# FinDA (Overheat Monitor) — 전체 파이프라인 (전처리 → 가설검증 → 라벨링 시행착오 → 최종모델)

이 노트북은 전처리, 관심지표 가설검증(H1/H2), 라벨링 시행착오(3분류→Buy/avoid→
Peak Give-back→Direct T+20 BHAR+q35), 최종 모델링/서빙 파이프라인을 **스토리라인
순서대로** 하나로 합친 것입니다. CSV 업로드나 원본 API 수집 코드는 포함하지
않았습니다.

## 실행 방법 — 로그인 없이 위에서 아래로 한 번에 실행

이 버전은 GCS 버킷(`gs://finda_project`)이 **읽기 전용으로 전체 공개**(`allUsers`:
`objectViewer`)되어 있다는 전제로 만들어졌습니다. 그래서:

- **읽기**: STAGE 3이 로그인 없이 공개 URL(`https://storage.googleapis.com/...`)로
  curated 원본을 직접 받습니다.
- **쓰기**: STAGE 11 이후 모든 계산 결과는 팀의 실제 GCS 버킷에 업로드하지 않고,
  전부 **이 세션의 로컬 폴더(`/content/finda_workspace/...`)에만** 저장합니다.
  누가, 몇 번을 실행해도 팀이 실제로 쓰는 GCS 데이터는 건드리지 않습니다.

즉 **Colab 메뉴에서 "런타임 → 모두 실행"을 한 번 누르면, 별도의 구글 로그인이나
권한 승인 없이 STAGE 1부터 STAGE 20까지 끝까지
실행됩니다.**

**Colab 사용 팁:** 각 STAGE는 `# STAGE N: ...` 형태의 최상위(H1) 마크다운 헤더로
시작합니다. Colab에서 헤더 왼쪽의 ▸ 화살표를 누르면 그 STAGE에 속한 모든 하위
셀(마크다운 소제목 + 코드)이 한 번에 접힙니다.

## STAGE 구성

| STAGE | 내용 | 상태 | 입력 |
|---|---|---|---|
| 1 | [부록·주석처리] STEP 0 — 현재 파일 및 환경 감사 | 참고용, 실행 안 됨 | Drive |
| 2 | [부록·주석처리] STEP 2A — 전체기간 표준데이터 재구축 (stock/attention/short_daily) | 참고용, 실행 안 됨 | Drive 원본 → GCS |
| 3 | GCS curated 6개 파일 로드 (공개 URL, 로그인 불필요) | 실행됨 — 여기부터 실제 실행 | GCS 공개 URL |
| 4 | [가설검증] 검색 관심 선행 신호 분석 | 실행됨 | STAGE 3 재사용 |
| 5 | [가설검증 H1] 관심 선행성 × 상승지속성 | 실행됨 Peak Give-back Legacy Gold | GCS derived (익명 클라이언트) |
| 6 | [가설검증 H2] 관심 지속성 × 상승지속성 | 실행됨 동일 | GCS derived (익명 클라이언트) |
| 7 | 3분류 라벨링 시도 | 폐기 | STAGE 3 재사용 |
| 8 | 매수/회피 모델 — Method 2: 이진화 점수모델(하락위험·상승지속) | 폐기 (탐색 근거로만 사용) | GCS 공개 URL 자체 재계산 |
| 9 | 매수/회피 모델 — Method 1: EWMA 상승/하락 장벽 ABCD 모델 (9개 전략) | 폐기 (First-touch가 최종 수익률을 보장 안 함) | GCS `events/modeling_eventset_10_20_60_v1.parquet` |
| 10 | Peak Give-back + 리크 발견/재검증 | 폐기(프레임워크) + 리크 발견은 최종 반영 | STAGE 3 재사용 |
| 11 | STEP 3 — 편입종목 + 시장일별데이터 최초 생성 | 최종 | 로컬(WORKSPACE_ROOT) |
| 12 | STEP 3 FIX — 과거편입 시장지표 재계산 | 최종 (STEP 3 패치) | 로컬 |
| 13 | STEP 4 — GCS+DuckDB Feature Group 기반 구축 | 최종 | 로컬 |
| 14 | STEP 5 — 이벤트셋 + Feature 생성 + 외국인수급 분석 | 최종 | 로컬 |
| 15 | STEP 6 — Direct T+20 BHAR × Q35 (**N=20/q=35% 선택 근거 삽입 + 최종 라벨 확정**) | 최종 | 로컬 |
| 16 | STEP 6B — Recall/Precision/Threshold 실제 분류성능 검증 | 최종 | 로컬 |
| 17 | STEP 6C — RF @ T+5 관심·시장·공매도 제거실험(Recipe ablation) | 최종 | 로컬 |
| 18 | STEP 6D — T+7/T+10 모델 최종확정 v2 | 최종 | 로컬 |
| 19 | STEP 7 — Serving Layer 생성 | 최종 | 로컬 |
| 20 | 급등이후 T+20 경로를 가장 잘 구분하는 사건 이전 20D 원천변수 분석 | 사후 EDA | 로컬 + STAGE 3 재사용 |


## 데이터 재사용 / 중복 다운로드 제거

STAGE 3에서 받은 `stock_daily`, `attention_raw_daily`, `attention_daily`, `short_daily`는
노트북 안에서 값이 바뀌지 않으므로, STAGE 4·7·10·20이 다시 다운로드하지 않고
재사용합니다. 단 **`market_daily`·`membership_daily`는 STAGE 11·12가 값을 새로
계산해서 같은 로컬 경로에 덮어쓰므로**, 그 이후 이 두 파일을 쓰는 STAGE(STAGE 20 등)는
STAGE 3의 초기 로드분이 아니라 그 시점의 최신 로컬 파일을 다시 읽습니다.


In [ ]:
import os
from pathlib import Path

# ==============================================================================
# [CONFIG]
# ==============================================================================
GCP_PROJECT_ID = os.environ.get("FINDA_GCP_PROJECT_ID", "project-7c970b4d-7375-4f3f-a24")
GCS_BUCKET_NAME = os.environ.get("FINDA_GCS_BUCKET", "finda_project")

# WORKSPACE_ROOT: 이 노트북이 만드는 모든 중간산출물(GCS 미러/DuckDB/모델/서빙 파일)의
# 로컬 작업공간입니다. 심사용 실행에서는 팀 GCS 버킷을 절대 쓰지 않도록(읽기 전용
# 공개 버킷에서 curated 원본만 받아오고, 그 이후 모든 계산 결과는 여기 로컬에만
# 저장) 예전에 Drive 경로였던 DRIVE_REFACTOR_ROOT를 로컬 경로로 재정의합니다.
# (변수명은 하위 STAGE 코드와의 호환을 위해 그대로 유지합니다.)
WORKSPACE_ROOT = Path(os.environ.get("FINDA_WORKSPACE_ROOT", "/content/finda_workspace"))
DRIVE_REFACTOR_ROOT = WORKSPACE_ROOT
DRIVE_ROOT = Path(os.environ.get("FINDA_DRIVE_ROOT", "/content/drive/MyDrive"))
DRIVE_PROJECT_ROOT = Path(os.environ.get("FINDA_DRIVE_PROJECT_ROOT", str(DRIVE_ROOT / "FinDA 프로젝트")))

WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)

try:
    from IPython.display import display  # noqa: F401
except ImportError:
    def display(*args, **kwargs):
        for a in args:
            print(a)

# ------------------------------------------------------------------------------
# Drive 마운트 함수 — 더 이상 자동으로 호출하지 않습니다 (읽기전용 공개 버킷 +
# 로컬 저장으로 전환하면서 Drive 자체가 필요 없어졌습니다). STAGE 8·9(보류 중)의
# 코드가 참조할 수 있어 정의만 남겨둡니다.
# ------------------------------------------------------------------------------
def ensure_drive_mounted(mountpoint="/content/drive"):
    import os
    if os.path.isdir(os.path.join(mountpoint, "MyDrive")):
        return  # 이미 마운트되어 있음
    from google.colab import drive
    drive.mount(mountpoint, force_remount=False)

# ------------------------------------------------------------------------------
# 공개 읽기전용 GCS 버킷에서 로그인 없이 파일을 받는 헬퍼.
# gs://{bucket}/{key} 형태의 객체를 https://storage.googleapis.com/{bucket}/{key}
# 공개 URL로 변환해 받습니다 (버킷이 allUsers:objectViewer로 공개되어 있어야 함).
# ------------------------------------------------------------------------------
import urllib.request

def gcs_public_download(key, dst_path, bucket=None):
    bucket = bucket or GCS_BUCKET_NAME
    url = f"https://storage.googleapis.com/{bucket}/{key}"
    dst_path = Path(dst_path)
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(url, str(dst_path))
    return dst_path

print("=" * 80)
print("FinDA 파이프라인 CONFIG")
print("=" * 80)
print("GCP_PROJECT_ID  :", GCP_PROJECT_ID)
print("GCS_BUCKET_NAME :", GCS_BUCKET_NAME)
print("WORKSPACE_ROOT  :", WORKSPACE_ROOT, "(로컬 — 팀 GCS/Drive에 쓰지 않음)")
print("=" * 80)

# STAGE 1: [부록 · 주석처리, 실행 안 됨] STEP 0 — 현재 파일 및 환경 감사

**태그:** 참고용

**원본 노트북:** `1__현재_파일_및_환경_감사.ipynb`

> Drive에 있는 원본 파일을 감사만 하고 수정하지 않는 코드입니다. GCS curated 데이터가 아닌 Drive 원본에 의존하므로 이 파일에서는 실행하지 않고 코드만 남겨둡니다.

## FinDA 리팩토링 — STEP 0. 현재 파일 및 환경 감사

이 노트북은 **GCS + DuckDB 파이프라인 리팩토링의 STEP 0만 수행**합니다.

### 목적
현재 Google Drive에 존재하는 FinDA 분석 파일을 감사하여 다음을 확인합니다.

- 파일명 / 경로 / 확장자 / 파일 크기
- 행 수 / 컬럼 수 / 컬럼명
- 날짜 컬럼 후보 / 시작일 / 종료일
- 종목코드 컬럼 후보 / 고유 종목 수
- 전체 중복행
- `code + date` 중복
- 컬럼별 결측
- `inf` 값
- 데이터 타입
- 파일 읽기 오류

그리고 리팩토링 관점에서 파일을 다음 5가지로 **1차 분류**합니다.

1. 그대로 사용할 후보
2. Parquet 변환 후보
3. 새로 계산할 데이터
4. Event-dependent 데이터
5. GCS 업로드 전 수정 필요

> **중요**
>
> - 이 노트북은 원본 파일을 수정·삭제·이동하지 않습니다.
> - GCS Bucket 생성은 하지 않습니다.
> - Parquet 변환도 아직 하지 않습니다.
> - Event / Target / Model 재생성도 하지 않습니다.
> - STEP 0 결과를 검토한 뒤에만 STEP 1로 넘어갑니다.

### 0-1. Google Drive 마운트

Drive 전체를 복사하는 것이 아니라 Colab에서 접근할 수 있도록 마운트만 합니다.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")

### 0-2. 감사 대상 폴더 지정

아래 `AUDIT_ROOTS`에 **실제 FinDA 데이터가 들어 있는 폴더만** 지정하세요.

예시 경로가 다르면 이 셀만 수정하면 됩니다.

In [ ]:
# from pathlib import Path

# # ============================================================
# # STEP 0 CONFIG
# # ============================================================

# AUDIT_ROOTS = [
#     Path(str(DRIVE_ROOT / "FinDA")),
# ]

# OUTPUT_DIR = Path(
#     "/content/drive/MyDrive/FinDA_refactor/outputs/step0_audit"
# )

# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# print("=== 감사 대상 ===")
# for p in AUDIT_ROOTS:
#     print(f"- {p} | exists={p.exists()}")

# print("\n=== 결과 저장 위치 ===")
# print(OUTPUT_DIR)

### 0-3. 라이브러리 준비

In [ ]:
# !pip -q install pyarrow openpyxl

# import re
# import gc
# import json
# import warnings

# import numpy as np
# import pandas as pd

# warnings.filterwarnings("ignore")

# pd.set_option("display.max_columns", 200)
# pd.set_option("display.max_colwidth", 200)

# print("라이브러리 준비 완료")

### 0-4. 분석 파일 탐색

PDF/PPT/이미지 등 문서 파일은 제외하고 실제 분석 데이터 파일만 찾습니다.

In [ ]:
# SUPPORTED_EXTENSIONS = {
#     ".csv",
#     ".parquet",
#     ".xlsx",
#     ".xls",
#     ".feather",
#     ".pkl",
#     ".pickle",
# }

# files = []

# for root in AUDIT_ROOTS:
#     if not root.exists():
#         print(f"⚠️ 존재하지 않는 경로: {root}")
#         continue

#     for path in root.rglob("*"):
#         if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS:
#             files.append(path)

# files = sorted(set(files))

# print(f"발견한 분석 파일: {len(files):,}개\n")

# for p in files[:50]:
#     print(p)

# if len(files) > 50:
#     print(f"... 외 {len(files)-50:,}개")

#### 확인
발견된 파일 수가 예상보다 지나치게 많거나 적다면 다음 셀로 넘어가기 전에 `AUDIT_ROOTS`를 확인하세요.

### 0-5. 날짜 / 종목코드 후보 정의

In [ ]:
# DATE_NAMES = {
#     "date",
#     "trade_date",
#     "trading_date",
#     "event_date",
#     "dt",
#     "day",
#     "trd_dd",
#     "base_date",
#     "일자",
#     "날짜",
#     "거래일",
#     "기준일",
# }

# CODE_NAMES = {
#     "code",
#     "ticker",
#     "symbol",
#     "stock_code",
#     "ticker_code",
#     "isu_cd",
#     "isu_srt_cd",
#     "short_code",
#     "종목코드",
#     "단축코드",
# }

# # 숫자 6자리뿐 아니라 0126Z0 같은 영숫자 코드도 허용
# CODE_PATTERN = re.compile(r"^[0-9A-Z]{6}$")

### 0-6. 파일 로더

원본 파일은 읽기만 합니다.
CSV는 `utf-8-sig → utf-8 → cp949` 순서로 시도합니다.

In [ ]:
# def read_data_file(path: Path):
#     ext = path.suffix.lower()

#     if ext == ".csv":
#         attempts = [
#             {"encoding": "utf-8-sig", "low_memory": False},
#             {"encoding": "utf-8", "low_memory": False},
#             {"encoding": "cp949", "low_memory": False},
#         ]

#         last_error = None

#         for kwargs in attempts:
#             try:
#                 return pd.read_csv(path, **kwargs)
#             except Exception as e:
#                 last_error = e

#         raise last_error

#     if ext == ".parquet":
#         return pd.read_parquet(path)

#     if ext in {".xlsx", ".xls"}:
#         return pd.read_excel(path)

#     if ext == ".feather":
#         return pd.read_feather(path)

#     if ext in {".pkl", ".pickle"}:
#         return pd.read_pickle(path)

#     raise ValueError(f"지원하지 않는 확장자: {ext}")

### 0-7. 날짜 / 종목코드 자동 탐지 함수

In [ ]:
# def detect_date_column(df):
#     candidates = []

#     for col in df.columns:
#         name = str(col).strip().lower()
#         name_score = 0

#         if name in DATE_NAMES:
#             name_score += 3

#         if "date" in name:
#             name_score += 2

#         if any(x in name for x in ["일자", "날짜", "거래일", "기준일"]):
#             name_score += 2

#         s = df[col]

#         if pd.api.types.is_datetime64_any_dtype(s):
#             parse_rate = 1.0
#         else:
#             sample = s.dropna()

#             if len(sample) > 5000:
#                 sample = sample.sample(5000, random_state=42)

#             if len(sample) == 0:
#                 parse_rate = 0.0
#             else:
#                 parsed = pd.to_datetime(sample, errors="coerce")
#                 parse_rate = float(parsed.notna().mean())

#         score = name_score + parse_rate

#         if name_score > 0 or parse_rate >= 0.8:
#             candidates.append({
#                 "column": str(col),
#                 "score": float(score),
#                 "parse_rate": float(parse_rate),
#             })

#     if not candidates:
#         return None, []

#     candidates = sorted(
#         candidates,
#         key=lambda x: (x["score"], x["parse_rate"]),
#         reverse=True,
#     )

#     return candidates[0]["column"], candidates


# def normalize_code_series(s):
#     s = s.astype("string").str.strip().str.upper()

#     # CSV에서 005930이 5930.0 형태로 읽힌 경우 보정
#     s = s.str.replace(r"\.0$", "", regex=True)

#     numeric_mask = s.str.fullmatch(r"\d{1,6}", na=False)
#     s.loc[numeric_mask] = s.loc[numeric_mask].str.zfill(6)

#     return s


# def detect_code_column(df):
#     candidates = []

#     for col in df.columns:
#         name = str(col).strip().lower()
#         name_score = 0

#         if name in CODE_NAMES:
#             name_score += 3

#         if "code" in name or "ticker" in name or "symbol" in name:
#             name_score += 2

#         if "종목" in name and "코드" in name:
#             name_score += 3

#         s = normalize_code_series(df[col])
#         sample = s.dropna()

#         if len(sample) > 5000:
#             sample = sample.sample(5000, random_state=42)

#         if len(sample):
#             format_rate = float(
#                 sample.str.match(CODE_PATTERN, na=False).mean()
#             )
#         else:
#             format_rate = 0.0

#         score = name_score + format_rate

#         if name_score > 0 or format_rate >= 0.8:
#             candidates.append({
#                 "column": str(col),
#                 "score": float(score),
#                 "format_rate": float(format_rate),
#             })

#     if not candidates:
#         return None, []

#     candidates = sorted(
#         candidates,
#         key=lambda x: (x["score"], x["format_rate"]),
#         reverse=True,
#     )

#     return candidates[0]["column"], candidates

### 0-8. 파일 감사 함수

각 파일에서 다음 항목을 검사합니다.

- 파일 기본 정보
- 날짜/종목코드 후보
- 날짜 범위
- 종목 수
- 중복행
- `code + date` 중복
- 컬럼별 결측
- 컬럼별 `inf`
- dtype

In [ ]:
# def audit_dataframe(df, path: Path):

#     # --------------------------------------------------------
#     # 기본 정보
#     # --------------------------------------------------------
#     n_rows = len(df)
#     n_cols = len(df.columns)
#     file_size_mb = path.stat().st_size / (1024 ** 2)

#     # --------------------------------------------------------
#     # 날짜 / 코드 후보
#     # --------------------------------------------------------
#     date_col, date_candidates = detect_date_column(df)
#     code_col, code_candidates = detect_code_column(df)

#     start_date = None
#     end_date = None

#     if date_col is not None:
#         parsed_date = pd.to_datetime(df[date_col], errors="coerce")

#         if parsed_date.notna().any():
#             start_date = parsed_date.min()
#             end_date = parsed_date.max()

#     # --------------------------------------------------------
#     # 종목 수
#     # --------------------------------------------------------
#     unique_codes = None

#     if code_col is not None:
#         codes = normalize_code_series(df[code_col])
#         unique_codes = int(codes.nunique(dropna=True))

#     # --------------------------------------------------------
#     # 전체 행 중복
#     # --------------------------------------------------------
#     try:
#         duplicate_rows = int(df.duplicated().sum())
#     except Exception:
#         duplicate_rows = None

#     # --------------------------------------------------------
#     # code + date 중복
#     # --------------------------------------------------------
#     code_date_duplicates = None

#     if code_col is not None and date_col is not None:
#         temp = pd.DataFrame({
#             "_code": normalize_code_series(df[code_col]),
#             "_date": pd.to_datetime(df[date_col], errors="coerce"),
#         })

#         valid = temp["_code"].notna() & temp["_date"].notna()

#         # 중복 키에 속한 모든 행 수
#         code_date_duplicates = int(
#             temp.loc[valid]
#                 .duplicated(
#                     subset=["_code", "_date"],
#                     keep=False,
#                 )
#                 .sum()
#         )

#     # --------------------------------------------------------
#     # 컬럼 프로파일
#     # --------------------------------------------------------
#     column_records = []
#     total_missing = 0
#     total_inf = 0

#     for col in df.columns:
#         s = df[col]

#         null_count = int(s.isna().sum())
#         null_pct = (
#             null_count / n_rows
#             if n_rows > 0
#             else np.nan
#         )

#         inf_count = 0

#         if pd.api.types.is_numeric_dtype(s):
#             arr = pd.to_numeric(s, errors="coerce")
#             inf_count = int(np.isinf(arr).sum())

#         try:
#             unique_count = int(s.nunique(dropna=True))
#         except Exception:
#             unique_count = None

#         total_missing += null_count
#         total_inf += inf_count

#         column_records.append({
#             "file_name": path.name,
#             "file_path": str(path),
#             "column": str(col),
#             "dtype": str(s.dtype),
#             "null_count": null_count,
#             "null_pct": null_pct,
#             "inf_count": inf_count,
#             "unique_count": unique_count,
#         })

#     record = {
#         "file_name": path.name,
#         "file_path": str(path),
#         "extension": path.suffix.lower(),
#         "file_size_mb": round(file_size_mb, 3),
#         "rows": n_rows,
#         "columns": n_cols,
#         "column_names": json.dumps(
#             [str(x) for x in df.columns],
#             ensure_ascii=False,
#         ),
#         "date_column": date_col,
#         "date_candidates": json.dumps(
#             date_candidates,
#             ensure_ascii=False,
#             default=str,
#         ),
#         "start_date": start_date,
#         "end_date": end_date,
#         "code_column": code_col,
#         "code_candidates": json.dumps(
#             code_candidates,
#             ensure_ascii=False,
#             default=str,
#         ),
#         "unique_codes": unique_codes,
#         "duplicate_rows": duplicate_rows,
#         "code_date_duplicates": code_date_duplicates,
#         "total_missing_cells": total_missing,
#         "total_inf_cells": total_inf,
#     }

#     return record, column_records

### 0-9. 전체 파일 감사 실행

읽기 오류가 발생한 파일은 전체 실행을 멈추지 않고 따로 기록합니다.

In [ ]:
# file_records = []
# column_records = []
# errors = []

# for i, path in enumerate(files, start=1):

#     print(
#         f"[{i:>3}/{len(files)}] {path.name}",
#         end=" ... "
#     )

#     try:
#         df = read_data_file(path)

#         file_record, cols = audit_dataframe(
#             df,
#             path,
#         )

#         file_records.append(file_record)
#         column_records.extend(cols)

#         print(
#             f"OK | "
#             f"{len(df):,} rows × "
#             f"{len(df.columns):,} cols"
#         )

#         del df
#         gc.collect()

#     except Exception as e:

#         print("❌ ERROR")

#         errors.append({
#             "file_name": path.name,
#             "file_path": str(path),
#             "error_type": type(e).__name__,
#             "error_message": str(e),
#         })

#         gc.collect()


# file_audit = pd.DataFrame(file_records)
# column_audit = pd.DataFrame(column_records)
# error_audit = pd.DataFrame(errors)

# print("\n==========================")
# print("STEP 0 감사 완료")
# print("==========================")
# print("성공:", len(file_audit))
# print("실패:", len(error_audit))

### 0-10. GCS 업로드 전 문제 후보 탐지

현재 단계에서는 문제를 **수정하지 않고 표시만** 합니다.

In [ ]:
# issue_records = []

# for _, r in file_audit.iterrows():

#     path = r["file_path"]
#     name = r["file_name"]

#     def add_issue(severity, issue):
#         issue_records.append({
#             "file_name": name,
#             "file_path": path,
#             "severity": severity,
#             "issue": issue,
#         })

#     if r["rows"] == 0:
#         add_issue(
#             "CRITICAL",
#             "0행 파일"
#         )

#     if (
#         pd.notna(r["duplicate_rows"])
#         and r["duplicate_rows"] > 0
#     ):
#         add_issue(
#             "WARNING",
#             f"전체 중복행 {int(r['duplicate_rows']):,}건"
#         )

#     if (
#         pd.notna(r["code_date_duplicates"])
#         and r["code_date_duplicates"] > 0
#     ):
#         add_issue(
#             "CRITICAL",
#             f"code + date 중복 "
#             f"{int(r['code_date_duplicates']):,}행"
#         )

#     if r["total_inf_cells"] > 0:
#         add_issue(
#             "CRITICAL",
#             f"inf 값 {int(r['total_inf_cells']):,}개"
#         )

#     if r["date_column"] is None:
#         add_issue(
#             "INFO",
#             "날짜 컬럼 자동 탐지 실패"
#         )

#     if r["code_column"] is None:
#         add_issue(
#             "INFO",
#             "종목코드 컬럼 자동 탐지 실패"
#         )

# issues = pd.DataFrame(issue_records)

# display(issues.head(100))

### 0-11. 결측 / inf / 중복키 리포트

In [ ]:
# missing_report = (
#     column_audit[
#         column_audit["null_count"] > 0
#     ]
#     .sort_values(
#         ["file_name", "null_pct"],
#         ascending=[True, False],
#     )
#     .reset_index(drop=True)
# )

# inf_report = (
#     column_audit[
#         column_audit["inf_count"] > 0
#     ]
#     .sort_values(
#         ["file_name", "inf_count"],
#         ascending=[True, False],
#     )
#     .reset_index(drop=True)
# )

# duplicate_key_report = (
#     file_audit[
#         file_audit["code_date_duplicates"].fillna(0) > 0
#     ]
#     [
#         [
#             "file_name",
#             "rows",
#             "code_column",
#             "date_column",
#             "code_date_duplicates",
#             "file_path",
#         ]
#     ]
#     .sort_values(
#         "code_date_duplicates",
#         ascending=False,
#     )
#     .reset_index(drop=True)
# )

# print("결측 포함 컬럼 수:", len(missing_report))
# print("inf 포함 컬럼 수:", len(inf_report))
# print("code+date 중복 파일 수:", len(duplicate_key_report))

### 0-12. 파일 요약 보기

In [ ]:
# summary_cols = [
#     "file_name",
#     "extension",
#     "file_size_mb",
#     "rows",
#     "columns",
#     "date_column",
#     "start_date",
#     "end_date",
#     "code_column",
#     "unique_codes",
#     "duplicate_rows",
#     "code_date_duplicates",
#     "total_missing_cells",
#     "total_inf_cells",
# ]

# display(
#     file_audit[
#         summary_cols
#     ].sort_values(
#         "file_size_mb",
#         ascending=False,
#     )
# )

### 0-13. 리팩토링 관점 1차 자동 분류

이 분류는 **최종 판정이 아니라 후보 분류**입니다.

특히 기존의 Event / Target / Model / OOF 산출물은 정상 파일이어도 새 파이프라인의 Source of Truth로 바로 사용하지 않고, **parity/reference 용도**로 보존하는 것이 기본 방향입니다.

In [ ]:
# def classify_file(row):

#     name = row["file_name"].lower()
#     path = row["file_path"].lower()
#     ext = row["extension"]

#     combined = name + " " + path

#     critical_problem = (
#         row["rows"] == 0
#         or row["total_inf_cells"] > 0
#         or (
#             pd.notna(row["code_date_duplicates"])
#             and row["code_date_duplicates"] > 0
#         )
#     )

#     # ------------------------------------------------
#     # Event-dependent
#     # ------------------------------------------------
#     event_dependent_keywords = [
#         "lead_days",
#         "lead_day",
#         "event_attention",
#         "event_feature",
#     ]

#     if any(
#         x in combined
#         for x in event_dependent_keywords
#     ):
#         return "4_Event-dependent 데이터"

#     # ------------------------------------------------
#     # 새 config 체계에서 재생성할 영역
#     # ------------------------------------------------
#     rebuild_keywords = [
#         "event",
#         "target",
#         "label",
#         "model_result",
#         "prediction",
#         "oof",
#         "fold",
#     ]

#     if any(
#         x in combined
#         for x in rebuild_keywords
#     ):
#         return "3_새로 계산할 데이터"

#     # ------------------------------------------------
#     # 데이터 문제
#     # ------------------------------------------------
#     if critical_problem:
#         return "5_GCS 업로드 전 수정 필요"

#     # ------------------------------------------------
#     # 이미 parquet
#     # ------------------------------------------------
#     if ext == ".parquet":
#         return "1_그대로 사용할 후보"

#     # ------------------------------------------------
#     # 내용은 살리되 parquet 변환 후보
#     # ------------------------------------------------
#     if ext in {
#         ".csv",
#         ".xlsx",
#         ".xls",
#         ".feather",
#         ".pkl",
#         ".pickle",
#     }:
#         return "2_Parquet 변환 후보"

#     return "검토 필요"


# classification = file_audit.copy()

# classification["auto_classification"] = (
#     classification.apply(
#         classify_file,
#         axis=1,
#     )
# )

# display(
#     classification[
#         [
#             "file_name",
#             "extension",
#             "rows",
#             "columns",
#             "start_date",
#             "end_date",
#             "unique_codes",
#             "auto_classification",
#             "file_path",
#         ]
#     ].sort_values(
#         ["auto_classification", "file_name"]
#     )
# )

### 0-14. 목표 Canonical Artifact 존재 여부

현재 Drive에 아래 산출물 이름이 이미 존재하는지 **참고용으로만** 확인합니다.

- stock_daily
- attention_daily
- short_daily
- market_daily
- event_master
- event_attention_features
- target H10 / H20 / H40

In [ ]:
# EXPECTED_ARTIFACTS = {
#     "stock_daily": [
#         "stock_daily"
#     ],
#     "attention_daily": [
#         "attention_daily"
#     ],
#     "short_daily": [
#         "short_daily"
#     ],
#     "market_daily": [
#         "market_daily"
#     ],
#     "event_master": [
#         "event_master"
#     ],
#     "event_attention_features": [
#         "event_attention"
#     ],
#     "target_h10": [
#         "target_peak_giveback_h10"
#     ],
#     "target_h20": [
#         "target_peak_giveback_h20"
#     ],
#     "target_h40": [
#         "target_peak_giveback_h40"
#     ],
# }

# artifact_records = []

# for artifact, keywords in EXPECTED_ARTIFACTS.items():

#     matches = []

#     for p in files:
#         p_lower = str(p).lower()

#         if any(
#             keyword.lower() in p_lower
#             for keyword in keywords
#         ):
#             matches.append(str(p))

#     artifact_records.append({
#         "artifact": artifact,
#         "status": (
#             "FOUND"
#             if matches
#             else "MISSING"
#         ),
#         "matches": "\n".join(matches),
#     })

# artifact_status = pd.DataFrame(
#     artifact_records
# )

# display(artifact_status)

# STAGE 2: [부록 · 주석처리, 실행 안 됨] STEP 2A — 전체기간 표준 Canonical Daily Feature Store 재구축

**태그:** 참고용

**원본 노트북:** `2A__최종_전체기간_표준데이터_재구축.ipynb`

> stock_daily / attention_daily / short_daily 세 curated 파일을 Drive 원본(두 기간 병합)에서 재생성해 GCS에 업로드하는 코드입니다. Drive 원본에 의존하므로 실행하지 않고 코드만 남겨둡니다.

## 전체기간 Canonical Daily Feature Store 재생성

### 범위

실제 보유 데이터 전체기간:

```text
2021-06-11 ~ 2026-08-10
```

을 기준으로 다음 세 파일을 최종 재생성합니다.

```text
gs://finda_project/curated/

├── stock_daily.parquet
├── attention_daily.parquet
└── short_daily.parquet
```

---

## 입력 데이터

### 주가

두 기간을 병합합니다.

```text
kospi200_20210611_20260611_v3
+
kospi200_20260612_20260810_v3
```

### 관심지표

이미 전체기간이 병합된 파일 하나를 사용합니다.

```text
ATTENTION_SET_20210611_20260810.parquet
```

### 공매도

두 기간의 원천 cache를 병합합니다.

```text
kospi200_short_analysis_20210611_20260611
+
kospi200_short_analysis_20260612_20260810
```

---

## 기존 전처리 원칙 유지

#### 주가
- `001260`은 legacy/raw에는 남기되 canonical에서 제외
- `0126Z0`은 정상 유지
- `ticker`가 primary key
- `trading_value_growth`의 ±inf는 NaN으로 정규화
- 다른 inf는 자동 처리하지 않고 중단

#### 관심지표
- Event-independent 9개 Feature만 canonical에 저장
- calendar day 기준 유지
- rolling 초기 결측 유지
- 결측을 0으로 대치하지 않음

#### 주가 × 관심지표 Universe
- 정상 주가 종목이 관심지표에 없다는 이유만으로 stock_daily에서 삭제하지 않음
- stock + attention 존재 → 모델 Attention 사용 가능
- stock만 존재 → stock에는 보존, Attention 모델 eligibility에서 제외
- attention만 존재 → canonical attention에서 제외하고 QA에 기록
- `001260`은 기존 검증 결과에 따라 명시적으로 제외

#### 공매도
- raw의 `short_balance_share`와 `short_balance_shares`를 모두 인식
- canonical 이름은 `short_balance_shares`
- 공매도 금지기간의 구조적 NaN 유지
- 미래정보 Feature 저장 금지
- `999999.99` sentinel 사용 금지
- 0 → 양수 신규발생은 별도 flag로 표현
- 전체 NaN Feature가 생기면 삭제하지 않고 **실패 처리**

---

## 매우 중요

```text
데이터 보유기간 = 2021-06-11 ~ 2026-08-10
```

과

```text
Development / OOT / Target Horizon에 실제 사용할 기간
```

은 서로 다릅니다.

이번 노트북은 **보유 데이터 전체를 GCS에 저장**하기만 합니다.

모델링 기간 선택은 이후 config에서 별도로 수행합니다.

### 1. Google Drive 연결

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive", force_remount=False)

### 2. 라이브러리

In [ ]:
# !pip -q install pyarrow

# from pathlib import Path
# import hashlib
# import json
# import warnings

# import numpy as np
# import pandas as pd
# import pyarrow.parquet as pq

# warnings.filterwarnings("ignore")

# pd.set_option("display.max_columns", 250)
# pd.set_option("display.width", 280)
# pd.set_option("display.max_rows", 300)

# print("준비 완료")

### 3. 프로젝트 / 소스 경로

In [ ]:
# PROJECT_ID = GCP_PROJECT_ID
# BUCKET = GCS_BUCKET_NAME

# BASE = Path(DRIVE_PROJECT_ROOT)
# REFACTOR_ROOT = Path(DRIVE_REFACTOR_ROOT)

# # ─────────────────────────────────────────────
# # STOCK
# # ─────────────────────────────────────────────
# STOCK_OLD_DIR = BASE / "cache" / "kospi200_20210611_20260611_v3"
# STOCK_NEW_DIR = BASE / "cache" / "kospi200_20260612_20260810_v3"

# # 기존 34-column schema 계약을 읽기 위한 이전 canonical
# PREVIOUS_STOCK_CANONICAL = REFACTOR_ROOT / "curated" / "stock_daily.parquet"

# # ─────────────────────────────────────────────
# # ATTENTION
# # ─────────────────────────────────────────────
# ATTENTION_CANDIDATES = [
#     BASE / "attention_set_v1" / "ATTENTION_SET_20210611_20260810.parquet",
#     BASE / "cache" / "attention_set_v1" / "ATTENTION_SET_20210611_20260810.parquet",
# ]

# ATTENTION_SOURCE = next(
#     (p for p in ATTENTION_CANDIDATES if p.exists()),
#     None,
# )

# # ─────────────────────────────────────────────
# # SHORT
# # ─────────────────────────────────────────────
# SHORT_OLD_ROOT = BASE / "cache" / "kospi200_short_analysis_20210611_20260611"
# SHORT_NEW_ROOT = BASE / "cache" / "kospi200_short_analysis_20260612_20260810"

# # ─────────────────────────────────────────────
# # OUTPUT
# # ─────────────────────────────────────────────
# CURATED_DIR = REFACTOR_ROOT / "curated"
# OUTPUT_DIR = REFACTOR_ROOT / "outputs" / "step2_final_fullrange"

# CURATED_DIR.mkdir(parents=True, exist_ok=True)
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# STOCK_OUT = CURATED_DIR / "stock_daily.parquet"
# ATTENTION_OUT = CURATED_DIR / "attention_daily.parquet"
# SHORT_OUT = CURATED_DIR / "short_daily.parquet"

# START_DATE = pd.Timestamp("2021-06-11")
# END_DATE = pd.Timestamp("2026-08-10")

# EXPLICIT_STOCK_EXCLUSIONS = {
#     "001260": (
#         "기존 cache의 ticker_name 오매핑이 확인되었고 "
#         "Historical KOSPI200/Attention universe 대상이 아니므로 canonical에서 제외"
#     )
# }

# for label, p in {
#     "STOCK OLD": STOCK_OLD_DIR,
#     "STOCK NEW": STOCK_NEW_DIR,
#     "SHORT OLD": SHORT_OLD_ROOT,
#     "SHORT NEW": SHORT_NEW_ROOT,
# }.items():
#     print(label, ":", p, "| exists:", p.exists())
#     if not p.exists():
#         raise FileNotFoundError(p)

# print("ATTENTION:", ATTENTION_SOURCE)

# if ATTENTION_SOURCE is None:
#     raise FileNotFoundError("ATTENTION_SET_20210611_20260810.parquet을 찾지 못했습니다.")

### 4. 공통 함수

In [ ]:
# def normalize_ticker_value(x):
#     if pd.isna(x):
#         return pd.NA

#     s = str(x).strip().upper()

#     if s.endswith(".0"):
#         s = s[:-2]

#     if s.isdigit():
#         s = s.zfill(6)

#     return s


# def normalize_ticker_series(s):
#     return s.map(normalize_ticker_value).astype("string")


# def numeric_clean(s):
#     if pd.api.types.is_numeric_dtype(s):
#         return pd.to_numeric(s, errors="coerce")

#     return pd.to_numeric(
#         s.astype("string")
#          .str.replace(",", "", regex=False)
#          .str.strip(),
#         errors="coerce",
#     )


# def sha256(path, block=1024 * 1024):
#     h = hashlib.sha256()

#     with open(path, "rb") as f:
#         while True:
#             chunk = f.read(block)
#             if not chunk:
#                 break
#             h.update(chunk)

#     return h.hexdigest()


# def safe_ratio_pct(num, den):
#     num = pd.to_numeric(num, errors="coerce")
#     den = pd.to_numeric(den, errors="coerce")

#     out = pd.Series(np.nan, index=num.index, dtype="float64")

#     ok = (
#         num.notna()
#         & den.notna()
#         & den.gt(0)
#     )

#     out.loc[ok] = (
#         num.loc[ok]
#         / den.loc[ok]
#         * 100.0
#     )

#     return out

## PART A — STOCK DAILY 전체기간 재생성

### 5. 기존 canonical stock schema 계약 확인

In [ ]:
# if not PREVIOUS_STOCK_CANONICAL.exists():
#     raise FileNotFoundError(
#         "기존 stock_daily.parquet이 없어 34-column schema 계약을 확인할 수 없습니다: "
#         + str(PREVIOUS_STOCK_CANONICAL)
#     )

# previous_stock = pd.read_parquet(PREVIOUS_STOCK_CANONICAL)
# STOCK_CANONICAL_COLUMNS = list(previous_stock.columns)

# print("기존 canonical columns:", len(STOCK_CANONICAL_COLUMNS))
# print(STOCK_CANONICAL_COLUMNS)

# required_stock_cols = {
#     "ticker",
#     "ticker_name",
#     "date",
#     "open",
#     "high",
#     "low",
#     "close",
#     "volume",
#     "trading_value",
#     "listed_shares",
#     "market_cap",
# }

# missing_contract = required_stock_cols - set(STOCK_CANONICAL_COLUMNS)

# if missing_contract:
#     raise RuntimeError(
#         "기존 canonical schema에 필수 stock 컬럼이 없습니다: "
#         + ", ".join(sorted(missing_contract))
#     )

### 6. 주가 old/new 파일 및 Universe 비교

In [ ]:
# import re

# STOCK_FILE_RE = re.compile(r"^[0-9A-Z]{6}\.csv$", re.IGNORECASE)

# def split_stock_csvs(stock_dir):
#     all_csv = sorted(stock_dir.glob("*.csv"))

#     ticker_csv = [
#         p for p in all_csv
#         if STOCK_FILE_RE.fullmatch(p.name)
#     ]

#     non_ticker_csv = [
#         p for p in all_csv
#         if not STOCK_FILE_RE.fullmatch(p.name)
#     ]

#     return all_csv, ticker_csv, non_ticker_csv


# stock_old_all_csv, stock_old_files, stock_old_non_ticker = split_stock_csvs(
#     STOCK_OLD_DIR
# )

# stock_new_all_csv, stock_new_files, stock_new_non_ticker = split_stock_csvs(
#     STOCK_NEW_DIR
# )

# old_file_tickers = {
#     normalize_ticker_value(p.stem)
#     for p in stock_old_files
# }

# new_file_tickers = {
#     normalize_ticker_value(p.stem)
#     for p in stock_new_files
# }

# stock_universe_source_compare = pd.DataFrame([{
#     "old_all_csv_n": len(stock_old_all_csv),
#     "old_ticker_csv_n": len(stock_old_files),
#     "old_non_ticker_csv_n": len(stock_old_non_ticker),

#     "new_all_csv_n": len(stock_new_all_csv),
#     "new_ticker_csv_n": len(stock_new_files),
#     "new_non_ticker_csv_n": len(stock_new_non_ticker),

#     "old_ticker_n": len(old_file_tickers),
#     "new_ticker_n": len(new_file_tickers),
#     "intersection_n": len(old_file_tickers & new_file_tickers),
#     "old_only_n": len(old_file_tickers - new_file_tickers),
#     "new_only_n": len(new_file_tickers - old_file_tickers),
#     "union_n_before_explicit_exclusion": len(
#         old_file_tickers | new_file_tickers
#     ),
# }])

# display(stock_universe_source_compare)

# print("\n[OLD] 종목파일이 아닌 CSV (skip 예정)")
# for p in stock_old_non_ticker:
#     print("-", p.name, "| bytes:", p.stat().st_size)

# print("\n[NEW] 종목파일이 아닌 CSV (skip 예정)")
# for p in stock_new_non_ticker:
#     print("-", p.name, "| bytes:", p.stat().st_size)

# print("\nold only ticker:", sorted(old_file_tickers - new_file_tickers))
# print("new only ticker:", sorted(new_file_tickers - old_file_tickers))

# print(
#     "\n※ 위 non-ticker CSV는 파일명이 6자리 종목코드 형식이 아니므로 "
#     "주가 canonical source에서 제외합니다."
# )

### 7. 주가 두 기간 로드 + schema 검증

In [ ]:
# stock_frames = []
# stock_read_errors = []
# stock_schema_issues = []

# for period_name, files in [
#     ("20210611_20260611", stock_old_files),
#     ("20260612_20260810", stock_new_files),
# ]:
#     for path in files:
#         try:
#             # 6자리 ticker 파일은 빈 파일도 오류로 봄
#             if path.stat().st_size == 0:
#                 raise ValueError(
#                     "6자리 ticker 파일이지만 파일 크기가 0 byte"
#                 )

#             x = pd.read_csv(
#                 path,
#                 dtype={"ticker": "string"},
#                 low_memory=False,
#             )

#             if x.empty and len(x.columns) == 0:
#                 raise ValueError(
#                     "6자리 ticker 파일이지만 header/data가 없음"
#                 )

#             file_cols = list(x.columns)

#             missing_cols = [
#                 c for c in STOCK_CANONICAL_COLUMNS
#                 if c not in file_cols
#             ]

#             if missing_cols:
#                 stock_schema_issues.append({
#                     "file": str(path),
#                     "file_name": path.name,
#                     "file_ticker": normalize_ticker_value(path.stem),
#                     "period": period_name,
#                     "missing_column_n": len(missing_cols),
#                     "missing_columns": "|".join(missing_cols),
#                 })
#                 continue

#             # canonical 34-column contract만 유지
#             x = x[STOCK_CANONICAL_COLUMNS].copy()

#             x["ticker"] = normalize_ticker_series(
#                 x["ticker"]
#             )

#             x["date"] = pd.to_datetime(
#                 x["date"],
#                 errors="coerce",
#             ).dt.normalize()

#             file_ticker = normalize_ticker_value(path.stem)
#             internal_tickers = (
#                 x["ticker"]
#                 .dropna()
#                 .astype(str)
#                 .unique()
#                 .tolist()
#             )

#             # 실제 종목파일이면 파일명 ticker와 내부 ticker가 반드시 일치
#             if (
#                 len(internal_tickers) != 1
#                 or internal_tickers[0] != file_ticker
#             ):
#                 raise ValueError(
#                     f"filename ticker={file_ticker}, "
#                     f"internal ticker={internal_tickers[:5]}"
#                 )

#             x["_source_period"] = period_name
#             x["_source_file"] = path.name

#             stock_frames.append(x)

#         except Exception as e:
#             stock_read_errors.append({
#                 "file": str(path),
#                 "file_name": path.name,
#                 "file_ticker": normalize_ticker_value(path.stem),
#                 "period": period_name,
#                 "error": repr(e),
#             })

# print("6자리 ticker CSV 대상:", len(stock_old_files) + len(stock_new_files))
# print("읽기 성공 file n:", len(stock_frames))
# print("schema issue:", len(stock_schema_issues))
# print("read error:", len(stock_read_errors))

# if stock_schema_issues:
#     print("\n⚠️ 실제 6자리 ticker 파일 schema issue")
#     display(pd.DataFrame(stock_schema_issues).head(100))

# if stock_read_errors:
#     print("\n⚠️ 실제 6자리 ticker 파일 read error")
#     display(pd.DataFrame(stock_read_errors).head(100))

# # 중요:
# # non-ticker 보조 CSV는 앞 셀에서 이미 skip했으므로,
# # 여기서 발생하는 오류는 실제 ticker 파일 문제임.
# if stock_schema_issues or stock_read_errors:
#     raise RuntimeError(
#         "실제 6자리 ticker CSV에서 schema/read issue가 있습니다. "
#         "이 경우는 자동 무시하지 않고 원본 확인이 필요합니다."
#     )

# stock_raw_all = pd.concat(
#     stock_frames,
#     ignore_index=True,
# )

# stock_raw_all = stock_raw_all[
#     stock_raw_all["date"].between(
#         START_DATE,
#         END_DATE,
#         inclusive="both",
#     )
# ].copy()

# print("\nconcat:", stock_raw_all.shape)
# print("ticker:", stock_raw_all["ticker"].nunique())
# print(
#     "date:",
#     stock_raw_all["date"].min(),
#     "~",
#     stock_raw_all["date"].max(),
# )

# print("✅ 실제 ticker 파일만 대상으로 stock source load 완료")

### 8. `001260` 명시적 제외

In [ ]:
# print(
#     "001260 제외 전 rows:",
#     int(stock_raw_all["ticker"].eq("001260").sum())
# )

# print(
#     "0126Z0 rows:",
#     int(stock_raw_all["ticker"].eq("0126Z0").sum())
# )

# stock_raw_all = stock_raw_all[
#     ~stock_raw_all["ticker"].isin(
#         EXPLICIT_STOCK_EXCLUSIONS.keys()
#     )
# ].copy()

# assert not stock_raw_all["ticker"].eq("001260").any()
# assert stock_raw_all["ticker"].eq("0126Z0").any()

# print("✅ 001260 제외 / 0126Z0 유지")

### 9. old/new 경계 중복 및 값 충돌 검사

In [ ]:
# dup = stock_raw_all[
#     stock_raw_all.duplicated(
#         ["ticker", "date"],
#         keep=False,
#     )
# ].copy()

# print("중복 key 포함 rows:", len(dup))

# conflicts = []

# if len(dup):
#     value_cols = [
#         c for c in STOCK_CANONICAL_COLUMNS
#         if c not in ["ticker", "date"]
#     ]

#     for (ticker, date), g in dup.groupby(
#         ["ticker", "date"],
#         sort=False,
#     ):
#         for c in value_cols:
#             vals = (
#                 g[c]
#                 .dropna()
#                 .astype(str)
#                 .unique()
#             )

#             if len(vals) > 1:
#                 conflicts.append({
#                     "ticker": ticker,
#                     "date": date,
#                     "column": c,
#                     "values": "|".join(vals[:10]),
#                 })

# stock_conflicts = pd.DataFrame(conflicts)

# print("conflict:", len(stock_conflicts))

# if len(stock_conflicts):
#     display(stock_conflicts.head(50))
#     raise RuntimeError(
#         "주가 old/new에서 동일 ticker-date의 값이 다릅니다."
#     )

# stock_daily = (
#     stock_raw_all
#     .sort_values(["ticker", "date"])
#     .drop_duplicates(
#         ["ticker", "date"],
#         keep="first",
#     )
#     .drop(
#         columns=[
#             "_source_period",
#             "_source_file",
#         ]
#     )
#     .reset_index(drop=True)
# )

# print("stock_daily:", stock_daily.shape)
# print("ticker:", stock_daily["ticker"].nunique())
# print("period:", stock_daily["date"].min(), "~", stock_daily["date"].max())

### 10. 기존 주가 inf 처리 규칙 재적용

In [ ]:
# inf_rows = []

# for c in stock_daily.select_dtypes(include=[np.number]).columns:
#     x = pd.to_numeric(stock_daily[c], errors="coerce")
#     n = int(np.isinf(x).sum())

#     if n:
#         inf_rows.append({
#             "column": c,
#             "inf_n": n,
#         })

# stock_inf_before = pd.DataFrame(inf_rows)

# print("inf before:")
# display(stock_inf_before)

# unexpected_inf = []

# if len(stock_inf_before):
#     unexpected_inf = stock_inf_before.loc[
#         ~stock_inf_before["column"].isin(["trading_value_growth"]),
#         "column",
#     ].tolist()

# if unexpected_inf:
#     raise RuntimeError(
#         "예상하지 못한 stock inf 컬럼: "
#         + ", ".join(unexpected_inf)
#     )

# if "trading_value_growth" in stock_daily.columns:
#     x = pd.to_numeric(
#         stock_daily["trading_value_growth"],
#         errors="coerce",
#     )
#     mask = np.isinf(x)

#     print("trading_value_growth inf → NaN:", int(mask.sum()))

#     stock_daily.loc[
#         mask,
#         "trading_value_growth",
#     ] = np.nan

# remaining_stock_inf = 0

# for c in stock_daily.select_dtypes(include=[np.number]).columns:
#     remaining_stock_inf += int(
#         np.isinf(
#             pd.to_numeric(stock_daily[c], errors="coerce")
#         ).sum()
#     )

# assert remaining_stock_inf == 0

# print("✅ stock inf 처리 완료")

### 11. stock_daily QA

In [ ]:
# stock_key_dups = int(
#     stock_daily.duplicated(
#         ["ticker", "date"],
#         keep=False,
#     ).sum()
# )

# stock_row_dups = int(stock_daily.duplicated().sum())
# stock_date_na = int(stock_daily["date"].isna().sum())

# stock_bad_ticker = int(
#     (
#         ~stock_daily["ticker"].str.match(
#             r"^[0-9A-Z]{6}$",
#             na=False,
#         )
#     ).sum()
# )

# stock_qa = pd.DataFrame([{
#     "rows": len(stock_daily),
#     "columns": len(stock_daily.columns),
#     "ticker_n": int(stock_daily["ticker"].nunique()),
#     "start_date": stock_daily["date"].min(),
#     "end_date": stock_daily["date"].max(),
#     "ticker_date_duplicate_rows": stock_key_dups,
#     "duplicate_rows": stock_row_dups,
#     "date_na_rows": stock_date_na,
#     "bad_ticker_format_rows": stock_bad_ticker,
#     "remaining_inf_cells": remaining_stock_inf,
#     "contains_001260": bool(stock_daily["ticker"].eq("001260").any()),
#     "contains_0126Z0": bool(stock_daily["ticker"].eq("0126Z0").any()),
# }])

# display(stock_qa)

# assert stock_daily["date"].min() == START_DATE
# assert stock_daily["date"].max() == END_DATE
# assert stock_key_dups == 0
# assert stock_row_dups == 0
# assert stock_date_na == 0
# assert stock_bad_ticker == 0
# assert remaining_stock_inf == 0
# assert not stock_daily["ticker"].eq("001260").any()
# assert stock_daily["ticker"].eq("0126Z0").any()

# print("✅ stock_daily QA 통과")

## PART B — ATTENTION DAILY 전체기간 재생성

### 12. 전체기간 Attention source 로드

In [ ]:
# ATTENTION_FEATURES = [
#     "search_pct_50",
#     "search_ratio_7_30",
#     "search_slope_7",
#     "search_high_share_7",
#     "news_pct_50",
#     "news_ratio_7_30",
#     "news_slope_7",
#     "news_high_share_7",
#     "search_news_level_gap",
# ]

# ATTENTION_COLUMNS = [
#     "ticker",
#     "ticker_name",
#     "date",
#     *ATTENTION_FEATURES,
# ]

# attention_raw = pd.read_parquet(ATTENTION_SOURCE)

# missing_attention_cols = [
#     c for c in ATTENTION_COLUMNS
#     if c not in attention_raw.columns
# ]

# if missing_attention_cols:
#     raise RuntimeError(
#         "Attention 필수 컬럼 누락: "
#         + ", ".join(missing_attention_cols)
#     )

# attention_raw = attention_raw[
#     ATTENTION_COLUMNS
# ].copy()

# attention_raw["ticker"] = normalize_ticker_series(
#     attention_raw["ticker"]
# )

# attention_raw["date"] = pd.to_datetime(
#     attention_raw["date"],
#     errors="coerce",
# ).dt.normalize()

# attention_raw = attention_raw[
#     attention_raw["date"].between(
#         START_DATE,
#         END_DATE,
#         inclusive="both",
#     )
# ].copy()

# print("raw:", attention_raw.shape)
# print("ticker:", attention_raw["ticker"].nunique())
# print("period:", attention_raw["date"].min(), "~", attention_raw["date"].max())

### 13. 기존 주가↔관심지표 Universe 처리 기준 재적용

In [ ]:
# stock_tickers = set(
#     stock_daily["ticker"]
#     .dropna()
#     .astype(str)
#     .unique()
# )

# attention_tickers = set(
#     attention_raw["ticker"]
#     .dropna()
#     .astype(str)
#     .unique()
# )

# stock_attention_common = sorted(
#     stock_tickers & attention_tickers
# )

# stock_only = sorted(
#     stock_tickers - attention_tickers
# )

# attention_only = sorted(
#     attention_tickers - stock_tickers
# )

# universe_alignment = pd.DataFrame([
#     {
#         "status": "STOCK_AND_ATTENTION",
#         "ticker_n": len(stock_attention_common),
#         "treatment": "stock 유지 / Attention 모델 사용 가능",
#     },
#     {
#         "status": "STOCK_ONLY",
#         "ticker_n": len(stock_only),
#         "treatment": (
#             "stock_daily에는 유지. "
#             "시장지표 계산에 사용 가능. "
#             "Attention 포함 모델에서는 eligibility 제외/검토"
#         ),
#     },
#     {
#         "status": "ATTENTION_ONLY",
#         "ticker_n": len(attention_only),
#         "treatment": "stock key가 없으므로 canonical attention에서 제외 후 QA 기록",
#     },
# ])

# display(universe_alignment)

# print("STOCK ONLY:", stock_only)
# print("ATTENTION ONLY:", attention_only)

# attention_daily = (
#     attention_raw[
#         attention_raw["ticker"].isin(stock_tickers)
#     ]
#     .sort_values(["ticker", "date"])
#     .reset_index(drop=True)
# )

# print("canonical attention:", attention_daily.shape)
# print("ticker:", attention_daily["ticker"].nunique())
# print("period:", attention_daily["date"].min(), "~", attention_daily["date"].max())

### 14. attention_daily QA

In [ ]:
# attention_key_dups = int(
#     attention_daily.duplicated(
#         ["ticker", "date"],
#         keep=False,
#     ).sum()
# )

# attention_row_dups = int(
#     attention_daily.duplicated().sum()
# )

# attention_date_na = int(
#     attention_daily["date"].isna().sum()
# )

# attention_bad_ticker = int(
#     (
#         ~attention_daily["ticker"].str.match(
#             r"^[0-9A-Z]{6}$",
#             na=False,
#         )
#     ).sum()
# )

# attention_inf = 0

# for c in ATTENTION_FEATURES:
#     attention_inf += int(
#         np.isinf(
#             pd.to_numeric(attention_daily[c], errors="coerce")
#         ).sum()
#     )

# attention_qa = pd.DataFrame([{
#     "rows": len(attention_daily),
#     "columns": len(attention_daily.columns),
#     "ticker_n": int(attention_daily["ticker"].nunique()),
#     "start_date": attention_daily["date"].min(),
#     "end_date": attention_daily["date"].max(),
#     "ticker_date_duplicate_rows": attention_key_dups,
#     "duplicate_rows": attention_row_dups,
#     "date_na_rows": attention_date_na,
#     "bad_ticker_format_rows": attention_bad_ticker,
#     "remaining_inf_cells": attention_inf,
#     "stock_only_ticker_n": len(stock_only),
#     "attention_only_ticker_n": len(attention_only),
# }])

# display(attention_qa)

# assert attention_daily["date"].min() == START_DATE
# assert attention_daily["date"].max() == END_DATE
# assert attention_key_dups == 0
# assert attention_row_dups == 0
# assert attention_date_na == 0
# assert attention_bad_ticker == 0
# assert attention_inf == 0

# print("✅ attention_daily QA 통과")

### 15. Attention 결측은 삭제하지 않고 기록

In [ ]:
# attention_missing = pd.DataFrame([
#     {
#         "feature": c,
#         "missing_n": int(attention_daily[c].isna().sum()),
#         "missing_pct": float(attention_daily[c].isna().mean()),
#         "valid_n": int(attention_daily[c].notna().sum()),
#         "first_valid_date": (
#             attention_daily.loc[
#                 attention_daily[c].notna(),
#                 "date",
#             ].min()
#         ),
#         "last_valid_date": (
#             attention_daily.loc[
#                 attention_daily[c].notna(),
#                 "date",
#             ].max()
#         ),
#     }
#     for c in ATTENTION_FEATURES
# ])

# display(attention_missing)

## PART C — SHORT DAILY 전체기간 원천 재구축

### 16. 공매도 raw 컬럼 alias 정의

In [ ]:
# CORE_SHORT_ALIASES = {
#     "ticker": [
#         "ticker",
#         "code",
#     ],
#     "date": [
#         "date",
#         "TRD_DD",
#     ],
#     "short_volume": [
#         "short_volume",
#         "CVSRTSELL_TRDVOL",
#     ],
#     "short_value": [
#         "short_value",
#         "CVSRTSELL_TRDVAL",
#     ],
#     "short_balance_shares": [
#         # 기존 결측 원인이었던 단수형을 가장 먼저 인식
#         "short_balance_share",
#         "short_balance_shares",
#         "STR_CONST_VAL1",
#     ],
#     "short_balance_value": [
#         "short_balance_value",
#         "STR_CONST_VAL2",
#     ],
# }


# def get_columns(path):
#     if path.suffix.lower() == ".parquet":
#         return pq.ParquetFile(path).schema.names

#     return pd.read_csv(path, nrows=0).columns.tolist()


# def find_alias(cols, aliases):
#     lookup = {
#         str(c).strip().lower(): c
#         for c in cols
#     }

#     for alias in aliases:
#         if alias.lower() in lookup:
#             return lookup[alias.lower()]

#     return None

### 17. old/new 공매도 폴더에서 raw layer 자동 선택

In [ ]:
# def choose_short_raw_dir(root):
#     records = []

#     for ext in ("*.csv", "*.parquet"):
#         for p in root.rglob(ext):
#             s = str(p).lower()

#             # 이미 가공된 panel/export는 원천에서 제외
#             if (
#                 "exports" in s
#                 or "panel_daily" in s
#                 or p.name.lower() == "short_daily.parquet"
#             ):
#                 continue

#             try:
#                 cols = get_columns(p)

#                 mapping = {
#                     k: find_alias(cols, aliases)
#                     for k, aliases
#                     in CORE_SHORT_ALIASES.items()
#                 }

#                 core_hit_n = sum(
#                     mapping[k] is not None
#                     for k in [
#                         "short_volume",
#                         "short_value",
#                         "short_balance_shares",
#                         "short_balance_value",
#                     ]
#                 )

#                 if (
#                     mapping["ticker"] is not None
#                     and mapping["date"] is not None
#                     and core_hit_n >= 3
#                 ):
#                     records.append({
#                         "path": str(p),
#                         "parent": str(p.parent),
#                         "core_hit_n": core_hit_n,
#                     })

#             except Exception:
#                 pass

#     candidate = pd.DataFrame(records)

#     if candidate.empty:
#         raise RuntimeError(
#             f"공매도 원천 cache 후보를 찾지 못했습니다: {root}"
#         )

#     parent_summary = (
#         candidate
#         .groupby("parent", as_index=False)
#         .agg(
#             file_n=("path", "size"),
#             full_core_file_n=(
#                 "core_hit_n",
#                 lambda x: int((x == 4).sum()),
#             ),
#         )
#         .sort_values(
#             ["full_core_file_n", "file_n"],
#             ascending=False,
#         )
#     )

#     selected = Path(
#         parent_summary.iloc[0]["parent"]
#     )

#     return selected, parent_summary


# SHORT_OLD_RAW_DIR, short_old_parent_summary = choose_short_raw_dir(
#     SHORT_OLD_ROOT
# )

# SHORT_NEW_RAW_DIR, short_new_parent_summary = choose_short_raw_dir(
#     SHORT_NEW_ROOT
# )

# print("OLD RAW DIR:", SHORT_OLD_RAW_DIR)
# display(short_old_parent_summary.head(10))

# print("NEW RAW DIR:", SHORT_NEW_RAW_DIR)
# display(short_new_parent_summary.head(10))

### 18. 공매도 두 기간 raw 로드 및 표준화

In [ ]:
# CORE_SHORT_VALUES = [
#     "short_volume",
#     "short_value",
#     "short_balance_shares",
#     "short_balance_value",
# ]


# def load_short_raw_dir(raw_dir, period_label):
#     files = []

#     for ext in ("*.csv", "*.parquet"):
#         files.extend(raw_dir.glob(ext))

#     frames = []
#     errors = []

#     for path in sorted(files):
#         try:
#             cols = get_columns(path)

#             mapping = {
#                 k: find_alias(cols, aliases)
#                 for k, aliases
#                 in CORE_SHORT_ALIASES.items()
#             }

#             if (
#                 mapping["ticker"] is None
#                 or mapping["date"] is None
#             ):
#                 continue

#             usecols = [
#                 v for v in mapping.values()
#                 if v is not None
#             ]

#             if path.suffix.lower() == ".parquet":
#                 x = pd.read_parquet(
#                     path,
#                     columns=usecols,
#                 )
#             else:
#                 x = pd.read_csv(
#                     path,
#                     usecols=usecols,
#                     low_memory=False,
#                 )

#             rename = {
#                 source: canonical
#                 for canonical, source
#                 in mapping.items()
#                 if source is not None
#             }

#             x = x.rename(columns=rename)

#             for c in CORE_SHORT_VALUES:
#                 if c not in x.columns:
#                     x[c] = np.nan

#             x = x[
#                 [
#                     "ticker",
#                     "date",
#                     *CORE_SHORT_VALUES,
#                 ]
#             ].copy()

#             x["ticker"] = normalize_ticker_series(
#                 x["ticker"]
#             )

#             x["date"] = pd.to_datetime(
#                 x["date"],
#                 errors="coerce",
#             ).dt.normalize()

#             for c in CORE_SHORT_VALUES:
#                 x[c] = numeric_clean(x[c])

#             x["_source_period"] = period_label
#             x["_source_file"] = path.name

#             frames.append(x)

#         except Exception as e:
#             errors.append({
#                 "path": str(path),
#                 "error": repr(e),
#             })

#     if not frames:
#         raise RuntimeError(
#             f"로드 가능한 공매도 원천파일이 없습니다: {raw_dir}"
#         )

#     return (
#         pd.concat(frames, ignore_index=True),
#         pd.DataFrame(errors),
#     )


# short_old_raw, short_old_errors = load_short_raw_dir(
#     SHORT_OLD_RAW_DIR,
#     "20210611_20260611",
# )

# short_new_raw, short_new_errors = load_short_raw_dir(
#     SHORT_NEW_RAW_DIR,
#     "20260612_20260810",
# )

# print(
#     "OLD:",
#     short_old_raw.shape,
#     short_old_raw["date"].min(),
#     "~",
#     short_old_raw["date"].max(),
# )

# print(
#     "NEW:",
#     short_new_raw.shape,
#     short_new_raw["date"].min(),
#     "~",
#     short_new_raw["date"].max(),
# )

# print(
#     "read errors:",
#     len(short_old_errors),
#     len(short_new_errors),
# )

# if len(short_old_errors):
#     display(short_old_errors.head(30))

# if len(short_new_errors):
#     display(short_new_errors.head(30))

# if len(short_old_errors) or len(short_new_errors):
#     raise RuntimeError(
#         "공매도 raw read error가 있으므로 중단합니다."
#     )

### 19. 공매도 old/new 병합 + key 충돌 검사

In [ ]:
# short_raw_all = pd.concat(
#     [short_old_raw, short_new_raw],
#     ignore_index=True,
# )

# short_raw_all = short_raw_all[
#     short_raw_all["date"].between(
#         START_DATE,
#         END_DATE,
#         inclusive="both",
#     )
# ].copy()

# # stock canonical에 없는 explicit bad ticker는 같이 제거
# short_raw_all = short_raw_all[
#     ~short_raw_all["ticker"].isin(
#         EXPLICIT_STOCK_EXCLUSIONS.keys()
#     )
# ].copy()

# dup = short_raw_all[
#     short_raw_all.duplicated(
#         ["ticker", "date"],
#         keep=False,
#     )
# ].copy()

# short_conflicts = []

# if len(dup):
#     for (ticker, date), g in dup.groupby(
#         ["ticker", "date"],
#         sort=False,
#     ):
#         for c in CORE_SHORT_VALUES:
#             vals = (
#                 g[c]
#                 .dropna()
#                 .unique()
#             )

#             if len(vals) > 1:
#                 short_conflicts.append({
#                     "ticker": ticker,
#                     "date": date,
#                     "column": c,
#                     "values": "|".join(
#                         map(str, vals[:10])
#                     ),
#                 })

# short_conflict_df = pd.DataFrame(
#     short_conflicts
# )

# print("중복 key 포함 rows:", len(dup))
# print("값 충돌:", len(short_conflict_df))

# if len(short_conflict_df):
#     display(short_conflict_df.head(50))
#     raise RuntimeError(
#         "공매도 old/new 동일 key의 원천값이 서로 다릅니다."
#     )

# short_raw_full = (
#     short_raw_all
#     .sort_values(["ticker", "date"])
#     .groupby(
#         ["ticker", "date"],
#         as_index=False,
#     )[CORE_SHORT_VALUES]
#     .first()
# )

# print("raw full:", short_raw_full.shape)
# print("ticker:", short_raw_full["ticker"].nunique())
# print("period:", short_raw_full["date"].min(), "~", short_raw_full["date"].max())
# print("key dups:", short_raw_full.duplicated(["ticker", "date"]).sum())

### 20. 원천 4개 컬럼 유효값 QA

In [ ]:
# short_raw_core_qa = pd.DataFrame([
#     {
#         "column": c,
#         "rows": len(short_raw_full),
#         "non_null_n": int(
#             short_raw_full[c].notna().sum()
#         ),
#         "non_null_pct": float(
#             short_raw_full[c].notna().mean()
#         ),
#         "nonzero_n": int(
#             short_raw_full[c].fillna(0).ne(0).sum()
#         ),
#         "min": (
#             float(short_raw_full[c].min())
#             if short_raw_full[c].notna().any()
#             else np.nan
#         ),
#         "max": (
#             float(short_raw_full[c].max())
#             if short_raw_full[c].notna().any()
#             else np.nan
#         ),
#     }
#     for c in CORE_SHORT_VALUES
# ])

# display(short_raw_core_qa)

# if (short_raw_core_qa["non_null_n"] == 0).any():
#     raise RuntimeError(
#         "공매도 핵심 원천 4개 중 전체 결측 컬럼이 있습니다."
#     )

# print("✅ 공매도 핵심 원천 4개 값 존재")

### 21. stock trading key에 raw short Left Join

In [ ]:
# stock_for_short = stock_daily[
#     [
#         "ticker",
#         "ticker_name",
#         "date",
#         "volume",
#         "trading_value",
#         "listed_shares",
#         "market_cap",
#     ]
# ].copy()

# short_work = stock_for_short.merge(
#     short_raw_full,
#     on=["ticker", "date"],
#     how="left",
#     validate="one_to_one",
#     indicator="_raw_merge",
# )

# print(
#     short_work["_raw_merge"]
#     .value_counts(dropna=False)
# )

# raw_missing_stock_key_n = int(
#     short_work["_raw_merge"]
#     .eq("left_only")
#     .sum()
# )

# print(
#     "stock trading day 중 raw short row 없음:",
#     raw_missing_stock_key_n,
# )

# short_work = short_work.drop(
#     columns="_raw_merge"
# )

### 22. 공매도 제도 상태 flag 재생성

In [ ]:
# # 기존 프로젝트에서 사용한 공매도 전면 금지기간
# SHORT_BAN_START = pd.Timestamp("2023-11-06")
# SHORT_BAN_END = pd.Timestamp("2025-03-30")
# FULL_REOPEN_DATE = pd.Timestamp("2025-03-31")

# short_work["short_ban_period"] = (
#     short_work["date"]
#     .between(
#         SHORT_BAN_START,
#         SHORT_BAN_END,
#         inclusive="both",
#     )
#     .astype("int8")
# )

# short_work["short_sale_available"] = (
#     1 - short_work["short_ban_period"]
# ).astype("int8")

# short_work["post_full_reopen"] = (
#     short_work["date"]
#     .ge(FULL_REOPEN_DATE)
#     .astype("int8")
# )

# print(
#     "ban rows:",
#     int(short_work["short_ban_period"].sum())
# )

### 23. 핵심 비율 재계산

In [ ]:
# short_work[
#     "short_volume_ratio_pct_stock_denominator"
# ] = safe_ratio_pct(
#     short_work["short_volume"],
#     short_work["volume"],
# )

# short_work[
#     "short_value_ratio_pct"
# ] = safe_ratio_pct(
#     short_work["short_value"],
#     short_work["trading_value"],
# )

# short_work[
#     "short_balance_ratio_pct"
# ] = safe_ratio_pct(
#     short_work["short_balance_shares"],
#     short_work["listed_shares"],
# )

# print("✅ ratio 계산")

### 24. 5D / 20D / prior40 공매도 Feature 계산

In [ ]:
# short_work = short_work.sort_values(
#     ["ticker", "date"]
# ).reset_index(drop=True)

# # 현재일 포함 rolling
# short_work[
#     "short_value_ratio_5d_mean"
# ] = (
#     short_work
#     .groupby("ticker")["short_value_ratio_pct"]
#     .rolling(5, min_periods=5)
#     .mean()
#     .reset_index(level=0, drop=True)
# )

# short_work[
#     "short_value_ratio_20d_mean"
# ] = (
#     short_work
#     .groupby("ticker")["short_value_ratio_pct"]
#     .rolling(20, min_periods=20)
#     .mean()
#     .reset_index(level=0, drop=True)
# )

# # 현재일 제외 직전 40 거래일
# short_work["_short_value_lag1"] = (
#     short_work
#     .groupby("ticker")["short_value"]
#     .shift(1)
# )

# short_work[
#     "prior40_short_value_avg"
# ] = (
#     short_work
#     .groupby("ticker")["_short_value_lag1"]
#     .rolling(40, min_periods=1)
#     .mean()
#     .reset_index(level=0, drop=True)
# )

# short_work[
#     "prior40_trade_count"
# ] = (
#     short_work["_short_value_lag1"]
#     .notna()
#     .astype(int)
#     .groupby(short_work["ticker"])
#     .rolling(40, min_periods=1)
#     .sum()
#     .reset_index(level=0, drop=True)
# )

# print("✅ rolling short features 계산")

### 25. `999999.99` sentinel 없이 증가배수 재계산

In [ ]:
# cur = pd.to_numeric(
#     short_work["short_value"],
#     errors="coerce",
# )

# avg40 = pd.to_numeric(
#     short_work["prior40_short_value_avg"],
#     errors="coerce",
# )

# short_work[
#     "short_value_increase_multiple_40d"
# ] = np.nan

# normal_mask = (
#     cur.notna()
#     & avg40.notna()
#     & avg40.gt(0)
# )

# short_work.loc[
#     normal_mask,
#     "short_value_increase_multiple_40d",
# ] = (
#     cur.loc[normal_mask]
#     / avg40.loc[normal_mask]
# )

# from_zero = (
#     cur.gt(0)
#     & avg40.eq(0)
# )

# short_work[
#     "short_value_increase_from_zero_flag"
# ] = (
#     from_zero
#     .fillna(False)
#     .astype("int8")
# )

# print(
#     "0 → 양수 신규발생:",
#     int(
#         short_work[
#             "short_value_increase_from_zero_flag"
#         ].sum()
#     ),
# )

### 26. 잔고 변화 + T-2 공개시차 Feature 재계산

In [ ]:
# short_work[
#     "short_balance_change_5d_pp"
# ] = (
#     short_work
#     .groupby("ticker")["short_balance_ratio_pct"]
#     .diff(5)
# )

# short_work[
#     "short_balance_change_20d_pp"
# ] = (
#     short_work
#     .groupby("ticker")["short_balance_ratio_pct"]
#     .diff(20)
# )

# # 사건일 T에서 보수적으로 T-2 거래일 잔고만 사용
# short_work[
#     "short_balance_public_t2_pct"
# ] = (
#     short_work
#     .groupby("ticker")["short_balance_ratio_pct"]
#     .shift(2)
# )

# short_work[
#     "short_balance_public_change_5d_pp"
# ] = (
#     short_work
#     .groupby("ticker")["short_balance_public_t2_pct"]
#     .diff(5)
# )

# short_work[
#     "short_balance_reported_flag"
# ] = (
#     short_work["short_balance_shares"]
#     .notna()
#     .astype("int8")
# )

# short_work[
#     "short_balance_missing_flag"
# ] = (
#     short_work["short_balance_shares"]
#     .isna()
#     .astype("int8")
# )

# print("✅ balance features 계산")

### 27. canonical short_daily 선택

In [ ]:
# SHORT_FEATURES = [
#     "short_volume",
#     "short_value",
#     "short_balance_shares",
#     "short_balance_value",

#     "short_volume_ratio_pct_stock_denominator",
#     "short_value_ratio_pct",
#     "short_balance_ratio_pct",

#     "short_value_ratio_5d_mean",
#     "short_value_ratio_20d_mean",

#     "prior40_short_value_avg",
#     "prior40_trade_count",
#     "short_value_increase_multiple_40d",
#     "short_value_increase_from_zero_flag",

#     "short_balance_change_5d_pp",
#     "short_balance_change_20d_pp",

#     "short_balance_public_t2_pct",
#     "short_balance_public_change_5d_pp",

#     "short_balance_reported_flag",
#     "short_balance_missing_flag",

#     "short_ban_period",
#     "short_sale_available",
#     "post_full_reopen",
# ]

# short_daily = short_work[
#     [
#         "ticker",
#         "ticker_name",
#         "date",
#         *SHORT_FEATURES,
#     ]
# ].copy()

# print("short_daily:", short_daily.shape)
# print("ticker:", short_daily["ticker"].nunique())
# print("period:", short_daily["date"].min(), "~", short_daily["date"].max())

### 28. 공매도 전체 NaN / inf / leakage QA

In [ ]:
# SHORT_KEYS = {
#     "ticker",
#     "ticker_name",
#     "date",
# }

# all_null_short_features = [
#     c for c in short_daily.columns
#     if c not in SHORT_KEYS
#     and short_daily[c].isna().all()
# ]

# short_inf_rows = []

# for c in short_daily.select_dtypes(include=[np.number]).columns:
#     n = int(
#         np.isinf(
#             pd.to_numeric(short_daily[c], errors="coerce")
#         ).sum()
#     )

#     if n:
#         short_inf_rows.append({
#             "column": c,
#             "inf_n": n,
#         })

# short_inf_df = pd.DataFrame(
#     short_inf_rows
# )

# leakage_like_cols = [
#     c for c in short_daily.columns
#     if any(
#         kw in c.lower()
#         for kw in [
#             "fwd_",
#             "future",
#             "forward",
#             "target",
#             "label",
#         ]
#     )
# ]

# print("전체 NaN short features:", all_null_short_features)
# print("leakage-like:", leakage_like_cols)
# display(short_inf_df)

# if all_null_short_features:
#     raise RuntimeError(
#         "공매도 재구축 후에도 전체 NaN Feature 존재: "
#         + ", ".join(all_null_short_features)
#     )

# if leakage_like_cols:
#     raise RuntimeError(
#         "미래정보 의심 short 컬럼 존재: "
#         + ", ".join(leakage_like_cols)
#     )

# if len(short_inf_df):
#     raise RuntimeError(
#         "short_daily에 inf가 존재합니다."
#     )

# print("✅ short feature QA 통과")

### 29. 공매도 금지기간 구조적 결측 확인

In [ ]:
# ban = short_daily["short_ban_period"].eq(1)

# ban_missing_qa = pd.DataFrame([
#     {
#         "feature": c,
#         "ban_rows": int(ban.sum()),
#         "ban_missing_n": int(
#             short_daily.loc[ban, c]
#             .isna()
#             .sum()
#         ),
#         "ban_missing_pct": float(
#             short_daily.loc[ban, c]
#             .isna()
#             .mean()
#         ) if ban.any() else np.nan,
#         "nonban_missing_n": int(
#             short_daily.loc[~ban, c]
#             .isna()
#             .sum()
#         ),
#         "nonban_missing_pct": float(
#             short_daily.loc[~ban, c]
#             .isna()
#             .mean()
#         ) if (~ban).any() else np.nan,
#     }
#     for c in [
#         "short_volume",
#         "short_value",
#         "short_balance_shares",
#         "short_balance_value",
#         "short_volume_ratio_pct_stock_denominator",
#         "short_value_ratio_pct",
#         "short_balance_ratio_pct",
#     ]
# ])

# display(ban_missing_qa)

# print(
#     "※ 구조적 결측을 0으로 변환하지 않습니다."
# )

### 30. short_daily 최종 key QA

In [ ]:
# short_key_dups = int(
#     short_daily.duplicated(
#         ["ticker", "date"],
#         keep=False,
#     ).sum()
# )

# short_row_dups = int(
#     short_daily.duplicated().sum()
# )

# short_bad_ticker = int(
#     (
#         ~short_daily["ticker"].str.match(
#             r"^[0-9A-Z]{6}$",
#             na=False,
#         )
#     ).sum()
# )

# short_qa = pd.DataFrame([{
#     "rows": len(short_daily),
#     "columns": len(short_daily.columns),
#     "ticker_n": int(short_daily["ticker"].nunique()),
#     "start_date": short_daily["date"].min(),
#     "end_date": short_daily["date"].max(),
#     "ticker_date_duplicate_rows": short_key_dups,
#     "duplicate_rows": short_row_dups,
#     "all_null_feature_n": len(all_null_short_features),
#     "raw_missing_stock_key_n": raw_missing_stock_key_n,
#     "contains_001260": bool(short_daily["ticker"].eq("001260").any()),
#     "contains_0126Z0": bool(short_daily["ticker"].eq("0126Z0").any()),
# }])

# display(short_qa)

# assert len(short_daily) == len(stock_daily)
# assert short_daily["ticker"].nunique() == stock_daily["ticker"].nunique()
# assert short_daily["date"].min() == START_DATE
# assert short_daily["date"].max() == END_DATE
# assert short_key_dups == 0
# assert short_row_dups == 0
# assert len(all_null_short_features) == 0
# assert not short_daily["ticker"].eq("001260").any()
# assert short_daily["ticker"].eq("0126Z0").any()

# print("✅ short_daily QA 통과")

## PART D — 세 Canonical Dataset 정합성 QA

### 31. 세 데이터 전체기간 요약

In [ ]:
# final_summary = pd.DataFrame([
#     {
#         "dataset": "stock_daily",
#         "rows": len(stock_daily),
#         "columns": len(stock_daily.columns),
#         "ticker_n": stock_daily["ticker"].nunique(),
#         "start_date": stock_daily["date"].min(),
#         "end_date": stock_daily["date"].max(),
#         "time_grain": "trading_day",
#     },
#     {
#         "dataset": "attention_daily",
#         "rows": len(attention_daily),
#         "columns": len(attention_daily.columns),
#         "ticker_n": attention_daily["ticker"].nunique(),
#         "start_date": attention_daily["date"].min(),
#         "end_date": attention_daily["date"].max(),
#         "time_grain": "calendar_day",
#     },
#     {
#         "dataset": "short_daily",
#         "rows": len(short_daily),
#         "columns": len(short_daily.columns),
#         "ticker_n": short_daily["ticker"].nunique(),
#         "start_date": short_daily["date"].min(),
#         "end_date": short_daily["date"].max(),
#         "time_grain": "trading_day",
#     },
# ])

# display(final_summary)

### 32. 2026-06-11 → 2026-06-12 경계 QA

In [ ]:
# boundary_date = pd.Timestamp("2026-06-11")

# boundary_qa = pd.DataFrame([
#     {
#         "dataset": name,
#         "rows_2026_06_11": int(
#             df["date"].eq(
#                 pd.Timestamp("2026-06-11")
#             ).sum()
#         ),
#         "rows_2026_06_12": int(
#             df["date"].eq(
#                 pd.Timestamp("2026-06-12")
#             ).sum()
#         ),
#         "max_on_or_before_0611": (
#             df.loc[
#                 df["date"] <= boundary_date,
#                 "date",
#             ].max()
#         ),
#         "min_after_0611": (
#             df.loc[
#                 df["date"] > boundary_date,
#                 "date",
#             ].min()
#         ),
#     }
#     for name, df in [
#         ("stock", stock_daily),
#         ("attention", attention_daily),
#         ("short", short_daily),
#     ]
# ])

# display(boundary_qa)

### 33. stock ↔ short key 완전 일치 검증

In [ ]:
# stock_key = pd.MultiIndex.from_frame(
#     stock_daily[
#         ["ticker", "date"]
#     ]
# )

# short_key = pd.MultiIndex.from_frame(
#     short_daily[
#         ["ticker", "date"]
#     ]
# )

# stock_only_key = stock_key.difference(
#     short_key
# )

# short_only_key = short_key.difference(
#     stock_key
# )

# stock_short_key_qa = pd.DataFrame([{
#     "stock_key_n": len(stock_key),
#     "short_key_n": len(short_key),
#     "stock_only_key_n": len(stock_only_key),
#     "short_only_key_n": len(short_only_key),
# }])

# display(stock_short_key_qa)

# assert len(stock_only_key) == 0
# assert len(short_only_key) == 0

# print("✅ stock-short key 완전 일치")

## PART E — 로컬 canonical 저장 + Metadata

### 34. 세 Parquet 저장

In [ ]:
# stock_daily.to_parquet(
#     STOCK_OUT,
#     index=False,
#     engine="pyarrow",
#     compression="zstd",
# )

# attention_daily.to_parquet(
#     ATTENTION_OUT,
#     index=False,
#     engine="pyarrow",
#     compression="zstd",
# )

# short_daily.to_parquet(
#     SHORT_OUT,
#     index=False,
#     engine="pyarrow",
#     compression="zstd",
# )

# hashes = {
#     "stock_daily": sha256(STOCK_OUT),
#     "attention_daily": sha256(ATTENTION_OUT),
#     "short_daily": sha256(SHORT_OUT),
# }

# print("STOCK:", STOCK_OUT, hashes["stock_daily"])
# print("ATTENTION:", ATTENTION_OUT, hashes["attention_daily"])
# print("SHORT:", SHORT_OUT, hashes["short_daily"])

### 35. QA 산출물 저장

In [ ]:
# stock_universe_source_compare.to_csv(
#     OUTPUT_DIR / "stock_source_universe_compare.csv",
#     index=False,
#     encoding="utf-8-sig",
# )

# universe_alignment.to_csv(
#     OUTPUT_DIR / "stock_attention_universe_alignment.csv",
#     index=False,
#     encoding="utf-8-sig",
# )

# pd.DataFrame({
#     "ticker": stock_only,
#     "status": "STOCK_ONLY",
# }).to_csv(
#     OUTPUT_DIR / "stock_only_attention_missing_tickers.csv",
#     index=False,
#     encoding="utf-8-sig",
# )

# pd.DataFrame({
#     "ticker": attention_only,
#     "status": "ATTENTION_ONLY",
# }).to_csv(
#     OUTPUT_DIR / "attention_only_tickers.csv",
#     index=False,
#     encoding="utf-8-sig",
# )

# stock_qa.to_csv(
#     OUTPUT_DIR / "stock_daily_qa.csv",
#     index=False,
#     encoding="utf-8-sig",
# )

# attention_qa.to_csv(
#     OUTPUT_DIR / "attention_daily_qa.csv",
#     index=False,
#     encoding="utf-8-sig",
# )

# attention_missing.to_csv(
#     OUTPUT_DIR / "attention_missing_summary.csv",
#     index=False,
#     encoding="utf-8-sig",
# )

# short_raw_core_qa.to_csv(
#     OUTPUT_DIR / "short_raw_core_qa.csv",
#     index=False,
#     encoding="utf-8-sig",
# )

# ban_missing_qa.to_csv(
#     OUTPUT_DIR / "short_ban_missing_qa.csv",
#     index=False,
#     encoding="utf-8-sig",
# )

# short_qa.to_csv(
#     OUTPUT_DIR / "short_daily_qa.csv",
#     index=False,
#     encoding="utf-8-sig",
# )

# final_summary.to_csv(
#     OUTPUT_DIR / "canonical_fullrange_summary.csv",
#     index=False,
#     encoding="utf-8-sig",
# )

# boundary_qa.to_csv(
#     OUTPUT_DIR / "period_boundary_qa.csv",
#     index=False,
#     encoding="utf-8-sig",
# )

# stock_short_key_qa.to_csv(
#     OUTPUT_DIR / "stock_short_key_qa.csv",
#     index=False,
#     encoding="utf-8-sig",
# )

# print("QA 저장:", OUTPUT_DIR)

### 36. Metadata 생성

In [ ]:
# metadata_dir = OUTPUT_DIR / "metadata"
# metadata_dir.mkdir(parents=True, exist_ok=True)

# stock_metadata = {
#     "artifact": "stock_daily.parquet",
#     "available_start": str(START_DATE.date()),
#     "available_end": str(END_DATE.date()),
#     "time_grain": "trading_day",
#     "rows": int(len(stock_daily)),
#     "columns": int(len(stock_daily.columns)),
#     "ticker_n": int(stock_daily["ticker"].nunique()),
#     "sources": [
#         str(STOCK_OLD_DIR),
#         str(STOCK_NEW_DIR),
#     ],
#     "explicit_exclusions": EXPLICIT_STOCK_EXCLUSIONS,
#     "stock_only_attention_missing_tickers": stock_only,
#     "parquet_sha256": hashes["stock_daily"],
# }

# attention_metadata = {
#     "artifact": "attention_daily.parquet",
#     "available_start": str(START_DATE.date()),
#     "available_end": str(END_DATE.date()),
#     "time_grain": "calendar_day",
#     "rows": int(len(attention_daily)),
#     "columns": int(len(attention_daily.columns)),
#     "ticker_n": int(attention_daily["ticker"].nunique()),
#     "source": str(ATTENTION_SOURCE),
#     "features": ATTENTION_FEATURES,
#     "stock_only_tickers_not_in_attention": stock_only,
#     "attention_only_tickers_removed_from_canonical": attention_only,
#     "missing_policy": (
#         "rolling/source structural missing preserved; "
#         "model preprocessing must be fold-safe"
#     ),
#     "parquet_sha256": hashes["attention_daily"],
# }

# short_metadata = {
#     "artifact": "short_daily.parquet",
#     "available_start": str(START_DATE.date()),
#     "available_end": str(END_DATE.date()),
#     "time_grain": "trading_day",
#     "rows": int(len(short_daily)),
#     "columns": int(len(short_daily.columns)),
#     "ticker_n": int(short_daily["ticker"].nunique()),
#     "sources": [
#         str(SHORT_OLD_RAW_DIR),
#         str(SHORT_NEW_RAW_DIR),
#     ],
#     "column_normalization": {
#         "short_balance_share": "short_balance_shares",
#     },
#     "ban_period": {
#         "start": str(SHORT_BAN_START.date()),
#         "end": str(SHORT_BAN_END.date()),
#         "missing_policy": "preserve NaN; do not fill with zero",
#     },
#     "balance_public_lag_trading_days": 2,
#     "sentinel_policy": (
#         "999999.99 not used; zero-to-positive transition "
#         "stored in short_value_increase_from_zero_flag"
#     ),
#     "raw_missing_stock_key_n": raw_missing_stock_key_n,
#     "parquet_sha256": hashes["short_daily"],
# }

# for name, meta in [
#     ("stock_daily_metadata.json", stock_metadata),
#     ("attention_daily_metadata.json", attention_metadata),
#     ("short_daily_metadata.json", short_metadata),
# ]:
#     path = metadata_dir / name

#     with open(path, "w", encoding="utf-8") as f:
#         json.dump(
#             meta,
#             f,
#             ensure_ascii=False,
#             indent=2,
#             default=str,
#         )

#     print(path)

## PART F — GCS Canonical 교체

### 37. Google Cloud 인증

In [ ]:
# from google.colab import auth
# auth.authenticate_user()

# !gcloud config set project {PROJECT_ID}

# print("✅ 인증 완료")

### 38. 업로드 직전 최종 상태

In [ ]:
# print("=" * 100)
# print("FinDA FULL-RANGE CANONICAL")
# print("=" * 100)

# display(final_summary)

# print("\nstock-only Attention missing ticker:", stock_only)
# print("attention-only ticker:", attention_only)
# print("short all-null features:", all_null_short_features)

# assert stock_daily["date"].max() == END_DATE
# assert attention_daily["date"].max() == END_DATE
# assert short_daily["date"].max() == END_DATE

# assert stock_key_dups == 0
# assert attention_key_dups == 0
# assert short_key_dups == 0

# assert len(all_null_short_features) == 0
# assert len(stock_only_key) == 0
# assert len(short_only_key) == 0

# print("✅ GCS 교체 조건 모두 통과")

### 39. 세 canonical Parquet + metadata GCS 업로드

In [ ]:
# GCS_STOCK = f"gs://{BUCKET}/curated/stock_daily.parquet"
# GCS_ATTENTION = f"gs://{BUCKET}/curated/attention_daily.parquet"
# GCS_SHORT = f"gs://{BUCKET}/curated/short_daily.parquet"

# !gcloud storage cp "{STOCK_OUT}" "{GCS_STOCK}"
# !gcloud storage cp "{ATTENTION_OUT}" "{GCS_ATTENTION}"
# !gcloud storage cp "{SHORT_OUT}" "{GCS_SHORT}"

# !gcloud storage cp "{metadata_dir / 'stock_daily_metadata.json'}" "gs://{BUCKET}/metadata/stock_daily_metadata.json"
# !gcloud storage cp "{metadata_dir / 'attention_daily_metadata.json'}" "gs://{BUCKET}/metadata/attention_daily_metadata.json"
# !gcloud storage cp "{metadata_dir / 'short_daily_metadata.json'}" "gs://{BUCKET}/metadata/short_daily_metadata.json"

# print("✅ GCS canonical 업로드 완료")

### 40. GCS 재다운로드 + SHA256 최종 검증

In [ ]:
# verify_dir = Path("/content/finda_gcs_verify")
# verify_dir.mkdir(parents=True, exist_ok=True)

# verify_paths = {
#     "stock_daily": verify_dir / "stock_daily.parquet",
#     "attention_daily": verify_dir / "attention_daily.parquet",
#     "short_daily": verify_dir / "short_daily.parquet",
# }

# !gcloud storage cp "{GCS_STOCK}" "{verify_paths['stock_daily']}"
# !gcloud storage cp "{GCS_ATTENTION}" "{verify_paths['attention_daily']}"
# !gcloud storage cp "{GCS_SHORT}" "{verify_paths['short_daily']}"

# verify_hashes = {
#     k: sha256(v)
#     for k, v in verify_paths.items()
# }

# checksum_qa = pd.DataFrame([
#     {
#         "dataset": k,
#         "local_sha256": hashes[k],
#         "gcs_sha256": verify_hashes[k],
#         "sha256_match": hashes[k] == verify_hashes[k],
#     }
#     for k in hashes
# ])

# display(checksum_qa)

# assert checksum_qa["sha256_match"].all()

# print("✅ 세 파일 checksum 모두 일치")

### 41. GCS 데이터 자체 최종 QA

In [ ]:
# gcs_stock = pd.read_parquet(
#     verify_paths["stock_daily"]
# )

# gcs_attention = pd.read_parquet(
#     verify_paths["attention_daily"]
# )

# gcs_short = pd.read_parquet(
#     verify_paths["short_daily"]
# )

# for df in [gcs_stock, gcs_attention, gcs_short]:
#     df["date"] = pd.to_datetime(
#         df["date"],
#         errors="coerce",
#     )

# gcs_final_qa = pd.DataFrame([
#     {
#         "dataset": "stock_daily",
#         "rows": len(gcs_stock),
#         "columns": len(gcs_stock.columns),
#         "ticker_n": gcs_stock["ticker"].nunique(),
#         "start_date": gcs_stock["date"].min(),
#         "end_date": gcs_stock["date"].max(),
#         "key_duplicate_rows": int(
#             gcs_stock.duplicated(
#                 ["ticker", "date"],
#                 keep=False,
#             ).sum()
#         ),
#     },
#     {
#         "dataset": "attention_daily",
#         "rows": len(gcs_attention),
#         "columns": len(gcs_attention.columns),
#         "ticker_n": gcs_attention["ticker"].nunique(),
#         "start_date": gcs_attention["date"].min(),
#         "end_date": gcs_attention["date"].max(),
#         "key_duplicate_rows": int(
#             gcs_attention.duplicated(
#                 ["ticker", "date"],
#                 keep=False,
#             ).sum()
#         ),
#     },
#     {
#         "dataset": "short_daily",
#         "rows": len(gcs_short),
#         "columns": len(gcs_short.columns),
#         "ticker_n": gcs_short["ticker"].nunique(),
#         "start_date": gcs_short["date"].min(),
#         "end_date": gcs_short["date"].max(),
#         "key_duplicate_rows": int(
#             gcs_short.duplicated(
#                 ["ticker", "date"],
#                 keep=False,
#             ).sum()
#         ),
#     },
# ])

# display(gcs_final_qa)

# assert gcs_final_qa["end_date"].eq(END_DATE).all()
# assert gcs_final_qa["key_duplicate_rows"].eq(0).all()

# print("✅ STEP 2 전체기간 canonical 재생성 완료")

# STAGE 3: GCS curated에서 5개 canonical 파일 불러오기

**태그:** 실행됨 — 여기부터 실제 실행

**원본 노트북:** `(신규 작성 — 원본 노트북 없음)`

> 여기서부터 실제로 실행되는 코드입니다. 위 STAGE 1·2가 만드는 결과물과 그 외 이미 GCS에 적재된 short_daily/market_daily/membership_daily를 포함해 curated 5개 파일을 불러오고 기본 스키마/기간/종목 수를 확인합니다.

In [ ]:
# 로그인 없이 공개 읽기전용 버킷(gs://finda_project, allUsers:objectViewer)에서
# curated 6개 파일을 직접 받아옵니다. 이후 STAGE 11 이후에서도 그대로 쓸 수 있도록
# WORKSPACE_ROOT/curated/ 에 저장합니다 (STAGE 11 이후가 기대하는 경로와 동일).
import pandas as pd

CURATED_DIR = WORKSPACE_ROOT / "curated"
CURATED_DIR.mkdir(parents=True, exist_ok=True)

# short_daily / market_daily / membership_daily 세 개는 뒤쪽 STEP 3(STAGE 11)·
# STEP 3 FIX(STAGE 12)가 값을 새로 계산해서 이 같은 로컬 경로에 덮어씁니다. 그래서
# market_daily / membership_daily는 이후 STAGE들이 이 시점의 값을 그대로 재사용하면
# 안 되고, 그 STAGE가 실행되는 시점의 최신(로컬) 버전을 다시 읽어야 합니다.
# stock_daily / attention_raw_daily / attention_daily / short_daily는 이 노트북
# 안에서 값이 바뀌지 않으므로 안전하게 재사용합니다.
CURATED_FILES = [
    "stock_daily.parquet",
    "attention_raw_daily.parquet",
    "attention_daily.parquet",
    "short_daily.parquet",
    "market_daily.parquet",
    "membership_daily.parquet",
]

for fname in CURATED_FILES:
    dst = CURATED_DIR / fname
    gcs_public_download(f"curated/{fname}", dst)
    print(f"다운로드 완료(공개 URL, 로그인 불필요): curated/{fname} -> {dst}")

stock_daily = pd.read_parquet(CURATED_DIR / "stock_daily.parquet")
attention_raw_daily = pd.read_parquet(CURATED_DIR / "attention_raw_daily.parquet")
attention_daily = pd.read_parquet(CURATED_DIR / "attention_daily.parquet")
short_daily = pd.read_parquet(CURATED_DIR / "short_daily.parquet")
market_daily = pd.read_parquet(CURATED_DIR / "market_daily.parquet")
membership_daily = pd.read_parquet(CURATED_DIR / "membership_daily.parquet")

print()
print("=" * 80)
print("curated 데이터 요약")
print("=" * 80)
for name, df in [
    ("stock_daily", stock_daily),
    ("attention_raw_daily", attention_raw_daily),
    ("attention_daily", attention_daily),
    ("short_daily", short_daily),
    ("market_daily", market_daily),
    ("membership_daily", membership_daily),
]:
    print(f"\n[{name}] shape={df.shape}")
    print("columns:", list(df.columns))
    if "date" in df.columns:
        dates = pd.to_datetime(df["date"], errors="coerce")
        print("date range:", dates.min(), "~", dates.max())
    if "ticker" in df.columns:
        print("unique tickers:", df["ticker"].nunique())

# STAGE 4: [가설검증] 검색 관심 선행 신호 분석 — 사전 검색관심도 5분위별 사후 경로

**태그:** 실행됨

**원본 노트북:** `검색_관심_선행_신호_분석_GCS_실제KOSPI200_최종.ipynb`

## 검색 관심 선행 신호 분석 — GCS 원자료·급등 기준

- 주가 고정기간: 2023-12-15 ~ 2026-06-11
- 급등일: 최근 20거래일 수익률 ≥ 20% **AND** 당일 거래량 ≥ 직전 20거래일 평균의 1.5배
- 연속 충족 시 최초 진입일, 동일 종목 20거래일 cooldown
- 시장지수: GCS `market_daily.parquet`의 `market_proxy_index`
- 검색 관심도: GCS `attention_raw_daily.parquet`의 `search_index`
- 사건 전 t-7~t-1 합계와 t-83~t-28의 8개 기준 주를 비교
- 사건 이후 1~120거래일 시장조정 누적수익률

GCS 실제 원자료 기간은 실행 시 자동 출력하며, 원자료가 없는 날짜를 0으로 채우지 않습니다.

In [ ]:
# 0. GCS 인증 및 고정 표본 로드
!pip -q install -U finance-datareader pykrx yfinance
import subprocess, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import FinanceDataReader as fdr
from pykrx import stock as krx_stock

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
OUTPUT_DIR=Path("/content/gcs_outputs/search_leading_signal_criterion3")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
# (재사용) STAGE 3에서 이미 받아온 stock_daily / attention_raw_daily를 그대로 사용합니다.
# 이 두 파일은 노트북 내 다른 STAGE가 값을 바꾸지 않으므로 재다운로드가 필요 없습니다.
stock=stock_daily.copy()
attention_raw=attention_raw_daily.copy()
for d in [stock,attention_raw]:
    d["ticker"]=d["ticker"].astype(str).str.strip().str.upper().str.zfill(6)
    d["date"]=pd.to_datetime(d["date"],errors="coerce").dt.tz_localize(None).dt.normalize()
start,end=pd.Timestamp("2023-12-15"),pd.Timestamp("2026-06-11")
ANALYSIS_START,ANALYSIS_END=start,end
stock=stock.loc[stock.date.between(start,end)].dropna(subset=["ticker","date"]).sort_values(["ticker","date"]).drop_duplicates(["ticker","date"],keep="last").reset_index(drop=True)
for c in ["close","volume"]: stock[c]=pd.to_numeric(stock[c],errors="coerce")
required_raw={"ticker","date","search_index"}
if missing:=required_raw-set(attention_raw.columns): raise ValueError(f"attention_raw_daily 필수 열 누락: {sorted(missing)}")
attention_raw["search_index"]=pd.to_numeric(attention_raw["search_index"],errors="coerce")
if "search_raw_available" in attention_raw.columns:
    available=attention_raw["search_raw_available"]
    if str(available.dtype)!="boolean" and available.dtype!=bool:
        available=available.astype(str).str.strip().str.upper().map({"TRUE":True,"FALSE":False,"1":True,"0":False})
    attention_raw.loc[available.eq(False),"search_index"]=np.nan
attention_raw=attention_raw.dropna(subset=["ticker","date"]).sort_values(["ticker","date"]).drop_duplicates(["ticker","date"],keep="last").reset_index(drop=True)
print("GCS 검색지수 실제 기간:",attention_raw.loc[attention_raw.search_index.notna(),"date"].min(),"~",attention_raw.loc[attention_raw.search_index.notna(),"date"].max())
print("표본 확인:",stock.shape,stock.ticker.nunique(),stock.date.min(),"~",stock.date.max())

In [ ]:
# 0-1. Colab 한글 폰트 설치 및 적용
import glob
import os
import subprocess
import matplotlib.font_manager as fm

subprocess.run(
    ["apt-get", "install", "-y", "-qq", "fonts-nanum"],
    check=True,
)
subprocess.run(["fc-cache", "-fv"], stdout=subprocess.DEVNULL, check=True)

font_candidates = glob.glob("/usr/share/fonts/truetype/nanum/*.ttf")
if not font_candidates:
    raise FileNotFoundError("설치된 나눔 폰트 파일을 찾을 수 없습니다.")

for font_file in font_candidates:
    fm.fontManager.addfont(font_file)

preferred = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
font_path = preferred if os.path.exists(preferred) else font_candidates[0]
korean_font = fm.FontProperties(fname=font_path).get_name()

# sns.set_theme 실행 뒤에 적용해야 폰트 설정이 유지됨
plt.rcParams.update({
    "font.family": korean_font,
    "font.sans-serif": [korean_font],
    "axes.unicode_minus": False,
})

print("한글 폰트 적용 완료:", korean_font)
print("폰트 경로:", font_path)

In [ ]:
# 1. KOSPI200과 급등 기준 사건 생성
stock["stock_return"]=stock.groupby("ticker")["close"].pct_change(fill_method=None)
stock["valid_price_return"]=stock.stock_return.notna()&stock.stock_return.abs().le(.35)&stock.close.gt(0)&stock.volume.gt(0)
# ============================================================
# 4. 실제 KOSPI200 지수 — PyKRX 1028, 실패 시 Yahoo ^KS200
# market_proxy_index는 사용하지 않음
# ============================================================
def load_actual_kospi200(start_date, end_date):
    errors = []
    try:
        raw = fdr.DataReader(
            "KS200", start_date.strftime("%Y-%m-%d"), end_date.strftime("%Y-%m-%d")
        )
        if raw is not None and not raw.empty and "Close" in raw.columns:
            result = raw[["Close"]].reset_index().rename(columns={"Date":"date", "Close":"market_close"})
            if "date" not in result.columns:
                result = result.rename(columns={result.columns[0]: "date"})
            return result[["date", "market_close"]], "FinanceDataReader KOSPI200(KS200)"
        errors.append("FinanceDataReader 결과가 비어 있거나 Close 열이 없음")
    except Exception as exc:
        errors.append(f"FinanceDataReader: {type(exc).__name__}: {exc}")

    try:
        raw = krx_stock.get_index_ohlcv_by_date(
            start_date.strftime("%Y%m%d"), end_date.strftime("%Y%m%d"), "1028"
        )
        if raw is not None and not raw.empty:
            result = raw.reset_index()
            result = result.rename(columns={result.columns[0]: "date", "종가": "market_close"})
            if "market_close" in result.columns:
                return result[["date", "market_close"]], "PyKRX KOSPI200(1028)"
        errors.append("PyKRX 결과가 비어 있거나 종가 열이 없음")
    except Exception as exc:
        errors.append(f"PyKRX: {type(exc).__name__}: {exc}")

    try:
        raw = yf.download(
            "^KS200", start=start_date.strftime("%Y-%m-%d"),
            end=(end_date + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
            auto_adjust=False, progress=False, threads=False,
        )
        if isinstance(raw.columns, pd.MultiIndex):
            raw.columns = raw.columns.get_level_values(0)
        if raw is not None and not raw.empty:
            price_col = "Adj Close" if "Adj Close" in raw.columns else "Close"
            result = raw[[price_col]].reset_index().rename(columns={"Date":"date", price_col:"market_close"})
            return result, "Yahoo Finance KOSPI200(^KS200)"
        errors.append("Yahoo Finance 결과가 비어 있음")
    except Exception as exc:
        errors.append(f"Yahoo: {type(exc).__name__}: {exc}")

    raise RuntimeError("실제 KOSPI200 다운로드 실패. 시장 프록시로 대체하지 않습니다.\n" + "\n".join(errors))

market, market_source = load_actual_kospi200(ANALYSIS_START, ANALYSIS_END)
market["date"] = pd.to_datetime(market["date"], errors="coerce").dt.tz_localize(None).dt.normalize()
market["market_close"] = pd.to_numeric(market["market_close"], errors="coerce")
market = market.dropna(subset=["date", "market_close"])
market = market.loc[market["date"].between(ANALYSIS_START, ANALYSIS_END)]
market = market.sort_values("date").drop_duplicates("date", keep="last").reset_index(drop=True)
if market.empty:
    raise ValueError("고정 분석기간의 실제 KOSPI200 자료가 비어 있습니다.")
market["market_return"] = market["market_close"].pct_change(fill_method=None)
market["market_return_5d"] = market["market_close"].pct_change(periods=5, fill_method=None)
market["market_volatility_5d"] = market["market_return"].rolling(5, min_periods=5).std()
market["market_volatility_60d"] = market["market_return"].rolling(60, min_periods=60).std()
print("시장자료 출처:", market_source)
print("실제 KOSPI200:", market.shape, market["date"].min(), "~", market["date"].max())

panel=stock.merge(market[["date","market_close","market_return"]],on="date",how="left",validate="many_to_one")
panel["return_20d"]=panel.groupby("ticker")["close"].pct_change(20,fill_method=None)
panel["return_20d_pct"]=panel.return_20d*100
panel["prior_volume_avg_20d"]=panel.groupby("ticker")["volume"].transform(lambda x:x.shift(1).rolling(20,min_periods=20).mean())
panel["event_volume_ratio"]=panel.volume/panel.prior_volume_avg_20d.replace(0,np.nan)
panel["valid_return_window_20d"]=panel.groupby("ticker")["valid_price_return"].transform(lambda x:x.astype(int).rolling(20,min_periods=20).sum().eq(20))
panel["runup_condition"]=panel.valid_return_window_20d&panel.return_20d.ge(.20)&panel.event_volume_ratio.ge(1.5)
panel["previous_condition"]=panel.groupby("ticker")["runup_condition"].shift(1).fillna(False).astype(bool)
panel["candidate"]=panel.runup_condition&~panel.previous_condition
panel["trading_day"]=panel.groupby("ticker").cumcount(); panel["surge_event"]=0
keep=[]
for ticker,g in panel.groupby("ticker",sort=False):
    last=None
    for idx,row in g.loc[g.candidate].iterrows():
        day=int(row.trading_day)
        if last is None or day-last>20: keep.append(idx); last=day
panel.loc[keep,"surge_event"]=1
events=panel.loc[panel.surge_event.eq(1),["ticker","date","return_20d_pct","event_volume_ratio"]].copy()
print("현재 GCS 사건 수:",len(events))

In [ ]:
# 2. 원본과 동일한 사건 직전 7일 비정상 검색 관심도와 5분위
# 사전 7일: t-7~t-1
# 평상시 기준: t-83~t-28에 위치한 완전한 8개 주

def complete_sum(series, start_date, end_date, expected_days=7):
    window=series.loc[start_date:end_date]
    return window.sum() if window.notna().sum()>=expected_days else np.nan

search_series={
    ticker:g.sort_values("date").set_index("date")["search_index"]
    for ticker,g in attention_raw.groupby("ticker",sort=False)
}

rows=[]
for row in events[["ticker","date"]].itertuples(index=False):
    value=np.nan
    if row.ticker in search_series:
        series=search_series[row.ticker]
        pre_sum=complete_sum(series,row.date-pd.Timedelta(days=7),row.date-pd.Timedelta(days=1))
        baseline=[]
        for week_number in range(8):
            baseline_end=row.date-pd.Timedelta(days=28+7*week_number)
            baseline_start=baseline_end-pd.Timedelta(days=6)
            baseline.append(complete_sum(series,baseline_start,baseline_end))
        baseline=np.asarray(baseline,dtype=float)
        if pd.notna(pre_sum) and np.isfinite(baseline).sum()==8:
            value=np.log1p(pre_sum)-np.median(np.log1p(baseline))
    rows.append({"ticker":row.ticker,"date":row.date,"search_pre_7d_abnormal":value})

event_path=events.merge(pd.DataFrame(rows),on=["ticker","date"],how="left",validate="one_to_one")
raw=event_path["search_pre_7d_abnormal"].replace([np.inf,-np.inf],np.nan)
lo,hi=raw.quantile([.01,.99]); winsorized=raw.clip(lo,hi)
event_path["search_attention_z"]=(winsorized-winsorized.mean())/winsorized.std()
labels=["Q1 최저","Q2","Q3","Q4","Q5 최고"]
valid=event_path.search_attention_z.notna(); event_path["search_attention_quintile"]=pd.NA
event_path.loc[valid,"search_attention_quintile"]=pd.qcut(event_path.loc[valid,"search_attention_z"],5,labels=labels,duplicates="drop")
print("전체 급등 사건:",len(event_path))
print("원자료로 관심도 계산 가능:",int(valid.sum()))
print(event_path.search_attention_quintile.value_counts(sort=False,dropna=False))

In [ ]:
# 3. 사건 이후 1~120거래일 시장조정 누적수익률
HORIZONS=list(range(1,121)); price=panel[["ticker","date","close"]].copy().sort_values(["ticker","date"])
for h in HORIZONS:
    price[f"stock_{h}d"]=price.groupby("ticker")["close"].shift(-h)/price.close-1
    market[f"market_{h}d"]=market.market_close.shift(-h)/market.market_close-1
price=price.merge(market[["date"]+[f"market_{h}d" for h in HORIZONS]],on="date",how="left",validate="many_to_one")
for h in HORIZONS: price[f"forward_abnormal_return_{h}d_pct"]=(price[f"stock_{h}d"]-price[f"market_{h}d"])*100
cols=["ticker","date"]+[f"forward_abnormal_return_{h}d_pct" for h in HORIZONS]
event_path=event_path.merge(price[cols],on=["ticker","date"],how="left",validate="one_to_one")
long=event_path.melt(id_vars=["ticker","date","search_attention_z","search_attention_quintile"],value_vars=cols[2:],var_name="hvar",value_name="abnormal_return")
long["horizon"]=long.hvar.str.extract(r"(\d+)",expand=False).astype(int); long=long.dropna(subset=["abnormal_return","search_attention_quintile"])
long["return_w"]=long.groupby("horizon")["abnormal_return"].transform(lambda x:x.clip(x.quantile(.01),x.quantile(.99)))
summary=long.groupby(["search_attention_quintile","horizon"],observed=True).agg(mean_return=("return_w","mean"),std=("return_w","std"),events=("return_w","count")).reset_index()
summary["se"]=summary["std"]/np.sqrt(summary["events"]); summary["lo"]=summary.mean_return-1.96*summary.se; summary["hi"]=summary.mean_return+1.96*summary.se
summary.to_csv(OUTPUT_DIR/"search_quintile_future_path.csv",index=False,encoding="utf-8-sig")

In [ ]:
# 4. 시각화
colors={"Q1 최저":"#2166ac","Q2":"#67a9cf","Q3":"#999999","Q4":"#ef8a62","Q5 최고":"#b2182b"}
fig,ax=plt.subplots(figsize=(15,8))
for q in labels:
    g=summary.loc[summary.search_attention_quintile.eq(q)].sort_values("horizon")
    ax.plot(g.horizon,g.mean_return,label=q,color=colors[q],linewidth=2.8 if q in ["Q1 최저","Q5 최고"] else 1.7)
    if q in ["Q1 최저","Q5 최고"]: ax.fill_between(g.horizon,g.lo,g.hi,color=colors[q],alpha=.12)
ax.axhline(0,color="black",linewidth=1)
for h in [5,20,60,120]: ax.axvline(h,color="gray",linestyle=":",linewidth=.8)
ax.set(title="사전 검색 관심도 5분위별 사건 이후 시장조정 누적수익률",xlabel="급등 사건 이후 거래일",ylabel="시장조정 누적수익률 (%)")
ax.legend(title="사전 검색 관심도 5분위",ncol=5,loc="upper center",bbox_to_anchor=(.5,-.12),frameon=False)
plt.tight_layout(); plt.savefig(OUTPUT_DIR/"search_quintile_future_path.png",dpi=300,bbox_inches="tight",facecolor="white"); plt.show()

# STAGE 5: [가설검증 H1] 관심 선행성 × 상승지속성 — Gold 데이터 기반

**태그:** 실행됨

**원본 노트북:** `01_관심선행_vs_가격경로.ipynb`

## FinDA — 관심지표 선행성 × 상승지속성 분석
### Gold Layer 재사용 버전

#### 연구 질문
> 급등 Event 이전에 검색·뉴스 관심이 더 먼저 나타난 사건일수록 이후 상승이 더 오래 지속되는가?

이 노트북은 **현재 FinDA 리팩토링의 실제 Gold 데이터 계약을 그대로 재사용**합니다.

#### 실제 입력
- `derived/events/event_master.parquet`
- `derived/events/event_snapshots_long.parquet`
- `derived/events/event_targets_long.parquet`
- `derived/events/event_bhar_path_long.parquet`

#### 핵심 원칙
1. 관심 lead를 `attention_daily`에서 다시 계산하지 않습니다.
2. STEP 5 Gold에 이미 저장된 `search_lead_days`, `news_lead_days`를 사용합니다.
3. 이 lead는 **달력일(calendar day) 차이**입니다.
4. 이번 분석의 최근 선행창은 `0~20 calendar days`로 제한합니다.
5. Target은 Peak Give-back Gold의 **`prediction_offset == 0`**만 사용합니다.
6. H=10/20/40을 비교합니다.
7. T시점 Control은 `event_snapshots_long`의 **T+0 snapshot**에 이미 저장된 `_T` / `_cutoff` 변수만 사용합니다.
8. T 이후 정보(`early_*`, 누적수급, sustain 등)는 Control에 넣지 않습니다.

> 참고: `event_targets_long`은 현재 최종 Direct T+20 모델 Target이 아니라 Peak Give-back 기반 Legacy/Robustness Gold입니다. 이번 분석은 예측모델 선정이 아니라 관심 선행성과 미래 경로의 관계를 보는 설명형/강건성 분석입니다.

### 0. 패키지 설치

In [ ]:
!pip -q install "google-cloud-storage>=2.16" "pyarrow>=15" "pandas>=2.1" "numpy>=1.26" "scipy>=1.11" "statsmodels>=0.14" "matplotlib>=3.8"

### 1. 인증 / Config

In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
import json, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from google.cloud import storage
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 220)

@dataclass(frozen=True)
class CFGClass:
    project_id: str = GCP_PROJECT_ID
    bucket: str = "finda_project"
    recent_lead_max_calendar_days: int = 20
    simultaneous_tolerance_calendar_days: int = 1
    horizons: tuple = (10, 20, 40)
    min_group_n: int = 10
    min_reg_n: int = 50
    output_prefix: str = "outputs/attention_lead_analysis_gold"
    local_cache: str = "/content/finda_attention_lead_gold"

CFG = CFGClass()
print(asdict(CFG))

LOCAL = Path(CFG.local_cache)
LOCAL.mkdir(parents=True, exist_ok=True)
(LOCAL / "figures").mkdir(exist_ok=True)

client = storage.Client.create_anonymous_client()  # 공개 읽기전용 버킷, 로그인 불필요
bucket = client.bucket(CFG.bucket)
print(f"✅ GCS: gs://{CFG.bucket}")

### 2. 실제 Gold 경로 고정

이 프로젝트의 STEP 5가 만든 재사용 Gold를 직접 사용합니다.

In [ ]:
PATHS = {
    "event_master": "derived/events/event_master.parquet",
    "snapshots": "derived/events/event_snapshots_long.parquet",
    "targets": "derived/events/event_targets_long.parquet",
    "bhar_path": "derived/events/event_bhar_path_long.parquet",
}

for key, gcs_path in PATHS.items():
    ok = bucket.blob(gcs_path).exists(client)
    print(f"{'✅' if ok else '❌'} {key:12s} -> gs://{CFG.bucket}/{gcs_path}")
    if not ok:
        raise FileNotFoundError(f"필수 Gold 파일 없음: gs://{CFG.bucket}/{gcs_path}")

def download_gold(key):
    gcs_path = PATHS[key]
    dst = LOCAL / Path(gcs_path).name
    if not dst.exists() or dst.stat().st_size == 0:
        bucket.blob(gcs_path).download_to_filename(str(dst))
        print("downloaded:", gcs_path, "->", dst)
    else:
        print("cached:", dst)
    return dst

LOCAL_FILES = {k: download_gold(k) for k in PATHS}

def read_parquet(key):
    return pd.read_parquet(LOCAL_FILES[key])

event_master = read_parquet("event_master")
snapshots = read_parquet("snapshots")
targets_raw = read_parquet("targets")
bhar_path = read_parquet("bhar_path")

print("\nShapes")
for name, df in [
    ("event_master", event_master),
    ("snapshots", snapshots),
    ("targets_raw", targets_raw),
    ("bhar_path", bhar_path),
]:
    print(name, df.shape)

### 3. Gold 계약 QA

In [ ]:
EXPECTED = {
    "event_master": [
        "event_id","ticker","event_date",
        "ret_20d_T","volume_ratio_prior20_T",
        "market_cap_T",
        "search_lead_days","news_lead_days",
    ],
    "snapshots": [
        "event_id","ticker","event_date","prediction_offset",
        "ret_20d_T","volume_ratio_prior20_T",
        "search_lead_days","news_lead_days",
    ],
    "targets": [
        "event_id","ticker","event_date","prediction_offset","target_horizon",
        "future_peak_bhar","future_end_bhar",
        "future_giveback_ratio","future_true_mdd",
        "label_class","binary_label",
    ],
    "bhar_path": [
        "event_id","ticker","event_date","prediction_offset",
        "future_step","bhar",
    ],
}

for name, df in [
    ("event_master", event_master),
    ("snapshots", snapshots),
    ("targets", targets_raw),
    ("bhar_path", bhar_path),
]:
    missing = [c for c in EXPECTED[name] if c not in df.columns]
    if missing:
        raise KeyError(f"{name} Gold contract 누락: {missing}\n실제 컬럼={list(df.columns)}")
    print(f"✅ {name} contract PASS")

for df in [event_master, snapshots, targets_raw, bhar_path]:
    df["event_id"] = df["event_id"].astype(str)
    df["ticker"] = df["ticker"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(6)
    df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce").dt.normalize()

snapshots["prediction_offset"] = pd.to_numeric(snapshots["prediction_offset"], errors="coerce")
targets_raw["prediction_offset"] = pd.to_numeric(targets_raw["prediction_offset"], errors="coerce")
targets_raw["target_horizon"] = pd.to_numeric(targets_raw["target_horizon"], errors="coerce")
bhar_path["prediction_offset"] = pd.to_numeric(bhar_path["prediction_offset"], errors="coerce")
bhar_path["future_step"] = pd.to_numeric(bhar_path["future_step"], errors="coerce")

assert event_master["event_id"].is_unique, "event_master event_id 중복"
assert not snapshots.duplicated(["event_id","prediction_offset"]).any(), "snapshot key 중복"
assert not targets_raw.duplicated(["event_id","prediction_offset","target_horizon"]).any(), "target key 중복"
assert not bhar_path.duplicated(["event_id","prediction_offset","future_step"]).any(), "BHAR path key 중복"

print("\nEvent master")
print("rows:", len(event_master))
print("tickers:", event_master["ticker"].nunique())
print("period:", event_master["event_date"].min(), "~", event_master["event_date"].max())

print("\nSnapshot offsets:", sorted(snapshots["prediction_offset"].dropna().unique().tolist()))
print("Target offsets:", sorted(targets_raw["prediction_offset"].dropna().unique().tolist()))
print("Target horizons:", sorted(targets_raw["target_horizon"].dropna().unique().tolist()))
print("BHAR path offsets:", sorted(bhar_path["prediction_offset"].dropna().unique().tolist()))

### 4. T+0 Gold Snapshot 추출

`event_snapshots_long`은 Event × prediction offset 구조입니다.  
이번 연구는 사건일 T에서 과거 관심 선행성과 이후 경로를 보는 것이므로 **offset=0만 사용**합니다.

여기에는 STEP 5가 이미 계산한:
- T시점 가격·거래량
- T시점 수급
- T시점 관심 lead
- T시점 시장 상태(`market_*_cutoff`, offset=0)

가 들어 있습니다.

In [ ]:
snapshot_t0 = snapshots.loc[snapshots["prediction_offset"].eq(0)].copy()

assert snapshot_t0["event_id"].is_unique, "T+0 snapshot event_id 중복"
assert len(snapshot_t0) > 0

# event_master와 lead parity 확인
parity_cols = ["event_id","search_lead_days","news_lead_days"]
em_lead = event_master[parity_cols].copy()
sp_lead = snapshot_t0[parity_cols].copy()

parity = em_lead.merge(
    sp_lead,
    on="event_id",
    how="inner",
    suffixes=("_event_master","_snapshot"),
    validate="one_to_one",
)

def same_or_na(a,b):
    return (a.eq(b)) | (a.isna() & b.isna())

search_parity = same_or_na(
    parity["search_lead_days_event_master"],
    parity["search_lead_days_snapshot"],
).mean()

news_parity = same_or_na(
    parity["news_lead_days_event_master"],
    parity["news_lead_days_snapshot"],
).mean()

print("snapshot_t0:", snapshot_t0.shape)
print("event match:", parity["event_id"].nunique())
print("search lead parity:", f"{search_parity:.2%}")
print("news lead parity:", f"{news_parity:.2%}")

assert search_parity == 1.0, "event_master vs snapshot search lead 불일치"
assert news_parity == 1.0, "event_master vs snapshot news lead 불일치"

print("✅ T+0 Gold lead parity PASS")

### 5. 최근 관심 선행 변수 생성

STEP 5의 원본 `search_lead_days`, `news_lead_days`는:
> `event_date - last_attention_entry_date`

로 만든 **달력일 차이**입니다.

In [ ]:
feature = snapshot_t0.copy()

feature = feature.rename(columns={
    "search_lead_days": "search_lead_days_gold",
    "news_lead_days": "news_lead_days_gold",
})

def recent_lead(s, max_days):
    x = pd.to_numeric(s, errors="coerce")
    return x.where(x.between(0, max_days, inclusive="both"))

feature["search_lead_days"] = recent_lead(
    feature["search_lead_days_gold"],
    CFG.recent_lead_max_calendar_days,
)
feature["news_lead_days"] = recent_lead(
    feature["news_lead_days_gold"],
    CFG.recent_lead_max_calendar_days,
)

feature["search_news_lead_gap"] = (
    feature["search_lead_days"] - feature["news_lead_days"]
)

s = feature["search_lead_days"].notna()
n = feature["news_lead_days"].notna()
gap = feature["search_news_lead_gap"]

tol = CFG.simultaneous_tolerance_calendar_days

feature["attention_timing_type"] = np.select(
    [
        ~s & ~n,
        s & ~n,
        ~s & n,
        s & n & (gap > tol),
        s & n & (gap < -tol),
        s & n & (gap.abs() <= tol),
    ],
    [
        "No Attention",
        "Search Only",
        "News Only",
        "Search First",
        "News First",
        "Simultaneous",
    ],
    default="Unknown",
)

def lead_group(v):
    if pd.isna(v):
        return "No signal"
    if v <= 2:
        return "0-2D"
    if v <= 5:
        return "3-5D"
    if v <= 10:
        return "6-10D"
    return "11-20D"

feature["search_lead_group"] = feature["search_lead_days"].map(lead_group)
feature["news_lead_group"] = feature["news_lead_days"].map(lead_group)

for c in ["search_lead_days","news_lead_days"]:
    assert feature[c].dropna().between(0, CFG.recent_lead_max_calendar_days).all()

print("=== Gold raw lead ===")
display(
    feature[["search_lead_days_gold","news_lead_days_gold"]]
    .describe()
    .T
)

print("\n=== Recent <=20 calendar-day signal ===")
print("search signal:", f"{feature['search_lead_days'].notna().mean():.2%}")
print("news signal  :", f"{feature['news_lead_days'].notna().mean():.2%}")
display(feature["attention_timing_type"].value_counts(dropna=False).to_frame("N"))

### 6. T+0 H10/H20/H40 Target 표준화

Peak Give-back Gold를 그대로 재사용합니다.

In [ ]:
target_t0 = targets_raw.loc[
    targets_raw["prediction_offset"].eq(0)
    & targets_raw["target_horizon"].isin(CFG.horizons)
].copy()

target_t0 = target_t0.rename(columns={
    "target_horizon": "horizon",
    "future_peak_bhar": "peak_return",
    "future_end_bhar": "final_return",
    "future_giveback_ratio": "giveback",
    "future_true_mdd": "mdd",
})

OUTCOMES = ["peak_return","final_return","giveback","mdd"]

for c in OUTCOMES + ["binary_label"]:
    target_t0[c] = pd.to_numeric(target_t0[c], errors="coerce")

target_t0["horizon"] = pd.to_numeric(target_t0["horizon"], errors="coerce").astype("Int64")

assert not target_t0.duplicated(["event_id","horizon"]).any()
assert set(target_t0["horizon"].dropna().astype(int).unique()) == set(CFG.horizons)

print("target_t0:", target_t0.shape)
display(target_t0["horizon"].value_counts().sort_index().to_frame("N"))

print("\nLabel distribution")
display(
    target_t0.groupby(["horizon","label_class"], dropna=False)
    .size()
    .rename("N")
    .reset_index()
)

print("\nOutcome availability")
display(target_t0.groupby("horizon")[OUTCOMES].count())

### 7. 최종 분석 Mart 생성

**Gold T+0 Snapshot 1행/Event × T+0 Peak Give-back Target H10/H20/H40**를 결합합니다.

In [ ]:
analysis = feature.merge(
    target_t0[
        [
            "event_id","ticker","event_date","horizon",
            "peak_return","final_return","giveback","mdd",
            "label_class","binary_label",
            "target_start_date","target_end_date",
        ]
    ],
    on=["event_id","ticker","event_date"],
    how="inner",
    validate="one_to_many",
)

analysis["event_year"] = analysis["event_date"].dt.year.astype(int)

dup = analysis.duplicated(["event_id","horizon"]).sum()
assert dup == 0, f"analysis event×horizon 중복={dup}"
assert len(analysis) > 0

event_h_n = analysis.groupby("event_id")["horizon"].nunique()
common_ids = event_h_n[event_h_n.eq(len(CFG.horizons))].index
analysis_common = analysis[analysis["event_id"].isin(common_ids)].copy()

feature_ids = set(feature["event_id"])
target_ids = set(target_t0["event_id"])

print("========== ANALYSIS MART QA ==========")
print("T+0 Gold feature events :", feature["event_id"].nunique())
print("T+0 target events       :", target_t0["event_id"].nunique())
print("matched analysis events :", analysis["event_id"].nunique())
print("analysis rows           :", len(analysis))
print("match rate vs features  :", f"{analysis['event_id'].nunique()/feature['event_id'].nunique():.2%}")
print("common H10/H20/H40 events:", analysis_common["event_id"].nunique())
print("duplicate event×horizon :", dup)

display(analysis["horizon"].value_counts().sort_index().to_frame("N"))

### 8. Horizon / Lead 표본 QA

In [ ]:
qa_rows=[]
for h,g in analysis.groupby("horizon"):
    row={
        "horizon":int(h),
        "rows":len(g),
        "events":g["event_id"].nunique(),
        "tickers":g["ticker"].nunique(),
        "search_signal_recent20_%":round(g["search_lead_days"].notna().mean()*100,2),
        "news_signal_recent20_%":round(g["news_lead_days"].notna().mean()*100,2),
    }
    for c in OUTCOMES:
        row[f"{c}_N"]=int(g[c].notna().sum())
        row[f"{c}_median"]=float(g[c].median()) if g[c].notna().any() else np.nan
    qa_rows.append(row)

horizon_qa=pd.DataFrame(qa_rows).sort_values("horizon")
display(horizon_qa)

print("\nLead group counts — event level")
display(feature["search_lead_group"].value_counts(dropna=False).to_frame("Search N"))
display(feature["news_lead_group"].value_counts(dropna=False).to_frame("News N"))

### 9. Lead 분포

In [ ]:
lead_distribution = pd.DataFrame([
    {
        "source":"search",
        "all_gold_nonnull":int(feature["search_lead_days_gold"].notna().sum()),
        "recent20_nonnull":int(feature["search_lead_days"].notna().sum()),
        "recent20_median":feature["search_lead_days"].median(),
        "recent20_mean":feature["search_lead_days"].mean(),
    },
    {
        "source":"news",
        "all_gold_nonnull":int(feature["news_lead_days_gold"].notna().sum()),
        "recent20_nonnull":int(feature["news_lead_days"].notna().sum()),
        "recent20_median":feature["news_lead_days"].median(),
        "recent20_mean":feature["news_lead_days"].mean(),
    },
])
display(lead_distribution)

fig, ax = plt.subplots(figsize=(8,4))
for c,label in [("search_lead_days","Search"),("news_lead_days","News")]:
    x=feature[c].dropna()
    ax.hist(x, bins=np.arange(-.5,21.5,1), alpha=.45, label=label)
ax.set_title("Recent attention lead distribution")
ax.set_xlabel("Calendar days before event")
ax.set_ylabel("Event count")
ax.legend()
fig.tight_layout()
fig.savefig(LOCAL/"figures"/"lead_distribution_calendar_days.png",dpi=180,bbox_inches="tight")
plt.show()

### 10. Lead Group / Timing Type별 Outcome

In [ ]:
def outcome_summary(df, group_col):
    return (
        df.groupby(["horizon",group_col], dropna=False)[OUTCOMES]
        .agg(["count","mean","median"])
        .pipe(lambda x: x.set_axis([f"{a}_{b}" for a,b in x.columns], axis=1))
        .reset_index()
    )

search_group_outcomes = outcome_summary(analysis, "search_lead_group")
news_group_outcomes = outcome_summary(analysis, "news_lead_group")
timing_type_outcomes = outcome_summary(analysis, "attention_timing_type")

display(search_group_outcomes)
display(news_group_outcomes)
display(timing_type_outcomes)

### 11. Spearman — 선행일수와 미래 경로

In [ ]:
spearman_rows=[]

for h in CFG.horizons:
    gh=analysis[analysis["horizon"].eq(h)]
    for lead in ["search_lead_days","news_lead_days"]:
        for outcome in OUTCOMES:
            z=gh[[lead,outcome]].dropna()
            if len(z)>=20 and z[lead].nunique()>1 and z[outcome].nunique()>1:
                rho,p=stats.spearmanr(z[lead],z[outcome])
                spearman_rows.append({
                    "horizon":h,
                    "lead_variable":lead,
                    "outcome":outcome,
                    "n":len(z),
                    "rho":rho,
                    "p_value":p,
                })

spearman_results=pd.DataFrame(spearman_rows)
display(spearman_results.sort_values(["lead_variable","outcome","horizon"]))

### 12. Kruskal-Wallis — 그룹별 미래 경로 차이

In [ ]:
def kruskal_results_for(df, group_col):
    rows=[]
    for h in CFG.horizons:
        gh=df[df["horizon"].eq(h)]
        for outcome in ["final_return","giveback","mdd"]:
            arrays=[]
            used=[]
            excluded=[]
            for name,g in gh.groupby(group_col, dropna=False):
                x=g[outcome].dropna().to_numpy()
                if len(x)>=CFG.min_group_n:
                    arrays.append(x)
                    used.append(f"{name}(n={len(x)})")
                else:
                    excluded.append(f"{name}(n={len(x)})")
            if len(arrays)>=2:
                stat,p=stats.kruskal(*arrays)
                rows.append({
                    "horizon":h,
                    "group_variable":group_col,
                    "outcome":outcome,
                    "H_stat":stat,
                    "p_value":p,
                    "groups_used":" | ".join(used),
                    "groups_excluded":" | ".join(excluded),
                })
    return pd.DataFrame(rows)

kruskal_results=pd.concat([
    kruskal_results_for(analysis,"search_lead_group"),
    kruskal_results_for(analysis,"news_lead_group"),
    kruskal_results_for(analysis,"attention_timing_type"),
],ignore_index=True)

display(kruskal_results)

### 13. `label_class` 구성 비교

`binary_label`은 neutral이 NaN이므로 전체 상승지속성 분석의 Main Outcome으로 쓰지 않습니다.  
3개 클래스(`sustained`, `overheat_reversal`, `neutral`)를 모두 유지합니다.

In [ ]:
def label_share(df, group_col):
    x=(
        df.groupby(["horizon",group_col,"label_class"],dropna=False)
        .size().rename("N").reset_index()
    )
    x["share"]=x["N"]/x.groupby(["horizon",group_col])["N"].transform("sum")
    return x

search_label_summary=label_share(analysis,"search_lead_group")
news_label_summary=label_share(analysis,"news_lead_group")
timing_label_summary=label_share(analysis,"attention_timing_type")

display(search_label_summary)
display(timing_label_summary)

### 14. 실제 Gold `event_bhar_path_long`로 미래 경로 시각화

주가 canonical을 다시 읽지 않고 STEP 5의 BHAR Gold를 그대로 재사용합니다.  
`prediction_offset == 0`, `future_step <= 40`만 사용합니다.

In [ ]:
path_t0 = bhar_path.loc[
    bhar_path["prediction_offset"].eq(0)
    & bhar_path["future_step"].between(1, max(CFG.horizons))
].copy()

path_t0 = path_t0.merge(
    feature[
        ["event_id","search_lead_group","news_lead_group","attention_timing_type"]
    ],
    on="event_id",
    how="inner",
    validate="many_to_one",
)

def median_path(df, group_col):
    return (
        df.groupby([group_col,"future_step"],dropna=False)
        .agg(median_bhar=("bhar","median"),n=("bhar","count"))
        .reset_index()
    )

search_path=median_path(path_t0,"search_lead_group")
news_path=median_path(path_t0,"news_lead_group")
timing_path=median_path(path_t0,"attention_timing_type")

ORDER=["No signal","0-2D","3-5D","6-10D","11-20D"]

fig,ax=plt.subplots(figsize=(9,5))
for grp in ORDER:
    z=search_path[search_path["search_lead_group"].eq(grp)]
    if len(z):
        ax.plot(z["future_step"],z["median_bhar"],label=grp)
ax.axhline(0,linewidth=.8)
ax.set_title("Median BHAR path by search lead group")
ax.set_xlabel("Trading steps after event T")
ax.set_ylabel("Median BHAR")
ax.legend(ncol=2)
fig.tight_layout()
fig.savefig(LOCAL/"figures"/"search_lead_median_bhar_path.png",dpi=180,bbox_inches="tight")
plt.show()

fig,ax=plt.subplots(figsize=(9,5))
for grp in ORDER:
    z=news_path[news_path["news_lead_group"].eq(grp)]
    if len(z):
        ax.plot(z["future_step"],z["median_bhar"],label=grp)
ax.axhline(0,linewidth=.8)
ax.set_title("Median BHAR path by news lead group")
ax.set_xlabel("Trading steps after event T")
ax.set_ylabel("Median BHAR")
ax.legend(ncol=2)
fig.tight_layout()
fig.savefig(LOCAL/"figures"/"news_lead_median_bhar_path.png",dpi=180,bbox_inches="tight")
plt.show()

### 15. T시점 통제변수 — 실제 Gold 컬럼만 사용

이전 버전처럼 `stock_daily + market_daily`에서 generic alias를 다시 만들지 않습니다.

#### Core controls
실제 STEP 5/Gold 계약에서 사건일 T에 알 수 있는 값만 사용:
- `ret_20d_T`
- `volume_ratio_prior20_T`
- `volatility_20d_T`
- `log_market_cap_T` ← `market_cap_T`에서 변환
- `foreign_net_buy_to_mcap_T`
- `institution_net_buy_to_mcap_T`
- `individual_net_buy_to_mcap_T`
- `market_volatility_20d_cutoff` (`offset=0`이므로 T 시장 상태)
- `market_decline_ratio_cutoff` (`offset=0`)

`per_T/pbr_T`는 결측으로 표본을 과도하게 줄일 수 있어 Core에서 제외합니다.

In [ ]:
analysis_reg=analysis.copy()

if "market_cap_T" in analysis_reg.columns:
    mcap=pd.to_numeric(analysis_reg["market_cap_T"],errors="coerce")
    analysis_reg["log_market_cap_T"]=np.log(mcap.where(mcap>0))

CONTROL_CANDIDATES=[
    "ret_20d_T",
    "volume_ratio_prior20_T",
    "volatility_20d_T",
    "log_market_cap_T",
    "foreign_net_buy_to_mcap_T",
    "institution_net_buy_to_mcap_T",
    "individual_net_buy_to_mcap_T",
    "market_volatility_20d_cutoff",
    "market_decline_ratio_cutoff",
]

CONTROL_FORBIDDEN_TOKENS=[
    "early_",
    "_cum",
    "sustain",
    "future_",
    "giveback_T",
    "volume_change_vs_T",
]

existing_controls=[]
control_qa=[]

for c in CONTROL_CANDIDATES:
    exists=c in analysis_reg.columns
    nonnull_pct=(
        pd.to_numeric(analysis_reg[c],errors="coerce").notna().mean()
        if exists else 0.0
    )
    forbidden=any(tok.lower() in c.lower() for tok in CONTROL_FORBIDDEN_TOKENS)
    use=exists and not forbidden and nonnull_pct>=0.60

    control_qa.append({
        "control":c,
        "exists":exists,
        "nonnull_%":round(nonnull_pct*100,2),
        "forbidden":forbidden,
        "use_core":use,
    })

    if use:
        analysis_reg[c]=pd.to_numeric(analysis_reg[c],errors="coerce")
        existing_controls.append(c)

control_qa=pd.DataFrame(control_qa)
display(control_qa)

if len(existing_controls)<3:
    print("⚠️ Core control이 3개 미만입니다. OLS는 최소 조정모형으로 실행하거나 SKIP될 수 있습니다.")

print("사용 controls:", existing_controls)

### 16. 조정 OLS — 안전한 matrix 방식

- lead가 있는 사건들 안에서 **선행일수 1 calendar day 증가**와 Outcome 관계를 봅니다.
- `No signal`은 OLS에 숫자 0으로 강제하지 않습니다. 그룹 비교에서 따로 봅니다.
- 동일 ticker의 반복 Event를 고려해 ticker cluster-robust SE를 사용합니다.
- 개별 모델 실패는 전체 노트북을 죽이지 않고 `status/error`에 기록합니다.

In [ ]:
def fit_adjusted_ols(df, lead, outcome, controls):
    base_cols=["ticker","event_year",lead,outcome]+controls
    z=df[base_cols].copy()

    for c in [lead,outcome]+controls:
        z[c]=pd.to_numeric(z[c],errors="coerce")

    z=z.replace([np.inf,-np.inf],np.nan).dropna()

    if len(z)<CFG.min_reg_n:
        return {
            "status":"SKIPPED",
            "reason":f"N<{CFG.min_reg_n}",
            "N":len(z),
        }

    if z[lead].nunique()<3 or z[outcome].nunique()<3:
        return {
            "status":"SKIPPED",
            "reason":"insufficient variation",
            "N":len(z),
        }

    X=z[[lead]+controls].astype(float).copy()
    year_dummies=pd.get_dummies(
        z["event_year"].astype(str),
        prefix="year",
        drop_first=True,
        dtype=float,
    )
    X=pd.concat([X.reset_index(drop=True),year_dummies.reset_index(drop=True)],axis=1)
    X=sm.add_constant(X,has_constant="add")
    y=z[outcome].astype(float).reset_index(drop=True)

    # 상수/완전 공선성에 가까운 컬럼 제거
    nunique=X.nunique(dropna=False)
    keep=[c for c in X.columns if c=="const" or nunique[c]>1]
    X=X[keep]

    try:
        model=sm.OLS(y,X)
        if z["ticker"].nunique()>=10:
            res=model.fit(
                cov_type="cluster",
                cov_kwds={"groups":z["ticker"].reset_index(drop=True)},
            )
            cov="cluster(ticker)"
        else:
            res=model.fit(cov_type="HC3")
            cov="HC3"

        if lead not in res.params.index:
            return {
                "status":"SKIPPED",
                "reason":"lead dropped from design matrix",
                "N":len(z),
            }

        ci=res.conf_int().loc[lead]

        return {
            "status":"PASS",
            "reason":"",
            "N":len(z),
            "tickers":z["ticker"].nunique(),
            "coef":float(res.params[lead]),
            "std_err":float(res.bse[lead]),
            "p_value":float(res.pvalues[lead]),
            "ci_low":float(ci.iloc[0]),
            "ci_high":float(ci.iloc[1]),
            "r2":float(res.rsquared),
            "cov_type":cov,
            "controls":" | ".join(controls),
        }

    except Exception as e:
        return {
            "status":"ERROR",
            "reason":f"{type(e).__name__}: {e}",
            "N":len(z),
        }

reg_rows=[]

for h in CFG.horizons:
    gh=analysis_reg[analysis_reg["horizon"].eq(h)]

    for lead in ["search_lead_days","news_lead_days"]:
        for outcome in ["final_return","giveback","mdd"]:
            r=fit_adjusted_ols(gh,lead,outcome,existing_controls)
            r.update({
                "horizon":h,
                "lead_variable":lead,
                "outcome":outcome,
            })
            reg_rows.append(r)

adjusted_regression=pd.DataFrame(reg_rows)

display(
    adjusted_regression[
        [
            "horizon","lead_variable","outcome","status","N",
            *[c for c in ["coef","p_value","ci_low","ci_high","r2","reason"] if c in adjusted_regression.columns]
        ]
    ]
)

print("\nOLS status")
display(adjusted_regression["status"].value_counts(dropna=False).to_frame("N"))

### 17. Common Sample 강건성

H10/H20/H40이 모두 존재하는 동일 Event만 사용해 Spearman을 다시 계산합니다.

In [ ]:
common_spearman_rows=[]

for h in CFG.horizons:
    gh=analysis_common[analysis_common["horizon"].eq(h)]
    for lead in ["search_lead_days","news_lead_days"]:
        for outcome in OUTCOMES:
            z=gh[[lead,outcome]].dropna()
            if len(z)>=20 and z[lead].nunique()>1 and z[outcome].nunique()>1:
                rho,p=stats.spearmanr(z[lead],z[outcome])
                common_spearman_rows.append({
                    "horizon":h,
                    "lead_variable":lead,
                    "outcome":outcome,
                    "n":len(z),
                    "rho":rho,
                    "p_value":p,
                })

common_spearman=pd.DataFrame(common_spearman_rows)
display(common_spearman)

### 18. 결과 저장 / GCS 업로드

In [ ]:
ARTIFACTS={
    "analysis_event_horizon.parquet":analysis,
    "analysis_common_sample.parquet":analysis_common,
    "gold_t0_features.parquet":feature,
    "horizon_qa.csv":horizon_qa,
    "lead_distribution.csv":lead_distribution,
    "search_group_outcomes.csv":search_group_outcomes,
    "news_group_outcomes.csv":news_group_outcomes,
    "timing_type_outcomes.csv":timing_type_outcomes,
    "spearman_results.csv":spearman_results,
    "common_sample_spearman.csv":common_spearman,
    "kruskal_results.csv":kruskal_results,
    "search_label_summary.csv":search_label_summary,
    "news_label_summary.csv":news_label_summary,
    "timing_label_summary.csv":timing_label_summary,
    "search_median_bhar_path.csv":search_path,
    "news_median_bhar_path.csv":news_path,
    "timing_median_bhar_path.csv":timing_path,
    "control_qa.csv":control_qa,
    "adjusted_regression.csv":adjusted_regression,
}

def upload(local_path, gcs_name):
    # 심사용 실행에서는 팀 GCS에 쓰지 않고 로컬 저장만 확인합니다.
    print("로컬 저장 완료(GCS 업로드 생략):", local_path)

for name,df in ARTIFACTS.items():
    p=LOCAL/name
    if name.endswith(".parquet"):
        df.to_parquet(p,index=False)
    else:
        df.to_csv(p,index=False,encoding="utf-8-sig")
    upload(p,f"{CFG.output_prefix}/{name}")

for p in sorted((LOCAL/"figures").glob("*.png")):
    upload(p,f"{CFG.output_prefix}/figures/{p.name}")

meta={
    "config":asdict(CFG),
    "sources":PATHS,
    "lead_definition":{
        "source":"STEP5 Gold search_lead_days/news_lead_days",
        "unit":"calendar days",
        "analysis_recent_window":"0-20 calendar days",
    },
    "target_definition":{
        "source":"event_targets_long Peak Give-back legacy/robustness target",
        "prediction_offset":0,
        "horizons":list(CFG.horizons),
    },
    "control_source":"event_snapshots_long prediction_offset=0 only",
}
meta_path=LOCAL/"analysis_metadata.json"
meta_path.write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding="utf-8")
upload(meta_path,f"{CFG.output_prefix}/analysis_metadata.json")

UPLOAD_DONE=True
print("✅ 결과 저장 완료")

### 19. 최종 QA

In [ ]:
checks=[]

def ck(name,cond,detail=""):
    checks.append({
        "check":name,
        "status":"PASS" if bool(cond) else "FAIL",
        "detail":detail,
    })

ck("Gold event_master load",len(event_master)>0,str(event_master.shape))
ck("Gold T+0 snapshot unique",snapshot_t0["event_id"].is_unique,str(snapshot_t0.shape))
ck("Gold lead parity",search_parity==1 and news_parity==1,
   f"search={search_parity:.2%}, news={news_parity:.2%}")
ck("T+0 targets",len(target_t0)>0,str(target_t0.shape))

for h in CFG.horizons:
    ck(f"H{h} target",(target_t0["horizon"]==h).any(),f"N={(target_t0['horizon']==h).sum()}")

ck("Analysis Mart",len(analysis)>0,str(analysis.shape))
ck("No event×horizon duplicate",
   not analysis.duplicated(["event_id","horizon"]).any())
ck("BHAR Gold path",len(path_t0)>0,str(path_t0.shape))
ck("Spearman",not spearman_results.empty,f"rows={len(spearman_results)}")
ck("Kruskal",not kruskal_results.empty,f"rows={len(kruskal_results)}")
ck("Adjusted OLS object",len(adjusted_regression)>0,
   adjusted_regression["status"].value_counts().to_dict())
ck("GCS output",globals().get("UPLOAD_DONE",False),
   f"gs://{CFG.bucket}/{CFG.output_prefix}/")

final_qa=pd.DataFrame(checks)
display(final_qa)

failed=final_qa[final_qa["status"].eq("FAIL")]
if len(failed):
    raise RuntimeError(
        "Final QA FAIL:\n"+failed[["check","detail"]].to_string(index=False)
    )

print("✅ FINAL QA — structural FAIL 없음")

### 20. 해석 순서

1. **최근 20 calendar-day 검색/뉴스 신호 비율**을 확인
2. Lead Group별 `final_return / giveback / mdd` 방향 확인
3. Spearman의 부호와 H10/H20/H40 일관성 확인
4. Kruskal-Wallis로 그룹 차이 확인
5. `label_class` 3분류 구성에서도 같은 패턴인지 확인
6. BHAR path로 실제 미래 경로 형태 확인
7. T시점 Core Control 조정 후 OLS lead 계수 방향이 유지되는지 확인
8. Common Sample에서도 결론이 유지되는지 확인

#### 해석 문장
- 가능: “관심 선행 정도와 이후 상승 지속성 사이에 연관성이 관찰되었다.”
- 금지: “관심 선행이 상승 지속을 유발했다.”

#### 주의
`event_targets_long`은 최종 Direct T+20 q35 Target이 아니라 Peak Give-back Legacy/Robustness Target입니다.  
따라서 이 결과는 **관심 선행성과 주가 경로의 설명분석**으로 사용하고, 최종 예측모델 성능 주장과 섞지 않습니다.

# STAGE 6: [가설검증 H2] 관심 지속성 × 상승지속성 — Gold 데이터 기반

**태그:** 실행됨

**원본 노트북:** `02_관심지속성_vs_상승지속성.ipynb`

## FinDA — H2 관심 지속성 × 상승지속성 추가 분석

### 연구 질문

> 단순히 관심이 주가 급등보다 먼저 나타나는 것보다, **높은 관심이 사건일 T 직전까지 지속된 경우** 이후 상승 지속성이 더 높은가?

#### H2
- 검색: `search_high_share_7`이 높을수록 이후 `final_return` ↑, `giveback` ↓, MDD 개선?
- 뉴스: `news_high_share_7`이 높을수록 이후 `final_return` ↑, `giveback` ↓, MDD 개선?
- 선행 신호가 실제로 있었던 사건에서는 `lead_days × high_share_7` 결합효과가 존재하는가?

#### 데이터 계층
- Silver: `curated/attention_daily.parquet`
- Gold: `derived/events/event_snapshots_long.parquet`
- Gold: `derived/events/event_targets_long.parquet`
- Gold: `derived/events/event_bhar_path_long.parquet`

#### 중요한 원칙
1. Event를 다시 정의하지 않는다.
2. 기존 Gold의 `search_lead_days`, `news_lead_days`를 그대로 사용한다.
3. `high_share_7`은 T 시점에 이미 계산되어 있는 과거 7일 관심 지속성 변수다.
4. T 이후 관심정보는 사용하지 않는다.
5. Main H2는 **최근 20 calendar-day lead 신호가 있는 사건 안에서** high-share 효과를 본다.
6. 전체 Event 분석은 보조 결과로 같이 제시한다.
7. H10/H20/H40 Peak Give-back outcome을 동일하게 사용한다.

### 0. 패키지 설치

In [ ]:
!pip -q install "google-cloud-storage>=2.16" "pyarrow>=15" "pandas>=2.1" "numpy>=1.26" "scipy>=1.11" "statsmodels>=0.14" "matplotlib>=3.8"

### 1. 인증 / Config

In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
import json, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from google.cloud import storage
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 220)

@dataclass(frozen=True)
class AnalysisConfig:
    project_id: str = GCP_PROJECT_ID
    bucket: str = "finda_project"
    recent_lead_max_calendar_days: int = 20
    horizons: tuple = (10, 20, 40)
    min_group_n: int = 10
    min_reg_n: int = 50
    output_prefix: str = "outputs/attention_persistence_analysis_gold"
    local_cache: str = "/content/finda_attention_persistence"

CFG = AnalysisConfig()
print(asdict(CFG))

LOCAL = Path(CFG.local_cache)
LOCAL.mkdir(parents=True, exist_ok=True)
(LOCAL / "figures").mkdir(exist_ok=True)

client = storage.Client.create_anonymous_client()  # 공개 읽기전용 버킷, 로그인 불필요
bucket = client.bucket(CFG.bucket)
print(f"✅ GCS: gs://{CFG.bucket}")

### 2. 실제 canonical / Gold 경로

In [ ]:
PATHS = {
    "attention": "curated/attention_daily.parquet",
    "snapshots": "derived/events/event_snapshots_long.parquet",
    "targets": "derived/events/event_targets_long.parquet",
    "bhar_path": "derived/events/event_bhar_path_long.parquet",
}

for key, path in PATHS.items():
    ok = bucket.blob(path).exists(client)
    print(f"{'✅' if ok else '❌'} {key:10s} -> gs://{CFG.bucket}/{path}")
    if not ok:
        raise FileNotFoundError(f"필수 파일 없음: gs://{CFG.bucket}/{path}")

def download(key):
    src = PATHS[key]
    dst = LOCAL / Path(src).name
    if not dst.exists() or dst.stat().st_size == 0:
        bucket.blob(src).download_to_filename(str(dst))
        print("downloaded:", src, "->", dst)
    else:
        print("cached:", dst)
    return dst

LOCAL_FILES = {k: download(k) for k in PATHS}

attention = pd.read_parquet(LOCAL_FILES["attention"])
snapshots = pd.read_parquet(LOCAL_FILES["snapshots"])
targets_raw = pd.read_parquet(LOCAL_FILES["targets"])
bhar_path = pd.read_parquet(LOCAL_FILES["bhar_path"])

print("attention :", attention.shape)
print("snapshots :", snapshots.shape)
print("targets   :", targets_raw.shape)
print("bhar_path :", bhar_path.shape)

### 3. Key / 컬럼 QA

In [ ]:
def norm_ticker(s):
    return (
        s.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(6)
    )

for df in [attention, snapshots, targets_raw, bhar_path]:
    if "ticker" in df.columns:
        df["ticker"] = norm_ticker(df["ticker"])

attention["date"] = pd.to_datetime(attention["date"], errors="coerce").dt.normalize()

for df in [snapshots, targets_raw, bhar_path]:
    df["event_id"] = df["event_id"].astype(str)
    df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce").dt.normalize()

snapshots["prediction_offset"] = pd.to_numeric(snapshots["prediction_offset"], errors="coerce")
targets_raw["prediction_offset"] = pd.to_numeric(targets_raw["prediction_offset"], errors="coerce")
targets_raw["target_horizon"] = pd.to_numeric(targets_raw["target_horizon"], errors="coerce")
bhar_path["prediction_offset"] = pd.to_numeric(bhar_path["prediction_offset"], errors="coerce")
bhar_path["future_step"] = pd.to_numeric(bhar_path["future_step"], errors="coerce")

ATTENTION_REQUIRED = [
    "ticker","date",
    "search_pct_50","search_ratio_7_30","search_slope_7","search_high_share_7",
    "news_pct_50","news_ratio_7_30","news_slope_7","news_high_share_7",
]
SNAP_REQUIRED = [
    "event_id","ticker","event_date","prediction_offset",
    "search_lead_days","news_lead_days",
    "ret_20d_T","volume_ratio_prior20_T","volatility_20d_T","market_cap_T",
    "foreign_net_buy_to_mcap_T","institution_net_buy_to_mcap_T","individual_net_buy_to_mcap_T",
    "market_volatility_20d_cutoff","market_decline_ratio_cutoff",
]
TARGET_REQUIRED = [
    "event_id","ticker","event_date","prediction_offset","target_horizon",
    "future_peak_bhar","future_end_bhar","future_giveback_ratio","future_true_mdd",
    "label_class","binary_label",
]

for name, df, req in [
    ("attention", attention, ATTENTION_REQUIRED),
    ("snapshots", snapshots, SNAP_REQUIRED),
    ("targets", targets_raw, TARGET_REQUIRED),
]:
    missing = [c for c in req if c not in df.columns]
    if missing:
        raise KeyError(f"{name} 필수 컬럼 누락: {missing}")

assert not attention.duplicated(["ticker","date"]).any(), "attention ticker+date 중복"
assert not snapshots.duplicated(["event_id","prediction_offset"]).any(), "snapshot key 중복"
assert not targets_raw.duplicated(["event_id","prediction_offset","target_horizon"]).any(), "target key 중복"

for c in ["search_high_share_7","news_high_share_7"]:
    x = pd.to_numeric(attention[c], errors="coerce")
    bad = x.dropna().loc[lambda s: (s < 0) | (s > 1)]
    assert bad.empty, f"{c} 범위 오류"

print("✅ INPUT CONTRACT QA PASS")

### 4. T+0 Gold Snapshot + 사건일 Attention 결합

`event_snapshots_long[prediction_offset=0]`에서:
- 기존 Gold lead
- 사건 T 통제변수

를 가져오고,

`attention_daily[ticker, event_date]`에서:
- `high_share_7`
- `slope_7`
- `ratio_7_30`
- `pct_50`

를 붙입니다.

T 이후 attention은 사용하지 않습니다.

In [ ]:
snapshot_t0 = snapshots.loc[snapshots["prediction_offset"].eq(0)].copy()
assert snapshot_t0["event_id"].is_unique

att_event = attention[
    [
        "ticker","date",
        "search_pct_50","search_ratio_7_30","search_slope_7","search_high_share_7",
        "news_pct_50","news_ratio_7_30","news_slope_7","news_high_share_7",
    ]
].rename(columns={"date":"event_date"}).copy()

feature = snapshot_t0.merge(
    att_event,
    on=["ticker","event_date"],
    how="left",
    validate="one_to_one",
)

match_any = (
    feature["search_high_share_7"].notna()
    | feature["news_high_share_7"].notna()
).mean()

print("snapshot_t0:", snapshot_t0.shape)
print("feature    :", feature.shape)
print("attention match(any high_share):", f"{match_any:.2%}")

if match_any < 0.95:
    print("⚠️ 사건일 Attention exact match가 95% 미만입니다. 누락 종목/날짜를 확인하세요.")

for c in [
    "search_high_share_7","news_high_share_7",
    "search_slope_7","news_slope_7",
    "search_ratio_7_30","news_ratio_7_30",
    "search_lead_days","news_lead_days",
]:
    feature[c] = pd.to_numeric(feature[c], errors="coerce")

# Gold lead는 calendar-day 차이
def recent_lead(s):
    return s.where(s.between(0, CFG.recent_lead_max_calendar_days, inclusive="both"))

feature["search_lead_recent20"] = recent_lead(feature["search_lead_days"])
feature["news_lead_recent20"] = recent_lead(feature["news_lead_days"])

display(
    feature[
        [
            "event_id","ticker","event_date",
            "search_lead_recent20","search_high_share_7","search_slope_7","search_ratio_7_30",
            "news_lead_recent20","news_high_share_7","news_slope_7","news_ratio_7_30",
        ]
    ].head(20)
)

### 5. Persistence Group 정의

`high_share_7`은 0~1 비율이다.

발표용 그룹은 결과를 본 뒤 최적화하지 않고 고정한다.

- **Low:** `high_share_7 <= 2/7`
- **Mid:** `2/7 < high_share_7 <= 4/7`
- **High:** `high_share_7 > 4/7`

Main 통계검정에서는 그룹값보다 연속형 `high_share_7`을 우선 사용한다.

In [ ]:
def persistence_group(v):
    if pd.isna(v):
        return "Missing"
    if v <= 2/7:
        return "Low (<=2/7)"
    if v <= 4/7:
        return "Mid (3-4/7)"
    return "High (>=5/7)"

for source in ["search","news"]:
    feature[f"{source}_persistence_group"] = (
        feature[f"{source}_high_share_7"].map(persistence_group)
    )

print("Search persistence groups")
display(feature["search_persistence_group"].value_counts(dropna=False).to_frame("N"))

print("News persistence groups")
display(feature["news_persistence_group"].value_counts(dropna=False).to_frame("N"))

persistence_distribution = pd.DataFrame({
    "search_high_share_7": feature["search_high_share_7"].describe(),
    "news_high_share_7": feature["news_high_share_7"].describe(),
})
display(persistence_distribution)

### 6. T+0 H10/H20/H40 Target

In [ ]:
target = targets_raw.loc[
    targets_raw["prediction_offset"].eq(0)
    & targets_raw["target_horizon"].isin(CFG.horizons)
].copy()

target = target.rename(columns={
    "target_horizon":"horizon",
    "future_peak_bhar":"peak_return",
    "future_end_bhar":"final_return",
    "future_giveback_ratio":"giveback",
    "future_true_mdd":"mdd",
})

OUTCOMES = ["peak_return","final_return","giveback","mdd"]
MAIN_OUTCOMES = ["final_return","giveback","mdd"]

for c in OUTCOMES + ["binary_label"]:
    target[c] = pd.to_numeric(target[c], errors="coerce")
target["horizon"] = pd.to_numeric(target["horizon"], errors="coerce").astype("Int64")

assert not target.duplicated(["event_id","horizon"]).any()
assert set(target["horizon"].dropna().astype(int).unique()) == set(CFG.horizons)

print("target:", target.shape)
display(target["horizon"].value_counts().sort_index().to_frame("N"))

### 7. Analysis Mart 생성

In [ ]:
analysis = feature.merge(
    target[
        [
            "event_id","ticker","event_date","horizon",
            "peak_return","final_return","giveback","mdd",
            "label_class","binary_label",
        ]
    ],
    on=["event_id","ticker","event_date"],
    how="inner",
    validate="one_to_many",
)

analysis["event_year"] = analysis["event_date"].dt.year.astype(int)

assert len(analysis) > 0
assert not analysis.duplicated(["event_id","horizon"]).any()

h_count = analysis.groupby("event_id")["horizon"].nunique()
common_ids = h_count[h_count.eq(len(CFG.horizons))].index
analysis_common = analysis[analysis["event_id"].isin(common_ids)].copy()

print("analysis:", analysis.shape)
print("event N :", analysis["event_id"].nunique())
print("common event N:", analysis_common["event_id"].nunique())
display(analysis["horizon"].value_counts().sort_index().to_frame("N"))

### 8. H2 분석 표본 정의

두 표본을 구분합니다.

#### A. All Events
지속성 지표 자체가 미래 경로와 관련되는지 확인.

#### B. Lead-conditioned (Main H2)
해당 source에서 **0~20 calendar-day lead signal이 실제 존재한 사건만** 사용.

이 표본이 H2의 핵심이다.

> “먼저 나타난 관심 중에서, T 직전까지 높은 상태를 오래 유지한 경우가 더 나은가?”

In [ ]:
sample_rows = []

for source in ["search","news"]:
    high = f"{source}_high_share_7"
    lead = f"{source}_lead_recent20"

    for sample_name, mask in [
        ("all_events", pd.Series(True, index=analysis.index)),
        ("lead_conditioned", analysis[lead].notna()),
    ]:
        d = analysis.loc[mask]
        sample_rows.append({
            "source":source,
            "sample":sample_name,
            "rows":len(d),
            "events":d["event_id"].nunique(),
            "high_share_nonnull":int(d[high].notna().sum()),
            "lead_nonnull":int(d[lead].notna().sum()),
        })

sample_qa = pd.DataFrame(sample_rows)
display(sample_qa)

### 9. High-share 연속형 Spearman + Holm 보정

In [ ]:
spearman_rows = []

for source in ["search","news"]:
    high = f"{source}_high_share_7"
    lead = f"{source}_lead_recent20"

    for sample_name, mask in [
        ("all_events", pd.Series(True, index=analysis.index)),
        ("lead_conditioned", analysis[lead].notna()),
    ]:
        for h in CFG.horizons:
            gh = analysis.loc[mask & analysis["horizon"].eq(h)]

            # Main H2 high_share
            for outcome in OUTCOMES:
                z = gh[[high,outcome]].dropna()
                if len(z) >= 20 and z[high].nunique() > 1 and z[outcome].nunique() > 1:
                    rho, p = stats.spearmanr(z[high], z[outcome])
                    spearman_rows.append({
                        "source":source,
                        "sample":sample_name,
                        "feature":"high_share_7",
                        "horizon":h,
                        "outcome":outcome,
                        "n":len(z),
                        "rho":rho,
                        "p_value":p,
                    })

            # Secondary indicators
            for feature_suffix in ["slope_7","ratio_7_30"]:
                feat = f"{source}_{feature_suffix}"
                for outcome in MAIN_OUTCOMES:
                    z = gh[[feat,outcome]].replace([np.inf,-np.inf],np.nan).dropna()
                    if len(z) >= 20 and z[feat].nunique() > 1 and z[outcome].nunique() > 1:
                        rho, p = stats.spearmanr(z[feat], z[outcome])
                        spearman_rows.append({
                            "source":source,
                            "sample":sample_name,
                            "feature":feature_suffix,
                            "horizon":h,
                            "outcome":outcome,
                            "n":len(z),
                            "rho":rho,
                            "p_value":p,
                        })

spearman_results = pd.DataFrame(spearman_rows)

# Holm은 source × sample × feature family 안에서 적용
spearman_results["p_holm"] = np.nan
for keys, idx in spearman_results.groupby(["source","sample","feature"]).groups.items():
    p = spearman_results.loc[idx, "p_value"].to_numpy()
    spearman_results.loc[idx, "p_holm"] = multipletests(p, method="holm")[1]

primary_spearman = spearman_results[
    (spearman_results["feature"] == "high_share_7")
    & (spearman_results["sample"] == "lead_conditioned")
].copy()

print("=== MAIN H2: lead-conditioned high_share_7 ===")
display(primary_spearman.sort_values(["source","outcome","horizon"]))

print("=== 전체 Spearman ===")
display(spearman_results.sort_values(["source","sample","feature","outcome","horizon"]))

### 10. Persistence Group × Outcome

In [ ]:
PERSIST_ORDER = ["Low (<=2/7)", "Mid (3-4/7)", "High (>=5/7)"]

def group_outcome_summary(df, source, lead_conditioned=True):
    group = f"{source}_persistence_group"
    lead = f"{source}_lead_recent20"

    d = df[df[lead].notna()].copy() if lead_conditioned else df.copy()
    d = d[d[group].isin(PERSIST_ORDER)]

    out = (
        d.groupby(["horizon",group], observed=True)[OUTCOMES]
        .agg(["count","mean","median"])
    )
    out.columns = [f"{a}_{b}" for a,b in out.columns]
    return out.reset_index()

search_group = group_outcome_summary(analysis, "search", True)
news_group = group_outcome_summary(analysis, "news", True)

display(search_group)
display(news_group)

### 11. Kruskal-Wallis — Persistence Group

In [ ]:
kruskal_rows = []

for source in ["search","news"]:
    group = f"{source}_persistence_group"
    lead = f"{source}_lead_recent20"

    for sample_name, mask in [
        ("all_events", pd.Series(True, index=analysis.index)),
        ("lead_conditioned", analysis[lead].notna()),
    ]:
        for h in CFG.horizons:
            gh = analysis.loc[mask & analysis["horizon"].eq(h)]

            for outcome in MAIN_OUTCOMES:
                arrays = []
                used = []
                excluded = []

                for gname in PERSIST_ORDER:
                    x = gh.loc[gh[group].eq(gname), outcome].dropna().to_numpy()
                    if len(x) >= CFG.min_group_n:
                        arrays.append(x)
                        used.append(f"{gname}(n={len(x)})")
                    else:
                        excluded.append(f"{gname}(n={len(x)})")

                if len(arrays) >= 2:
                    stat, p = stats.kruskal(*arrays)
                    kruskal_rows.append({
                        "source":source,
                        "sample":sample_name,
                        "horizon":h,
                        "outcome":outcome,
                        "H_stat":stat,
                        "p_value":p,
                        "groups_used":" | ".join(used),
                        "groups_excluded":" | ".join(excluded),
                    })

kruskal_results = pd.DataFrame(kruskal_rows)
kruskal_results["p_holm"] = np.nan

for keys, idx in kruskal_results.groupby(["source","sample"]).groups.items():
    p = kruskal_results.loc[idx, "p_value"].to_numpy()
    kruskal_results.loc[idx, "p_holm"] = multipletests(p, method="holm")[1]

display(kruskal_results)

### 12. Persistence Group × label_class

In [ ]:
def label_share(df, source):
    group = f"{source}_persistence_group"
    lead = f"{source}_lead_recent20"

    d = df[df[lead].notna() & df[group].isin(PERSIST_ORDER)].copy()

    x = (
        d.groupby(["horizon",group,"label_class"], observed=True)
        .size()
        .rename("N")
        .reset_index()
    )
    x["share"] = x["N"] / x.groupby(["horizon",group])["N"].transform("sum")
    return x

search_label = label_share(analysis, "search")
news_label = label_share(analysis, "news")

display(search_label)
display(news_label)

### 13. 조정 OLS — high_share main effect

통제:
- `ret_20d_T`
- `volume_ratio_prior20_T`
- `volatility_20d_T`
- 로그 시가총액
- 외국인/기관/개인 순매수 비율
- 시장 변동성 / 하락종목 비율
- 연도
- ticker cluster-robust SE

Main H2는 lead-conditioned sample에서:

`outcome ~ high_share_7 + lead_days + controls + year`

를 본다.

`high_share_7` 계수는 0→1 전체 범위 효과이므로, 결과표에는 **0.1(10%p) 증가당 효과**도 함께 저장한다.

In [ ]:
analysis_reg = analysis.copy()

mcap = pd.to_numeric(analysis_reg["market_cap_T"], errors="coerce")
analysis_reg["log_market_cap_T"] = np.log(mcap.where(mcap > 0))

CONTROL_CANDIDATES = [
    "ret_20d_T",
    "volume_ratio_prior20_T",
    "volatility_20d_T",
    "log_market_cap_T",
    "foreign_net_buy_to_mcap_T",
    "institution_net_buy_to_mcap_T",
    "individual_net_buy_to_mcap_T",
    "market_volatility_20d_cutoff",
    "market_decline_ratio_cutoff",
]

controls = []
for c in CONTROL_CANDIDATES:
    if c in analysis_reg.columns:
        analysis_reg[c] = pd.to_numeric(analysis_reg[c], errors="coerce")
        if analysis_reg[c].notna().mean() >= 0.60:
            controls.append(c)

print("controls:", controls)

def fit_cluster_ols(df, source, outcome, interaction=False):
    high = f"{source}_high_share_7"
    lead = f"{source}_lead_recent20"

    cols = ["ticker","event_year",high,lead,outcome] + controls
    z = df[cols].replace([np.inf,-np.inf],np.nan).dropna().copy()

    if len(z) < CFG.min_reg_n:
        return {"status":"SKIPPED","reason":f"N<{CFG.min_reg_n}","N":len(z)}

    # Centering
    z["high_c"] = z[high] - z[high].mean()
    z["lead_c"] = z[lead] - z[lead].mean()

    X_cols = ["high_c","lead_c"] + controls

    if interaction:
        z["high_x_lead"] = z["high_c"] * z["lead_c"]
        X_cols.append("high_x_lead")

    X = z[X_cols].astype(float).reset_index(drop=True)
    year_dummies = pd.get_dummies(
        z["event_year"].astype(str),
        prefix="year",
        drop_first=True,
        dtype=float,
    ).reset_index(drop=True)

    X = pd.concat([X, year_dummies], axis=1)
    X = sm.add_constant(X, has_constant="add")
    y = z[outcome].astype(float).reset_index(drop=True)

    # constant columns 제거
    keep = [c for c in X.columns if c=="const" or X[c].nunique(dropna=False) > 1]
    X = X[keep]

    try:
        model = sm.OLS(y, X)
        if z["ticker"].nunique() >= 10:
            res = model.fit(
                cov_type="cluster",
                cov_kwds={"groups":z["ticker"].reset_index(drop=True)},
            )
            cov = "cluster(ticker)"
        else:
            res = model.fit(cov_type="HC3")
            cov = "HC3"

        ci = res.conf_int()

        out = {
            "status":"PASS",
            "reason":"",
            "N":len(z),
            "tickers":z["ticker"].nunique(),
            "cov_type":cov,
            "high_share_coef":float(res.params.get("high_c", np.nan)),
            "high_share_p":float(res.pvalues.get("high_c", np.nan)),
            "high_share_ci_low":float(ci.loc["high_c",0]) if "high_c" in ci.index else np.nan,
            "high_share_ci_high":float(ci.loc["high_c",1]) if "high_c" in ci.index else np.nan,
            "effect_per_0p1_high_share":float(res.params.get("high_c", np.nan) * 0.1),
            "lead_coef":float(res.params.get("lead_c", np.nan)),
            "lead_p":float(res.pvalues.get("lead_c", np.nan)),
            "r2":float(res.rsquared),
        }

        if interaction:
            out.update({
                "interaction_coef":float(res.params.get("high_x_lead", np.nan)),
                "interaction_p":float(res.pvalues.get("high_x_lead", np.nan)),
            })

        return out

    except Exception as e:
        return {
            "status":"ERROR",
            "reason":f"{type(e).__name__}: {e}",
            "N":len(z),
        }

ols_rows = []
interaction_rows = []

for h in CFG.horizons:
    gh = analysis_reg[analysis_reg["horizon"].eq(h)]

    for source in ["search","news"]:
        # lead-conditioned sample; fit 함수가 high/lead 모두 dropna
        for outcome in MAIN_OUTCOMES:
            r = fit_cluster_ols(gh, source, outcome, interaction=False)
            r.update({"horizon":h,"source":source,"outcome":outcome})
            ols_rows.append(r)

            ri = fit_cluster_ols(gh, source, outcome, interaction=True)
            ri.update({"horizon":h,"source":source,"outcome":outcome})
            interaction_rows.append(ri)

adjusted_persistence_ols = pd.DataFrame(ols_rows)
lead_x_persistence_ols = pd.DataFrame(interaction_rows)

print("=== Main adjusted persistence OLS ===")
display(adjusted_persistence_ols)

print("=== Lead × persistence interaction OLS ===")
display(lead_x_persistence_ols)

### 14. BHAR path — Persistence Group

In [ ]:
path_t0 = bhar_path.loc[
    bhar_path["prediction_offset"].eq(0)
    & bhar_path["future_step"].between(1, max(CFG.horizons))
].copy()

path_t0 = path_t0.merge(
    feature[
        [
            "event_id",
            "search_lead_recent20","search_persistence_group",
            "news_lead_recent20","news_persistence_group",
        ]
    ],
    on="event_id",
    how="inner",
    validate="many_to_one",
)

def median_path(source):
    lead = f"{source}_lead_recent20"
    group = f"{source}_persistence_group"

    d = path_t0[
        path_t0[lead].notna()
        & path_t0[group].isin(PERSIST_ORDER)
    ].copy()

    return (
        d.groupby([group,"future_step"], observed=True)
        .agg(median_bhar=("bhar","median"), n=("bhar","count"))
        .reset_index()
    )

search_path = median_path("search")
news_path = median_path("news")

for source, tab in [("search",search_path),("news",news_path)]:
    group = f"{source}_persistence_group"

    fig, ax = plt.subplots(figsize=(9,5))
    for gname in PERSIST_ORDER:
        z = tab[tab[group].eq(gname)]
        if len(z):
            ax.plot(z["future_step"], z["median_bhar"], label=gname)

    ax.axhline(0, linewidth=.8)
    ax.set_title(f"Median BHAR path by {source} persistence")
    ax.set_xlabel("Trading steps after event T")
    ax.set_ylabel("Median BHAR")
    ax.legend()
    fig.tight_layout()
    fig.savefig(
        LOCAL/"figures"/f"{source}_persistence_median_bhar.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()

### 15. Common Sample 강건성

In [ ]:
common_rows = []

for source in ["search","news"]:
    high = f"{source}_high_share_7"
    lead = f"{source}_lead_recent20"

    for h in CFG.horizons:
        gh = analysis_common[
            analysis_common["horizon"].eq(h)
            & analysis_common[lead].notna()
        ]

        for outcome in OUTCOMES:
            z = gh[[high,outcome]].dropna()
            if len(z) >= 20 and z[high].nunique() > 1 and z[outcome].nunique() > 1:
                rho, p = stats.spearmanr(z[high], z[outcome])
                common_rows.append({
                    "source":source,
                    "horizon":h,
                    "outcome":outcome,
                    "n":len(z),
                    "rho":rho,
                    "p_value":p,
                })

common_spearman = pd.DataFrame(common_rows)
display(common_spearman)

### 16. 결과 저장 / GCS 업로드

In [ ]:
ARTIFACTS = {
    "persistence_feature_t0.parquet": feature,
    "analysis_event_horizon.parquet": analysis,
    "analysis_common_sample.parquet": analysis_common,
    "sample_qa.csv": sample_qa,
    "persistence_distribution.csv": persistence_distribution,
    "primary_spearman.csv": primary_spearman,
    "all_spearman.csv": spearman_results,
    "search_persistence_group_outcomes.csv": search_group,
    "news_persistence_group_outcomes.csv": news_group,
    "kruskal_results.csv": kruskal_results,
    "search_label_summary.csv": search_label,
    "news_label_summary.csv": news_label,
    "adjusted_persistence_ols.csv": adjusted_persistence_ols,
    "lead_x_persistence_ols.csv": lead_x_persistence_ols,
    "common_sample_spearman.csv": common_spearman,
    "search_persistence_bhar_path.csv": search_path,
    "news_persistence_bhar_path.csv": news_path,
}

def upload_file(local_path, gcs_path):
    # 심사용 실행에서는 팀 GCS에 쓰지 않고 로컬 저장만 확인합니다.
    print("로컬 저장 완료(GCS 업로드 생략):", local_path)

for name, df in ARTIFACTS.items():
    p = LOCAL / name
    if name.endswith(".parquet"):
        df.to_parquet(p, index=False)
    else:
        df.to_csv(p, index=False, encoding="utf-8-sig")
    upload_file(p, f"{CFG.output_prefix}/{name}")

for p in sorted((LOCAL/"figures").glob("*.png")):
    upload_file(p, f"{CFG.output_prefix}/figures/{p.name}")

meta = {
    "config":asdict(CFG),
    "sources":PATHS,
    "primary_hypothesis":"lead-conditioned high_share_7 -> H10/H20/H40 future path",
    "persistence_group":{
        "low":"<= 2/7",
        "mid":"2/7 < share <= 4/7",
        "high":"> 4/7",
    },
    "leakage_policy":"T 이후 attention feature 사용 안 함",
}

meta_path = LOCAL / "analysis_metadata.json"
meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
upload_file(meta_path, f"{CFG.output_prefix}/analysis_metadata.json")

UPLOAD_DONE = True
print("✅ 결과 저장 완료")

### 17. Final QA

In [ ]:
checks = []

def ck(name, condition, detail=""):
    checks.append({
        "check":name,
        "status":"PASS" if bool(condition) else "FAIL",
        "detail":detail,
    })

ck("attention load", len(attention)>0, str(attention.shape))
ck("T+0 snapshot", len(snapshot_t0)>0 and snapshot_t0["event_id"].is_unique, str(snapshot_t0.shape))
ck("event attention join", len(feature)==len(snapshot_t0), str(feature.shape))
ck("persistence range search",
   feature["search_high_share_7"].dropna().between(0,1).all())
ck("persistence range news",
   feature["news_high_share_7"].dropna().between(0,1).all())
ck("H10/H20/H40 target",
   set(target["horizon"].dropna().astype(int).unique())==set(CFG.horizons))
ck("analysis mart", len(analysis)>0, str(analysis.shape))
ck("primary H2 Spearman", not primary_spearman.empty, f"rows={len(primary_spearman)}")
ck("Kruskal", not kruskal_results.empty, f"rows={len(kruskal_results)}")
ck("Adjusted OLS", len(adjusted_persistence_ols)>0,
   adjusted_persistence_ols["status"].value_counts(dropna=False).to_dict())
ck("Interaction OLS", len(lead_x_persistence_ols)>0,
   lead_x_persistence_ols["status"].value_counts(dropna=False).to_dict())
ck("BHAR paths", len(search_path)>0 and len(news_path)>0)
ck("GCS output", globals().get("UPLOAD_DONE",False),
   f"gs://{CFG.bucket}/{CFG.output_prefix}/")

final_qa = pd.DataFrame(checks)
display(final_qa)

failed = final_qa[final_qa["status"].eq("FAIL")]
if len(failed):
    raise RuntimeError(
        "FINAL QA FAIL:\n" + failed[["check","detail"]].to_string(index=False)
    )

print("✅ FINAL QA — structural FAIL 없음")

### 18. H2 해석 순서

#### Main
1. `primary_spearman.csv`에서 **lead-conditioned high_share_7** 방향을 본다.
2. `final_return ↑`, `giveback ↓`, `mdd ↑(덜 음수)` 방향이 H20/H40에서 반복되는지 확인한다.
3. Holm 보정 후에도 남는지 본다.
4. Persistence Group의 median outcome과 BHAR path가 같은 방향인지 확인한다.
5. 조정 OLS에서 `high_share_7` 효과가 유지되는지 본다.

#### Lead × Persistence
`interaction_p`가 유의하고 방향이 경제적으로 일관되면:
> “관심이 더 일찍 시작되었을 때, 높은 관심이 유지되는 정도에 따라 이후 경로가 달라지는 관계가 관찰됐다.”

#### 반대 결과
high-share가 높을수록 give-back이 커지거나 final return이 낮다면:
> “관심의 장기 지속은 정보 신호라기보다 이미 과열된 대중 관심의 누적일 수 있다.”

#### 금지
- 인과관계 주장
- 결과를 보고 persistence group threshold 변경
- T 이후 attention을 설명변수에 추가

# STAGE 7: 3분류(관망형/단기과열형/지속상승형) 라벨링 시도

**태그:** 폐기됨

**원본 노트북:** `3class_분류_시도.ipynb`

## KOSPI200 급등 후 3분류(관망형/단기과열형/지속상승형) 라벨링 — 통합 재실행본

CSV 업로드 없이 **GCS(`gs://finda_project/curated/`)만 사용**합니다.

### 이 노트북에서 재현하는 방법론
1. BHAR/MDD 절대임계값
2. Overheat Ratio (v1 / v2 tertile, extreme 25·75분위)
3. CAR Slope / ZCAR (Z=1.5 표준화 초과수익)
4. 비지도 클러스터링 (KMeans)
5. Mann-Kendall 추세검정

업종대비 상대라벨은 GCS curated에 업종지수 데이터가 없어 이번 분석에서 제외합니다.

각 라벨에 대해 **Logistic / RandomForest / XGBoost**를 walk-forward로 비교하고,
마지막에 SMOTE + class_weight + 임계값조정을 적용합니다.

공시(DART) 데이터 관련 실험은 이번 정리에서 제외합니다.

### STEP 0: 환경 설정 + 데이터 로드 (GCS only)

In [ ]:
import pandas as pd
import numpy as np
import os
import subprocess
from google.colab import auth
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, f1_score, confusion_matrix
from xgboost import XGBClassifier
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

SAVE_DIR = '/content/kospi200_project'  # 로컬 저장 (팀 GCS/Drive에 쓰지 않음)
os.makedirs(SAVE_DIR, exist_ok=True)

# ── 가격 데이터 ──
# (재사용) STAGE 3에서 이미 받아온 stock_daily / attention_daily를 그대로 사용합니다.
price_all = stock_daily.copy()
price_all = price_all.rename(columns={'ticker': '종목코드', 'date': '날짜'})
price_all['날짜'] = pd.to_datetime(price_all['날짜'])
price_all_clean = price_all[price_all['종목코드'] != '001260'].copy()
print(f"price_all_clean: {price_all_clean.shape}, 종목수: {price_all_clean['종목코드'].nunique()}")

# ── 관심지표 데이터 (파생 9개 Feature, 재사용: STAGE 3의 attention_daily) ──
attention_df = attention_daily.copy()
attention_df = attention_df.rename(columns={'ticker': '종목코드', 'date': '날짜'})
attention_df['날짜'] = pd.to_datetime(attention_df['날짜'])
attention_df = attention_df[attention_df['종목코드'] != '001260'].copy()
print(f"attention_df: {attention_df.shape}, 종목수: {attention_df['종목코드'].nunique()}")

### STEP 1: 시장지수(시가총액가중) + 종목별 기본 피처

In [ ]:
price = price_all_clean.sort_values(['종목코드', '날짜']).reset_index(drop=True)

price_wide_close = price.pivot(index='날짜', columns='종목코드', values='close')
ret_wide = price_wide_close.pct_change(fill_method=None)
mktcap_wide = price.pivot(index='날짜', columns='종목코드', values='market_cap').shift(1)
weights = mktcap_wide.div(mktcap_wide.sum(axis=1), axis=0)
market_daily_return = (ret_wide * weights).sum(axis=1)

market_index = pd.DataFrame({'날짜': market_daily_return.index})
market_index['시장일수익률'] = market_daily_return.values
market_index = market_index.sort_values('날짜').reset_index(drop=True)
market_index['시장지수'] = (1 + market_index['시장일수익률'].fillna(0)).cumprod() * 100
market_idx_series = market_index.set_index('날짜')['시장지수']
print(f"market_index: {market_index.shape}")

df = price.merge(market_index[['날짜', '시장일수익률', '시장지수']], on='날짜', how='left')
df['일수익률'] = df.groupby('종목코드')['close'].pct_change(fill_method=None)
g = df.groupby('종목코드', group_keys=False)

df['로그시가총액'] = np.log(df['market_cap'].replace(0, np.nan))
df['거래량_20일평균'] = g['volume'].apply(lambda x: x.rolling(20).mean())
df['거래량증가율_20일평균대비'] = df['volume'] / df['거래량_20일평균']
df['수익률_20일'] = g['close'].apply(lambda x: x.pct_change(20, fill_method=None))
df['5일_변동성'] = g['일수익률'].apply(lambda x: x.rolling(5).std())
df['60일_변동성'] = g['일수익률'].apply(lambda x: x.rolling(60).std())
df['52주최고가'] = g['close'].apply(lambda x: x.rolling(252, min_periods=60).max())
df['high52w_gap'] = df['close'] / df['52주최고가'] - 1
df['외국인_보유비중_레벨'] = df['foreign_ownership_pct']
df['개인_외국인_반대'] = np.sign(df['individual_net_buy_value']) * np.sign(df['foreign_net_buy_value']) < 0

print(f"STEP1 후 df: {df.shape}")

### STEP 2: 급등 Event 정의

- 최근 20거래일 수익률 ≥ 20%
- 당일 거래량 / 직전 20거래일 평균 거래량 ≥ 1.5
- 해당일 실제 KOSPI200 구성종목 (GCS curated universe 자체가 Historical KOSPI200 기준이므로 별도 필터 불요)
- 동일 종목에서 사건 발생 후 20거래일 cooldown

In [ ]:
df = df.sort_values(['종목코드', '날짜']).reset_index(drop=True)
g = df.groupby('종목코드', group_keys=False)

df['조건_수익률'] = df['수익률_20일'] >= 0.20
df['조건_거래량'] = df['거래량증가율_20일평균대비'] >= 1.5
df['급등조건충족'] = df['조건_수익률'] & df['조건_거래량']
print(f"급등조건 충족 거래일 수: {df['급등조건충족'].sum()}")

def find_event_days_cooldown(group, cooldown=20):
    cond = group['급등조건충족'].values
    flag = np.zeros(len(cond), dtype=bool)
    prev = False
    last_kept = -np.inf
    for i in range(len(cond)):
        if cond[i] and not prev:
            if i - last_kept >= cooldown:
                flag[i] = True
                last_kept = i
        prev = cond[i]
    return flag

df['사건일여부'] = g.apply(
    lambda x: pd.Series(find_event_days_cooldown(x), index=x.index),
    include_groups=False
).values

events = df[df['사건일여부']].copy().reset_index(drop=True)
print(f"탐지된 사건 수: {len(events)}, 종목 수: {events['종목코드'].nunique()}")

ticker_groups = {t: gdf.reset_index(drop=True) for t, gdf in df.groupby('종목코드')}

def get_pos(code_, date):
    gdf = ticker_groups.get(code_)
    if gdf is None:
        return None, None
    arr = gdf.index[gdf['날짜'] == date]
    return gdf, (arr[0] if len(arr) > 0 else None)

### STEP 3: BHAR 경로(path) 계산 — 여러 라벨 방식이 공유하는 공통 함수

`checkpoints`(5,10,15,20,30,40,60거래일)마다 시장조정수익률(BHAR)을 계산해
peak/final/MDD/slope/Mann-Kendall 등 여러 라벨링 방식에서 재사용합니다.

In [ ]:
CHECKPOINTS = (5, 10, 15, 20, 30, 40, 60)

def compute_bhar_path(code_, event_date, checkpoints=CHECKPOINTS):
    gdf, pos0 = get_pos(code_, event_date)
    if pos0 is None:
        return None
    close0 = gdf.loc[pos0, 'close']
    mkt0 = market_idx_series.get(event_date)
    if mkt0 is None:
        return None
    path = {}
    for h in checkpoints:
        pos_h = pos0 + h
        if pos_h >= len(gdf):
            continue
        stock_ret = gdf.loc[pos_h, 'close'] / close0 - 1
        mkt_date = gdf.loc[pos_h, '날짜']
        mkt_h = market_idx_series.get(mkt_date)
        if mkt_h is None:
            continue
        path[h] = stock_ret - (mkt_h / mkt0 - 1)
    return path if len(path) > 0 else None

def compute_bhar_mdd(row, horizons=(20, 60)):
    code_, event_date = row['종목코드'], row['날짜']
    gdf, pos0 = get_pos(code_, event_date)
    result = {f'BHAR{h}': np.nan for h in horizons}
    result['MDD20'] = np.nan
    if pos0 is None:
        return pd.Series(result)
    close0 = gdf.loc[pos0, 'close']
    mkt0 = market_idx_series.get(event_date)
    for h in horizons:
        pos_h = pos0 + h
        if pos_h >= len(gdf) or mkt0 is None:
            continue
        mkt_h = market_idx_series.get(gdf.loc[pos_h, '날짜'])
        if mkt_h is None:
            continue
        stock_ret = gdf.loc[pos_h, 'close'] / close0 - 1
        result[f'BHAR{h}'] = stock_ret - (mkt_h / mkt0 - 1)
    pos_end20 = min(pos0 + 20, len(gdf) - 1)
    if pos_end20 > pos0 and mkt0 is not None:
        window = gdf.loc[pos0:pos_end20].copy()
        window['stock_cum'] = window['close'] / close0 - 1
        window['mkt_cum'] = window['날짜'].map(market_idx_series) / mkt0 - 1
        window['bhar_path'] = window['stock_cum'] - window['mkt_cum']
        result['MDD20'] = (window['bhar_path'] - window['bhar_path'].cummax()).min()
    return pd.Series(result)

bhar_results = events.apply(compute_bhar_mdd, axis=1)
events = pd.concat([events, bhar_results], axis=1)

path_dicts = events.apply(lambda r: compute_bhar_path(r['종목코드'], r['날짜']), axis=1)
events['bhar_path'] = path_dicts
print(f"BHAR/MDD + path 계산 완료: {events.shape}")
print(f"BHAR60 결측(우측절단) 비율: {events['BHAR60'].isna().mean():.3f}")

### STEP 4: 추가 피처 (리드타임, 과거 급등이력, 관심지표 등)

`search_news_lead_diff`(검색·뉴스 리드타임 차이), `prior_surge_count`(과거 급등이력 횟수)를 계산합니다.
관심지표 상위10% 신규진입 플래그는 GCS `attention_daily`의 `search_pct_50`/`news_pct_50` ≥ 0.90 기준으로 정의합니다.

In [ ]:
attention_sorted = attention_df.sort_values(['종목코드', '날짜']).reset_index(drop=True)
attention_sorted['검색_상위10pct'] = attention_sorted['search_pct_50'] >= 0.90
attention_sorted['뉴스_상위10pct'] = attention_sorted['news_pct_50'] >= 0.90
attention_sorted['검색_신규진입'] = attention_sorted['검색_상위10pct'] & (
    ~attention_sorted.groupby('종목코드')['검색_상위10pct'].shift(1).fillna(False))
attention_sorted['뉴스_신규진입'] = attention_sorted['뉴스_상위10pct'] & (
    ~attention_sorted.groupby('종목코드')['뉴스_상위10pct'].shift(1).fillna(False))

search_groups = {t: gdf.reset_index(drop=True) for t, gdf in attention_sorted.groupby('종목코드')}

def find_lead_days(code_, event_date, flag_col, window_days=28):
    gdf = search_groups.get(code_)
    if gdf is None:
        return np.nan
    window_start = event_date - pd.Timedelta(days=window_days)
    cand = gdf[(gdf['날짜'] > window_start) & (gdf['날짜'] <= event_date) & (gdf[flag_col])]
    if len(cand) == 0:
        return np.nan
    return (event_date - cand['날짜'].min()).days

events = events.sort_values(['종목코드', '날짜']).reset_index(drop=True)
events['검색_리드타임'] = events.apply(lambda r: find_lead_days(r['종목코드'], r['날짜'], '검색_신규진입'), axis=1)
events['뉴스_리드타임'] = events.apply(lambda r: find_lead_days(r['종목코드'], r['날짜'], '뉴스_신규진입'), axis=1)
events['search_news_lead_diff'] = events['검색_리드타임'] - events['뉴스_리드타임']

prior_count = {}
prior_counts = []
for _, ev in events.iterrows():
    code_ = ev['종목코드']
    prior_count[code_] = prior_count.get(code_, 0)
    prior_counts.append(prior_count[code_])
    prior_count[code_] += 1
events['prior_surge_count'] = prior_counts

print(f"피처 추가 완료: {events.shape}")
events.to_pickle(f'{SAVE_DIR}/events_3class_base.pkl')

### STEP 5: 라벨 정의 — 방법론별로 분리

아래 5-1 ~ 5-6은 서로 독립적으로 실행 가능합니다. 각 셀은 `events` 위에 새 라벨 컬럼을 추가합니다.

#### 5-1. BHAR/MDD 절대임계값 (기본 + 완화 버전 + BHAR-only 버전)

In [ ]:
def classify_bhar_mdd(row, bhar_thresh=-0.05, mdd_thresh=-0.15, up_thresh=0.05):
    if pd.isna(row['BHAR20']) or pd.isna(row['MDD20']):
        return np.nan
    if row['BHAR20'] <= bhar_thresh or row['MDD20'] <= mdd_thresh:
        return '단기과열형'
    if pd.isna(row['BHAR60']):
        return np.nan
    if row['BHAR20'] >= up_thresh and row['BHAR60'] >= up_thresh:
        return '지속상승형'
    return '관망형'

def classify_bhar_only(row, bhar_thresh=-0.05, up_thresh=0.05):
    # MDD 조건 제거 버전
    if pd.isna(row['BHAR20']):
        return np.nan
    if row['BHAR20'] <= bhar_thresh:
        return '단기과열형'
    if pd.isna(row['BHAR60']):
        return np.nan
    if row['BHAR20'] >= up_thresh and row['BHAR60'] >= up_thresh:
        return '지속상승형'
    return '관망형'

events['label_bhar_mdd'] = events.apply(classify_bhar_mdd, axis=1)
events['label_bhar_mdd_relaxed'] = events.apply(lambda r: classify_bhar_mdd(r, bhar_thresh=-0.03), axis=1)
events['label_bhar_only'] = events.apply(classify_bhar_only, axis=1)

for col in ['label_bhar_mdd', 'label_bhar_mdd_relaxed', 'label_bhar_only']:
    print(f"--- {col} ---")
    print(events[col].value_counts())
    print()

#### 5-2. Overheat Ratio (v1 raw / v2 min_peak 필터 / extreme 25·75분위)

In [ ]:
def compute_overheat_ratio(code_, event_date, window=20):
    path = compute_bhar_path(code_, event_date, checkpoints=range(1, window + 1))
    if not path:
        return np.nan
    vals = np.array(list(path.values()))
    peak = vals.max()
    final = vals[-1]
    if peak <= 0:
        return np.nan
    giveback = max(peak - final, 0)
    return giveback / abs(peak) if peak != 0 else np.nan

def compute_overheat_ratio_v2(code_, event_date, window=20, min_peak=0.03):
    path = compute_bhar_path(code_, event_date, checkpoints=range(1, window + 1))
    if not path:
        return np.nan
    vals = np.array(list(path.values()))
    peak = vals.max()
    if peak < min_peak:
        return np.nan
    final = vals[-1]
    giveback = max(peak - final, 0)
    return giveback / peak

events['overheat_ratio'] = events.apply(lambda r: compute_overheat_ratio(r['종목코드'], r['날짜']), axis=1)
events['overheat_ratio_v2'] = events.apply(lambda r: compute_overheat_ratio_v2(r['종목코드'], r['날짜']), axis=1)

# v1: tertile(33/67)
valid1 = events['overheat_ratio'].dropna()
q33_1, q67_1 = valid1.quantile([0.33, 0.67])
def classify_tertile(r, q33, q67, col):
    v = r[col]
    if pd.isna(v):
        return np.nan
    if v >= q67:
        return '단기과열형'
    if v <= q33:
        return '지속상승형'
    return '관망형'
events['label_ratio_v1'] = events.apply(lambda r: classify_tertile(r, q33_1, q67_1, 'overheat_ratio'), axis=1)

# v2: tertile(33/67), min_peak 필터로 '미과열형' 자연 발생
valid2 = events['overheat_ratio_v2'].dropna()
q33_2, q67_2 = valid2.quantile([0.33, 0.67])
def classify_tertile_v2(r):
    v = r['overheat_ratio_v2']
    if pd.isna(v):
        return '미과열형'
    if v >= q67_2:
        return '단기과열형'
    if v <= q33_2:
        return '지속상승형'
    return '관망형'
events['label_ratio_v2'] = events.apply(classify_tertile_v2, axis=1)

# extreme 25/75분위 (미과열형 별도)
q25_2, q75_2 = valid2.quantile([0.25, 0.75])
def classify_extreme(r):
    v = r['overheat_ratio_v2']
    if pd.isna(v):
        return '미과열형'
    if v >= q75_2:
        return '단기과열형'
    if v <= q25_2:
        return '지속상승형'
    return '관망형'
events['label_extreme'] = events.apply(classify_extreme, axis=1)

for col in ['label_ratio_v1', 'label_ratio_v2', 'label_extreme']:
    print(f"--- {col} ---")
    print(events[col].value_counts())
    print()

#### 5-4. CAR Slope 회귀기울기 & ZCAR(Z=1.5 표준화)

In [ ]:
def compute_slope_label(code_, event_date, checkpoints=CHECKPOINTS):
    path = compute_bhar_path(code_, event_date, checkpoints)
    if not path or len(path) < 3:
        return np.nan, np.nan
    xs = np.array(list(path.keys()))
    ys = np.array(list(path.values()))
    slope, intercept, r, p, se = stats.linregress(xs, ys)
    return slope, ys[-1]

slope_results = events.apply(lambda r: compute_slope_label(r['종목코드'], r['날짜']), axis=1)
events['car_slope'] = slope_results.apply(lambda x: x[0])
events['car_final'] = slope_results.apply(lambda x: x[1])

valid_slope = events['car_slope'].dropna()
q33_sl, q67_sl = valid_slope.quantile([0.33, 0.67])
def classify_slope(v):
    if pd.isna(v):
        return np.nan
    if v <= q33_sl:
        return '단기과열형'
    if v >= q67_sl:
        return '지속상승형'
    return '관망형'
events['label_slope'] = events['car_slope'].apply(classify_slope)

# ── ZCAR: 전체 사건 CAR20 표준편차로 표준화 ──
Z_THRESHOLD = 1.5
car20 = events['BHAR20']
car20_std = car20.std()
events['zcar20'] = car20 / car20_std

def classify_zcar(v, z=Z_THRESHOLD):
    if pd.isna(v):
        return np.nan
    if v <= -z:
        return '단기과열형'
    if v >= z:
        return '지속상승형'
    return '관망형'
events['label_zcar'] = events['zcar20'].apply(classify_zcar)

print("--- label_slope ---")
print(events['label_slope'].value_counts())
print("\n--- label_zcar ---")
print(events['label_zcar'].value_counts())

#### 5-5. 비지도 클러스터링 (KMeans)

In [ ]:
def bhar_path_vector(code_, event_date, checkpoints=CHECKPOINTS):
    path = compute_bhar_path(code_, event_date, checkpoints)
    if not path or len(path) < len(checkpoints):
        return None
    return [path.get(h, np.nan) for h in checkpoints]

path_vectors = events.apply(lambda r: bhar_path_vector(r['종목코드'], r['날짜']), axis=1)
valid_mask = path_vectors.apply(lambda x: x is not None)
X_path = np.array([v for v in path_vectors[valid_mask]])

scaler_km = StandardScaler()
X_path_scaled = scaler_km.fit_transform(X_path)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_path_scaled)

# 클러스터 중심의 마지막 체크포인트 BHAR 순으로 정렬해 의미 부여
centers_final = kmeans.cluster_centers_[:, -1]
order = np.argsort(centers_final)  # 낮은 순 = 단기과열형, 높은 순 = 지속상승형
cluster_name_map = {order[0]: '단기과열형', order[1]: '관망형', order[2]: '지속상승형'}

events['label_cluster'] = np.nan
events.loc[valid_mask, 'label_cluster'] = pd.Series(cluster_labels, index=events.index[valid_mask]).map(cluster_name_map)

print(events['label_cluster'].value_counts())

#### 5-6. Mann-Kendall 추세검정

In [ ]:
!pip install pymannkendall --quiet
import pymannkendall as mk

def compute_mk_label(code_, event_date, checkpoints=CHECKPOINTS):
    path = compute_bhar_path(code_, event_date, checkpoints)
    if not path or len(path) < 3:
        return np.nan, np.nan, np.nan
    vals = list(path.values())
    result = mk.original_test(vals)
    return result.p, result.Tau, result.trend

mk_results = events.apply(lambda r: compute_mk_label(r['종목코드'], r['날짜']), axis=1)
events['mk_pval'] = mk_results.apply(lambda x: x[0])
events['mk_tau'] = mk_results.apply(lambda x: x[1])
events['mk_trend'] = mk_results.apply(lambda x: x[2])

def classify_mk(row):
    if pd.isna(row['mk_trend']):
        return np.nan
    if row['mk_trend'] == 'decreasing':
        return '단기과열형'
    if row['mk_trend'] == 'increasing':
        return '지속상승형'
    return '관망형'

events['label_mk'] = events.apply(classify_mk, axis=1)
print(events['mk_pval'].describe())
print()
print(events['label_mk'].value_counts())

events.to_pickle(f'{SAVE_DIR}/events_3class_all_labels.pkl')

### STEP 6: 공통 모델링 — Logistic / RandomForest / XGBoost, walk-forward 5-fold

아래 함수 하나로 위에서 만든 모든 라벨 컬럼을 동일한 조건으로 비교합니다.

In [ ]:
FEATURES_BASE = [
    '로그시가총액', '거래량증가율_20일평균대비', '수익률_20일',
    '5일_변동성', '60일_변동성', 'high52w_gap',
    '외국인_보유비중_레벨', '개인_외국인_반대',
    '검색_리드타임', '뉴스_리드타임', 'search_news_lead_diff', 'prior_surge_count',
]

def prepare_X(data, cols):
    X = data[cols].copy()
    X = X.replace([np.inf, -np.inf], np.nan)
    for col in cols:
        if X[col].dtype == bool:
            X[col] = X[col].astype(int)
        if X[col].isnull().any():
            X[col] = X[col].fillna(X[col].median())
    return X

def run_walkforward(data, label_col, features=FEATURES_BASE, n_folds=5, model_names=('logistic', 'rf', 'xgboost')):
    d = data.dropna(subset=[label_col]).sort_values('날짜').reset_index(drop=True)
    if d[label_col].nunique() < 2:
        print(f"{label_col}: 클래스가 2개 미만이라 스킵")
        return None
    le = LabelEncoder()
    d['label_enc'] = le.fit_transform(d[label_col])
    X = prepare_X(d, features)
    y = d['label_enc']
    fold_bounds = np.linspace(0, len(d), n_folds + 2).astype(int)

    results = {}
    for model_name in model_names:
        aucs, f1s = [], []
        all_true, all_pred = [], []
        for i in range(1, n_folds + 1):
            train_idx = d.index[:fold_bounds[i]]
            test_idx = d.index[fold_bounds[i]:fold_bounds[i + 1]]
            if len(test_idx) == 0 or y.loc[train_idx].nunique() < 2:
                continue
            X_train, y_train = X.loc[train_idx], y.loc[train_idx]
            X_test, y_test = X.loc[test_idx], y.loc[test_idx]

            if model_name == 'logistic':
                model = LogisticRegression(max_iter=1000, class_weight='balanced')
            elif model_name == 'rf':
                model = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight='balanced', random_state=42)
            else:
                model = XGBClassifier(objective='multi:softprob', num_class=y.nunique(),
                                       eval_metric='mlogloss', random_state=42, verbosity=0)
            model.fit(X_train, y_train)
            proba = model.predict_proba(X_test)
            pred = model.predict(X_test)
            try:
                auc = roc_auc_score(y_test, proba, multi_class='ovr', average='macro')
            except ValueError:
                auc = np.nan
            f1 = f1_score(y_test, pred, average='macro')
            aucs.append(auc)
            f1s.append(f1)
            all_true.extend(y_test.tolist())
            all_pred.extend(pred.tolist())

        results[model_name] = {
            'mean_auc': np.nanmean(aucs), 'mean_f1': np.nanmean(f1s),
            'all_true': all_true, 'all_pred': all_pred, 'classes': le.classes_
        }
    return results

LABEL_COLS = [
    'label_bhar_mdd', 'label_bhar_mdd_relaxed', 'label_bhar_only',
    'label_ratio_v1', 'label_ratio_v2', 'label_extreme',
    'label_slope', 'label_zcar', 'label_cluster', 'label_mk',
]

summary_rows = []
all_results = {}
for label_col in LABEL_COLS:
    if label_col not in events.columns:
        continue
    print(f"===== {label_col} =====")
    res = run_walkforward(events, label_col)
    if res is None:
        continue
    all_results[label_col] = res
    for model_name, r in res.items():
        summary_rows.append({
            '라벨': label_col, '모델': model_name,
            'macro_AUC': round(r['mean_auc'], 3), 'macro_F1': round(r['mean_f1'], 3)
        })
    print()

summary_df = pd.DataFrame(summary_rows)
print("=" * 60)
print("전체 요약")
print("=" * 60)
print(summary_df.sort_values('macro_AUC', ascending=False).to_string(index=False))
summary_df.to_csv(f'{SAVE_DIR}/label_comparison_summary.csv', index=False)

### STEP 7: 대표 라벨(BHAR/MDD 절대임계값) confusion matrix 상세 확인

In [ ]:
target_label = 'label_bhar_mdd'
if target_label in all_results:
    for model_name, r in all_results[target_label].items():
        cm = confusion_matrix(r['all_true'], r['all_pred'])
        cm_df = pd.DataFrame(cm, index=[f'실제_{c}' for c in r['classes']],
                              columns=[f'예측_{c}' for c in r['classes']])
        print(f"--- {target_label} / {model_name} ---")
        print(cm_df)
        print()

### STEP 8: SMOTE + class_weight + 임계값조정

대표 라벨(`label_bhar_mdd`)에 대해 클래스 불균형 처리 기법을 추가 적용합니다.

In [ ]:
from imblearn.over_sampling import SMOTE

target_label = 'label_bhar_mdd'
d = events.dropna(subset=[target_label]).sort_values('날짜').reset_index(drop=True)
le_final = LabelEncoder()
d['label_enc'] = le_final.fit_transform(d[target_label])
X = prepare_X(d, FEATURES_BASE)
y = d['label_enc']

n_folds = 5
fold_bounds = np.linspace(0, len(d), n_folds + 2).astype(int)

all_true, all_pred = [], []
for i in range(1, n_folds + 1):
    train_idx = d.index[:fold_bounds[i]]
    test_idx = d.index[fold_bounds[i]:fold_bounds[i + 1]]
    if len(test_idx) == 0 or y.loc[train_idx].nunique() < 2:
        continue
    X_train, y_train = X.loc[train_idx], y.loc[train_idx]
    X_test, y_test = X.loc[test_idx], y.loc[test_idx]

    min_c = y_train.value_counts().min()
    if min_c >= 6:
        sm = SMOTE(random_state=42, k_neighbors=min(5, min_c - 1))
        X_tr, y_tr = sm.fit_resample(X_train, y_train)
    else:
        X_tr, y_tr = X_train, y_train

    model = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight='balanced', random_state=42)
    model.fit(X_tr, y_tr)
    pred = model.predict(X_test)
    all_true.extend(y_test.tolist())
    all_pred.extend(pred.tolist())

print("=== Confusion Matrix (SMOTE + class_weight) ===")
cm = confusion_matrix(all_true, all_pred)
cm_df = pd.DataFrame(cm, index=[f'실제_{c}' for c in le_final.classes_],
                      columns=[f'예측_{c}' for c in le_final.classes_])
print(cm_df)

print("\n=== classification_report ===")
print(classification_report(all_true, all_pred, target_names=le_final.classes_))

### STEP 9: 최종 산출물 저장

In [ ]:
events.to_pickle(f'{SAVE_DIR}/events_3class_FINAL.pkl')
summary_df.to_csv(f'{SAVE_DIR}/label_comparison_summary_FINAL.csv', index=False)
print("저장 완료:")
print("1. events_3class_FINAL.pkl (전체 라벨 + 피처)")
print("2. label_comparison_summary_FINAL.csv (라벨×모델별 macro AUC/F1 요약)")

# STAGE 8: 매수/회피 모델 — Method 2: 이진화 점수모델(하락위험·상승지속)

**태그:** 폐기됨 ("절대적 매매 규칙 아님, 3분류 실패 이후 탐색 근거로만 사용")

**원본 노트북:** `회피_매수_모델.ipynb`

> GCS 공개 curated 데이터에서 사건·A/B/C/D 피처·라벨(`target_downside_path`/`target_persistence_path`, 60일 BHAR/MDD 경로 기반)을 전부 자체 계산하는 완결된 코드입니다. 외부 파일 의존이 없어 별도 수정 없이 바로 실행됩니다.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import requests
from io import BytesIO
import numpy as np
import pandas as pd
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

SAVE_OUTPUTS = True

BUCKET = "finda_project"
GCS_MARKET_KEY = "curated/market_daily.parquet"
GCS_STOCK_KEY = "curated/stock_daily.parquet"
GCS_MEMBERSHIP_KEY = "curated/membership_daily.parquet"
GCS_ATTENTION_RAW_KEY = "curated/attention_raw_daily.parquet"
GCS_ATTENTION_KEY = "curated/attention_daily.parquet"
GCS_SHORT_KEY = "curated/short_daily.parquet"

ROOT = WORKSPACE_ROOT / "buy_avoid_mixed_strategy"
OUTPUT_DIR = ROOT / "outputs"
if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVELOPMENT_END = pd.Timestamp("2024-12-31")
START_DATE = pd.Timestamp("2021-06-11")
END_DATE = pd.Timestamp("2026-06-11")
PRICE_THRESHOLD = 0.20
VOLUME_THRESHOLD = 1.50
COOLDOWN_TRADING_DAYS = 60
POSITIVE_RATIO_CUT = 0.60
TARGET_DOWNSIDE = "target_downside_path"
TARGET_PERSISTENCE = "target_persistence_path"
OBSERVABLE_FLAG = "경로60일관측가능"
RANK_PCTS = (0.10, 0.20, 0.30)
TOP_PCT = 0.20
RANDOM_STATE = 42

GROUP_A = [
    "price_volatility_20d", "price_return_5d", "price_return_20d",
    "daily_return", "volume_to_avg_20d", "volume_avg5_to_avg20",
    "stock_amihud_20d",
]
GROUP_B_ADDITIONS = [
    "search_to_avg20", "institution_net_buy_ratio_20d",
    "individual_net_buy_ratio_5d", "foreign_ownership_change_20d_pp",
    "foreign_net_buy_ratio_5d", "market_relative_pbr",
]
GROUP_C_ADDITIONS = [
    "market_drawdown_60d", "high_volatility_regime_probability",
]
GROUP_D_ADDITIONS = [
    "foreign_flow_transition_5d_vs_prev20d",
    "institution_flow_transition_5d_vs_prev20d",
    "search_accel_5d10d", "news_accel_5d10d",
    "short_balance_change_5d", "short_balance_change_20d",
    "short_value_to_avg20",
]
GROUPS = {
    "A_종목기본": GROUP_A,
    "B_수급검색가치": list(dict.fromkeys(GROUP_A + GROUP_B_ADDITIONS)),
    "C_시장보조추가": list(dict.fromkeys(GROUP_A + GROUP_B_ADDITIONS + GROUP_C_ADDITIONS)),
    "D_이탈가속공매도": list(dict.fromkeys(
        GROUP_A + GROUP_B_ADDITIONS + GROUP_C_ADDITIONS + GROUP_D_ADDITIONS
    )),
}


def read_public_gcs_parquet(key):
    """FinDA 공개 GCS Parquet을 디스크에 저장하지 않고 메모리에서 읽는다."""
    url = f"https://storage.googleapis.com/{BUCKET}/{key}"
    response = requests.get(url, timeout=180)
    response.raise_for_status()
    try:
        return pd.read_parquet(BytesIO(response.content))
    except ImportError as error:
        raise ImportError("Parquet을 읽으려면 pyarrow가 필요합니다: pip install pyarrow") from error


def rebuild_high_volatility_probability_from_gcs():
    """GCS market_proxy_return으로 2상태 Markov 고변동성 확률을 재계산한다."""
    market = read_public_gcs_parquet(GCS_MARKET_KEY).copy()
    if "date" not in market.columns or "market_proxy_return" not in market.columns:
        raise KeyError(
            "curated/market_daily.parquet에 date 또는 market_proxy_return 열이 없습니다."
        )
    market["date"] = pd.to_datetime(market["date"], errors="coerce").dt.normalize()
    market["market_proxy_return"] = pd.to_numeric(
        market["market_proxy_return"], errors="coerce"
    )
    market = market.sort_values("date").drop_duplicates("date", keep="last").reset_index(drop=True)

    valid = market["market_proxy_return"].notna()
    development = market["date"].le(DEVELOPMENT_END) & valid
    dev_returns = market.loc[development, "market_proxy_return"].astype(float)
    if len(dev_returns) < 300:
        raise ValueError(f"Markov 모형 학습 표본이 부족합니다: {len(dev_returns)}")

    dev_model = MarkovRegression(
        dev_returns, k_regimes=2, trend="c", switching_variance=True
    )
    fitted = dev_model.fit(
        disp=False,
        maxiter=1000,
        em_iter=20,
        search_reps=0,  # 무작위 시작점 탐색을 제거해 실행 결과를 고정
    )

    all_returns = market.loc[valid, "market_proxy_return"].astype(float)
    full_model = MarkovRegression(
        all_returns, k_regimes=2, trend="c", switching_variance=True
    )
    filtered = full_model.filter(fitted.params)
    probabilities = filtered.filtered_marginal_probabilities
    variances = {
        regime: float(fitted.params[f"sigma2[{regime}]"])
        for regime in range(2)
    }
    high_regime = max(variances, key=variances.get)
    market["high_volatility_regime_probability"] = np.nan
    market.loc[valid, "high_volatility_regime_probability"] = (
        probabilities[high_regime].to_numpy()
    )
    return market[["date", "high_volatility_regime_probability"]]


def _normalize_panel(frame):
    result = frame.copy()
    result["ticker"] = (
        result["ticker"].astype("string").str.replace(r"\.0$", "", regex=True).str.upper().str.zfill(6)
    )
    result["date"] = pd.to_datetime(result["date"], errors="coerce").dt.tz_localize(None).dt.normalize()
    return result.dropna(subset=["ticker", "date"]).sort_values(["ticker", "date"])


def _rolling(grouped, window, min_periods, how="mean", shift=0):
    def calculate(series):
        source = series.shift(shift) if shift else series
        rolling = source.rolling(window, min_periods=min_periods)
        return rolling.mean() if how == "mean" else rolling.sum()
    return grouped.transform(calculate)


def _drawdown(growth):
    path = np.r_[1.0, np.asarray(growth, dtype=float)]
    return float(np.min(path / np.maximum.accumulate(path) - 1.0))


def build_all_from_gcs():
    """사건, A/B/C/D 변수와 60일 경로 라벨을 공개 GCS에서 전부 재계산한다."""
    stock = _normalize_panel(read_public_gcs_parquet(GCS_STOCK_KEY))
    membership = _normalize_panel(read_public_gcs_parquet(GCS_MEMBERSHIP_KEY))
    attention_raw = _normalize_panel(read_public_gcs_parquet(GCS_ATTENTION_RAW_KEY))
    attention = _normalize_panel(read_public_gcs_parquet(GCS_ATTENTION_KEY))
    short = _normalize_panel(read_public_gcs_parquet(GCS_SHORT_KEY))
    membership = membership.drop_duplicates(["ticker", "date"], keep="last")
    attention_raw = attention_raw.drop_duplicates(["ticker", "date"], keep="last")
    attention = attention.drop_duplicates(["ticker", "date"], keep="last")
    short = short.drop_duplicates(["ticker", "date"], keep="last")
    market = read_public_gcs_parquet(GCS_MARKET_KEY).copy()
    market["date"] = pd.to_datetime(market["date"], errors="coerce").dt.normalize()
    market = market.sort_values("date").drop_duplicates("date", keep="last")

    required_stock = {
        "ticker", "date", "close", "volume", "trading_value", "pbr",
        "foreign_net_buy_value", "institution_net_buy_value",
        "individual_net_buy_value", "foreign_ownership_pct",
    }
    if missing := required_stock - set(stock.columns):
        raise KeyError(f"stock_daily 필수 열 누락: {sorted(missing)}")
    if not {"ticker", "date", "is_member"}.issubset(membership.columns):
        raise KeyError("membership_daily에 ticker/date/is_member가 필요합니다.")

    stock = stock.loc[stock["date"].between(START_DATE, END_DATE)].copy()
    for column in required_stock - {"ticker", "date"}:
        stock[column] = pd.to_numeric(stock[column], errors="coerce")
    stock = stock.drop_duplicates(["ticker", "date"], keep="last").reset_index(drop=True)
    grouped = stock.groupby("ticker", sort=False)

    # A: 주가·거래량·유동성
    stock["daily_return"] = grouped["close"].pct_change(fill_method=None)
    stock["price_return_5d"] = grouped["close"].pct_change(5, fill_method=None)
    stock["price_return_20d"] = grouped["close"].pct_change(20, fill_method=None)
    stock["price_volatility_20d"] = grouped["daily_return"].transform(
        lambda x: x.rolling(20, min_periods=15).std(ddof=1)
    )
    prior_volume20 = grouped["volume"].transform(
        lambda x: x.shift(1).rolling(20, min_periods=20).mean()
    )
    stock["volume_to_avg_20d"] = stock["volume"] / prior_volume20.replace(0, np.nan)
    volume5 = grouped["volume"].transform(lambda x: x.rolling(5, min_periods=5).mean())
    stock["volume_avg5_to_avg20"] = volume5 / prior_volume20.replace(0, np.nan)
    stock["_amihud"] = stock["daily_return"].abs() / stock["trading_value"].replace(0, np.nan)
    stock["stock_amihud_20d"] = grouped["_amihud"].transform(
        lambda x: x.rolling(20, min_periods=15).mean()
    )

    # B/D 수급: 최근 5일과 직전 20일의 순매수/거래대금 비율
    tv5 = grouped["trading_value"].transform(lambda x: x.rolling(5, min_periods=3).sum())
    tv20 = grouped["trading_value"].transform(lambda x: x.rolling(20, min_periods=10).sum())
    tv_prev20 = grouped["trading_value"].transform(lambda x: x.shift(5).rolling(20, min_periods=10).sum())
    for actor, source in [
        ("foreign", "foreign_net_buy_value"),
        ("institution", "institution_net_buy_value"),
        ("individual", "individual_net_buy_value"),
    ]:
        value5 = grouped[source].transform(lambda x: x.rolling(5, min_periods=3).sum())
        value20 = grouped[source].transform(lambda x: x.rolling(20, min_periods=10).sum())
        previous20 = grouped[source].transform(lambda x: x.shift(5).rolling(20, min_periods=10).sum())
        stock[f"{actor}_net_buy_ratio_5d"] = value5 / tv5.replace(0, np.nan)
        stock[f"{actor}_net_buy_ratio_20d"] = value20 / tv20.replace(0, np.nan)
        stock[f"{actor}_flow_transition_5d_vs_prev20d"] = (
            value5 / tv5.replace(0, np.nan) - previous20 / tv_prev20.replace(0, np.nan)
        )
    stock["foreign_ownership_change_20d_pp"] = grouped["foreign_ownership_pct"].diff(20)
    pbr_positive = stock["pbr"].where(stock["pbr"].gt(0))
    stock["market_relative_pbr"] = pbr_positive - pbr_positive.groupby(stock["date"]).transform("median")

    # 사건: 20일 +20%, 거래량 1.5배, 최초 진입, 동일 종목 60거래일 cooldown
    member_keys = membership.loc[pd.to_numeric(membership["is_member"], errors="coerce").eq(1), ["ticker", "date"]]
    stock = stock.merge(member_keys.assign(is_member=True), on=["ticker", "date"], how="left")
    stock["is_member"] = stock["is_member"].fillna(False).astype(bool)
    condition = stock["is_member"] & stock["price_return_20d"].ge(PRICE_THRESHOLD) & stock["volume_to_avg_20d"].ge(VOLUME_THRESHOLD)
    previous = condition.groupby(stock["ticker"]).shift(1, fill_value=False)
    stock["event_candidate"] = condition & ~previous
    stock["trading_position"] = stock.groupby("ticker").cumcount()
    chosen = []
    for _, group in stock.loc[stock["event_candidate"]].groupby("ticker", sort=False):
        last = -10**9
        for index, row in group.sort_values("trading_position").iterrows():
            position = int(row["trading_position"])
            if position - last > COOLDOWN_TRADING_DAYS:
                chosen.append(index)
                last = position
    events = stock.loc[chosen].copy().sort_values(["date", "ticker"]).reset_index(drop=True)

    # 검색 변수: GCS raw search_index에서 계산
    if "search_index" not in attention_raw.columns:
        raise KeyError("attention_raw_daily에 search_index가 없습니다.")
    attention_raw["search_index"] = pd.to_numeric(attention_raw["search_index"], errors="coerce")
    if "search_raw_available" in attention_raw.columns:
        available = as_bool(attention_raw["search_raw_available"])
        attention_raw.loc[~available, "search_index"] = np.nan
    ag = attention_raw.groupby("ticker", sort=False)
    search_prior20 = ag["search_index"].transform(lambda x: x.shift(1).rolling(20, min_periods=10).mean())
    search5 = ag["search_index"].transform(lambda x: x.rolling(5, min_periods=3).mean())
    search20 = ag["search_index"].transform(lambda x: x.rolling(20, min_periods=10).mean())
    attention_raw["search_to_avg20"] = attention_raw["search_index"] / search_prior20.replace(0, np.nan)
    attention_raw["_search_ratio"] = search5 / search20.replace(0, np.nan)
    attention_raw["search_accel_5d10d"] = ag["_search_ratio"].transform(lambda x: x - x.shift(10))
    events = events.merge(
        attention_raw[["ticker", "date", "search_to_avg20", "search_accel_5d10d"]],
        on=["ticker", "date"], how="left", validate="one_to_one",
    )

    # GCS attention_daily에는 정제 뉴스 지표가 있으므로 7/30 비율의 10일 변화를 사용한다.
    if "news_ratio_7_30" in attention.columns:
        attention["news_ratio_7_30"] = pd.to_numeric(attention["news_ratio_7_30"], errors="coerce")
        attention["news_accel_5d10d"] = attention.groupby("ticker")["news_ratio_7_30"].diff(10)
        events = events.merge(
            attention[["ticker", "date", "news_accel_5d10d"]],
            on=["ticker", "date"], how="left", validate="one_to_one",
        )
    else:
        events["news_accel_5d10d"] = np.nan

    # 공매도 D 변수
    for column in ["short_balance_shares", "short_value"]:
        if column not in short.columns:
            raise KeyError(f"short_daily에 {column}이 없습니다.")
        short[column] = pd.to_numeric(short[column], errors="coerce")
    sg = short.groupby("ticker", sort=False)
    short["short_balance_change_5d"] = short["short_balance_shares"] / sg["short_balance_shares"].shift(5).replace(0, np.nan) - 1
    short["short_balance_change_20d"] = short["short_balance_shares"] / sg["short_balance_shares"].shift(20).replace(0, np.nan) - 1
    short5 = sg["short_value"].transform(lambda x: x.rolling(5, min_periods=3).mean())
    short20 = sg["short_value"].transform(lambda x: x.rolling(20, min_periods=10).mean())
    short["short_value_to_avg20"] = short5 / short20.replace(0, np.nan) - 1
    events = events.merge(
        short[["ticker", "date", "short_balance_change_5d", "short_balance_change_20d", "short_value_to_avg20"]],
        on=["ticker", "date"], how="left", validate="one_to_one",
    )

    # C 시장 변수와 GCS 시장수익률 기반 Markov 확률
    high_vol = rebuild_high_volatility_probability_from_gcs()
    market_features = market[["date", "market_drawdown_60d", "market_proxy_return"]].merge(
        high_vol, on="date", how="left", validate="one_to_one"
    )
    events = events.merge(market_features, on="date", how="left", validate="many_to_one")

    # 사건 이후 정확히 60 시장거래일의 BHAR/MDD 경로
    market_features = market_features.sort_values("date").reset_index(drop=True)
    market_dates = market_features["date"].tolist()
    market_pos = {date: i for i, date in enumerate(market_dates)}
    market_return = market_features.set_index("date")["market_proxy_return"]
    price_maps = {
        ticker: group.set_index("date")["close"]
        for ticker, group in stock.groupby("ticker", sort=False)
    }
    path_rows = []
    for event in events.itertuples(index=False):
        result = {"경로60일관측가능": False}
        price_series = price_maps.get(event.ticker)
        if event.date not in market_pos or price_series is None or event.date not in price_series.index:
            path_rows.append(result); continue
        future_dates = market_dates[market_pos[event.date] + 1: market_pos[event.date] + 61]
        if len(future_dates) < 60 or not set(future_dates).issubset(price_series.index):
            path_rows.append(result); continue
        base = float(price_series.loc[event.date])
        future = price_series.reindex(future_dates).to_numpy(float)
        market_r = market_return.reindex(future_dates).to_numpy(float)
        if base <= 0 or not np.isfinite(future).all() or not np.isfinite(market_r).all():
            path_rows.append(result); continue
        stock_growth = future / base
        market_growth = np.cumprod(1 + market_r)
        bhar_growth = stock_growth / market_growth
        bhar = bhar_growth - 1
        for horizon in [5, 10, 20, 40, 60]:
            result[f"경로_BHAR_{horizon}일"] = float(bhar[horizon - 1])
        result.update({
            "경로_BHAR_10일내최대": float(bhar[:10].max()),
            "경로_BHAR_20일내최소": float(bhar[:20].min()),
            "경로_10일고점대비60일반납률": float(bhar_growth[-1] / bhar_growth[np.argmax(bhar[:10])] - 1),
            "경로_60일고점대비종점거리": float(bhar_growth[-1] / bhar_growth[np.argmax(bhar)] - 1),
            "경로_20_60일_BHAR양수비율": float(np.mean(bhar[19:60] > 1e-12)),
            "경로_종목MDD_60일": _drawdown(stock_growth),
            "경로60일관측가능": True,
        })
        path_rows.append(result)
    events = pd.concat([events.reset_index(drop=True), pd.DataFrame(path_rows)], axis=1)

    # 2021~2024에서만 경계 산출 후 경로 라벨 고정 적용
    dev = events.loc[events["date"].le(DEVELOPMENT_END) & events["경로60일관측가능"]]
    pullback_cut = dev["경로_10일고점대비60일반납률"].quantile(.30)
    peak_cut = dev["경로_60일고점대비종점거리"].quantile(.30)
    mdd_cut = dev["경로_종목MDD_60일"].quantile(.30)
    observable = events["경로60일관측가능"]
    immediate = observable & events["경로_BHAR_10일"].le(0) & events["경로_BHAR_20일"].lt(0) & events["경로_BHAR_60일"].le(0)
    overheat = observable & ~immediate & events["경로_BHAR_10일내최대"].gt(0) & events["경로_10일고점대비60일반납률"].le(pullback_cut) & events["경로_BHAR_60일"].lt(0)
    recovery = observable & ~immediate & ~overheat & events["경로_BHAR_20일내최소"].lt(0) & events["경로_BHAR_60일"].gt(0) & events["경로_20_60일_BHAR양수비율"].ge(POSITIVE_RATIO_CUT) & events["경로_60일고점대비종점거리"].gt(peak_cut)
    sustained = observable & ~immediate & ~overheat & ~recovery & events["경로_BHAR_20일"].gt(0) & events["경로_BHAR_60일"].gt(0) & events["경로_20_60일_BHAR양수비율"].ge(POSITIVE_RATIO_CUT) & events["경로_60일고점대비종점거리"].gt(peak_cut)
    volatility_risk = (recovery | sustained) & events["경로_종목MDD_60일"].le(mdd_cut)
    recovery &= ~volatility_risk
    sustained &= ~volatility_risk
    events[TARGET_DOWNSIDE] = (immediate | overheat | volatility_risk).where(observable)
    events[TARGET_PERSISTENCE] = (recovery | sustained).where(observable)
    events["분석기간"] = np.where(events["date"].le(DEVELOPMENT_END), "기준설정(2021~2024)", "미래검증(2025~2026)")

    feature_columns = list(dict.fromkeys(GROUPS["D_이탈가속공매도"]))
    keep = ["ticker", "date", "분석기간", "경로60일관측가능", TARGET_DOWNSIDE, TARGET_PERSISTENCE, "경로_BHAR_60일", "경로_종목MDD_60일", *feature_columns]
    model_ready = events[keep].copy()
    outcomes = events[["ticker", "date", "경로_BHAR_60일", "경로_종목MDD_60일", TARGET_DOWNSIDE, TARGET_PERSISTENCE, "경로60일관측가능"]].copy()
    return model_ready, outcomes


def as_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype("boolean").fillna(False).astype(bool)
    text = series.astype("string").str.strip().str.lower()
    return text.isin({"true", "1", "yes", "y"})


def usable_features(frame, requested):
    selected = []
    for feature in requested:
        if feature not in frame.columns:
            continue
        values = pd.to_numeric(frame[feature], errors="coerce").replace(
            [np.inf, -np.inf], np.nan
        )
        if values.notna().sum() >= 20 and values.nunique(dropna=True) >= 3:
            selected.append(feature)
    return selected


def clean_numeric(frame, features):
    return frame[features].apply(pd.to_numeric, errors="coerce").replace(
        [np.inf, -np.inf], np.nan
    )


def make_model(features):
    preprocess = ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler()),
        ]), features)
    ])
    classifier = LogisticRegression(
        penalty="l2", C=0.5, class_weight="balanced", solver="liblinear",
        max_iter=5000, random_state=RANDOM_STATE,
    )
    return Pipeline([("preprocess", preprocess), ("classifier", classifier)])


def rank_oof_scores(data, features):
    ordered = data.sort_values(["date", "ticker"], kind="stable").reset_index(drop=True)
    dates = np.array(sorted(ordered["date"].dropna().unique()))
    frames = []
    for fold, (train_idx, valid_idx) in enumerate(TimeSeriesSplit(n_splits=4).split(dates), 1):
        train_dates = set(dates[train_idx])
        valid_dates = set(dates[valid_idx])
        train = ordered[ordered["date"].isin(train_dates)].copy()
        valid = ordered[ordered["date"].isin(valid_dates)].copy()

        down = make_model(features)
        down.fit(clean_numeric(train, features), train[TARGET_DOWNSIDE].astype(int))
        p_down = down.predict_proba(clean_numeric(valid, features))[:, 1]

        safe = train[~train[TARGET_DOWNSIDE]].copy()
        persist = make_model(features)
        persist.fit(clean_numeric(safe, features), safe[TARGET_PERSISTENCE].astype(int))
        p_persist = persist.predict_proba(clean_numeric(valid, features))[:, 1]

        part = valid[["ticker", "date", TARGET_DOWNSIDE, TARGET_PERSISTENCE]].copy()
        part["fold"] = fold
        part["하락위험점수"] = p_down
        part["상승지속점수"] = (1.0 - p_down) * p_persist
        frames.append(part)
    return pd.concat(frames, ignore_index=True)


def rank_future_scores(development, validation, features):
    down = make_model(features)
    down.fit(clean_numeric(development, features), development[TARGET_DOWNSIDE].astype(int))

    safe = development[~development[TARGET_DOWNSIDE]].copy()
    persist = make_model(features)
    persist.fit(clean_numeric(safe, features), safe[TARGET_PERSISTENCE].astype(int))

    p_down = down.predict_proba(clean_numeric(validation, features))[:, 1]
    p_persist = persist.predict_proba(clean_numeric(validation, features))[:, 1]
    result = validation[[
        "ticker", "date", TARGET_DOWNSIDE, TARGET_PERSISTENCE,
        "경로_BHAR_60일", "경로_종목MDD_60일",
    ]].copy()
    result["하락위험점수"] = p_down
    result["상승지속점수"] = (1.0 - p_down) * p_persist
    return result


def assign_action(scores, down_cut, up_cut):
    result = scores.copy()
    avoid = result["하락위험점수"].ge(down_cut)
    buy = ~avoid & result["상승지속점수"].ge(up_cut)
    result["판정"] = "관망"
    result.loc[buy, "판정"] = "매수후보"
    result.loc[avoid, "판정"] = "회피후보"
    return result


# 1) FinDA 공개 GCS에서 사건·변수·라벨을 전부 재계산한다.
data, outcomes = build_all_from_gcs()

# 2) 학습 데이터 준비

# 공유 링크 순서가 반대여도 열 구성으로 자동 교정한다.
MODEL_MARKERS = {"price_volatility_20d", "volume_to_avg_20d", "search_to_avg20"}
OUTCOME_MARKERS = {"경로_BHAR_60일", "경로_종목MDD_60일"}
if not MODEL_MARKERS.intersection(data.columns) and MODEL_MARKERS.intersection(outcomes.columns):
    data, outcomes = outcomes, data

missing_model_markers = sorted(MODEL_MARKERS - set(data.columns))
missing_outcome_markers = sorted(OUTCOME_MARKERS - set(outcomes.columns))
if missing_model_markers:
    raise ValueError(f"GCS에서 재계산한 모델 데이터에 열이 없습니다: {missing_model_markers}")
if missing_outcome_markers:
    raise ValueError(f"GCS에서 재계산한 경로 데이터에 열이 없습니다: {missing_outcome_markers}")

for frame in (data, outcomes):
    frame["ticker"] = frame["ticker"].astype("string").str.zfill(6)
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce").dt.normalize()

needed_outcomes = ["경로_BHAR_60일", "경로_종목MDD_60일"]
missing = [column for column in needed_outcomes if column not in data.columns]
if missing:
    data = data.merge(
        outcomes[["ticker", "date", *missing]],
        on=["ticker", "date"], how="left", validate="one_to_one",
    )
for column in [TARGET_DOWNSIDE, TARGET_PERSISTENCE, OBSERVABLE_FLAG]:
    data[column] = as_bool(data[column])
data = data[data[OBSERVABLE_FLAG]].copy()
development = data[data["date"].le(DEVELOPMENT_END)].copy()
validation = data[data["date"].gt(DEVELOPMENT_END)].copy()

# 3) 원본과 동일하게 OOF 경계와 미래 판정 생성
boundary_rows = []
prediction_frames = []
for model_name, requested in GROUPS.items():
    features = usable_features(development, requested)
    if not features:
        raise ValueError(
            f"{model_name}: 사용 가능한 모델 변수가 0개입니다. "
            f"GCS 원자료와 계산 열을 확인하세요. 요청 변수: {requested}"
        )
    oof = rank_oof_scores(development, features)
    future = rank_future_scores(development, validation, features)
    for pct in RANK_PCTS:
        down_cut = float(oof["하락위험점수"].quantile(1.0 - pct))
        up_cut = float(oof["상승지속점수"].quantile(1.0 - pct))
        boundary_rows.append({
            "모델": model_name, "상위선정비율": pct,
            "하락위험점수경계": down_cut, "상승지속점수경계": up_cut,
            "경계산출자료": "2021~2024 날짜단위 TimeSeriesSplit OOF",
        })
        if pct == TOP_PCT:
            judged = assign_action(future, down_cut, up_cut)
            judged["모델"] = model_name
            judged["상위선정비율"] = pct
            prediction_frames.append(judged)

boundaries = pd.DataFrame(boundary_rows)
predictions = pd.concat(prediction_frames, ignore_index=True)
if SAVE_OUTPUTS:
    boundaries.to_csv(OUTPUT_DIR / "02_개발OOF_점수경계.csv", index=False, encoding="utf-8-sig")
    predictions.to_csv(
        OUTPUT_DIR / "03_미래사건별_상위20pct_판정.csv",
        index=False, encoding="utf-8-sig", date_format="%Y-%m-%d",
    )

# 4) B는 매수, C는 회피에 사용하는 혼합전략
base = [
    "ticker", "date", TARGET_DOWNSIDE, TARGET_PERSISTENCE,
    "경로_BHAR_60일", "경로_종목MDD_60일",
]
b = predictions[predictions["모델"].eq("B_수급검색가치")][
    base + ["하락위험점수", "상승지속점수"]
].rename(columns={
    "하락위험점수": "B_하락위험점수", "상승지속점수": "B_상승지속점수",
})
c = predictions[predictions["모델"].eq("C_시장보조추가")][
    ["ticker", "date", "하락위험점수"]
].rename(columns={"하락위험점수": "C_하락위험점수"})
hybrid = b.merge(c, on=["ticker", "date"], how="inner", validate="one_to_one")

boundary20 = boundaries[boundaries["상위선정비율"].round(2).eq(TOP_PCT)]
def boundary(model, column):
    return float(boundary20.loc[boundary20["모델"].eq(model), column].iloc[0])

b_down = boundary("B_수급검색가치", "하락위험점수경계")
b_up = boundary("B_수급검색가치", "상승지속점수경계")
c_down = boundary("C_시장보조추가", "하락위험점수경계")
hybrid["판정"] = "관망"
hybrid.loc[
    hybrid["B_상승지속점수"].ge(b_up) & hybrid["B_하락위험점수"].lt(b_down),
    "판정",
] = "매수후보"
hybrid.loc[hybrid["C_하락위험점수"].ge(c_down), "판정"] = "회피후보"
hybrid["연도"] = hybrid["date"].dt.year
hybrid["실제_하락위험"] = hybrid[TARGET_DOWNSIDE]
hybrid["실제_상승지속"] = ~hybrid[TARGET_DOWNSIDE] & hybrid[TARGET_PERSISTENCE]
hybrid["실제_관망"] = ~hybrid["실제_하락위험"] & ~hybrid["실제_상승지속"]

yearly = (
    hybrid.groupby(["연도", "판정"], as_index=False)
    .agg(
        사건수=("ticker", "size"),
        실제하락위험비율=("실제_하락위험", "mean"),
        실제상승지속비율=("실제_상승지속", "mean"),
        실제관망비율=("실제_관망", "mean"),
        BHAR60평균=("경로_BHAR_60일", "mean"),
        BHAR60중앙값=("경로_BHAR_60일", "median"),
        BHAR60양수비율=("경로_BHAR_60일", lambda x: x.gt(0).mean()),
        종목MDD60평균=("경로_종목MDD_60일", "mean"),
        종목MDD60중앙값=("경로_종목MDD_60일", "median"),
    )
)
yearly["전체대비비율"] = yearly["사건수"] / yearly.groupby("연도")["사건수"].transform("sum")
if SAVE_OUTPUTS:
    yearly.to_csv(
        OUTPUT_DIR / "03_혼합전략_연도별성과.csv", index=False, encoding="utf-8-sig"
    )

# 5) 최종 표
buy = yearly[yearly["판정"].eq("매수후보")].set_index("연도")
avoid = yearly[yearly["판정"].eq("회피후보")].set_index("연도")
years = [2025, 2026]
values = [[
    str(year),
    f'{buy.loc[year, "실제상승지속비율"]:.2%}',
    f'{buy.loc[year, "실제하락위험비율"]:.2%}',
    f'{buy.loc[year, "BHAR60중앙값"]:+.2%}',
    f'{avoid.loc[year, "실제하락위험비율"]:.2%}',
    f'{avoid.loc[year, "BHAR60중앙값"]:+.2%}',
] for year in years]

# Matplotlib 글꼴에 의존하지 않는 HTML 출력: 한글이 모든 노트북 환경에서 표시된다.
rows_html = "".join(
    f"""
    <tr>
      <td>{row[0]}</td>
      <td class="good">{row[1]}</td>
      <td>{row[2]}</td>
      <td class="good">{row[3]}</td>
      <td>{row[4]}</td>
      <td class="risk">{row[5]}</td>
    </tr>
    """
    for row in values
)

report_html = f"""
<div style="font-family:Arial,'Apple SD Gothic Neo','Malgun Gothic',sans-serif;
            max-width:1050px;color:#172033;line-height:1.65">
  <h2 style="color:#1677f2;margin-bottom:6px">상위 구간에서 확인된 실제 경로 차이</h2>
  <table style="border-collapse:collapse;width:100%;font-size:17px;text-align:center">
    <thead>
      <tr style="background:#dceafa">
        <th>연도</th><th>매수<br>정밀도</th><th>위험<br>혼입률</th>
        <th>매수 BHAR<br>중앙값</th><th>회피<br>정밀도</th><th>회피 BHAR<br>중앙값</th>
      </tr>
    </thead>
    <tbody>{rows_html}</tbody>
  </table>

  <style>
    table th, table td {{border:1px solid #20242b;padding:18px 10px}}
    .good {{color:#159957;font-weight:700}}
    .risk {{color:#ef2b2d;font-weight:700}}
  </style>
</div>
"""

try:
    from IPython.display import HTML, display
    display(HTML(report_html))
except ImportError:
    print(yearly.to_string(index=False))

# STAGE 9: 매수/회피 모델 — Method 1: EWMA 상승/하락 장벽 ABCD 모델

**태그:** 폐기됨 (First-touch는 최종 수익률을 보장하지 않음, 탐색 실험으로 해석)

**원본 노트북:** `FinDA_모델링준비_이벤트_성과평가_10_20_60D_Colab(1).ipynb` + `01_장벽_Target_생성_EWMA_모듈형.ipynb` + `02_9개_장벽전략_OOF_강건성_비교.ipynb` + `03_...시각화_해석.ipynb` + `04_ABCD_매수회피_모델링_10_20_60D.ipynb` + `05_...결과_시각화_해석.ipynb`

> STAGE 9는 Legacy ABCD 실험 재현 구간입니다. 사건목록은 검증 완료된 GCS artifact `events/modeling_eventset_10_20_60_v1.parquet`(2,240건)을 내려받아 사용하며, 이 노트북에서는 Eventset을 다시 만들거나 GCS에 업로드하지 않습니다.


## STAGE 9-0 — Modeling Eventset GCS 로드

`modeling_eventset_10_20_60_v1.parquet`은 별도 모델링 준비 Notebook에서 이미 정상 생성·검증되어 GCS에 업로드된 공용 Artifact를 사용합니다.

- Source: `gs://finda_project/events/modeling_eventset_10_20_60_v1.parquet`
- Local mirror: `WORKSPACE_ROOT/cache/modeling_preparation_v1/modeling_eventset_10_20_60_v1.parquet`
- Expected events: **2,240**
- 이 노트북에서는 **재생성하지 않음**
- 이 노트북에서는 **GCS에 업로드하지 않음**
- Drive raw/cache 경로에 의존하지 않음

STAGE 9-A 이후는 로컬에 내려받은 동일 parquet을 사건목록 source로 사용합니다.


In [ ]:
# ============================================================
# STAGE 9-0. Modeling Eventset — 검증 완료 GCS Artifact 로드
# ============================================================
# Source of Truth:
#   gs://finda_project/events/modeling_eventset_10_20_60_v1.parquet
#
# 이 셀은:
#   1) 공개 GCS에서 parquet을 로컬로 다운로드
#   2) 2,240건 / key / 필수 컬럼 QA
#   3) downstream 공통 객체(events/modeling_events)를 준비
#
# 하지 않는 것:
#   - Drive raw/cache 읽기
#   - Eventset 재생성
#   - attention_raw_daily로 Eventset 재구축
#   - GCS 업로드

from pathlib import Path
import numpy as np
import pandas as pd

PREP_DIR = WORKSPACE_ROOT / "cache" / "modeling_preparation_v1"
PREP_DIR.mkdir(parents=True, exist_ok=True)

MODELING_EVENTSET_GCS_KEY = "events/modeling_eventset_10_20_60_v1.parquet"
MODELING_EVENTSET_PATH = PREP_DIR / "modeling_eventset_10_20_60_v1.parquet"

HORIZONS = [10, 20, 60]
EXPECTED_EVENT_N = 2240


def normalize_ticker(s: pd.Series) -> pd.Series:
    s = (
        s.astype(str)
         .str.strip()
         .str.upper()
         .str.replace(r"\.0$", "", regex=True)
    )
    numeric = s.str.fullmatch(r"\d+", na=False)
    s.loc[numeric] = s.loc[numeric].str.zfill(6)
    return s


def normalize_date(s: pd.Series) -> pd.Series:
    out = pd.to_datetime(s, errors="coerce")
    if getattr(out.dt, "tz", None) is not None:
        out = out.dt.tz_convert(None)
    return out.dt.normalize()


# ------------------------------------------------------------
# 1. GCS -> local (읽기 전용)
# ------------------------------------------------------------
gcs_public_download(
    MODELING_EVENTSET_GCS_KEY,
    MODELING_EVENTSET_PATH,
)

print("✅ GCS artifact 다운로드 완료")
print(
    f"gs://{GCS_BUCKET_NAME}/{MODELING_EVENTSET_GCS_KEY}"
    f" -> {MODELING_EVENTSET_PATH}"
)


# ------------------------------------------------------------
# 2. 로드
# ------------------------------------------------------------
modeling_events = pd.read_parquet(MODELING_EVENTSET_PATH)

if "ticker" in modeling_events.columns:
    modeling_events["ticker"] = normalize_ticker(modeling_events["ticker"])

if "event_date" in modeling_events.columns:
    modeling_events["event_date"] = normalize_date(modeling_events["event_date"])


# ------------------------------------------------------------
# 3. Source Contract QA
# ------------------------------------------------------------
required_cols = [
    "event_id",
    "ticker",
    "ticker_name",
    "event_date",
    "news_count",
    "search_index",
]

for h in HORIZONS:
    required_cols += [
        f"eligible_{h}d",
        f"outcome_end_date_{h}d",
        f"stock_return_{h}d",
        f"market_return_{h}d",
        f"BHAR_{h}d",
        f"MDD_{h}d",
    ]

missing_cols = [
    c for c in required_cols
    if c not in modeling_events.columns
]

if missing_cols:
    raise ValueError(
        "modeling_eventset 필수 컬럼 누락:\n"
        + "\n".join(missing_cols)
    )

if len(modeling_events) != EXPECTED_EVENT_N:
    raise ValueError(
        f"modeling_eventset 사건 수 불일치: "
        f"expected={EXPECTED_EVENT_N:,}, actual={len(modeling_events):,}"
    )

if not modeling_events["event_id"].is_unique:
    raise ValueError("modeling_eventset의 event_id가 유일하지 않습니다.")

if modeling_events.duplicated(["ticker", "event_date"]).any():
    raise ValueError("modeling_eventset에 ticker + event_date 중복이 있습니다.")

if modeling_events["event_date"].isna().any():
    raise ValueError("event_date 파싱 실패 행이 있습니다.")

if modeling_events["news_count"].isna().any():
    raise ValueError("공통 Eventset인데 news_count 결측이 존재합니다.")

if modeling_events["search_index"].isna().any():
    raise ValueError("공통 Eventset인데 search_index 결측이 존재합니다.")


# eligible outcome QA
for h in HORIZONS:
    eligible = modeling_events[f"eligible_{h}d"].fillna(False).astype(bool)

    outcome_cols = [
        f"stock_return_{h}d",
        f"market_return_{h}d",
        f"BHAR_{h}d",
        f"MDD_{h}d",
    ]

    if eligible.any():
        if modeling_events.loc[eligible, outcome_cols].isna().any().any():
            raise ValueError(f"{h}D eligible outcome에 NaN이 있습니다.")

        vals = modeling_events.loc[eligible, outcome_cols].to_numpy(dtype=float)
        if not np.isfinite(vals).all():
            raise ValueError(f"{h}D eligible outcome에 inf가 있습니다.")

        if (modeling_events.loc[eligible, f"MDD_{h}d"] > 1e-12).any():
            raise ValueError(f"{h}D MDD에 양수 값이 있습니다.")


# ------------------------------------------------------------
# 4. Downstream contract
# ------------------------------------------------------------
events = modeling_events.copy()
events_abcd = modeling_events.copy()

print("\n========== MODELING EVENTSET QA ==========")
print("shape:", modeling_events.shape)
print("event count:", len(modeling_events))
print("ticker count:", modeling_events["ticker"].nunique())
print(
    "period:",
    modeling_events["event_date"].min(),
    "~",
    modeling_events["event_date"].max(),
)
print("event_id unique:", modeling_events["event_id"].is_unique)
print(
    "ticker/event_date duplicate:",
    modeling_events.duplicated(["ticker", "event_date"]).sum(),
)

for h in HORIZONS:
    print(
        f"{h}D eligible:",
        int(modeling_events[f"eligible_{h}d"].fillna(False).sum()),
        "/",
        len(modeling_events),
    )

print("\n✅ STAGE 9-0 완료 — 기존 GCS Artifact를 읽기만 합니다.")


## STAGE 9-A — EWMA Barrier Input / Target 생성 (구 01번)

STAGE 9-0에서 GCS로부터 내려받은 `modeling_eventset_10_20_60_v1.parquet`을 사건목록으로 사용합니다.


In [ ]:
# ============================================================
# 1. 설정 + 가격패널 준비
# ============================================================
# 사건목록은 STAGE 9-0이 GCS에서 내려받은 modeling_eventset parquet을 명시적으로 다시 읽어 사용합니다.
# 기존의 parquet 부재 보완용 임시 Event Builder는 제거했습니다.

HORIZONS = [10, 20, 60]
HISTORY_LOOKBACK = 120
EWMA_HALFLIVES = [5, 10, 20]
EWMA_MIN_PERIODS = 20
MULTIPLIER_GRID = [1.0, 1.5, 2.0]

# 기본은 종가로 장벽 터치 여부 판정 ('close' 또는 'high_low')
TOUCH_PRICE_MODE = 'close'
# high_low 모드에서 같은 날 상/하 장벽을 모두 터치하면 순서를 알 수 없으므로 기본 제외
SAME_DAY_POLICY = 'exclude'  # 'exclude' or 'both_positive'

print('HORIZONS:', HORIZONS)
print('EWMA_HALFLIVES:', EWMA_HALFLIVES)
print('MULTIPLIER_GRID:', MULTIPLIER_GRID)
print('TOUCH_PRICE_MODE:', TOUCH_PRICE_MODE)

PREP_DIR = WORKSPACE_ROOT / 'cache' / 'modeling_preparation_v1'
MODELING_EVENTSET_PATH = PREP_DIR / 'modeling_eventset_10_20_60_v1.parquet'
INPUT_DIR = PREP_DIR / 'barrier_inputs'
TARGET_DIR = PREP_DIR / 'barrier_targets'
INPUT_DIR.mkdir(parents=True, exist_ok=True)
TARGET_DIR.mkdir(parents=True, exist_ok=True)

HISTORY_PATH = INPUT_DIR / 'event_return_history_long.parquet'
FUTURE_PATH = INPUT_DIR / 'event_future_path_10_20_60_long.parquet'
VOL_PATH = INPUT_DIR / 'event_volatility_ewma_candidates.parquet'
MANIFEST_PATH = TARGET_DIR / 'barrier_strategy_manifest.csv'

if not MODELING_EVENTSET_PATH.exists():
    raise FileNotFoundError(
        f'{MODELING_EVENTSET_PATH}가 없습니다. STAGE 9-0 GCS Artifact 로드 셀을 먼저 실행하세요.'
    )

modeling_events = pd.read_parquet(MODELING_EVENTSET_PATH)
events = modeling_events.copy()
events['ticker'] = normalize_ticker(events['ticker'])
events['event_date'] = normalize_date(events['event_date'])

required_event_cols = ['event_id', 'ticker', 'ticker_name', 'event_date'] + [f'eligible_{h}d' for h in HORIZONS]
missing_event_cols = [c for c in required_event_cols if c not in events.columns]
if missing_event_cols:
    raise ValueError(f'modeling_eventset 필수 컬럼 누락: {missing_event_cols}')
if not events['event_id'].is_unique:
    raise ValueError('modeling_eventset의 event_id가 유일하지 않습니다.')

print('modeling_events:', modeling_events.shape)
print('사건 수:', events['event_id'].nunique())


def strategy_slug(halflife, multiplier):
    return f"ewma_hl{halflife}_k{str(multiplier).replace('.', 'p')}"


def first_true_day(days, mask):
    idx = np.flatnonzero(mask)
    return np.nan if len(idx) == 0 else int(days[idx[0]])

# ------------------------------------------------------------
# 가격패널 — 기존 장벽 분석과 동일한 기간으로 고정
# ------------------------------------------------------------

LEGACY_PRICE_START = pd.Timestamp('2021-06-11')
LEGACY_PRICE_END = pd.Timestamp('2026-06-11')

daily = stock_daily.copy()

need = [
    'ticker',
    'ticker_name',
    'date',
    'open',
    'high',
    'low',
    'close',
    'daily_return',
]

missing = [c for c in need if c not in daily.columns]
if missing:
    raise ValueError(f'stock_daily에 필수 컬럼 누락: {missing}')

daily = daily[need].copy()

daily['ticker'] = normalize_ticker(daily['ticker'])
daily['date'] = normalize_date(daily['date'])

# 중요:
# modeling_eventset_10_20_60_v1.parquet과
# 원본 EWMA Barrier 분석은 2026-06-11까지의 V3를 사용했음.
# 2026-08-10 확장분을 넣으면 eligible_horizon이 달라짐.
daily = daily[
    daily['date'].between(
        LEGACY_PRICE_START,
        LEGACY_PRICE_END,
        inclusive='both'
    )
].copy()

daily = (
    daily
    .dropna(subset=['ticker', 'date'])
    .sort_values(['ticker', 'date'])
    .reset_index(drop=True)
)

if daily.duplicated(['ticker', 'date']).any():
    raise ValueError('가격패널 ticker-date 중복')

print('daily shape:', daily.shape)
print('종목:', daily['ticker'].nunique())
print('기간:', daily['date'].min(), '~', daily['date'].max())

assert daily['date'].max() == LEGACY_PRICE_END

In [ ]:
# ============================================================
# 2. 연속 거래 block — 14일 초과 공백은 새 block
# ============================================================
daily['gap'] = daily.groupby('ticker')['date'].diff().dt.days
daily['new_block'] = daily['gap'].isna() | (daily['gap'] > 14)
daily['ticker_block'] = daily.groupby('ticker')['new_block'].cumsum()
daily['block_pos'] = daily.groupby(['ticker','ticker_block']).cumcount()
daily = daily.drop(columns=['gap','new_block'])

event_keys = events[['event_id','ticker','event_date']].merge(
    daily[['ticker','date','ticker_block','block_pos']],
    left_on=['ticker','event_date'], right_on=['ticker','date'],
    how='left', validate='one_to_one'
).drop(columns=['date'])

if event_keys['block_pos'].isna().any():
    display(event_keys[event_keys['block_pos'].isna()].head(20))
    raise ValueError('사건일이 가격패널과 매칭되지 않는 행이 있습니다.')

print('사건일 key 매칭 완료:', len(event_keys))

In [ ]:
# ============================================================
# 3. event_return_history_long (사건 이전 ~ T 수익률 history)
# ============================================================
blocks = {
    key: g.sort_values('block_pos').reset_index(drop=True)
    for key, g in daily.groupby(['ticker','ticker_block'], sort=False)
}

hist_parts=[]
for r in event_keys.itertuples(index=False):
    g = blocks[(r.ticker, r.ticker_block)]
    pos = int(r.block_pos)
    start = max(0, pos - HISTORY_LOOKBACK)
    x = g.iloc[start:pos+1][['ticker','ticker_name','date','daily_return','close']].copy()
    x['event_id'] = r.event_id
    x['event_date'] = r.event_date
    x['relative_day'] = np.arange(-len(x)+1, 1, dtype=int)
    hist_parts.append(x)

history_long = pd.concat(hist_parts, ignore_index=True)
history_long = history_long[['event_id','ticker','ticker_name','event_date','relative_day','date','daily_return','close']]
history_long.to_parquet(HISTORY_PATH, index=False, compression='zstd')

print('저장:', HISTORY_PATH)
print('shape:', history_long.shape)
print('사건 수:', history_long['event_id'].nunique())
display(history_long.head())

In [ ]:
# ============================================================
# 4. event_future_path_10_20_60_long (T ~ T+60 가격 경로)
# ============================================================
MAX_H = max(HORIZONS)
future_parts=[]

for r in event_keys.itertuples(index=False):
    g = blocks[(r.ticker, r.ticker_block)]
    pos = int(r.block_pos)
    end = min(len(g)-1, pos + MAX_H)
    x = g.iloc[pos:end+1][['ticker','ticker_name','date','open','high','low','close']].copy()
    event_close = float(g.iloc[pos]['close'])
    if not np.isfinite(event_close) or event_close <= 0:
        raise ValueError(f'{r.event_id}: 사건일 close 비정상')
    x['event_id'] = r.event_id
    x['event_date'] = r.event_date
    x['forward_day'] = np.arange(len(x), dtype=int)
    x['close_return_from_event'] = x['close'] / event_close - 1.0
    x['high_return_from_event'] = x['high'] / event_close - 1.0
    x['low_return_from_event'] = x['low'] / event_close - 1.0
    future_parts.append(x)

future_long = pd.concat(future_parts, ignore_index=True)
future_long = future_long[['event_id','ticker','ticker_name','event_date','forward_day','date',
                           'close_return_from_event','high_return_from_event','low_return_from_event']]
future_long.to_parquet(FUTURE_PATH, index=False, compression='zstd')

print('저장:', FUTURE_PATH)
print('shape:', future_long.shape)
print('사건 수:', future_long['event_id'].nunique())
display(future_long.head())

In [ ]:
# ============================================================
# 5. modeling_eventset outcome/eligibility QA
# ============================================================
max_forward = future_long.groupby('event_id')['forward_day'].max()
for h in HORIZONS:
    expected_eligible = events['event_id'].map(max_forward).ge(h).fillna(False)
    if not expected_eligible.equals(events[f'eligible_{h}d'].astype(bool).reset_index(drop=True)):
        mismatch = int((expected_eligible.to_numpy() != events[f'eligible_{h}d'].astype(bool).to_numpy()).sum())
        raise AssertionError(f'eligible_{h}d가 future path와 맞지 않습니다: mismatch={mismatch}')

required_outcome_cols = []
for h in HORIZONS:
    required_outcome_cols += [
        f'eligible_{h}d', f'outcome_end_date_{h}d', f'stock_return_{h}d',
        f'market_return_{h}d', f'BHAR_{h}d', f'MDD_{h}d',
    ]
missing = [c for c in required_outcome_cols if c not in events.columns]
if missing:
    raise ValueError(f'modeling_eventset outcome 컬럼 누락: {missing}')

for h in HORIZONS:
    cols = [f'stock_return_{h}d', f'market_return_{h}d', f'BHAR_{h}d', f'MDD_{h}d']
    eligible = events[f'eligible_{h}d'].astype(bool)
    if eligible.any():
        vals = events.loc[eligible, cols].replace([np.inf, -np.inf], np.nan)
        if vals.isna().any().any():
            raise AssertionError(f'{h}D eligible outcome에 NaN/inf가 있습니다.')
        if (events.loc[eligible, f'MDD_{h}d'] > 1e-12).any():
            raise AssertionError(f'{h}D MDD 양수 발생')

print(events[[f'eligible_{h}d' for h in HORIZONS]].mean())
print(events[[f'outcome_end_date_{h}d' for h in HORIZONS]].notna().mean())
print(events[[f'BHAR_{h}d' for h in HORIZONS]].describe())
print(events[[f'MDD_{h}d' for h in HORIZONS]].describe())
print('Barrier target input event count:', len(events))


In [ ]:
# ============================================================
# 6. EWMA volatility 후보
# ============================================================
vol_rows=[]

for event_id, g in history_long.groupby('event_id', sort=False):
    g = g.sort_values('relative_day')
    r = pd.to_numeric(g['daily_return'], errors='coerce').dropna()
    meta = g.iloc[-1]
    row = {
        'event_id': event_id,
        'ticker': meta['ticker'],
        'ticker_name': meta['ticker_name'],
        'event_date': meta['event_date'],
        'history_n_returns': len(r),
        'history_lookback_max': HISTORY_LOOKBACK,
        'ewma_min_periods': EWMA_MIN_PERIODS,
    }
    for hl in EWMA_HALFLIVES:
        if len(r) < EWMA_MIN_PERIODS:
            vol = np.nan
        else:
            vol = float(r.ewm(halflife=hl, adjust=False, min_periods=EWMA_MIN_PERIODS).std(bias=False).iloc[-1])
        row[f'ewma_vol_hl{hl}'] = vol
    vol_rows.append(row)

vol_candidates = pd.DataFrame(vol_rows)
vol_candidates.to_parquet(VOL_PATH, index=False, compression='zstd')

print('저장:', VOL_PATH)
vol_cols = [f'ewma_vol_hl{hl}' for hl in EWMA_HALFLIVES]
display(pd.DataFrame({
    'non_missing_n': vol_candidates[vol_cols].notna().sum(),
    'missing_n': vol_candidates[vol_cols].isna().sum(),
    'coverage_pct': vol_candidates[vol_cols].notna().mean()*100,
}))

In [ ]:
def generate_targets(events_df, future_df, vol_df, halflife, multiplier=1.0,
                     multiplier_by_horizon=None, touch_price_mode='close',
                     same_day_policy='exclude'):
    vol_col = f'ewma_vol_hl{halflife}'
    if vol_col not in vol_df.columns: raise KeyError(vol_col)
    if touch_price_mode not in {'close','high_low'}: raise ValueError('touch_price_mode')
    if same_day_policy not in {'exclude','both_positive'}: raise ValueError('same_day_policy')

    needed_elig = [f'eligible_{h}d' for h in HORIZONS]
    miss = [c for c in needed_elig if c not in events_df.columns]
    if miss: raise KeyError(f'eventset eligibility 컬럼 없음: {miss}')

    meta = events_df[['event_id','ticker','ticker_name','event_date'] + needed_elig].merge(
        vol_df[['event_id','history_n_returns',vol_col]], on='event_id', how='left', validate='one_to_one'
    )
    future_groups = {eid:g.sort_values('forward_day') for eid,g in future_df.groupby('event_id', sort=False)}
    rows=[]

    for ev in meta.itertuples(index=False):
        ewma_vol = getattr(ev, vol_col)
        g = future_groups.get(ev.event_id)

        for h in HORIZONS:
            eligible = bool(getattr(ev, f'eligible_{h}d'))
            k = multiplier_by_horizon.get(h, multiplier) if multiplier_by_horizon else multiplier

            upper=lower=upper_day=lower_day=np.nan
            same_day=False
            first_touch='ineligible'
            buy=risk=np.nan

            if eligible and np.isfinite(ewma_vol) and ewma_vol >= 0 and g is not None:
                upper = float(k * ewma_vol)
                lower = float(-k * ewma_vol)
                p = g[(g['forward_day']>=1) & (g['forward_day']<=h)].copy()
                days = p['forward_day'].to_numpy(dtype=int)

                if touch_price_mode == 'close':
                    rr = p['close_return_from_event'].to_numpy(dtype=float)
                    upper_day = first_true_day(days, np.isfinite(rr) & (rr >= upper))
                    lower_day = first_true_day(days, np.isfinite(rr) & (rr <= lower))
                else:
                    hi = p['high_return_from_event'].to_numpy(dtype=float)
                    lo = p['low_return_from_event'].to_numpy(dtype=float)
                    upper_day = first_true_day(days, np.isfinite(hi) & (hi >= upper))
                    lower_day = first_true_day(days, np.isfinite(lo) & (lo <= lower))

                up = np.isfinite(upper_day); dn = np.isfinite(lower_day)
                if not up and not dn:
                    first_touch='neither'; buy=0; risk=0
                elif up and not dn:
                    first_touch='upper'; buy=1; risk=0
                elif dn and not up:
                    first_touch='lower'; buy=0; risk=1
                elif upper_day < lower_day:
                    first_touch='upper'; buy=1; risk=0
                elif lower_day < upper_day:
                    first_touch='lower'; buy=0; risk=1
                else:
                    same_day=True; first_touch='both_same_day'
                    if same_day_policy == 'both_positive': buy=1; risk=1

            rows.append({
                'event_id': ev.event_id, 'ticker': ev.ticker, 'ticker_name': ev.ticker_name,
                'event_date': ev.event_date, 'horizon': h, 'eligible_horizon': eligible,
                'volatility_strategy': 'EWMA_T_inclusive', 'ewma_halflife': halflife,
                'history_n_returns': ev.history_n_returns, 'touch_price_mode': touch_price_mode,
                'barrier_multiplier': float(k), 'ewma_daily_vol': ewma_vol,
                'upper_barrier_return': upper, 'lower_barrier_return': lower,
                'upper_first_day': upper_day, 'lower_first_day': lower_day,
                'same_day_both_flag': same_day, 'first_touch': first_touch,
                'buy_target': buy, 'risk_target': risk,
            })
    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# 7. 후보 전략별 target 파일 생성 (EWMA halflife 3종 x multiplier 3종 = 9개 전략)
# ============================================================
import json  # ← 이거 추가

manifest_rows=[]

for hl in EWMA_HALFLIVES:
    for k in MULTIPLIER_GRID:
        ...

manifest_rows=[]

for hl in EWMA_HALFLIVES:
    for k in MULTIPLIER_GRID:
        slug = strategy_slug(hl, k)
        sdir = TARGET_DIR / slug
        sdir.mkdir(parents=True, exist_ok=True)
        target_path = sdir / 'targets.parquet'
        config_path = sdir / 'config.json'

        t = generate_targets(
            events, future_long, vol_candidates,
            halflife=hl, multiplier=k,
            multiplier_by_horizon=None,
            touch_price_mode=TOUCH_PRICE_MODE,
            same_day_policy=SAME_DAY_POLICY
        )
        t['strategy_name'] = slug
        t.to_parquet(target_path, index=False, compression='zstd')

        config = {
            'strategy_name': slug,
            'volatility_strategy': 'EWMA_T_inclusive',
            'market_volatility_used': False,
            'history_lookback': HISTORY_LOOKBACK,
            'ewma_halflife': hl,
            'ewma_min_periods': EWMA_MIN_PERIODS,
            'barrier_multiplier': k,
            'horizons': HORIZONS,
            'touch_price_mode': TOUCH_PRICE_MODE,
            'same_day_policy': SAME_DAY_POLICY,
            'note': 'Candidate strategy only; not a final adopted threshold.'
        }
        with open(config_path, 'w', encoding='utf-8') as f:
            json.dump(config, f, ensure_ascii=False, indent=2)

        for h in HORIZONS:
            d = t[(t['horizon']==h) & t['eligible_horizon']].copy()
            valid = d[d['buy_target'].notna() & d['risk_target'].notna()].copy()
            manifest_rows.append({
                'strategy_name': slug, 'ewma_halflife': hl, 'barrier_multiplier': k,
                'horizon': h, 'eligible_n': len(d), 'valid_target_n': len(valid),
                'buy_target_rate': valid['buy_target'].mean() if len(valid) else np.nan,
                'risk_target_rate': valid['risk_target'].mean() if len(valid) else np.nan,
                'neither_rate': (valid['first_touch']=='neither').mean() if len(valid) else np.nan,
                'same_day_both_n': int(d['same_day_both_flag'].sum()),
                'target_path': str(target_path),
            })
        print('✅', slug)

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(MANIFEST_PATH, index=False, encoding='utf-8-sig')
print('Manifest:', MANIFEST_PATH)
display(manifest)

In [ ]:
# ============================================================
# 8. 2021~2024 / 2025~2026 진단
# ============================================================
diag=[]
for hl in EWMA_HALFLIVES:
    for k in MULTIPLIER_GRID:
        slug = strategy_slug(hl,k)
        t = pd.read_parquet(TARGET_DIR / slug / 'targets.parquet')
        t['period'] = np.where(t['event_date'].dt.year <= 2024, '2021_2024', '2025_2026')
        for (period,h), d in t.groupby(['period','horizon']):
            d = d[d['eligible_horizon'] & d['buy_target'].notna() & d['risk_target'].notna()]
            diag.append({
                'strategy': slug, 'period': period, 'horizon': h, 'n': len(d),
                'buy_rate': d['buy_target'].mean() if len(d) else np.nan,
                'risk_rate': d['risk_target'].mean() if len(d) else np.nan,
                'neither_rate': (d['first_touch']=='neither').mean() if len(d) else np.nan,
            })

diagnostic = pd.DataFrame(diag)
display(diagnostic.sort_values(['strategy','period','horizon']))

## STAGE 9-D — A/B/C/D 매수·회피 모델링 10D/20D/60D (구 04번, eventset·V3·관심지표 GCS로 재구성)

In [ ]:
# ============================================================
# 0. Google Drive + 패키지
# ============================================================

from pathlib import Path
import sys
import subprocess
import importlib.util

# LightGBM이 없는 런타임만 설치
if importlib.util.find_spec("lightgbm") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "lightgbm"
    ])

import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
import json
import warnings

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("pandas:", pd.__version__)
print("lightgbm:", lgb.__version__)

### 1. 설정

**장벽 전략을 바꾸고 싶으면 `TARGET_STRATEGY`만 변경하세요.**

현재 기본값 `ewma_hl10_k1p5`는 최종 기준으로 강제하는 값이 아니라,
half-life 5/10/20과 multiplier 1.0/1.5/2.0의 중앙 조합을 **1차 baseline**으로 사용하는 것입니다.

이 노트북의 A/B/C/D 비교가 끝난 뒤, 같은 조건에서 다른 target 전략으로 다시 실행할 수 있습니다.

In [ ]:
# ============================================================
# 1. 경로 / 핵심 설정
# ============================================================
# EVENTSET_PATH는 STAGE 9-0에서 GCS로부터 내려받은 modeling_eventset parquet입니다.
# 아래 A/B/C/D 코드는 기존 변수명(events)을 유지해 같은 eventset과 barrier target을 사용합니다.

PREP_DIR = WORKSPACE_ROOT / "cache" / "modeling_preparation_v1"
MODELING_EVENTSET_PATH = PREP_DIR / "modeling_eventset_10_20_60_v1.parquet"

# ------------------------------------------------------------
# 장벽 전략
# ------------------------------------------------------------

# [수정] 원본 04번 기본값은 "ewma_hl10_k2p0"였으나, 02번(9개 전략 비교)과
# 05번(결과 시각화)이 둘 다 "ewma_hl10_k1p5"를 하드코딩해서 찾고 있어
# (원본 파일들 사이에 있던 불일치) 여기서 k1p5로 맞춥니다.
TARGET_STRATEGY = "ewma_hl10_k1p5"

TARGET_PATH = (
    PREP_DIR
    / "barrier_targets"
    / TARGET_STRATEGY
    / "targets.parquet"
)

if not TARGET_PATH.exists():
    available = sorted(
        p.parent.name
        for p in (PREP_DIR / "barrier_targets").glob("*/targets.parquet")
    )
    print("\n사용 가능한 target 전략:")
    print(available)
    raise FileNotFoundError(
        f"TARGET_STRATEGY='{TARGET_STRATEGY}' 파일을 찾지 못했습니다."
    )

# ------------------------------------------------------------
# 분석 horizon
# ------------------------------------------------------------

HORIZONS = [10, 20, 60]

# ------------------------------------------------------------
# 시간 분리
# ------------------------------------------------------------

TRAIN_SELECTION_END = pd.Timestamp("2024-12-31")
OOT_START = pd.Timestamp("2025-01-01")

# walk-forward validation 구간
CV_WINDOWS = [
    ("2022H2", "2022-07-01", "2022-12-31"),
    ("2023H1", "2023-01-01", "2023-06-30"),
    ("2023H2", "2023-07-01", "2023-12-31"),
    ("2024H1", "2024-01-01", "2024-06-30"),
    ("2024H2", "2024-07-01", "2024-12-31"),
]

TOP_Q_LIST = [0.10, 0.20, 0.30]

# ------------------------------------------------------------
# 출력
# ------------------------------------------------------------

RUN_DIR = (
    PREP_DIR
    / "modeling_runs"
    / TARGET_STRATEGY
)

MODEL_DIR = RUN_DIR / "models"

RUN_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_PATH = RUN_DIR / "event_features_ABCD_v1.parquet"
FEATURE_MANIFEST_PATH = RUN_DIR / "feature_manifest.json"

OOF_PRED_PATH = RUN_DIR / "predictions_oof_2021_2024.parquet"
OOT_PRED_PATH = RUN_DIR / "predictions_oot_2025_2026.parquet"

THRESHOLD_PATH = RUN_DIR / "score_thresholds_top10_20_30.csv"
MODEL_METRIC_PATH = RUN_DIR / "model_metrics.csv"
CANDIDATE_EVAL_PATH = RUN_DIR / "candidate_evaluation_oot.csv"
TOP20_COMPARE_PATH = RUN_DIR / "comparison_summary_top20.csv"

print("EVENTSET:", MODELING_EVENTSET_PATH.exists(), MODELING_EVENTSET_PATH)
print("TARGET:", TARGET_PATH.exists(), TARGET_PATH)
print("RUN_DIR:", RUN_DIR)


In [ ]:
# ============================================================
# 2. 공통 유틸
# ============================================================

def normalize_ticker(s: pd.Series) -> pd.Series:
    s = (
        s.astype(str)
         .str.strip()
         .str.upper()
         .str.replace(r"\.0$", "", regex=True)
    )
    numeric = s.str.fullmatch(r"\d+", na=False)
    s.loc[numeric] = s.loc[numeric].str.zfill(6)
    return s


def normalize_date(s: pd.Series) -> pd.Series:
    out = pd.to_datetime(s, errors="coerce")
    if getattr(out.dt, "tz", None) is not None:
        out = out.dt.tz_convert(None)
    return out.dt.normalize()


def safe_divide(a, b):
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")

    out = np.where(
        np.isfinite(a) &
        np.isfinite(b) &
        (b != 0),
        a / b,
        np.nan
    )

    return pd.Series(out, index=a.index)


def first_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def clean_numeric_frame(X):
    X = X.copy()

    for c in X.columns:
        if pd.api.types.is_bool_dtype(X[c]):
            X[c] = X[c].astype("int8")
        else:
            X[c] = pd.to_numeric(X[c], errors="coerce")

    X = X.replace([np.inf, -np.inf], np.nan)

    # 100% missing 또는 상수 컬럼 제거
    keep = []
    for c in X.columns:
        if X[c].notna().sum() == 0:
            continue

        if X[c].nunique(dropna=True) <= 1:
            continue

        keep.append(c)

    return X[keep]


def metric_dict(y_true, score, threshold=0.5):
    y_true = np.asarray(y_true)
    score = np.asarray(score)

    valid = np.isfinite(y_true) & np.isfinite(score)

    y = y_true[valid].astype(int)
    s = score[valid]

    out = {
        "n": len(y),
        "positive_rate": y.mean() if len(y) else np.nan,
        "roc_auc": np.nan,
        "pr_auc": np.nan,
        "balanced_accuracy_0p5": np.nan,
        "precision_0p5": np.nan,
        "recall_0p5": np.nan,
        "f1_0p5": np.nan,
    }

    if len(y) == 0:
        return out

    pred = (s >= threshold).astype(int)

    if np.unique(y).size == 2:
        out["roc_auc"] = roc_auc_score(y, s)
        out["pr_auc"] = average_precision_score(y, s)
        out["balanced_accuracy_0p5"] = balanced_accuracy_score(y, pred)
        out["precision_0p5"] = precision_score(y, pred, zero_division=0)
        out["recall_0p5"] = recall_score(y, pred, zero_division=0)
        out["f1_0p5"] = f1_score(y, pred, zero_division=0)

    return out


def make_lgbm():
    return lgb.LGBMClassifier(
        objective="binary",
        n_estimators=350,
        learning_rate=0.03,
        num_leaves=15,
        max_depth=4,
        min_child_samples=20,
        subsample=0.90,
        colsample_bytree=0.90,
        reg_alpha=0.10,
        reg_lambda=0.50,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
    )

### 3. 고정 eventset + target 로드

`BHAR`, `MDD`, 미래 수익률은 feature에서 자동 배제합니다.

Target 파일의 `buy_target`, `risk_target`은 horizon별로 long format이므로
모델 학습 때 해당 horizon만 선택합니다.

In [ ]:
# ============================================================
# 3. Eventset + target (STAGE 9-0에서 GCS로부터 로드한 events를 재사용)
# ============================================================

if "events" not in globals():
    if not MODELING_EVENTSET_PATH.exists():
        raise FileNotFoundError(MODELING_EVENTSET_PATH)
    events = pd.read_parquet(MODELING_EVENTSET_PATH)

events_abcd = events.copy()
targets = pd.read_parquet(TARGET_PATH)

events_abcd["ticker"] = normalize_ticker(events_abcd["ticker"])
events_abcd["event_date"] = normalize_date(events_abcd["event_date"])

targets["ticker"] = normalize_ticker(targets["ticker"])
targets["event_date"] = normalize_date(targets["event_date"])

print("EVENTS:", events_abcd.shape)
print("TARGETS:", targets.shape)

print("사건 수:", events_abcd["event_id"].nunique())
print("target 사건 수:", targets["event_id"].nunique())

if not events_abcd["event_id"].is_unique:
    raise ValueError("eventset의 event_id가 유일하지 않습니다.")

expected_target_rows = len(events_abcd) * len(HORIZONS)

if len(targets) != expected_target_rows:
    print(
        f"⚠️ target 행 기대값={expected_target_rows:,}, 실제={len(targets):,}"
    )

# target 분포
target_diag = (
    targets[
        targets["eligible_horizon"] &
        targets["buy_target"].notna() &
        targets["risk_target"].notna()
    ]
    .groupby("horizon")
    .agg(
        n=("event_id", "size"),
        buy_rate=("buy_target", "mean"),
        risk_rate=("risk_target", "mean"),
        neither_rate=("first_touch", lambda s: (s == "neither").mean()),
    )
    .reset_index()
)

display(target_diag)

# 아래 셀들은 원본에서 "events"라는 이름을 그대로 참조하므로 여기서 갱신
events = events_abcd


## 4. 사건일용 BASE historical feature 생성

A/B/C/D 비교의 공통 BASE를 풍부하게 만들기 위해 V3 일별 데이터에서
**사건일 T까지의 정보만** 사용해 추가 피처를 계산합니다.

포함:
- 1D / 5D / 10D / 20D 수익률
- 5D / 20D / 60D 종목 변동성
- 최근 10D 상승일 비율 / 최대 일간수익률
- 가격 가속도
- 당일 고저폭
- 거래량 / 거래대금 / 유동성
- PER / PBR / EPS / 로그 시총
- 개인 / 외국인 / 기관 수급
  - 당일 비율
  - 5D / 10D 누적 비율
- 외국인 보유비율 및 5D / 20D 변화

시장지수 기반 시장 변동성은 현재 버전에서는 추가하지 않습니다.

In [ ]:
# ============================================================
# 4. 가격 패널 (STAGE 9-A의 daily는 장벽 계산용 8개 컬럼만 남긴 축소판이라,
#    이 STAGE(BASE feature 생성)에 필요한 전체 컬럼을 위해 stock_daily에서
#    다시 만듭니다. 001260은 분석 universe와 동일하게 제외합니다.)
# ============================================================

daily = stock_daily.copy()
daily["ticker"] = normalize_ticker(daily["ticker"])
daily["date"] = normalize_date(daily["date"])
daily = daily[daily["ticker"] != "001260"].copy()

daily = (
    daily
    .dropna(subset=["ticker", "date"])
    .sort_values(["ticker", "date"])
    .reset_index(drop=True)
)

if daily.duplicated(["ticker", "date"]).any():
    raise ValueError("가격 패널 ticker-date 중복이 있습니다.")

print("daily:", daily.shape)
print("종목:", daily["ticker"].nunique())
print("기간:", daily["date"].min(), "~", daily["date"].max())

In [ ]:
# ============================================================
# 5. 연속 block + BASE feature engineering
# ============================================================

daily["date_gap_days"] = daily.groupby("ticker")["date"].diff().dt.days
daily["new_block"] = (
    daily["date_gap_days"].isna() |
    (daily["date_gap_days"] > 14)
)
daily["ticker_block_fe"] = daily.groupby("ticker")["new_block"].cumsum()

G = ["ticker", "ticker_block_fe"]

# -------------------------------
# 가격
# -------------------------------

daily["feat_return_1d"] = (
    daily.groupby(G)["close"].pct_change(fill_method=None)
)

daily["feat_return_5d"] = (
    daily["close"] /
    daily.groupby(G)["close"].shift(5)
    - 1
)

daily["feat_return_10d"] = (
    daily["close"] /
    daily.groupby(G)["close"].shift(10)
    - 1
)

daily["feat_return_20d"] = (
    daily["close"] /
    daily.groupby(G)["close"].shift(20)
    - 1
)

daily["feat_vol_5d"] = (
    daily.groupby(G)["feat_return_1d"]
    .transform(lambda s: s.rolling(5, min_periods=5).std())
)

daily["feat_vol_20d"] = (
    daily.groupby(G)["feat_return_1d"]
    .transform(lambda s: s.rolling(20, min_periods=20).std())
)

daily["feat_vol_60d"] = (
    daily.groupby(G)["feat_return_1d"]
    .transform(lambda s: s.rolling(60, min_periods=40).std())
)

daily["feat_up_ratio_10d"] = (
    daily.groupby(G)["feat_return_1d"]
    .transform(
        lambda s: s.gt(0).astype(float)
        .rolling(10, min_periods=10)
        .mean()
    )
)

daily["feat_max_daily_return_10d"] = (
    daily.groupby(G)["feat_return_1d"]
    .transform(lambda s: s.rolling(10, min_periods=10).max())
)

# 기간별 평균 일간 수익률 차이로 간단한 가속도 표현
daily["feat_price_accel_1v5"] = (
    daily["feat_return_1d"]
    - daily["feat_return_5d"] / 5.0
)

daily["feat_price_accel_5v10"] = (
    daily["feat_return_5d"] / 5.0
    - daily["feat_return_10d"] / 10.0
)

daily["feat_high_low_range_pct"] = safe_divide(
    daily["high"] - daily["low"],
    daily["close"]
)

# -------------------------------
# 거래량 / 거래대금 / 유동성
# -------------------------------

daily["feat_prev20_volume_mean"] = (
    daily.groupby(G)["volume"]
    .transform(
        lambda s: s.shift(1).rolling(20, min_periods=20).mean()
    )
)

daily["feat_volume_ratio_20d"] = safe_divide(
    daily["volume"],
    daily["feat_prev20_volume_mean"]
)

daily["feat_log_trading_value"] = np.log1p(
    pd.to_numeric(daily["trading_value"], errors="coerce").clip(lower=0)
)

daily["feat_turnover_value_to_mcap"] = safe_divide(
    daily["trading_value"],
    daily["market_cap"]
)

daily["feat_log_market_cap"] = np.log1p(
    pd.to_numeric(daily["market_cap"], errors="coerce").clip(lower=0)
)

# -------------------------------
# 수급
# -------------------------------

FLOW_COLS = {
    "foreign": "foreign_net_buy_value",
    "individual": "individual_net_buy_value",
    "institution": "institution_net_buy_value",
}

for prefix, col in FLOW_COLS.items():
    if col not in daily.columns:
        continue

    daily[f"feat_{prefix}_net_buy_ratio_1d"] = safe_divide(
        daily[col],
        daily["trading_value"]
    )

    for w in [5, 10]:
        num = (
            daily.groupby(G)[col]
            .transform(
                lambda s, w=w:
                s.rolling(w, min_periods=w).sum()
            )
        )

        den = (
            daily.groupby(G)["trading_value"]
            .transform(
                lambda s, w=w:
                s.rolling(w, min_periods=w).sum()
            )
        )

        daily[f"feat_{prefix}_net_buy_ratio_{w}d"] = safe_divide(
            num, den
        )

if "foreign_ownership_pct" in daily.columns:
    daily["feat_foreign_ownership_pct"] = pd.to_numeric(
        daily["foreign_ownership_pct"],
        errors="coerce"
    )

    daily["feat_foreign_ownership_change_5d"] = (
        daily["feat_foreign_ownership_pct"]
        - daily.groupby(G)["feat_foreign_ownership_pct"].shift(5)
    )

    daily["feat_foreign_ownership_change_20d"] = (
        daily["feat_foreign_ownership_pct"]
        - daily.groupby(G)["feat_foreign_ownership_pct"].shift(20)
    )

daily = daily.drop(columns=["date_gap_days", "new_block"])

print("BASE historical feature 생성 완료")

### 6. 관심 feature

검색과 뉴스는 별도 원천에서 계산합니다.

#### 검색
- 사건일 검색지수
- 7일 / 30일 평균
- 제공된 rolling50 평균 / 백분위
- 검색지수 ÷ rolling50 평균
- 상위 10% flag

#### 뉴스
- 사건일 기사수
- 최근 7일 기사수 합계
- 최근 30일 일평균 기사수
- 당일 기사수 ÷ 최근 30일 평균

뉴스·검색은 주말에도 발생하므로 **calendar-day 원천에서 rolling을 계산한 뒤 사건일에 붙입니다.**

In [ ]:
# ============================================================
# 6. 관심 feature 생성 (STAGE 3의 attention_raw_daily를 재사용해 원자료부터 재계산)
# ============================================================
# 원본은 news_daily_20210611_20260611.csv / naver_datalab_final.csv를 Drive에서
# 직접 읽었으나, 두 CSV의 원자료(종목코드/날짜/뉴스기사수/검색지수)는
# attention_raw_daily.parquet(GCS curated)에 이미 동일하게 들어있어 그대로 재사용합니다.
# "롤링50_평균"/"롤링50_백분위"/"검색관심_상위10pct"는 raw search_index에서
# 직접 롤링 50거래일 평균/백분위로 계산합니다.

news = attention_raw_daily.copy()
news = news.rename(columns={"ticker": "종목코드", "date": "날짜"})
news["종목코드"] = normalize_ticker(news["종목코드"])
news["날짜"] = normalize_date(news["날짜"])

search = attention_raw_daily.copy()
search = search.rename(columns={"ticker": "종목코드", "date": "날짜"})
search["종목코드"] = normalize_ticker(search["종목코드"])
search["날짜"] = normalize_date(search["날짜"])

# -------------------------------
# 뉴스
# -------------------------------

news = news.sort_values(["종목코드", "날짜"]).copy()

news["feat_news_count"] = pd.to_numeric(
    news["news_count"],
    errors="coerce"
)

news["feat_news_7d_sum"] = (
    news.groupby("종목코드")["feat_news_count"]
    .transform(lambda s: s.rolling(7, min_periods=1).sum())
)

news["feat_news_30d_mean"] = (
    news.groupby("종목코드")["feat_news_count"]
    .transform(lambda s: s.rolling(30, min_periods=7).mean())
)

news["feat_news_spike_vs_30d"] = safe_divide(
    news["feat_news_count"],
    news["feat_news_30d_mean"]
)

news_features = news[
    [
        "종목코드",
        "날짜",
        "feat_news_count",
        "feat_news_7d_sum",
        "feat_news_30d_mean",
        "feat_news_spike_vs_30d",
    ]
].rename(
    columns={
        "종목코드": "ticker",
        "날짜": "event_date",
    }
)

# -------------------------------
# 검색 (raw search_index에서 롤링50거래일 평균/백분위 직접 계산)
# -------------------------------

search = search.sort_values(["종목코드", "날짜"]).copy()

search["feat_search_index"] = pd.to_numeric(
    search["search_index"],
    errors="coerce"
)

search["feat_search_7d_mean"] = (
    search.groupby("종목코드")["feat_search_index"]
    .transform(lambda s: s.rolling(7, min_periods=3).mean())
)

search["feat_search_30d_mean"] = (
    search.groupby("종목코드")["feat_search_index"]
    .transform(lambda s: s.rolling(30, min_periods=7).mean())
)

search["feat_search_rolling50_mean"] = (
    search.groupby("종목코드")["feat_search_index"]
    .transform(lambda s: s.rolling(50, min_periods=20).mean())
)

search["feat_search_rolling50_percentile"] = (
    search.groupby("종목코드")["feat_search_index"]
    .transform(lambda s: s.rolling(50, min_periods=20).rank(pct=True))
)

search["feat_search_vs_rolling50"] = safe_divide(
    search["feat_search_index"],
    search["feat_search_rolling50_mean"]
)

search["feat_search_top10pct"] = (
    search["feat_search_rolling50_percentile"].ge(0.90).astype("Int64")
)

search_features = search[
    [
        "종목코드",
        "날짜",
        "feat_search_index",
        "feat_search_7d_mean",
        "feat_search_30d_mean",
        "feat_search_rolling50_mean",
        "feat_search_rolling50_percentile",
        "feat_search_vs_rolling50",
        "feat_search_top10pct",
    ]
].rename(
    columns={
        "종목코드": "ticker",
        "날짜": "event_date",
    }
)

print("news features:", news_features.shape)
print("search features:", search_features.shape)

### 7. 사건 단위 feature table

공매도 feature는 이미 `modeling_eventset`에 사건일 기준으로 존재하므로 그대로 가져옵니다.

**모델에 사용하는 공매도 잔고는 공개시점 보정 변수만 사용**하며,
`short_balance_ratio_pct_raw`는 모델 feature에서 제외합니다.

In [ ]:
# ============================================================
# 7. 사건일 BASE / ATTENTION / SHORT feature table
# ============================================================

# V3 일별에서 사건일 historical feature 추출
HIST_FEATURE_COLS = [
    c for c in daily.columns
    if c.startswith("feat_")
]

event_hist = (
    events[["event_id", "ticker", "event_date"]]
    .merge(
        daily[
            ["ticker", "date"] + HIST_FEATURE_COLS
        ],
        left_on=["ticker", "event_date"],
        right_on=["ticker", "date"],
        how="left",
        validate="one_to_one"
    )
    .drop(columns=["date"])
)

feature_df = events.copy()

feature_df = (
    feature_df
    .merge(
        event_hist.drop(columns=["ticker", "event_date"]),
        on="event_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        news_features,
        on=["ticker", "event_date"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        search_features,
        on=["ticker", "event_date"],
        how="left",
        validate="one_to_one"
    )
)

print("feature_df:", feature_df.shape)

if len(feature_df) != len(events):
    raise AssertionError("feature 병합 후 사건 수가 달라졌습니다.")

In [ ]:
# ============================================================
# 8. A/B/C/D feature set 확정
# ============================================================

# -------------------------------
# BASE
# -------------------------------

BASE_FEATURES = [
    # 기업 가치 / 규모
    "per",
    "pbr",
    "eps",
    "feat_log_market_cap",

    # 가격 / 모멘텀
    "feat_return_1d",
    "feat_return_5d",
    "feat_return_10d",
    "feat_return_20d",
    "feat_price_accel_1v5",
    "feat_price_accel_5v10",
    "feat_up_ratio_10d",
    "feat_max_daily_return_10d",
    "feat_vol_5d",
    "feat_vol_20d",
    "feat_vol_60d",
    "feat_high_low_range_pct",

    # 거래량 / 거래대금 / 유동성
    "feat_volume_ratio_20d",
    "feat_log_trading_value",
    "feat_turnover_value_to_mcap",
    "trading_value_growth",

    # 외국인 / 개인 / 기관 수급
    "feat_foreign_net_buy_ratio_1d",
    "feat_foreign_net_buy_ratio_5d",
    "feat_foreign_net_buy_ratio_10d",

    "feat_individual_net_buy_ratio_1d",
    "feat_individual_net_buy_ratio_5d",
    "feat_individual_net_buy_ratio_10d",

    "feat_institution_net_buy_ratio_1d",
    "feat_institution_net_buy_ratio_5d",
    "feat_institution_net_buy_ratio_10d",

    "feat_foreign_ownership_pct",
    "feat_foreign_ownership_change_5d",
    "feat_foreign_ownership_change_20d",
]

# -------------------------------
# ATTENTION
# -------------------------------

ATTENTION_FEATURES = [
    "feat_news_count",
    "feat_news_7d_sum",
    "feat_news_30d_mean",
    "feat_news_spike_vs_30d",

    "feat_search_index",
    "feat_search_7d_mean",
    "feat_search_30d_mean",
    "feat_search_rolling50_mean",
    "feat_search_rolling50_percentile",
    "feat_search_vs_rolling50",
    "feat_search_top10pct",
]

# -------------------------------
# SHORT
# -------------------------------

SHORT_FEATURES = [
    "short_value_ratio_pct",
    "short_volume_ratio_pct",
    "short_value_ratio_5d_mean",
    "short_value_ratio_20d_mean",
    "short_balance_public_t2_pct",
    "short_balance_public_change_5d_pp",
    "short_balance_public_change_20d_pp",
    "short_ban_period",
    "post_full_reopen",
]

# 실제 존재하는 컬럼만 남기기
def existing(cols):
    return [c for c in cols if c in feature_df.columns]

BASE_FEATURES = existing(BASE_FEATURES)
ATTENTION_FEATURES = existing(ATTENTION_FEATURES)
SHORT_FEATURES = existing(SHORT_FEATURES)

FEATURE_SETS = {
    "A": BASE_FEATURES,
    "B": BASE_FEATURES + ATTENTION_FEATURES,
    "C": BASE_FEATURES + ATTENTION_FEATURES + SHORT_FEATURES,
    "D": BASE_FEATURES + SHORT_FEATURES,
}

print("BASE:", len(BASE_FEATURES))
print("ATTENTION:", len(ATTENTION_FEATURES))
print("SHORT:", len(SHORT_FEATURES))

for model_name, cols in FEATURE_SETS.items():
    print(model_name, len(cols), "features")

# outcome / target 누수 방지 검증
FORBIDDEN_SUBSTRINGS = [
    "BHAR_",
    "MDD_",
    "stock_return_",
    "market_return_",
    "outcome_end_date_",
    "fwd_",
    "buy_target",
    "risk_target",
    "upper_first_day",
    "lower_first_day",
]

for model_name, cols in FEATURE_SETS.items():
    bad = [
        c for c in cols
        if any(x in c for x in FORBIDDEN_SUBSTRINGS)
    ]

    if bad:
        raise ValueError(
            f"{model_name} feature에 누수 가능 컬럼이 있습니다: {bad}"
        )

print("✅ feature leakage 검사 통과")

In [ ]:
# ============================================================
# 9. Feature 품질 / 저장
# ============================================================

all_features = sorted(
    set(
        BASE_FEATURES
        + ATTENTION_FEATURES
        + SHORT_FEATURES
    )
)

quality = pd.DataFrame({
    "feature": all_features,
    "non_missing_n": [
        feature_df[c].notna().sum()
        for c in all_features
    ],
    "missing_pct": [
        feature_df[c].isna().mean() * 100
        for c in all_features
    ],
    "nunique": [
        feature_df[c].nunique(dropna=True)
        for c in all_features
    ],
}).sort_values("missing_pct", ascending=False)

display(quality)

feature_df.to_parquet(
    FEATURE_PATH,
    index=False,
    compression="zstd"
)

manifest = {
    "target_strategy": TARGET_STRATEGY,
    "base_features": BASE_FEATURES,
    "attention_features": ATTENTION_FEATURES,
    "short_features": SHORT_FEATURES,
    "feature_sets": FEATURE_SETS,
    "note": (
        "BHAR/MDD/future outcomes are excluded from all feature sets. "
        "Market-index volatility features are not used in this version."
    ),
}

with open(FEATURE_MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("✅ feature table 저장:", FEATURE_PATH)

## 10. Purged expanding walk-forward CV

각 horizon별 미래 target이 있기 때문에 단순한 날짜 split만으로는 부족합니다.

예를 들어 60D target을 가진 2022-06 사건의 미래 경로가 2022-09까지 이어지는데,
2022-07부터 validation을 시작하면 미래정보가 validation 기간과 겹칩니다.

따라서 fold마다:

- validation은 특정 날짜 구간
- train은 validation 이전 사건
- **train 사건의 `outcome_end_date_h`가 validation 시작일보다 이전인 경우만 사용**

하도록 purge합니다.

In [ ]:
# ============================================================
# 10. CV split 생성 함수
# ============================================================

def get_horizon_dataset(horizon):
    t = targets[
        (targets["horizon"] == horizon)
        & (targets["eligible_horizon"])
        & targets["buy_target"].notna()
        & targets["risk_target"].notna()
    ].copy()

    cols = [
        "event_id",
        "buy_target",
        "risk_target",
        "first_touch",
        "ewma_daily_vol",
        "upper_barrier_return",
        "lower_barrier_return",
    ]

    t = t[cols]

    d = feature_df.merge(
        t,
        on="event_id",
        how="inner",
        validate="one_to_one"
    )

    eligible_col = f"eligible_{horizon}d"
    end_col = f"outcome_end_date_{horizon}d"

    if eligible_col not in d.columns or end_col not in d.columns:
        raise KeyError(
            f"{horizon}D eligible/outcome_end_date 컬럼이 없습니다."
        )

    d[end_col] = pd.to_datetime(
        d[end_col],
        errors="coerce"
    )

    d = d[
        d[eligible_col]
        & d[end_col].notna()
    ].copy()

    return d


def make_purged_folds(d, horizon):
    end_col = f"outcome_end_date_{horizon}d"

    folds = []

    for fold_name, start, end in CV_WINDOWS:
        val_start = pd.Timestamp(start)
        val_end = pd.Timestamp(end)

        val_mask = (
            (d["event_date"] >= val_start)
            & (d["event_date"] <= val_end)
        )

        train_mask = (
            (d["event_date"] < val_start)
            & (d[end_col] < val_start)
        )

        train_idx = d.index[train_mask].to_numpy()
        val_idx = d.index[val_mask].to_numpy()

        if len(train_idx) == 0 or len(val_idx) == 0:
            continue

        # 같은 날짜가 양쪽에 들어가지 않는지 검증
        train_dates = set(d.loc[train_idx, "event_date"])
        val_dates = set(d.loc[val_idx, "event_date"])

        overlap = train_dates & val_dates

        if overlap:
            raise AssertionError(
                f"{fold_name}: 동일 날짜 train/val overlap"
            )

        folds.append({
            "fold": fold_name,
            "val_start": val_start,
            "val_end": val_end,
            "train_idx": train_idx,
            "val_idx": val_idx,
        })

    return folds

## 11. OOF 학습

각 horizon마다 A/B/C/D 각각에 대해:

- `buy_target` 모델
- `risk_target` 모델

을 별도로 학습합니다.

따라서 horizon 하나당 총 8개 binary classifier가 존재합니다.

OOF
= 2021~2024 내부에서 기준 만들기

OOT
= 만들어진 기준으로 2025~2026 진짜 미래 테스트

Fold 1
과거 데이터로 학습
→ 2022 하반기 예측

Fold 2
더 늘어난 과거 데이터로 학습
→ 2023 상반기 예측

Fold 3
더 늘어난 과거 데이터로 학습
→ 2023 하반기 예측

Fold 4
→ 2024 상반기 예측

Fold 5
→ 2024 하반기 예측

In [ ]:
# ============================================================
# 11. OOF 예측 생성
# ============================================================

oof_rows = []
metric_rows = []

for horizon in HORIZONS:
    print("\n" + "=" * 90)
    print(f"{horizon}D")
    print("=" * 90)

    d = get_horizon_dataset(horizon)
    folds = make_purged_folds(d, horizon)

    print("전체 target 가능 사건:", len(d))
    print("fold 수:", len(folds))

    for model_name, feature_cols in FEATURE_SETS.items():
        X_all = clean_numeric_frame(d[feature_cols])

        if X_all.shape[1] == 0:
            raise RuntimeError(f"{model_name}: 사용 가능한 feature가 없습니다.")

        used_features = X_all.columns.tolist()

        pred_store = {
            "buy": pd.Series(np.nan, index=d.index, dtype=float),
            "risk": pd.Series(np.nan, index=d.index, dtype=float),
        }

        fold_name_store = pd.Series(
            pd.NA,
            index=d.index,
            dtype="object"
        )

        for fold in folds:
            tr = fold["train_idx"]
            va = fold["val_idx"]

            X_train = X_all.loc[tr]
            X_val = X_all.loc[va]

            fold_name_store.loc[va] = fold["fold"]

            for target_kind, target_col in [
                ("buy", "buy_target"),
                ("risk", "risk_target"),
            ]:
                y_train = d.loc[tr, target_col].astype(int)
                y_val = d.loc[va, target_col].astype(int)

                # binary class가 모두 존재해야 학습 가능
                if y_train.nunique() < 2:
                    print(
                        f"⚠️ {horizon}D {model_name} {fold['fold']} "
                        f"{target_kind}: train class 1종 → fold skip"
                    )
                    continue

                clf = make_lgbm()

                clf.fit(
                    X_train,
                    y_train
                )

                score = clf.predict_proba(X_val)[:, 1]

                pred_store[target_kind].loc[va] = score

        # --------------------------------------
        # OOF 행 저장
        # --------------------------------------

        valid_oof = (
            pred_store["buy"].notna()
            & pred_store["risk"].notna()
        )

        tmp = d.loc[valid_oof, [
            "event_id",
            "ticker",
            "ticker_name",
            "event_date",
            "buy_target",
            "risk_target",
            f"BHAR_{horizon}d",
            f"MDD_{horizon}d",
        ]].copy()

        tmp["horizon"] = horizon
        tmp["model"] = model_name
        tmp["fold"] = fold_name_store.loc[valid_oof].values
        tmp["buy_score"] = pred_store["buy"].loc[valid_oof].values
        tmp["risk_score"] = pred_store["risk"].loc[valid_oof].values

        oof_rows.append(tmp)

        # --------------------------------------
        # OOF classifier metric
        # --------------------------------------

        for target_kind, target_col, score_col in [
            ("buy", "buy_target", "buy_score"),
            ("risk", "risk_target", "risk_score"),
        ]:
            m = metric_dict(
                tmp[target_col],
                tmp[score_col]
            )

            metric_rows.append({
                "period": "OOF_2021_2024",
                "horizon": horizon,
                "model": model_name,
                "target": target_kind,
                **m,
            })

        print(
            f"{model_name}: OOF n={len(tmp):,}, features={len(used_features)}"
        )

oof_pred = pd.concat(
    oof_rows,
    ignore_index=True
)

print("\nOOF shape:", oof_pred.shape)

## 12. OOF score에서 Top 10 / 20 / 30% 경계 산출

각 **horizon × model**마다 별도 경계를 계산합니다.

예:
- A 20D risk score의 Top 20% 경계
- B 20D risk score의 Top 20% 경계

는 서로 다를 수 있습니다.

2025~2026에는 이 숫자를 다시 계산하지 않습니다.

In [ ]:
# ============================================================
# 12. OOF score threshold
# ============================================================

threshold_rows = []

for (horizon, model_name), g in oof_pred.groupby(
    ["horizon", "model"]
):
    for q in TOP_Q_LIST:
        threshold_rows.append({
            "horizon": horizon,
            "model": model_name,
            "top_q": q,
            "buy_threshold": g["buy_score"].quantile(1 - q),
            "risk_threshold": g["risk_score"].quantile(1 - q),
            "oof_n": len(g),
        })

thresholds = pd.DataFrame(threshold_rows)

display(
    thresholds.sort_values(
        ["horizon", "model", "top_q"]
    )
)

thresholds.to_csv(
    THRESHOLD_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("✅ threshold 저장:", THRESHOLD_PATH)

## 13. 2021 ~ 2024 최종 학습 → 2025 ~ 2026 OOT 예측

최종 train에서도 horizon별 미래 경로가 2025로 넘어가는 2024년 말 사건은 제외합니다.

즉 `outcome_end_date_h <= 2024-12-31`인 사건만 최종 학습에 사용합니다.

In [ ]:
# ============================================================
# 13. Final fit + OOT prediction
# ============================================================

oot_rows = []

for horizon in HORIZONS:
    d = get_horizon_dataset(horizon)
    end_col = f"outcome_end_date_{horizon}d"

    final_train_mask = (
        (d["event_date"] <= TRAIN_SELECTION_END)
        & (d[end_col] <= TRAIN_SELECTION_END)
    )

    oot_mask = (
        d["event_date"] >= OOT_START
    )

    train_idx = d.index[final_train_mask]
    test_idx = d.index[oot_mask]

    print(
        f"\n{horizon}D final train={len(train_idx):,}, "
        f"OOT={len(test_idx):,}"
    )

    if len(train_idx) == 0 or len(test_idx) == 0:
        continue

    for model_name, feature_cols in FEATURE_SETS.items():
        X_all = clean_numeric_frame(d[feature_cols])

        used_features = X_all.columns.tolist()

        X_train = X_all.loc[train_idx]
        X_test = X_all.loc[test_idx]

        scores = {}

        for target_kind, target_col in [
            ("buy", "buy_target"),
            ("risk", "risk_target"),
        ]:
            y_train = d.loc[train_idx, target_col].astype(int)
            y_test = d.loc[test_idx, target_col].astype(int)

            if y_train.nunique() < 2:
                raise RuntimeError(
                    f"{horizon}D {model_name} {target_kind}: "
                    "final train에 class가 하나뿐입니다."
                )

            clf = make_lgbm()
            clf.fit(X_train, y_train)

            score = clf.predict_proba(X_test)[:, 1]
            scores[target_kind] = score

            # 모델 저장
            model_path = (
                MODEL_DIR
                / f"{horizon}D_{model_name}_{target_kind}.joblib"
            )

            joblib.dump(
                {
                    "model": clf,
                    "features": used_features,
                    "horizon": horizon,
                    "model_name": model_name,
                    "target": target_kind,
                    "target_strategy": TARGET_STRATEGY,
                },
                model_path
            )

            # OOT classifier metric
            m = metric_dict(y_test, score)

            metric_rows.append({
                "period": "OOT_2025_2026",
                "horizon": horizon,
                "model": model_name,
                "target": target_kind,
                **m,
            })

        tmp = d.loc[test_idx, [
            "event_id",
            "ticker",
            "ticker_name",
            "event_date",
            "buy_target",
            "risk_target",
            f"BHAR_{horizon}d",
            f"MDD_{horizon}d",
        ]].copy()

        tmp["horizon"] = horizon
        tmp["model"] = model_name
        tmp["buy_score"] = scores["buy"]
        tmp["risk_score"] = scores["risk"]

        oot_rows.append(tmp)

oot_pred = pd.concat(
    oot_rows,
    ignore_index=True
)

model_metrics = pd.DataFrame(metric_rows)

print("\nOOT shape:", oot_pred.shape)

display(
    model_metrics.sort_values(
        ["period", "horizon", "model", "target"]
    )
)

oof_pred.to_parquet(
    OOF_PRED_PATH,
    index=False,
    compression="zstd"
)

oot_pred.to_parquet(
    OOT_PRED_PATH,
    index=False,
    compression="zstd"
)

model_metrics.to_csv(
    MODEL_METRIC_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("✅ prediction / metric 저장 완료")

## 14. 2025~2026 매수 / 회피 / 관망 후보 생성

OOF에서 계산한 경계를 그대로 적용합니다.

우선순위:
1. risk score가 threshold 이상 → 회피
2. 아니면서 buy score가 threshold 이상 → 매수
3. 나머지 → 관망

따라서 두 score가 동시에 높아도 **회피 우선**입니다.

In [ ]:
# ============================================================
# 14. 후보 decision 생성
# ============================================================

decision_parts = []

for row in thresholds.itertuples(index=False):
    g = oot_pred[
        (oot_pred["horizon"] == row.horizon)
        & (oot_pred["model"] == row.model)
    ].copy()

    risk_hit = g["risk_score"] >= row.risk_threshold
    buy_hit = g["buy_score"] >= row.buy_threshold

    g["top_q"] = row.top_q
    g["buy_threshold"] = row.buy_threshold
    g["risk_threshold"] = row.risk_threshold

    g["decision"] = np.select(
        [
            risk_hit,
            (~risk_hit) & buy_hit,
        ],
        [
            "avoid",
            "buy",
        ],
        default="hold"
    )

    decision_parts.append(g)

oot_decisions = pd.concat(
    decision_parts,
    ignore_index=True
)

display(
    oot_decisions[
        oot_decisions["top_q"] == 0.20
    ]
    .groupby(["horizon", "model", "decision"])
    .size()
    .reset_index(name="n")
)

## 15. 후보 품질 평가

각 horizon × A/B/C/D × Top q마다 다음을 계산합니다.

#### 매수 후보
- 실제 `buy_target=1` 비율 ↑
- 실제 `risk_target=1` 비율 ↓
- BHAR ↑
- MDD 덜 나쁨(0에 가까움)

#### 회피 후보
- 실제 `risk_target=1` 비율 ↑
- 실제 `buy_target=1` 비율 ↓
- BHAR 낮음
- MDD 더 나쁨

또한 `buy BHAR - avoid BHAR` gap도 계산합니다.

In [ ]:
# ============================================================
# 15. Candidate evaluation
# ============================================================

eval_rows = []

for (horizon, model_name, q), g in oot_decisions.groupby(
    ["horizon", "model", "top_q"]
):
    bhar_col = f"BHAR_{horizon}d"
    mdd_col = f"MDD_{horizon}d"

    row = {
        "horizon": horizon,
        "model": model_name,
        "top_q": q,
        "total_n": len(g),
    }

    for decision in ["buy", "avoid", "hold"]:
        d = g[g["decision"] == decision]

        prefix = decision

        row[f"{prefix}_n"] = len(d)
        row[f"{prefix}_share"] = len(d) / len(g) if len(g) else np.nan

        row[f"{prefix}_actual_buy_rate"] = (
            d["buy_target"].mean()
            if len(d) else np.nan
        )

        row[f"{prefix}_actual_risk_rate"] = (
            d["risk_target"].mean()
            if len(d) else np.nan
        )

        row[f"{prefix}_BHAR_mean"] = (
            d[bhar_col].mean()
            if len(d) else np.nan
        )

        row[f"{prefix}_BHAR_median"] = (
            d[bhar_col].median()
            if len(d) else np.nan
        )

        row[f"{prefix}_MDD_mean"] = (
            d[mdd_col].mean()
            if len(d) else np.nan
        )

        row[f"{prefix}_MDD_median"] = (
            d[mdd_col].median()
            if len(d) else np.nan
        )

    row["buy_minus_avoid_BHAR_mean_gap"] = (
        row.get("buy_BHAR_mean", np.nan)
        - row.get("avoid_BHAR_mean", np.nan)
    )

    row["buy_minus_avoid_BHAR_median_gap"] = (
        row.get("buy_BHAR_median", np.nan)
        - row.get("avoid_BHAR_median", np.nan)
    )

    eval_rows.append(row)

candidate_eval = pd.DataFrame(eval_rows)

candidate_eval.to_csv(
    CANDIDATE_EVAL_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("========== Top 20% 결과 ==========")

top20 = candidate_eval[
    candidate_eval["top_q"] == 0.20
].copy()

display(
    top20[
        [
            "horizon",
            "model",
            "buy_n",
            "buy_actual_buy_rate",
            "buy_actual_risk_rate",
            "avoid_n",
            "avoid_actual_risk_rate",
            "avoid_actual_buy_rate",
            "buy_BHAR_mean",
            "avoid_BHAR_mean",
            "buy_minus_avoid_BHAR_mean_gap",
            "buy_MDD_mean",
            "avoid_MDD_mean",
        ]
    ]
    .sort_values(["horizon", "model"])
)

## 16. A/B/C/D 비교 요약 — 메인 Top 20%

관심지표 효과:
- **A → B**
- **D → C**

공매도 효과:
- **A → D**
- **B → C**

아래 표에서는 매수 측과 회피 측의 핵심 품질 차이를 자동 계산합니다.

In [ ]:
# ============================================================
# 16. Top20 model comparison
# ============================================================

KEY_METRICS = [
    "buy_actual_buy_rate",
    "buy_actual_risk_rate",
    "avoid_actual_risk_rate",
    "avoid_actual_buy_rate",
    "buy_BHAR_mean",
    "avoid_BHAR_mean",
    "buy_minus_avoid_BHAR_mean_gap",
    "buy_MDD_mean",
    "avoid_MDD_mean",
]

COMPARES = [
    ("attention_without_short", "A", "B"),
    ("attention_with_short", "D", "C"),
    ("short_without_attention", "A", "D"),
    ("short_with_attention", "B", "C"),
]

compare_rows = []

top20_indexed = top20.set_index(["horizon", "model"])

for horizon in HORIZONS:
    for comparison, base_model, added_model in COMPARES:
        key_base = (horizon, base_model)
        key_added = (horizon, added_model)

        if key_base not in top20_indexed.index or key_added not in top20_indexed.index:
            continue

        b = top20_indexed.loc[key_base]
        a = top20_indexed.loc[key_added]

        row = {
            "horizon": horizon,
            "comparison": comparison,
            "from_model": base_model,
            "to_model": added_model,
        }

        for metric in KEY_METRICS:
            row[f"{metric}_from"] = b[metric]
            row[f"{metric}_to"] = a[metric]
            row[f"{metric}_delta"] = a[metric] - b[metric]

        compare_rows.append(row)

comparison_top20 = pd.DataFrame(compare_rows)

comparison_top20.to_csv(
    TOP20_COMPARE_PATH,
    index=False,
    encoding="utf-8-sig"
)

display(comparison_top20)

## 17. 2025와 2026을 별도로 확인

전체 OOT 결과가 좋아도 2026년에만 성능이 무너질 수 있습니다.

따라서 Top 20% 기준으로 2025 / 2026 후보 성과를 별도 확인합니다.
이 표는 **threshold를 다시 조정하지 않고**, 이미 고정한 과거 threshold를 그대로 적용한 결과입니다.

In [ ]:
# ============================================================
# 17. 연도별 OOT 평가
# ============================================================

year_rows = []

g20 = oot_decisions[
    oot_decisions["top_q"] == 0.20
].copy()

g20["year"] = g20["event_date"].dt.year

for (year, horizon, model_name), g in g20.groupby(
    ["year", "horizon", "model"]
):
    bhar_col = f"BHAR_{horizon}d"
    mdd_col = f"MDD_{horizon}d"

    buy = g[g["decision"] == "buy"]
    avoid = g[g["decision"] == "avoid"]

    year_rows.append({
        "year": year,
        "horizon": horizon,
        "model": model_name,
        "n": len(g),

        "buy_n": len(buy),
        "buy_actual_buy_rate": buy["buy_target"].mean() if len(buy) else np.nan,
        "buy_actual_risk_rate": buy["risk_target"].mean() if len(buy) else np.nan,
        "buy_BHAR_mean": buy[bhar_col].mean() if len(buy) else np.nan,
        "buy_MDD_mean": buy[mdd_col].mean() if len(buy) else np.nan,

        "avoid_n": len(avoid),
        "avoid_actual_risk_rate": avoid["risk_target"].mean() if len(avoid) else np.nan,
        "avoid_actual_buy_rate": avoid["buy_target"].mean() if len(avoid) else np.nan,
        "avoid_BHAR_mean": avoid[bhar_col].mean() if len(avoid) else np.nan,
        "avoid_MDD_mean": avoid[mdd_col].mean() if len(avoid) else np.nan,
    })

year_eval = pd.DataFrame(year_rows)

display(
    year_eval.sort_values(
        ["year", "horizon", "model"]
    )
)

## 18. Feature importance 저장 / 확인

최종 학습된 LightGBM 모델의 gain importance를 확인합니다.

나중에 대시보드에서 SHAP을 사용할 수 있도록 `.joblib` 모델도 이미 저장했습니다.

In [ ]:
# ============================================================
# 18. Feature importance
# ============================================================

importance_rows = []

for model_path in sorted(MODEL_DIR.glob("*.joblib")):
    obj = joblib.load(model_path)

    clf = obj["model"]
    features = obj["features"]

    booster = clf.booster_
    gains = booster.feature_importance(
        importance_type="gain"
    )

    total_gain = gains.sum()

    for feature, gain in zip(features, gains):
        importance_rows.append({
            "horizon": obj["horizon"],
            "model": obj["model_name"],
            "target": obj["target"],
            "feature": feature,
            "gain": gain,
            "gain_share": (
                gain / total_gain
                if total_gain > 0
                else np.nan
            ),
        })

feature_importance = pd.DataFrame(
    importance_rows
)

importance_path = RUN_DIR / "feature_importance_gain.csv"

feature_importance.to_csv(
    importance_path,
    index=False,
    encoding="utf-8-sig"
)

for horizon in HORIZONS:
    print(f"\n========== {horizon}D C model BUY top features ==========")

    display(
        feature_importance[
            (feature_importance["horizon"] == horizon)
            & (feature_importance["model"] == "C")
            & (feature_importance["target"] == "buy")
        ]
        .sort_values("gain_share", ascending=False)
        .head(15)
    )

    print(f"{horizon}D C model RISK top features")

    display(
        feature_importance[
            (feature_importance["horizon"] == horizon)
            & (feature_importance["model"] == "C")
            & (feature_importance["target"] == "risk")
        ]
        .sort_values("gain_share", ascending=False)
        .head(15)
    )

## 19. 최종 산출물

실행이 끝나면:

```text
cache/modeling_preparation_v1/
└── modeling_runs/
    └── ewma_hl10_k1p5/
        ├── event_features_ABCD_v1.parquet
        ├── feature_manifest.json
        ├── predictions_oof_2021_2024.parquet
        ├── predictions_oot_2025_2026.parquet
        ├── score_thresholds_top10_20_30.csv
        ├── model_metrics.csv
        ├── candidate_evaluation_oot.csv
        ├── comparison_summary_top20.csv
        ├── feature_importance_gain.csv
        └── models/
            └── *.joblib
```

가 생성됩니다.

#### 결과를 볼 때 우선순위

메인 가설은 **관심지표 추가효과**입니다.

1. `A → B`
2. `D → C`

두 비교에서 동시에:
- 매수 후보의 실제 buy target 비율 ↑
- 매수 후보 risk target 비율 ↓
- 회피 후보 risk target 비율 ↑
- 회피 후보 buy target 비율 ↓
- BHAR gap 개선

이 나타나는지를 우선 확인합니다.

공매도 효과는:
- `A → D`
- `B → C`

로 별도 확인합니다.

In [ ]:
# ============================================================
# 19. 최종 저장 확인
# ============================================================

artifacts = [
    FEATURE_PATH,
    FEATURE_MANIFEST_PATH,
    OOF_PRED_PATH,
    OOT_PRED_PATH,
    THRESHOLD_PATH,
    MODEL_METRIC_PATH,
    CANDIDATE_EVAL_PATH,
    TOP20_COMPARE_PATH,
    RUN_DIR / "feature_importance_gain.csv",
]

print("========== 산출물 ==========")

for p in artifacts:
    print(
        "✅" if p.exists() else "❌",
        p
    )

print("\n모델 파일 수:", len(list(MODEL_DIR.glob("*.joblib"))))
print("\nRUN_DIR:")
print(RUN_DIR)

print("\n✅ A/B/C/D 모델링 파이프라인 완료")

## STAGE 9-B — 9개 EWMA 장벽전략 OOF 강건성 비교 (구 02번)

In [ ]:
# ============================================================
# 0. Drive + 패키지
# ============================================================

from pathlib import Path
import sys
import subprocess
import importlib.util
import json
import warnings

if importlib.util.find_spec("lightgbm") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "lightgbm"
    ])

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("pandas:", pd.__version__)
print("lightgbm:", lgb.__version__)

In [ ]:
# ============================================================
# 1. 경로 / 설정
# ============================================================

BASE = WORKSPACE_ROOT
PREP_DIR = BASE / "cache/modeling_preparation_v1"

# 이전 A/B/C/D 모델링 노트북에서 생성된 공통 feature table 사용
BASELINE_RUN_DIR = (
    PREP_DIR
    / "modeling_runs"
    / "ewma_hl10_k1p5"
)

FEATURE_PATH = BASELINE_RUN_DIR / "event_features_ABCD_v1.parquet"
FEATURE_MANIFEST_PATH = BASELINE_RUN_DIR / "feature_manifest.json"

TARGET_ROOT = PREP_DIR / "barrier_targets"

OUT_DIR = PREP_DIR / "barrier_strategy_comparison_oof"
OUT_DIR.mkdir(parents=True, exist_ok=True)

HORIZONS = [10, 20, 60]
TOP_Q_LIST = [0.10, 0.20, 0.30]

# 2025 이후는 전략 선택에 사용하지 않음
SELECTION_END = pd.Timestamp("2024-12-31")

CV_WINDOWS = [
    ("2022H2", "2022-07-01", "2022-12-31"),
    ("2023H1", "2023-01-01", "2023-06-30"),
    ("2023H2", "2023-07-01", "2023-12-31"),
    ("2024H1", "2024-01-01", "2024-06-30"),
    ("2024H2", "2024-07-01", "2024-12-31"),
]

print("FEATURE:", FEATURE_PATH.exists(), FEATURE_PATH)
print("MANIFEST:", FEATURE_MANIFEST_PATH.exists(), FEATURE_MANIFEST_PATH)
print("TARGET_ROOT:", TARGET_ROOT.exists(), TARGET_ROOT)
print("OUT_DIR:", OUT_DIR)

if not FEATURE_PATH.exists():
    raise FileNotFoundError(
        "event_features_ABCD_v1.parquet가 없습니다.\n"
        "먼저 기존 A/B/C/D 모델링 노트북을 feature 저장 셀까지 실행하세요."
    )

if not FEATURE_MANIFEST_PATH.exists():
    raise FileNotFoundError(FEATURE_MANIFEST_PATH)

### 2. 9개 target 전략 자동 검색

폴더 이름을 직접 하드코딩하지 않고 `barrier_targets/*/targets.parquet`를 검색합니다.

In [ ]:
# ============================================================
# 2. target 전략 검색
# ============================================================

strategy_paths = sorted(TARGET_ROOT.glob("*/targets.parquet"))

if not strategy_paths:
    raise FileNotFoundError("barrier target 파일이 없습니다.")

strategy_map = {
    p.parent.name: p
    for p in strategy_paths
}

print("전략 수:", len(strategy_map))

for name in strategy_map:
    print("✅", name)

EXPECTED = {
    "ewma_hl5_k1p0", "ewma_hl5_k1p5", "ewma_hl5_k2p0",
    "ewma_hl10_k1p0", "ewma_hl10_k1p5", "ewma_hl10_k2p0",
    "ewma_hl20_k1p0", "ewma_hl20_k1p5", "ewma_hl20_k2p0",
}

missing_expected = EXPECTED - set(strategy_map)

if missing_expected:
    print("\n⚠️ 예상 전략 중 누락:")
    print(sorted(missing_expected))

### 3. 공통 feature / A·B·C·D feature set 로드

9개 전략의 차이는 **target만 다르고 feature는 완전히 동일**해야 합니다.

그래서 이전 모델링 노트북에서 만든 동일한 사건 feature table과 feature manifest를 재사용합니다.

In [ ]:
# ============================================================
# 3. 공통 feature table
# ============================================================

features = pd.read_parquet(FEATURE_PATH)

with open(FEATURE_MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)

FEATURE_SETS = manifest["feature_sets"]

features["event_date"] = pd.to_datetime(
    features["event_date"],
    errors="coerce"
)

print("feature shape:", features.shape)
print("사건 수:", features["event_id"].nunique())
print("기간:", features["event_date"].min(), "~", features["event_date"].max())

for model_name, cols in FEATURE_SETS.items():
    print(model_name, len(cols), "features")

In [ ]:
# ============================================================
# 4. 공통 함수
# ============================================================

def clean_numeric_frame(X):
    X = X.copy()

    for c in X.columns:
        if pd.api.types.is_bool_dtype(X[c]):
            X[c] = X[c].astype("int8")
        else:
            X[c] = pd.to_numeric(X[c], errors="coerce")

    X = X.replace([np.inf, -np.inf], np.nan)

    keep = []

    for c in X.columns:
        if X[c].notna().sum() == 0:
            continue

        if X[c].nunique(dropna=True) <= 1:
            continue

        keep.append(c)

    return X[keep]


def make_lgbm():
    return lgb.LGBMClassifier(
        objective="binary",
        n_estimators=350,
        learning_rate=0.03,
        num_leaves=15,
        max_depth=4,
        min_child_samples=20,
        subsample=0.90,
        colsample_bytree=0.90,
        reg_alpha=0.10,
        reg_lambda=0.50,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
    )


def metric_dict(y_true, score):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(score, dtype=float)

    valid = np.isfinite(y) & np.isfinite(s)
    y = y[valid]
    s = s[valid]

    out = {
        "n": len(y),
        "positive_rate": np.nan,
        "roc_auc": np.nan,
        "pr_auc": np.nan,
        "pr_lift_vs_prevalence": np.nan,
        "balanced_accuracy_0p5": np.nan,
        "precision_0p5": np.nan,
        "recall_0p5": np.nan,
        "f1_0p5": np.nan,
    }

    if len(y) == 0:
        return out

    prevalence = y.mean()
    out["positive_rate"] = prevalence

    pred = (s >= 0.5).astype(int)

    if np.unique(y).size == 2:
        out["roc_auc"] = roc_auc_score(y, s)
        out["pr_auc"] = average_precision_score(y, s)

        if prevalence > 0:
            out["pr_lift_vs_prevalence"] = (
                out["pr_auc"] / prevalence
            )

        out["balanced_accuracy_0p5"] = balanced_accuracy_score(y, pred)
        out["precision_0p5"] = precision_score(y, pred, zero_division=0)
        out["recall_0p5"] = recall_score(y, pred, zero_division=0)
        out["f1_0p5"] = f1_score(y, pred, zero_division=0)

    return out


def make_purged_folds(d, horizon):
    end_col = f"outcome_end_date_{horizon}d"

    folds = []

    for fold_name, start, end in CV_WINDOWS:
        val_start = pd.Timestamp(start)
        val_end = pd.Timestamp(end)

        val_mask = (
            (d["event_date"] >= val_start)
            & (d["event_date"] <= val_end)
        )

        train_mask = (
            (d["event_date"] < val_start)
            & (d[end_col] < val_start)
        )

        train_idx = d.index[train_mask].to_numpy()
        val_idx = d.index[val_mask].to_numpy()

        if len(train_idx) == 0 or len(val_idx) == 0:
            continue

        train_dates = set(d.loc[train_idx, "event_date"])
        val_dates = set(d.loc[val_idx, "event_date"])

        if train_dates & val_dates:
            raise AssertionError(
                f"{fold_name}: 동일 event_date가 train/val에 동시에 존재"
            )

        folds.append({
            "fold": fold_name,
            "train_idx": train_idx,
            "val_idx": val_idx,
        })

    return folds

### 5. 먼저 target 자체의 건강성 확인

모델을 돌리기 전에 각 전략의:
- `buy_target=1` 비율
- `risk_target=1` 비율
- 둘 다 0인 `neither` 비율
- 유효 사건 수

를 2021~2024에서 확인합니다.

장벽이 너무 좁으면 거의 모든 사건이 빠르게 upper/lower를 터치하고,
너무 넓으면 대부분 `neither`가 될 수 있습니다.

In [ ]:
# ============================================================
# 5. target 분포 진단 — 2021~2024 only
# ============================================================

target_health_rows = []

for strategy, path in strategy_map.items():
    t = pd.read_parquet(path)

    t["event_date"] = pd.to_datetime(
        t["event_date"],
        errors="coerce"
    )

    t = t[
        (t["event_date"] <= SELECTION_END)
        & t["eligible_horizon"]
        & t["buy_target"].notna()
        & t["risk_target"].notna()
    ].copy()

    for h, d in t.groupby("horizon"):
        target_health_rows.append({
            "strategy": strategy,
            "horizon": int(h),
            "n": len(d),
            "buy_rate": d["buy_target"].mean(),
            "risk_rate": d["risk_target"].mean(),
            "neither_rate": (d["first_touch"] == "neither").mean(),
            "upper_first_rate": (d["first_touch"] == "upper").mean(),
            "lower_first_rate": (d["first_touch"] == "lower").mean(),
        })

target_health = pd.DataFrame(target_health_rows)

display(
    target_health.sort_values(
        ["horizon", "strategy"]
    )
)

target_health.to_csv(
    OUT_DIR / "01_target_health_2021_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

## 6. 9개 전략 × 3 horizon × A/B/C/D OOF 학습

**2025~2026은 전혀 사용하지 않습니다.**

각 전략별로 동일한 purged expanding walk-forward fold를 사용하여
`buy_score`, `risk_score` OOF prediction을 생성합니다.

In [ ]:
# ============================================================
# 6. 전략별 OOF prediction
# ============================================================

all_oof_parts = []
all_metric_rows = []

for strategy, target_path in strategy_map.items():
    print("\n" + "=" * 100)
    print("STRATEGY:", strategy)
    print("=" * 100)

    targets = pd.read_parquet(target_path)
    targets["event_date"] = pd.to_datetime(
        targets["event_date"],
        errors="coerce"
    )

    for horizon in HORIZONS:
        end_col = f"outcome_end_date_{horizon}d"

        t = targets[
            (targets["horizon"] == horizon)
            & targets["eligible_horizon"]
            & targets["buy_target"].notna()
            & targets["risk_target"].notna()
            & (targets["event_date"] <= SELECTION_END)
        ][
            [
                "event_id",
                "buy_target",
                "risk_target",
                "first_touch",
            ]
        ].copy()

        d = features.merge(
            t,
            on="event_id",
            how="inner",
            validate="one_to_one"
        )

        d[end_col] = pd.to_datetime(
            d[end_col],
            errors="coerce"
        )

        d = d[
            (d["event_date"] <= SELECTION_END)
            & d[end_col].notna()
        ].copy()

        folds = make_purged_folds(d, horizon)

        print(
            f"{horizon}D: n={len(d):,}, folds={len(folds)}"
        )

        for model_name, feature_cols in FEATURE_SETS.items():
            X_all = clean_numeric_frame(
                d[feature_cols]
            )

            buy_score = pd.Series(
                np.nan,
                index=d.index,
                dtype=float
            )

            risk_score = pd.Series(
                np.nan,
                index=d.index,
                dtype=float
            )

            fold_store = pd.Series(
                pd.NA,
                index=d.index,
                dtype="object"
            )

            for fold in folds:
                tr = fold["train_idx"]
                va = fold["val_idx"]

                X_train = X_all.loc[tr]
                X_val = X_all.loc[va]

                fold_store.loc[va] = fold["fold"]

                for target_col, score_store in [
                    ("buy_target", buy_score),
                    ("risk_target", risk_score),
                ]:
                    y_train = d.loc[tr, target_col].astype(int)

                    if y_train.nunique() < 2:
                        continue

                    clf = make_lgbm()
                    clf.fit(
                        X_train,
                        y_train
                    )

                    score_store.loc[va] = (
                        clf.predict_proba(X_val)[:, 1]
                    )

            valid = (
                buy_score.notna()
                & risk_score.notna()
            )

            tmp = d.loc[
                valid,
                [
                    "event_id",
                    "ticker",
                    "ticker_name",
                    "event_date",
                    "buy_target",
                    "risk_target",
                    f"BHAR_{horizon}d",
                    f"MDD_{horizon}d",
                ]
            ].copy()

            tmp["strategy"] = strategy
            tmp["horizon"] = horizon
            tmp["model"] = model_name
            tmp["fold"] = fold_store.loc[valid].values
            tmp["buy_score"] = buy_score.loc[valid].values
            tmp["risk_score"] = risk_score.loc[valid].values

            all_oof_parts.append(tmp)

            for target_name, target_col, score_col in [
                ("buy", "buy_target", "buy_score"),
                ("risk", "risk_target", "risk_score"),
            ]:
                m = metric_dict(
                    tmp[target_col],
                    tmp[score_col]
                )

                all_metric_rows.append({
                    "strategy": strategy,
                    "horizon": horizon,
                    "model": model_name,
                    "target": target_name,
                    **m,
                })

oof = pd.concat(
    all_oof_parts,
    ignore_index=True
)

metrics = pd.DataFrame(
    all_metric_rows
)

print("\nOOF total shape:", oof.shape)

oof.to_parquet(
    OUT_DIR / "02_all_strategy_oof_predictions.parquet",
    index=False,
    compression="zstd"
)

metrics.to_csv(
    OUT_DIR / "03_all_strategy_oof_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    metrics.sort_values(
        ["horizon", "strategy", "model", "target"]
    )
)

### 7. OOF ROC-AUC / PR-AUC 요약

전략 간 PR-AUC는 target prevalence가 달라 직접 비교가 조심스러우므로
`PR-AUC ÷ positive rate`인 **PR lift**도 같이 봅니다.

- ROC-AUC > 0.5: 랜덤보다 방향성 있음
- PR lift > 1: 해당 target의 단순 prevalence baseline보다 개선

In [ ]:
# ============================================================
# 7. metric pivot
# ============================================================

metric_pivot = (
    metrics
    .pivot_table(
        index=["strategy", "horizon"],
        columns=["model", "target"],
        values=[
            "roc_auc",
            "pr_auc",
            "pr_lift_vs_prevalence",
        ]
    )
)

display(metric_pivot)

metric_pivot.to_csv(
    OUT_DIR / "04_oof_metric_pivot.csv",
    encoding="utf-8-sig"
)

## 8. OOF Top 10 / 20 / 30% 후보 생성

여기서는 각 전략의 OOF score 자체에서 percentile 경계를 만들고,
동일한 OOF prediction에서 후보 성격을 비교합니다.

이 값은 **전략 선택용 2021~2024 내부 진단**이며 2025~2026을 보지 않습니다.

In [ ]:
# ============================================================
# 8. OOF 후보 결정
# ============================================================

decision_parts = []

for (strategy, horizon, model_name), g in oof.groupby(
    ["strategy", "horizon", "model"]
):
    for q in TOP_Q_LIST:
        buy_threshold = g["buy_score"].quantile(1 - q)
        risk_threshold = g["risk_score"].quantile(1 - q)

        d = g.copy()

        risk_hit = (
            d["risk_score"] >= risk_threshold
        )

        buy_hit = (
            d["buy_score"] >= buy_threshold
        )

        d["top_q"] = q
        d["buy_threshold"] = buy_threshold
        d["risk_threshold"] = risk_threshold

        d["decision"] = np.select(
            [
                risk_hit,
                (~risk_hit) & buy_hit,
            ],
            [
                "avoid",
                "buy",
            ],
            default="hold"
        )

        decision_parts.append(d)

oof_decisions = pd.concat(
    decision_parts,
    ignore_index=True
)

print("OOF decisions:", oof_decisions.shape)

In [ ]:
# ============================================================
# 9. OOF 후보 품질 평가
# ============================================================

candidate_rows = []

for (strategy, horizon, model_name, q), g in oof_decisions.groupby(
    ["strategy", "horizon", "model", "top_q"]
):
    bhar_col = f"BHAR_{horizon}d"
    mdd_col = f"MDD_{horizon}d"

    row = {
        "strategy": strategy,
        "horizon": horizon,
        "model": model_name,
        "top_q": q,
        "total_n": len(g),
    }

    for decision in ["buy", "avoid", "hold"]:
        d = g[
            g["decision"] == decision
        ]

        row[f"{decision}_n"] = len(d)

        row[f"{decision}_actual_buy_rate"] = (
            d["buy_target"].mean()
            if len(d) else np.nan
        )

        row[f"{decision}_actual_risk_rate"] = (
            d["risk_target"].mean()
            if len(d) else np.nan
        )

        row[f"{decision}_BHAR_mean"] = (
            d[bhar_col].mean()
            if len(d) else np.nan
        )

        row[f"{decision}_MDD_mean"] = (
            d[mdd_col].mean()
            if len(d) else np.nan
        )

    row["buy_minus_avoid_BHAR_gap"] = (
        row.get("buy_BHAR_mean", np.nan)
        - row.get("avoid_BHAR_mean", np.nan)
    )

    candidate_rows.append(row)

candidate_eval = pd.DataFrame(
    candidate_rows
)

candidate_eval.to_csv(
    OUT_DIR / "05_oof_candidate_evaluation.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    candidate_eval[
        candidate_eval["top_q"] == 0.20
    ].sort_values(
        ["horizon", "strategy", "model"]
    )
)

## 10. 관심지표 효과 A→B / D→C

메인 가설에 맞춰 관심지표 추가 전후의 변화량만 뽑습니다.

좋은 방향:

- `buy_actual_buy_rate` : **증가**
- `buy_actual_risk_rate` : **감소**
- `avoid_actual_risk_rate` : **증가**
- `avoid_actual_buy_rate` : **감소**
- `buy_minus_avoid_BHAR_gap` : 보조적으로 **증가**

특히 20D Top20을 메인으로 봅니다.

In [ ]:
# ============================================================
# 10. 관심 효과 비교
# ============================================================

ATTENTION_COMPARES = [
    ("A_to_B", "A", "B"),
    ("D_to_C", "D", "C"),
]

compare_rows = []

idx = candidate_eval.set_index(
    ["strategy", "horizon", "top_q", "model"]
)

for strategy in strategy_map:
    for horizon in HORIZONS:
        for q in TOP_Q_LIST:
            for comparison, from_model, to_model in ATTENTION_COMPARES:
                k_from = (
                    strategy, horizon, q, from_model
                )
                k_to = (
                    strategy, horizon, q, to_model
                )

                if (
                    k_from not in idx.index
                    or k_to not in idx.index
                ):
                    continue

                a = idx.loc[k_from]
                b = idx.loc[k_to]

                row = {
                    "strategy": strategy,
                    "horizon": horizon,
                    "top_q": q,
                    "comparison": comparison,
                }

                metrics_to_compare = [
                    "buy_actual_buy_rate",
                    "buy_actual_risk_rate",
                    "avoid_actual_risk_rate",
                    "avoid_actual_buy_rate",
                    "buy_minus_avoid_BHAR_gap",
                    "buy_MDD_mean",
                    "avoid_MDD_mean",
                ]

                for m in metrics_to_compare:
                    row[f"{m}_from"] = a[m]
                    row[f"{m}_to"] = b[m]
                    row[f"{m}_delta"] = (
                        b[m] - a[m]
                    )

                # 4개 target 방향성 점수
                correct_signs = [
                    row["buy_actual_buy_rate_delta"] > 0,
                    row["buy_actual_risk_rate_delta"] < 0,
                    row["avoid_actual_risk_rate_delta"] > 0,
                    row["avoid_actual_buy_rate_delta"] < 0,
                ]

                row["target_direction_score"] = int(
                    sum(correct_signs)
                )

                compare_rows.append(row)

attention_compare = pd.DataFrame(
    compare_rows
)

attention_compare.to_csv(
    OUT_DIR / "06_attention_effect_oof.csv",
    index=False,
    encoding="utf-8-sig"
)

print("========== 20D / Top20 관심 효과 ==========")

main20 = attention_compare[
    (attention_compare["horizon"] == 20)
    & (attention_compare["top_q"] == 0.20)
].copy()

display(
    main20[
        [
            "strategy",
            "comparison",
            "target_direction_score",
            "buy_actual_buy_rate_delta",
            "buy_actual_risk_rate_delta",
            "avoid_actual_risk_rate_delta",
            "avoid_actual_buy_rate_delta",
            "buy_minus_avoid_BHAR_gap_delta",
        ]
    ].sort_values(
        [
            "target_direction_score",
            "strategy",
            "comparison",
        ],
        ascending=[False, True, True]
    )
)

## 11. 전략별 20D 관심효과 강건성 점수

A→B와 D→C 각각 4개의 target 방향성 지표가 있으므로
한 전략당 최대 **8점**입니다.

이 점수는 전략 자동 확정용이 아니라
**“관심지표 효과가 얼마나 일관되게 같은 방향인가”를 빠르게 보는 요약**입니다.

In [ ]:
# ============================================================
# 11. 20D Top20 robustness summary
# ============================================================

robustness_20d = (
    main20
    .groupby("strategy")
    .agg(
        attention_direction_score_8=(
            "target_direction_score",
            "sum"
        ),
        AB_direction_score_4=(
            "target_direction_score",
            lambda s: (
                s[
                    main20.loc[s.index, "comparison"]
                    == "A_to_B"
                ].sum()
            )
        ),
        DC_direction_score_4=(
            "target_direction_score",
            lambda s: (
                s[
                    main20.loc[s.index, "comparison"]
                    == "D_to_C"
                ].sum()
            )
        ),
        mean_BHAR_gap_delta=(
            "buy_minus_avoid_BHAR_gap_delta",
            "mean"
        ),
    )
    .reset_index()
    .sort_values(
        [
            "attention_direction_score_8",
            "mean_BHAR_gap_delta",
        ],
        ascending=False
    )
)

display(robustness_20d)

robustness_20d.to_csv(
    OUT_DIR / "07_20D_top20_attention_robustness.csv",
    index=False,
    encoding="utf-8-sig"
)

## 12. 전략 선택용 OOF 요약표

최종 선택은 하나의 숫자만 보고 하지 않는 것을 권장합니다.

우선 확인 순서:

1. **target health**
   - buy/risk/neither가 지나치게 한쪽으로 쏠리지 않는가
2. **OOF 모델 분류력**
   - B/C의 buy/risk ROC-AUC가 랜덤 0.5보다 안정적으로 높은가
   - PR lift가 1보다 높은가
3. **20D 관심효과**
   - A→B와 D→C에서 target direction score가 높은가
4. BHAR/MDD는 보조 확인

아래 표는 이 핵심 정보만 한 번에 붙입니다.

In [ ]:
# ============================================================
# 12. 최종 OOF selection summary
# ============================================================

# 20D target health
health20 = target_health[
    target_health["horizon"] == 20
][
    [
        "strategy",
        "n",
        "buy_rate",
        "risk_rate",
        "neither_rate",
    ]
].copy()

# 20D B/C 평균 ROC / PR lift
m20 = metrics[
    (metrics["horizon"] == 20)
    & (metrics["model"].isin(["B", "C"]))
].copy()

metric20 = (
    m20.groupby("strategy")
    .agg(
        BC_mean_roc_auc=("roc_auc", "mean"),
        BC_min_roc_auc=("roc_auc", "min"),
        BC_mean_pr_lift=("pr_lift_vs_prevalence", "mean"),
        BC_min_pr_lift=("pr_lift_vs_prevalence", "min"),
    )
    .reset_index()
)

selection_summary = (
    health20
    .merge(
        metric20,
        on="strategy",
        how="left"
    )
    .merge(
        robustness_20d,
        on="strategy",
        how="left"
    )
)

selection_summary = selection_summary.sort_values(
    [
        "attention_direction_score_8",
        "BC_mean_roc_auc",
        "BC_mean_pr_lift",
    ],
    ascending=False
)

display(selection_summary)

selection_summary.to_csv(
    OUT_DIR / "08_strategy_selection_summary_OOF_ONLY.csv",
    index=False,
    encoding="utf-8-sig"
)

## 13. 저장 결과

실행 후 아래 파일들을 보면 됩니다.

```text
barrier_strategy_comparison_oof/
├── 01_target_health_2021_2024.csv
├── 02_all_strategy_oof_predictions.parquet
├── 03_all_strategy_oof_metrics.csv
├── 04_oof_metric_pivot.csv
├── 05_oof_candidate_evaluation.csv
├── 06_attention_effect_oof.csv
├── 07_20D_top20_attention_robustness.csv
└── 08_strategy_selection_summary_OOF_ONLY.csv
```

가장 먼저 보여줄 파일은:

1. `08_strategy_selection_summary_OOF_ONLY.csv`
2. `07_20D_top20_attention_robustness.csv`
3. 필요하면 `03_all_strategy_oof_metrics.csv`

입니다.

**이 노트북에서는 2025~2026을 이용한 전략 선택을 하지 않습니다.**

In [ ]:
# ============================================================
# 13. 산출물 확인
# ============================================================

artifacts = sorted(OUT_DIR.glob("*"))

print("========== 생성 파일 ==========")

for p in artifacts:
    print("✅", p.name)

print("\nOUT_DIR:")
print(OUT_DIR)

print("\n✅ 9개 장벽 전략 OOF 강건성 비교 완료")

## STAGE 9-C — 9개 전략 강건성 결과 시각화 & 해석 (구 03번)

In [ ]:
# ============================================================
# 0. Drive 연결 + 경로
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

BASE = WORKSPACE_ROOT
OUT_DIR = (
    BASE
    / "cache/modeling_preparation_v1/barrier_strategy_comparison_oof"
)

FILES = {
    "target_health": OUT_DIR / "01_target_health_2021_2024.csv",
    "oof_predictions": OUT_DIR / "02_all_strategy_oof_predictions.parquet",
    "metrics": OUT_DIR / "03_all_strategy_oof_metrics.csv",
    "metric_pivot": OUT_DIR / "04_oof_metric_pivot.csv",
    "candidate_eval": OUT_DIR / "05_oof_candidate_evaluation.csv",
    "attention_effect": OUT_DIR / "06_attention_effect_oof.csv",
    "robustness": OUT_DIR / "07_20D_top20_attention_robustness.csv",
    "selection": OUT_DIR / "08_strategy_selection_summary_OOF_ONLY.csv",
}

print("========== 파일 확인 ==========")
for name, path in FILES.items():
    print("✅" if path.exists() else "❌", name, "->", path)

required = [
    FILES["target_health"],
    FILES["metrics"],
    FILES["candidate_eval"],
    FILES["attention_effect"],
    FILES["robustness"],
    FILES["selection"],
]

missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "필수 산출물이 없습니다:\n" + "\n".join(missing)
    )

In [ ]:
# ============================================================
# 1. 데이터 로드
# ============================================================

target_health = pd.read_csv(FILES["target_health"])
metrics = pd.read_csv(FILES["metrics"])
candidate_eval = pd.read_csv(FILES["candidate_eval"])
attention_effect = pd.read_csv(FILES["attention_effect"])
robustness = pd.read_csv(FILES["robustness"])
selection = pd.read_csv(FILES["selection"])

print("target_health:", target_health.shape)
print("metrics:", metrics.shape)
print("candidate_eval:", candidate_eval.shape)
print("attention_effect:", attention_effect.shape)
print("robustness:", robustness.shape)
print("selection:", selection.shape)

display(selection)

## 1. 전략 선택 요약표를 먼저 읽는 법

`08_strategy_selection_summary_OOF_ONLY.csv`는 **20D 기준 핵심 정보만 압축한 표**입니다.

#### `n`
20D에서 target을 계산할 수 있었던 2021~2024 사건 수입니다.

#### `buy_rate`
상승 장벽을 하락 장벽보다 먼저 터치한 비율입니다.

#### `risk_rate`
하락 장벽을 상승 장벽보다 먼저 터치한 비율입니다.

#### `neither_rate`
20거래일 동안 둘 다 터치하지 않은 비율입니다.

#### `BC_mean_roc_auc`
관심지표가 들어간 B/C 모델의 buy/risk OOF ROC-AUC 평균입니다.

#### `BC_min_roc_auc`
B/C의 여러 분류 중 **가장 낮은 ROC-AUC**입니다.

평균만 좋은 전략보다 최악의 경우도 0.5 이상인 전략이 더 안정적입니다.

#### `BC_mean_pr_lift`
PR-AUC를 해당 target의 원래 발생비율로 나눈 값의 평균입니다.

- 1보다 크면 prevalence baseline보다 낫다는 뜻
- 1보다 작으면 positive를 특별히 잘 골랐다고 보기 어려움

#### `attention_direction_score_8`
20D Top20에서 관심지표를 추가했을 때 기대 방향으로 개선된 지표 개수입니다.

- A→B : 최대 4점
- D→C : 최대 4점
- 합계 : 최대 8점

#### `mean_BHAR_gap_delta`
관심지표 추가 후 **매수 후보 BHAR - 회피 후보 BHAR 격차가 평균적으로 얼마나 개선됐는지**입니다.
양수일수록 경제적 분리 측면에서 좋은 방향입니다.

## 2. Target Health — 장벽이 너무 쉽거나 어려운가?

좋은 장벽은 무조건 `buy/risk/neither`가 정확히 1/3일 필요는 없습니다.

다만:

- `neither ≈ 0%` → 장벽이 너무 좁아 거의 모든 사건이 쉽게 분류될 가능성
- `neither`가 지나치게 높음 → 장벽이 너무 넓어 target이 잘 안 생김

을 의심할 수 있습니다.

현재 9개 전략의 20D target 구성을 시각화합니다.

In [ ]:
# ============================================================
# 2. 20D Target Health
# ============================================================

health20 = target_health[
    target_health["horizon"].eq(20)
].copy()

# selection 순서와 맞춤
order = selection["strategy"].tolist()
health20["strategy"] = pd.Categorical(
    health20["strategy"],
    categories=order,
    ordered=True
)
health20 = health20.sort_values("strategy")

plot_df = health20.set_index("strategy")[
    ["buy_rate", "risk_rate", "neither_rate"]
]

ax = plot_df.plot(
    kind="bar",
    stacked=True,
    figsize=(13, 6)
)

ax.set_title("20D — 장벽 전략별 Target 구성")
ax.set_xlabel("Strategy")
ax.set_ylabel("Rate")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

display(
    health20[
        ["strategy", "n", "buy_rate", "risk_rate", "neither_rate"]
    ]
)

#### 해석 포인트

예를 들어 `neither_rate = 0.004`라면 **0.4%만 미도달**이라는 뜻입니다.

즉 그 전략은 거의 모든 사건이 20D 안에 위/아래 장벽 중 하나를 터치하므로,
**장벽이 상당히 좁은 편**이라고 볼 수 있습니다.

반대로 `neither_rate = 0.133`이면 13.3%가 미도달이라서
장벽이 조금 더 까다롭게 설정된 것입니다.

## 3. OOF ROC-AUC — 모델이 이 Target을 배울 수 있는가?

여기서 가장 중요한 질문은:

> **“이 장벽으로 만든 target이 feature들로 실제 예측 가능한가?”**

입니다.

방향성 점수가 높더라도 ROC-AUC가 0.5보다 낮으면,
관심지표를 넣었을 때 일부 후보군 지표가 우연히 좋아졌을 가능성을 배제하기 어렵습니다.

따라서 전략 선택에서 **OOF 분류력을 함께 보는 것이 중요**합니다.

In [ ]:
# ============================================================
# 3-1. 20D B/C ROC-AUC
# ============================================================

m20 = metrics[
    metrics["horizon"].eq(20)
    & metrics["model"].isin(["B", "C"])
].copy()

# B/C × buy/risk
m20["series"] = (
    m20["model"].astype(str)
    + "_"
    + m20["target"].astype(str)
)

pivot_auc = m20.pivot(
    index="strategy",
    columns="series",
    values="roc_auc"
)

pivot_auc = pivot_auc.reindex(order)

ax = pivot_auc.plot(
    kind="bar",
    figsize=(14, 6)
)

ax.axhline(0.5, linestyle="--", linewidth=1)
ax.set_title("20D — B/C 모델 OOF ROC-AUC")
ax.set_xlabel("Strategy")
ax.set_ylabel("ROC-AUC")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 3-2. 전략별 Mean / Min ROC-AUC
# ============================================================

d = selection.set_index("strategy").reindex(order)

ax = d[
    ["BC_mean_roc_auc", "BC_min_roc_auc"]
].plot(
    kind="bar",
    figsize=(13, 6)
)

ax.axhline(0.5, linestyle="--", linewidth=1)
ax.set_title("20D — B/C OOF ROC-AUC 평균과 최저값")
ax.set_xlabel("Strategy")
ax.set_ylabel("ROC-AUC")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 4. PR Lift — 실제 positive를 baseline보다 잘 찾는가?

PR-AUC는 target의 원래 발생비율에 영향을 받습니다.

그래서 전략 간 비교에서는:

`PR lift = PR-AUC / positive prevalence`

를 사용합니다.

- **1.0** : 단순 발생비율 수준
- **> 1.0** : positive를 baseline보다 잘 찾음
- **< 1.0** : 특별한 개선이 없거나 오히려 약함

`BC_min_pr_lift > 1`이면 B/C의 buy/risk 중 가장 약한 경우까지 baseline보다 낫다는 뜻이라 꽤 보수적인 기준입니다.

In [ ]:
# ============================================================
# 4. PR lift 시각화
# ============================================================

d = selection.set_index("strategy").reindex(order)

ax = d[
    ["BC_mean_pr_lift", "BC_min_pr_lift"]
].plot(
    kind="bar",
    figsize=(13, 6)
)

ax.axhline(1.0, linestyle="--", linewidth=1)
ax.set_title("20D — B/C PR Lift 평균과 최저값")
ax.set_xlabel("Strategy")
ax.set_ylabel("PR Lift")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 5. 관심지표 방향성 점수 — A→B, D→C

FinDA의 핵심 가설은 관심지표의 추가효과입니다.

20D Top20에서 관심지표 추가 후 다음 4가지가 좋아지는지 확인합니다.

1. 매수 후보의 `buy_target` 비율 ↑
2. 매수 후보의 `risk_target` 비율 ↓
3. 회피 후보의 `risk_target` 비율 ↑
4. 회피 후보의 `buy_target` 비율 ↓

이를:
- A→B에서 최대 4점
- D→C에서 최대 4점

으로 계산한 것이 `attention_direction_score_8`입니다.

하지만 **점수만 높다고 최종 전략으로 고르면 안 됩니다.**
OOF ROC-AUC와 PR lift가 같이 받쳐줘야 합니다.

In [ ]:
# ============================================================
# 5. 방향성 점수
# ============================================================

r = robustness.set_index("strategy").reindex(order)

ax = r[
    ["AB_direction_score_4", "DC_direction_score_4"]
].plot(
    kind="bar",
    stacked=True,
    figsize=(13, 6)
)

ax.set_title("20D Top20 — 관심지표 추가효과 방향성 점수")
ax.set_xlabel("Strategy")
ax.set_ylabel("Score (max 8)")
ax.set_ylim(0, 8)
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 6. 관심지표가 실제로 무엇을 개선했나?

단순 점수 대신 A→B와 D→C의 실제 변화량을 봅니다.

좋은 방향:

- `buy_actual_buy_rate_delta` > 0
- `buy_actual_risk_rate_delta` < 0
- `avoid_actual_risk_rate_delta` > 0
- `avoid_actual_buy_rate_delta` < 0

아래 heatmap은 좋은 방향이면 **양수로 변환**하여 보여줍니다.

즉 네 지표 모두 값이 높을수록 관심지표가 후보를 원하는 방향으로 개선했다는 뜻입니다.

In [ ]:
# ============================================================
# 6. 20D Top20 관심효과 signed improvement heatmap
# ============================================================

main20 = attention_effect[
    attention_effect["horizon"].eq(20)
    & np.isclose(attention_effect["top_q"], 0.20)
].copy()

main20["buy_good"] = main20["buy_actual_buy_rate_delta"]
main20["buy_risk_reduction"] = -main20["buy_actual_risk_rate_delta"]
main20["avoid_risk_good"] = main20["avoid_actual_risk_rate_delta"]
main20["avoid_buy_reduction"] = -main20["avoid_actual_buy_rate_delta"]

heat = main20.pivot_table(
    index="strategy",
    columns="comparison",
    values=[
        "buy_good",
        "buy_risk_reduction",
        "avoid_risk_good",
        "avoid_buy_reduction",
    ]
)

# 컬럼 단순화
heat.columns = [
    f"{metric}|{comp}"
    for metric, comp in heat.columns
]

heat = heat.reindex(order)

arr = heat.to_numpy(dtype=float)

fig, ax = plt.subplots(figsize=(16, 7))
im = ax.imshow(arr, aspect="auto")

ax.set_yticks(np.arange(len(heat.index)))
ax.set_yticklabels(heat.index)

ax.set_xticks(np.arange(len(heat.columns)))
ax.set_xticklabels(heat.columns, rotation=60, ha="right")

for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        if np.isfinite(arr[i, j]):
            ax.text(
                j, i,
                f"{arr[i, j]*100:+.1f}%p",
                ha="center",
                va="center",
                fontsize=8
            )

ax.set_title(
    "20D Top20 — 관심지표 추가 후 Target 개선 방향\n"
    "(양수 = 원하는 방향)"
)
fig.colorbar(im, ax=ax, label="Improvement")
plt.tight_layout()
plt.show()

## 7. BHAR gap 개선 — 경제적 성과도 좋아졌는가?

Target을 잘 맞히는 것과 최종 투자성과는 동일하지 않습니다.

그래서 보조적으로:

`매수 후보 평균 BHAR - 회피 후보 평균 BHAR`

의 차이를 봅니다.

`mean_BHAR_gap_delta > 0`이면 관심지표를 추가했을 때
매수/회피 후보의 실제 시장 대비 성과 차이가 더 벌어진 것입니다.

In [ ]:
# ============================================================
# 7. Mean BHAR gap delta
# ============================================================

d = selection.set_index("strategy").reindex(order)

ax = d["mean_BHAR_gap_delta"].plot(
    kind="bar",
    figsize=(13, 5)
)

ax.axhline(0, linewidth=1)
ax.set_title("20D Top20 — 관심지표 추가 후 평균 BHAR Gap 변화")
ax.set_xlabel("Strategy")
ax.set_ylabel("BHAR gap delta")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 8. 한 장으로 보는 전략 비교

이제 네 축을 동시에 봅니다.

- X축: `BC_mean_roc_auc`
- Y축: `attention_direction_score_8`
- 점 크기: `BC_mean_pr_lift`
- 라벨: 전략명

우상단에 가까울수록:
- target 자체가 상대적으로 예측 가능하고
- 관심효과도 일관된 전략

이라고 볼 수 있습니다.

In [ ]:
# ============================================================
# 8. ROC-AUC vs 방향성 score scatter
# ============================================================

fig, ax = plt.subplots(figsize=(11, 7))

x = selection["BC_mean_roc_auc"].to_numpy()
y = selection["attention_direction_score_8"].to_numpy()
sizes = 700 * selection["BC_mean_pr_lift"].to_numpy()

ax.scatter(x, y, s=sizes, alpha=0.65)

for _, row in selection.iterrows():
    ax.annotate(
        row["strategy"],
        (row["BC_mean_roc_auc"], row["attention_direction_score_8"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=9
    )

ax.axvline(0.5, linestyle="--", linewidth=1)
ax.axhline(4, linestyle="--", linewidth=1)

ax.set_title(
    "20D 전략 비교 — OOF 분류력 vs 관심효과 방향성\n"
    "점 크기 = 평균 PR Lift"
)
ax.set_xlabel("B/C Mean ROC-AUC")
ax.set_ylabel("Attention Direction Score / 8")
ax.set_ylim(-0.5, 8.5)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 9. 보수적 안정성 기준

전략 선택 시 평균만 보면 일부 buy/risk 모델이 매우 약해도 가려질 수 있습니다.

그래서 다음 두 조건을 따로 확인합니다.

- `BC_min_roc_auc > 0.5`
- `BC_min_pr_lift > 1.0`

둘 다 만족하면:

> B/C 모델의 buy/risk 중 **가장 약한 경우조차**
> 랜덤/발생비율 baseline보다 높았다.

는 뜻입니다.

이것은 최종 전략을 고를 때 꽤 강한 안정성 기준입니다.

In [ ]:
# ============================================================
# 9. 안정성 조건
# ============================================================

stable = selection.copy()

stable["all_BC_auc_above_random"] = (
    stable["BC_min_roc_auc"] > 0.5
)

stable["all_BC_pr_above_baseline"] = (
    stable["BC_min_pr_lift"] > 1.0
)

stable["both_stable"] = (
    stable["all_BC_auc_above_random"]
    & stable["all_BC_pr_above_baseline"]
)

display(
    stable[
        [
            "strategy",
            "BC_mean_roc_auc",
            "BC_min_roc_auc",
            "BC_mean_pr_lift",
            "BC_min_pr_lift",
            "attention_direction_score_8",
            "mean_BHAR_gap_delta",
            "both_stable",
        ]
    ].sort_values(
        [
            "both_stable",
            "attention_direction_score_8",
            "BC_mean_roc_auc",
        ],
        ascending=False
    )
)

## 10. 자동 결론 — 어떤 전략이 가장 균형적인가?

여기서는 전략을 **기계적으로 점수 하나로 확정하지 않고**,
다음 우선순위로 추천 후보를 정리합니다.

#### 1순위
B/C의 가장 약한 buy/risk 모델까지:
- ROC-AUC > 0.5
- PR lift > 1

를 모두 만족하는가?

#### 2순위
관심지표 방향성 점수가 높은가?

#### 3순위
BHAR gap도 양의 방향으로 개선되는가?

이렇게 보면 **“방향성만 좋아 보이는 전략”과 “실제로 target을 학습할 수 있는 전략”을 구분**할 수 있습니다.

In [ ]:
# ============================================================
# 10. 자동 전략 추천
# ============================================================

ranked = stable.copy()

# 안정성을 가장 먼저, 이후 방향성 / 평균 AUC / PR / BHAR
ranked = ranked.sort_values(
    [
        "both_stable",
        "attention_direction_score_8",
        "BC_mean_roc_auc",
        "BC_mean_pr_lift",
        "mean_BHAR_gap_delta",
    ],
    ascending=False
).reset_index(drop=True)

best = ranked.iloc[0]

print("=" * 95)
print("전략 선택 자동 해석")
print("=" * 95)

print("\n[추천 1순위]")
print(best["strategy"])

print(
    f"- Attention direction score: "
    f"{int(best['attention_direction_score_8'])}/8"
)
print(
    f"- B/C mean ROC-AUC: "
    f"{best['BC_mean_roc_auc']:.3f}"
)
print(
    f"- B/C min ROC-AUC: "
    f"{best['BC_min_roc_auc']:.3f}"
)
print(
    f"- B/C mean PR lift: "
    f"{best['BC_mean_pr_lift']:.3f}"
)
print(
    f"- B/C min PR lift: "
    f"{best['BC_min_pr_lift']:.3f}"
)
print(
    f"- Mean BHAR gap delta: "
    f"{best['mean_BHAR_gap_delta']*100:+.2f}%p"
)

print("\n[해석]")

if bool(best["both_stable"]):
    print(
        "이 전략은 B/C의 가장 약한 buy/risk 조합까지 "
        "ROC-AUC 0.5와 PR-lift 1을 모두 넘었습니다."
    )
else:
    print(
        "모든 B/C buy/risk 조합이 랜덤/baseline을 넘는 수준은 아닙니다. "
        "따라서 결과를 강한 예측력으로 해석하면 안 됩니다."
    )

if best["attention_direction_score_8"] >= 6:
    print(
        "관심지표 추가효과도 비교적 일관된 방향으로 나타났습니다."
    )
elif best["attention_direction_score_8"] >= 4:
    print(
        "관심지표 추가효과는 절반 이상에서 좋은 방향이지만 완전히 일관적이지는 않습니다."
    )
else:
    print(
        "관심지표 추가효과의 방향성은 약하거나 혼재합니다."
    )

if best["mean_BHAR_gap_delta"] > 0:
    print(
        "BHAR 기준 경제적 분리도 역시 평균적으로 개선되는 방향입니다."
    )
else:
    print(
        "BHAR 기준 경제적 분리도는 개선되지 않았습니다."
    )

print("\n[전체 순위 참고]")
display(
    ranked[
        [
            "strategy",
            "both_stable",
            "attention_direction_score_8",
            "BC_mean_roc_auc",
            "BC_min_roc_auc",
            "BC_mean_pr_lift",
            "BC_min_pr_lift",
            "mean_BHAR_gap_delta",
            "neither_rate",
        ]
    ]
)

## 11. 현재 결과를 바탕으로 한 해석 원칙

현재 요약표에서 특히 주의해야 할 패턴은 다음과 같습니다.

#### `ewma_hl20_k1p5`
관심 방향성 점수는 높을 수 있지만,
ROC-AUC가 0.5 아래이고 PR lift도 1 아래라면:

> **관심 추가 후 후보 구성은 원하는 방향으로 일부 움직였지만,
> target 자체를 안정적으로 예측한다고 보기는 어렵다.**

라고 해석해야 합니다.

#### `ewma_hl10_k2p0`
만약:
- `BC_min_roc_auc > 0.5`
- `BC_min_pr_lift > 1`
- 관심 방향성도 절반 이상 개선
- BHAR gap delta > 0

를 동시에 만족한다면:

> **최고 방향성 점수는 아니더라도,
> 분류력·안정성·경제적 방향성을 함께 고려했을 때 가장 균형적인 후보**

라고 볼 수 있습니다.

즉 **attention_direction_score만으로 전략을 고르지 않는 것이 핵심**입니다.

## 12. 10D / 20D / 60D 전체 OOF 성능도 확인

위의 전략 선택 요약은 프로젝트 메인 horizon인 **20D 중심**입니다.

하지만 최종 보고에서는:
- 10D
- 20D
- 60D

에서 해당 전략의 OOF ROC-AUC가 어떻게 변하는지도 같이 보여주는 것이 좋습니다.

선택 후보 전략에 대해 horizon별 B/C buy/risk 성능을 시각화합니다.

In [ ]:
# ============================================================
# 12. 추천전략 horizon별 OOF 성능
# ============================================================

BEST_STRATEGY = best["strategy"]

best_metrics = metrics[
    metrics["strategy"].eq(BEST_STRATEGY)
    & metrics["model"].isin(["B", "C"])
].copy()

for metric_name, baseline in [
    ("roc_auc", 0.5),
    ("pr_lift_vs_prevalence", 1.0),
]:
    p = best_metrics.pivot_table(
        index="horizon",
        columns=["model", "target"],
        values=metric_name
    )

    ax = p.plot(
        kind="bar",
        figsize=(12, 6)
    )

    ax.axhline(baseline, linestyle="--", linewidth=1)
    ax.set_title(
        f"{BEST_STRATEGY} — 10D/20D/60D {metric_name}"
    )
    ax.set_xlabel("Horizon")
    ax.set_ylabel(metric_name)
    ax.grid(axis="y", alpha=0.25)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

display(
    best_metrics[
        [
            "horizon",
            "model",
            "target",
            "positive_rate",
            "roc_auc",
            "pr_auc",
            "pr_lift_vs_prevalence",
        ]
    ].sort_values(
        ["horizon", "model", "target"]
    )
)

## 13. Top10 / Top20 / Top30 민감도 분석

Top20에서만 관심효과가 좋아지고,
Top10이나 Top30에서는 완전히 사라진다면 결과가 threshold에 민감할 수 있습니다.

따라서 추천 전략에 대해 A→B, D→C의 방향성 점수가
Top10 / Top20 / Top30에서 얼마나 유지되는지 봅니다.

In [ ]:
# ============================================================
# 13. Top-q 민감도
# ============================================================

ae = attention_effect[
    attention_effect["strategy"].eq(BEST_STRATEGY)
].copy()

def calc_direction_score(row):
    return int(
        (row["buy_actual_buy_rate_delta"] > 0)
        + (row["buy_actual_risk_rate_delta"] < 0)
        + (row["avoid_actual_risk_rate_delta"] > 0)
        + (row["avoid_actual_buy_rate_delta"] < 0)
    )

ae["direction_score_4"] = ae.apply(
    calc_direction_score,
    axis=1
)

score_q = (
    ae.groupby(
        ["horizon", "top_q"]
    )["direction_score_4"]
    .sum()
    .reset_index(name="direction_score_8")
)

for h in [10, 20, 60]:
    d = score_q[
        score_q["horizon"].eq(h)
    ].copy()

    ax = d.plot(
        kind="bar",
        x="top_q",
        y="direction_score_8",
        legend=False,
        figsize=(8, 4)
    )

    ax.set_title(
        f"{BEST_STRATEGY} — {h}D Top-q 관심효과 방향성"
    )
    ax.set_xlabel("Top q")
    ax.set_ylabel("Direction Score / 8")
    ax.set_ylim(0, 8)
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

display(score_q)

## 14. 최종 보고서용 결론 자동 생성

아래 셀은 현재 OOF 결과만 사용하여
발표/노션에 적기 쉬운 형태로 결론을 출력합니다.

2025~2026 OOT 성능은 **이 전략을 확정한 뒤 별도로 확인**해야 합니다.

In [ ]:
# ============================================================
# 14. 보고서용 결론
# ============================================================

top = ranked.iloc[0]
runner = ranked.iloc[1] if len(ranked) > 1 else None

print("========== 보고서용 결론 ==========\n")

print(
    f"1. 9개 EWMA 장벽 전략을 2021~2024년 OOF 구간에서 비교한 결과, "
    f"분류 안정성과 관심지표 효과를 함께 고려했을 때 "
    f"`{top['strategy']}`가 가장 균형적인 후보로 나타났다."
)

print(
    f"2. 해당 전략의 20D B/C 평균 ROC-AUC는 "
    f"{top['BC_mean_roc_auc']:.3f}, 최저 ROC-AUC는 "
    f"{top['BC_min_roc_auc']:.3f}였으며, 평균 PR lift는 "
    f"{top['BC_mean_pr_lift']:.3f}, 최저 PR lift는 "
    f"{top['BC_min_pr_lift']:.3f}였다."
)

print(
    f"3. 관심지표 추가 후 기대 방향으로 개선된 지표는 "
    f"8개 중 {int(top['attention_direction_score_8'])}개였고, "
    f"Buy-Avoid BHAR gap은 평균 "
    f"{top['mean_BHAR_gap_delta']*100:+.2f}%p 변화했다."
)

if bool(top["both_stable"]):
    print(
        "4. 특히 B/C의 가장 약한 buy/risk 조합에서도 "
        "ROC-AUC가 0.5를 넘고 PR lift가 1을 넘었다는 점에서, "
        "단순히 방향성 점수만 높은 전략보다 target의 예측 가능성이 상대적으로 안정적이었다."
    )
else:
    print(
        "4. 다만 B/C의 모든 buy/risk 조합이 랜덤 및 prevalence baseline을 "
        "동시에 넘지는 못해 절대적인 예측력은 제한적이었다."
    )

print(
    "5. 따라서 이 전략을 장벽 기준의 최종 후보로 먼저 고정한 뒤, "
    "2025~2026 OOT 구간에서 A/B/C/D를 다시 평가하여 "
    "관심지표 효과가 실제 미래 구간에서도 유지되는지 최종 검증하는 것이 적절하다."
)

print(
    "\n※ 이 결론은 장벽전략 선택용 2021~2024 OOF 결과에만 기반하며, "
    "2025~2026 결과를 전략 선택에 사용하지 않았다."
)

In [ ]:
# ============================================================
# 15. 시각화용 요약 파일 저장
# ============================================================

VIZ_DIR = OUT_DIR / "visualization_summary"
VIZ_DIR.mkdir(parents=True, exist_ok=True)

ranked.to_csv(
    VIZ_DIR / "strategy_ranked_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

health20.to_csv(
    VIZ_DIR / "target_health_20D.csv",
    index=False,
    encoding="utf-8-sig"
)

score_q.to_csv(
    VIZ_DIR / "best_strategy_topq_sensitivity.csv",
    index=False,
    encoding="utf-8-sig"
)

best_metrics.to_csv(
    VIZ_DIR / "best_strategy_horizon_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✅ 저장 완료")
print(VIZ_DIR)

## STAGE 9-E — A/B/C/D 모델링 결과 시각화 & 해석 (구 05번)

In [ ]:
# ============================================================
# 0. Drive 연결 + 라이브러리
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import json
import warnings

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

BASE = WORKSPACE_ROOT
RUN_DIR = (
    BASE
    / "cache/modeling_preparation_v1/modeling_runs/ewma_hl10_k1p5"
)

PATHS = {
    "features": RUN_DIR / "event_features_ABCD_v1.parquet",
    "manifest": RUN_DIR / "feature_manifest.json",
    "oof": RUN_DIR / "predictions_oof_2021_2024.parquet",
    "oot": RUN_DIR / "predictions_oot_2025_2026.parquet",
    "thresholds": RUN_DIR / "score_thresholds_top10_20_30.csv",
    "metrics": RUN_DIR / "model_metrics.csv",
    "candidate": RUN_DIR / "candidate_evaluation_oot.csv",
    "comparison": RUN_DIR / "comparison_summary_top20.csv",
    "importance": RUN_DIR / "feature_importance_gain.csv",
}

print("========== 파일 확인 ==========")
for name, p in PATHS.items():
    print("✅" if p.exists() else "❌", name, "->", p)

missing = [str(p) for p in PATHS.values() if not p.exists()]
if missing:
    raise FileNotFoundError("누락 파일:\n" + "\n".join(missing))

In [ ]:
# ============================================================
# 1. 데이터 로드
# ============================================================

features = pd.read_parquet(PATHS["features"])
oof = pd.read_parquet(PATHS["oof"])
oot = pd.read_parquet(PATHS["oot"])

thresholds = pd.read_csv(PATHS["thresholds"])
metrics = pd.read_csv(PATHS["metrics"])
candidate = pd.read_csv(PATHS["candidate"])
comparison = pd.read_csv(PATHS["comparison"])
importance = pd.read_csv(PATHS["importance"])

with open(PATHS["manifest"], "r", encoding="utf-8") as f:
    manifest = json.load(f)

for d in [features, oof, oot]:
    if "event_date" in d.columns:
        d["event_date"] = pd.to_datetime(d["event_date"], errors="coerce")

print("features:", features.shape)
print("OOF:", oof.shape)
print("OOT:", oot.shape)
print("metrics:", metrics.shape)
print("candidate:", candidate.shape)
print("comparison:", comparison.shape)
print("importance:", importance.shape)

print("\nTarget strategy:", manifest.get("target_strategy"))

## 1. 먼저 용어부터 쉽게 이해하기

### OOF — Out-Of-Fold

**2021~2024 내부 검증용 예측값**입니다.

어떤 사건을 예측할 때, 그 사건을 학습에 직접 사용하지 않은 모델이 예측하도록 만든 점수입니다.  
그래서 학습 데이터를 다시 맞혀보는 것보다 훨씬 현실적인 성능 추정치입니다.

이 OOF 점수로:
- `buy_score` 상위 10/20/30%
- `risk_score` 상위 10/20/30%

경계를 정합니다.

---

### OOT — Out-Of-Time

**완전히 미래인 2025~2026 테스트**입니다.

2021 ~ 2024만 보고 학습·기준 설정을 끝낸 뒤,
그 기준을 2025 ~ 2026에 그대로 적용합니다.

즉 OOT가 실제 실전 상황에 가장 가까운 평가입니다.

---

### ROC-AUC

모델이 **양성 target을 음성 target보다 더 높은 점수로 정렬하는 능력**입니다.

- 0.5 ≈ 랜덤
- 0.6대 = 약한 분리력
- 0.7대 = 꽤 의미 있는 분리력
- 1.0 = 완벽

다만 주식 데이터는 잡음이 크므로 숫자 하나만으로 판단하면 안 됩니다.

---

### PR-AUC

Positive class, 즉 `buy_target=1` 또는 `risk_target=1`을  
**얼마나 정확하게 찾아내는지**에 더 민감한 지표입니다.

특히 클래스 비율이 불균형할 때 ROC-AUC보다 실질적인 정보를 줄 수 있습니다.

---

### BHAR — Buy-and-Hold Abnormal Return

사건 이후 해당 종목을 들고 갔을 때의 수익률에서  
시장 benchmark 수익률을 뺀 **시장 대비 초과성과**입니다.

- 높을수록 좋음
- 매수 후보 BHAR은 높아야 함
- 회피 후보 BHAR은 낮아야 함

---

### MDD — Maximum Drawdown

사건 이후 경로에서 고점 대비 가장 크게 빠진 폭입니다.

예:
- `MDD = -0.10` → 최대 낙폭 -10%
- `MDD = -0.30` → 최대 낙폭 -30%

따라서:
- 매수 후보 MDD → 0에 가까울수록 좋음
- 회피 후보 MDD → 더 음수일수록 실제 위험종목을 잘 잡은 것

---

### Top 20%

2021~2024 OOF 점수 기준으로:
- risk score 상위 20% → 회피 후보
- buy score 상위 20% → 매수 후보
- 둘 다면 회피 우선
- 나머지 → 관망

중요한 점은 **2025~2026에서 다시 상위 20%를 뽑는 게 아니라 과거 threshold를 고정 적용**한다는 것입니다.

그래서 OOT에서 매수/회피 후보가 정확히 20%가 아닐 수 있습니다.

## 2. OOF vs OOT — 모델 자체 분류 성능

먼저 후보군의 실제 투자 성과를 보기 전에,
모델이 `buy_target`, `risk_target` 자체를 얼마나 구분하는지 확인합니다.

**OOF가 괜찮은데 OOT에서 크게 떨어지면**  
2025~2026 시장구조 변화, 즉 **distribution shift / domain shift** 가능성을 의심할 수 있습니다.

In [ ]:
# ============================================================
# 2-1. ROC-AUC 시각화
# ============================================================

def plot_metric_by_horizon(metric_name, target_name):
    d = metrics[
        metrics["target"].eq(target_name)
    ].copy()

    periods = ["OOF_2021_2024", "OOT_2025_2026"]
    models = ["A", "B", "C", "D"]
    horizons = [10, 20, 60]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

    for ax, h in zip(axes, horizons):
        sub = d[d["horizon"].eq(h)]

        x = np.arange(len(models))
        width = 0.35

        for i, period in enumerate(periods):
            vals = []
            for m in models:
                row = sub[
                    sub["model"].eq(m) &
                    sub["period"].eq(period)
                ]
                vals.append(
                    row[metric_name].iloc[0]
                    if len(row) else np.nan
                )

            ax.bar(
                x + (i - 0.5) * width,
                vals,
                width=width,
                label=period
            )

        ax.axhline(0.5, linestyle="--", linewidth=1)
        ax.set_title(f"{h}D")
        ax.set_xticks(x)
        ax.set_xticklabels(models)
        ax.set_xlabel("Model")
        ax.grid(axis="y", alpha=0.25)

    axes[0].set_ylabel(metric_name)
    axes[-1].legend()
    fig.suptitle(f"{target_name.upper()} target — {metric_name}")
    plt.tight_layout()
    plt.show()

plot_metric_by_horizon("roc_auc", "buy")
plot_metric_by_horizon("roc_auc", "risk")

In [ ]:
# ============================================================
# 2-2. PR-AUC 시각화
# ============================================================

plot_metric_by_horizon("pr_auc", "buy")
plot_metric_by_horizon("pr_auc", "risk")

In [ ]:
# ============================================================
# 2-3. OOF → OOT 성능 변화 표
# ============================================================

metric_compare = (
    metrics
    .pivot_table(
        index=["horizon", "model", "target"],
        columns="period",
        values=["roc_auc", "pr_auc"]
    )
)

display(metric_compare)

#### 이 그래프를 보는 법

예를 들어 **20D B 모델**이 A 모델보다 OOF와 OOT에서 모두 높은 ROC-AUC/PR-AUC를 보인다면:

> 가격·거래량·수급만 봤을 때보다 검색·뉴스 관심정보를 추가했을 때  
> `buy_target` 또는 `risk_target`을 더 잘 정렬했다.

라고 해석할 수 있습니다.

반대로 OOF에서는 좋아졌는데 OOT에서는 악화됐다면:

> 과거 검증에서는 유효했던 신호가 2025~2026에서는 충분히 일반화되지 않았다.

라고 보는 것이 안전합니다.

## 3. Top 20% 후보 수 — 과거 threshold가 미래에 어떻게 작동했나

2025~2026에 threshold를 다시 계산하지 않았기 때문에,
각 모델의 매수·회피 후보 수가 다르게 나올 수 있습니다.

이 차이는 오류가 아니라 **score distribution 변화**를 보여줍니다.

In [ ]:
# ============================================================
# 3. Top20 후보 수
# ============================================================

top20 = candidate[
    np.isclose(candidate["top_q"], 0.20)
].copy()

count_plot = top20[
    ["horizon", "model", "buy_n", "avoid_n", "hold_n"]
].copy()

for h in [10, 20, 60]:
    d = count_plot[count_plot["horizon"].eq(h)].set_index("model")
    d = d.reindex(["A", "B", "C", "D"])

    ax = d[["buy_n", "avoid_n", "hold_n"]].plot(
        kind="bar",
        figsize=(9, 5)
    )

    ax.set_title(f"{h}D — OOT Top20 후보 수")
    ax.set_xlabel("Model")
    ax.set_ylabel("Number of events")
    ax.grid(axis="y", alpha=0.25)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 4. Top20 후보의 Target 적중률

이제 모델이 실제로:
- 매수 후보에서 `buy_target=1`을 더 많이 뽑았는지
- 회피 후보에서 `risk_target=1`을 더 많이 뽑았는지

확인합니다.

#### 좋은 모델이라면

**매수 후보**
- `actual_buy_rate` ↑
- `actual_risk_rate` ↓

**회피 후보**
- `actual_risk_rate` ↑
- `actual_buy_rate` ↓

In [ ]:
# ============================================================
# 4-1. 매수 후보 target 비율
# ============================================================

for h in [10, 20, 60]:
    d = top20[top20["horizon"].eq(h)].set_index("model")
    d = d.reindex(["A", "B", "C", "D"])

    ax = d[
        ["buy_actual_buy_rate", "buy_actual_risk_rate"]
    ].plot(
        kind="bar",
        figsize=(9, 5)
    )

    ax.set_title(f"{h}D — 매수 후보의 실제 Target 비율")
    ax.set_xlabel("Model")
    ax.set_ylabel("Rate")
    ax.set_ylim(0, 1)
    ax.grid(axis="y", alpha=0.25)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 4-2. 회피 후보 target 비율
# ============================================================

for h in [10, 20, 60]:
    d = top20[top20["horizon"].eq(h)].set_index("model")
    d = d.reindex(["A", "B", "C", "D"])

    ax = d[
        ["avoid_actual_risk_rate", "avoid_actual_buy_rate"]
    ].plot(
        kind="bar",
        figsize=(9, 5)
    )

    ax.set_title(f"{h}D — 회피 후보의 실제 Target 비율")
    ax.set_xlabel("Model")
    ax.set_ylabel("Rate")
    ax.set_ylim(0, 1)
    ax.grid(axis="y", alpha=0.25)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 5. BHAR — 실제 경제적 성과

Target을 잘 맞혔다고 해서 반드시 최종 수익률도 좋은 것은 아닙니다.

예를 들어 어떤 종목이 초반에 상승장벽을 먼저 넘었더라도,
이후 크게 하락하면 `buy_target=1`인데 BHAR는 음수일 수 있습니다.

그래서 **Target 적중률과 BHAR를 반드시 같이 봐야 합니다.**

#### 우리가 원하는 방향
- Buy BHAR > Hold BHAR > Avoid BHAR
- 특히 `Buy - Avoid BHAR gap`이 클수록 후보 선별이 잘 된 것

In [ ]:
# ============================================================
# 5-1. Buy / Avoid / Hold 평균 BHAR
# ============================================================

for h in [10, 20, 60]:
    d = top20[top20["horizon"].eq(h)].set_index("model")
    d = d.reindex(["A", "B", "C", "D"])

    ax = d[
        ["buy_BHAR_mean", "hold_BHAR_mean", "avoid_BHAR_mean"]
    ].plot(
        kind="bar",
        figsize=(9, 5)
    )

    ax.axhline(0, linewidth=1)
    ax.set_title(f"{h}D — 후보군 평균 BHAR")
    ax.set_xlabel("Model")
    ax.set_ylabel("BHAR")
    ax.grid(axis="y", alpha=0.25)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 5-2. Buy - Avoid BHAR gap
# ============================================================

pivot_gap = (
    top20
    .pivot(
        index="horizon",
        columns="model",
        values="buy_minus_avoid_BHAR_mean_gap"
    )
    .reindex(columns=["A", "B", "C", "D"])
)

ax = pivot_gap.plot(
    kind="bar",
    figsize=(10, 5)
)

ax.axhline(0, linewidth=1)
ax.set_title("Top20 — Buy minus Avoid BHAR Mean Gap")
ax.set_xlabel("Horizon")
ax.set_ylabel("BHAR gap")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

display(pivot_gap)

## 6. MDD — 실제 위험 분리

BHAR가 최종 성과라면 MDD는 **중간에 얼마나 심하게 깨졌는가**를 봅니다.

#### 우리가 원하는 방향

- 매수 후보 MDD → 덜 음수
- 회피 후보 MDD → 더 음수

예:

```text
매수 MDD = -12%
회피 MDD = -25%
```

라면 모델이 실제 낙폭 위험을 어느 정도 분리했다고 볼 수 있습니다.

In [ ]:
# ============================================================
# 6. Buy / Avoid / Hold 평균 MDD
# ============================================================

for h in [10, 20, 60]:
    d = top20[top20["horizon"].eq(h)].set_index("model")
    d = d.reindex(["A", "B", "C", "D"])

    ax = d[
        ["buy_MDD_mean", "hold_MDD_mean", "avoid_MDD_mean"]
    ].plot(
        kind="bar",
        figsize=(9, 5)
    )

    ax.axhline(0, linewidth=1)
    ax.set_title(f"{h}D — 후보군 평균 MDD")
    ax.set_xlabel("Model")
    ax.set_ylabel("MDD")
    ax.grid(axis="y", alpha=0.25)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 7. 관심지표 추가효과 — FinDA 핵심 가설

여기가 가장 중요합니다.

### A → B
가격·거래량·수급 모델에 **검색·뉴스 관심지표를 추가**

### D → C
가격·거래량·수급+공매도 모델에 **검색·뉴스 관심지표를 추가**

#### 좋은 방향

- 매수 `buy_target` 비율: **+**
- 매수 `risk_target` 비율: **-**
- 회피 `risk_target` 비율: **+**
- 회피 `buy_target` 비율: **-**
- Buy-Avoid BHAR gap: **+**

특히 두 비교에서 같은 방향이 반복되면 관심지표의 추가효과가 더 설득력 있습니다.

In [ ]:
# ============================================================
# 7-1. 관심지표 추가효과 — Target delta
# ============================================================

attention = comparison[
    comparison["comparison"].isin([
        "attention_without_short",
        "attention_with_short"
    ])
].copy()

attention_metrics = [
    ("buy_actual_buy_rate_delta", "매수 후보 buy-target Δ"),
    ("buy_actual_risk_rate_delta", "매수 후보 risk-target Δ"),
    ("avoid_actual_risk_rate_delta", "회피 후보 risk-target Δ"),
    ("avoid_actual_buy_rate_delta", "회피 후보 buy-target Δ"),
]

for col, title in attention_metrics:
    p = attention.pivot(
        index="horizon",
        columns="comparison",
        values=col
    )

    ax = p.plot(
        kind="bar",
        figsize=(10, 5)
    )

    ax.axhline(0, linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("Horizon")
    ax.set_ylabel("Delta")
    ax.grid(axis="y", alpha=0.25)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 7-2. 관심지표 추가효과 — BHAR gap delta
# ============================================================

p = attention.pivot(
    index="horizon",
    columns="comparison",
    values="buy_minus_avoid_BHAR_mean_gap_delta"
)

ax = p.plot(
    kind="bar",
    figsize=(10, 5)
)

ax.axhline(0, linewidth=1)
ax.set_title("관심지표 추가 후 Buy-Avoid BHAR gap 변화")
ax.set_xlabel("Horizon")
ax.set_ylabel("BHAR gap delta")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

display(
    attention[
        [
            "horizon",
            "comparison",
            "buy_actual_buy_rate_delta",
            "buy_actual_risk_rate_delta",
            "avoid_actual_risk_rate_delta",
            "avoid_actual_buy_rate_delta",
            "buy_minus_avoid_BHAR_mean_gap_delta",
        ]
    ]
)

## 8. 공매도 추가효과

공매도의 독립적 효과는:

- **A → D**
- **B → C**

로 봅니다.

공매도가 추가됐는데:
- 회피 후보의 actual risk 비율이 올라가고
- 회피 후보 BHAR가 더 낮아지고
- MDD가 더 나빠지면

공매도 변수가 위험 선별에 유효했다고 볼 수 있습니다.

반대로 방향이 흔들리면:
> 현재 구성한 공매도 feature가 추가적인 예측력을 안정적으로 제공하지 못했다.

라고 해석하는 것이 맞습니다.

In [ ]:
# ============================================================
# 8. 공매도 추가효과
# ============================================================

short_effect = comparison[
    comparison["comparison"].isin([
        "short_without_attention",
        "short_with_attention"
    ])
].copy()

short_metrics = [
    ("avoid_actual_risk_rate_delta", "회피 후보 risk-target Δ"),
    ("avoid_actual_buy_rate_delta", "회피 후보 buy-target Δ"),
    ("buy_minus_avoid_BHAR_mean_gap_delta", "Buy-Avoid BHAR gap Δ"),
]

for col, title in short_metrics:
    p = short_effect.pivot(
        index="horizon",
        columns="comparison",
        values=col
    )

    ax = p.plot(
        kind="bar",
        figsize=(10, 5)
    )

    ax.axhline(0, linewidth=1)
    ax.set_title("공매도 추가효과 — " + title)
    ax.set_xlabel("Horizon")
    ax.set_ylabel("Delta")
    ax.grid(axis="y", alpha=0.25)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 9. 2025 vs 2026 — 성능이 같은가?

2025~2026을 하나로 묶으면 2026년의 시장 변화가 가려질 수 있습니다.

여기서는 **과거에 고정한 Top20 threshold를 그대로 유지한 채**
2025와 2026 결과를 나눠봅니다.

이건 threshold를 다시 튜닝하는 것이 아니라,
**모델의 시간적 안정성(temporal stability)**을 확인하는 진단입니다.

In [ ]:
# ============================================================
# 9. OOT Top20 decision 재생성
# ============================================================

decision_parts = []

thr20 = thresholds[
    np.isclose(thresholds["top_q"], 0.20)
].copy()

for row in thr20.itertuples(index=False):
    g = oot[
        (oot["horizon"] == row.horizon)
        & (oot["model"] == row.model)
    ].copy()

    risk_hit = g["risk_score"] >= row.risk_threshold
    buy_hit = g["buy_score"] >= row.buy_threshold

    g["decision"] = np.select(
        [
            risk_hit,
            (~risk_hit) & buy_hit,
        ],
        [
            "avoid",
            "buy",
        ],
        default="hold"
    )

    decision_parts.append(g)

oot_top20 = pd.concat(
    decision_parts,
    ignore_index=True
)

oot_top20["year"] = oot_top20["event_date"].dt.year

year_rows = []

for (year, h, model), g in oot_top20.groupby(
    ["year", "horizon", "model"]
):
    bhar_col = f"BHAR_{h}d"
    mdd_col = f"MDD_{h}d"

    buy = g[g["decision"].eq("buy")]
    avoid = g[g["decision"].eq("avoid")]

    year_rows.append({
        "year": year,
        "horizon": h,
        "model": model,
        "n": len(g),
        "buy_n": len(buy),
        "avoid_n": len(avoid),
        "buy_actual_buy_rate": buy["buy_target"].mean() if len(buy) else np.nan,
        "avoid_actual_risk_rate": avoid["risk_target"].mean() if len(avoid) else np.nan,
        "buy_BHAR_mean": buy[bhar_col].mean() if len(buy) else np.nan,
        "avoid_BHAR_mean": avoid[bhar_col].mean() if len(avoid) else np.nan,
        "buy_MDD_mean": buy[mdd_col].mean() if len(buy) else np.nan,
        "avoid_MDD_mean": avoid[mdd_col].mean() if len(avoid) else np.nan,
    })

year_eval = pd.DataFrame(year_rows)

display(
    year_eval.sort_values(
        ["horizon", "model", "year"]
    )
)

In [ ]:
# ============================================================
# 9-2. 2025 vs 2026 BHAR gap
# ============================================================

year_eval["BHAR_gap"] = (
    year_eval["buy_BHAR_mean"]
    - year_eval["avoid_BHAR_mean"]
)

for h in [10, 20, 60]:
    d = year_eval[
        year_eval["horizon"].eq(h)
    ].pivot(
        index="model",
        columns="year",
        values="BHAR_gap"
    ).reindex(["A", "B", "C", "D"])

    ax = d.plot(
        kind="bar",
        figsize=(9, 5)
    )

    ax.axhline(0, linewidth=1)
    ax.set_title(f"{h}D — 2025 vs 2026 Buy-Avoid BHAR gap")
    ax.set_xlabel("Model")
    ax.set_ylabel("BHAR gap")
    ax.grid(axis="y", alpha=0.25)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 10. Feature Importance — 모델이 무엇을 많이 봤나?

LightGBM의 `gain importance`는:

> 어떤 변수를 사용한 분기가 모델의 손실을 얼마나 많이 줄였는가

를 누적한 값입니다.

즉 값이 크다고 해서 **인과관계**가 있다는 뜻은 아니고,
모델의 예측 과정에서 많이 기여했다는 뜻입니다.

특히:
- B에서 관심변수가 상위권에 있는가?
- C에서 공매도변수가 상위권에 있는가?
- Buy 모델과 Risk 모델이 서로 다른 변수를 보는가?

를 보면 좋습니다.

In [ ]:
# ============================================================
# 10. C 모델 Top feature — horizon × target
# ============================================================

for h in [10, 20, 60]:
    for target in ["buy", "risk"]:
        d = importance[
            (importance["horizon"].eq(h))
            & (importance["model"].eq("C"))
            & (importance["target"].eq(target))
        ].sort_values(
            "gain_share",
            ascending=False
        ).head(12)

        if len(d) == 0:
            continue

        plot_d = d.sort_values("gain_share")

        ax = plot_d.plot(
            kind="barh",
            x="feature",
            y="gain_share",
            legend=False,
            figsize=(9, 6)
        )

        ax.set_title(f"{h}D C model — {target.upper()} Top Features")
        ax.set_xlabel("Gain share")
        ax.set_ylabel("")
        ax.grid(axis="x", alpha=0.25)
        plt.tight_layout()
        plt.show()

## 11. 관심/공매도 변수는 실제로 상위권인가?

Feature importance를 family별로 묶어보면
개별 변수보다 전체적으로 어느 정보군이 많이 사용됐는지 볼 수 있습니다.

주의:
- 이 역시 **예측 기여도**이지 인과효과가 아닙니다.
- 모델 A에는 attention/short가 없으므로 비교 대상은 주로 B/C/D입니다.

In [ ]:
# ============================================================
# 11. Feature family importance
# ============================================================

ATTENTION_KEYS = [
    "search",
    "news",
]

SHORT_KEYS = [
    "short_",
]

def feature_family(name):
    name = str(name)

    if any(k in name for k in ATTENTION_KEYS):
        return "ATTENTION"

    if any(k in name for k in SHORT_KEYS):
        return "SHORT"

    if "foreign" in name:
        return "FOREIGN_FLOW"

    if "individual" in name or "institution" in name:
        return "OTHER_FLOW"

    if (
        "return" in name
        or "vol_" in name
        or "price" in name
        or "high_low" in name
    ):
        return "PRICE_VOLATILITY"

    if "volume" in name or "trading_value" in name or "turnover" in name:
        return "VOLUME_LIQUIDITY"

    if name in ["per", "pbr", "eps"] or "market_cap" in name:
        return "VALUE_SIZE"

    return "OTHER"

importance2 = importance.copy()
importance2["family"] = importance2["feature"].map(feature_family)

family_imp = (
    importance2
    .groupby(
        ["horizon", "model", "target", "family"],
        as_index=False
    )["gain_share"]
    .sum()
)

for h in [10, 20, 60]:
    for target in ["buy", "risk"]:
        d = family_imp[
            (family_imp["horizon"].eq(h))
            & (family_imp["model"].isin(["B", "C", "D"]))
            & (family_imp["target"].eq(target))
        ]

        p = d.pivot(
            index="model",
            columns="family",
            values="gain_share"
        ).fillna(0)

        ax = p.plot(
            kind="bar",
            stacked=True,
            figsize=(11, 6)
        )

        ax.set_title(f"{h}D — {target.upper()} Feature Family Importance")
        ax.set_xlabel("Model")
        ax.set_ylabel("Gain share")
        ax.grid(axis="y", alpha=0.25)
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.show()

## 12. 핵심 결과 자동 요약

아래 셀은 현재 저장된 결과를 읽어서,
**숫자 기반으로 자동 요약 문장**을 생성합니다.

해석 우선순위는:

1. 관심지표 효과 A→B / D→C
2. 20D의 일관성
3. 공매도 추가효과
4. OOF → OOT 성능 변화
5. 2025 vs 2026 안정성

입니다.

In [ ]:
# ============================================================
# 12. 자동 해석
# ============================================================

def pctp(x):
    if pd.isna(x):
        return "NA"
    return f"{x * 100:+.2f}%p"


def pct(x):
    if pd.isna(x):
        return "NA"
    return f"{x * 100:.2f}%"


print("=" * 90)
print("FinDA A/B/C/D 모델링 자동 요약")
print("=" * 90)

# ------------------------------------------------------------
# 20D 관심 효과
# ------------------------------------------------------------

att20 = attention[
    attention["horizon"].eq(20)
].copy()

print("\n[1] 20D 관심지표 추가효과")

for row in att20.itertuples(index=False):
    name = (
        "A→B"
        if row.comparison == "attention_without_short"
        else "D→C"
    )

    print(f"\n{name}")
    print(
        "  매수 buy-target:",
        pctp(row.buy_actual_buy_rate_delta)
    )
    print(
        "  매수 risk-target:",
        pctp(row.buy_actual_risk_rate_delta)
    )
    print(
        "  회피 risk-target:",
        pctp(row.avoid_actual_risk_rate_delta)
    )
    print(
        "  회피 buy-target:",
        pctp(row.avoid_actual_buy_rate_delta)
    )
    print(
        "  Buy-Avoid BHAR gap:",
        pctp(row.buy_minus_avoid_BHAR_mean_gap_delta)
    )

# ------------------------------------------------------------
# 방향성 점수
# ------------------------------------------------------------

def attention_direction_score(row):
    score = 0

    score += int(row["buy_actual_buy_rate_delta"] > 0)
    score += int(row["buy_actual_risk_rate_delta"] < 0)
    score += int(row["avoid_actual_risk_rate_delta"] > 0)
    score += int(row["avoid_actual_buy_rate_delta"] < 0)

    return score

attention_scored = attention.copy()
attention_scored["direction_score_4"] = attention_scored.apply(
    attention_direction_score,
    axis=1
)

print("\n[2] 관심효과 방향성 점수 — 4점 만점")

display(
    attention_scored[
        [
            "horizon",
            "comparison",
            "direction_score_4",
            "buy_minus_avoid_BHAR_mean_gap_delta",
        ]
    ].sort_values(
        ["horizon", "comparison"]
    )
)

# ------------------------------------------------------------
# 공매도
# ------------------------------------------------------------

print("\n[3] 공매도 추가효과 — Top20")

short_summary = short_effect[
    [
        "horizon",
        "comparison",
        "avoid_actual_risk_rate_delta",
        "avoid_actual_buy_rate_delta",
        "buy_minus_avoid_BHAR_mean_gap_delta",
    ]
].copy()

display(short_summary)

# ------------------------------------------------------------
# OOF/OOT AUC gap
# ------------------------------------------------------------

auc = metrics.pivot_table(
    index=["horizon", "model", "target"],
    columns="period",
    values="roc_auc"
).reset_index()

if (
    "OOF_2021_2024" in auc.columns
    and "OOT_2025_2026" in auc.columns
):
    auc["OOT_minus_OOF"] = (
        auc["OOT_2025_2026"]
        - auc["OOF_2021_2024"]
    )

    print("\n[4] OOF → OOT ROC-AUC 변화")
    display(
        auc.sort_values(
            ["horizon", "model", "target"]
        )
    )

# ------------------------------------------------------------
# 한 줄 결론
# ------------------------------------------------------------

score20 = attention_scored[
    attention_scored["horizon"].eq(20)
]["direction_score_4"].sum()

print("\n" + "=" * 90)

if score20 >= 7:
    print(
        "핵심 결론: 현재 ewma_hl10_k1p5 기준에서는 "
        "20D에서 관심지표 추가효과가 매우 일관된 방향으로 나타납니다."
    )
elif score20 >= 5:
    print(
        "핵심 결론: 현재 ewma_hl10_k1p5 기준에서는 "
        "20D에서 관심지표 추가효과가 대체로 긍정적이지만 일부 지표는 혼재합니다."
    )
else:
    print(
        "핵심 결론: 현재 ewma_hl10_k1p5 기준에서는 "
        "20D 관심지표 효과가 충분히 일관적이지 않습니다."
    )

print(
    "단, 이 결론은 아직 하나의 장벽 전략 결과이므로 "
    "9개 장벽 전략 OOF 강건성 비교 후 최종 확정해야 합니다."
)

## 13. 발표/보고서용 정리 템플릿

아래 문장은 결과표를 보고 숫자만 넣어서 사용할 수 있는 형식입니다.

---

### 모델 검증 구조

> 본 분석은 시간순서에 따른 정보누수를 방지하기 위해 2021~ 2024년을 개발 구간으로 설정하고 Purged Expanding Walk-Forward 방식으로 OOF 예측값을 생성하였다. 매수·회피 후보의 점수 임계값은 해당 OOF 예측분포에서 산출하였으며, 2025~2026년 Out-of-Time 구간에는 이를 고정 적용하였다.

### 관심지표 효과

> 관심지표의 추가효과는 비관심 모델 A와 관심 모델 B, 공매도 포함 비관심 모델 D와 관심 모델 C를 비교하여 확인하였다. 특히 20거래일 구간에서는 관심지표 추가 후 매수 후보의 상승장벽 선행 비율과 회피 후보의 하락장벽 선행 비율이 동시에 개선되는지를 핵심 기준으로 평가하였다.

### BHAR / MDD

> 장벽 기반 target은 미래 경로의 선행 방향을 측정하는 반면, BHAR와 MDD는 각각 실제 시장 대비 초과성과와 경로상 최대낙폭을 나타낸다. 따라서 target 적중률뿐 아니라 매수 후보와 회피 후보 간 BHAR·MDD 격차를 함께 평가하여 실제 투자 의사결정 관점의 유효성을 검증하였다.

### 해석상 주의

> 본 결과는 현재 `ewma_hl10_k1p5` 장벽 전략에 대한 결과이며, 최종 결론은 EWMA half-life와 barrier multiplier를 변경한 9개 전략의 2021~2024 OOF 강건성 비교 이후 확정한다.

In [ ]:
# ============================================================
# 14. 핵심 표 저장
# ============================================================

VIZ_OUT = RUN_DIR / "visualization_summary"
VIZ_OUT.mkdir(parents=True, exist_ok=True)

top20.to_csv(
    VIZ_OUT / "top20_candidate_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

attention_scored.to_csv(
    VIZ_OUT / "attention_effect_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

year_eval.to_csv(
    VIZ_OUT / "year_2025_2026_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

family_imp.to_csv(
    VIZ_OUT / "feature_family_importance.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✅ 시각화 요약표 저장 완료")
print(VIZ_OUT)

# STAGE 10: Peak Give-back Binary — 리크 발견 前/後 전체 과정

**태그:** 폐기됨 (time-to-peak bias) / 리크 발견-재검증은 최종 반영

**원본 노트북:** `peak_giveback_leaky_and_noleak.ipynb`

## KOSPI200 과열 진단 모델 — Peak Give-back Binary (리크 발견 前/後 전체 과정, GCS-only)

이 노트북은 **CSV 업로드 없이 전부 GCS(`gs://finda_project/curated/`)에서 데이터를 불러와서** 실행합니다.

### 이 노트북이 재현하는 전체 흐름
1. 사건 정의: 20일 수익률≥20% AND 거래량비≥1.5 (동시충족), cooldown 20거래일
2. 라벨: Peak Give-back, horizon **40일** + MDD20 ≤ -25% 우선조건(override)
3. **[1단계] 리크 포함 버전** — 피처 20개(`vol_sustain_5d/10d/20d/30d`, `vol_trend_7_13` 포함)로
   먼저 성능을 확인한다. 이 시점에는 이 5개 변수가 사건일(T) 이후 정보를 담고 있다는 사실을
   아직 모른다고 가정하고 진행한다.
4. **[리크 발견]** Feature Importance를 다시 들여다보며 `vol_sustain_*`, `vol_trend_7_13`이
   실제로는 사건일 T **이후**의 거래량 데이터로 계산된 값임을 확인한다 (STEP 5 코드에서
   `pos0+1`부터, 즉 사건일 다음날부터 값을 집계하고 있음).
5. **[2단계] 리크 제거 후 재검증** — 위 5개 변수를 제거한 15개 피처로 동일한 조건(RandomForest,
   5-fold walk-forward + SMOTE, 확신도 기반 선택출력, 10-seed 안정성 검증)으로 다시 학습하고,
   리크 포함 버전과 정량적으로 직접 비교한다.
6. 모델: RandomForest (`max_depth=5, min_samples_leaf=10, n_estimators=300`)
7. 보조지표: KNN(K=40) 기반 유사사례 이웃과열비율 (리크 포함 피처 기준)

### 결론 미리보기
리크 포함 버전(20개 피처)은 confidence≥0.4 기준 precision 0.704 / recall 0.594를 보였지만,
리크 제거 후(15개 피처)에는 precision 0.473 / recall 0.242로 크게 하락했다. 이는 초기 고성능의
상당 부분이 미래정보(사건 이후 거래량)를 몰래 사용한 결과였음을 보여준다. 이 발견이 이후
Peak Give-back 라벨 자체의 구조적 문제(Target-Feature 순환, Time-to-peak bias, MDD 경로 편향)를
재검토하고 Direct T+20 BHAR + q35로 전환하는 계기가 되었다.

### STEP 0: 환경 설정 + 데이터 로드 (GCS only)

In [ ]:
import pandas as pd
import numpy as np
import os
import subprocess
import pickle
import itertools
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score
from scipy.stats import spearmanr
from imblearn.over_sampling import SMOTE

SAVE_DIR = '/content/kospi200_project'  # 로컬 저장 (팀 GCS/Drive에 쓰지 않음)
os.makedirs(SAVE_DIR, exist_ok=True)

# ── 가격 데이터 ──
# (재사용) STAGE 3에서 이미 받아온 stock_daily를 그대로 사용합니다.
price_all = stock_daily.copy()
price_all = price_all.rename(columns={'ticker': '종목코드'})
price_all_clean = price_all[price_all['종목코드'] != '001260'].copy()
print(f"price_all_clean: {price_all_clean.shape}, 종목수: {price_all_clean['종목코드'].nunique()}")

# ── 관심지표 데이터 (파생 9개 Feature, 재사용: STAGE 3의 attention_daily) ──
# search_pct_50/news_pct_50 등은 STEP 5의 '상위10% 신규진입' 리드타임 계산에 계속 사용합니다.
attention_df = attention_daily.copy()
attention_df = attention_df.rename(columns={'ticker': '종목코드', 'date': '날짜'})
attention_df['날짜'] = pd.to_datetime(attention_df['날짜'])
attention_df = attention_df.dropna(subset=['날짜']).copy()
attention_df = attention_df[attention_df['종목코드'] != '001260'].copy()
print(f"attention_df: {attention_df.shape}, 종목수: {attention_df['종목코드'].nunique()}")


price_codes = set(price_all_clean['종목코드'].unique())
attention_codes = set(attention_df['종목코드'].unique())
print(f"\nprice == attention: {price_codes == attention_codes}")

### STEP 1~3: 시장지수 + 종목별 일별 피처 + 관심지표 병합

In [ ]:
# ── STEP 1: 시장지수(KOSPI200 proxy, 시가총액가중) ──
price = price_all_clean.copy()
price['date'] = pd.to_datetime(price['date'])
price = price.rename(columns={'date': '날짜'})
price = price.sort_values(['종목코드', '날짜']).reset_index(drop=True)

price_wide_close = price.pivot(index='날짜', columns='종목코드', values='close')
ret_wide = price_wide_close.pct_change(fill_method=None)
mktcap_wide = price.pivot(index='날짜', columns='종목코드', values='market_cap').shift(1)
weights = mktcap_wide.div(mktcap_wide.sum(axis=1), axis=0)
market_daily_return = (ret_wide * weights).sum(axis=1)

market_index = pd.DataFrame({'날짜': market_daily_return.index})
market_index['시장일수익률'] = market_daily_return.values
market_index = market_index.sort_values('날짜').reset_index(drop=True)
market_index['시장지수'] = (1 + market_index['시장일수익률'].fillna(0)).cumprod() * 100
print(f"market_index: {market_index.shape}")

# ── STEP 2: 종목별 일별 피처 ──
df = price.merge(market_index[['날짜', '시장일수익률', '시장지수']], on='날짜', how='left')
df['일수익률'] = df.groupby('종목코드')['close'].pct_change(fill_method=None)
g = df.groupby('종목코드', group_keys=False)

df['로그시가총액'] = np.log(df['market_cap'].replace(0, np.nan))
df['시장_60일변동성'] = market_index.set_index('날짜')['시장일수익률'].rolling(60).std().reindex(df['날짜']).values
df['5일_변동성'] = g['일수익률'].apply(lambda x: x.rolling(5).std())
df['60일_변동성'] = g['일수익률'].apply(lambda x: x.rolling(60).std())
df['거래량_20일평균'] = g['volume'].apply(lambda x: x.rolling(20).mean())
df['거래량증가율_20일평균대비'] = df['volume'] / df['거래량_20일평균']
df['amihud_daily'] = df['일수익률'].abs() / (df['trading_value'] / 1e8).replace(0, np.nan)
df['amihud_20일'] = g['amihud_daily'].apply(lambda x: x.rolling(20).mean())
df['개인_5일누적'] = g['individual_net_buy_value'].apply(lambda x: x.rolling(5).sum())
df['거래대금_5일누적'] = g['trading_value'].apply(lambda x: x.rolling(5).sum())
df['개인_순매수비율_5일'] = df['개인_5일누적'] / df['거래대금_5일누적']
df['외국인_순매수_20일누적금액'] = g['foreign_net_buy_value'].apply(lambda x: x.rolling(20).sum())
df['외국인_보유비중_레벨'] = df['foreign_ownership_pct']
print(f"STEP2 후 df: {df.shape}")

# ── STEP 3: 관심지표 병합 (GCS attention_daily의 파생 9개 Feature 그대로 사용) ──
attention_sorted = attention_df.sort_values(['종목코드', '날짜']).reset_index(drop=True)

ATTENTION_COLS = [
    'search_pct_50', 'search_ratio_7_30', 'search_slope_7', 'search_high_share_7',
    'news_pct_50', 'news_ratio_7_30', 'news_slope_7', 'news_high_share_7',
    'search_news_level_gap',
]

df = df.merge(attention_sorted[['종목코드', '날짜'] + ATTENTION_COLS], on=['종목코드', '날짜'], how='left')

# 아래 STEP5(리드타임 계산) 셀이 naver_sorted/news_sorted 이름을 참조하므로 유지
naver_sorted = attention_sorted.copy()
news_sorted = attention_sorted.copy()

print(f"\n최종 df shape: {df.shape}")
df.to_pickle(f'{SAVE_DIR}/df_step3_checkpoint.pkl')
market_index.to_pickle(f'{SAVE_DIR}/market_index_checkpoint.pkl')

### STEP 4: 사건(Event) 탐지 + BHAR/MDD 계산

In [ ]:
df = df.sort_values(['종목코드', '날짜']).reset_index(drop=True)
g = df.groupby('종목코드', group_keys=False)

df['수익률_20일'] = g['close'].apply(lambda x: x.pct_change(20, fill_method=None))
df['조건_수익률'] = df['수익률_20일'] >= 0.20
df['조건_거래량'] = df['거래량증가율_20일평균대비'] >= 1.5
df['급등조건충족'] = df['조건_수익률'] & df['조건_거래량']
print(f"급등조건 충족 거래일 수: {df['급등조건충족'].sum()}")

def find_event_days_trading_cooldown(group, cooldown_trading_days=20):
    cond = group['급등조건충족'].values
    events_flag = np.zeros(len(cond), dtype=bool)
    prev = False
    last_kept_pos = -np.inf
    for i in range(len(cond)):
        if cond[i] and not prev:
            if i - last_kept_pos >= cooldown_trading_days:
                events_flag[i] = True
                last_kept_pos = i
        prev = cond[i]
    return events_flag

df['사건일여부'] = g.apply(
    lambda x: pd.Series(find_event_days_trading_cooldown(x), index=x.index),
    include_groups=False
).values

events = df[df['사건일여부']].copy().reset_index(drop=True)
print(f"탐지된 사건 수: {len(events)}, 종목 수: {events['종목코드'].nunique()}")

# ── BHAR/MDD 계산 (peak give-back 라벨링 및 극단낙폭 안전장치에 필요) ──
df = df.sort_values(['종목코드', '날짜']).reset_index(drop=True)
ticker_groups = {t: gdf.reset_index(drop=True) for t, gdf in df.groupby('종목코드')}
market_idx_series = market_index.set_index('날짜')['시장지수']

def get_pos(code, date):
    gdf = ticker_groups.get(code)
    if gdf is None:
        return None, None
    arr = gdf.index[gdf['날짜'] == date]
    return gdf, (arr[0] if len(arr) > 0 else None)

def compute_bhar_mdd(row, horizons=(20, 60)):
    code, event_date = row['종목코드'], row['날짜']
    gdf, pos0 = get_pos(code, event_date)
    result = {f'BHAR{h}': np.nan for h in horizons}
    result['MDD20'] = np.nan
    if pos0 is None:
        return pd.Series(result)
    close0 = gdf.loc[pos0, 'close']
    for h in horizons:
        pos_h = pos0 + h
        if pos_h >= len(gdf):
            continue
        p_end = gdf.loc[pos_h, 'close']
        stock_ret = p_end / close0 - 1
        mkt_start = market_idx_series.get(event_date)
        mkt_end_date = gdf.loc[pos_h, '날짜']
        mkt_end = market_idx_series.get(mkt_end_date)
        if mkt_start is None or mkt_end is None:
            continue
        mkt_ret = mkt_end / mkt_start - 1
        result[f'BHAR{h}'] = stock_ret - mkt_ret
    pos_end20 = min(pos0 + 20, len(gdf) - 1)
    if pos_end20 > pos0:
        window = gdf.loc[pos0:pos_end20].copy()
        window['stock_cum'] = window['close'] / close0 - 1
        mkt0 = market_idx_series.get(event_date)
        window['mkt_cum'] = window['날짜'].map(market_idx_series) / mkt0 - 1
        window['bhar_path'] = window['stock_cum'] - window['mkt_cum']
        running_max = window['bhar_path'].cummax()
        drawdown = window['bhar_path'] - running_max
        result['MDD20'] = drawdown.min()
    return pd.Series(result)

bhar_results = events.apply(compute_bhar_mdd, axis=1)
events = pd.concat([events, bhar_results], axis=1)
print(f"BHAR/MDD 계산 완료: {events.shape}")

events.to_pickle(f'{SAVE_DIR}/events_step4_checkpoint.pkl')
df.to_pickle(f'{SAVE_DIR}/df_step4_checkpoint.pkl')

### STEP 5: 리드타임 / 과거급등이력 / 거래량지속(vol_sustain) 피처

⚠️ 아래 `vol_sustain_20d`, `vol_sustain_30d`, `vol_trend_7_13`는 **사건일 이후** 거래량 정보를 담고 있어
리크 가능성이 있는 피처입니다. 이 노트북은 **리크 제거 前 버전**이므로 그대로 포함해서 계산합니다.

'상위10% 관심 신규진입' 플래그는 GCS의 `search_pct_50`/`news_pct_50`(이미 계산된 rolling percentile)이
0.90 이상인 시점으로 정의합니다.

In [ ]:
naver_lead = naver_sorted.sort_values(['종목코드','날짜']).copy()
naver_lead['검색관심_상위10pct'] = naver_lead['search_pct_50'] >= 0.90
naver_lead['검색_신규진입'] = naver_lead['검색관심_상위10pct'] & (~naver_lead.groupby('종목코드')['검색관심_상위10pct'].shift(1).fillna(False))

news_lead = news_sorted.sort_values(['종목코드','날짜']).copy()
news_lead['뉴스관심_상위10pct'] = news_lead['news_pct_50'] >= 0.90
news_lead['뉴스_신규진입'] = news_lead['뉴스관심_상위10pct'] & (~news_lead.groupby('종목코드')['뉴스관심_상위10pct'].shift(1).fillna(False))

search_groups = {t: gdf.reset_index(drop=True) for t, gdf in naver_lead.groupby('종목코드')}
news_groups = {t: gdf.reset_index(drop=True) for t, gdf in news_lead.groupby('종목코드')}

def find_lead_days(groups_dict, code, event_date, flag_col, window_days=28):
    gdf = groups_dict.get(code)
    if gdf is None:
        return np.nan
    window_start = event_date - pd.Timedelta(days=window_days)
    cand = gdf[(gdf['날짜'] > window_start) & (gdf['날짜'] <= event_date) & (gdf[flag_col])]
    if len(cand) == 0:
        return np.nan
    entry_date = cand['날짜'].min()
    return (event_date - entry_date).days

events = events.sort_values(['종목코드', '날짜']).reset_index(drop=True)
prior_count = {}
extra_rows = []

for _, ev in events.iterrows():
    code, edate = ev['종목코드'], ev['날짜']
    search_lead = find_lead_days(search_groups, code, edate, '검색_신규진입')
    news_lead_ = find_lead_days(news_groups, code, edate, '뉴스_신규진입')
    lead_diff = search_lead - news_lead_ if pd.notna(search_lead) and pd.notna(news_lead_) else np.nan

    prior_count[code] = prior_count.get(code, 0)
    hist_count = prior_count[code]
    prior_count[code] += 1

    gdf, pos0 = get_pos(code, edate)
    vol_feats = {'vol_sustain_5d': np.nan, 'vol_sustain_10d': np.nan,
                 'vol_sustain_20d': np.nan, 'vol_sustain_30d': np.nan, 'vol_trend_7_13': np.nan}
    if pos0 is not None:
        for h, key in [(5,'vol_sustain_5d'),(10,'vol_sustain_10d'),(20,'vol_sustain_20d'),(30,'vol_sustain_30d')]:
            pos_end = min(pos0 + h, len(gdf) - 1)
            if pos_end > pos0:
                vol_feats[key] = gdf.loc[pos0+1:pos_end, '거래량증가율_20일평균대비'].mean()
        pos_end7 = min(pos0+7, len(gdf)-1)
        pos_end20 = min(pos0+20, len(gdf)-1)
        first7 = gdf.loc[pos0+1:pos_end7, '거래량증가율_20일평균대비'].mean() if pos_end7>pos0 else np.nan
        next13 = gdf.loc[pos0+8:pos_end20, '거래량증가율_20일평균대비'].mean() if pos_end20>pos0+7 else np.nan
        if pd.notna(first7) and first7 != 0:
            vol_feats['vol_trend_7_13'] = next13 / first7

    extra_rows.append({
        '종목코드': code, '날짜': edate,
        '검색_리드타임': search_lead, '뉴스_리드타임': news_lead_, '검색_뉴스_리드타임차이': lead_diff,
        '과거_급등이력_횟수': hist_count,
        **vol_feats
    })

extra_df = pd.DataFrame(extra_rows)
events = events.merge(extra_df, on=['종목코드', '날짜'], how='left')
print(f"피처 완성 후 shape: {events.shape}")
events.to_pickle(f'{SAVE_DIR}/events_step5_checkpoint.pkl')

### [1단계] 리크 포함 버전 — 최종 피처 목록 (20개, vol_sustain 5개 포함)

이 시점에는 아래 5개 변수(`vol_sustain_5d/10d/20d/30d`, `vol_trend_7_13`)가 사건일 이후
정보를 담고 있다는 사실을 **아직 모른다고 가정**하고 그대로 포함해서 진행한다.
(리크 발견 및 재검증은 이 노트북 뒤쪽 STEP 6~7에서 다룬다.)

`search_ratio_7_30`/`news_ratio_7_30`(7일평균/30일평균 비율)을 검색·뉴스 관심 강도
지표로 사용합니다. 원래 공식(log1p 7일합 − log1p 8주중앙값 기반 `검색_로그관심도`)은
`급등기준3개_관심지표_GCS_실제KOSPI200_최종.ipynb`에서 확인했지만, 이 노트북이 처음
horizon=40일을 채택했던 시점의 수치와 재현성을 맞추기 위해 의도적으로 이 대체 변수를
그대로 유지합니다.

공매도 변수는 A/B 비교에서 분류 성능에 유의미한 개선이 없어 최종 목록에서 제외했습니다.

In [ ]:
FEATURES_FINAL = [
    # 검색·뉴스 (연구질문 핵심, 무조건 유지) — 재현성 위해 40일 도출 당시 변수 유지
    'search_ratio_7_30', '검색_리드타임', 'news_ratio_7_30', '뉴스_리드타임', '검색_뉴스_리드타임차이',
    # 기업규모
    '로그시가총액',
    # 변동성
    '시장_60일변동성', '5일_변동성', '60일_변동성',
    # 거래량·유동성
    '거래량증가율_20일평균대비', 'amihud_20일',
    # 수급
    '개인_순매수비율_5일', '외국인_순매수_20일누적금액', '외국인_보유비중_레벨',
    # 가격 모멘텀
    '수익률_20일',
    # 거래량 지속성 (⚠️ 리크 제거 前 버전이라 그대로 포함)
    'vol_sustain_5d', 'vol_sustain_10d', 'vol_sustain_20d', 'vol_sustain_30d', 'vol_trend_7_13',
]

print(f"최종 피처 수: {len(FEATURES_FINAL)}개")

def prepare_X(df, cols):
    X = df[cols].copy()
    X = X.replace([np.inf, -np.inf], np.nan)
    for col in cols:
        if X[col].isnull().any():
            X[col] = X[col].fillna(X[col].median())
    return X

In [ ]:
# horizon 스윕에서 RandomForest 하이퍼파라미터가 필요하므로 미리 정의한다.
# (아래 [1단계] 모델 섹션에서 다시 한 번 정의되지만 같은 값이라 문제 없다)
BEST_PARAMS_40 = {'max_depth': 5, 'min_samples_leaf': 10, 'n_estimators': 300}
print(f"BEST_PARAMS_40 미리 정의 완료: {BEST_PARAMS_40}")

### 관측기간(Horizon) 40일 선택 근거

Peak Give-back 라벨(MDD20≤-25% 우선조건 포함, peak_return≥10% AND giveback_ratio≥50%)을 계산할 때,
"사건일 이후 며칠까지의 경로를 볼 것인가"를 정하는 horizon 파라미터가 별도로 필요했다. 특정 값 하나만
보고 판단하면 우연히 좋게 나온 결과인지 알 수 없기 때문에, **사전에 후보군(20/30/40/60/90일)을 고정한
뒤 동일한 조건으로 일괄 비교**하는 방식을 택했다.

#### 1. 검증 방법

같은 라벨링 로직(MDD override + peak_return/giveback_ratio, 즉 실제 채택된 `classify_40`과 동일한
로직)에 horizon 값만 바꿔가며 5개 후보를 동일한 조건(RandomForest, walk-forward)으로 비교한다.
`MDD20`은 사건일 기준 고정 20일 창으로 계산되는 값이라 horizon과 무관하며, `events`에 이미
계산된 값을 그대로 재사용한다.

```
horizons_to_test = [20, 30, 40, 60, 90]
```

In [ ]:
HORIZONS_TO_TEST = [20, 30, 40, 60, 90]

# MDD20은 사건일 기준 고정 20일 창으로 계산되므로 horizon과 무관하게 events의 값을 그대로 재사용한다.
def compute_peak_giveback_binary(row, horizon):
    code_, event_date = row['종목코드'], row['날짜']
    gdf, pos0 = get_pos(code_, event_date)
    empty = pd.Series({'peak_return': np.nan, 'final_return': np.nan, 'giveback_ratio': np.nan})
    if pos0 is None:
        return empty
    pos_end = pos0 + horizon
    if pos_end >= len(gdf):
        return empty
    close0 = gdf.loc[pos0, 'close']
    mkt0 = market_idx_series.get(event_date)
    if mkt0 is None:
        return empty
    window = gdf.loc[pos0:pos_end].copy()
    window['stock_cum'] = window['close'] / close0 - 1
    window['mkt_cum'] = window['날짜'].map(market_idx_series) / mkt0 - 1
    window['bhar_path'] = window['stock_cum'] - window['mkt_cum']
    peak_idx = window['bhar_path'].idxmax()
    peak_return = window.loc[peak_idx, 'bhar_path']
    final_return = window['bhar_path'].iloc[-1]
    giveback_ratio = (peak_return - final_return) / peak_return if peak_return > 0.01 else np.nan
    return pd.Series({'peak_return': peak_return, 'final_return': final_return, 'giveback_ratio': giveback_ratio})


def classify_binary_with_mdd(row, peak_thresh=0.10, giveback_thresh=0.50, mdd_thresh=-0.25):
    """실제 채택된 classify_40과 동일한 로직(MDD override 포함)을 이진 라벨로 표현"""
    if pd.notna(row.get('MDD20')) and row['MDD20'] <= mdd_thresh:
        return 1
    if pd.isna(row['peak_return']):
        return np.nan
    if row['peak_return'] < peak_thresh:
        return 0
    if pd.isna(row['giveback_ratio']):
        return np.nan
    return int(row['giveback_ratio'] >= giveback_thresh)


def evaluate_horizon(horizon, features=FEATURES_FINAL, n_folds=5, seed=42):
    h_results = events.apply(lambda r: compute_peak_giveback_binary(r, horizon), axis=1)
    ev_h = pd.concat([events.copy(), h_results], axis=1)
    # MDD20은 events에 이미 계산돼 있음 (compute_bhar_mdd, horizon과 무관한 고정 20일 창)
    ev_h['label_h'] = ev_h.apply(classify_binary_with_mdd, axis=1)

    d = ev_h.dropna(subset=['label_h'] + features).sort_values('날짜').reset_index(drop=True)
    if len(d) < 50:
        return None

    X = prepare_X(d, features)
    y = d['label_h'].astype(int)

    fold_bounds = np.linspace(0, len(d), n_folds + 2).astype(int)
    oof_proba = np.full(len(d), np.nan)

    for i in range(1, n_folds + 1):
        train_idx = d.index[:fold_bounds[i]]
        test_idx = d.index[fold_bounds[i]:fold_bounds[i + 1]]
        if len(test_idx) == 0 or y.loc[train_idx].nunique() < 2:
            continue
        X_train, y_train = X.loc[train_idx], y.loc[train_idx]
        X_test = X.loc[test_idx]
        model = RandomForestClassifier(**BEST_PARAMS_40, class_weight='balanced', random_state=seed)
        model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)[:, 1]
        test_positions = d.index.get_indexer(test_idx)
        oof_proba[test_positions] = proba

    d['oof_proba'] = oof_proba
    valid = d.dropna(subset=['oof_proba'])
    if valid['label_h'].nunique() < 2:
        return None

    pred = (valid['oof_proba'] >= 0.5).astype(int)
    macro_f1 = f1_score(valid['label_h'], pred, average='macro')
    try:
        auc = roc_auc_score(valid['label_h'], valid['oof_proba'])
    except ValueError:
        auc = np.nan
    tp = ((pred == 1) & (valid['label_h'] == 1)).sum()
    recall = tp / max((valid['label_h'] == 1).sum(), 1)
    precision = tp / max((pred == 1).sum(), 1)

    return {
        'horizon': horizon, '표본수': len(d),
        'macro F1': round(macro_f1, 3), 'ROC-AUC': round(auc, 3),
        'recall': round(recall, 3), 'precision': round(precision, 3),
    }


horizon_results = []
for h in HORIZONS_TO_TEST:
    print(f"horizon={h}일 평가 중... (MDD override 포함)")
    res = evaluate_horizon(h)
    if res is not None:
        horizon_results.append(res)

horizon_df = pd.DataFrame(horizon_results)
print("\n=== Horizon 후보 5개 비교 결과 (MDD override 포함 — 실제 채택 라벨과 동일 로직) ===")
print(horizon_df.to_string(index=False))

HORIZON_BEST = int(horizon_df.sort_values('macro F1', ascending=False).iloc[0]['horizon'])
print(f"\nmacro F1 기준 최우수 horizon: {HORIZON_BEST}일")

#### 3. 안정성 재검증 — 10회 시드 검증

40일로 채택을 확정한 뒤, 이 개선이 우연이 아닌지 확인하기 위해 하이퍼파라미터를 재탐색
(`max_depth=5, min_samples_leaf=10, n_estimators=300` = `BEST_PARAMS_40`)하고 10회 시드로
반복 검증한다. 확신도(confidence) 기반 선택출력 방식도 이때부터 함께 적용한다.

In [ ]:
h_results_40 = events.apply(lambda r: compute_peak_giveback_binary(r, 40), axis=1)
ev_40_pre = pd.concat([events.copy(), h_results_40], axis=1)
ev_40_pre['label_h'] = ev_40_pre.apply(classify_binary_with_mdd, axis=1)

d40 = ev_40_pre.dropna(subset=['label_h'] + FEATURES_FINAL).sort_values('날짜').reset_index(drop=True)
X40_h = prepare_X(d40, FEATURES_FINAL)
y40_h = d40['label_h'].astype(int)

n_folds = 5
fold_bounds_40h = np.linspace(0, len(d40), n_folds + 2).astype(int)

SEEDS_H = [42, 1, 7, 13, 21, 33, 55, 77, 99, 123]
thresholds_h = [0.3, 0.4]
seed_results_h40 = {t: {'precision': [], 'recall': [], 'coverage': []} for t in thresholds_h}

for seed in SEEDS_H:
    oof_s = np.full(len(d40), np.nan)
    for i in range(1, n_folds + 1):
        train_idx = d40.index[:fold_bounds_40h[i]]
        test_idx = d40.index[fold_bounds_40h[i]:fold_bounds_40h[i + 1]]
        if len(test_idx) == 0 or y40_h.loc[train_idx].nunique() < 2:
            continue
        X_train, y_train = X40_h.loc[train_idx], y40_h.loc[train_idx]
        X_test = X40_h.loc[test_idx]
        model = RandomForestClassifier(**BEST_PARAMS_40, class_weight='balanced', random_state=seed)
        model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)[:, 1]
        test_positions = d40.index.get_indexer(test_idx)
        oof_s[test_positions] = proba

    temp = d40.copy()
    temp['oof_proba_s'] = oof_s
    temp['oof_pred_s'] = (temp['oof_proba_s'] >= 0.5).astype(int)
    temp['확신도_s'] = np.abs(temp['oof_proba_s'] - 0.5) * 2
    valid_s = temp.dropna(subset=['oof_proba_s'])

    for t in thresholds_h:
        subset = valid_s[valid_s['확신도_s'] >= t]
        pred_pos = subset[subset['oof_pred_s'] == 1]
        precision = (pred_pos['label_h'] == 1).mean() if len(pred_pos) > 0 else np.nan
        recall = ((subset['oof_pred_s'] == 1) & (subset['label_h'] == 1)).sum() / max((subset['label_h'] == 1).sum(), 1)
        seed_results_h40[t]['precision'].append(precision)
        seed_results_h40[t]['recall'].append(recall)
        seed_results_h40[t]['coverage'].append(len(subset))

print("=== Horizon=40일, 10회 시드 안정성 검증 (확신도 기반, MDD override 포함) ===")
for t in thresholds_h:
    p = [x for x in seed_results_h40[t]['precision'] if not np.isnan(x)]
    r = seed_results_h40[t]['recall']
    c = seed_results_h40[t]['coverage']
    print(f"\n확신도 >= {t}:")
    print(f"  precision: 평균 {np.mean(p):.3f} (표준편차 {np.std(p):.3f}, 범위 [{np.min(p):.3f}, {np.max(p):.3f}])")
    print(f"  recall:    평균 {np.mean(r):.3f} (표준편차 {np.std(r):.3f}, 범위 [{np.min(r):.3f}, {np.max(r):.3f}])")
    print(f"  평균 표본수: {np.mean(c):.0f}")

#### 4. 결론

5개 후보 중 40일이 판별력(macro F1·AUC)과 표본 확보 양쪽에서 가장 우수했고, 이 결과가 10회
시드 검증에서도 안정적으로 재현되어 최종 채택했다. 단, 이는 20/30/40/60/90일 5개 이산 후보에
대한 grid sweep과 반복검증이며, 40일 근방(예: 35~45일)을 촘촘히 훑는 정밀 민감도 분석이나
통계적 유의성 검정까지 수행한 것은 아니라는 점은 명확히 해둔다.

이렇게 확정된 horizon=40일 위에 이후 MDD20≤-25% 우선조건(override)을 추가해 `classify_40`
라벨(3-way→binary)로 발전시킨다 (다음 셀 참고).

### peak_return / giveback_ratio / MDD 임계값에 대한 안내

horizon(40일)과 달리 `peak_return≥10%`, `giveback_ratio≥50%`, `MDD20≤-25%` 세 임계값은
별도의 민감도 분석(sweep) 없이 관례적 판단으로 설정했다.

one-at-a-time 방식으로 스윕을 시도해본 결과(별도 실행, 이 노트북에는 포함하지 않음),
`giveback_ratio`·`MDD`는 채택값 근방에서 안정적이거나 오히려 역U자 정점을 보이는 등
납득할 만한 패턴이 있었던 반면, `peak_return`은 임계값을 좁힐수록(20%까지) macro F1이
계속 단조 증가하는 패턴을 보였다. 이는 "컷오프를 좁힐수록 기계적으로 더 잘 갈리는"
현상에 가까워, 이 스윕 방법론 자체(one-at-a-time, joint 탐색 아님, 통계 검정 없음)를
신뢰할 만한 근거로 삼기 어렵다고 판단했다.

따라서 이 세 임계값은 **"검증된 최적값"이 아니라 "관례적으로 설정한 값"**이라는 점을
보고서에 명시하며, 별도의 정당화 코드는 이 노트북에 포함하지 않는다. 이는 이후 13절에서
Peak Give-back 라벨 자체를 재설계(MDD 경로 편향 등)하게 된 배경과도 일관된다.

### 라벨: Peak Give-back, Horizon 40일 (리크 제거 前, MDD 우선조건 포함)

- `peak_return`: horizon(40일) 내 최고 BHAR (시장초과수익 경로 기준)
- `final_return`: horizon 시점의 최종 BHAR
- `giveback_ratio`: (peak - final) / peak, 고점 대비 반납 비율
- 극단 낙폭(MDD20 ≤ -25%)은 giveback 계산과 무관하게 단기과열형으로 우선 분류(override)

In [ ]:
HORIZON_FINAL = 40

def compute_peak_giveback(row, horizon=HORIZON_FINAL):
    code_, event_date = row['종목코드'], row['날짜']
    gdf, pos0 = get_pos(code_, event_date)
    if pos0 is None:
        return pd.Series({'peak_return': np.nan, 'final_return': np.nan, 'giveback_ratio': np.nan})
    pos_end = pos0 + horizon
    if pos_end >= len(gdf):
        return pd.Series({'peak_return': np.nan, 'final_return': np.nan, 'giveback_ratio': np.nan})
    close0 = gdf.loc[pos0, 'close']
    mkt0 = market_idx_series.get(event_date)
    if mkt0 is None:
        return pd.Series({'peak_return': np.nan, 'final_return': np.nan, 'giveback_ratio': np.nan})
    window = gdf.loc[pos0:pos_end].copy()
    window['stock_cum'] = window['close'] / close0 - 1
    window['mkt_cum'] = window['날짜'].map(market_idx_series) / mkt0 - 1
    window['bhar_path'] = window['stock_cum'] - window['mkt_cum']
    peak_idx = window['bhar_path'].idxmax()
    peak_return = window.loc[peak_idx, 'bhar_path']
    final_return = window['bhar_path'].iloc[-1]
    giveback_ratio = (peak_return - final_return) / peak_return if peak_return > 0.01 else np.nan
    return pd.Series({'peak_return': peak_return, 'final_return': final_return, 'giveback_ratio': giveback_ratio})

events_40 = events.copy()
h40_results = events_40.apply(compute_peak_giveback, axis=1)
events_40 = pd.concat([events_40, h40_results], axis=1)

def classify_40(row):
    if pd.notna(row.get('MDD20')) and row['MDD20'] <= -0.25:
        return '단기과열형'
    if pd.isna(row['peak_return']):
        return np.nan
    if row['peak_return'] < 0.10:
        return '관망형'
    if pd.isna(row['giveback_ratio']):
        return np.nan
    if row['giveback_ratio'] >= 0.5:
        return '단기과열형'
    elif row['giveback_ratio'] <= 0.2:
        return '지속상승형'
    return '관망형'

events_40['label_40'] = events_40.apply(classify_40, axis=1)
model_df_40 = events_40.dropna(subset=['label_40']).copy().sort_values('날짜').reset_index(drop=True)
model_df_40['label_40_enc'] = (model_df_40['label_40'] == '단기과열형').astype(int)

print(f"최종 표본 수: {len(model_df_40)}")
print(model_df_40['label_40'].value_counts())

X_40 = prepare_X(model_df_40, FEATURES_FINAL)
y_40 = model_df_40['label_40_enc']

### [1단계] 모델: RandomForest (BEST_PARAMS_40) + Walk-forward OOF + 확신도 기반 선택출력 (리크 포함, 20개 피처)

Binary 분류 (단기과열형=1 vs 나머지=0). 5-fold walk-forward, 각 fold 학습 시 SMOTE로 클래스 불균형 보정.
확신도(confidence) = |예측확률 - 0.5| × 2

⚠️ 아래 결과는 리크가 포함된(vol_sustain 계열 포함) 20개 피처 기준이며, 이후 STEP 6~7에서
리크를 발견하고 제거한 뒤 재검증한다.

In [ ]:
BEST_PARAMS_40 = {'max_depth': 5, 'min_samples_leaf': 10, 'n_estimators': 300}

n_folds = 5
fold_bounds_40 = np.linspace(0, len(model_df_40), n_folds + 2).astype(int)

oof_proba_40 = np.full(len(model_df_40), np.nan)

for i in range(1, n_folds + 1):
    train_idx = model_df_40.index[:fold_bounds_40[i]]
    test_idx = model_df_40.index[fold_bounds_40[i]:fold_bounds_40[i+1]]
    if len(test_idx) == 0 or y_40.loc[train_idx].nunique() < 2:
        continue
    X_train, y_train = X_40.loc[train_idx], y_40.loc[train_idx]
    X_test = X_40.loc[test_idx]
    min_c = y_train.value_counts().min()
    if min_c >= 6:
        sm = SMOTE(random_state=42, k_neighbors=min(5, min_c-1))
        X_tr, y_tr = sm.fit_resample(X_train, y_train)
    else:
        X_tr, y_tr = X_train, y_train
    model = RandomForestClassifier(**BEST_PARAMS_40, class_weight='balanced', random_state=42)
    model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_test)[:, 1]
    test_positions = model_df_40.index.get_indexer(test_idx)
    oof_proba_40[test_positions] = proba

model_df_40['oof_proba'] = oof_proba_40
model_df_40['oof_pred'] = (model_df_40['oof_proba'] >= 0.5).astype(int)
model_df_40['확신도'] = np.abs(model_df_40['oof_proba'] - 0.5) * 2

valid_40 = model_df_40.dropna(subset=['oof_proba'])

print("=== 확신도 구간별 성능 ===")
for threshold in [0.0, 0.1, 0.2, 0.3, 0.4]:
    subset = valid_40[valid_40['확신도'] >= threshold]
    if len(subset) == 0:
        continue
    coverage = len(subset) / len(valid_40)
    pred_pos = subset[subset['oof_pred'] == 1]
    precision_pos = (pred_pos['label_40_enc'] == 1).mean() if len(pred_pos) > 0 else np.nan
    recall_pos = ((subset['oof_pred']==1) & (subset['label_40_enc']==1)).sum() / max((subset['label_40_enc']==1).sum(), 1)
    print(f"확신도 >= {threshold:.1f}: 표본 {len(subset):4d}건 (커버리지 {coverage:.1%}), "
          f"precision {precision_pos:.3f}, recall {recall_pos:.3f}")

### [1단계] 10-seed 안정성 검증 (리크 포함, 20개 피처)

In [ ]:
SEEDS = [42, 1, 7, 13, 21, 33, 55, 77, 99, 123]
thresholds_40 = [0.3, 0.4]
seed_results_40 = {t: {'precision': [], 'recall': [], 'coverage': []} for t in thresholds_40}

for seed in SEEDS:
    oof_s = np.full(len(model_df_40), np.nan)
    for i in range(1, n_folds + 1):
        train_idx = model_df_40.index[:fold_bounds_40[i]]
        test_idx = model_df_40.index[fold_bounds_40[i]:fold_bounds_40[i+1]]
        if len(test_idx) == 0 or y_40.loc[train_idx].nunique() < 2:
            continue
        X_train, y_train = X_40.loc[train_idx], y_40.loc[train_idx]
        X_test = X_40.loc[test_idx]
        min_c = y_train.value_counts().min()
        if min_c >= 6:
            sm = SMOTE(random_state=seed, k_neighbors=min(5, min_c-1))
            X_tr, y_tr = sm.fit_resample(X_train, y_train)
        else:
            X_tr, y_tr = X_train, y_train
        model = RandomForestClassifier(**BEST_PARAMS_40, class_weight='balanced', random_state=seed)
        model.fit(X_tr, y_tr)
        proba = model.predict_proba(X_test)[:, 1]
        test_positions = model_df_40.index.get_indexer(test_idx)
        oof_s[test_positions] = proba

    temp = model_df_40.copy()
    temp['oof_proba_s'] = oof_s
    temp['oof_pred_s'] = (temp['oof_proba_s'] >= 0.5).astype(int)
    temp['확신도_s'] = np.abs(temp['oof_proba_s'] - 0.5) * 2
    valid_s = temp.dropna(subset=['oof_proba_s'])

    for t in thresholds_40:
        subset = valid_s[valid_s['확신도_s'] >= t]
        pred_pos = subset[subset['oof_pred_s'] == 1]
        precision = (pred_pos['label_40_enc'] == 1).mean() if len(pred_pos) > 0 else np.nan
        recall = ((subset['oof_pred_s']==1) & (subset['label_40_enc']==1)).sum() / max((subset['label_40_enc']==1).sum(),1)
        seed_results_40[t]['precision'].append(precision)
        seed_results_40[t]['recall'].append(recall)
        seed_results_40[t]['coverage'].append(len(subset))

print("=== 10회 시드 반복 안정성 검증 ===")
for t in thresholds_40:
    p = [x for x in seed_results_40[t]['precision'] if not np.isnan(x)]
    r = seed_results_40[t]['recall']
    c = seed_results_40[t]['coverage']
    print(f"\n확신도 >= {t}:")
    print(f"  precision: 평균 {np.mean(p):.3f}, 표준편차 {np.std(p):.3f}, 범위 [{np.min(p):.3f}, {np.max(p):.3f}]")
    print(f"  recall:    평균 {np.mean(r):.3f}, 표준편차 {np.std(r):.3f}, 범위 [{np.min(r):.3f}, {np.max(r):.3f}]")
    print(f"  평균 표본수: {np.mean(c):.0f}")

### KNN 보조지표 (K=40)

RF가 확신도 낮은(0.4 미만) 사건에 대해 보조적으로 참고할 수 있는 "유사 과거사례 기반" 지표.
시간순을 지켜서(현재 시점보다 과거 사건들 중에서만) 이웃을 탐색해 미래 데이터 누설을 방지.

In [ ]:
X_full = prepare_X(model_df_40, FEATURES_FINAL)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_full), index=X_full.index, columns=X_full.columns)

model_df_40_sorted = model_df_40.sort_values('날짜').reset_index(drop=True)
X_scaled_sorted = X_scaled.loc[model_df_40_sorted.index].reset_index(drop=True)

K_final = 40
neighbor_rates_final = np.full(len(model_df_40_sorted), np.nan)

for i in range(K_final, len(model_df_40_sorted)):
    past_X = X_scaled_sorted.iloc[:i]
    past_y = model_df_40_sorted['label_40_enc'].iloc[:i]
    if len(past_X) < K_final:
        continue
    nn = NearestNeighbors(n_neighbors=K_final)
    nn.fit(past_X)
    _, idx = nn.kneighbors(X_scaled_sorted.iloc[[i]])
    neighbor_rates_final[i] = past_y.iloc[idx[0]].mean()

model_df_40_sorted['이웃과열비율_K40'] = neighbor_rates_final
valid_final = model_df_40_sorted.dropna(subset=['이웃과열비율_K40']).copy()

corr_nn, p_nn = spearmanr(valid_final['이웃과열비율_K40'], valid_final['label_40_enc'])
print(f"이웃과열비율 vs 실제라벨 상관관계: {corr_nn:.3f} (p={p_nn:.4f})")

top20 = valid_final[valid_final['이웃과열비율_K40'] >= valid_final['이웃과열비율_K40'].quantile(0.8)]
bottom20 = valid_final[valid_final['이웃과열비율_K40'] <= valid_final['이웃과열비율_K40'].quantile(0.2)]

print(f"\n상위 20% 이웃비율 그룹 (고위험 추정): 실제 과열비율 {top20['label_40_enc'].mean():.3f}, 표본 {len(top20)}건")
print(f"하위 20% 이웃비율 그룹 (저위험 추정): 실제 과열비율 {bottom20['label_40_enc'].mean():.3f}, 표본 {len(bottom20)}건")

model_df_40 = model_df_40_sorted.copy()

### [1단계] 최종모델 학습 + 성능 결과 요약 (리크 포함, 20개 피처)

이 결과는 아직 리크를 제거하지 않은 상태의 성능이다. 이 시점에서는 이 성능을 최종 성능으로 오인할 수 있는 상태다.

In [ ]:
X_40 = prepare_X(model_df_40, FEATURES_FINAL)
y_40 = model_df_40['label_40_enc']

final_model_40 = RandomForestClassifier(**BEST_PARAMS_40, class_weight='balanced', random_state=42)
final_model_40.fit(X_40, y_40)

print(f"X_40 shape: {X_40.shape}")
print(f"final_model_40 재학습 완료, 피처 수: {len(FEATURES_FINAL)}개")

print("=" * 60)
print("KOSPI200 과열 진단 모델 — 최종 성능 요약 (리크 제거 前)")
print("=" * 60)

print(f"\n[데이터]")
print(f"  전체 사건 수: {len(model_df_40)}건")
print(f"  단기과열형(1): {(model_df_40['label_40_enc']==1).sum()}건")
print(f"  나머지(0): {(model_df_40['label_40_enc']==0).sum()}건")
print(f"  피처 수: {len(FEATURES_FINAL)}개")
print(f"  라벨: Peak Give-back (horizon={HORIZON_FINAL}일)")
print(f"  모델: RandomForest {BEST_PARAMS_40}")

print(f"\n[확신도 구간별 성능 (seed=42 단일 실행)]")
summary_rows = []
for threshold in [0.0, 0.1, 0.2, 0.3, 0.4]:
    subset = valid_40[valid_40['확신도'] >= threshold]
    if len(subset) == 0:
        continue
    coverage = len(subset) / len(valid_40)
    pred_pos = subset[subset['oof_pred'] == 1]
    precision_pos = (pred_pos['label_40_enc'] == 1).mean() if len(pred_pos) > 0 else np.nan
    recall_pos = ((subset['oof_pred']==1) & (subset['label_40_enc']==1)).sum() / max((subset['label_40_enc']==1).sum(), 1)
    summary_rows.append({
        '확신도threshold': threshold, '표본수': len(subset), '커버리지': f"{coverage:.1%}",
        'precision': round(precision_pos, 3), 'recall': round(recall_pos, 3)
    })
print(pd.DataFrame(summary_rows).to_string(index=False))

print(f"\n[10-seed 안정성 검증 — 확신도 >= 0.4]")
t_final = 0.4
p = [x for x in seed_results_40[t_final]['precision'] if not np.isnan(x)]
r = seed_results_40[t_final]['recall']
c = seed_results_40[t_final]['coverage']
print(f"  precision: 평균 {np.mean(p):.3f} (표준편차 {np.std(p):.3f}, 범위 [{np.min(p):.3f}, {np.max(p):.3f}])")
print(f"  recall:    평균 {np.mean(r):.3f} (표준편차 {np.std(r):.3f}, 범위 [{np.min(r):.3f}, {np.max(r):.3f}])")
print(f"  coverage:  평균 {np.mean(c)/len(model_df_40):.1%} (평균 표본수 {np.mean(c):.0f}건)")

print(f"\n[KNN 보조지표 (K={K_final})]")
corr_nn, p_nn = spearmanr(valid_final['이웃과열비율_K40'], valid_final['label_40_enc'])
print(f"  이웃과열비율 vs 실제라벨 상관관계: {corr_nn:.3f} (p={p_nn:.4f})")
print(f"  상위 20% 그룹 실제과열비율: {top20['label_40_enc'].mean():.3f} (n={len(top20)})")
print(f"  하위 20% 그룹 실제과열비율: {bottom20['label_40_enc'].mean():.3f} (n={len(bottom20)})")

print(f"\n[전체 피처 중요도 상위 10개]")
importance_final = pd.Series(final_model_40.feature_importances_, index=FEATURES_FINAL).sort_values(ascending=False)
print(importance_final.head(10).round(4).to_string())

print("\n" + "=" * 60)

### STEP 6: 리크 발견 — Feature Importance 재검토

[1단계]에서 학습한 `final_model_40`의 Feature Importance 상위권에 `vol_sustain_10d`, `vol_sustain_20d`,
`vol_sustain_5d`, `vol_sustain_30d`, `vol_trend_7_13`이 나란히 올라와 있다.

이 변수들의 실제 계산 로직(STEP 5)을 다시 보면:

```python
for h, key in [(5,'vol_sustain_5d'),(10,'vol_sustain_10d'),(20,'vol_sustain_20d'),(30,'vol_sustain_30d')]:
    pos_end = min(pos0 + h, len(gdf) - 1)
    if pos_end > pos0:
        vol_feats[key] = gdf.loc[pos0+1:pos_end, '거래량증가율_20일평균대비'].mean()
```

`gdf.loc[pos0+1:pos_end, ...]` — 즉 **사건일(T) 다음날부터 T+h까지**의 거래량 데이터를 평균 낸 값이다.
T 시점에 이 값을 안다고 가정하고 모델을 만들었지만, 실제로는 T **이후** 정보가 이미 섞여 들어가 있었다.
`vol_trend_7_13`도 동일한 방식(T+1~T+7, T+8~T+20 평균 비율)으로 계산되어 같은 문제를 가진다.

**즉 이 5개 변수는 미래정보 누수(Data Leakage) 변수였다.**

In [ ]:
print("=== [1단계] 리크 포함 모델의 Feature Importance 상위 10개 (재확인) ===")
print(importance_final.head(10).round(4).to_string())

LEAK_FEATURES = ['vol_sustain_5d', 'vol_sustain_10d', 'vol_sustain_20d', 'vol_sustain_30d', 'vol_trend_7_13']
leak_rank = importance_final.index.get_indexer(LEAK_FEATURES)
print("\n=== 리크 의심 변수 5개의 순위 (0-indexed) ===")
for feat, rank in zip(LEAK_FEATURES, leak_rank):
    print(f"  {feat}: 전체 {len(importance_final)}개 중 {rank+1}위 (importance={importance_final[feat]:.4f})")

### STEP 7: 리크 제거 후 재검증 — 15개 피처로 동일 조건 재학습

리크 변수 5개(`vol_sustain_5d/10d/20d/30d`, `vol_trend_7_13`)를 제거하고, 나머지 15개 피처만으로
**동일한 조건**(RandomForest `BEST_PARAMS_40`, 5-fold walk-forward + SMOTE, 확신도 기반 선택출력)에서
다시 검증한다. 라벨(`label_40_enc`)과 fold 구성(`fold_bounds_40`)은 [1단계]와 동일하게 유지해
"리크를 뺐다"는 것 외에는 아무것도 바뀌지 않도록 한다.

In [ ]:
FEATURES_NO_LEAK = [f for f in FEATURES_FINAL if f not in LEAK_FEATURES]
print(f"[2단계] 리크 제거 후 피처 수: {len(FEATURES_NO_LEAK)}개")
print(FEATURES_NO_LEAK)

X_40_noleak = prepare_X(model_df_40, FEATURES_NO_LEAK)
y_40_noleak = y_40  # 라벨은 동일

oof_proba_40_noleak = np.full(len(model_df_40), np.nan)

for i in range(1, n_folds + 1):
    train_idx = model_df_40.index[:fold_bounds_40[i]]
    test_idx = model_df_40.index[fold_bounds_40[i]:fold_bounds_40[i+1]]
    if len(test_idx) == 0 or y_40_noleak.loc[train_idx].nunique() < 2:
        continue
    X_train, y_train = X_40_noleak.loc[train_idx], y_40_noleak.loc[train_idx]
    X_test = X_40_noleak.loc[test_idx]
    min_c = y_train.value_counts().min()
    if min_c >= 6:
        sm = SMOTE(random_state=42, k_neighbors=min(5, min_c-1))
        X_tr, y_tr = sm.fit_resample(X_train, y_train)
    else:
        X_tr, y_tr = X_train, y_train
    model = RandomForestClassifier(**BEST_PARAMS_40, class_weight='balanced', random_state=42)
    model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_test)[:, 1]
    test_positions = model_df_40.index.get_indexer(test_idx)
    oof_proba_40_noleak[test_positions] = proba

model_df_40['oof_proba_noleak'] = oof_proba_40_noleak
model_df_40['oof_pred_noleak'] = (model_df_40['oof_proba_noleak'] >= 0.5).astype(int)
model_df_40['확신도_noleak'] = np.abs(model_df_40['oof_proba_noleak'] - 0.5) * 2

valid_40_noleak = model_df_40.dropna(subset=['oof_proba_noleak'])

print("\n=== [2단계] 리크 제거 후 확신도 구간별 성능 ===")
summary_rows_noleak = []
for threshold in [0.0, 0.1, 0.2, 0.3, 0.4]:
    subset = valid_40_noleak[valid_40_noleak['확신도_noleak'] >= threshold]
    if len(subset) == 0:
        continue
    coverage = len(subset) / len(valid_40_noleak)
    pred_pos = subset[subset['oof_pred_noleak'] == 1]
    precision_pos = (pred_pos['label_40_enc'] == 1).mean() if len(pred_pos) > 0 else np.nan
    recall_pos = ((subset['oof_pred_noleak']==1) & (subset['label_40_enc']==1)).sum() / max((subset['label_40_enc']==1).sum(), 1)
    summary_rows_noleak.append({
        '확신도threshold': threshold, '표본수': len(subset), '커버리지': f"{coverage:.1%}",
        'precision': round(precision_pos, 3), 'recall': round(recall_pos, 3)
    })
print(pd.DataFrame(summary_rows_noleak).to_string(index=False))

### [2단계] 10-seed 안정성 검증 (리크 제거 후, 15개 피처)

In [ ]:
seed_results_40_noleak = {t: {'precision': [], 'recall': [], 'coverage': []} for t in thresholds_40}

for seed in SEEDS:
    oof_s = np.full(len(model_df_40), np.nan)
    for i in range(1, n_folds + 1):
        train_idx = model_df_40.index[:fold_bounds_40[i]]
        test_idx = model_df_40.index[fold_bounds_40[i]:fold_bounds_40[i+1]]
        if len(test_idx) == 0 or y_40_noleak.loc[train_idx].nunique() < 2:
            continue
        X_train, y_train = X_40_noleak.loc[train_idx], y_40_noleak.loc[train_idx]
        X_test = X_40_noleak.loc[test_idx]
        min_c = y_train.value_counts().min()
        if min_c >= 6:
            sm = SMOTE(random_state=seed, k_neighbors=min(5, min_c-1))
            X_tr, y_tr = sm.fit_resample(X_train, y_train)
        else:
            X_tr, y_tr = X_train, y_train
        model = RandomForestClassifier(**BEST_PARAMS_40, class_weight='balanced', random_state=seed)
        model.fit(X_tr, y_tr)
        proba = model.predict_proba(X_test)[:, 1]
        test_positions = model_df_40.index.get_indexer(test_idx)
        oof_s[test_positions] = proba

    temp = model_df_40.copy()
    temp['oof_proba_s'] = oof_s
    temp['oof_pred_s'] = (temp['oof_proba_s'] >= 0.5).astype(int)
    temp['확신도_s'] = np.abs(temp['oof_proba_s'] - 0.5) * 2
    valid_s = temp.dropna(subset=['oof_proba_s'])

    for t in thresholds_40:
        subset = valid_s[valid_s['확신도_s'] >= t]
        pred_pos = subset[subset['oof_pred_s'] == 1]
        precision = (pred_pos['label_40_enc'] == 1).mean() if len(pred_pos) > 0 else np.nan
        recall = ((subset['oof_pred_s']==1) & (subset['label_40_enc']==1)).sum() / max((subset['label_40_enc']==1).sum(),1)
        seed_results_40_noleak[t]['precision'].append(precision)
        seed_results_40_noleak[t]['recall'].append(recall)
        seed_results_40_noleak[t]['coverage'].append(len(subset))

print("=== [2단계] 10회 시드 반복 안정성 검증 (리크 제거 후) ===")
for t in thresholds_40:
    p = [x for x in seed_results_40_noleak[t]['precision'] if not np.isnan(x)]
    r = seed_results_40_noleak[t]['recall']
    c = seed_results_40_noleak[t]['coverage']
    print(f"\n확신도 >= {t}:")
    print(f"  precision: 평균 {np.mean(p):.3f}, 표준편차 {np.std(p):.3f}, 범위 [{np.min(p):.3f}, {np.max(p):.3f}]")
    print(f"  recall:    평균 {np.mean(r):.3f}, 표준편차 {np.std(r):.3f}, 범위 [{np.min(r):.3f}, {np.max(r):.3f}]")
    print(f"  평균 표본수: {np.mean(c):.0f}")

### 리크 포함 vs 리크 제거 — 직접 비교 (확신도 ≥ 0.4, 10-seed 평균)

In [ ]:
t_cmp = 0.4

p_leak = np.mean([x for x in seed_results_40[t_cmp]['precision'] if not np.isnan(x)])
r_leak = np.mean(seed_results_40[t_cmp]['recall'])
c_leak = np.mean(seed_results_40[t_cmp]['coverage'])
cov_leak = c_leak / len(model_df_40)

p_noleak = np.mean([x for x in seed_results_40_noleak[t_cmp]['precision'] if not np.isnan(x)])
r_noleak = np.mean(seed_results_40_noleak[t_cmp]['recall'])
c_noleak = np.mean(seed_results_40_noleak[t_cmp]['coverage'])
cov_noleak = c_noleak / len(model_df_40)

comparison = pd.DataFrame({
    '리크 포함(20개 피처)': [round(p_leak,3), round(r_leak,3), f"{c_leak:.0f}건 ({cov_leak:.1%})"],
    '리크 제거(15개 피처)': [round(p_noleak,3), round(r_noleak,3), f"{c_noleak:.0f}건 ({cov_noleak:.1%})"],
    '변화': [round(p_noleak-p_leak,3), round(r_noleak-r_leak,3), f"{(cov_noleak-cov_leak)*100:+.1f}%p"]
}, index=['Precision', 'Recall', '표본수(커버리지)'])

print("=== 리크 포함 vs 리크 제거 — 직접 비교 (확신도≥0.4, 10-seed 평균) ===")
print(comparison.to_string())

print(f"\nRecall이 {r_leak:.3f} -> {r_noleak:.3f}로 하락했다 — "
      f"실제 과열사건 10건 중 {r_leak*10:.1f}건 가까이 잡아내던 모델이, "
      f"리크를 제거하자 {r_noleak*10:.1f}건 수준으로 떨어졌다는 뜻이다.")

### 리크 제거 후 Feature Importance 재확인

In [ ]:
final_model_40_noleak = RandomForestClassifier(**BEST_PARAMS_40, class_weight='balanced', random_state=42)
final_model_40_noleak.fit(X_40_noleak, y_40_noleak)

importance_noleak = pd.Series(final_model_40_noleak.feature_importances_, index=FEATURES_NO_LEAK).sort_values(ascending=False)
print("=== 리크 제거 후 Feature Importance 상위 10개 ===")
print(importance_noleak.head(10).round(4).to_string())

print("\n리크 변수 제거 후에는 변동성 계열(시장_60일변동성, 60일_변동성, 5일_변동성 등)이 상위를 차지하고,")
print("vol_sustain 계열의 빈자리를 검색·뉴스·수급 변수들이 메웠지만 이전만큼 뚜렷한 신호를 주지는 못한다.")

### STEP 6~7 결론

초기(1단계, 리크 포함 20개 피처)에는 confidence≥0.4 기준 precision 0.70대, recall 0.59대의
높은 성능이 관찰됐다. 하지만 Feature Importance 재검토(STEP 6) 과정에서 상위권 변수들이
사건일 이후 거래량 데이터를 사용해 계산된 Data Leakage 변수임이 드러났다.

이 변수들을 제거하고 재검증(STEP 7, 15개 피처)한 결과 precision·recall 모두 크게 하락했다 —
특히 recall이 크게 무너져, 실제 과열사건 10건 중 6건 가까이 잡아내던 모델이 2~3건밖에
잡지 못하는 수준으로 떨어졌다. 안정성(10-seed 표준편차)도 함께 악화됐다.

**따라서 리크 포함 버전의 성능은 최종 성능으로 채택하지 않고 폐기한다.**
사건일(T) 시점 정보만으로는 이 정도의 예측력을 낼 수 없다는 사실이 이 재검증으로 명확히 드러났으며,
이 발견이 이후 "왜 사건 이후의 거래량·가격경로는 그렇게 강한 신호였을까?"라는 질문— 즉
T+3/T+5/T+7/T+10 Prediction Timing 비교와 Direct T+20 BHAR + q35로의 전환으로 이어졌다.

### 최종 산출물 저장

리크 포함(1단계)·리크 제거(2단계) 버전을 모두 저장한다. `model_df_40`에는
두 버전의 OOF 예측 결과(`oof_proba`/`oof_proba_noleak` 등)가 함께 들어있다.

In [ ]:
model_df_40.to_pickle('/content/model_df_40_FINAL_leaky_and_noleak.pkl')
events_40.to_pickle('/content/events_40_FINAL.pkl')

with open('/content/final_model_40_leaky.pkl', 'wb') as f:
    pickle.dump(final_model_40, f)

with open('/content/final_model_40_noleak.pkl', 'wb') as f:
    pickle.dump(final_model_40_noleak, f)

comparison.to_csv('/content/leak_vs_noleak_comparison.csv')

print("저장 완료:")
print("1. model_df_40_FINAL_leaky_and_noleak.pkl (라벨링+피처 데이터, 리크 포함/제거 OOF 예측 모두 포함)")
print("2. events_40_FINAL.pkl (사건 데이터 전체)")
print("3. final_model_40_leaky.pkl (학습된 최종 모델, 리크 포함 20개 피처)")
print("4. final_model_40_noleak.pkl (학습된 최종 모델, 리크 제거 15개 피처)")
print("5. leak_vs_noleak_comparison.csv (리크 포함 vs 제거 직접 비교표)")

# STAGE 11: STEP 3 — 편입종목(Membership) + 시장일별데이터(Market Daily) 최초 생성

**태그:** 최종

**원본 노트북:** `3__편입종목_시장일별데이터_생성.ipynb`

## FinDA STEP 3 — Membership + Market Daily Feature Store

이번 노트북은 한 번에 다음을 수행합니다.

1. canonical `stock_daily.parquet` 로드
2. `membership_daily.parquet` 생성
3. Markov 제외 시장변수 7개 계산
4. `market_daily.parquet` 생성
5. Config / Metadata / Feature Dictionary 저장
6. QA
7. GCS 업로드 + checksum 검증
8. 선택적으로 pykrx KOSPI200 지수(`1028`) reference 저장

> `membership_daily`가 `market_daily`를 대체하는 것이 아닙니다.
>
> - membership = **그 날짜에 어떤 종목을 시장 계산에 포함하는가**
> - market = **그 구성종목으로 계산한 시장 상태가 무엇인가**

`market_high_vol_regime_prob`은 Markov Switching을 학습해야 생성되는 fold-dependent Feature이므로 여기서는 만들지 않습니다.

### 1. Drive 연결

### 2. 라이브러리

In [ ]:
!pip -q install pyarrow pyyaml pykrx

from pathlib import Path
import hashlib
import json
import warnings

import numpy as np
import pandas as pd
import yaml

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 280)
pd.set_option("display.max_rows", 300)

print("준비 완료")

### 3. 경로

In [ ]:
PROJECT_ID = GCP_PROJECT_ID
BUCKET = GCS_BUCKET_NAME

REFACTOR_ROOT = Path(DRIVE_REFACTOR_ROOT)
CURATED_DIR = REFACTOR_ROOT / "curated"
CONFIG_DIR = REFACTOR_ROOT / "configs"
OUTPUT_DIR = REFACTOR_ROOT / "outputs" / "step3_market"
METADATA_DIR = OUTPUT_DIR / "metadata"
REFERENCE_DIR = REFACTOR_ROOT / "reference"

for p in [
    CURATED_DIR,
    CONFIG_DIR,
    OUTPUT_DIR,
    METADATA_DIR,
    REFERENCE_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

STOCK_PATH = CURATED_DIR / "stock_daily.parquet"
MEMBERSHIP_PATH = CURATED_DIR / "membership_daily.parquet"
MARKET_PATH = CURATED_DIR / "market_daily.parquet"
CONFIG_PATH = CONFIG_DIR / "market_features.yaml"
INDEX_REFERENCE_PATH = REFERENCE_DIR / "kospi200_index_daily.parquet"

START_DATE = pd.Timestamp("2021-06-11")
END_DATE = pd.Timestamp("2026-08-10")

if not STOCK_PATH.exists():
    raise FileNotFoundError(STOCK_PATH)

print("stock:", STOCK_PATH)
print("membership:", MEMBERSHIP_PATH)
print("market:", MARKET_PATH)
print("config:", CONFIG_PATH)

### 4. Config 생성 / 로드

In [ ]:
DEFAULT_CONFIG = {
    "data": {
        "available_start": "2021-06-11",
        "available_end": "2026-08-10",
    },
    "membership": {
        "builder": "stock_daily_canonical_keys",
        "is_member_value": 1,
    },
    "market": {
        "proxy_return": {
            "aggregation": "median",
            "source_return_column": "daily_return",
        },
        "proxy_index": {
            "base_value": 100.0,
        },
        "drawdown": {
            "window": 60,
            "min_periods": 60,
        },
        "volatility": {
            "window": 20,
            "min_periods": 20,
            "annualize": False,
        },
        "trading_value_change": {
            "recent_window": 5,
            "baseline_window": 20,
            "baseline_shift": 5,
        },
        "below_ma": {
            "window": 20,
            "min_periods": 20,
        },
        "crash_ratio": {
            "threshold": -0.03,
        },
        "illiquidity": {
            "recent_window": 20,
            "baseline_window": 252,
            "baseline_shift": 20,
            "robust_scale": 1.4826,
        },
        "markov": {
            "store_in_market_daily": False,
            "fold_dependent": True,
            "states": 2,
            "probability_type": "filtered",
        },
    },
}

if not CONFIG_PATH.exists():
    with open(CONFIG_PATH, "w", encoding="utf-8") as f:
        yaml.safe_dump(
            DEFAULT_CONFIG,
            f,
            allow_unicode=True,
            sort_keys=False,
        )
    print("config 생성:", CONFIG_PATH)

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

print(yaml.safe_dump(CFG, allow_unicode=True, sort_keys=False))

assert CFG["market"]["markov"]["store_in_market_daily"] is False
assert CFG["market"]["markov"]["fold_dependent"] is True

print("✅ config 로드 완료")

### 5. 공통 함수

In [ ]:
def sha256(path, block=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(block)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()


def safe_ratio(num, den):
    num = pd.to_numeric(num, errors="coerce")
    den = pd.to_numeric(den, errors="coerce")

    out = pd.Series(
        np.nan,
        index=num.index,
        dtype="float64",
    )

    ok = (
        num.notna()
        & den.notna()
        & den.ne(0)
    )

    out.loc[ok] = (
        num.loc[ok]
        / den.loc[ok]
    )

    return out


def cross_section_ratio(condition, valid):
    valid_n = int(valid.sum())

    if valid_n == 0:
        return np.nan

    return float(
        (condition & valid).sum()
        / valid_n
    )


def rolling_mad(values):
    x = pd.Series(values).dropna().to_numpy()

    if len(x) == 0:
        return np.nan

    med = np.median(x)

    return np.median(
        np.abs(x - med)
    )

## PART A — stock source

### 6. `stock_daily` 로드 / 필수 컬럼 검증

In [ ]:
stock = pd.read_parquet(STOCK_PATH)

stock["date"] = pd.to_datetime(
    stock["date"],
    errors="coerce",
).dt.normalize()

required = [
    "ticker",
    "ticker_name",
    "date",
    "close",
    "daily_return",
    "trading_value",
]

missing = [
    c for c in required
    if c not in stock.columns
]

if missing:
    raise RuntimeError(
        "시장 Feature 생성 필수 컬럼 누락: "
        + ", ".join(missing)
    )

stock = (
    stock[
        stock["date"].between(
            START_DATE,
            END_DATE,
            inclusive="both",
        )
    ]
    .sort_values(["ticker", "date"])
    .reset_index(drop=True)
)

print("shape:", stock.shape)
print("ticker:", stock["ticker"].nunique())
print("period:", stock["date"].min(), "~", stock["date"].max())
print("key duplicate:", stock.duplicated(["ticker", "date"]).sum())

assert stock["date"].min() == START_DATE
assert stock["date"].max() == END_DATE
assert stock.duplicated(["ticker", "date"]).sum() == 0
assert not stock["ticker"].eq("001260").any()

print("✅ stock source QA 통과")

## PART B — membership_daily

### 7. `membership_daily` 생성

현재 `stock_daily`는 Historical KOSPI200 universe 기준으로 정리된 canonical 패널이므로,
그 `ticker × trading day` key를 별도 membership reference table로 materialize합니다.

향후 공식 Historical constituent history를 별도로 확보하면 **membership builder만 교체**할 수 있습니다.

In [ ]:
membership_daily = (
    stock[
        ["date", "ticker", "ticker_name"]
    ]
    .drop_duplicates(
        ["date", "ticker"]
    )
    .copy()
)

membership_daily["is_member"] = np.int8(
    CFG["membership"]["is_member_value"]
)

membership_daily[
    "membership_source"
] = "stock_daily_canonical_keys"

membership_daily = (
    membership_daily[
        [
            "date",
            "ticker",
            "ticker_name",
            "is_member",
            "membership_source",
        ]
    ]
    .sort_values(["date", "ticker"])
    .reset_index(drop=True)
)

print("shape:", membership_daily.shape)
print("ticker:", membership_daily["ticker"].nunique())
print("period:", membership_daily["date"].min(), "~", membership_daily["date"].max())
print("key duplicate:", membership_daily.duplicated(["date", "ticker"]).sum())

assert len(membership_daily) == len(stock)
assert membership_daily.duplicated(["date", "ticker"]).sum() == 0
assert membership_daily["is_member"].eq(1).all()

print("✅ membership_daily 생성 완료")

### 8. Membership coverage QA

In [ ]:
membership_count = (
    membership_daily
    .groupby("date")
    .size()
    .rename("member_n")
)

membership_count_qa = pd.DataFrame([{
    "trading_days": len(membership_count),
    "min_member_n": int(membership_count.min()),
    "median_member_n": float(membership_count.median()),
    "max_member_n": int(membership_count.max()),
    "days_below_190": int((membership_count < 190).sum()),
    "days_above_205": int((membership_count > 205).sum()),
}])

display(membership_count_qa)

display(
    membership_count
    .sort_values()
    .head(20)
    .to_frame()
)

## PART C — 종목 수준 계산

### 9. 종목별 MA20 / 비유동성

In [ ]:
work = stock[
    [
        "date",
        "ticker",
        "ticker_name",
        "close",
        "daily_return",
        "trading_value",
    ]
].copy()

ma_window = int(
    CFG["market"]["below_ma"]["window"]
)

ma_min_periods = int(
    CFG["market"]["below_ma"]["min_periods"]
)

work["stock_ma20"] = (
    work
    .groupby("ticker")["close"]
    .transform(
        lambda s: s.rolling(
            ma_window,
            min_periods=ma_min_periods,
        ).mean()
    )
)

work["stock_below_ma20"] = (
    work["close"].notna()
    & work["stock_ma20"].notna()
    & work["close"].lt(
        work["stock_ma20"]
    )
)

valid_illiq = (
    work["daily_return"].notna()
    & work["trading_value"].notna()
    & work["trading_value"].gt(0)
)

work["stock_illiquidity"] = np.nan

work.loc[
    valid_illiq,
    "stock_illiquidity",
] = (
    work.loc[
        valid_illiq,
        "daily_return",
    ].abs()
    / work.loc[
        valid_illiq,
        "trading_value",
    ]
)

print("MA20 valid:", int(work["stock_ma20"].notna().sum()))
print("illiquidity valid:", int(work["stock_illiquidity"].notna().sum()))

## PART D — market_daily

### 10. 일별 단면 기본 집계

In [ ]:
crash_threshold = float(
    CFG["market"]["crash_ratio"]["threshold"]
)

daily_records = []

for date, g in work.groupby("date", sort=True):
    ret_valid = g["daily_return"].notna()

    ma_valid = (
        g["close"].notna()
        & g["stock_ma20"].notna()
    )

    illiq_valid = (
        g["stock_illiquidity"].notna()
    )

    trading_value_valid = (
        g["trading_value"].notna()
    )

    daily_records.append({
        "date": date,

        "market_member_n": int(len(g)),

        "market_return_observed_n": int(
            ret_valid.sum()
        ),

        "market_proxy_return": (
            float(
                g.loc[
                    ret_valid,
                    "daily_return",
                ].median()
            )
            if ret_valid.any()
            else np.nan
        ),

        "market_trading_value_total": (
            float(
                g.loc[
                    trading_value_valid,
                    "trading_value",
                ].sum()
            )
            if trading_value_valid.any()
            else np.nan
        ),

        "market_decline_ratio": (
            cross_section_ratio(
                g["daily_return"].lt(0),
                ret_valid,
            )
        ),

        "market_ma20_observed_n": int(
            ma_valid.sum()
        ),

        "market_below_ma20_ratio": (
            cross_section_ratio(
                g["close"].lt(
                    g["stock_ma20"]
                ),
                ma_valid,
            )
        ),

        "market_crash_3pct_ratio": (
            cross_section_ratio(
                g["daily_return"].le(
                    crash_threshold
                ),
                ret_valid,
            )
        ),

        "market_illiq_observed_n": int(
            illiq_valid.sum()
        ),

        "market_illiquidity_median": (
            float(
                g.loc[
                    illiq_valid,
                    "stock_illiquidity",
                ].median()
            )
            if illiq_valid.any()
            else np.nan
        ),
    })

market_daily = (
    pd.DataFrame(daily_records)
    .sort_values("date")
    .reset_index(drop=True)
)

display(market_daily.head())
print("rows:", len(market_daily))
print("period:", market_daily["date"].min(), "~", market_daily["date"].max())

### 11. 시장 대용지수 / 60D Drawdown / 20D Volatility

In [ ]:
base_value = float(
    CFG["market"]["proxy_index"]["base_value"]
)

drawdown_window = int(
    CFG["market"]["drawdown"]["window"]
)

drawdown_min = int(
    CFG["market"]["drawdown"]["min_periods"]
)

vol_window = int(
    CFG["market"]["volatility"]["window"]
)

vol_min = int(
    CFG["market"]["volatility"]["min_periods"]
)

market_daily[
    "market_proxy_index"
] = (
    base_value
    * (
        1.0
        + market_daily[
            "market_proxy_return"
        ].fillna(0)
    ).cumprod()
)

rolling_high = (
    market_daily[
        "market_proxy_index"
    ]
    .rolling(
        drawdown_window,
        min_periods=drawdown_min,
    )
    .max()
)

market_daily[
    "market_drawdown_60d"
] = (
    market_daily[
        "market_proxy_index"
    ]
    / rolling_high
    - 1.0
)

market_daily[
    "market_volatility_20d"
] = (
    market_daily[
        "market_proxy_return"
    ]
    .rolling(
        vol_window,
        min_periods=vol_min,
    )
    .std()
)

print("✅ drawdown / volatility 계산")

### 12. 시장 거래대금 변화 — 최근 5D / 그 이전 20D

In [ ]:
recent_w = int(
    CFG["market"]["trading_value_change"]["recent_window"]
)

baseline_w = int(
    CFG["market"]["trading_value_change"]["baseline_window"]
)

baseline_shift = int(
    CFG["market"]["trading_value_change"]["baseline_shift"]
)

market_daily[
    "market_trading_value_recent5_mean"
] = (
    market_daily[
        "market_trading_value_total"
    ]
    .rolling(
        recent_w,
        min_periods=recent_w,
    )
    .mean()
)

market_daily[
    "market_trading_value_previous20_mean"
] = (
    market_daily[
        "market_trading_value_total"
    ]
    .shift(
        baseline_shift
    )
    .rolling(
        baseline_w,
        min_periods=baseline_w,
    )
    .mean()
)

market_daily[
    "market_trading_value_change_5_20"
] = safe_ratio(
    market_daily[
        "market_trading_value_recent5_mean"
    ],
    market_daily[
        "market_trading_value_previous20_mean"
    ],
)

print("✅ trading value change 계산")

### 13. 시장 비유동성 충격 — recent20 vs 이전 252

In [ ]:
ill_cfg = CFG["market"]["illiquidity"]

recent_w = int(
    ill_cfg["recent_window"]
)

baseline_w = int(
    ill_cfg["baseline_window"]
)

baseline_shift = int(
    ill_cfg["baseline_shift"]
)

robust_scale = float(
    ill_cfg["robust_scale"]
)

illiq = market_daily[
    "market_illiquidity_median"
]

market_daily[
    "market_illiquidity_recent20_median"
] = (
    illiq
    .rolling(
        recent_w,
        min_periods=recent_w,
    )
    .median()
)

baseline_source = (
    illiq
    .shift(
        baseline_shift
    )
)

market_daily[
    "market_illiquidity_baseline252_median"
] = (
    baseline_source
    .rolling(
        baseline_w,
        min_periods=baseline_w,
    )
    .median()
)

market_daily[
    "market_illiquidity_baseline252_mad"
] = (
    baseline_source
    .rolling(
        baseline_w,
        min_periods=baseline_w,
    )
    .apply(
        rolling_mad,
        raw=False,
    )
)

denom = (
    robust_scale
    * market_daily[
        "market_illiquidity_baseline252_mad"
    ]
)

market_daily[
    "market_illiquidity_shock"
] = np.nan

valid = (
    market_daily[
        "market_illiquidity_recent20_median"
    ].notna()
    & market_daily[
        "market_illiquidity_baseline252_median"
    ].notna()
    & denom.notna()
    & denom.gt(0)
)

market_daily.loc[
    valid,
    "market_illiquidity_shock",
] = (
    (
        market_daily.loc[
            valid,
            "market_illiquidity_recent20_median",
        ]
        - market_daily.loc[
            valid,
            "market_illiquidity_baseline252_median",
        ]
    )
    / denom.loc[valid]
)

print("✅ illiquidity shock 계산")

### 14. 최종 컬럼 정리

In [ ]:
MARKET_MODEL_FEATURES = [
    "market_drawdown_60d",
    "market_volatility_20d",
    "market_trading_value_change_5_20",
    "market_decline_ratio",
    "market_below_ma20_ratio",
    "market_crash_3pct_ratio",
    "market_illiquidity_shock",
]

MARKET_SUPPORT_COLUMNS = [
    "market_member_n",
    "market_return_observed_n",
    "market_proxy_return",
    "market_proxy_index",
    "market_trading_value_total",
    "market_trading_value_recent5_mean",
    "market_trading_value_previous20_mean",
    "market_ma20_observed_n",
    "market_illiq_observed_n",
    "market_illiquidity_median",
    "market_illiquidity_recent20_median",
    "market_illiquidity_baseline252_median",
    "market_illiquidity_baseline252_mad",
]

market_daily = market_daily[
    [
        "date",
        *MARKET_MODEL_FEATURES,
        *MARKET_SUPPORT_COLUMNS,
    ]
].copy()

print("최종 Model Features:")
for c in MARKET_MODEL_FEATURES:
    print("-", c)

print("\nSupport / QA:")
for c in MARKET_SUPPORT_COLUMNS:
    print("-", c)

assert (
    "market_high_vol_regime_prob"
    not in market_daily.columns
)

print("\n✅ Markov 변수 제외 확인")

### 15. market_daily 결측 / inf / 범위 QA

In [ ]:
market_missing = pd.DataFrame([
    {
        "column": c,
        "missing_n": int(
            market_daily[c].isna().sum()
        ),
        "missing_pct": float(
            market_daily[c].isna().mean()
        ),
        "first_valid_date": (
            market_daily.loc[
                market_daily[c].notna(),
                "date",
            ].min()
            if market_daily[c].notna().any()
            else pd.NaT
        ),
    }
    for c in market_daily.columns
    if c != "date"
])

display(market_missing)

inf_records = []

for c in market_daily.select_dtypes(
    include=[np.number]
).columns:
    n = int(
        np.isinf(
            pd.to_numeric(
                market_daily[c],
                errors="coerce",
            )
        ).sum()
    )

    if n:
        inf_records.append({
            "column": c,
            "inf_n": n,
        })

inf_df = pd.DataFrame(
    inf_records,
    columns=["column", "inf_n"],
)

display(inf_df)

assert market_daily["date"].duplicated().sum() == 0
assert market_daily["date"].min() == START_DATE
assert market_daily["date"].max() == END_DATE
assert len(inf_df) == 0

for c in [
    "market_decline_ratio",
    "market_below_ma20_ratio",
    "market_crash_3pct_ratio",
]:
    s = market_daily[c].dropna()

    assert s.between(
        0,
        1,
        inclusive="both",
    ).all(), c

assert (
    market_daily[
        "market_drawdown_60d"
    ]
    .dropna()
    .le(1e-12)
    .all()
)

print("✅ market_daily QA 통과")

### 16. 마지막 20거래일 시장변수

In [ ]:
display(
    market_daily[
        [
            "date",
            *MARKET_MODEL_FEATURES,
            "market_proxy_return",
            "market_member_n",
            "market_return_observed_n",
            "market_ma20_observed_n",
        ]
    ].tail(20)
)

## PART E — 선택적 pykrx reference

### 17. KOSPI200 실제 지수 reference 수집

이 데이터는 **7개 시장변수 계산에는 사용하지 않습니다.**

구성종목 기반 `market_proxy_index`와 실제 KOSPI200 지수의 방향을 비교하거나 QA할 때 사용할 수 있도록 별도 reference로 저장합니다.

실패해도 STEP 3는 계속 진행합니다.

In [ ]:
pykrx_reference_status = {
    "success": False,
    "error": None,
}

try:
    from pykrx import stock as krx_stock

    idx = krx_stock.get_index_ohlcv(
        START_DATE.strftime("%Y%m%d"),
        END_DATE.strftime("%Y%m%d"),
        "1028",
    )

    if idx is None or idx.empty:
        raise RuntimeError(
            "pykrx KOSPI200 index result is empty"
        )

    idx = idx.reset_index()

    rename_candidates = {
        "날짜": "date",
        "시가": "open",
        "고가": "high",
        "저가": "low",
        "종가": "close",
        "거래량": "volume",
        "거래대금": "trading_value",
    }

    idx = idx.rename(
        columns={
            k: v
            for k, v
            in rename_candidates.items()
            if k in idx.columns
        }
    )

    if "date" not in idx.columns:
        idx = idx.rename(
            columns={
                idx.columns[0]: "date"
            }
        )

    idx["date"] = pd.to_datetime(
        idx["date"],
        errors="coerce",
    ).dt.normalize()

    idx = (
        idx
        .sort_values("date")
        .drop_duplicates("date")
        .reset_index(drop=True)
    )

    idx.to_parquet(
        INDEX_REFERENCE_PATH,
        index=False,
        engine="pyarrow",
        compression="zstd",
    )

    pykrx_reference_status = {
        "success": True,
        "rows": int(len(idx)),
        "start_date": str(
            idx["date"].min().date()
        ),
        "end_date": str(
            idx["date"].max().date()
        ),
        "path": str(
            INDEX_REFERENCE_PATH
        ),
        "error": None,
    }

    display(idx.head())
    display(idx.tail())

    print(
        "✅ reference 저장:",
        INDEX_REFERENCE_PATH,
    )

except Exception as e:
    pykrx_reference_status = {
        "success": False,
        "error": repr(e),
    }

    print(
        "⚠️ pykrx reference 수집 실패 — "
        "membership/market 생성에는 영향 없음"
    )
    print(repr(e))

print(pykrx_reference_status)

## PART F — 저장 / Metadata

### 18. Parquet 저장

In [ ]:
membership_daily.to_parquet(
    MEMBERSHIP_PATH,
    index=False,
    engine="pyarrow",
    compression="zstd",
)

market_daily.to_parquet(
    MARKET_PATH,
    index=False,
    engine="pyarrow",
    compression="zstd",
)

membership_hash = sha256(
    MEMBERSHIP_PATH
)

market_hash = sha256(
    MARKET_PATH
)

print("membership:", MEMBERSHIP_PATH)
print("sha256:", membership_hash)

print("\nmarket:", MARKET_PATH)
print("sha256:", market_hash)

### 19. Feature Dictionary / Metadata

In [ ]:
feature_dictionary = {
    "market_drawdown_60d": {
        "role": "model_feature",
        "formula": "proxy_index / rolling_60d_max(proxy_index) - 1",
        "direction": "more negative = deeper correction",
        "fold_dependent": False,
    },
    "market_volatility_20d": {
        "role": "model_feature",
        "formula": "rolling_20d_std(market_proxy_return)",
        "direction": "higher = higher volatility",
        "annualized": False,
        "fold_dependent": False,
    },
    "market_trading_value_change_5_20": {
        "role": "model_feature",
        "formula": "mean(T-4:T) / mean(T-24:T-5)",
        "direction": ">1 = recent trading value increased",
        "fold_dependent": False,
    },
    "market_decline_ratio": {
        "role": "model_feature",
        "formula": "count(return < 0) / count(observed return)",
        "direction": "higher = broader decline",
        "fold_dependent": False,
    },
    "market_below_ma20_ratio": {
        "role": "model_feature",
        "formula": "count(close < stock_MA20) / count(observed stock_MA20)",
        "direction": "higher = weaker internal trend",
        "fold_dependent": False,
    },
    "market_crash_3pct_ratio": {
        "role": "model_feature",
        "formula": "count(return <= -0.03) / count(observed return)",
        "direction": "higher = broader severe shock",
        "fold_dependent": False,
    },
    "market_illiquidity_shock": {
        "role": "model_feature",
        "formula": (
            "(recent20_median_illiq - previous252_median_illiq) / "
            "(1.4826 * previous252_MAD)"
        ),
        "direction": "higher positive = worse liquidity",
        "fold_dependent": False,
    },
    "market_high_vol_regime_prob": {
        "role": "future_fold_feature",
        "canonical_market_daily": False,
        "source": "market_proxy_return",
        "model": "2-state Markov Switching",
        "probability_type": "filtered",
        "fold_dependent": True,
    },
}

FEATURE_DICT_PATH = (
    METADATA_DIR
    / "market_feature_dictionary.json"
)

with open(
    FEATURE_DICT_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        feature_dictionary,
        f,
        ensure_ascii=False,
        indent=2,
    )

membership_metadata = {
    "artifact": "membership_daily.parquet",
    "rows": int(len(membership_daily)),
    "ticker_n": int(
        membership_daily["ticker"].nunique()
    ),
    "start_date": str(
        membership_daily["date"].min().date()
    ),
    "end_date": str(
        membership_daily["date"].max().date()
    ),
    "time_grain": "trading_day_x_ticker",
    "builder": CFG["membership"]["builder"],
    "parquet_sha256": membership_hash,
}

market_metadata = {
    "artifact": "market_daily.parquet",
    "rows": int(len(market_daily)),
    "start_date": str(
        market_daily["date"].min().date()
    ),
    "end_date": str(
        market_daily["date"].max().date()
    ),
    "time_grain": "trading_day",
    "model_features": MARKET_MODEL_FEATURES,
    "support_columns": MARKET_SUPPORT_COLUMNS,
    "markov_stored": False,
    "markov_policy": CFG["market"]["markov"],
    "config_path": str(CONFIG_PATH),
    "pykrx_reference": pykrx_reference_status,
    "parquet_sha256": market_hash,
}

for filename, obj in [
    (
        "membership_daily_metadata.json",
        membership_metadata,
    ),
    (
        "market_daily_metadata.json",
        market_metadata,
    ),
]:
    path = METADATA_DIR / filename

    with open(
        path,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            obj,
            f,
            ensure_ascii=False,
            indent=2,
            default=str,
        )

    print(path)

print("feature dictionary:", FEATURE_DICT_PATH)

### 20. Local 최종 QA

In [ ]:
final_qa = pd.DataFrame([
    {
        "dataset": "membership_daily",
        "rows": len(membership_daily),
        "columns": len(membership_daily.columns),
        "ticker_n": membership_daily["ticker"].nunique(),
        "start_date": membership_daily["date"].min(),
        "end_date": membership_daily["date"].max(),
        "duplicate_key_rows": int(
            membership_daily.duplicated(
                ["date", "ticker"],
                keep=False,
            ).sum()
        ),
    },
    {
        "dataset": "market_daily",
        "rows": len(market_daily),
        "columns": len(market_daily.columns),
        "ticker_n": np.nan,
        "start_date": market_daily["date"].min(),
        "end_date": market_daily["date"].max(),
        "duplicate_key_rows": int(
            market_daily["date"]
            .duplicated(
                keep=False
            )
            .sum()
        ),
    },
])

display(final_qa)

assert final_qa["duplicate_key_rows"].eq(0).all()
assert (
    "market_high_vol_regime_prob"
    not in market_daily.columns
)

print("✅ STEP 3 local QA 완료")

## PART G — GCS

### 21. GCP 인증

In [ ]:
# 심사용 실행에서는 GCS 업로드를 생략하고 로컬 저장만 확인합니다.
print("✅ (GCS 인증 생략 — 로컬 전용 실행)")

### 22. GCS 업로드

In [ ]:
# 심사용 실행에서는 팀 GCS에 쓰지 않습니다. membership_daily/market_daily는
# 이미 CURATED_DIR(로컬)에 저장되어 있으므로 그 사실만 확인합니다.
assert MEMBERSHIP_PATH.exists() and MARKET_PATH.exists()
assert (METADATA_DIR / 'membership_daily_metadata.json').exists()
assert (METADATA_DIR / 'market_daily_metadata.json').exists()
assert FEATURE_DICT_PATH.exists()
print("✅ 로컬 저장 확인 완료:", MEMBERSHIP_PATH, MARKET_PATH)

### 23. 재다운로드 + checksum

In [ ]:
# 심사용 실행에서는 재다운로드 없이 로컬 파일 자체를 그대로 사용합니다.
membership_hash_reloaded = sha256(MEMBERSHIP_PATH)
market_hash_reloaded = sha256(MARKET_PATH)

checksum_qa = pd.DataFrame([
    {"dataset": "membership_daily", "sha256_match": membership_hash_reloaded == membership_hash},
    {"dataset": "market_daily", "sha256_match": market_hash_reloaded == market_hash},
])
display(checksum_qa)
assert checksum_qa["sha256_match"].all()
print("✅ membership_daily / market_daily 로컬 canonical 완료")

# STAGE 12: STEP 3 FIX — 과거편입 기준 시장지표 재계산

**태그:** 최종 (STEP 3 패치)

**원본 노트북:** `2__과거편입_시장지표_생성.ipynb`

## FinDA STEP 3 FIX — Historical Membership + Market Daily 생성

- `stock_daily.parquet` 자체는 수정하지 않음
- `attention_daily.parquet`, `short_daily.parquet`도 수정하지 않음
- `membership_daily.parquet`만 역사적 구성종목 기준으로 수정
- 수정된 membership으로 `market_daily.parquet` 재계산
- 기존 GCS 파일은 archive로 백업한 뒤 교체
- Markov 변수는 여전히 `market_daily`에 저장하지 않음

핵심 수정:
`stock_daily`에 존재하는 종목을 곧바로 "그날 KOSPI200 구성종목"으로 간주하지 않습니다.

### 1. Drive 연결

### 2. 라이브러리

In [ ]:
!pip -q install pyarrow pyyaml

from pathlib import Path
from datetime import datetime
import hashlib
import json
import shutil

import numpy as np
import pandas as pd
import yaml

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 300)
pd.set_option("display.max_rows", 300)

print("✅ 준비 완료")

### 3. 경로

In [ ]:
PROJECT_ID = GCP_PROJECT_ID
BUCKET = GCS_BUCKET_NAME

REFACTOR_ROOT = Path(DRIVE_REFACTOR_ROOT)
CURATED_DIR = REFACTOR_ROOT / "curated"
CONFIG_DIR = REFACTOR_ROOT / "configs"
OUTPUT_DIR = REFACTOR_ROOT / "outputs" / "step3_membership_fix"
METADATA_DIR = OUTPUT_DIR / "metadata"

for p in [CURATED_DIR, CONFIG_DIR, OUTPUT_DIR, METADATA_DIR]:
    p.mkdir(parents=True, exist_ok=True)

STOCK_PATH = CURATED_DIR / "stock_daily.parquet"
OLD_MEMBERSHIP_PATH = CURATED_DIR / "membership_daily.parquet"
OLD_MARKET_PATH = CURATED_DIR / "market_daily.parquet"
CONFIG_PATH = CONFIG_DIR / "market_features.yaml"

START_DATE = pd.Timestamp("2021-06-11")
END_DATE = pd.Timestamp("2026-08-10")

for p in [STOCK_PATH, OLD_MEMBERSHIP_PATH, OLD_MARKET_PATH, CONFIG_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

print("stock:", STOCK_PATH)
print("old membership:", OLD_MEMBERSHIP_PATH)
print("old market:", OLD_MARKET_PATH)
print("config:", CONFIG_PATH)

### 4. Config / 함수

In [ ]:
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

def sha256(path, block=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(block)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def safe_ratio(num, den):
    num = pd.to_numeric(num, errors="coerce")
    den = pd.to_numeric(den, errors="coerce")
    out = pd.Series(np.nan, index=num.index, dtype="float64")
    ok = num.notna() & den.notna() & den.ne(0)
    out.loc[ok] = num.loc[ok] / den.loc[ok]
    return out

def cross_section_ratio(condition, valid):
    valid_n = int(valid.sum())
    if valid_n == 0:
        return np.nan
    return float((condition & valid).sum() / valid_n)

def rolling_mad(values):
    x = pd.Series(values).dropna().to_numpy()
    if len(x) == 0:
        return np.nan
    med = np.median(x)
    return np.median(np.abs(x - med))

assert CFG["market"]["markov"]["store_in_market_daily"] is False
assert CFG["market"]["markov"]["fold_dependent"] is True

print("✅ config / 함수 준비 완료")

### 5. 기존 canonical 로드

In [ ]:
stock = pd.read_parquet(STOCK_PATH)
old_membership = pd.read_parquet(OLD_MEMBERSHIP_PATH)
old_market = pd.read_parquet(OLD_MARKET_PATH)

for df in [stock, old_membership, old_market]:
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()

stock = stock.sort_values(["ticker", "date"]).reset_index(drop=True)
old_membership = old_membership.sort_values(["date", "ticker"]).reset_index(drop=True)
old_market = old_market.sort_values("date").reset_index(drop=True)

print("stock:", stock.shape)
print("old membership:", old_membership.shape)
print("old market:", old_market.shape)

display(
    old_membership.groupby("date").size()
    .value_counts().sort_index()
    .rename("trading_days").to_frame()
)

### 6. Historical membership patch 정의

현재 QA에서 확인된 장기 201/202 구간 중, 편입과 동시에 기존 구성종목이 빠져야 했던 구간을 membership에서 교정합니다.

> 이 셀은 `stock_daily` 행을 삭제하지 않습니다. **membership에서만 제외**합니다.

In [ ]:
PATCHES = pd.DataFrame([
    {
        "effective_start": "2021-09-10",
        "effective_end": "2021-12-10",
        "exclude_ticker": "001060",
        "exclude_name": "JW중외제약",
        "reason": "2021-09 special inclusion replacement",
    },
    {
        "effective_start": "2021-09-10",
        "effective_end": "2021-12-10",
        "exclude_ticker": "115390",
        "exclude_name": "락앤락",
        "reason": "2021-09 special inclusion replacement",
    },
    {
        "effective_start": "2022-03-11",
        "effective_end": "2022-06-09",
        "exclude_ticker": "049770",
        "exclude_name": "동원F&B",
        "reason": "2022-03 special inclusion replacement",
    },
    {
        "effective_start": "2023-04-03",
        "effective_end": "2023-04-24",
        "exclude_ticker": "008560",
        "exclude_name": "메리츠증권",
        "reason": "2023-04 special change replacement",
    },
])

PATCHES["effective_start"] = pd.to_datetime(PATCHES["effective_start"])
PATCHES["effective_end"] = pd.to_datetime(PATCHES["effective_end"])

display(PATCHES)

### 7. Membership patch 적용

In [ ]:
membership_daily = old_membership.copy()

patch_audit = []

for _, p in PATCHES.iterrows():
    mask = (
        membership_daily["date"].between(
            p["effective_start"],
            p["effective_end"],
            inclusive="both",
        )
        & membership_daily["ticker"].eq(p["exclude_ticker"])
    )

    patch_audit.append({
        "effective_start": p["effective_start"],
        "effective_end": p["effective_end"],
        "exclude_ticker": p["exclude_ticker"],
        "exclude_name": p["exclude_name"],
        "removed_rows": int(mask.sum()),
        "reason": p["reason"],
    })

    membership_daily = membership_daily.loc[~mask].copy()

membership_daily["membership_source"] = "historical_membership_patched_v1"

membership_daily = (
    membership_daily
    .sort_values(["date", "ticker"])
    .reset_index(drop=True)
)

patch_audit = pd.DataFrame(patch_audit)

display(patch_audit)

assert patch_audit["removed_rows"].gt(0).all()
assert membership_daily.duplicated(["date", "ticker"]).sum() == 0

print("기존 rows:", len(old_membership))
print("수정 rows:", len(membership_daily))
print("제거 rows:", len(old_membership) - len(membership_daily))
print("✅ patch 적용 완료")

### 8. 수정 후 Membership count QA

In [ ]:
member_count = (
    membership_daily
    .groupby("date")
    .size()
    .rename("member_n")
)

display(
    member_count.value_counts().sort_index()
    .rename("trading_days").to_frame()
)

# QA 결과상 실제 201 상태를 허용할 구간
EXPECTED_201_WINDOWS = [
    (
        pd.Timestamp("2024-03-15"),
        pd.Timestamp("2024-06-13"),
        "2024 special-change period",
    ),
    (
        pd.Timestamp("2024-09-30"),
        pd.Timestamp("2024-12-12"),
        "2024 spin-off/special-change period",
    ),
]

unexpected = []

for d, n in member_count.items():
    if n == 200:
        continue

    allowed_201 = (
        n == 201
        and any(start <= d <= end for start, end, _ in EXPECTED_201_WINDOWS)
    )

    if not allowed_201:
        unexpected.append({"date": d, "member_n": int(n)})

unexpected_df = pd.DataFrame(
    unexpected,
    columns=["date", "member_n"],
)

display(unexpected_df)

assert len(unexpected_df) == 0, (
    "예상하지 않은 구성종목 수가 남아 있습니다. "
    "GCS overwrite 전에 membership 이력을 추가 확인하세요."
)

assert int(member_count.min()) == 200
assert int(member_count.max()) == 201

print("✅ membership count QA 통과")
print("200종목 거래일:", int((member_count == 200).sum()))
print("201종목 거래일:", int((member_count == 201).sum()))

### 9. 주요 경계 QA

In [ ]:
BOUNDARY_DATES = [
    "2021-09-09", "2021-09-10", "2021-12-10", "2021-12-13",
    "2022-03-10", "2022-03-11", "2022-06-09", "2022-06-10",
    "2023-03-31", "2023-04-03", "2023-04-24", "2023-04-25",
    "2024-03-14", "2024-03-15", "2024-06-13", "2024-06-14",
    "2024-09-27", "2024-09-30", "2024-12-12", "2024-12-13",
]

boundary = (
    member_count
    .reindex(pd.to_datetime(BOUNDARY_DATES))
    .rename_axis("date")
    .to_frame()
)

display(boundary)

assert member_count.loc["2021-09-10":"2021-12-10"].eq(200).all()
assert member_count.loc["2022-03-11":"2022-06-09"].eq(200).all()
assert member_count.loc["2023-04-03":"2023-04-24"].eq(200).all()

assert member_count.loc["2024-03-15":"2024-06-13"].eq(201).all()
assert member_count.loc["2024-09-30":"2024-12-12"].eq(201).all()

print("✅ 주요 경계 QA 통과")

## PART C — corrected membership으로 `market_daily` 재계산

MA20은 종목 자체의 과거 가격으로 먼저 계산한 뒤,
`stock_daily INNER JOIN membership_daily`로 시장 단면에 포함할 종목만 남깁니다.

### 10. Market 계산용 종목 패널

In [ ]:
required_stock_cols = [
    "date",
    "ticker",
    "ticker_name",
    "close",
    "daily_return",
    "trading_value",
]

missing = [c for c in required_stock_cols if c not in stock.columns]
if missing:
    raise RuntimeError(f"stock 필수 컬럼 누락: {missing}")

stock_work = stock[required_stock_cols].copy()

ma_window = int(CFG["market"]["below_ma"]["window"])
ma_min_periods = int(CFG["market"]["below_ma"]["min_periods"])

stock_work["stock_ma20"] = (
    stock_work
    .groupby("ticker")["close"]
    .transform(
        lambda s: s.rolling(
            ma_window,
            min_periods=ma_min_periods,
        ).mean()
    )
)

member_keys = membership_daily[["date", "ticker"]].copy()

work = (
    member_keys
    .merge(
        stock_work,
        on=["date", "ticker"],
        how="left",
        validate="one_to_one",
    )
    .sort_values(["ticker", "date"])
    .reset_index(drop=True)
)

assert len(work) == len(membership_daily)

work["stock_below_ma20"] = (
    work["close"].notna()
    & work["stock_ma20"].notna()
    & work["close"].lt(work["stock_ma20"])
)

valid_illiq = (
    work["daily_return"].notna()
    & work["trading_value"].notna()
    & work["trading_value"].gt(0)
)

work["stock_illiquidity"] = np.nan
work.loc[valid_illiq, "stock_illiquidity"] = (
    work.loc[valid_illiq, "daily_return"].abs()
    / work.loc[valid_illiq, "trading_value"]
)

print("market source rows:", len(work))
print("✅ corrected market source panel 생성")

### 11. 일별 시장 단면 집계

In [ ]:
crash_threshold = float(
    CFG["market"]["crash_ratio"]["threshold"]
)

daily_records = []

for date, g in work.groupby("date", sort=True):
    ret_valid = g["daily_return"].notna()
    ma_valid = g["close"].notna() & g["stock_ma20"].notna()
    illiq_valid = g["stock_illiquidity"].notna()
    tv_valid = g["trading_value"].notna()

    daily_records.append({
        "date": date,
        "market_member_n": int(len(g)),
        "market_return_observed_n": int(ret_valid.sum()),

        "market_proxy_return": (
            float(g.loc[ret_valid, "daily_return"].median())
            if ret_valid.any()
            else np.nan
        ),

        "market_trading_value_total": (
            float(g.loc[tv_valid, "trading_value"].sum())
            if tv_valid.any()
            else np.nan
        ),

        "market_decline_ratio": cross_section_ratio(
            g["daily_return"].lt(0),
            ret_valid,
        ),

        "market_ma20_observed_n": int(ma_valid.sum()),

        "market_below_ma20_ratio": cross_section_ratio(
            g["close"].lt(g["stock_ma20"]),
            ma_valid,
        ),

        "market_crash_3pct_ratio": cross_section_ratio(
            g["daily_return"].le(crash_threshold),
            ret_valid,
        ),

        "market_illiq_observed_n": int(illiq_valid.sum()),

        "market_illiquidity_median": (
            float(g.loc[illiq_valid, "stock_illiquidity"].median())
            if illiq_valid.any()
            else np.nan
        ),
    })

market_daily = (
    pd.DataFrame(daily_records)
    .sort_values("date")
    .reset_index(drop=True)
)

print("rows:", len(market_daily))
print("period:", market_daily["date"].min(), "~", market_daily["date"].max())
print("✅ 단면 집계 완료")

### 12. Proxy index / Drawdown / Volatility

In [ ]:
base_value = float(CFG["market"]["proxy_index"]["base_value"])

drawdown_window = int(CFG["market"]["drawdown"]["window"])
drawdown_min = int(CFG["market"]["drawdown"]["min_periods"])

vol_window = int(CFG["market"]["volatility"]["window"])
vol_min = int(CFG["market"]["volatility"]["min_periods"])

market_daily["market_proxy_index"] = (
    base_value
    * (1.0 + market_daily["market_proxy_return"].fillna(0)).cumprod()
)

rolling_high = (
    market_daily["market_proxy_index"]
    .rolling(drawdown_window, min_periods=drawdown_min)
    .max()
)

market_daily["market_drawdown_60d"] = (
    market_daily["market_proxy_index"] / rolling_high - 1.0
)

market_daily["market_volatility_20d"] = (
    market_daily["market_proxy_return"]
    .rolling(vol_window, min_periods=vol_min)
    .std()
)

print("✅ drawdown / volatility 계산")

### 13. 시장 거래대금 변화

In [ ]:
tv_cfg = CFG["market"]["trading_value_change"]

recent_w = int(tv_cfg["recent_window"])
baseline_w = int(tv_cfg["baseline_window"])
baseline_shift = int(tv_cfg["baseline_shift"])

market_daily["market_trading_value_recent5_mean"] = (
    market_daily["market_trading_value_total"]
    .rolling(recent_w, min_periods=recent_w)
    .mean()
)

market_daily["market_trading_value_previous20_mean"] = (
    market_daily["market_trading_value_total"]
    .shift(baseline_shift)
    .rolling(baseline_w, min_periods=baseline_w)
    .mean()
)

market_daily["market_trading_value_change_5_20"] = safe_ratio(
    market_daily["market_trading_value_recent5_mean"],
    market_daily["market_trading_value_previous20_mean"],
)

print("✅ trading value change 계산")

### 14. 시장 비유동성 충격

In [ ]:
ill_cfg = CFG["market"]["illiquidity"]

recent_w = int(ill_cfg["recent_window"])
baseline_w = int(ill_cfg["baseline_window"])
baseline_shift = int(ill_cfg["baseline_shift"])
robust_scale = float(ill_cfg["robust_scale"])

illiq = market_daily["market_illiquidity_median"]

market_daily["market_illiquidity_recent20_median"] = (
    illiq
    .rolling(recent_w, min_periods=recent_w)
    .median()
)

baseline_source = illiq.shift(baseline_shift)

market_daily["market_illiquidity_baseline252_median"] = (
    baseline_source
    .rolling(baseline_w, min_periods=baseline_w)
    .median()
)

market_daily["market_illiquidity_baseline252_mad"] = (
    baseline_source
    .rolling(baseline_w, min_periods=baseline_w)
    .apply(rolling_mad, raw=False)
)

denom = robust_scale * market_daily["market_illiquidity_baseline252_mad"]

market_daily["market_illiquidity_shock"] = np.nan

valid = (
    market_daily["market_illiquidity_recent20_median"].notna()
    & market_daily["market_illiquidity_baseline252_median"].notna()
    & denom.notna()
    & denom.gt(0)
)

market_daily.loc[valid, "market_illiquidity_shock"] = (
    (
        market_daily.loc[valid, "market_illiquidity_recent20_median"]
        - market_daily.loc[valid, "market_illiquidity_baseline252_median"]
    )
    / denom.loc[valid]
)

print("✅ illiquidity shock 계산")

### 15. 최종 컬럼

In [ ]:
MARKET_MODEL_FEATURES = [
    "market_drawdown_60d",
    "market_volatility_20d",
    "market_trading_value_change_5_20",
    "market_decline_ratio",
    "market_below_ma20_ratio",
    "market_crash_3pct_ratio",
    "market_illiquidity_shock",
]

MARKET_SUPPORT_COLUMNS = [
    "market_member_n",
    "market_return_observed_n",
    "market_proxy_return",
    "market_proxy_index",
    "market_trading_value_total",
    "market_trading_value_recent5_mean",
    "market_trading_value_previous20_mean",
    "market_ma20_observed_n",
    "market_illiq_observed_n",
    "market_illiquidity_median",
    "market_illiquidity_recent20_median",
    "market_illiquidity_baseline252_median",
    "market_illiquidity_baseline252_mad",
]

market_daily = market_daily[
    ["date", *MARKET_MODEL_FEATURES, *MARKET_SUPPORT_COLUMNS]
].copy()

assert "market_high_vol_regime_prob" not in market_daily.columns

print("market_daily:", market_daily.shape)
print("✅ Markov 변수 없음")

### 16. 기존 `market_daily` 대비 변경량

In [ ]:
compare = (
    old_market[["date", *MARKET_MODEL_FEATURES]]
    .merge(
        market_daily[["date", *MARKET_MODEL_FEATURES]],
        on="date",
        how="inner",
        suffixes=("_old", "_new"),
        validate="one_to_one",
    )
)

change_rows = []

for c in MARKET_MODEL_FEATURES:
    old_c = compare[f"{c}_old"]
    new_c = compare[f"{c}_new"]

    diff = (new_c - old_c).abs()

    changed = (
        diff.gt(1e-15)
        | (old_c.isna() ^ new_c.isna())
    )

    change_rows.append({
        "feature": c,
        "changed_days": int(changed.sum()),
        "mean_abs_diff": float(diff.mean(skipna=True)),
        "max_abs_diff": float(diff.max(skipna=True)),
    })

market_change_summary = pd.DataFrame(change_rows)

display(market_change_summary)

changed_any = pd.Series(False, index=compare.index)

for c in MARKET_MODEL_FEATURES:
    old_c = compare[f"{c}_old"]
    new_c = compare[f"{c}_new"]

    changed_any |= (
        (old_c - new_c).abs().gt(1e-15)
        | (old_c.isna() ^ new_c.isna())
    )

changed_dates = compare.loc[changed_any, ["date"]]

print("하나 이상의 시장 Feature가 바뀐 거래일:", len(changed_dates))

if len(changed_dates):
    print("최초:", changed_dates["date"].min())
    print("최종:", changed_dates["date"].max())

display(changed_dates.head(30))

### 17. 최종 QA

In [ ]:
qa = pd.DataFrame([
    {
        "dataset": "membership_daily",
        "rows": len(membership_daily),
        "columns": len(membership_daily.columns),
        "ticker_n": membership_daily["ticker"].nunique(),
        "start_date": membership_daily["date"].min(),
        "end_date": membership_daily["date"].max(),
        "duplicate_key_rows": int(
            membership_daily.duplicated(
                ["date", "ticker"],
                keep=False,
            ).sum()
        ),
    },
    {
        "dataset": "market_daily",
        "rows": len(market_daily),
        "columns": len(market_daily.columns),
        "ticker_n": np.nan,
        "start_date": market_daily["date"].min(),
        "end_date": market_daily["date"].max(),
        "duplicate_key_rows": int(
            market_daily["date"].duplicated(keep=False).sum()
        ),
    },
])

display(qa)

display(
    membership_daily.groupby("date").size()
    .value_counts().sort_index()
    .rename("trading_days").to_frame()
)

inf_records = []

for c in market_daily.select_dtypes(include=[np.number]).columns:
    n = int(
        np.isinf(
            pd.to_numeric(market_daily[c], errors="coerce")
        ).sum()
    )
    if n:
        inf_records.append({"column": c, "inf_n": n})

inf_df = pd.DataFrame(inf_records, columns=["column", "inf_n"])
display(inf_df)

assert qa["duplicate_key_rows"].eq(0).all()
assert membership_daily["date"].min() == START_DATE
assert membership_daily["date"].max() == END_DATE
assert market_daily["date"].min() == START_DATE
assert market_daily["date"].max() == END_DATE
assert len(inf_df) == 0
assert member_count.min() == 200
assert member_count.max() == 201

for c in [
    "market_decline_ratio",
    "market_below_ma20_ratio",
    "market_crash_3pct_ratio",
]:
    assert market_daily[c].dropna().between(0, 1, inclusive="both").all()

assert market_daily["market_drawdown_60d"].dropna().le(1e-12).all()

print("✅ FINAL QA 통과")

### 18. 임시 파일 저장

In [ ]:
TEMP_DIR = OUTPUT_DIR / "rebuilt"
TEMP_DIR.mkdir(parents=True, exist_ok=True)

TEMP_MEMBERSHIP = TEMP_DIR / "membership_daily.parquet"
TEMP_MARKET = TEMP_DIR / "market_daily.parquet"

membership_daily.to_parquet(
    TEMP_MEMBERSHIP,
    index=False,
    engine="pyarrow",
    compression="zstd",
)

market_daily.to_parquet(
    TEMP_MARKET,
    index=False,
    engine="pyarrow",
    compression="zstd",
)

membership_hash = sha256(TEMP_MEMBERSHIP)
market_hash = sha256(TEMP_MARKET)

print("membership:", TEMP_MEMBERSHIP)
print("sha256:", membership_hash)

print("\nmarket:", TEMP_MARKET)
print("sha256:", market_hash)

### 19. Audit / Metadata

In [ ]:
PATCH_AUDIT_PATH = METADATA_DIR / "membership_patch_audit.csv"
CHANGE_SUMMARY_PATH = METADATA_DIR / "market_change_summary_after_membership_fix.csv"
META_PATH = METADATA_DIR / "step3_membership_market_fix_metadata.json"

patch_audit.to_csv(
    PATCH_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig",
)

market_change_summary.to_csv(
    CHANGE_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

metadata = {
    "artifact_version": "historical_membership_patched_v1",
    "generated_at": datetime.now().isoformat(),
    "period": {
        "start": str(START_DATE.date()),
        "end": str(END_DATE.date()),
    },
    "membership_policy": {
        "base": "previous membership_daily",
        "patch_audit_path": str(PATCH_AUDIT_PATH),
        "allowed_member_counts": [200, 201],
    },
    "market_policy": {
        "source": "stock_daily INNER JOIN corrected membership_daily",
        "model_features": MARKET_MODEL_FEATURES,
        "markov_stored": False,
        "markov_fold_dependent": True,
    },
    "sha256": {
        "membership_daily": membership_hash,
        "market_daily": market_hash,
    },
}

with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(PATCH_AUDIT_PATH)
print(CHANGE_SUMMARY_PATH)
print(META_PATH)

## PART E — GCS 교체

기존 GCS `membership_daily` / `market_daily`를 archive로 먼저 백업합니다.

### 20. GCP 인증

In [ ]:
# 심사용 실행에서는 GCS 업로드를 생략하고 로컬 저장만 확인합니다.
print("✅ (GCS 인증 생략 — 로컬 전용 실행)")

### 21. 기존 GCS 파일 archive 백업

In [ ]:
# 심사용 실행에서는 팀 GCS의 archive에 백업하지 않고, 로컬에 타임스탬프 백업만 남깁니다.
RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
ARCHIVE_DIR = WORKSPACE_ROOT / "archive" / f"before_membership_fix_{RUN_TS}"
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

if MEMBERSHIP_PATH.exists():
    shutil.copy2(MEMBERSHIP_PATH, ARCHIVE_DIR / "membership_daily.parquet")
if MARKET_PATH.exists():
    shutil.copy2(MARKET_PATH, ARCHIVE_DIR / "market_daily.parquet")

print("✅ 기존 파일 로컬 백업 완료")
print(ARCHIVE_DIR)

### 22. Drive canonical + GCS 교체

In [ ]:
NEW_MEMBERSHIP_PATH = CURATED_DIR / "membership_daily.parquet"
NEW_MARKET_PATH = CURATED_DIR / "market_daily.parquet"

shutil.copy2(TEMP_MEMBERSHIP, NEW_MEMBERSHIP_PATH)
shutil.copy2(TEMP_MARKET, NEW_MARKET_PATH)

assert sha256(NEW_MEMBERSHIP_PATH) == membership_hash
assert sha256(NEW_MARKET_PATH) == market_hash

# 심사용 실행에서는 팀 GCS에 쓰지 않습니다. 로컬 CURATED_DIR이 이미 최신본으로
# 교체되었으므로(위 shutil.copy2), 이후 STAGE들은 이 로컬 파일을 그대로 씁니다.
print("✅ 로컬 canonical 교체 완료:", NEW_MEMBERSHIP_PATH, NEW_MARKET_PATH)

### 23. GCS 재다운로드 checksum

In [ ]:
# 심사용 실행에서는 재다운로드 없이 로컬 파일을 그대로 검증합니다.
verify_qa = pd.DataFrame([
    {"dataset": "membership_daily", "sha256_match": sha256(NEW_MEMBERSHIP_PATH) == membership_hash},
    {"dataset": "market_daily", "sha256_match": sha256(NEW_MARKET_PATH) == market_hash},
])

display(verify_qa)
assert verify_qa["sha256_match"].all()

verify_membership = pd.read_parquet(NEW_MEMBERSHIP_PATH)
verify_market = pd.read_parquet(NEW_MARKET_PATH)

verify_membership["date"] = pd.to_datetime(verify_membership["date"])
verify_market["date"] = pd.to_datetime(verify_market["date"])

print("\n로컬 membership rows:", len(verify_membership))
print(
    "count distribution:",
    verify_membership.groupby("date").size()
    .value_counts().sort_index().to_dict()
)

print("\n로컬 market rows:", len(verify_market))
print("period:", verify_market["date"].min(), "~", verify_market["date"].max())

print("\n✅ STEP 3 FIX 완료")

# STAGE 13: STEP 4 — GCS + DuckDB Feature Group 기반 구축

**태그:** 최종

**원본 노트북:** `4__GCS___DuckDB_Feature_Group_기반.ipynb`

## FinDA STEP 4 — GCS + DuckDB Feature Group Foundation

목적:

```text
GCS canonical Parquet
        ↓
DuckDB
        ↓
canonical.*  원천 view
feature.*    변수군별 view
analysis.*   P / P+F / P+I / P+FI / P+ALLFLOW 비교 view
meta.*       feature registry / 실험 recipe / leakage 정책
```

이번 단계에서는 **Event/Target을 아직 만들지 않습니다.**  
먼저 `stock_daily` 안에 섞여 있는 가격·거래량 / 외국인 / 기관 / 개인 / 기업가치 변수를 분리합니다.

특히 `analysis.daily_P`는 **가격·거래량만 포함**하며 외국인·기관·개인·PER/PBR/EPS·관심·공매도·시장변수를 포함하지 않습니다.

DuckDB는 우선 GCS Parquet을 `gcsfs`로 직접 읽고, Colab 인증 문제 발생 시 Drive mirror로 자동 fallback합니다.

In [ ]:
# 1. packages
!pip -q install duckdb gcsfs fsspec pyarrow pyyaml

from pathlib import Path
from datetime import datetime
import json, subprocess
import duckdb, gcsfs, pandas as pd, numpy as np, yaml

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 320)

print("DuckDB:", duckdb.__version__)

In [ ]:
# 2. 프로젝트/버킷 이름만 참조 (인증 불필요 — 로컬 curated 파일을 직접 씁니다)
PROJECT_ID = GCP_PROJECT_ID
BUCKET = GCS_BUCKET_NAME
print("✅ 준비 완료 (로컬 전용, GCS 인증 불필요)")

In [ ]:
# 3. Paths / sources
ROOT = Path(DRIVE_REFACTOR_ROOT)
DUCKDB_DIR = ROOT / "duckdb"
CONFIG_DIR = ROOT / "configs"
META_DIR = ROOT / "metadata"
CURATED_DIR = ROOT / "curated"  # STAGE 3·11·12가 쓰는 로컬 curated 폴더와 동일 경로

for p in [DUCKDB_DIR, CONFIG_DIR, META_DIR, CURATED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DB_PATH = DUCKDB_DIR / "finda.duckdb"
FEATURE_CONFIG_PATH = CONFIG_DIR / "feature_groups.yaml"
MANIFEST_PATH = META_DIR / "duckdb_manifest.json"

FILES = {
    "stock_daily": "stock_daily.parquet",
    "attention_daily": "attention_daily.parquet",
    "short_daily": "short_daily.parquet",
    "membership_daily": "membership_daily.parquet",
    "market_daily": "market_daily.parquet",
}

print("DB:", DB_PATH)

### PART A — GCS 연결

공식 DuckDB Python 방식대로 `gcsfs` filesystem을 등록합니다.  
직접 연결이 안 되면 GCS Source of Truth를 Drive mirror로 내려받아 같은 View 구조를 만듭니다.

In [ ]:
# 4. Source 설정 — 로컬 curated 폴더(STAGE 3·11·12가 이미 채워둔 파일)를 직접 사용합니다.
# (원래는 GCS에 gcsfs로 직접 연결하고, 실패하면 gcloud로 Drive에 미러링하는 이중 경로였으나,
# 이 노트북은 심사용 실행에서 팀 GCS를 아예 거치지 않도록 처음부터 로컬 파일만 사용합니다.)
con = duckdb.connect(str(DB_PATH))
gcs_error = None
SOURCE_MODE = "local"
SOURCE_URIS = {name: str(CURATED_DIR / filename) for name, filename in FILES.items()}

for name, uri in SOURCE_URIS.items():
    assert Path(uri).exists(), f"필수 파일 없음: {uri}"

print("SOURCE_MODE =", SOURCE_MODE)
for k, v in SOURCE_URIS.items():
    print("-", k, "=>", v)

### PART B — Canonical Views

In [ ]:
# 5. Schemas + canonical views
for schema in ["canonical", "feature", "analysis", "meta"]:
    con.execute(f"CREATE SCHEMA IF NOT EXISTS {schema}")

for name, uri in SOURCE_URIS.items():
    con.execute(f'''
        CREATE OR REPLACE VIEW canonical.{name} AS
        SELECT * FROM read_parquet('{uri}')
    ''')

display(con.execute('''
    SELECT table_schema, table_name, table_type
    FROM information_schema.tables
    WHERE table_schema='canonical'
    ORDER BY table_name
''').df())

print("✅ canonical views 생성")

In [ ]:
# 6. Canonical QA
queries = {
"stock_daily": "SELECT COUNT(*) AS \"rows\", COUNT(DISTINCT ticker) ticker_n, MIN(date) start_date, MAX(date) end_date, COUNT(*)-COUNT(DISTINCT (ticker,date)) duplicate_keys FROM canonical.stock_daily",
"attention_daily": "SELECT COUNT(*) AS \"rows\", COUNT(DISTINCT ticker) ticker_n, MIN(date) start_date, MAX(date) end_date, COUNT(*)-COUNT(DISTINCT (ticker,date)) duplicate_keys FROM canonical.attention_daily",
"short_daily": "SELECT COUNT(*) AS \"rows\", COUNT(DISTINCT ticker) ticker_n, MIN(date) start_date, MAX(date) end_date, COUNT(*)-COUNT(DISTINCT (ticker,date)) duplicate_keys FROM canonical.short_daily",
"membership_daily": "SELECT COUNT(*) AS \"rows\", COUNT(DISTINCT ticker) ticker_n, MIN(date) start_date, MAX(date) end_date, COUNT(*)-COUNT(DISTINCT (ticker,date)) duplicate_keys FROM canonical.membership_daily",
"market_daily": "SELECT COUNT(*) AS \"rows\", NULL::BIGINT ticker_n, MIN(date) start_date, MAX(date) end_date, COUNT(*)-COUNT(DISTINCT date) duplicate_keys FROM canonical.market_daily",
}

rows=[]
for name,q in queries.items():
    r=con.execute(q).df().iloc[0].to_dict()
    r["dataset"]=name
    rows.append(r)

canonical_qa=pd.DataFrame(rows)[["dataset","rows","ticker_n","start_date","end_date","duplicate_keys"]]
display(canonical_qa)

expected={
    "stock_daily":253111,
    "attention_daily":380088,
    "short_daily":253111,
    "membership_daily":252911,
    "market_daily":1264,
}
assert canonical_qa["duplicate_keys"].eq(0).all()
for k,v in expected.items():
    actual=int(canonical_qa.loc[canonical_qa.dataset.eq(k),"rows"].iloc[0])
    assert actual==v,(k,actual,v)

print("✅ canonical QA 통과")

### PART C — Feature Group Config

Feature를 물리적으로 여러 Parquet으로 복제하지 않고 DuckDB View로 분리합니다.

- `price_volume`: 순수 기준군
- `foreign_flow`
- `institution_flow`
- `individual_flow`
- `value`
- `attention`
- `market`
- `short`는 canonical passthrough

T+5 사건변수는 이후 Event Builder에서 생성합니다.

In [ ]:
# 7. Feature group config
CFG = {
"daily_groups":{
"price_volume":[
    "ticker","ticker_name","market","date","open","high","low","close","volume",
    "market_cap","trading_value","listed_shares","yahoo_close","adj_close",
    "yahoo_shares_outstanding","daily_return","return_5d","adj_daily_return",
    "adj_return_5d","volume_to_20d_avg","trading_value_growth","close_diff_pct"
],
"foreign_flow":["ticker","date","foreign_net_buy_value","foreign_ownership_pct","foreign_holding_shares"],
"institution_flow":["ticker","date","institution_net_buy_value"],
"individual_flow":["ticker","date","individual_net_buy_value"],
"value":["ticker","date","per","pbr","eps","market_cap"],
"attention":[
    "ticker","date","search_pct_50","search_ratio_7_30","search_slope_7",
    "search_high_share_7","news_pct_50","news_ratio_7_30","news_slope_7",
    "news_high_share_7","search_news_level_gap"
],
"market":[
    "date","market_drawdown_60d","market_volatility_20d",
    "market_trading_value_change_5_20","market_decline_ratio",
    "market_below_ma20_ratio","market_crash_3pct_ratio",
    "market_illiquidity_shock","market_proxy_return"
],
},
"planned_t5_groups":{
"t5_price_volume":["early_giveback_t5","volume_sustain_t5"],
"t5_foreign":["foreign_net_buy_t5","foreign_net_buy_t5_ratio"],
"t5_institution":["institution_net_buy_t5","institution_net_buy_t5_ratio"],
"t5_individual":["individual_net_buy_t5","individual_net_buy_t5_ratio"],
},
"experiment_recipes":{
"P":["price_volume"],
"P_F":["price_volume","foreign_flow"],
"P_I":["price_volume","institution_flow"],
"P_FI":["price_volume","foreign_flow","institution_flow"],
"P_ALLFLOW":["price_volume","foreign_flow","institution_flow","individual_flow"],
"P_ATT":["price_volume","attention"],
"P_FI_ATT":["price_volume","foreign_flow","institution_flow","attention"],
},
"policies":{
"attention_lead":{"stage":"after_event_builder","canonical_daily":False},
"markov_high_vol_probability":{"stage":"inside_each_walk_forward_fold","canonical_daily":False,"probability":"filtered"},
"t5_cutoff":{"prediction_time":"T+5","features":"<=T+5","target":">T+5 only"},
}
}

with open(FEATURE_CONFIG_PATH,"w",encoding="utf-8") as f:
    yaml.safe_dump(CFG,f,allow_unicode=True,sort_keys=False)

print(FEATURE_CONFIG_PATH)

### PART D — 변수군 View

In [ ]:
# 8. Pure price-volume base
base_cols=", ".join(CFG["daily_groups"]["price_volume"])
con.execute(f'''
CREATE OR REPLACE VIEW feature.stock_base_daily AS
SELECT {base_cols}
FROM canonical.stock_daily
''')

base_schema=con.execute("DESCRIBE feature.stock_base_daily").df()
display(base_schema)

forbidden_tokens=[
    "foreign","institution","individual","net_buy","ownership","holding",
    "per","pbr","eps","short","search","news"
]
leaked=[
    c for c in base_schema["column_name"].astype(str)
    if any(t in c.lower() for t in forbidden_tokens)
]
assert not leaked, leaked

print("✅ 순수 가격·거래량 base 생성 / leakage guard 통과")

In [ ]:
# 9. Flow / value views
con.execute('''
CREATE OR REPLACE VIEW feature.flow_foreign_daily AS
SELECT ticker,date,foreign_net_buy_value,foreign_ownership_pct,foreign_holding_shares
FROM canonical.stock_daily
''')

con.execute('''
CREATE OR REPLACE VIEW feature.flow_institution_daily AS
SELECT ticker,date,institution_net_buy_value
FROM canonical.stock_daily
''')

con.execute('''
CREATE OR REPLACE VIEW feature.flow_individual_daily AS
SELECT ticker,date,individual_net_buy_value
FROM canonical.stock_daily
''')

con.execute('''
CREATE OR REPLACE VIEW feature.flow_all_daily AS
SELECT ticker,date,
       individual_net_buy_value,foreign_net_buy_value,institution_net_buy_value,
       other_corporation_net_buy_value,total_net_buy_value,
       foreign_ownership_pct,foreign_holding_shares
FROM canonical.stock_daily
''')

con.execute('''
CREATE OR REPLACE VIEW feature.value_daily AS
SELECT ticker,date,per,pbr,eps,market_cap
FROM canonical.stock_daily
''')

con.execute("CREATE OR REPLACE VIEW feature.attention_daily AS SELECT * FROM canonical.attention_daily")
con.execute("CREATE OR REPLACE VIEW feature.short_daily AS SELECT * FROM canonical.short_daily")
con.execute("CREATE OR REPLACE VIEW feature.market_daily AS SELECT * FROM canonical.market_daily")
con.execute("CREATE OR REPLACE VIEW feature.membership_daily AS SELECT * FROM canonical.membership_daily WHERE is_member=1")

print("✅ feature views 생성")

### PART E — 비교실험용 Daily Views

모든 View는 **동일한 membership sample**을 사용하며 변수만 추가됩니다.

In [ ]:
# 10. P / P+F / P+I / P+FI / P+ALLFLOW
con.execute('''
CREATE OR REPLACE VIEW analysis.daily_P AS
SELECT b.*
FROM feature.stock_base_daily b
INNER JOIN feature.membership_daily m USING(ticker,date)
''')

con.execute('''
CREATE OR REPLACE VIEW analysis.daily_P_F AS
SELECT b.*, f.foreign_net_buy_value, f.foreign_ownership_pct, f.foreign_holding_shares
FROM analysis.daily_P b
LEFT JOIN feature.flow_foreign_daily f USING(ticker,date)
''')

con.execute('''
CREATE OR REPLACE VIEW analysis.daily_P_I AS
SELECT b.*, i.institution_net_buy_value
FROM analysis.daily_P b
LEFT JOIN feature.flow_institution_daily i USING(ticker,date)
''')

con.execute('''
CREATE OR REPLACE VIEW analysis.daily_P_FI AS
SELECT b.*,
       f.foreign_net_buy_value,f.foreign_ownership_pct,f.foreign_holding_shares,
       i.institution_net_buy_value
FROM analysis.daily_P b
LEFT JOIN feature.flow_foreign_daily f USING(ticker,date)
LEFT JOIN feature.flow_institution_daily i USING(ticker,date)
''')

con.execute('''
CREATE OR REPLACE VIEW analysis.daily_P_ALLFLOW AS
SELECT b.*,
       a.individual_net_buy_value,a.foreign_net_buy_value,a.institution_net_buy_value,
       a.other_corporation_net_buy_value,a.total_net_buy_value,
       a.foreign_ownership_pct,a.foreign_holding_shares
FROM analysis.daily_P b
LEFT JOIN feature.flow_all_daily a USING(ticker,date)
''')

print("✅ comparison daily views 생성")

In [ ]:
# 11. 동일 sample key QA
VIEWS=["daily_P","daily_P_F","daily_P_I","daily_P_FI","daily_P_ALLFLOW"]
out=[]
for v in VIEWS:
    r=con.execute(f'''
        SELECT COUNT(*) "rows",
               COUNT(DISTINCT (ticker,date)) unique_keys,
               COUNT(*)-COUNT(DISTINCT (ticker,date)) duplicate_keys
        FROM analysis.{v}
    ''').df().iloc[0].to_dict()
    r["view"]=v
    out.append(r)

view_qa=pd.DataFrame(out)[["view","rows","unique_keys","duplicate_keys"]]
display(view_qa)

assert view_qa["duplicate_keys"].eq(0).all()
assert view_qa["rows"].nunique()==1
assert view_qa["unique_keys"].nunique()==1
assert int(view_qa["rows"].iloc[0])==252911

print("✅ 모든 비교군 동일 sample")

In [ ]:
# 12. 비교군별 추가 컬럼 확인
base_cols=set(con.execute("DESCRIBE analysis.daily_P").df()["column_name"])
summary=[]

for v in VIEWS:
    cols=con.execute(f"DESCRIBE analysis.{v}").df()["column_name"].tolist()
    added=[c for c in cols if c not in base_cols]
    summary.append({
        "view":v,
        "column_n":len(cols),
        "added_vs_P":", ".join(added) if added else "(none)"
    })

comparison_summary=pd.DataFrame(summary)
display(comparison_summary)

print("✅ 변수군 분리 확인")

### PART F — Meta Registry

In [ ]:
# 13. Feature registry / experiment recipe / policy tables
registry=[]

for group,cols in CFG["daily_groups"].items():
    for c in cols:
        registry.append({
            "feature_group":group,
            "column_name":c,
            "stage":"daily",
            "available_at":"T/current",
        })

for group,cols in CFG["planned_t5_groups"].items():
    for c in cols:
        registry.append({
            "feature_group":group,
            "column_name":c,
            "stage":"post_event",
            "available_at":"T+5",
        })

reg_df=pd.DataFrame(registry)
con.register("_reg_df",reg_df)
con.execute("CREATE OR REPLACE TABLE meta.feature_registry AS SELECT * FROM _reg_df")
con.unregister("_reg_df")

recipes=[]
for recipe,groups in CFG["experiment_recipes"].items():
    for n,g in enumerate(groups,1):
        recipes.append({"recipe":recipe,"group_order":n,"feature_group":g})

recipe_df=pd.DataFrame(recipes)
con.register("_recipe_df",recipe_df)
con.execute("CREATE OR REPLACE TABLE meta.experiment_recipe AS SELECT * FROM _recipe_df")
con.unregister("_recipe_df")

policy_df=pd.DataFrame([
{"policy":"base_no_flow","rule":"daily_P에는 외국인/기관/개인/보유정보 없음"},
{"policy":"attention_lead","rule":"Event 생성 이후 계산"},
{"policy":"markov","rule":"각 Walk-forward Train fold에서 fit; filtered probability"},
{"policy":"T5_cutoff","rule":"Feature <= T+5, Target > T+5 only"},
])
con.register("_policy_df",policy_df)
con.execute("CREATE OR REPLACE TABLE meta.pipeline_policy AS SELECT * FROM _policy_df")
con.unregister("_policy_df")

display(con.execute("SELECT * FROM meta.experiment_recipe ORDER BY recipe,group_order").df())
print("✅ meta registry 생성")

### PART G — 최종 QA / Manifest

In [ ]:
# 14. Membership + object inventory
membership_qa=con.execute('''
WITH x AS (
    SELECT date,COUNT(*) member_n
    FROM canonical.membership_daily
    WHERE is_member=1
    GROUP BY date
)
SELECT member_n,COUNT(*) trading_days
FROM x
GROUP BY member_n
ORDER BY member_n
''').df()

display(membership_qa)

actual=dict(zip(membership_qa.member_n.astype(int),membership_qa.trading_days.astype(int)))
assert actual=={200:1153,201:111},actual

inventory=con.execute('''
SELECT table_schema,table_name,table_type
FROM information_schema.tables
WHERE table_schema IN ('canonical','feature','analysis','meta')
ORDER BY table_schema,table_name
''').df()

display(inventory)
print("✅ corrected membership 확인")

In [ ]:
# 15. Manifest + checkpoint
manifest={
    "built_at":datetime.now().isoformat(),
    "duckdb_version":duckdb.__version__,
    "db_path":str(DB_PATH),
    "project_id":PROJECT_ID,
    "bucket":BUCKET,
    "source_mode":SOURCE_MODE,
    "gcs_direct_error":gcs_error,
    "feature_config":str(FEATURE_CONFIG_PATH),
    "comparison_views":VIEWS,
    "policies":{
        "P_has_no_flow":True,
        "attention_lead_event_dependent":True,
        "markov_fold_dependent":True,
        "T5_target_after_cutoff_only":True,
    },
}
with open(MANIFEST_PATH,"w",encoding="utf-8") as f:
    json.dump(manifest,f,ensure_ascii=False,indent=2,default=str)

con.execute("CHECKPOINT")
print("DB:",DB_PATH)
print("Manifest:",MANIFEST_PATH)
print("DB size MB:",round(DB_PATH.stat().st_size/1024/1024,2))
print("✅ checkpoint 완료")

In [ ]:
# 16. Reconnect smoke test
con.close()
con=duckdb.connect(str(DB_PATH))

if SOURCE_MODE=="direct_gcs":
    try:
        duckdb.register_filesystem(gcs_fs)
    except Exception:
        gcs_fs=gcsfs.GCSFileSystem(project=PROJECT_ID,token="google_default")
        duckdb.register_filesystem(gcs_fs)

smoke=con.execute('''
SELECT
 (SELECT COUNT(*) FROM canonical.stock_daily) stock_rows,
 (SELECT COUNT(*) FROM canonical.membership_daily) membership_rows,
 (SELECT COUNT(*) FROM canonical.market_daily) market_rows,
 (SELECT COUNT(*) FROM analysis.daily_P) p_rows,
 (SELECT COUNT(*) FROM analysis.daily_P_F) pf_rows,
 (SELECT COUNT(*) FROM analysis.daily_P_I) pi_rows,
 (SELECT COUNT(*) FROM analysis.daily_P_FI) pfi_rows
''').df()

display(smoke)

assert int(smoke.stock_rows.iloc[0])==253111
assert int(smoke.membership_rows.iloc[0])==252911
assert int(smoke.market_rows.iloc[0])==1264
assert smoke[["p_rows","pf_rows","pi_rows","pfi_rows"]].nunique(axis=1).iloc[0]==1

print("✅ DuckDB 재접속 Smoke Test 통과")

# STAGE 14: STEP 5 — Event Set + T/T+3/T+5/T+10 Feature Builder + 외국인수급 분석

**태그:** 최종

**원본 노트북:** `5__이벤트셋_사건일_기반_Feature생성_및_외국인수급_분석.ipynb`

> STEP 4의 finda.duckdb를 그대로 사용합니다.

## Event Set + T/T+3/T+5/T+10 Feature Builder + T+5 외국인 수급 설명분석 / Development Threshold 탐색

이번 노트북은 STEP 4의 `finda.duckdb`를 그대로 사용합니다.

#### 핵심 산출물
- `event_master`
- `event_snapshots_long`
- `event_targets_long`
- `event_model_frame_long`
- `event_bhar_path_long`

#### T+5 외국인 수급 추가 분석
1. `foreign_flow_ratio_T5` 5분위
2. 분위별 Peak Give-back 과열 발생률
3. 분위별 T+5 이후 median BHAR path
4. `early_giveback_T5 × foreign_flow_ratio_T5` 조합별 과열률
5. 통계검정
6. `5분위 → 10분위 → breakpoint 후보`
7. threshold는 **2021~2024 Development에서만 선택**
8. 2025~2026 OOT에는 고정 threshold 그대로 적용

#### Event 정의
- 20거래일 수익률 ≥ 20%
- 당일 거래량 / **직전 20거래일 평균 거래량** ≥ 1.5
- 동일 종목 20거래일 cooldown
- 해당일 KOSPI200 membership 유효

#### Clean Peak Give-back Target
Prediction cutoff 이후 미래만 사용:
- 단기과열: `Peak BHAR >= 10%` AND `Give-back >= 50%`
- 지속상승: `Peak BHAR >= 10%` AND `Give-back <= 20%`
- 그 외 neutral
- MDD는 라벨에 쓰지 않고 보조지표로만 저장

#### 외국인 T+5 수급비율
`foreign_flow_ratio_T5 = T+1~T+5 외국인 순매수 누적 / T 시가총액`

In [ ]:
# 1. packages / imports
!pip -q install duckdb pyarrow pyyaml scipy statsmodels matplotlib

from pathlib import Path
from datetime import datetime
import json, hashlib, subprocess
import duckdb
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency, kruskal, fisher_exact
import statsmodels.api as sm
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 360)
pd.set_option("display.max_rows", 250)

print("DuckDB:", duckdb.__version__)

In [ ]:
# 2. Paths / config
PROJECT_ID = GCP_PROJECT_ID
BUCKET = GCS_BUCKET_NAME

ROOT = Path(DRIVE_REFACTOR_ROOT)
DB_PATH = ROOT / "duckdb" / "finda.duckdb"
CONFIG_DIR = ROOT / "configs"
DERIVED_DIR = ROOT / "derived" / "events"
ANALYSIS_DIR = ROOT / "outputs" / "t5_foreign_flow_analysis"
META_DIR = ROOT / "metadata"

for p in [CONFIG_DIR, DERIVED_DIR, ANALYSIS_DIR, META_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CFG_PATH = CONFIG_DIR / "event_t5_analysis.yaml"
META_PATH = META_DIR / "event_t5_analysis_manifest.json"

CFG = {
    "event": {
        "return_threshold": 0.20,
        "volume_ratio_threshold": 1.50,
        "cooldown_trading_days": 20,
    },
    "prediction_offsets": [0, 3, 5, 10],
    "target_horizons": [10, 20, 40],
    "label": {
        "peak_bhar_min": 0.10,
        "overheat_giveback_min": 0.50,
        "sustained_giveback_max": 0.20,
        "mdd_in_label": False,
    },
    "foreign_analysis": {
        "prediction_offset": 5,
        "main_horizon": 20,
        "path_horizon": 40,
        "development_end": "2024-12-31",
        "oot_start": "2025-01-01",
        "oot_end": "2026-08-10",
        "min_threshold_group_n": 30,
    },
}

with open(CFG_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(CFG, f, allow_unicode=True, sort_keys=False)

assert DB_PATH.exists(), DB_PATH
print(CFG_PATH)

### PART A — GCS → Drive mirror 최신 동기화 후 DuckDB 연결

STEP 4에서 direct GCS 인증은 사용하지 않고 `drive_mirror`로 확정했으므로,
노트북 시작 시 canonical 5개만 다시 동기화합니다.

In [ ]:
# 심사용 실행에서는 GCS를 거치지 않습니다. STAGE 13이 이미 로컬 curated 폴더를
# 기준으로 DuckDB canonical view를 만들어뒀고, view는 쿼리할 때마다 그 시점의
# 로컬 파일을 다시 읽으므로(스냅샷이 아님) STAGE 11·12가 갱신한 최신 membership/
# market이 자동으로 반영됩니다. 별도 동기화가 필요 없습니다.
print("✅ 로컬 canonical 최신 상태 확인 (별도 동기화 불필요)")

In [ ]:
# 4. Connect + canonical QA
con = duckdb.connect(str(DB_PATH))
for schema in ["canonical", "feature", "analysis", "meta", "event"]:
    con.execute(f"CREATE SCHEMA IF NOT EXISTS {schema}")

qa = con.execute("""
SELECT
    (SELECT COUNT(*) FROM canonical.stock_daily) AS stock_count,
    (SELECT COUNT(*) FROM canonical.attention_daily) AS attention_count,
    (SELECT COUNT(*) FROM canonical.short_daily) AS short_count,
    (SELECT COUNT(*) FROM canonical.membership_daily) AS membership_count,
    (SELECT COUNT(*) FROM canonical.market_daily) AS market_count
""").df()

display(qa)
assert int(qa.loc[0, "stock_count"]) == 253111
assert int(qa.loc[0, "membership_count"]) == 252911
assert int(qa.loc[0, "market_count"]) == 1264
print("✅ canonical QA")

### PART B — Daily source + T시점 파생변수

In [ ]:
# 5. Load canonical data
stock = con.execute("""
SELECT
    ticker, ticker_name, market, date,
    open, high, low, close, volume, market_cap, trading_value, listed_shares,
    per, pbr, eps,
    individual_net_buy_value, foreign_net_buy_value, institution_net_buy_value,
    other_corporation_net_buy_value, total_net_buy_value,
    foreign_ownership_pct, foreign_holding_shares,
    daily_return, return_5d, adj_daily_return, adj_return_5d,
    volume_to_20d_avg, trading_value_growth, close_diff_pct
FROM canonical.stock_daily
ORDER BY ticker, date
""").df()

membership = con.execute("""
SELECT ticker, date
FROM canonical.membership_daily
WHERE is_member = 1
ORDER BY ticker, date
""").df()

attention = con.execute("""
SELECT * FROM canonical.attention_daily
ORDER BY ticker, date
""").df()

market = con.execute("""
SELECT * FROM canonical.market_daily
ORDER BY date
""").df()

short = con.execute("""
SELECT * FROM canonical.short_daily
ORDER BY ticker, date
""").df()

for df in [stock, membership, attention, market, short]:
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()

for df in [stock, membership, attention, short]:
    df["ticker"] = df["ticker"].astype(str).str.zfill(6)

print(stock.shape, membership.shape, attention.shape, market.shape, short.shape)

In [ ]:
# 6. T-available derived variables
stock = stock.sort_values(["ticker", "date"]).reset_index(drop=True)
stock["ticker_tidx"] = stock.groupby("ticker").cumcount()
g = stock.groupby("ticker", group_keys=False)

for w in [1, 5, 10, 20]:
    stock[f"ret_{w}d_calc"] = g["close"].pct_change(w)

for w in [5, 20, 60]:
    stock[f"volatility_{w}d_calc"] = (
        g["ret_1d_calc"].rolling(w, min_periods=w).std().reset_index(level=0, drop=True)
    )

# Event denominator: T를 제외한 직전 20거래일
stock["volume_prior20_mean"] = g["volume"].transform(
    lambda s: s.shift(1).rolling(20, min_periods=20).mean()
)
stock["trading_value_prior20_mean"] = g["trading_value"].transform(
    lambda s: s.shift(1).rolling(20, min_periods=20).mean()
)
stock["volume_ratio_prior20_calc"] = stock["volume"] / stock["volume_prior20_mean"]
stock["trading_value_ratio_prior20_calc"] = (
    stock["trading_value"] / stock["trading_value_prior20_mean"]
)

stock["turnover_calc"] = np.where(
    stock["market_cap"].gt(0),
    stock["trading_value"] / stock["market_cap"],
    np.nan,
)
stock["high_low_range_calc"] = np.where(
    stock["close"].gt(0),
    (stock["high"] - stock["low"]) / stock["close"],
    np.nan,
)
stock["up_day_flag"] = stock["ret_1d_calc"].gt(0).astype(float)
stock["up_day_share_10_calc"] = (
    g["up_day_flag"].rolling(10, min_periods=10).mean().reset_index(level=0, drop=True)
)
stock["max_daily_return_10_calc"] = (
    g["ret_1d_calc"].rolling(10, min_periods=10).max().reset_index(level=0, drop=True)
)

FLOW_COLS = {
    "foreign": "foreign_net_buy_value",
    "institution": "institution_net_buy_value",
    "individual": "individual_net_buy_value",
}
for prefix, col in FLOW_COLS.items():
    stock[f"{prefix}_net_buy_5d_calc"] = (
        g[col].rolling(5, min_periods=1).sum().reset_index(level=0, drop=True)
    )
    stock[f"{prefix}_net_buy_10d_calc"] = (
        g[col].rolling(10, min_periods=1).sum().reset_index(level=0, drop=True)
    )
    stock[f"{prefix}_net_buy_to_mcap_calc"] = np.where(
        stock["market_cap"].gt(0),
        stock[col] / stock["market_cap"],
        np.nan,
    )

stock["foreign_ownership_change_5d_calc"] = g["foreign_ownership_pct"].diff(5)
stock["foreign_ownership_change_10d_calc"] = g["foreign_ownership_pct"].diff(10)

print("✅ T시점 파생변수")

In [ ]:
# 7. Attention new-entry / lead prep
ATT_THR = 0.90
attention = attention.sort_values(["ticker", "date"]).reset_index(drop=True)
ag = attention.groupby("ticker", group_keys=False)

attention["search_high_flag"] = attention["search_pct_50"].ge(ATT_THR)
attention["news_high_flag"] = attention["news_pct_50"].ge(ATT_THR)

attention["search_prev_high"] = ag["search_high_flag"].shift(1).fillna(False).astype(bool)
attention["news_prev_high"] = ag["news_high_flag"].shift(1).fillna(False).astype(bool)

attention["search_new_entry_flag"] = attention["search_high_flag"] & ~attention["search_prev_high"]
attention["news_new_entry_flag"] = attention["news_high_flag"] & ~attention["news_prev_high"]

attention["search_entry_date"] = attention["date"].where(attention["search_new_entry_flag"])
attention["news_entry_date"] = attention["date"].where(attention["news_new_entry_flag"])
attention["search_last_entry_date"] = attention.groupby("ticker")["search_entry_date"].ffill()
attention["news_last_entry_date"] = attention.groupby("ticker")["news_entry_date"].ffill()

print("✅ attention lead 준비")

### PART C — Event Builder

In [ ]:
# 8. Candidate + cooldown
panel = stock.merge(
    membership.assign(is_member=1),
    on=["ticker", "date"],
    how="left",
    validate="one_to_one",
)
panel["is_member"] = panel["is_member"].fillna(0).astype(int)

att_cols = [
    "ticker","date","search_pct_50","news_pct_50","search_news_level_gap",
    "search_high_flag","news_high_flag",
    "search_new_entry_flag","news_new_entry_flag",
    "search_last_entry_date","news_last_entry_date",
]
panel = panel.merge(attention[att_cols], on=["ticker","date"], how="left", validate="one_to_one")
panel["search_lead_days_calc"] = (panel["date"] - panel["search_last_entry_date"]).dt.days
panel["news_lead_days_calc"] = (panel["date"] - panel["news_last_entry_date"]).dt.days

panel["candidate_flag"] = (
    panel["is_member"].eq(1)
    & panel["ret_20d_calc"].ge(CFG["event"]["return_threshold"])
    & panel["volume_ratio_prior20_calc"].ge(CFG["event"]["volume_ratio_threshold"])
)
candidates = panel[panel["candidate_flag"]].copy()

COOLDOWN = int(CFG["event"]["cooldown_trading_days"])
accepted = []

for ticker, grp in candidates.groupby("ticker", sort=False):
    grp = grp.sort_values("ticker_tidx")
    last_tidx = None
    for idx, row in grp.iterrows():
        tidx = int(row["ticker_tidx"])
        if last_tidx is None or (tidx - last_tidx) > COOLDOWN:
            accepted.append(idx)
            last_tidx = tidx

event_master = candidates.loc[accepted].sort_values(["date","ticker"]).reset_index(drop=True)
event_master["event_id"] = event_master["ticker"] + "_" + event_master["date"].dt.strftime("%Y%m%d")
event_master = event_master.rename(columns={"date":"event_date","ticker_tidx":"event_tidx"})

assert event_master["event_id"].is_unique
tmp = event_master[["ticker","event_tidx"]].sort_values(["ticker","event_tidx"]).copy()
tmp["gap"] = tmp.groupby("ticker")["event_tidx"].diff()
assert not (tmp["gap"].dropna() <= COOLDOWN).any()

print("accepted event_n:", len(event_master))
print("ticker_n:", event_master["ticker"].nunique())
print("period:", event_master["event_date"].min(), "~", event_master["event_date"].max())

In [ ]:
# 9. Event Master T snapshot
keep = [
    "event_id","ticker","ticker_name","market","event_date","event_tidx",
    "ret_1d_calc","ret_5d_calc","ret_10d_calc","ret_20d_calc",
    "volatility_5d_calc","volatility_20d_calc","volatility_60d_calc",
    "volume_prior20_mean","volume_ratio_prior20_calc","trading_value_ratio_prior20_calc",
    "turnover_calc","high_low_range_calc","up_day_share_10_calc","max_daily_return_10_calc",
    "close","volume","trading_value","market_cap",
    "foreign_net_buy_value","foreign_net_buy_5d_calc","foreign_net_buy_10d_calc",
    "foreign_net_buy_to_mcap_calc","foreign_ownership_pct",
    "foreign_ownership_change_5d_calc","foreign_ownership_change_10d_calc",
    "institution_net_buy_value","institution_net_buy_5d_calc","institution_net_buy_10d_calc",
    "institution_net_buy_to_mcap_calc",
    "individual_net_buy_value","individual_net_buy_5d_calc","individual_net_buy_10d_calc",
    "individual_net_buy_to_mcap_calc",
    "per","pbr","eps",
    "search_pct_50","news_pct_50","search_news_level_gap",
    "search_high_flag","news_high_flag","search_new_entry_flag","news_new_entry_flag",
    "search_lead_days_calc","news_lead_days_calc",
]
event_master = event_master[keep].copy()

rename_map = {
    "ret_1d_calc":"ret_1d_T","ret_5d_calc":"ret_5d_T","ret_10d_calc":"ret_10d_T","ret_20d_calc":"ret_20d_T",
    "volatility_5d_calc":"volatility_5d_T","volatility_20d_calc":"volatility_20d_T","volatility_60d_calc":"volatility_60d_T",
    "volume_prior20_mean":"volume_prior20_mean_T","volume_ratio_prior20_calc":"volume_ratio_prior20_T",
    "trading_value_ratio_prior20_calc":"trading_value_ratio_prior20_T",
    "turnover_calc":"turnover_T","high_low_range_calc":"high_low_range_T",
    "up_day_share_10_calc":"up_day_share_10_T","max_daily_return_10_calc":"max_daily_return_10_T",
    "close":"close_T","volume":"volume_T","trading_value":"trading_value_T","market_cap":"market_cap_T",
    "foreign_net_buy_value":"foreign_net_buy_T","foreign_net_buy_5d_calc":"foreign_net_buy_5d_T",
    "foreign_net_buy_10d_calc":"foreign_net_buy_10d_T","foreign_net_buy_to_mcap_calc":"foreign_net_buy_to_mcap_T",
    "foreign_ownership_pct":"foreign_ownership_pct_T",
    "foreign_ownership_change_5d_calc":"foreign_ownership_change_5d_T",
    "foreign_ownership_change_10d_calc":"foreign_ownership_change_10d_T",
    "institution_net_buy_value":"institution_net_buy_T",
    "institution_net_buy_5d_calc":"institution_net_buy_5d_T","institution_net_buy_10d_calc":"institution_net_buy_10d_T",
    "institution_net_buy_to_mcap_calc":"institution_net_buy_to_mcap_T",
    "individual_net_buy_value":"individual_net_buy_T",
    "individual_net_buy_5d_calc":"individual_net_buy_5d_T","individual_net_buy_10d_calc":"individual_net_buy_10d_T",
    "individual_net_buy_to_mcap_calc":"individual_net_buy_to_mcap_T",
    "per":"per_T","pbr":"pbr_T","eps":"eps_T",
    "search_pct_50":"search_pct_50_T","news_pct_50":"news_pct_50_T",
    "search_news_level_gap":"search_news_level_gap_T",
    "search_high_flag":"search_high_T","news_high_flag":"news_high_T",
    "search_new_entry_flag":"search_new_entry_T","news_new_entry_flag":"news_new_entry_T",
    "search_lead_days_calc":"search_lead_days","news_lead_days_calc":"news_lead_days",
}
event_master = event_master.rename(columns=rename_map)
display(event_master.head())
print(event_master.shape)

### PART D — T/T+3/T+5/T+10 Snapshot Builder

In [ ]:
# 10. Lookups
stock_lookup = stock.set_index(["ticker","ticker_tidx"], drop=False).sort_index()
stock_by_ticker = {
    t:gdf.sort_values("ticker_tidx").copy()
    for t,gdf in stock.groupby("ticker", sort=False)
}
attention_by_ticker = {
    t:gdf.sort_values("date").copy()
    for t,gdf in attention.groupby("ticker", sort=False)
}
market_lookup = market.set_index("date", drop=False).sort_index()
short_lookup = short.set_index(["ticker","date"], drop=False).sort_index()

PRED_OFFSETS = [int(x) for x in CFG["prediction_offsets"]]

In [ ]:
# 11. Build snapshots
snapshot_rows = []

for _, ev in event_master.iterrows():
    ticker = ev["ticker"]
    event_tidx = int(ev["event_tidx"])
    event_date = pd.Timestamp(ev["event_date"])
    sg = stock_by_ticker[ticker]

    for offset in PRED_OFFSETS:
        cutoff_tidx = event_tidx + offset
        if (ticker, cutoff_tidx) not in stock_lookup.index:
            continue

        crow = stock_lookup.loc[(ticker, cutoff_tidx)]
        cutoff_date = pd.Timestamp(crow["date"])
        early = sg[sg["ticker_tidx"].between(event_tidx, cutoff_tidx, inclusive="both")].copy()
        post_t = early[early["ticker_tidx"] > event_tidx].copy()

        close_T = ev["close_T"]
        close_cutoff = crow["close"]
        early_return = (
            close_cutoff / close_T - 1.0
            if pd.notna(close_T) and close_T > 0 and pd.notna(close_cutoff)
            else np.nan
        )

        ret_path = early["close"] / close_T - 1.0
        early_peak = float(ret_path.max()) if ret_path.notna().any() else np.nan
        early_giveback = (
            (early_peak - early_return) / early_peak
            if pd.notna(early_peak) and early_peak > 0 and pd.notna(early_return)
            else np.nan
        )

        close_path = early["close"].astype(float)
        early_mdd = (
            float((close_path / close_path.cummax() - 1.0).min())
            if close_path.notna().any() else np.nan
        )

        volume_sustain = (
            float(post_t["volume"].mean() / ev["volume_prior20_mean_T"])
            if offset > 0 and len(post_t) and pd.notna(ev["volume_prior20_mean_T"]) and ev["volume_prior20_mean_T"] > 0
            else np.nan
        )

        row = {
            "event_id":ev["event_id"],"ticker":ticker,"ticker_name":ev["ticker_name"],
            "event_date":event_date,"event_tidx":event_tidx,
            "prediction_offset":offset,"prediction_cutoff":f"T+{offset}",
            "cutoff_tidx":cutoff_tidx,"cutoff_date":cutoff_date,
            "early_return":early_return,
            "early_peak_return":early_peak,
            "early_giveback_ratio":early_giveback,
            "early_true_mdd":early_mdd,
            "volume_sustain":volume_sustain,
            "volume_change_vs_T":(
                float(crow["volume"] / ev["volume_T"] - 1.0)
                if pd.notna(crow["volume"]) and pd.notna(ev["volume_T"]) and ev["volume_T"] > 0
                else np.nan
            ),
        }

        for c in event_master.columns:
            if c not in ["event_id","ticker","ticker_name","event_date","event_tidx"]:
                row[c] = ev[c]

        for prefix, col in FLOW_COLS.items():
            cum_val = (
                float(pd.to_numeric(post_t[col], errors="coerce").sum(min_count=1))
                if offset > 0 and len(post_t) else np.nan
            )
            row[f"{prefix}_net_buy_cum"] = cum_val
            row[f"{prefix}_flow_ratio"] = (
                cum_val / ev["market_cap_T"]
                if pd.notna(cum_val) and pd.notna(ev["market_cap_T"]) and ev["market_cap_T"] > 0
                else np.nan
            )

        row["foreign_ownership_change"] = (
            float(crow["foreign_ownership_pct"] - ev["foreign_ownership_pct_T"])
            if pd.notna(crow["foreign_ownership_pct"]) and pd.notna(ev["foreign_ownership_pct_T"])
            else np.nan
        )

        agdf = attention_by_ticker.get(ticker)
        aw = (
            agdf[agdf["date"].between(event_date, cutoff_date, inclusive="both")].copy()
            if agdf is not None else pd.DataFrame()
        )
        for prefix in ["search","news"]:
            col = f"{prefix}_pct_50"
            if len(aw) and aw[col].notna().any():
                row[f"{prefix}_mean_to_cutoff"] = float(aw[col].mean())
                row[f"{prefix}_max_to_cutoff"] = float(aw[col].max())
                row[f"{prefix}_last_at_cutoff"] = float(aw.sort_values("date")[col].dropna().iloc[-1])
                row[f"{prefix}_high_share_to_cutoff"] = float(aw[col].ge(ATT_THR).mean())
            else:
                row[f"{prefix}_mean_to_cutoff"] = np.nan
                row[f"{prefix}_max_to_cutoff"] = np.nan
                row[f"{prefix}_last_at_cutoff"] = np.nan
                row[f"{prefix}_high_share_to_cutoff"] = np.nan

            base = ev.get(f"{prefix}_pct_50_T", np.nan)
            last = row[f"{prefix}_last_at_cutoff"]
            row[f"{prefix}_sustain_ratio"] = (
                last / base if pd.notna(last) and pd.notna(base) and base != 0 else np.nan
            )

        row["search_news_level_gap_cutoff"] = (
            aw.sort_values("date").iloc[-1].get("search_news_level_gap", np.nan)
            if len(aw) else np.nan
        )

        if cutoff_date in market_lookup.index:
            mr = market_lookup.loc[cutoff_date]
            for c in [
                "market_drawdown_60d","market_volatility_20d","market_trading_value_change_5_20",
                "market_decline_ratio","market_below_ma20_ratio","market_crash_3pct_ratio",
                "market_illiquidity_shock","market_proxy_return","market_proxy_index"
            ]:
                if c in mr.index:
                    row[f"{c}_cutoff"] = mr[c]

        try:
            sr = short_lookup.loc[(ticker, cutoff_date)]
            if isinstance(sr, pd.DataFrame):
                sr = sr.iloc[-1]
            for c in short.columns:
                if c not in ["ticker","date","ticker_name"] and c.startswith("short_"):
                    row[f"{c}_cutoff"] = sr.get(c, np.nan)
        except KeyError:
            pass

        snapshot_rows.append(row)

snapshots = pd.DataFrame(snapshot_rows).sort_values(
    ["event_date","ticker","prediction_offset"]
).reset_index(drop=True)

mask5 = snapshots["prediction_offset"].eq(5)
snapshots["foreign_flow_ratio_T5"] = np.where(mask5, snapshots["foreign_flow_ratio"], np.nan)
snapshots["institution_flow_ratio_T5"] = np.where(mask5, snapshots["institution_flow_ratio"], np.nan)
snapshots["individual_flow_ratio_T5"] = np.where(mask5, snapshots["individual_flow_ratio"], np.nan)
snapshots["early_giveback_T5"] = np.where(mask5, snapshots["early_giveback_ratio"], np.nan)
snapshots["volume_sustain_T5"] = np.where(mask5, snapshots["volume_sustain"], np.nan)

print("snapshots:", snapshots.shape)
display(snapshots[[
    "event_id","prediction_offset","cutoff_date",
    "early_return","early_giveback_ratio","volume_sustain",
    "foreign_net_buy_cum","foreign_flow_ratio"
]].head(30))

### PART E — Clean Peak Give-back Target + BHAR path

In [ ]:
# 12. Target / BHAR path
HORIZONS = [int(x) for x in CFG["target_horizons"]]
PATH_H = int(CFG["foreign_analysis"]["path_horizon"])
PEAK_MIN = float(CFG["label"]["peak_bhar_min"])
OVERHEAT_GB = float(CFG["label"]["overheat_giveback_min"])
SUSTAINED_GB = float(CFG["label"]["sustained_giveback_max"])

target_rows = []
path_rows = []

for _, s in snapshots.iterrows():
    ticker = s["ticker"]
    cutoff_tidx = int(s["cutoff_tidx"])
    cutoff_date = pd.Timestamp(s["cutoff_date"])

    if cutoff_date not in market_lookup.index:
        continue

    cutoff_close = stock_lookup.loc[(ticker, cutoff_tidx), "close"]
    cutoff_market = market_lookup.loc[cutoff_date, "market_proxy_index"]
    if pd.isna(cutoff_close) or cutoff_close <= 0 or pd.isna(cutoff_market) or cutoff_market <= 0:
        continue

    future_all = stock_by_ticker[ticker][
        (stock_by_ticker[ticker]["ticker_tidx"] > cutoff_tidx)
        & (stock_by_ticker[ticker]["ticker_tidx"] <= cutoff_tidx + PATH_H)
    ].copy()

    future_all = future_all.merge(
        market[["date","market_proxy_index"]],
        on="date", how="left", validate="one_to_one"
    )
    future_all["future_step"] = np.arange(1, len(future_all)+1)
    future_all["stock_wealth"] = future_all["close"] / cutoff_close
    future_all["market_wealth"] = future_all["market_proxy_index"] / cutoff_market
    future_all["bhar"] = future_all["stock_wealth"] / future_all["market_wealth"] - 1.0

    for _, p in future_all.iterrows():
        path_rows.append({
            "event_id":s["event_id"],"ticker":ticker,"event_date":s["event_date"],
            "prediction_offset":int(s["prediction_offset"]),
            "cutoff_date":cutoff_date,
            "future_step":int(p["future_step"]),
            "path_date":p["date"],
            "bhar":p["bhar"],
        })

    for H in HORIZONS:
        fut = future_all[future_all["future_step"] <= H].copy()
        if len(fut) < H:
            continue

        peak_bhar = float(fut["bhar"].max())
        end_bhar = float(fut["bhar"].iloc[-1])
        giveback = (peak_bhar - end_bhar) / peak_bhar if peak_bhar > 0 else np.nan

        wealth = pd.concat([pd.Series([1.0]), fut["stock_wealth"].reset_index(drop=True)], ignore_index=True)
        future_true_mdd = float((wealth / wealth.cummax() - 1.0).min())

        if peak_bhar >= PEAK_MIN and pd.notna(giveback) and giveback >= OVERHEAT_GB:
            label_class, binary_label = "overheat_reversal", 1
        elif peak_bhar >= PEAK_MIN and pd.notna(giveback) and giveback <= SUSTAINED_GB:
            label_class, binary_label = "sustained", 0
        else:
            label_class, binary_label = "neutral", np.nan

        target_rows.append({
            "event_id":s["event_id"],"ticker":ticker,"event_date":s["event_date"],
            "prediction_offset":int(s["prediction_offset"]),
            "prediction_cutoff":s["prediction_cutoff"],
            "cutoff_date":cutoff_date,
            "target_horizon":H,
            "target_start_date":fut["date"].iloc[0],
            "target_end_date":fut["date"].iloc[-1],
            "future_peak_bhar":peak_bhar,
            "future_end_bhar":end_bhar,
            "future_giveback_ratio":giveback,
            "future_true_mdd":future_true_mdd,
            "label_class":label_class,
            "binary_label":binary_label,
        })

targets = pd.DataFrame(target_rows)
bhar_path = pd.DataFrame(path_rows)

assert (targets["target_start_date"] > targets["cutoff_date"]).all()
assert CFG["label"]["mdd_in_label"] is False

print("targets:", targets.shape)
print("bhar_path:", bhar_path.shape)

display(
    targets.groupby(["prediction_offset","target_horizon","label_class"])
    .size().rename("n").reset_index()
)

In [ ]:
# 13. Merge model frame
model_frame = snapshots.merge(
    targets,
    on=["event_id","ticker","event_date","prediction_offset","prediction_cutoff","cutoff_date"],
    how="inner",
    validate="one_to_many",
)
model_frame = model_frame.sort_values(
    ["event_date","ticker","prediction_offset","target_horizon"]
).reset_index(drop=True)

print("model_frame:", model_frame.shape)
print("event_n:", model_frame["event_id"].nunique())

## PART F — T+5 외국인 수급 분석

기본 설명분석 라벨은 `T+5 / H20 / overheat vs sustained`를 사용합니다.
분위 경계와 threshold는 **2021~2024 Development에서만 생성**합니다.

In [ ]:
# 14. Analysis frame
MAIN_H = int(CFG["foreign_analysis"]["main_horizon"])
DEV_END = pd.Timestamp(CFG["foreign_analysis"]["development_end"])
OOT_START = pd.Timestamp(CFG["foreign_analysis"]["oot_start"])
OOT_END = pd.Timestamp(CFG["foreign_analysis"]["oot_end"])

t5 = model_frame[
    model_frame["prediction_offset"].eq(5)
    & model_frame["target_horizon"].eq(MAIN_H)
].copy()

t5_bin = t5.dropna(
    subset=["binary_label","foreign_flow_ratio_T5","early_giveback_T5"]
).copy()
t5_bin["binary_label"] = t5_bin["binary_label"].astype(int)
t5_bin["foreign_flow_pct_T5"] = t5_bin["foreign_flow_ratio_T5"] * 100

dev = t5_bin[t5_bin["event_date"] <= DEV_END].copy()
oot = t5_bin[t5_bin["event_date"].between(OOT_START, OOT_END, inclusive="both")].copy()

print("T+5 H20 binary total:", len(t5_bin))
print("DEV:", len(dev), "OOT:", len(oot))

In [ ]:
# 15. DEV quintiles + OOT fixed application
def quantile_edges(series, q):
    x = pd.to_numeric(series, errors="coerce").dropna()
    return np.unique(x.quantile(np.linspace(0,1,q+1)).to_numpy())

def apply_edges(series, edges, prefix):
    inner = np.asarray(edges[1:-1], dtype=float)
    bins = np.r_[-np.inf, inner, np.inf]
    labels = [f"{prefix}{i}" for i in range(1, len(bins))]
    return pd.cut(series, bins=bins, labels=labels, include_lowest=True), bins

q5_edges = quantile_edges(dev["foreign_flow_ratio_T5"], 5)
dev["foreign_q5"], q5_bins = apply_edges(dev["foreign_flow_ratio_T5"], q5_edges, "Q")
oot["foreign_q5"], _ = apply_edges(oot["foreign_flow_ratio_T5"], q5_edges, "Q")

def q_summary(df, col):
    return (
        df.groupby(col, observed=False)
        .agg(
            n=("binary_label","size"),
            overheat_n=("binary_label","sum"),
            overheat_rate=("binary_label","mean"),
            foreign_ratio_median=("foreign_flow_ratio_T5","median"),
            foreign_ratio_min=("foreign_flow_ratio_T5","min"),
            foreign_ratio_max=("foreign_flow_ratio_T5","max"),
        )
        .reset_index()
    )

q5_dev = q_summary(dev, "foreign_q5")
q5_oot = q_summary(oot, "foreign_q5")

print("DEV q5 bins:", q5_bins)
print("[DEV]")
display(q5_dev)
print("[OOT, same bins]")
display(q5_oot)

In [ ]:
# 16. Statistical tests for quintiles
ct = pd.crosstab(dev["foreign_q5"], dev["binary_label"])
chi2, p_chi, dof, expected = chi2_contingency(ct)

dev["foreign_q5_ord"] = pd.to_numeric(
    dev["foreign_q5"].astype(str).str.extract(r"(\d+)")[0],
    errors="coerce",
)
trend = smf.glm(
    "binary_label ~ foreign_q5_ord",
    data=dev,
    family=sm.families.Binomial(),
).fit()

display(pd.DataFrame([
    {"test":"Chi-square across quintiles","stat":chi2,"p_value":p_chi},
    {
        "test":"Ordinal logistic trend",
        "stat":float(trend.params["foreign_q5_ord"]),
        "p_value":float(trend.pvalues["foreign_q5_ord"]),
    }
]))

In [ ]:
# 17. Median BHAR path by DEV-derived quintile
event_bins = pd.concat([
    dev[["event_id","foreign_q5"]].assign(split="DEV"),
    oot[["event_id","foreign_q5"]].assign(split="OOT"),
], ignore_index=True)

path5 = bhar_path[
    bhar_path["prediction_offset"].eq(5)
    & bhar_path["future_step"].le(int(CFG["foreign_analysis"]["path_horizon"]))
].merge(event_bins, on="event_id", how="inner", validate="many_to_one")

median_path = (
    path5.groupby(["split","foreign_q5","future_step"], observed=False)
    .agg(median_bhar=("bhar","median"), n=("bhar","count"))
    .reset_index()
)

display(median_path.head(20))

for split in ["DEV","OOT"]:
    fig, ax = plt.subplots(figsize=(10,5))
    tmp = median_path[median_path["split"].eq(split)]
    for q in sorted(tmp["foreign_q5"].dropna().astype(str).unique()):
        qdf = tmp[tmp["foreign_q5"].astype(str).eq(q)]
        ax.plot(qdf["future_step"], qdf["median_bhar"], label=q)
    ax.axhline(0, linewidth=1)
    ax.set_title(f"T+5 이후 median BHAR path by foreign-flow quintile — {split}")
    ax.set_xlabel("Trading days after T+5")
    ax.set_ylabel("Median BHAR")
    ax.legend()
    plt.show()

# H20/H40 endpoint nonparametric test on DEV
tests = []
for H in [20,40]:
    ep = path5[
        path5["split"].eq("DEV") & path5["future_step"].eq(H)
    ].dropna(subset=["bhar","foreign_q5"])
    groups = [g["bhar"].to_numpy() for _,g in ep.groupby("foreign_q5", observed=False) if len(g)]
    if len(groups) >= 2:
        stat,p = kruskal(*groups)
        tests.append({"horizon":H,"test":"Kruskal-Wallis","stat":stat,"p_value":p})
display(pd.DataFrame(tests))

In [ ]:
# 18. early_giveback_T5 × foreign Q5 grid
gb_edges = quantile_edges(dev["early_giveback_T5"], 3)
dev["giveback_q3"], gb_bins = apply_edges(dev["early_giveback_T5"], gb_edges, "G")
oot["giveback_q3"], _ = apply_edges(oot["early_giveback_T5"], gb_edges, "G")

def risk_grid(df):
    return (
        df.groupby(["giveback_q3","foreign_q5"], observed=False)
        .agg(n=("binary_label","size"), overheat_rate=("binary_label","mean"))
        .reset_index()
    )

grid_dev = risk_grid(dev)
grid_oot = risk_grid(oot)

print("[DEV rate]")
display(grid_dev.pivot(index="giveback_q3",columns="foreign_q5",values="overheat_rate"))
print("[DEV n]")
display(grid_dev.pivot(index="giveback_q3",columns="foreign_q5",values="n"))

print("[OOT rate]")
display(grid_oot.pivot(index="giveback_q3",columns="foreign_q5",values="overheat_rate"))
print("[OOT n]")
display(grid_oot.pivot(index="giveback_q3",columns="foreign_q5",values="n"))

# continuous interaction test, DEV only
for c in ["early_giveback_T5","foreign_flow_ratio_T5"]:
    mu, sd = dev[c].mean(), dev[c].std(ddof=0)
    dev[f"z_{c}"] = (dev[c]-mu)/sd if sd > 0 else 0.0

inter = smf.glm(
    "binary_label ~ z_early_giveback_T5 * z_foreign_flow_ratio_T5",
    data=dev,
    family=sm.families.Binomial(),
).fit()

display(pd.DataFrame({"coef":inter.params,"p_value":inter.pvalues}))

### PART G — 5분위 → 10분위 → breakpoint 후보 → OOT 고정 검증

In [ ]:
# 19. DEV deciles
q10_edges = quantile_edges(dev["foreign_flow_ratio_T5"], 10)
dev["foreign_q10"], q10_bins = apply_edges(dev["foreign_flow_ratio_T5"], q10_edges, "D")
oot["foreign_q10"], _ = apply_edges(oot["foreign_flow_ratio_T5"], q10_edges, "D")

q10_dev = q_summary(dev, "foreign_q10")
display(q10_dev)

fig, ax = plt.subplots(figsize=(10,5))
ax.plot(q10_dev["foreign_q10"].astype(str), q10_dev["overheat_rate"], marker="o")
ax.set_title("Development overheat rate by foreign_flow_ratio_T5 decile")
ax.set_xlabel("DEV decile")
ax.set_ylabel("Overheat rate")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# 20. Candidate thresholds from DEV decile boundaries
MIN_N = int(CFG["foreign_analysis"]["min_threshold_group_n"])
rows = []

for thr in q10_bins[1:-1]:
    low = dev[dev["foreign_flow_ratio_T5"] <= thr]
    high = dev[dev["foreign_flow_ratio_T5"] > thr]
    if len(low) < MIN_N or len(high) < MIN_N:
        continue

    low_rate = low["binary_label"].mean()
    high_rate = high["binary_label"].mean()

    table = np.array([
        [int(low["binary_label"].sum()), int(len(low)-low["binary_label"].sum())],
        [int(high["binary_label"].sum()), int(len(high)-high["binary_label"].sum())],
    ])
    odds_ratio, p = fisher_exact(table)

    rows.append({
        "threshold_decimal":float(thr),
        "threshold_pct":float(thr*100),
        "low_n":len(low),"high_n":len(high),
        "low_overheat_rate":low_rate,
        "high_overheat_rate":high_rate,
        "risk_difference_low_minus_high":low_rate-high_rate,
        "abs_risk_difference":abs(low_rate-high_rate),
        "odds_ratio":odds_ratio,
        "fisher_p":p,
    })

threshold_table = pd.DataFrame(rows).sort_values(
    ["abs_risk_difference","fisher_p"], ascending=[False,True]
)
display(threshold_table)

if len(threshold_table):
    locked_threshold = float(threshold_table.iloc[0]["threshold_decimal"])
else:
    locked_threshold = np.nan

print("LOCKED DEV threshold decimal:", locked_threshold)
print("LOCKED DEV threshold %:", None if pd.isna(locked_threshold) else locked_threshold*100)

In [ ]:
# 21. Locked threshold validation on DEV / OOT / yearly OOT
def threshold_eval(df, thr, split):
    x = df.dropna(subset=["foreign_flow_ratio_T5","binary_label"]).copy()
    x["foreign_low_flag"] = x["foreign_flow_ratio_T5"] <= thr

    low = x[x["foreign_low_flag"]]
    high = x[~x["foreign_low_flag"]]

    out = {
        "split":split,"threshold_decimal":thr,"threshold_pct":thr*100,
        "low_n":len(low),"high_n":len(high),
        "low_rate":low["binary_label"].mean() if len(low) else np.nan,
        "high_rate":high["binary_label"].mean() if len(high) else np.nan,
    }
    out["risk_difference"] = out["low_rate"] - out["high_rate"]

    if len(low) and len(high):
        table = np.array([
            [int(low["binary_label"].sum()), int(len(low)-low["binary_label"].sum())],
            [int(high["binary_label"].sum()), int(len(high)-high["binary_label"].sum())],
        ])
        odds_ratio, p = fisher_exact(table)
        out["odds_ratio"] = odds_ratio
        out["fisher_p"] = p
    else:
        out["odds_ratio"] = np.nan
        out["fisher_p"] = np.nan
    return out

locked_results = [
    threshold_eval(dev, locked_threshold, "DEV_2021_2024"),
    threshold_eval(oot, locked_threshold, "OOT_2025_2026"),
]
for year in [2025,2026]:
    locked_results.append(
        threshold_eval(oot[oot["event_date"].dt.year.eq(year)], locked_threshold, f"OOT_{year}")
    )

locked_results = pd.DataFrame(locked_results)
display(locked_results)

In [ ]:
# 22. Simple 2x2: early giveback × locked foreign threshold
giveback_median_dev = float(dev["early_giveback_T5"].median())

def two_by_two(df, split):
    x = df.copy()
    x["high_early_giveback"] = x["early_giveback_T5"] >= giveback_median_dev
    x["foreign_low_flag"] = x["foreign_flow_ratio_T5"] <= locked_threshold
    out = (
        x.groupby(["high_early_giveback","foreign_low_flag"])
        .agg(n=("binary_label","size"),overheat_rate=("binary_label","mean"))
        .reset_index()
    )
    out["split"] = split
    return out

display(two_by_two(dev,"DEV"))
display(two_by_two(oot,"OOT"))

### PART H — 저장 / DuckDB materialize / GCS 업로드

In [ ]:
# 23. Save Drive artifacts
OUT_EVENT = DERIVED_DIR / "event_master.parquet"
OUT_SNAPSHOT = DERIVED_DIR / "event_snapshots_long.parquet"
OUT_TARGET = DERIVED_DIR / "event_targets_long.parquet"
OUT_MODEL = DERIVED_DIR / "event_model_frame_long.parquet"
OUT_PATH = DERIVED_DIR / "event_bhar_path_long.parquet"

event_master.to_parquet(OUT_EVENT,index=False,compression="zstd")
snapshots.to_parquet(OUT_SNAPSHOT,index=False,compression="zstd")
targets.to_parquet(OUT_TARGET,index=False,compression="zstd")
model_frame.to_parquet(OUT_MODEL,index=False,compression="zstd")
bhar_path.to_parquet(OUT_PATH,index=False,compression="zstd")

q5_dev.to_csv(ANALYSIS_DIR/"foreign_flow_q5_dev.csv",index=False,encoding="utf-8-sig")
q5_oot.to_csv(ANALYSIS_DIR/"foreign_flow_q5_oot.csv",index=False,encoding="utf-8-sig")
q10_dev.to_csv(ANALYSIS_DIR/"foreign_flow_q10_dev.csv",index=False,encoding="utf-8-sig")
grid_dev.to_csv(ANALYSIS_DIR/"early_giveback_x_foreign_q5_dev.csv",index=False,encoding="utf-8-sig")
grid_oot.to_csv(ANALYSIS_DIR/"early_giveback_x_foreign_q5_oot.csv",index=False,encoding="utf-8-sig")
threshold_table.to_csv(ANALYSIS_DIR/"foreign_threshold_candidates_dev.csv",index=False,encoding="utf-8-sig")
locked_results.to_csv(ANALYSIS_DIR/"foreign_locked_threshold_oot.csv",index=False,encoding="utf-8-sig")
median_path.to_csv(ANALYSIS_DIR/"foreign_q5_median_bhar_path.csv",index=False,encoding="utf-8-sig")

print("✅ Drive 저장")

In [ ]:
# 24. Materialize into DuckDB
def replace_table(name, df):
    temp = "_tmp_" + name.replace(".","_")
    con.register(temp, df)
    con.execute(f"CREATE OR REPLACE TABLE {name} AS SELECT * FROM {temp}")
    con.unregister(temp)

replace_table("event.event_master", event_master)
replace_table("event.snapshots_long", snapshots)
replace_table("event.targets_long", targets)
replace_table("event.model_frame_long", model_frame)
replace_table("event.bhar_path_long", bhar_path)
replace_table("meta.foreign_flow_q5_dev", q5_dev)
replace_table("meta.foreign_flow_q10_dev", q10_dev)
replace_table("meta.foreign_threshold_candidates_dev", threshold_table)

con.execute("CHECKPOINT")

duck_qa = con.execute("""
SELECT
    (SELECT COUNT(*) FROM event.event_master) AS event_count,
    (SELECT COUNT(*) FROM event.snapshots_long) AS snapshot_count,
    (SELECT COUNT(*) FROM event.targets_long) AS target_count,
    (SELECT COUNT(*) FROM event.model_frame_long) AS model_frame_count,
    (SELECT COUNT(*) FROM event.bhar_path_long) AS path_count
""").df()

display(duck_qa)

In [ ]:
# 25. Manifest + GCS upload
def sha256(path):
    h = hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            chunk = f.read(1024*1024)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "built_at":datetime.now().isoformat(),
    "event_n":int(event_master["event_id"].nunique()),
    "prediction_offsets":PRED_OFFSETS,
    "target_horizons":HORIZONS,
    "locked_foreign_threshold_decimal":None if pd.isna(locked_threshold) else float(locked_threshold),
    "locked_foreign_threshold_pct":None if pd.isna(locked_threshold) else float(locked_threshold*100),
    "threshold_selection":"2021-2024 Development only",
    "threshold_validation":"2025-2026 OOT fixed",
    "label_mdd_used":False,
    "files":{
        "event_master":str(OUT_EVENT),
        "snapshots":str(OUT_SNAPSHOT),
        "targets":str(OUT_TARGET),
        "model_frame":str(OUT_MODEL),
        "bhar_path":str(OUT_PATH),
    },
    "sha256":{
        "event_master":sha256(OUT_EVENT),
        "snapshots":sha256(OUT_SNAPSHOT),
        "targets":sha256(OUT_TARGET),
        "model_frame":sha256(OUT_MODEL),
        "bhar_path":sha256(OUT_PATH),
    }
}
with open(META_PATH,"w",encoding="utf-8") as f:
    json.dump(manifest,f,ensure_ascii=False,indent=2,default=str)

event_files = [OUT_EVENT,OUT_SNAPSHOT,OUT_TARGET,OUT_MODEL,OUT_PATH,CFG_PATH,META_PATH]
analysis_files = list(ANALYSIS_DIR.glob("*.csv"))

# 심사용 실행에서는 팀 GCS에 업로드하지 않습니다 — 이미 로컬(DERIVED_DIR, ANALYSIS_DIR)에
# 저장되어 있으므로 그 사실만 확인합니다.
for p in event_files + analysis_files:
    assert p.exists(), p

print("✅ Manifest 작성 + 로컬 저장 확인 완료 (" + str(len(event_files)+len(analysis_files)) + "개 파일)")

In [ ]:
# 26. Final QA
final_qa = pd.DataFrame([
    {"dataset":"event_master","row_count":len(event_master),"event_n":event_master["event_id"].nunique()},
    {"dataset":"snapshots","row_count":len(snapshots),"event_n":snapshots["event_id"].nunique()},
    {"dataset":"targets","row_count":len(targets),"event_n":targets["event_id"].nunique()},
    {"dataset":"model_frame","row_count":len(model_frame),"event_n":model_frame["event_id"].nunique()},
    {"dataset":"bhar_path","row_count":len(bhar_path),"event_n":bhar_path["event_id"].nunique()},
])
display(final_qa)

assert event_master["event_id"].is_unique
assert not snapshots.duplicated(["event_id","prediction_offset"]).any()
assert not targets.duplicated(["event_id","prediction_offset","target_horizon"]).any()
assert (targets["target_start_date"] > targets["cutoff_date"]).all()

print("locked foreign threshold %:",
      None if pd.isna(locked_threshold) else locked_threshold*100)
print("✅ STEP 5 완료")

# STAGE 15: STEP 6 FINAL — Direct T+20 BHAR × Prediction Timing × Feature Ablation (최종 라벨 확정: N=20, q35)

**태그:** 최종 — 팀 확정 라벨 정의

**원본 노트북:** `6___T20_BHAR_Q35_시점별_제거실험.ipynb + Target_N20_q35_안정성및견고성검증.ipynb`

> 이 STAGE는 원본 STEP 6 노트북 사이에 'N=20, q=35% 최종 채택' 검증 노트북의 내용을 끼워넣은 것입니다. 흐름: STEP 6의 PART A~D(데이터 동기화 → Event/Target 생성 → Feature Groups 구성)까지 진행한 뒤, **그 라벨/피처를 실제로 확정하기 전에** N=20·q=35%를 선택한 근거(N 후보 스윕의 안정성 점수, q 후보 스윕과 T+5 fold×seed paired test)를 보여주고, 이어서 STEP 6의 PART E부터(공통 Train-only q35 적용)를 계속합니다.

## FinDA STEP 6 FINAL — Direct T+20 BHAR × Prediction Timing × Feature Ablation

### 이번 노트북의 최종 질문

> **급등 사건의 T+20 시장대비 부진 위험은 언제부터 잘 보이는가?**

Target은 항상 사건일 `T` 기준 **Direct T+20 BHAR**로 고정하고, 예측시점만 바꿉니다.

```text
T / T+3 / T+5 / T+7 / T+10  →  동일한 T+20 Target
```

#### Binary Target
- 각 학습구간의 `direct_bhar_T20` 하위 35% 이하 → `1`
- 나머지 → `0`
- q35는 **각 Train 구간에서만 계산**
- 전체 데이터로 고정 라벨을 미리 만들지 않음

#### 기본 변수 P
- T까지의 가격·수익률·변동성·거래량
- T 이후 Prediction Cutoff까지의 초기 가격·거래량 반응

#### 추가 변수군
- Foreign
- Institution
- Individual
- Attention
- Market
- Short

#### 모델 / 검증
- RandomForest / LightGBM / LogisticRegression
- 2021~2024 Purged Expanding Walk-Forward
- 2025 Temporal Test
- 2026 Stress Test

기존 STEP 5에서 만든 `event.event_master`, `event.snapshots_long`, `event.bhar_path_long`을 재사용하며, **기존에 없던 T+7 Snapshot만 추가 계산**합니다.

In [ ]:
# 1. packages / imports
!pip -q install duckdb pyarrow pyyaml scikit-learn imbalanced-learn lightgbm scipy matplotlib

from pathlib import Path
from datetime import datetime
import json, hashlib, subprocess, time, warnings

import duckdb
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    brier_score_loss
)
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 360)
pd.set_option('display.max_rows', 300)

print('DuckDB:', duckdb.__version__)

In [ ]:
# 2. Config
PROJECT_ID = GCP_PROJECT_ID
BUCKET = GCS_BUCKET_NAME

ROOT = Path(DRIVE_REFACTOR_ROOT)
DB_PATH = ROOT / 'duckdb' / 'finda.duckdb'
CURATED_DIR = ROOT / 'curated'
DERIVED_DIR = ROOT / 'derived' / 'events'
OUT_DIR = ROOT / 'outputs' / 'step6_direct_t20_timing'
CONFIG_DIR = ROOT / 'configs'
META_DIR = ROOT / 'metadata'

for p in [DERIVED_DIR, OUT_DIR, CONFIG_DIR, META_DIR]:
    p.mkdir(parents=True, exist_ok=True)

PRED_OFFSETS = [0, 3, 5, 7, 10]
TARGET_HORIZON = 20
LABEL_Q = 0.35

DEV_END = pd.Timestamp('2024-12-31')
TEST25_START = pd.Timestamp('2025-01-01')
TEST25_END = pd.Timestamp('2025-12-31')
TEST26_START = pd.Timestamp('2026-01-01')
TEST26_END = pd.Timestamp('2026-12-31')

N_WF_SPLITS = 5
INITIAL_TRAIN_FRAC = 0.35
USE_SMOTE = True
GRID_SEED = 42
STABILITY_SEEDS = [11,22,33,44,55,66,77,88,99,111]
MODELS = ['rf','lgbm','logit']

CFG = {
    'target': {
        'type':'direct_t20_bhar',
        'horizon':20,
        'quantile':0.35,
        'threshold_scope':'train_period_only',
    },
    'prediction_offsets':PRED_OFFSETS,
    'development_end':str(DEV_END.date()),
    'temporal_test_2025':[str(TEST25_START.date()),str(TEST25_END.date())],
    'stress_test_2026':[str(TEST26_START.date()),str(TEST26_END.date())],
    'walk_forward_splits':N_WF_SPLITS,
    'use_smote':USE_SMOTE,
    'models':MODELS,
}

CFG_PATH = CONFIG_DIR / 'step6_direct_t20_timing.yaml'
with open(CFG_PATH,'w',encoding='utf-8') as f:
    yaml.safe_dump(CFG,f,allow_unicode=True,sort_keys=False)

assert DB_PATH.exists(), DB_PATH
print(CFG_PATH)

### PART A — GCS Canonical 최신 동기화 + DuckDB 연결

DuckDB `canonical.*` View는 Drive mirror의 Parquet을 읽으므로, 시작할 때 GCS canonical 5개를 mirror로 동기화합니다.

In [ ]:
# 심사용 실행에서는 GCS를 거치지 않습니다. STAGE 13이 만든 canonical view가 이미
# 로컬 curated 폴더(STAGE 3·11·12가 채운 CURATED_DIR)를 그대로 가리키고 있고, 쿼리할
# 때마다 최신 파일을 다시 읽으므로 별도 동기화가 필요 없습니다.
print("✅ 로컬 canonical 최신 상태 확인 (별도 동기화 불필요)")

In [ ]:
# 4. DuckDB inventory / STEP 5 prerequisite
con = duckdb.connect(str(DB_PATH))
for schema in ['canonical','feature','analysis','event','meta']:
    con.execute(f'CREATE SCHEMA IF NOT EXISTS {schema}')

inventory = con.execute("""
SELECT table_schema, table_name, table_type
FROM information_schema.tables
WHERE table_schema IN ('canonical','feature','analysis','event','meta')
ORDER BY table_schema, table_name
""").df()
display(inventory)

required = {'event_master','snapshots_long','bhar_path_long'}
present = set(inventory.loc[inventory['table_schema'].eq('event'),'table_name'].astype(str))
missing = sorted(required-present)
assert not missing, f'STEP 5 prerequisite missing: {missing}'
print('✅ STEP 5 tables 확인')

### PART B — 기존 Event Master 재사용 + T+7 Snapshot 생성

기존 `event.snapshots_long`의 T/T+3/T+5/T+10은 그대로 재사용하고, T+7만 STEP 5와 동일한 방식으로 계산합니다.

기존 테이블은 덮어쓰지 않고 전체 Timing Snapshot을 새 `event.snapshots_timing_long`으로 저장합니다.

In [ ]:
# 5. Load only data required for T+7
EVENT_MASTER_SQL = 'SELECT * FROM event.event_master ORDER BY event_date, ticker'
event_master = con.execute(EVENT_MASTER_SQL).df()

stock = con.execute("""
SELECT
    ticker, ticker_name, date, close, volume, market_cap,
    individual_net_buy_value, foreign_net_buy_value,
    institution_net_buy_value, foreign_ownership_pct
FROM canonical.stock_daily
ORDER BY ticker, date
""").df()

attention = con.execute('SELECT * FROM canonical.attention_daily ORDER BY ticker, date').df()
market = con.execute('SELECT * FROM canonical.market_daily ORDER BY date').df()
short = con.execute('SELECT * FROM canonical.short_daily ORDER BY ticker, date').df()

for df in [stock, attention, market, short]:
    df['date'] = pd.to_datetime(df['date'],errors='coerce').dt.normalize()
event_master['event_date'] = pd.to_datetime(event_master['event_date'],errors='coerce').dt.normalize()

for df in [stock,attention,short]:
    df['ticker'] = df['ticker'].astype(str).str.zfill(6)
event_master['ticker'] = event_master['ticker'].astype(str).str.zfill(6)

# STEP5의 ticker_tidx와 동일하게 canonical stock 정렬 후 복원
stock = stock.sort_values(['ticker','date']).reset_index(drop=True)
stock['ticker_tidx'] = stock.groupby('ticker').cumcount()

print('event_master:',event_master.shape)
print('stock:',stock.shape)

In [ ]:
# 6. Build T+7 snapshots
stock_lookup = stock.set_index(['ticker','ticker_tidx'],drop=False).sort_index()
stock_by_ticker = {
    t:g.sort_values('ticker_tidx').copy()
    for t,g in stock.groupby('ticker',sort=False)
}
attention_by_ticker = {
    t:g.sort_values('date').copy()
    for t,g in attention.groupby('ticker',sort=False)
}
market_lookup = market.set_index('date',drop=False).sort_index()
short_lookup = short.set_index(['ticker','date'],drop=False).sort_index()

FLOW_COLS = {
    'foreign':'foreign_net_buy_value',
    'institution':'institution_net_buy_value',
    'individual':'individual_net_buy_value',
}
ATT_THR = 0.90
MARKET_COLS = [
    'market_drawdown_60d','market_volatility_20d',
    'market_trading_value_change_5_20','market_decline_ratio',
    'market_below_ma20_ratio','market_crash_3pct_ratio',
    'market_illiquidity_shock','market_proxy_return','market_proxy_index'
]

snapshot7_rows=[]

for _,ev in event_master.iterrows():
    ticker = ev['ticker']
    event_tidx = int(ev['event_tidx'])
    event_date = pd.Timestamp(ev['event_date'])
    cutoff_tidx = event_tidx + 7

    if ticker not in stock_by_ticker or (ticker,cutoff_tidx) not in stock_lookup.index:
        continue

    sg = stock_by_ticker[ticker]
    crow = stock_lookup.loc[(ticker,cutoff_tidx)]
    if isinstance(crow,pd.DataFrame):
        crow = crow.iloc[-1]
    cutoff_date = pd.Timestamp(crow['date'])

    early = sg[sg['ticker_tidx'].between(event_tidx,cutoff_tidx,inclusive='both')].copy()
    post_t = early[early['ticker_tidx']>event_tidx].copy()

    close_T = ev['close_T']
    close_cutoff = crow['close']
    early_return = (
        close_cutoff/close_T-1.0
        if pd.notna(close_T) and close_T>0 and pd.notna(close_cutoff)
        else np.nan
    )
    ret_path = early['close']/close_T-1.0
    early_peak = float(ret_path.max()) if ret_path.notna().any() else np.nan
    early_giveback = (
        (early_peak-early_return)/early_peak
        if pd.notna(early_peak) and early_peak>0 and pd.notna(early_return)
        else np.nan
    )
    close_path = pd.to_numeric(early['close'],errors='coerce')
    early_mdd = (
        float((close_path/close_path.cummax()-1.0).min())
        if close_path.notna().any() else np.nan
    )

    prior20 = ev.get('volume_prior20_mean_T',np.nan)
    volume_sustain = (
        float(pd.to_numeric(post_t['volume'],errors='coerce').mean()/prior20)
        if len(post_t) and pd.notna(prior20) and prior20>0 else np.nan
    )

    row = {
        'event_id':ev['event_id'],'ticker':ticker,
        'ticker_name':ev.get('ticker_name',np.nan),
        'event_date':event_date,'event_tidx':event_tidx,
        'prediction_offset':7,'prediction_cutoff':'T+7',
        'cutoff_tidx':cutoff_tidx,'cutoff_date':cutoff_date,
        'early_return':early_return,
        'early_peak_return':early_peak,
        'early_giveback_ratio':early_giveback,
        'early_true_mdd':early_mdd,
        'volume_sustain':volume_sustain,
        'volume_change_vs_T':(
            float(crow['volume']/ev['volume_T']-1.0)
            if pd.notna(crow['volume']) and pd.notna(ev.get('volume_T',np.nan)) and ev['volume_T']>0
            else np.nan
        ),
    }

    # copy all fixed T features
    for c in event_master.columns:
        if c not in ['event_id','ticker','ticker_name','event_date','event_tidx']:
            row[c]=ev[c]

    # cumulative investor flow T+1..T+7
    for prefix,col in FLOW_COLS.items():
        cum_val = (
            float(pd.to_numeric(post_t[col],errors='coerce').sum(min_count=1))
            if len(post_t) else np.nan
        )
        row[f'{prefix}_net_buy_cum']=cum_val
        row[f'{prefix}_flow_ratio']=(
            cum_val/ev['market_cap_T']
            if pd.notna(cum_val) and pd.notna(ev.get('market_cap_T',np.nan)) and ev['market_cap_T']>0
            else np.nan
        )

    row['foreign_ownership_change']=(
        float(crow['foreign_ownership_pct']-ev['foreign_ownership_pct_T'])
        if pd.notna(crow['foreign_ownership_pct']) and pd.notna(ev.get('foreign_ownership_pct_T',np.nan))
        else np.nan
    )

    # attention T..cutoff (calendar days)
    agdf = attention_by_ticker.get(ticker)
    aw = (
        agdf[agdf['date'].between(event_date,cutoff_date,inclusive='both')].copy()
        if agdf is not None else pd.DataFrame()
    )
    for prefix in ['search','news']:
        col=f'{prefix}_pct_50'
        if len(aw) and col in aw.columns and aw[col].notna().any():
            valid=aw.sort_values('date')[col].dropna()
            row[f'{prefix}_mean_to_cutoff']=float(aw[col].mean())
            row[f'{prefix}_max_to_cutoff']=float(aw[col].max())
            row[f'{prefix}_last_at_cutoff']=float(valid.iloc[-1])
            row[f'{prefix}_high_share_to_cutoff']=float(aw[col].ge(ATT_THR).mean())
        else:
            row[f'{prefix}_mean_to_cutoff']=np.nan
            row[f'{prefix}_max_to_cutoff']=np.nan
            row[f'{prefix}_last_at_cutoff']=np.nan
            row[f'{prefix}_high_share_to_cutoff']=np.nan

        base=ev.get(f'{prefix}_pct_50_T',np.nan)
        last=row[f'{prefix}_last_at_cutoff']
        row[f'{prefix}_sustain_ratio']=(
            last/base if pd.notna(last) and pd.notna(base) and base!=0 else np.nan
        )

    row['search_news_level_gap_cutoff']=(
        aw.sort_values('date').iloc[-1].get('search_news_level_gap',np.nan)
        if len(aw) else np.nan
    )

    # market at cutoff
    if cutoff_date in market_lookup.index:
        mr=market_lookup.loc[cutoff_date]
        if isinstance(mr,pd.DataFrame): mr=mr.iloc[-1]
        for c in MARKET_COLS:
            if c in mr.index: row[f'{c}_cutoff']=mr[c]

    # short at cutoff
    try:
        sr=short_lookup.loc[(ticker,cutoff_date)]
        if isinstance(sr,pd.DataFrame): sr=sr.iloc[-1]
        for c in short.columns:
            if c not in ['ticker','date','ticker_name'] and c.startswith('short_'):
                row[f'{c}_cutoff']=sr.get(c,np.nan)
    except KeyError:
        pass

    # convenience aliases
    row['foreign_flow_ratio_T7']=row.get('foreign_flow_ratio',np.nan)
    row['institution_flow_ratio_T7']=row.get('institution_flow_ratio',np.nan)
    row['individual_flow_ratio_T7']=row.get('individual_flow_ratio',np.nan)
    row['early_giveback_T7']=row.get('early_giveback_ratio',np.nan)
    row['volume_sustain_T7']=row.get('volume_sustain',np.nan)

    snapshot7_rows.append(row)

snapshot7=pd.DataFrame(snapshot7_rows).sort_values(['event_date','ticker']).reset_index(drop=True)
print('T+7 snapshots:',snapshot7.shape)
print('T+7 event_n:',snapshot7['event_id'].nunique())
display(snapshot7[['event_id','cutoff_date','early_return','early_giveback_ratio','volume_sustain','foreign_flow_ratio','institution_flow_ratio']].head(20))

In [ ]:
# 7. Combine T/T+3/T+5/T+7/T+10
existing_snapshots = con.execute("""
SELECT * FROM event.snapshots_long
WHERE prediction_offset IN (0,3,5,10)
ORDER BY event_date, ticker, prediction_offset
""").df()

for c in ['event_date','cutoff_date']:
    existing_snapshots[c]=pd.to_datetime(existing_snapshots[c],errors='coerce').dt.normalize()
    snapshot7[c]=pd.to_datetime(snapshot7[c],errors='coerce').dt.normalize()

snapshots_timing=pd.concat([existing_snapshots,snapshot7],ignore_index=True,sort=False)
snapshots_timing['ticker']=snapshots_timing['ticker'].astype(str).str.zfill(6)
snapshots_timing=(
    snapshots_timing[snapshots_timing['prediction_offset'].isin(PRED_OFFSETS)]
    .sort_values(['event_date','ticker','prediction_offset'])
    .drop_duplicates(['event_id','prediction_offset'],keep='last')
    .reset_index(drop=True)
)

assert not snapshots_timing.duplicated(['event_id','prediction_offset']).any()
offset_qa=(snapshots_timing.groupby('prediction_offset').agg(
    rows=('event_id','size'),events=('event_id','nunique'),
    min_date=('event_date','min'),max_date=('event_date','max')
).reset_index())
display(offset_qa)
assert set(PRED_OFFSETS).issubset(set(offset_qa['prediction_offset'].astype(int)))
print('✅ Timing snapshots ready')

### PART C — 고정 Direct T+20 Target

`event.bhar_path_long`의 `prediction_offset=0, future_step=20`만 사용합니다. Prediction Timing과 무관하게 같은 사건은 같은 Target을 가집니다.

In [ ]:
# 8. Direct T+20 continuous target
direct_t20=con.execute("""
SELECT
    event_id,ticker,event_date,
    cutoff_date AS target_anchor_date,
    path_date AS direct_t20_date,
    bhar AS direct_bhar_T20
FROM event.bhar_path_long
WHERE prediction_offset=0 AND future_step=20
ORDER BY event_date,ticker
""").df()

for c in ['event_date','target_anchor_date','direct_t20_date']:
    direct_t20[c]=pd.to_datetime(direct_t20[c],errors='coerce').dt.normalize()
direct_t20['ticker']=direct_t20['ticker'].astype(str).str.zfill(6)

assert direct_t20['event_id'].is_unique
assert direct_t20['direct_bhar_T20'].notna().all()
assert (direct_t20['direct_t20_date']>direct_t20['event_date']).all()

print('direct_t20:',direct_t20.shape)
print('event_n:',direct_t20['event_id'].nunique())
print('period:',direct_t20['event_date'].min(),'~',direct_t20['event_date'].max())
display(direct_t20.head())

In [ ]:
# 9. Final model frame
model_frame=snapshots_timing.merge(
    direct_t20,on=['event_id','ticker','event_date'],
    how='inner',validate='many_to_one'
)
model_frame['target_start_date']=model_frame['direct_t20_date']
model_frame['target_end_date']=model_frame['direct_t20_date']
model_frame=model_frame[model_frame['cutoff_date']<model_frame['direct_t20_date']].copy()
model_frame=model_frame.sort_values(['event_date','ticker','prediction_offset']).reset_index(drop=True)

same_target=model_frame.groupby('event_id').agg(
    bhar_n=('direct_bhar_T20','nunique'),
    target_date_n=('direct_t20_date','nunique')
)
assert (same_target['bhar_n']==1).all()
assert (same_target['target_date_n']==1).all()

display(model_frame.groupby('prediction_offset').agg(
    rows=('event_id','size'),events=('event_id','nunique'),
    bhar_mean=('direct_bhar_T20','mean'),bhar_median=('direct_bhar_T20','median')
).reset_index())
print('model_frame:',model_frame.shape)
print('✅ same Direct T+20 target across timings')

### PART D — Feature Groups

`P_T_ONLY`를 통제군으로 둡니다.

- `P_T_ONLY @ T+7` = 사건일 T 정보만 사용
- `P @ T+7` = T 정보 + T~T+7 초기 가격·거래량 반응

둘의 차이가 **추가 7일을 관찰한 가치**입니다.

In [ ]:
# 10. Feature groups
P_T=['ret_1d_T','ret_5d_T','ret_10d_T','ret_20d_T','volatility_5d_T','volatility_20d_T','volatility_60d_T','volume_ratio_prior20_T','trading_value_ratio_prior20_T','turnover_T','high_low_range_T','up_day_share_10_T','max_daily_return_10_T']
P_POST=['early_return','early_peak_return','early_giveback_ratio','early_true_mdd','volume_sustain','volume_change_vs_T']
F_T=['foreign_net_buy_T','foreign_net_buy_5d_T','foreign_net_buy_10d_T','foreign_net_buy_to_mcap_T','foreign_ownership_pct_T','foreign_ownership_change_5d_T','foreign_ownership_change_10d_T']
F_POST=['foreign_net_buy_cum','foreign_flow_ratio','foreign_ownership_change']
I_T=['institution_net_buy_T','institution_net_buy_5d_T','institution_net_buy_10d_T','institution_net_buy_to_mcap_T']
I_POST=['institution_net_buy_cum','institution_flow_ratio']
IND_T=['individual_net_buy_T','individual_net_buy_5d_T','individual_net_buy_10d_T','individual_net_buy_to_mcap_T']
IND_POST=['individual_net_buy_cum','individual_flow_ratio']
ATT_T=['search_pct_50_T','news_pct_50_T','search_news_level_gap_T','search_lead_days','news_lead_days']
ATT_POST=['search_mean_to_cutoff','search_max_to_cutoff','search_last_at_cutoff','search_high_share_to_cutoff','search_sustain_ratio','news_mean_to_cutoff','news_max_to_cutoff','news_last_at_cutoff','news_high_share_to_cutoff','news_sustain_ratio','search_news_level_gap_cutoff']
MARKET=['market_drawdown_60d_cutoff','market_volatility_20d_cutoff','market_trading_value_change_5_20_cutoff','market_decline_ratio_cutoff','market_below_ma20_ratio_cutoff','market_crash_3pct_ratio_cutoff','market_illiquidity_shock_cutoff']
SHORT=sorted([c for c in model_frame.columns if c.startswith('short_') and c.endswith('_cutoff')])

def existing(cols):
    return [c for c in cols if c in model_frame.columns and pd.api.types.is_numeric_dtype(model_frame[c])]

GROUPS={
    'P_T':existing(P_T),'P_POST':existing(P_POST),
    'F_T':existing(F_T),'F_POST':existing(F_POST),
    'I_T':existing(I_T),'I_POST':existing(I_POST),
    'IND_T':existing(IND_T),'IND_POST':existing(IND_POST),
    'ATT_T':existing(ATT_T),'ATT_POST':existing(ATT_POST),
    'MARKET':existing(MARKET),'SHORT':existing(SHORT),
}
for k,v in GROUPS.items():
    print(f'{k:10s} n={len(v):2d}',v)

In [ ]:
# 11. Feature recipes
def uniq(x): return list(dict.fromkeys(x))

RECIPES={
    'P_T_ONLY':uniq(GROUPS['P_T']),
    'P':uniq(GROUPS['P_T']+GROUPS['P_POST']),
    'P_F':uniq(GROUPS['P_T']+GROUPS['P_POST']+GROUPS['F_T']+GROUPS['F_POST']),
    'P_I':uniq(GROUPS['P_T']+GROUPS['P_POST']+GROUPS['I_T']+GROUPS['I_POST']),
    'P_FI':uniq(GROUPS['P_T']+GROUPS['P_POST']+GROUPS['F_T']+GROUPS['F_POST']+GROUPS['I_T']+GROUPS['I_POST']),
    'P_ALLFLOW':uniq(GROUPS['P_T']+GROUPS['P_POST']+GROUPS['F_T']+GROUPS['F_POST']+GROUPS['I_T']+GROUPS['I_POST']+GROUPS['IND_T']+GROUPS['IND_POST']),
    'P_ATT':uniq(GROUPS['P_T']+GROUPS['P_POST']+GROUPS['ATT_T']+GROUPS['ATT_POST']),
    'P_FI_ATT':uniq(GROUPS['P_T']+GROUPS['P_POST']+GROUPS['F_T']+GROUPS['F_POST']+GROUPS['I_T']+GROUPS['I_POST']+GROUPS['ATT_T']+GROUPS['ATT_POST']),
    'P_FI_MARKET':uniq(GROUPS['P_T']+GROUPS['P_POST']+GROUPS['F_T']+GROUPS['F_POST']+GROUPS['I_T']+GROUPS['I_POST']+GROUPS['MARKET']),
    'P_FI_SHORT':uniq(GROUPS['P_T']+GROUPS['P_POST']+GROUPS['F_T']+GROUPS['F_POST']+GROUPS['I_T']+GROUPS['I_POST']+GROUPS['SHORT']),
    'FULL':uniq(GROUPS['P_T']+GROUPS['P_POST']+GROUPS['F_T']+GROUPS['F_POST']+GROUPS['I_T']+GROUPS['I_POST']+GROUPS['IND_T']+GROUPS['IND_POST']+GROUPS['ATT_T']+GROUPS['ATT_POST']+GROUPS['MARKET']+GROUPS['SHORT']),
}

recipe_summary=pd.DataFrame([{'recipe':k,'feature_n':len(v),'features':', '.join(v)} for k,v in RECIPES.items()])
display(recipe_summary)

for recipe,feats in RECIPES.items():
    bad=[c for c in feats if any(tok in c.lower() for tok in ['direct_bhar','target_','label','future_'])]
    assert not bad,(recipe,bad)
print('✅ feature leakage guard')

### [삽입] N=20, q=35% 최종 채택 근거 — 최종 라벨 확정 전 검증

원본 노트북: `Target_N20_q35_안정성및견고성검증.ipynb`

바로 아래 STEP 6의 PART E(공통 Train-only q35 적용)에서 N=20·q=35%를 **주어진 값으로 사용하기 전에**, 이 값들을 어떻게 선택했는지 그 과정을 보여줍니다.

1. **셀 1** — GCS `stock_daily.parquet` 로드 + 시가총액가중 시장지수 + 정적변수(P_T_ONLY, 13개) 준비
2. **셀 2** — N(관측기간) 후보 스윕: Timing=T(정적변수만)에서, q 6개 후보에 대한 **"평균"이 아니라 "평균/표준편차 비율(안정성 점수)"**로 N 선택
3. **셀 3** — q(컷오프) 후보 스윕: N 고정 후 T+3/T+5/T+7/T+10 전체에서 q를 스윕 → 최종 q(1위) vs 근접후보(2위)를 T+5에서 fold×seed paired test로 검정

In [ ]:
# -*- coding: utf-8 -*-
"""
N=20, q=35% 최종 채택 - 전체 과정 통합본
=================================================================
셀 1: GCS 데이터 로드 + 공통 함수/변수 준비
셀 2: N(관측기간) 선택
       - Timing=T(정적변수만)에서 N 후보 스윕
       - q 6개 후보에 대한 평균이 아니라 "평균/표준편차 비율"로 선택
         (평균이 근소하게 높아도 q에 따라 성능이 출렁이면 신뢰할 수
          없는 target 정의이므로, 안정적으로 꾸준한 N을 우선한다)
셀 3: q(컷오프) 선택
       - 셀 2에서 확정된 N을 고정
       - 신호가 실제로 존재하는 T+3/T+5/T+7/T+10 전체에서 q 스윕
       - 최종 q(1위) vs 근접후보(2위)를 T+5에서 fold x seed paired test로 검정
"""

# =================================================================
# 1. GCS 데이터 로드 + 공통 함수/변수 준비
# =================================================================
import pandas as pd
import numpy as np
import subprocess
import warnings
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from scipy import stats

warnings.filterwarnings('ignore')

# ---- GCS 인증 + 데이터 로드 ----
# (재사용) STAGE 3에서 이미 받아온 stock_daily를 그대로 사용합니다.
price_all = stock_daily.copy()
price_all = price_all.rename(columns={'ticker': '종목코드', 'date': '날짜'})
price_all['날짜'] = pd.to_datetime(price_all['날짜'])
price_all = price_all[price_all['종목코드'] != '001260'].copy()
print(f"price_all: {price_all.shape}, 종목수: {price_all['종목코드'].nunique()}")

# ---- 시가총액가중 시장지수 ----
price = price_all.sort_values(['종목코드', '날짜']).reset_index(drop=True)

price_wide_close = price.pivot(index='날짜', columns='종목코드', values='close')
ret_wide = price_wide_close.pct_change(fill_method=None)
mktcap_wide = price.pivot(index='날짜', columns='종목코드', values='market_cap').shift(1)
weights = mktcap_wide.div(mktcap_wide.sum(axis=1), axis=0)
market_daily_return = (ret_wide * weights).sum(axis=1)

market_index = pd.DataFrame({'날짜': market_daily_return.index})
market_index['시장일수익률'] = market_daily_return.values
market_index = market_index.sort_values('날짜').reset_index(drop=True)
market_index['시장지수'] = (1 + market_index['시장일수익률'].fillna(0)).cumprod() * 100
market_idx_series = market_index.set_index('날짜')['시장지수']

df = price.merge(market_index[['날짜', '시장일수익률', '시장지수']], on='날짜', how='left')
df['일수익률'] = df.groupby('종목코드')['close'].pct_change(fill_method=None)
print(f"시장지수 계산 완료: {market_index.shape}")

# ---- 정적변수 P_T_ONLY (13개) ----
g = df.groupby('종목코드', group_keys=False)

df['거래량_20일평균'] = g['volume'].apply(lambda x: x.rolling(20).mean())
df['거래량증가율_20일평균대비'] = df['volume'] / df['거래량_20일평균']
df['수익률_20일'] = g['close'].apply(lambda x: x.pct_change(20, fill_method=None))

VALUE_COL = 'trading_value' if 'trading_value' in df.columns else None
df['거래대금_근사'] = df[VALUE_COL] if VALUE_COL else df['close'] * df['volume']

df['ret_1d_T']  = df['일수익률']
df['ret_5d_T']  = g['close'].apply(lambda x: x.pct_change(5,  fill_method=None))
df['ret_10d_T'] = g['close'].apply(lambda x: x.pct_change(10, fill_method=None))
df['ret_20d_T'] = df['수익률_20일']

df['volatility_5d_T']  = g['일수익률'].apply(lambda x: x.rolling(5).std())
df['volatility_20d_T'] = g['일수익률'].apply(lambda x: x.rolling(20).std())
df['volatility_60d_T'] = g['일수익률'].apply(lambda x: x.rolling(60).std())

df['volume_ratio_prior20_T'] = df['거래량증가율_20일평균대비']
df['거래대금_20일평균'] = g['거래대금_근사'].apply(lambda x: x.rolling(20).mean())
df['trading_value_ratio_prior20_T'] = df['거래대금_근사'] / df['거래대금_20일평균']
df['turnover_T'] = df['거래대금_근사'] / df['market_cap']
df['high_low_range_T'] = (df['high'] - df['low']) / df['close']
df['up_day_share_10_T'] = g['일수익률'].apply(
    lambda x: x.rolling(10).apply(lambda y: (y > 0).mean()))
df['max_daily_return_10_T'] = g['일수익률'].apply(lambda x: x.rolling(10).max())

STATIC_FEATURES = [
    'ret_1d_T', 'ret_5d_T', 'ret_10d_T', 'ret_20d_T',
    'volatility_5d_T', 'volatility_20d_T', 'volatility_60d_T',
    'volume_ratio_prior20_T', 'trading_value_ratio_prior20_T', 'turnover_T',
    'high_low_range_T', 'up_day_share_10_T', 'max_daily_return_10_T',
]
print("정적변수(P_T_ONLY) 13개 생성 완료")

# ---- 급등 Event 탐지 (20거래일 cooldown) ----
df = df.sort_values(['종목코드', '날짜']).reset_index(drop=True)
g = df.groupby('종목코드', group_keys=False)

df['조건_수익률'] = df['수익률_20일'] >= 0.20
df['조건_거래량'] = df['거래량증가율_20일평균대비'] >= 1.5
df['급등조건충족'] = df['조건_수익률'] & df['조건_거래량']

def find_event_days_cooldown(group, cooldown=20):
    cond = group['급등조건충족'].values
    flag = np.zeros(len(cond), dtype=bool)
    prev = False
    last_kept = -np.inf
    for i in range(len(cond)):
        if cond[i] and not prev:
            if i - last_kept >= cooldown:
                flag[i] = True
                last_kept = i
        prev = cond[i]
    return flag

df['사건일여부'] = g.apply(
    lambda x: pd.Series(find_event_days_cooldown(x), index=x.index),
    include_groups=False
).values

events = df[df['사건일여부']].copy().reset_index(drop=True)
print(f"탐지된 사건 수: {len(events)}, 종목 수: {events['종목코드'].nunique()}")

ticker_groups = {t: gdf.reset_index(drop=True) for t, gdf in df.groupby('종목코드')}

def get_pos(code_, date):
    gdf = ticker_groups.get(code_)
    if gdf is None:
        return None, None
    arr = gdf.index[gdf['날짜'] == date]
    return gdf, (arr[0] if len(arr) > 0 else None)

# ---- Direct T+N BHAR ----
def compute_direct_bhar_N(row, N):
    code_, event_date = row['종목코드'], row['날짜']
    gdf, pos0 = get_pos(code_, event_date)
    if pos0 is None:
        return np.nan
    pos_N = pos0 + N
    if pos_N >= len(gdf):
        return np.nan
    close0 = gdf.loc[pos0, 'close']
    mkt0 = market_idx_series.get(event_date)
    if mkt0 is None:
        return np.nan
    mkt_N = market_idx_series.get(gdf.loc[pos_N, '날짜'])
    if mkt_N is None:
        return np.nan
    stock_ret = gdf.loc[pos_N, 'close'] / close0 - 1
    mkt_ret = mkt_N / mkt0 - 1
    return stock_ret - mkt_ret

# ---- 사건일 T~T+n 초기반응 동적변수 6개 ----
DYNAMIC_FEATURES = [
    'early_return', 'early_peak_return', 'early_giveback_ratio',
    'early_true_mdd', 'volume_sustain', 'volume_change_vs_T',
]

def compute_dynamic_features_cutoff(row, n):
    code_, event_date = row['종목코드'], row['날짜']
    gdf, pos0 = get_pos(code_, event_date)
    empty = pd.Series({f: np.nan for f in DYNAMIC_FEATURES})
    if pos0 is None:
        return empty
    pos_n = pos0 + n
    if pos_n >= len(gdf):
        return empty
    close0 = gdf.loc[pos0, 'close']
    mkt0 = market_idx_series.get(event_date)
    if mkt0 is None:
        return empty

    window = gdf.loc[pos0 + 1: pos_n].copy()
    if len(window) == 0:
        return empty
    window['stock_cum'] = window['close'] / close0 - 1
    window['mkt_cum'] = window['날짜'].map(market_idx_series) / mkt0 - 1
    window['bhar_path'] = window['stock_cum'] - window['mkt_cum']

    peak = window['bhar_path'].max()
    final = window['bhar_path'].iloc[-1]
    early_giveback_ratio = (peak - final) / peak if peak > 0.01 else np.nan
    running_max = window['bhar_path'].cummax()
    early_true_mdd = (window['bhar_path'] - running_max).min()

    vol20 = gdf.loc[pos0, '거래량_20일평균'] if '거래량_20일평균' in gdf.columns else np.nan
    volume_sustain = (window['volume'].mean() / vol20
                       if (vol20 is not None and not pd.isna(vol20) and vol20 > 0) else np.nan)
    vol_T = gdf.loc[pos0, 'volume']
    vol_n = gdf.loc[pos_n, 'volume']
    volume_change_vs_T = vol_n / vol_T if (vol_T is not None and vol_T > 0) else np.nan

    return pd.Series({
        'early_return': final,
        'early_peak_return': peak,
        'early_giveback_ratio': early_giveback_ratio,
        'early_true_mdd': early_true_mdd,
        'volume_sustain': volume_sustain,
        'volume_change_vs_T': volume_change_vs_T,
    })

# ---- Purged Expanding Walk-Forward Fold ----
def make_purged_folds(events_df, n_folds=5, horizon_days=20):
    d = events_df.sort_values('날짜').reset_index(drop=True)
    fold_bounds = np.linspace(0, len(d), n_folds + 2).astype(int)
    folds = []
    for i in range(1, n_folds + 1):
        train_idx_raw = d.index[:fold_bounds[i]]
        test_idx = d.index[fold_bounds[i]:fold_bounds[i + 1]]
        if len(test_idx) == 0:
            continue
        test_start_date = d.loc[test_idx[0], '날짜']
        purge_mask = []
        for idx in train_idx_raw:
            code_, ev_date = d.loc[idx, '종목코드'], d.loc[idx, '날짜']
            gdf, pos0 = get_pos(code_, ev_date)
            if pos0 is None:
                purge_mask.append(False)
                continue
            pos_end = pos0 + horizon_days
            if pos_end >= len(gdf):
                purge_mask.append(False)
                continue
            target_end_date = gdf.loc[pos_end, '날짜']
            purge_mask.append(target_end_date < test_start_date)
        train_idx = train_idx_raw[np.array(purge_mask)]
        folds.append((train_idx, test_idx))
    return folds

# ---- Fold별 Rolling Quantile 라벨 ----
def make_fold_labels(d, train_idx, test_idx, bhar_col, q):
    train_vals = d.loc[train_idx, bhar_col].dropna()
    if len(train_vals) < 10:
        return None
    threshold = train_vals.quantile(q)
    label = (d[bhar_col] <= threshold).astype(int)
    return label, threshold

# ---- 고정 RandomForest 평가 함수 ----
FIXED_RF_PARAMS = dict(
    n_estimators=300, max_depth=5, min_samples_leaf=10,
    class_weight='balanced'
)

def evaluate_N_q(events_df, N, q, features, n_folds=5, seed=42, need_bhar=True):
    bhar_col = f'bhar_N{N}'
    if need_bhar:
        events_df[bhar_col] = events_df.apply(lambda r: compute_direct_bhar_N(r, N), axis=1)
    d = events_df.dropna(subset=[bhar_col] + features).sort_values('날짜').reset_index(drop=True)
    if len(d) < 50:
        return None

    folds = make_purged_folds(d, n_folds=n_folds, horizon_days=N)

    aucs, f1s, precs, recalls = [], [], [], []
    for train_idx, test_idx in folds:
        if len(train_idx) < 20 or len(test_idx) < 5:
            continue
        result = make_fold_labels(d, train_idx, test_idx, bhar_col, q)
        if result is None:
            continue
        label, _ = result
        y_train, y_test = label.loc[train_idx], label.loc[test_idx]
        if y_train.nunique() < 2 or y_test.nunique() < 2:
            continue

        X = d[features].copy()
        for c in features:
            X[c] = X[c].replace([np.inf, -np.inf], np.nan)
            X[c] = X[c].fillna(X.loc[train_idx, c].median())
        X_train, X_test = X.loc[train_idx], X.loc[test_idx]

        model = RandomForestClassifier(random_state=seed, **FIXED_RF_PARAMS)
        model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)[:, 1]
        pred = model.predict(X_test)

        aucs.append(roc_auc_score(y_test, proba))
        f1s.append(f1_score(y_test, pred))
        precs.append(precision_score(y_test, pred, zero_division=0))
        recalls.append(recall_score(y_test, pred, zero_division=0))

    if len(aucs) == 0:
        return None
    return {
        'ROC_AUC': round(float(np.mean(aucs)), 4),
        'F1': round(float(np.mean(f1s)), 4),
        'Precision': round(float(np.mean(precs)), 4),
        'Recall': round(float(np.mean(recalls)), 4),
        '유효표본수': len(d),
    }

print("\n셀 1 완료 - 데이터/함수 준비 완료")

In [ ]:
# =================================================================
# 2. N(관측기간) 선택
# - Timing=T(정적변수만)에서 N 후보를 스윕
# - q 6개 후보에 대한 "평균"이 아니라 "평균/표준편차 비율(안정성 점수)"로 선택
#   -> 평균이 근소하게 높아도 q에 따라 출렁이는 N은 신뢰할 수 없는 target
#      정의이므로 배제하고, 어떤 q를 쓰든 꾸준한 N을 우선한다
# =================================================================
N_CANDIDATES = [10, 15, 20, 25, 30, 35, 40, 50, 60]
Q_CANDIDATES_FOR_N = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45]

print("=" * 60)
print("셀 2: N 스윕 (Timing=T, 정적변수만, 안정성 점수 = 평균/표준편차)")
print("=" * 60)

stage_n_records = []
for N in N_CANDIDATES:
    events_n = events.copy()
    events_n[f'bhar_N{N}'] = events_n.apply(lambda r: compute_direct_bhar_N(r, N), axis=1)
    for q in Q_CANDIDATES_FOR_N:
        res = evaluate_N_q(events_n, N=N, q=q, features=STATIC_FEATURES, need_bhar=False)
        if res is not None:
            stage_n_records.append({'N': N, 'q': q, 'ROC_AUC': res['ROC_AUC']})

stage_n_df = pd.DataFrame(stage_n_records)

n_summary = stage_n_df.groupby('N')['ROC_AUC'].agg(['mean', 'std']).reset_index()
n_summary.columns = ['N', '평균AUC', '표준편차']
n_summary['안정성점수(평균/표준편차)'] = n_summary['평균AUC'] / n_summary['표준편차']
n_summary_by_score = n_summary.sort_values('안정성점수(평균/표준편차)', ascending=False)
n_summary_by_mean = n_summary.sort_values('평균AUC', ascending=False)

print("\n--- N별 상세 (q 후보별 AUC) ---")
print(stage_n_df.pivot(index='N', columns='q', values='ROC_AUC').to_string())

print("\n--- 참고: 평균 AUC 기준 순위 (이 기준만으로는 채택하지 않음) ---")
print(n_summary_by_mean.to_string(index=False))

print("\n--- 채택 기준: 안정성 점수(평균/표준편차) 순위 ---")
print(n_summary_by_score.to_string(index=False))

N_BEST = int(n_summary_by_score.iloc[0]['N'])
n_best_mean_rank = int(n_summary_by_mean.reset_index(drop=True).index[n_summary_by_mean['N'] == N_BEST][0]) + 1
print(f"\n확정: N = {N_BEST}  "
      f"(안정성점수={n_summary_by_score.iloc[0]['안정성점수(평균/표준편차)']:.1f}, "
      f"평균AUC 기준으로는 {n_best_mean_rank}위였으나 표준편차가 가장 작아 채택)")

stage_n_df.to_csv('/content/step2_N_selection_full.csv', index=False)
n_summary_by_score.to_csv('/content/step2_N_selection_summary.csv', index=False)
print("저장 완료: step2_N_selection_full.csv, step2_N_selection_summary.csv")

In [ ]:
# =================================================================
# 3. q(컷오프) 선택 (N=N_BEST 고정)
# - 신호가 실제로 존재하는 T+3/T+5/T+7/T+10 전체에서 q를 스윕
#   (Timing=T는 신호가 거의 없는 환경이라 q 선택 근거에서 제외)
# - 최종 확정된 q(1위)와 근접후보(2위)를 T+5에서
#   fold x seed paired test로 통계검정
# =================================================================
TIMING_CANDIDATES = {'T+3': 3, 'T+5': 5, 'T+7': 7, 'T+10': 10}
Q_CANDIDATES = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45]

print("\n" + "=" * 60)
print(f"셀 3-1: q 스윕 (N={N_BEST} 고정, T+3/T+5/T+7/T+10)")
print("=" * 60)

robustness_records = []
events_by_timing = {}
for timing_label, cutoff_n in TIMING_CANDIDATES.items():
    events_t = events.copy()
    events_t[f'bhar_N{N_BEST}'] = events_t.apply(lambda r: compute_direct_bhar_N(r, N_BEST), axis=1)
    dyn = events_t.apply(lambda r: compute_dynamic_features_cutoff(r, cutoff_n), axis=1)
    events_t = pd.concat([events_t, dyn], axis=1)
    events_by_timing[timing_label] = events_t
    features = STATIC_FEATURES + DYNAMIC_FEATURES

    for q in Q_CANDIDATES:
        res = evaluate_N_q(events_t, N=N_BEST, q=q, features=features, need_bhar=False)
        auc = res['ROC_AUC'] if res is not None else np.nan
        robustness_records.append({'Timing': timing_label, 'q': q, 'AUC': auc})

robust_df = pd.DataFrame(robustness_records)
pivot_auc = robust_df.pivot(index='q', columns='Timing', values='AUC')
pivot_auc = pivot_auc[['T+3', 'T+5', 'T+7', 'T+10']]
pivot_rank = pivot_auc.rank(ascending=False, axis=0)
pivot_rank['평균순위'] = pivot_rank.mean(axis=1)

print("\n--- AUC 표 ---")
print(pivot_auc.to_string())
print("\n--- 순위 표 (1=최고) ---")
print(pivot_rank.to_string())

q_best = float(pivot_rank['평균순위'].idxmin())
win_count = int((pivot_rank[['T+3', 'T+5', 'T+7', 'T+10']].loc[q_best] == 1).sum())
print(f"\n확정: q = {q_best:.0%}  "
      f"(4개 구간 중 {win_count}개에서 1위, 평균순위 {pivot_rank.loc[q_best,'평균순위']:.2f})")

pivot_auc.to_csv('/content/step3_q_selection_auc.csv')
pivot_rank.to_csv('/content/step3_q_selection_rank.csv')

# ---- 셀 3-2: 최종 확정 q vs 근접후보(2위) 통계검정 (T+5 기준) ----
print("\n" + "=" * 60)
q_2nd = float(pivot_auc.loc[:, 'T+5'].drop(q_best).idxmax())
print(f"셀 3-2: T+5에서 q={q_best:.0%}(1위) vs q={q_2nd:.0%}(2위) 통계검정")
print("=" * 60)

def paired_seed_comparison(events_df, N, features, q_a, q_b, n_folds=5, n_seeds=10):
    """q_a: 비교기준(2위), q_b: 검정대상(1위). diffs는 (q_b - q_a) 방향."""
    bhar_col = f'bhar_N{N}'
    d = events_df.dropna(subset=[bhar_col] + features).sort_values('날짜').reset_index(drop=True)
    folds = make_purged_folds(d, n_folds=n_folds, horizon_days=N)

    pair_a, pair_b = [], []
    for seed in range(n_seeds):
        for train_idx, test_idx in folds:
            if len(train_idx) < 20 or len(test_idx) < 5:
                continue
            res_a = make_fold_labels(d, train_idx, test_idx, bhar_col, q_a)
            res_b = make_fold_labels(d, train_idx, test_idx, bhar_col, q_b)
            if res_a is None or res_b is None:
                continue
            label_a, _ = res_a
            label_b, _ = res_b
            if (label_a.loc[train_idx].nunique() < 2 or label_a.loc[test_idx].nunique() < 2 or
                    label_b.loc[train_idx].nunique() < 2 or label_b.loc[test_idx].nunique() < 2):
                continue
            X = d[features].copy()
            for c in features:
                X[c] = X[c].replace([np.inf, -np.inf], np.nan)
                X[c] = X[c].fillna(X.loc[train_idx, c].median())
            X_train, X_test = X.loc[train_idx], X.loc[test_idx]

            model_a = RandomForestClassifier(random_state=seed, **FIXED_RF_PARAMS)
            model_a.fit(X_train, label_a.loc[train_idx])
            auc_a = roc_auc_score(label_a.loc[test_idx], model_a.predict_proba(X_test)[:, 1])

            model_b = RandomForestClassifier(random_state=seed, **FIXED_RF_PARAMS)
            model_b.fit(X_train, label_b.loc[train_idx])
            auc_b = roc_auc_score(label_b.loc[test_idx], model_b.predict_proba(X_test)[:, 1])

            pair_a.append(auc_a)
            pair_b.append(auc_b)

    pair_a, pair_b = np.array(pair_a), np.array(pair_b)
    diffs = pair_b - pair_a
    t_stat, p_ttest = stats.ttest_rel(pair_b, pair_a)
    try:
        w_stat, p_wilcoxon = stats.wilcoxon(pair_b, pair_a)
    except ValueError:
        w_stat, p_wilcoxon = np.nan, np.nan
    cohens_d = diffs.mean() / diffs.std(ddof=1) if diffs.std(ddof=1) > 0 else np.nan

    print(f"\n매칭쌍 개수: {len(pair_a)}개 (기대값: {n_folds}fold x {n_seeds}seed = {n_folds*n_seeds}개)")
    print(f"q={q_a:.0%} 평균 AUC: {pair_a.mean():.4f} (표준편차 {pair_a.std():.4f})")
    print(f"q={q_b:.0%} 평균 AUC: {pair_b.mean():.4f} (표준편차 {pair_b.std():.4f})")
    print(f"평균 차이({q_b:.0%}-{q_a:.0%}): {diffs.mean():.4f}")
    print(f"[1] Paired t-test: t={t_stat:.3f}, p={p_ttest:.4f} "
          f"{'-> 유의함' if p_ttest < 0.05 else '-> 유의하지 않음'}")
    print(f"[2] Wilcoxon: W={w_stat}, p={p_wilcoxon} "
          f"{'-> 유의함' if (not np.isnan(p_wilcoxon) and p_wilcoxon < 0.05) else '-> 유의하지 않음/계산불가'}")
    print(f"[3] Cohen's d: {cohens_d:.3f}")
    return {'p_ttest': p_ttest, 'p_wilcoxon': p_wilcoxon, 'cohens_d': cohens_d}

FEATURES_T5 = STATIC_FEATURES + DYNAMIC_FEATURES
sig_result = paired_seed_comparison(
    events_by_timing['T+5'], N=N_BEST, features=FEATURES_T5,
    q_a=q_2nd, q_b=q_best,
)

# ---- 최종 요약 ----
print("\n" + "=" * 60)
print("최종 요약")
print("=" * 60)
print(f"N = {N_BEST}  (Timing=T 스윕, 평균이 아닌 안정성점수(평균/표준편차) 기준 채택)")
print(f"q = {q_best:.0%}  (T+3/T+5/T+7/T+10 4개 구간 중 {win_count}개에서 1위)")
print(f"T+5에서 q={q_best:.0%} vs q={q_2nd:.0%}: p={sig_result['p_ttest']:.4f} "
      f"({'통계적으로 유의함' if sig_result['p_ttest'] < 0.05 else '유의하지 않음(단, 2위 후보가 1위를 반박하지도 않음)'})")

### STEP 6 계속 — PART E부터 (공통 Train-only q35 적용 및 최종 라벨/모델링)

위에서 확정된 N=20, q=35%를 그대로 사용해 원본 STEP 6 노트북의 나머지 부분(PART E~N)을 진행합니다.

### PART E — 공통 Train-only q35 + 공통 Walk-Forward 경계

Timing을 공정하게 비교하기 위해 **라벨 경계(q35)는 Feature/Model/Prediction Offset과 무관하게 같은 Train 기간의 Event Target만으로 계산**합니다.

즉 같은 Fold에서는 T/T+3/T+5/T+7/T+10 모두 같은 q35를 사용합니다.

In [ ]:
# 12. Common q35 by training period
TARGET_BASE=direct_t20[['event_id','event_date','direct_t20_date','direct_bhar_T20']].copy()

def q35_before(cut_date):
    hist=TARGET_BASE[
        (TARGET_BASE['event_date']<cut_date)
        & (TARGET_BASE['direct_t20_date']<cut_date)
    ]
    assert len(hist)>0
    return float(hist['direct_bhar_T20'].quantile(LABEL_Q))

def apply_fixed_label(train_df,eval_df,q35):
    tr=train_df.copy(); ev=eval_df.copy()
    tr['y']=(pd.to_numeric(tr['direct_bhar_T20'],errors='coerce')<=q35).astype(int)
    ev['y']=(pd.to_numeric(ev['direct_bhar_T20'],errors='coerce')<=q35).astype(int)
    return tr,ev

In [ ]:
# 13. Common development fold boundaries from target events
DEV_TARGET=TARGET_BASE[TARGET_BASE['event_date']<=DEV_END].copy()
dates=np.array(sorted(DEV_TARGET['event_date'].dropna().unique()))
initial_n=max(5,int(len(dates)*INITIAL_TRAIN_FRAC))
blocks=np.array_split(dates[initial_n:],N_WF_SPLITS)

COMMON_FOLDS=[]
for i,val_dates in enumerate(blocks,1):
    if len(val_dates)==0: continue
    val_start=pd.Timestamp(val_dates[0]); val_end=pd.Timestamp(val_dates[-1])
    q35=q35_before(val_start)
    COMMON_FOLDS.append({'fold':i,'val_start':val_start,'val_end':val_end,'q35':q35})

fold_boundaries=pd.DataFrame(COMMON_FOLDS)
fold_boundaries['q35_pct']=fold_boundaries['q35']*100
display(fold_boundaries)
print('✅ common fold boundaries/q35')

In [ ]:
# 14. Purge QA for each timing
qa_rows=[]
for offset in PRED_OFFSETS:
    x=model_frame[(model_frame['prediction_offset']==offset)&(model_frame['event_date']<=DEV_END)].copy()
    for f in COMMON_FOLDS:
        tr=x[(x['event_date']<f['val_start'])&(x['target_end_date']<f['val_start'])]
        va=x[x['event_date'].between(f['val_start'],f['val_end'],inclusive='both')]
        tr_l,va_l=apply_fixed_label(tr,va,f['q35'])
        qa_rows.append({
            'offset':offset,'fold':f['fold'],'train_n':len(tr),'val_n':len(va),
            'q35':f['q35'],'q35_pct':f['q35']*100,
            'train_positive_rate':tr_l['y'].mean() if len(tr_l) else np.nan,
            'val_positive_rate':va_l['y'].mean() if len(va_l) else np.nan,
            'purge_ok':bool(len(tr)==0 or tr['target_end_date'].max()<f['val_start'])
        })
fold_qa=pd.DataFrame(qa_rows)
display(fold_qa)
assert fold_qa['purge_ok'].all()
print('✅ purge QA')

### PART F — 모델/평가 함수

In [ ]:
# 15. Model factories
def make_model(name,seed):
    if name=='rf':
        return RandomForestClassifier(n_estimators=300,max_depth=5,min_samples_leaf=10,class_weight='balanced',random_state=seed,n_jobs=-1)
    if name=='lgbm':
        return LGBMClassifier(n_estimators=250,learning_rate=0.03,max_depth=4,num_leaves=15,min_child_samples=20,subsample=0.85,colsample_bytree=0.85,reg_lambda=1.0,class_weight='balanced',random_state=seed,verbosity=-1,n_jobs=-1)
    if name=='logit':
        return LogisticRegression(C=1.0,penalty='l2',solver='lbfgs',max_iter=3000,class_weight='balanced',random_state=seed)
    raise ValueError(name)

In [ ]:
# 16. Fit / predict
def fit_predict(train_df,eval_df,features,model_name,seed):
    feats=[c for c in features if c in train_df.columns and pd.api.types.is_numeric_dtype(train_df[c]) and train_df[c].notna().any()]
    if not feats: raise ValueError('No usable features')

    Xtr=train_df[feats].replace([np.inf,-np.inf],np.nan)
    Xev=eval_df[feats].replace([np.inf,-np.inf],np.nan)
    ytr=train_df['y'].astype(int).to_numpy()

    imputer=SimpleImputer(strategy='median')
    Xtr2=imputer.fit_transform(Xtr); Xev2=imputer.transform(Xev)
    scaler=None
    if model_name=='logit':
        scaler=StandardScaler(); Xtr2=scaler.fit_transform(Xtr2); Xev2=scaler.transform(Xev2)

    Xfit,yfit=Xtr2,ytr
    if USE_SMOTE:
        counts=pd.Series(ytr).value_counts()
        if len(counts)==2 and int(counts.min())>=2:
            k=min(5,int(counts.min())-1)
            Xfit,yfit=SMOTE(random_state=seed,k_neighbors=k).fit_resample(Xtr2,ytr)

    model=make_model(model_name,seed)
    model.fit(Xfit,yfit)
    prob=model.predict_proba(Xev2)[:,1]
    return {'prob':prob,'features':feats,'model':model,'imputer':imputer,'scaler':scaler}

In [ ]:
# 17. Metrics
def ece_score(y,prob,n_bins=10):
    y=np.asarray(y); prob=np.asarray(prob)
    edges=np.linspace(0,1,n_bins+1); ids=np.digitize(prob,edges[1:-1],right=True)
    ece=0.0
    for b in range(n_bins):
        m=ids==b
        if m.sum()==0: continue
        ece+=m.mean()*abs(y[m].mean()-prob[m].mean())
    return float(ece)

def metric_dict(y,prob,prob_threshold=0.5):
    y=np.asarray(y).astype(int); prob=np.asarray(prob).astype(float)
    pred=(prob>=prob_threshold).astype(int)
    out={'n':len(y),'positive_n':int(y.sum()),'positive_rate':float(y.mean()),'roc_auc':np.nan,'pr_auc':np.nan,
         'precision':precision_score(y,pred,zero_division=0),'recall':recall_score(y,pred,zero_division=0),
         'f1':f1_score(y,pred,zero_division=0),'brier':brier_score_loss(y,prob),'ece10':ece_score(y,prob,10)}
    if len(np.unique(y))==2:
        out['roc_auc']=roc_auc_score(y,prob); out['pr_auc']=average_precision_score(y,prob)
    return out

In [ ]:
# 18. Evaluate one configuration
def evaluate_config(base_df,offset,recipe,model_name,seed):
    x=base_df[base_df['prediction_offset'].eq(offset)].sort_values(['event_date','ticker']).reset_index(drop=True).copy()
    features=RECIPES[recipe]
    result_rows=[]; pred_rows=[]

    # Development Walk-Forward
    dev=x[x['event_date']<=DEV_END].copy()
    for f in COMMON_FOLDS:
        tr=dev[(dev['event_date']<f['val_start'])&(dev['target_end_date']<f['val_start'])].copy()
        va=dev[dev['event_date'].between(f['val_start'],f['val_end'],inclusive='both')].copy()
        if len(tr)<30 or len(va)<5: continue
        tr,va=apply_fixed_label(tr,va,f['q35'])
        if tr['y'].nunique()<2 or va['y'].nunique()<2: continue
        fit=fit_predict(tr,va,features,model_name,seed)
        result_rows.append({'prediction_offset':offset,'recipe':recipe,'model':model_name,'seed':seed,'split':'DEV_WF','fold':f['fold'],'label_q35':f['q35'],'feature_n':len(fit['features']),**metric_dict(va['y'],fit['prob'])})
        for eid,ed,y,p in zip(va['event_id'],va['event_date'],va['y'],fit['prob']):
            pred_rows.append({'prediction_offset':offset,'recipe':recipe,'model':model_name,'seed':seed,'split':'DEV_OOF','fold':f['fold'],'event_id':eid,'event_date':ed,'y':int(y),'prob':float(p),'label_q35':f['q35']})

    # 2025 temporal
    q25=q35_before(TEST25_START)
    tr=x[(x['event_date']<TEST25_START)&(x['target_end_date']<TEST25_START)].copy()
    te=x[x['event_date'].between(TEST25_START,TEST25_END,inclusive='both')].copy()
    if len(tr) and len(te):
        tr,te=apply_fixed_label(tr,te,q25)
        if tr['y'].nunique()==2 and te['y'].nunique()==2:
            fit=fit_predict(tr,te,features,model_name,seed)
            result_rows.append({'prediction_offset':offset,'recipe':recipe,'model':model_name,'seed':seed,'split':'TEST_2025','fold':0,'label_q35':q25,'feature_n':len(fit['features']),**metric_dict(te['y'],fit['prob'])})
            for eid,ed,y,p in zip(te['event_id'],te['event_date'],te['y'],fit['prob']):
                pred_rows.append({'prediction_offset':offset,'recipe':recipe,'model':model_name,'seed':seed,'split':'TEST_2025','fold':0,'event_id':eid,'event_date':ed,'y':int(y),'prob':float(p),'label_q35':q25})

    # 2026 stress
    q26=q35_before(TEST26_START)
    tr=x[(x['event_date']<TEST26_START)&(x['target_end_date']<TEST26_START)].copy()
    te=x[x['event_date'].between(TEST26_START,TEST26_END,inclusive='both')].copy()
    if len(tr) and len(te):
        tr,te=apply_fixed_label(tr,te,q26)
        if tr['y'].nunique()==2 and te['y'].nunique()==2:
            fit=fit_predict(tr,te,features,model_name,seed)
            result_rows.append({'prediction_offset':offset,'recipe':recipe,'model':model_name,'seed':seed,'split':'STRESS_2026','fold':0,'label_q35':q26,'feature_n':len(fit['features']),**metric_dict(te['y'],fit['prob'])})
            for eid,ed,y,p in zip(te['event_id'],te['event_date'],te['y'],fit['prob']):
                pred_rows.append({'prediction_offset':offset,'recipe':recipe,'model':model_name,'seed':seed,'split':'STRESS_2026','fold':0,'event_id':eid,'event_date':ed,'y':int(y),'prob':float(p),'label_q35':q26})

    return pd.DataFrame(result_rows),pd.DataFrame(pred_rows)

### PART G — 전체 Grid 실행

5 Timing × 11 Feature Recipes × 3 Models = **165 configurations**를 먼저 seed=42로 비교합니다.

In [ ]:
# 19. Full grid
GRID_RECIPES=['P_T_ONLY','P','P_F','P_I','P_FI','P_ALLFLOW','P_ATT','P_FI_ATT','P_FI_MARKET','P_FI_SHORT','FULL']
all_results=[]; all_predictions=[]
total=len(PRED_OFFSETS)*len(GRID_RECIPES)*len(MODELS); counter=0; started=time.time()

for offset in PRED_OFFSETS:
    for recipe in GRID_RECIPES:
        for model_name in MODELS:
            counter+=1
            if counter==1 or counter%10==0:
                print(f'[{counter}/{total}] T+{offset} | {recipe} | {model_name} | {(time.time()-started)/60:.1f} min')
            res,pred=evaluate_config(model_frame,offset,recipe,model_name,GRID_SEED)
            if len(res): all_results.append(res)
            if len(pred): all_predictions.append(pred)

results=pd.concat(all_results,ignore_index=True)
predictions=pd.concat(all_predictions,ignore_index=True)
print('results:',results.shape,'predictions:',predictions.shape,'elapsed min:',round((time.time()-started)/60,2))

### PART H — Development / 2025 / 2026 결과 요약

In [ ]:
# 20. Development fold mean / temporal-stress summary
dev_summary=(results[results['split'].eq('DEV_WF')].groupby(['prediction_offset','recipe','model'],as_index=False).agg(
    dev_auc_mean=('roc_auc','mean'),dev_auc_sd=('roc_auc','std'),dev_pr_auc_mean=('pr_auc','mean'),
    dev_recall_mean=('recall','mean'),dev_precision_mean=('precision','mean'),dev_f1_mean=('f1','mean'),fold_n=('fold','nunique')
))

test_summary=results[results['split'].isin(['TEST_2025','STRESS_2026'])][[
    'prediction_offset','recipe','model','split','roc_auc','pr_auc','precision','recall','f1','brier','ece10','positive_rate','label_q35'
]].copy()

display(dev_summary.sort_values('dev_auc_mean',ascending=False).head(50))

In [ ]:
# 21. Base P timing table
timing_rows=[]
for model_name in MODELS:
    for off in PRED_OFFSETS:
        d=dev_summary[(dev_summary['model']==model_name)&(dev_summary['recipe']=='P')&(dev_summary['prediction_offset']==off)]
        row={'model':model_name,'prediction_offset':off,'DEV_WF_AUC':float(d['dev_auc_mean'].iloc[0]) if len(d) else np.nan,'DEV_SD':float(d['dev_auc_sd'].iloc[0]) if len(d) else np.nan}
        for split,label in [('TEST_2025','TEST25_AUC'),('STRESS_2026','STRESS26_AUC')]:
            s=test_summary[(test_summary['model']==model_name)&(test_summary['recipe']=='P')&(test_summary['prediction_offset']==off)&(test_summary['split']==split)]
            row[label]=float(s['roc_auc'].iloc[0]) if len(s) else np.nan
        timing_rows.append(row)
p_timing=pd.DataFrame(timing_rows)
display(p_timing)

In [ ]:
# 22. Timing plots — P
for model_name in MODELS:
    s=p_timing[p_timing['model']==model_name].sort_values('prediction_offset')
    fig,ax=plt.subplots(figsize=(9,5))
    ax.plot(s['prediction_offset'],s['DEV_WF_AUC'],marker='o',label='DEV_WF mean')
    ax.plot(s['prediction_offset'],s['TEST25_AUC'],marker='o',label='TEST_2025')
    ax.plot(s['prediction_offset'],s['STRESS26_AUC'],marker='o',label='STRESS_2026')
    ax.axhline(0.5,linewidth=1)
    ax.set_xticks(PRED_OFFSETS); ax.set_xlabel('Prediction timing'); ax.set_ylabel('ROC-AUC')
    ax.set_title(f'Direct T+20 Q35 — P Timing / {model_name}')
    ax.legend(); plt.show()

### PART I — T+5 vs T+7 핵심 비교

T+7 추가 목적은 **T+5 이후에도 추가 정보 이득이 있는지** 확인하는 것입니다.

In [ ]:
# 23. T+5 vs T+7 table
timing57=[]
for recipe in ['P','P_F','P_I','P_FI','P_ALLFLOW','P_ATT','P_FI_MARKET','P_FI_SHORT']:
    for model_name in MODELS:
        row={'recipe':recipe,'model':model_name}
        for off in [5,7]:
            d=dev_summary[(dev_summary['prediction_offset']==off)&(dev_summary['recipe']==recipe)&(dev_summary['model']==model_name)]
            row[f'DEV_T{off}']=float(d['dev_auc_mean'].iloc[0]) if len(d) else np.nan
            for split,label in [('TEST_2025','TEST25'),('STRESS_2026','STRESS26')]:
                s=test_summary[(test_summary['prediction_offset']==off)&(test_summary['recipe']==recipe)&(test_summary['model']==model_name)&(test_summary['split']==split)]
                row[f'{label}_T{off}']=float(s['roc_auc'].iloc[0]) if len(s) else np.nan
        row['DEV_delta_T7_T5']=row['DEV_T7']-row['DEV_T5']
        row['TEST25_delta_T7_T5']=row['TEST25_T7']-row['TEST25_T5']
        row['STRESS26_delta_T7_T5']=row['STRESS26_T7']-row['STRESS26_T5']
        timing57.append(row)
timing57=pd.DataFrame(timing57)
display(timing57.sort_values('DEV_delta_T7_T5',ascending=False))

In [ ]:
# 24. Feature ablation at T+5 / T+7
ablation=[]
for off in [5,7]:
    for model_name in MODELS:
        base=dev_summary[(dev_summary['prediction_offset']==off)&(dev_summary['recipe']=='P')&(dev_summary['model']==model_name)]
        base_auc=float(base['dev_auc_mean'].iloc[0]) if len(base) else np.nan
        for recipe in GRID_RECIPES:
            d=dev_summary[(dev_summary['prediction_offset']==off)&(dev_summary['recipe']==recipe)&(dev_summary['model']==model_name)]
            if len(d)==0: continue
            row={'prediction_offset':off,'model':model_name,'recipe':recipe,'dev_auc':float(d['dev_auc_mean'].iloc[0]),'delta_vs_P':float(d['dev_auc_mean'].iloc[0])-base_auc}
            for split,label in [('TEST_2025','test25_auc'),('STRESS_2026','stress26_auc')]:
                s=test_summary[(test_summary['prediction_offset']==off)&(test_summary['recipe']==recipe)&(test_summary['model']==model_name)&(test_summary['split']==split)]
                row[label]=float(s['roc_auc'].iloc[0]) if len(s) else np.nan
            ablation.append(row)
ablation=pd.DataFrame(ablation)
display(ablation.sort_values(['prediction_offset','model','dev_auc'],ascending=[True,True,False]))

### PART J — Paired Bootstrap: T+7 vs T+5

동일 OOT Event의 T+5/T+7 Prediction을 직접 맞춰 AUC 차이를 bootstrap합니다.

In [ ]:
# 25. Paired bootstrap
def paired_bootstrap_timing(pred_df,recipe,model_name,split,off_a=5,off_b=7,seed=GRID_SEED,n_boot=2000,random_state=123):
    a=pred_df[(pred_df['recipe']==recipe)&(pred_df['model']==model_name)&(pred_df['split']==split)&(pred_df['prediction_offset']==off_a)&(pred_df['seed']==seed)][['event_id','y','prob']].rename(columns={'prob':'prob_a'})
    b=pred_df[(pred_df['recipe']==recipe)&(pred_df['model']==model_name)&(pred_df['split']==split)&(pred_df['prediction_offset']==off_b)&(pred_df['seed']==seed)][['event_id','y','prob']].rename(columns={'prob':'prob_b'})
    z=a.merge(b,on=['event_id','y'],how='inner',validate='one_to_one')
    if len(z)<20 or z['y'].nunique()<2: return None
    auc_a=roc_auc_score(z['y'],z['prob_a']); auc_b=roc_auc_score(z['y'],z['prob_b']); obs=auc_b-auc_a
    rng=np.random.default_rng(random_state); idx=np.arange(len(z)); diffs=[]
    for _ in range(n_boot):
        zz=z.iloc[rng.choice(idx,len(idx),replace=True)]
        if zz['y'].nunique()<2: continue
        diffs.append(roc_auc_score(zz['y'],zz['prob_b'])-roc_auc_score(zz['y'],zz['prob_a']))
    diffs=np.asarray(diffs); lo,hi=np.quantile(diffs,[0.025,0.975]); p=2*min(np.mean(diffs<=0),np.mean(diffs>=0))
    return {'recipe':recipe,'model':model_name,'split':split,'n':len(z),'auc_T5':auc_a,'auc_T7':auc_b,'delta_T7_minus_T5':obs,'ci95_low':lo,'ci95_high':hi,'bootstrap_p':p}

boot=[]
for recipe in ['P','P_FI']:
    for model_name in MODELS:
        for split in ['TEST_2025','STRESS_2026']:
            out=paired_bootstrap_timing(predictions,recipe,model_name,split)
            if out is not None: boot.append(out)
bootstrap_timing=pd.DataFrame(boot)
display(bootstrap_timing)

### PART K — Development 상위 후보 10-seed 안정성

최종 후보는 **Development Walk-Forward 성능으로만** 먼저 선정합니다. 2025/2026을 보고 후보를 고르지 않습니다.

In [ ]:
# 26. Select top 8 configs from Development only
top_cfg=dev_summary.sort_values('dev_auc_mean',ascending=False).head(8)[['prediction_offset','recipe','model']].drop_duplicates().reset_index(drop=True)
display(top_cfg)

In [ ]:
# 27. 10-seed stability
stability_rows=[]; started=time.time()
for i,cfgrow in top_cfg.iterrows():
    print(f"[{i+1}/{len(top_cfg)}] T+{cfgrow['prediction_offset']} | {cfgrow['recipe']} | {cfgrow['model']}")
    for seed in STABILITY_SEEDS:
        res,_=evaluate_config(model_frame,int(cfgrow['prediction_offset']),cfgrow['recipe'],cfgrow['model'],seed)
        if len(res): stability_rows.append(res)
stability=pd.concat(stability_rows,ignore_index=True)
stability_summary=(stability.groupby(['prediction_offset','recipe','model','split'],as_index=False).agg(
    roc_auc_mean=('roc_auc','mean'),roc_auc_sd=('roc_auc','std'),pr_auc_mean=('pr_auc','mean'),
    recall_mean=('recall','mean'),precision_mean=('precision','mean'),f1_mean=('f1','mean'),q35_mean=('label_q35','mean')
))
display(stability_summary.sort_values(['split','roc_auc_mean'],ascending=[True,False]))
print('elapsed min:',round((time.time()-started)/60,2))

### PART L — Label Threshold QA

In [ ]:
# 28. q35 must be independent of timing/feature/model in same evaluation period
q35_check=(results.groupby(['split','fold'])['label_q35'].agg(['min','max','nunique']).reset_index())
display(q35_check)
assert (q35_check['nunique']==1).all()
print('✅ common q35 across timing/recipe/model')

### PART M — 저장 / DuckDB / GCS

In [ ]:
# 29. Save files
SNAPSHOT_PATH=DERIVED_DIR/'event_snapshots_timing_long.parquet'
TARGET_PATH=DERIVED_DIR/'direct_t20_target.parquet'
FRAME_PATH=DERIVED_DIR/'direct_t20_model_frame_timing.parquet'

snapshots_timing.to_parquet(SNAPSHOT_PATH,index=False,compression='zstd')
direct_t20.to_parquet(TARGET_PATH,index=False,compression='zstd')
model_frame.to_parquet(FRAME_PATH,index=False,compression='zstd')

RESULT_PATH=OUT_DIR/'model_results_fold.csv'
PRED_PATH=OUT_DIR/'model_predictions.parquet'
DEV_PATH=OUT_DIR/'development_summary.csv'
TEST_PATH=OUT_DIR/'temporal_stress_summary.csv'
TIMING57_PATH=OUT_DIR/'timing_T5_vs_T7.csv'
ABLATION_PATH=OUT_DIR/'feature_ablation_T5_T7.csv'
BOOT_PATH=OUT_DIR/'paired_bootstrap_T5_T7.csv'
STABILITY_PATH=OUT_DIR/'stability_10seed.csv'

results.to_csv(RESULT_PATH,index=False,encoding='utf-8-sig')
predictions.to_parquet(PRED_PATH,index=False,compression='zstd')
dev_summary.to_csv(DEV_PATH,index=False,encoding='utf-8-sig')
test_summary.to_csv(TEST_PATH,index=False,encoding='utf-8-sig')
timing57.to_csv(TIMING57_PATH,index=False,encoding='utf-8-sig')
ablation.to_csv(ABLATION_PATH,index=False,encoding='utf-8-sig')
bootstrap_timing.to_csv(BOOT_PATH,index=False,encoding='utf-8-sig')
stability_summary.to_csv(STABILITY_PATH,index=False,encoding='utf-8-sig')
print('✅ Drive files saved')

In [ ]:
# 30. Materialize reusable tables in DuckDB
def replace_table(name,df):
    if df is None or len(df)==0: return
    temp='_tmp_'+name.replace('.','_')
    con.register(temp,df)
    con.execute(f'CREATE OR REPLACE TABLE {name} AS SELECT * FROM {temp}')
    con.unregister(temp)

replace_table('event.snapshots_timing_long',snapshots_timing)
replace_table('event.direct_t20_target',direct_t20)
replace_table('event.direct_t20_model_frame',model_frame)
replace_table('meta.step6_direct_t20_results',results)
replace_table('meta.step6_direct_t20_dev_summary',dev_summary)
replace_table('meta.step6_direct_t20_test_summary',test_summary)
replace_table('meta.step6_direct_t20_timing57',timing57)
replace_table('meta.step6_direct_t20_stability',stability_summary)
replace_table('meta.direct_t20_label_policy',pd.DataFrame([{
    'target':'direct_bhar_T20','target_horizon':20,'label_rule':'direct_bhar_T20 <= common train-period q35',
    'label_quantile':0.35,'prediction_offsets':'0,3,5,7,10','global_label_stored':False
}]))
con.execute('CHECKPOINT')
display(con.execute("""
SELECT
 (SELECT COUNT(*) FROM event.snapshots_timing_long) AS snapshot_rows,
 (SELECT COUNT(*) FROM event.direct_t20_target) AS target_rows,
 (SELECT COUNT(*) FROM event.direct_t20_model_frame) AS model_frame_rows
""").df())

In [ ]:
# 31. Manifest + GCS upload
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            chunk=f.read(1024*1024)
            if not chunk: break
            h.update(chunk)
    return h.hexdigest()

MANIFEST_PATH=META_DIR/'step6_direct_t20_timing_manifest.json'
manifest={
    'built_at':datetime.now().isoformat(),'config':CFG,
    'event_n':int(model_frame['event_id'].nunique()),'model_frame_rows':int(len(model_frame)),
    'files':{p.name:sha256(p) for p in [SNAPSHOT_PATH,TARGET_PATH,FRAME_PATH,RESULT_PATH,PRED_PATH,DEV_PATH,TEST_PATH,TIMING57_PATH,ABLATION_PATH,BOOT_PATH,STABILITY_PATH]}
}
with open(MANIFEST_PATH,'w',encoding='utf-8') as f:
    json.dump(manifest,f,ensure_ascii=False,indent=2,default=str)

# 심사용 실행에서는 팀 GCS에 업로드하지 않습니다 — 이미 로컬(DERIVED_DIR/OUT_DIR)에
# 저장되어 있으므로 그 사실만 확인합니다.
for p in [SNAPSHOT_PATH,TARGET_PATH,FRAME_PATH,RESULT_PATH,PRED_PATH,DEV_PATH,TEST_PATH,TIMING57_PATH,ABLATION_PATH,BOOT_PATH,STABILITY_PATH,CFG_PATH,MANIFEST_PATH]:
    assert p.exists(), p
print('✅ 로컬 저장 확인 완료')

### PART N — Final QA

In [ ]:
# 32. Final validation
checks=pd.DataFrame([
    {'check':'timings 0/3/5/7/10 exist','passed':set(PRED_OFFSETS).issubset(set(model_frame['prediction_offset'].astype(int)))},
    {'check':'unique event+timing','passed':not model_frame.duplicated(['event_id','prediction_offset']).any()},
    {'check':'same Direct T20 target across timings','passed':bool((same_target['bhar_n']==1).all() and (same_target['target_date_n']==1).all())},
    {'check':'prediction before target','passed':bool((model_frame['cutoff_date']<model_frame['direct_t20_date']).all())},
    {'check':'no fixed binary label in reusable frame','passed':not any(c in model_frame.columns for c in ['y','q35_label','direct_t20_label'])},
    {'check':'common q35 across timing/recipe/model','passed':bool((q35_check['nunique']==1).all())},
    {'check':'2025 result exists','passed':bool(results['split'].eq('TEST_2025').any())},
    {'check':'2026 stress result exists','passed':bool(results['split'].eq('STRESS_2026').any())},
])
display(checks)
assert checks['passed'].all()
print('✅ STEP 6 FINAL validation passed')

# STAGE 16: STEP 6B — Recall / Precision / Threshold 실제 분류성능 검증

**태그:** 최종

**원본 노트북:** `6B__Recall___Precision___Threshold_실제_분류성능_검증.ipynb`

> STEP 6 모델을 재학습하지 않고, 저장된 model_predictions.parquet을 재사용합니다.

## FinDA STEP 6B — Recall / Precision / Threshold 실제 분류성능 검증

이 노트북은 **이전 STEP 6 모델을 다시 학습하지 않습니다.**

STEP 6에서 저장한 `model_predictions.parquet`을 재사용해:

1. Development OOF에서만 Classification Threshold 선택
2. Threshold 고정
3. 2025 Temporal Test에 적용
4. 2026 Stress Test에 적용
5. Recall / Precision / F1 / Specificity / Balanced Accuracy 평가
6. Confusion Matrix / PR Curve / Threshold Curve 생성
7. Bootstrap 95% CI 계산

를 수행합니다.

---

### Threshold 전략

#### `F1_MAX`
Development OOF에서 F1 최대 threshold

#### `RECALL_80`
Development OOF에서 Recall ≥ 80%를 만족하는 threshold 중 Precision 최대

---

### 메인 후보

- RF + P @ T+5
- RF + P @ T+7
- RF + P @ T+10
- LightGBM + P @ T+5
- Logistic + P @ T+5
- RF + P+FI @ T+5

In [ ]:
# 1. imports
!pip -q install pyarrow scikit-learn matplotlib

from pathlib import Path
from datetime import datetime
import json, subprocess, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    balanced_accuracy_score,
    accuracy_score,
    precision_recall_curve,
    brier_score_loss,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 320)

print("✅ imports 완료")

In [ ]:
# 2. Paths / config
PROJECT_ID = GCP_PROJECT_ID
BUCKET = GCS_BUCKET_NAME

ROOT = Path(DRIVE_REFACTOR_ROOT)
STEP6_OUT = ROOT / "outputs" / "step6_direct_t20_timing"
OUT_DIR = ROOT / "outputs" / "step6b_classification_metrics"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PRED_PATH = STEP6_OUT / "model_predictions.parquet"

# STEP6 파일이 로컬에 없으면(예: STAGE 15를 건너뛰고 이 STAGE만 실행한 경우)
# 공개 읽기전용 버킷에서 로그인 없이 받아옵니다.
if not PRED_PATH.exists():
    STEP6_OUT.mkdir(parents=True, exist_ok=True)
    gcs_public_download("derived/modeling_direct_t20/model_predictions.parquet", PRED_PATH)

assert PRED_PATH.exists(), PRED_PATH

CANDIDATES = [
    {"name":"RF_P_T5",   "prediction_offset":5,  "recipe":"P",    "model":"rf"},
    {"name":"RF_P_T7",   "prediction_offset":7,  "recipe":"P",    "model":"rf"},
    {"name":"RF_P_T10",  "prediction_offset":10, "recipe":"P",    "model":"rf"},
    {"name":"LGBM_P_T5", "prediction_offset":5,  "recipe":"P",    "model":"lgbm"},
    {"name":"LOGIT_P_T5","prediction_offset":5,  "recipe":"P",    "model":"logit"},
    {"name":"RF_PFI_T5", "prediction_offset":5,  "recipe":"P_FI", "model":"rf"},
]

TARGET_RECALL = 0.80
BOOT_N = 3000
RANDOM_STATE = 20260816

print(PRED_PATH)

### PART A — 기존 Prediction QA

In [ ]:
# 3. Load predictions
pred = pd.read_parquet(PRED_PATH)

pred["event_date"] = pd.to_datetime(
    pred["event_date"], errors="coerce"
).dt.normalize()

required = {
    "prediction_offset","recipe","model","seed","split",
    "event_id","event_date","y","prob","label_q35"
}
missing = sorted(required - set(pred.columns))
assert not missing, missing

print("shape:", pred.shape)
print("splits:", pred["split"].value_counts().to_dict())
print("offsets:", sorted(pred["prediction_offset"].unique()))
print("recipes:", sorted(pred["recipe"].unique()))
print("models:", sorted(pred["model"].unique()))
print("seed:", sorted(pred["seed"].unique()))

display(pred.head())

In [ ]:
# 4. Candidate availability
candidate_rows = []

for c in CANDIDATES:
    q = pred[
        pred["prediction_offset"].eq(c["prediction_offset"])
        & pred["recipe"].eq(c["recipe"])
        & pred["model"].eq(c["model"])
    ]
    candidate_rows.append({
        **c,
        "rows":len(q),
        "dev_n":q[q["split"].eq("DEV_OOF")]["event_id"].nunique(),
        "test25_n":q[q["split"].eq("TEST_2025")]["event_id"].nunique(),
        "stress26_n":q[q["split"].eq("STRESS_2026")]["event_id"].nunique(),
    })

candidate_qa = pd.DataFrame(candidate_rows)
display(candidate_qa)

assert (candidate_qa[["dev_n","test25_n","stress26_n"]] > 0).all().all()
print("✅ candidate predictions 존재")

### PART B — Threshold 선택

Threshold는 **DEV_OOF에서만** 선택합니다.

2025/2026은 threshold 선택에 사용하지 않습니다.

In [ ]:
# 5. Metric helpers
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0,1]
    ).ravel()
    return tn / (tn + fp) if (tn+fp) else np.nan

def metric_at_threshold(y, prob, threshold):
    y = np.asarray(y).astype(int)
    prob = np.asarray(prob).astype(float)
    yhat = (prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y, yhat, labels=[0,1]
    ).ravel()

    return {
        "n":len(y),
        "positive_n":int(y.sum()),
        "positive_rate":float(y.mean()),
        "threshold":float(threshold),
        "roc_auc":roc_auc_score(y,prob) if len(np.unique(y))==2 else np.nan,
        "pr_auc":average_precision_score(y,prob) if len(np.unique(y))==2 else np.nan,
        "accuracy":accuracy_score(y,yhat),
        "balanced_accuracy":balanced_accuracy_score(y,yhat),
        "precision":precision_score(y,yhat,zero_division=0),
        "recall":recall_score(y,yhat,zero_division=0),
        "specificity":specificity_score(y,yhat),
        "f1":f1_score(y,yhat,zero_division=0),
        "brier":brier_score_loss(y,prob),
        "tn":int(tn),
        "fp":int(fp),
        "fn":int(fn),
        "tp":int(tp),
    }

def threshold_table(y, prob):
    thresholds = np.unique(
        np.r_[
            np.linspace(0.05,0.95,181),
            np.asarray(prob)
        ]
    )
    rows = []

    for t in thresholds:
        m = metric_at_threshold(y,prob,t)
        rows.append({
            "threshold":t,
            "precision":m["precision"],
            "recall":m["recall"],
            "f1":m["f1"],
            "specificity":m["specificity"],
            "balanced_accuracy":m["balanced_accuracy"],
            "pred_positive_n":m["tp"]+m["fp"],
        })

    return pd.DataFrame(rows).sort_values("threshold").reset_index(drop=True)

def choose_f1_max(tab):
    return (
        tab.sort_values(
            ["f1","recall","precision","threshold"],
            ascending=[False,False,False,False]
        )
        .iloc[0]
    )

def choose_recall_target(tab, target=0.80):
    eligible = tab[tab["recall"] >= target].copy()

    if len(eligible)==0:
        # 목표 recall 불가능 시 최대 recall
        return (
            tab.sort_values(
                ["recall","precision","threshold"],
                ascending=[False,False,False]
            ).iloc[0]
        )

    # Recall 목표 이상에서 Precision 최대.
    # 동률이면 더 높은 threshold -> 불필요한 경고 감소
    return (
        eligible.sort_values(
            ["precision","f1","threshold"],
            ascending=[False,False,False]
        ).iloc[0]
    )

In [ ]:
# 6. Choose thresholds from DEV only
threshold_selection = []
threshold_curves = {}

for c in CANDIDATES:
    dev = pred[
        pred["prediction_offset"].eq(c["prediction_offset"])
        & pred["recipe"].eq(c["recipe"])
        & pred["model"].eq(c["model"])
        & pred["split"].eq("DEV_OOF")
    ].copy()

    # 현재 전체 grid prediction은 seed=42 한 세트
    # 혹시 중복 seed가 들어오면 event별 평균으로 안전하게 집계
    dev = (
        dev.groupby(["event_id","event_date","y"],as_index=False)
        .agg(prob=("prob","mean"))
    )

    assert dev["event_id"].is_unique
    assert dev["y"].nunique()==2

    tab = threshold_table(dev["y"],dev["prob"])
    threshold_curves[c["name"]] = tab

    f1row = choose_f1_max(tab)
    r80row = choose_recall_target(tab,TARGET_RECALL)

    for strategy,row in [
        ("F1_MAX",f1row),
        ("RECALL_80",r80row),
    ]:
        threshold_selection.append({
            "candidate":c["name"],
            "prediction_offset":c["prediction_offset"],
            "recipe":c["recipe"],
            "model":c["model"],
            "strategy":strategy,
            "selected_threshold":float(row["threshold"]),
            "DEV_precision":float(row["precision"]),
            "DEV_recall":float(row["recall"]),
            "DEV_f1":float(row["f1"]),
            "DEV_specificity":float(row["specificity"]),
            "DEV_balanced_accuracy":float(row["balanced_accuracy"]),
        })

threshold_selection = pd.DataFrame(threshold_selection)
display(threshold_selection)

### PART C — 선택된 Threshold를 2025 / 2026에 고정 적용

In [ ]:
# 7. Evaluate DEV / 2025 / 2026
eval_rows = []

for c in CANDIDATES:
    for _,sel in threshold_selection[
        threshold_selection["candidate"].eq(c["name"])
    ].iterrows():

        threshold = float(sel["selected_threshold"])

        for split in ["DEV_OOF","TEST_2025","STRESS_2026"]:
            q = pred[
                pred["prediction_offset"].eq(c["prediction_offset"])
                & pred["recipe"].eq(c["recipe"])
                & pred["model"].eq(c["model"])
                & pred["split"].eq(split)
            ].copy()

            q = (
                q.groupby(["event_id","event_date","y"],as_index=False)
                .agg(prob=("prob","mean"))
            )

            if len(q)==0 or q["y"].nunique()<2:
                continue

            m = metric_at_threshold(
                q["y"],q["prob"],threshold
            )

            eval_rows.append({
                "candidate":c["name"],
                "prediction_offset":c["prediction_offset"],
                "recipe":c["recipe"],
                "model":c["model"],
                "strategy":sel["strategy"],
                "split":split,
                **m,
            })

classification_results = pd.DataFrame(eval_rows)

display(
    classification_results.sort_values(
        ["strategy","candidate","split"]
    )
)

### PART D — 메인 모델 RF + P @ T+5

발표에서 가장 중요하게 볼 표입니다.

In [ ]:
# 8. Main candidate summary
main_result = classification_results[
    classification_results["candidate"].eq("RF_P_T5")
].copy()

display(main_result[[
    "strategy","split","threshold",
    "roc_auc","pr_auc",
    "precision","recall","f1",
    "specificity","balanced_accuracy",
    "tn","fp","fn","tp"
]])

### PART E — Confusion Matrix

In [ ]:
# 9. Confusion matrices for RF_P_T5
def plot_confusion(cm, title):
    fig, ax = plt.subplots(figsize=(5,4))
    im = ax.imshow(cm)

    for i in range(2):
        for j in range(2):
            ax.text(
                j,i,str(cm[i,j]),
                ha="center",va="center"
            )

    ax.set_xticks([0,1])
    ax.set_yticks([0,1])
    ax.set_xticklabels(["Pred 0","Pred 1"])
    ax.set_yticklabels(["Actual 0","Actual 1"])
    ax.set_xlabel("Prediction")
    ax.set_ylabel("Actual")
    ax.set_title(title)
    plt.show()

for strategy in ["F1_MAX","RECALL_80"]:
    sel = threshold_selection[
        threshold_selection["candidate"].eq("RF_P_T5")
        & threshold_selection["strategy"].eq(strategy)
    ].iloc[0]

    threshold = float(sel["selected_threshold"])

    for split in ["TEST_2025","STRESS_2026"]:
        q = pred[
            pred["prediction_offset"].eq(5)
            & pred["recipe"].eq("P")
            & pred["model"].eq("rf")
            & pred["split"].eq(split)
        ].copy()

        q = (
            q.groupby(["event_id","y"],as_index=False)
            .agg(prob=("prob","mean"))
        )
        yhat = (q["prob"] >= threshold).astype(int)
        cm = confusion_matrix(q["y"],yhat,labels=[0,1])

        plot_confusion(
            cm,
            f"RF+P @ T+5 — {strategy} — {split}"
        )

### PART F — Threshold Curve

In [ ]:
# 10. Precision / Recall / F1 by threshold — RF P T+5
tab = threshold_curves["RF_P_T5"]

fig, ax = plt.subplots(figsize=(9,5))
ax.plot(tab["threshold"],tab["precision"],label="Precision")
ax.plot(tab["threshold"],tab["recall"],label="Recall")
ax.plot(tab["threshold"],tab["f1"],label="F1")
ax.axhline(TARGET_RECALL,linestyle="--",label="Recall target 0.80")

for strategy in ["F1_MAX","RECALL_80"]:
    t = float(
        threshold_selection[
            threshold_selection["candidate"].eq("RF_P_T5")
            & threshold_selection["strategy"].eq(strategy)
        ]["selected_threshold"].iloc[0]
    )
    ax.axvline(t,linestyle="--",label=f"{strategy}: {t:.3f}")

ax.set_xlim(0.05,0.95)
ax.set_ylim(0,1)
ax.set_xlabel("Classification threshold")
ax.set_ylabel("Metric")
ax.set_title("DEV OOF threshold trade-off — RF+P @ T+5")
ax.legend()
plt.show()

### PART G — Precision-Recall Curve

In [ ]:
# 11. PR curves
for split in ["DEV_OOF","TEST_2025","STRESS_2026"]:
    q = pred[
        pred["prediction_offset"].eq(5)
        & pred["recipe"].eq("P")
        & pred["model"].eq("rf")
        & pred["split"].eq(split)
    ].copy()

    q = (
        q.groupby(["event_id","y"],as_index=False)
        .agg(prob=("prob","mean"))
    )

    precision, recall, _ = precision_recall_curve(
        q["y"], q["prob"]
    )
    ap = average_precision_score(q["y"],q["prob"])

    fig, ax = plt.subplots(figsize=(6,5))
    ax.plot(recall,precision)
    ax.axhline(
        q["y"].mean(),
        linestyle="--",
        label=f"Base rate={q['y'].mean():.3f}"
    )
    ax.set_xlim(0,1)
    ax.set_ylim(0,1)
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(f"RF+P @ T+5 — {split} — PR-AUC={ap:.3f}")
    ax.legend()
    plt.show()

### PART H — Bootstrap 95% CI

2025 / 2026의 Precision, Recall, F1이 표본 변동에 얼마나 민감한지 Event-level bootstrap으로 확인합니다.

In [ ]:
# 12. Bootstrap confidence intervals
def bootstrap_metrics(
    y,
    prob,
    threshold,
    n_boot=3000,
    seed=20260816,
):
    df = pd.DataFrame({
        "y":np.asarray(y).astype(int),
        "prob":np.asarray(prob).astype(float),
    })

    obs = metric_at_threshold(
        df["y"],df["prob"],threshold
    )

    rng = np.random.default_rng(seed)
    idx = np.arange(len(df))
    boot = []

    for _ in range(n_boot):
        samp = rng.choice(idx,size=len(idx),replace=True)
        z = df.iloc[samp]

        if z["y"].nunique()<2:
            continue

        m = metric_at_threshold(
            z["y"],z["prob"],threshold
        )
        boot.append({
            k:m[k]
            for k in [
                "precision","recall","f1",
                "specificity","balanced_accuracy"
            ]
        })

    boot = pd.DataFrame(boot)

    out = {}
    for k in boot.columns:
        lo,hi = np.quantile(boot[k],[0.025,0.975])
        out[f"{k}_value"] = obs[k]
        out[f"{k}_ci_low"] = lo
        out[f"{k}_ci_high"] = hi

    return out

bootstrap_rows=[]

for strategy in ["F1_MAX","RECALL_80"]:
    threshold = float(
        threshold_selection[
            threshold_selection["candidate"].eq("RF_P_T5")
            & threshold_selection["strategy"].eq(strategy)
        ]["selected_threshold"].iloc[0]
    )

    for split in ["TEST_2025","STRESS_2026"]:
        q = pred[
            pred["prediction_offset"].eq(5)
            & pred["recipe"].eq("P")
            & pred["model"].eq("rf")
            & pred["split"].eq(split)
        ].copy()

        q = (
            q.groupby(["event_id","y"],as_index=False)
            .agg(prob=("prob","mean"))
        )

        out = bootstrap_metrics(
            q["y"],q["prob"],threshold,
            n_boot=BOOT_N,
            seed=RANDOM_STATE,
        )

        bootstrap_rows.append({
            "candidate":"RF_P_T5",
            "strategy":strategy,
            "split":split,
            "threshold":threshold,
            "n":len(q),
            **out,
        })

bootstrap_ci = pd.DataFrame(bootstrap_rows)
display(bootstrap_ci)

### PART I — 후보 모델 비교

같은 Threshold 전략에서 실제 분류성능까지 비교합니다.

In [ ]:
# 13. Candidate comparison
for strategy in ["F1_MAX","RECALL_80"]:
    print("==========",strategy,"==========")

    comp = classification_results[
        classification_results["strategy"].eq(strategy)
        & classification_results["split"].isin(
            ["TEST_2025","STRESS_2026"]
        )
    ].copy()

    display(comp[[
        "candidate","split","threshold",
        "roc_auc","pr_auc",
        "precision","recall","f1",
        "specificity","balanced_accuracy",
        "fp","fn","tp","tn"
    ]].sort_values(
        ["split","f1"],
        ascending=[True,False]
    ))

### PART J — Timing별 실제 탐지성능

RF + P에서 T/T+3/T+5/T+7/T+10의 실제 Recall/Precision을 비교합니다.

각 Timing은 자신의 DEV OOF에서 threshold를 고르고,
그 threshold를 2025/2026에 고정 적용합니다.

In [ ]:
# 14. RF P timing classification comparison
timing_candidates = []

for off in [0,3,5,7,10]:
    name = f"RF_P_T{off}"

    dev = pred[
        pred["prediction_offset"].eq(off)
        & pred["recipe"].eq("P")
        & pred["model"].eq("rf")
        & pred["split"].eq("DEV_OOF")
    ].copy()

    dev = (
        dev.groupby(["event_id","event_date","y"],as_index=False)
        .agg(prob=("prob","mean"))
    )

    tab = threshold_table(dev["y"],dev["prob"])
    f1row = choose_f1_max(tab)
    t = float(f1row["threshold"])

    for split in ["TEST_2025","STRESS_2026"]:
        q = pred[
            pred["prediction_offset"].eq(off)
            & pred["recipe"].eq("P")
            & pred["model"].eq("rf")
            & pred["split"].eq(split)
        ].copy()

        q = (
            q.groupby(["event_id","y"],as_index=False)
            .agg(prob=("prob","mean"))
        )

        m = metric_at_threshold(
            q["y"],q["prob"],t
        )

        timing_candidates.append({
            "prediction_offset":off,
            "split":split,
            **m,
        })

timing_classification = pd.DataFrame(timing_candidates)

display(timing_classification[[
    "prediction_offset","split","threshold",
    "roc_auc","precision","recall","f1",
    "specificity","balanced_accuracy",
    "fp","fn"
]])

### PART K — 경고 확률 Calibration / Risk Bin

예측확률을 5개 구간으로 나눠 실제 위험률이 함께 증가하는지 확인합니다.

In [ ]:
# 15. Risk bins — RF P T+5
risk_bin_rows=[]

for split in ["TEST_2025","STRESS_2026"]:
    q = pred[
        pred["prediction_offset"].eq(5)
        & pred["recipe"].eq("P")
        & pred["model"].eq("rf")
        & pred["split"].eq(split)
    ].copy()

    q = (
        q.groupby(["event_id","y"],as_index=False)
        .agg(prob=("prob","mean"))
    )

    q["risk_bin"] = pd.qcut(
        q["prob"],
        q=5,
        labels=["Q1","Q2","Q3","Q4","Q5"],
        duplicates="drop",
    )

    tab = (
        q.groupby("risk_bin",observed=True)
        .agg(
            n=("event_id","size"),
            mean_pred=("prob","mean"),
            actual_risk_rate=("y","mean"),
        )
        .reset_index()
    )
    tab["split"]=split
    risk_bin_rows.append(tab)

risk_bins = pd.concat(risk_bin_rows,ignore_index=True)
display(risk_bins)

### PART L — 저장

In [ ]:
# 16. Save outputs
THRESHOLD_PATH = OUT_DIR / "threshold_selection.csv"
RESULT_PATH = OUT_DIR / "classification_metrics.csv"
BOOT_PATH = OUT_DIR / "bootstrap_ci_main_model.csv"
TIMING_PATH = OUT_DIR / "timing_classification_metrics.csv"
RISK_PATH = OUT_DIR / "risk_bins.csv"

threshold_selection.to_csv(
    THRESHOLD_PATH,index=False,encoding="utf-8-sig"
)
classification_results.to_csv(
    RESULT_PATH,index=False,encoding="utf-8-sig"
)
bootstrap_ci.to_csv(
    BOOT_PATH,index=False,encoding="utf-8-sig"
)
timing_classification.to_csv(
    TIMING_PATH,index=False,encoding="utf-8-sig"
)
risk_bins.to_csv(
    RISK_PATH,index=False,encoding="utf-8-sig"
)

print("saved:", OUT_DIR)

In [ ]:
# 심사용 실행에서는 팀 GCS에 업로드하지 않습니다 — 이미 로컬(OUT_DIR)에
# 저장되어 있으므로 그 사실만 확인합니다.
for p in [
    THRESHOLD_PATH,
    RESULT_PATH,
    BOOT_PATH,
    TIMING_PATH,
    RISK_PATH,
]:
    assert p.exists(), p

print("✅ 로컬 저장 확인 완료")

# STAGE 17: STEP 6C — RandomForest @ T+5 관심·시장·공매도 증분효과 재검증 (Recipe Ablation)

**태그:** 최종

**원본 노트북:** `6C__랜덤포레스트_T5_관심_시장_공매도_제거실험.ipynb`

> P-only vs P+ATT/P+MARKET/P+SHORT 비교 — P단독 우세 결론의 근거.

## FinDA STEP 6C — RF @ T+5 관심·시장·공매도 증분효과 재검증

### 목적

현재 메인 후보인:

```text
RandomForest + P @ T+5
```

를 기준으로 고정하고,

- 관심지표
- 시장지표
- 공매도

를 **하나씩 또는 조합해서 추가했을 때** 실제로 성능이 개선되는지 다시 검증합니다.

---

### 비교 Recipes

```text
P
P_ATT
P_MARKET
P_SHORT
P_ATT_MARKET
P_ATT_SHORT
P_MARKET_SHORT
P_ATT_MARKET_SHORT
```

---

### 고정 조건

- Target: Direct T+20 BHAR
- Label: Train-fold q35 이하 = 1
- Timing: T+5
- Model: RandomForest
- RF: 300 trees / depth 5 / min leaf 10 / balanced
- SMOTE: Train 내부만
- DEV: Purged Expanding Walk-Forward
- 2025: Temporal Test
- 2026: Stress Test

---

### 이번 분석의 핵심

단순 AUC만 비교하지 않습니다.

각 추가 Recipe를 `P`와 동일 Event에서 직접 비교해:

- ΔROC-AUC
- ΔPR-AUC
- ΔPrecision
- ΔRecall
- ΔF1
- ΔBalanced Accuracy

의 **paired bootstrap 95% CI**까지 계산합니다.

In [ ]:
# 1. packages / imports
!pip -q install duckdb pyarrow scikit-learn imbalanced-learn scipy matplotlib

from pathlib import Path
from datetime import datetime
import json, subprocess, hashlib, warnings, time

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    balanced_accuracy_score,
    accuracy_score,
)
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 360)
pd.set_option("display.max_rows", 300)

print("✅ imports 완료")

In [ ]:
# 2. Config
PROJECT_ID = GCP_PROJECT_ID
BUCKET = GCS_BUCKET_NAME

ROOT = Path(DRIVE_REFACTOR_ROOT)
DB_PATH = ROOT / "duckdb" / "finda.duckdb"
OUT_DIR = ROOT / "outputs" / "step6c_att_market_short"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTION_OFFSET = 5
LABEL_Q = 0.35

DEV_END = pd.Timestamp("2024-12-31")
TEST25_START = pd.Timestamp("2025-01-01")
TEST25_END = pd.Timestamp("2025-12-31")
TEST26_START = pd.Timestamp("2026-01-01")
TEST26_END = pd.Timestamp("2026-12-31")

N_WF_SPLITS = 5
INITIAL_TRAIN_FRAC = 0.35

RF_SEED = 42
USE_SMOTE = True
TARGET_RECALL = 0.80
BOOT_N = 2000
BOOT_SEED = 20260816

assert DB_PATH.exists(), DB_PATH

print(DB_PATH)

## PART A — Direct T+20 Model Frame 로드

이전 STEP 6에서 저장한:

```text
event.direct_t20_model_frame
```

을 그대로 재사용합니다.

따라서 Event/Target/Snapshot은 다시 만들지 않습니다.

In [ ]:
# 3. Load model frame
con = duckdb.connect(str(DB_PATH))

inventory = con.execute("""
SELECT table_schema, table_name, table_type
FROM information_schema.tables
WHERE table_schema IN ('event','meta')
ORDER BY table_schema, table_name
""").df()

display(inventory)

tables = set(
    inventory.loc[inventory["table_schema"].eq("event"), "table_name"].astype(str)
)

assert "direct_t20_model_frame" in tables, (
    "event.direct_t20_model_frame가 없습니다. "
    "STEP 6 FINAL을 먼저 실행하세요."
)

model_frame = con.execute("""
SELECT *
FROM event.direct_t20_model_frame
WHERE prediction_offset = 5
ORDER BY event_date, ticker
""").df()

for c in [
    "event_date", "cutoff_date",
    "direct_t20_date", "target_start_date", "target_end_date"
]:
    if c in model_frame.columns:
        model_frame[c] = pd.to_datetime(
            model_frame[c], errors="coerce"
        ).dt.normalize()

model_frame["ticker"] = (
    model_frame["ticker"].astype(str).str.zfill(6)
)

print("shape:", model_frame.shape)
print("events:", model_frame["event_id"].nunique())
print("period:", model_frame["event_date"].min(), "~", model_frame["event_date"].max())

assert model_frame["event_id"].is_unique
assert model_frame["direct_bhar_T20"].notna().all()
assert (
    model_frame["cutoff_date"] < model_frame["direct_t20_date"]
).all()

print("✅ T+5 model frame 정상")

## PART B — Feature Group 정의

In [ ]:
# 4. Base P / Attention / Market / Short

P_T = [
    "ret_1d_T","ret_5d_T","ret_10d_T","ret_20d_T",
    "volatility_5d_T","volatility_20d_T","volatility_60d_T",
    "volume_ratio_prior20_T","trading_value_ratio_prior20_T",
    "turnover_T","high_low_range_T",
    "up_day_share_10_T","max_daily_return_10_T",
]

P_POST = [
    "early_return","early_peak_return","early_giveback_ratio",
    "early_true_mdd","volume_sustain","volume_change_vs_T",
]

ATT_T = [
    "search_pct_50_T",
    "news_pct_50_T",
    "search_news_level_gap_T",
    "search_lead_days",
    "news_lead_days",
]

ATT_POST = [
    "search_mean_to_cutoff",
    "search_max_to_cutoff",
    "search_last_at_cutoff",
    "search_high_share_to_cutoff",
    "search_sustain_ratio",
    "news_mean_to_cutoff",
    "news_max_to_cutoff",
    "news_last_at_cutoff",
    "news_high_share_to_cutoff",
    "news_sustain_ratio",
    "search_news_level_gap_cutoff",
]

MARKET = [
    "market_drawdown_60d_cutoff",
    "market_volatility_20d_cutoff",
    "market_trading_value_change_5_20_cutoff",
    "market_decline_ratio_cutoff",
    "market_below_ma20_ratio_cutoff",
    "market_crash_3pct_ratio_cutoff",
    "market_illiquidity_shock_cutoff",
]

SHORT = sorted([
    c for c in model_frame.columns
    if c.startswith("short_")
    and c.endswith("_cutoff")
    and pd.api.types.is_numeric_dtype(model_frame[c])
])

def existing_numeric(cols):
    return [
        c for c in cols
        if c in model_frame.columns
        and pd.api.types.is_numeric_dtype(model_frame[c])
        and model_frame[c].notna().any()
    ]

GROUPS = {
    "P": existing_numeric(P_T + P_POST),
    "ATT": existing_numeric(ATT_T + ATT_POST),
    "MARKET": existing_numeric(MARKET),
    "SHORT": existing_numeric(SHORT),
}

for k, v in GROUPS.items():
    print(f"\n[{k}] n={len(v)}")
    print(v)

In [ ]:
# 5. Recipes
def uniq(xs):
    return list(dict.fromkeys(xs))

RECIPES = {
    "P":
        uniq(GROUPS["P"]),

    "P_ATT":
        uniq(GROUPS["P"] + GROUPS["ATT"]),

    "P_MARKET":
        uniq(GROUPS["P"] + GROUPS["MARKET"]),

    "P_SHORT":
        uniq(GROUPS["P"] + GROUPS["SHORT"]),

    "P_ATT_MARKET":
        uniq(GROUPS["P"] + GROUPS["ATT"] + GROUPS["MARKET"]),

    "P_ATT_SHORT":
        uniq(GROUPS["P"] + GROUPS["ATT"] + GROUPS["SHORT"]),

    "P_MARKET_SHORT":
        uniq(GROUPS["P"] + GROUPS["MARKET"] + GROUPS["SHORT"]),

    "P_ATT_MARKET_SHORT":
        uniq(
            GROUPS["P"]
            + GROUPS["ATT"]
            + GROUPS["MARKET"]
            + GROUPS["SHORT"]
        ),
}

recipe_summary = pd.DataFrame([
    {
        "recipe": name,
        "feature_n": len(cols),
        "features": ", ".join(cols),
    }
    for name, cols in RECIPES.items()
])

display(recipe_summary)

# leakage guard
for recipe, feats in RECIPES.items():
    bad = [
        c for c in feats
        if any(
            token in c.lower()
            for token in [
                "direct_bhar",
                "target_",
                "future_",
                "label",
            ]
        )
    ]
    assert not bad, (recipe, bad)

assert len(GROUPS["P"]) > 0
assert len(GROUPS["ATT"]) > 0
assert len(GROUPS["MARKET"]) > 0

print("SHORT feature n:", len(GROUPS["SHORT"]))
print("✅ feature recipes 준비")

### 중요

만약 `SHORT feature n = 0`이면 현재 `direct_t20_model_frame`에 공매도 cutoff 변수가 저장되지 않은 것입니다.

그 경우 이 노트북은 중단하도록 아래 QA에서 확인합니다.

In [ ]:
# 6. Feature QA
group_qa = pd.DataFrame([
    {
        "group": k,
        "feature_n": len(v),
        "all_missing_feature_n": sum(
            model_frame[c].isna().all() for c in v
        ),
        "mean_missing_rate": (
            float(model_frame[v].isna().mean().mean())
            if len(v) else np.nan
        ),
    }
    for k, v in GROUPS.items()
])

display(group_qa)

assert len(GROUPS["SHORT"]) > 0, (
    "공매도 cutoff numeric feature가 model frame에 없습니다. "
    "STEP5/STEP6의 short cutoff 저장 상태를 확인해야 합니다."
)

print("✅ ATT / MARKET / SHORT 모두 비교 가능")

## PART C — Train-only q35 / Purged Expanding Walk-Forward

In [ ]:
# 7. Label helper
def apply_train_q35(train_df, eval_df, q=LABEL_Q):
    tr = train_df.copy()
    ev = eval_df.copy()

    target = (
        tr[["event_id", "direct_bhar_T20"]]
        .drop_duplicates("event_id")
        ["direct_bhar_T20"]
    )

    threshold = float(target.quantile(q))

    tr["y"] = (
        tr["direct_bhar_T20"] <= threshold
    ).astype(int)

    ev["y"] = (
        ev["direct_bhar_T20"] <= threshold
    ).astype(int)

    return tr, ev, threshold

In [ ]:
# 8. Same purged expanding fold logic as STEP 6
def make_purged_folds(
    dev_df,
    n_splits=5,
    initial_train_frac=0.35,
):
    x = (
        dev_df
        .sort_values("event_date")
        .reset_index(drop=True)
        .copy()
    )

    dates = np.array(
        sorted(
            pd.Series(
                x["event_date"].dropna().unique()
            )
        )
    )

    initial_n = max(
        5,
        int(len(dates) * initial_train_frac),
    )

    remain = dates[initial_n:]
    blocks = np.array_split(remain, n_splits)

    folds = []

    for i, val_dates in enumerate(blocks, 1):
        if len(val_dates) == 0:
            continue

        val_start = pd.Timestamp(val_dates[0])
        val_end = pd.Timestamp(val_dates[-1])

        train_mask = (
            (x["event_date"] < val_start)
            & (x["target_end_date"] < val_start)
        )

        val_mask = x["event_date"].between(
            val_start,
            val_end,
            inclusive="both",
        )

        folds.append({
            "fold": i,
            "train_idx": x.index[train_mask].to_numpy(),
            "val_idx": x.index[val_mask].to_numpy(),
            "val_start": val_start,
            "val_end": val_end,
        })

    return x, folds

In [ ]:
# 9. Fold QA
dev = model_frame[
    model_frame["event_date"] <= DEV_END
].copy()

dev, folds = make_purged_folds(
    dev,
    n_splits=N_WF_SPLITS,
    initial_train_frac=INITIAL_TRAIN_FRAC,
)

qa_rows = []

for f in folds:
    tr = dev.loc[f["train_idx"]].copy()
    va = dev.loc[f["val_idx"]].copy()

    tr_l, va_l, q35 = apply_train_q35(tr, va)

    qa_rows.append({
        "fold": f["fold"],
        "train_n": len(tr),
        "val_n": len(va),
        "q35": q35,
        "train_pos_rate": tr_l["y"].mean(),
        "val_pos_rate": va_l["y"].mean(),
        "train_target_end_max": tr["target_end_date"].max(),
        "val_start": f["val_start"],
        "purge_ok": bool(
            tr["target_end_date"].max()
            < f["val_start"]
        ),
    })

fold_qa = pd.DataFrame(qa_rows)
display(fold_qa)

assert fold_qa["purge_ok"].all()

print("✅ purge QA")

## PART D — RandomForest 학습 함수

In [ ]:
# 10. RF fixed model
def make_rf(seed=42):
    return RandomForestClassifier(
        n_estimators=300,
        max_depth=5,
        min_samples_leaf=10,
        class_weight="balanced",
        random_state=seed,
        n_jobs=-1,
    )

def fit_predict_rf(
    train_df,
    eval_df,
    features,
    seed=42,
):
    feats = [
        c for c in features
        if c in train_df.columns
        and pd.api.types.is_numeric_dtype(train_df[c])
        and train_df[c].notna().any()
    ]

    Xtr = (
        train_df[feats]
        .replace([np.inf, -np.inf], np.nan)
    )
    Xev = (
        eval_df[feats]
        .replace([np.inf, -np.inf], np.nan)
    )

    ytr = train_df["y"].astype(int).to_numpy()

    imputer = SimpleImputer(strategy="median")
    Xtr2 = imputer.fit_transform(Xtr)
    Xev2 = imputer.transform(Xev)

    Xfit, yfit = Xtr2, ytr

    if USE_SMOTE:
        counts = pd.Series(ytr).value_counts()

        if len(counts) == 2 and int(counts.min()) >= 2:
            k = min(5, int(counts.min()) - 1)

            sm = SMOTE(
                random_state=seed,
                k_neighbors=k,
            )

            Xfit, yfit = sm.fit_resample(
                Xtr2, ytr
            )

    model = make_rf(seed)
    model.fit(Xfit, yfit)

    prob = model.predict_proba(Xev2)[:, 1]

    return {
        "prob": prob,
        "model": model,
        "features": feats,
        "imputer": imputer,
    }

In [ ]:
# 11. Threshold-free metrics
def auc_metrics(y, prob):
    y = np.asarray(y).astype(int)
    prob = np.asarray(prob).astype(float)

    return {
        "roc_auc": (
            roc_auc_score(y, prob)
            if len(np.unique(y)) == 2
            else np.nan
        ),
        "pr_auc": (
            average_precision_score(y, prob)
            if len(np.unique(y)) == 2
            else np.nan
        ),
    }

## PART E — 8개 Recipe 모델 실행

In [ ]:
# 12. Evaluate one recipe
def evaluate_recipe(
    base_df,
    recipe,
    seed=RF_SEED,
):
    features = RECIPES[recipe]

    dev0 = base_df[
        base_df["event_date"] <= DEV_END
    ].copy()

    dev0, folds0 = make_purged_folds(
        dev0,
        N_WF_SPLITS,
        INITIAL_TRAIN_FRAC,
    )

    fold_rows = []
    pred_rows = []

    # DEV OOF
    for f in folds0:
        tr = dev0.loc[f["train_idx"]].copy()
        va = dev0.loc[f["val_idx"]].copy()

        tr, va, q35 = apply_train_q35(
            tr, va
        )

        fit = fit_predict_rf(
            tr,
            va,
            features,
            seed=seed,
        )

        m = auc_metrics(
            va["y"],
            fit["prob"],
        )

        fold_rows.append({
            "recipe": recipe,
            "split": "DEV_WF",
            "fold": f["fold"],
            "feature_n": len(fit["features"]),
            "label_q35": q35,
            "n": len(va),
            **m,
        })

        for eid, ed, y, p in zip(
            va["event_id"],
            va["event_date"],
            va["y"],
            fit["prob"],
        ):
            pred_rows.append({
                "recipe": recipe,
                "split": "DEV_OOF",
                "fold": f["fold"],
                "event_id": eid,
                "event_date": ed,
                "y": int(y),
                "prob": float(p),
                "label_q35": q35,
            })

    # 2025
    tr25 = base_df[
        (base_df["event_date"] < TEST25_START)
        & (
            base_df["target_end_date"]
            < TEST25_START
        )
    ].copy()

    te25 = base_df[
        base_df["event_date"].between(
            TEST25_START,
            TEST25_END,
            inclusive="both",
        )
    ].copy()

    tr25, te25, q25 = apply_train_q35(
        tr25, te25
    )

    fit25 = fit_predict_rf(
        tr25,
        te25,
        features,
        seed=seed,
    )

    for eid, ed, y, p in zip(
        te25["event_id"],
        te25["event_date"],
        te25["y"],
        fit25["prob"],
    ):
        pred_rows.append({
            "recipe": recipe,
            "split": "TEST_2025",
            "fold": 0,
            "event_id": eid,
            "event_date": ed,
            "y": int(y),
            "prob": float(p),
            "label_q35": q25,
        })

    # 2026 stress
    tr26 = base_df[
        (base_df["event_date"] < TEST26_START)
        & (
            base_df["target_end_date"]
            < TEST26_START
        )
    ].copy()

    te26 = base_df[
        base_df["event_date"].between(
            TEST26_START,
            TEST26_END,
            inclusive="both",
        )
    ].copy()

    tr26, te26, q26 = apply_train_q35(
        tr26, te26
    )

    fit26 = fit_predict_rf(
        tr26,
        te26,
        features,
        seed=seed,
    )

    for eid, ed, y, p in zip(
        te26["event_id"],
        te26["event_date"],
        te26["y"],
        fit26["prob"],
    ):
        pred_rows.append({
            "recipe": recipe,
            "split": "STRESS_2026",
            "fold": 0,
            "event_id": eid,
            "event_date": ed,
            "y": int(y),
            "prob": float(p),
            "label_q35": q26,
        })

    return (
        pd.DataFrame(fold_rows),
        pd.DataFrame(pred_rows),
    )

In [ ]:
# 13. Run all 8 recipes
all_fold_results = []
all_predictions = []

started = time.time()

for i, recipe in enumerate(RECIPES, 1):
    print(
        f"[{i}/{len(RECIPES)}] {recipe}"
    )

    fr, pr = evaluate_recipe(
        model_frame,
        recipe,
        seed=RF_SEED,
    )

    all_fold_results.append(fr)
    all_predictions.append(pr)

fold_results = pd.concat(
    all_fold_results,
    ignore_index=True,
)

predictions = pd.concat(
    all_predictions,
    ignore_index=True,
)

print("fold_results:", fold_results.shape)
print("predictions:", predictions.shape)
print(
    "elapsed min:",
    round((time.time() - started) / 60, 2)
)

## PART F — AUC / PR-AUC 비교

In [ ]:
# 14. DEV fold mean + pooled OOF + 2025/2026
dev_fold_summary = (
    fold_results
    .groupby("recipe", as_index=False)
    .agg(
        DEV_WF_AUC=("roc_auc", "mean"),
        DEV_WF_AUC_SD=("roc_auc", "std"),
        DEV_WF_PR_AUC=("pr_auc", "mean"),
        fold_n=("fold", "nunique"),
    )
)

pooled_rows = []

for recipe in RECIPES:
    for split in [
        "DEV_OOF",
        "TEST_2025",
        "STRESS_2026",
    ]:
        q = predictions[
            predictions["recipe"].eq(recipe)
            & predictions["split"].eq(split)
        ].copy()

        assert q["event_id"].is_unique

        m = auc_metrics(q["y"], q["prob"])

        pooled_rows.append({
            "recipe": recipe,
            "split": split,
            "n": len(q),
            "positive_rate": q["y"].mean(),
            **m,
        })

pooled = pd.DataFrame(pooled_rows)

auc_summary = (
    dev_fold_summary
    .merge(
        pooled.pivot(
            index="recipe",
            columns="split",
            values="roc_auc",
        ).add_suffix("_AUC").reset_index(),
        on="recipe",
        how="left",
    )
    .merge(
        pooled.pivot(
            index="recipe",
            columns="split",
            values="pr_auc",
        ).add_suffix("_PR_AUC").reset_index(),
        on="recipe",
        how="left",
    )
)

base_dev = float(
    auc_summary.loc[
        auc_summary["recipe"].eq("P"),
        "DEV_WF_AUC",
    ].iloc[0]
)

auc_summary["DEV_delta_vs_P"] = (
    auc_summary["DEV_WF_AUC"] - base_dev
)

display(
    auc_summary.sort_values(
        "DEV_WF_AUC",
        ascending=False,
    )
)

#### 여기서 먼저 확인

기존 STEP 6의 `RF + P @ T+5`가:

```text
DEV mean AUC ≈ 0.781
2025 AUC ≈ 0.787
2026 AUC ≈ 0.778
```

와 거의 동일하게 재현되는지 확인합니다.

이 값이 크게 다르면 다음 단계로 넘어가지 말고 데이터/모델 조건을 먼저 점검합니다.

In [ ]:
# 15. Baseline reproduction QA
baseline = auc_summary[
    auc_summary["recipe"].eq("P")
].copy()

display(baseline)

dev_auc = float(
    baseline["DEV_WF_AUC"].iloc[0]
)
test25_auc = float(
    baseline["TEST_2025_AUC"].iloc[0]
)
stress26_auc = float(
    baseline["STRESS_2026_AUC"].iloc[0]
)

print("P DEV:", dev_auc)
print("P 2025:", test25_auc)
print("P 2026:", stress26_auc)

assert abs(dev_auc - 0.781271) < 0.03
assert abs(test25_auc - 0.786934) < 0.03
assert abs(stress26_auc - 0.778200) < 0.03

print("✅ 기존 P 결과 재현 범위 통과")

## PART G — DEV OOF Threshold 선택

In [ ]:
# 16. Classification metric helpers
def specificity_score(y, yhat):
    tn, fp, fn, tp = confusion_matrix(
        y,
        yhat,
        labels=[0,1],
    ).ravel()

    return (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

def classification_metrics(
    y,
    prob,
    threshold,
):
    y = np.asarray(y).astype(int)
    prob = np.asarray(prob).astype(float)
    yhat = (prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y,
        yhat,
        labels=[0,1],
    ).ravel()

    return {
        "threshold": float(threshold),
        "precision": precision_score(
            y, yhat, zero_division=0
        ),
        "recall": recall_score(
            y, yhat, zero_division=0
        ),
        "f1": f1_score(
            y, yhat, zero_division=0
        ),
        "specificity": specificity_score(
            y, yhat
        ),
        "balanced_accuracy":
            balanced_accuracy_score(y, yhat),
        "accuracy": accuracy_score(y, yhat),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

def threshold_table(y, prob):
    grid = np.unique(
        np.r_[
            np.linspace(0.05, 0.95, 181),
            np.asarray(prob),
        ]
    )

    rows = []

    for t in grid:
        m = classification_metrics(
            y, prob, t
        )

        rows.append({
            "threshold": t,
            **{
                k: m[k]
                for k in [
                    "precision",
                    "recall",
                    "f1",
                    "specificity",
                    "balanced_accuracy",
                ]
            }
        })

    return (
        pd.DataFrame(rows)
        .sort_values("threshold")
        .reset_index(drop=True)
    )

def choose_f1_max(tab):
    return (
        tab.sort_values(
            [
                "f1",
                "recall",
                "precision",
                "threshold",
            ],
            ascending=[
                False,
                False,
                False,
                False,
            ],
        )
        .iloc[0]
    )

def choose_recall80(tab):
    z = tab[
        tab["recall"] >= TARGET_RECALL
    ].copy()

    if len(z) == 0:
        return (
            tab.sort_values(
                [
                    "recall",
                    "precision",
                    "f1",
                ],
                ascending=[
                    False,
                    False,
                    False,
                ],
            )
            .iloc[0]
        )

    return (
        z.sort_values(
            [
                "precision",
                "f1",
                "threshold",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        )
        .iloc[0]
    )

In [ ]:
# 17. Select thresholds using DEV OOF only
threshold_rows = []
threshold_curves = {}

for recipe in RECIPES:
    q = predictions[
        predictions["recipe"].eq(recipe)
        & predictions["split"].eq("DEV_OOF")
    ].copy()

    tab = threshold_table(
        q["y"],
        q["prob"],
    )

    threshold_curves[recipe] = tab

    f1row = choose_f1_max(tab)
    r80row = choose_recall80(tab)

    for strategy, row in [
        ("F1_MAX", f1row),
        ("RECALL_80", r80row),
    ]:
        threshold_rows.append({
            "recipe": recipe,
            "strategy": strategy,
            "selected_threshold":
                float(row["threshold"]),
            "DEV_precision":
                float(row["precision"]),
            "DEV_recall":
                float(row["recall"]),
            "DEV_f1":
                float(row["f1"]),
            "DEV_specificity":
                float(row["specificity"]),
            "DEV_balanced_accuracy":
                float(row["balanced_accuracy"]),
        })

threshold_selection = pd.DataFrame(
    threshold_rows
)

display(threshold_selection)

## PART H — 실제 Precision / Recall / F1

In [ ]:
# 18. Fixed-threshold future evaluation
class_rows = []

for _, sel in threshold_selection.iterrows():
    recipe = sel["recipe"]
    strategy = sel["strategy"]
    threshold = float(
        sel["selected_threshold"]
    )

    for split in [
        "DEV_OOF",
        "TEST_2025",
        "STRESS_2026",
    ]:
        q = predictions[
            predictions["recipe"].eq(recipe)
            & predictions["split"].eq(split)
        ].copy()

        m = classification_metrics(
            q["y"],
            q["prob"],
            threshold,
        )

        am = auc_metrics(
            q["y"],
            q["prob"],
        )

        class_rows.append({
            "recipe": recipe,
            "strategy": strategy,
            "split": split,
            "n": len(q),
            **am,
            **m,
        })

classification_results = pd.DataFrame(
    class_rows
)

display(
    classification_results.sort_values(
        ["strategy", "split", "f1"],
        ascending=[True, True, False],
    )
)

## PART I — P 대비 Paired Bootstrap

In [ ]:
# 19. Paired bootstrap helper
def paired_bootstrap_vs_p(
    pred_df,
    recipe,
    split,
    threshold_base,
    threshold_new,
    n_boot=BOOT_N,
    seed=BOOT_SEED,
):
    base = pred_df[
        pred_df["recipe"].eq("P")
        & pred_df["split"].eq(split)
    ][
        ["event_id", "y", "prob"]
    ].rename(
        columns={"prob": "prob_base"}
    )

    new = pred_df[
        pred_df["recipe"].eq(recipe)
        & pred_df["split"].eq(split)
    ][
        ["event_id", "y", "prob"]
    ].rename(
        columns={"prob": "prob_new"}
    )

    z = base.merge(
        new,
        on=["event_id", "y"],
        how="inner",
        validate="one_to_one",
    )

    y = z["y"].to_numpy()
    pb = z["prob_base"].to_numpy()
    pn = z["prob_new"].to_numpy()

    def metrics_pair(y_, pb_, pn_):
        out = {}

        out["roc_auc"] = (
            roc_auc_score(y_, pn_)
            - roc_auc_score(y_, pb_)
        )

        out["pr_auc"] = (
            average_precision_score(y_, pn_)
            - average_precision_score(y_, pb_)
        )

        mb = classification_metrics(
            y_, pb_, threshold_base
        )
        mn = classification_metrics(
            y_, pn_, threshold_new
        )

        for k in [
            "precision",
            "recall",
            "f1",
            "balanced_accuracy",
        ]:
            out[k] = mn[k] - mb[k]

        return out

    obs = metrics_pair(y, pb, pn)

    rng = np.random.default_rng(seed)
    idx = np.arange(len(z))

    boots = {
        k: []
        for k in obs
    }

    for _ in range(n_boot):
        samp = rng.choice(
            idx,
            size=len(idx),
            replace=True,
        )

        ys = y[samp]

        if len(np.unique(ys)) < 2:
            continue

        d = metrics_pair(
            ys,
            pb[samp],
            pn[samp],
        )

        for k, v in d.items():
            boots[k].append(v)

    rows = []

    for metric, observed in obs.items():
        arr = np.asarray(
            boots[metric],
            dtype=float,
        )

        lo, hi = np.quantile(
            arr,
            [0.025, 0.975],
        )

        p = 2 * min(
            np.mean(arr <= 0),
            np.mean(arr >= 0),
        )

        rows.append({
            "recipe": recipe,
            "split": split,
            "metric": metric,
            "delta_vs_P": observed,
            "ci95_low": lo,
            "ci95_high": hi,
            "bootstrap_p": p,
            "n": len(z),
        })

    return pd.DataFrame(rows)

In [ ]:
# 20. Bootstrap for F1_MAX threshold
bootstrap_rows = []

base_threshold = float(
    threshold_selection[
        threshold_selection["recipe"].eq("P")
        & threshold_selection["strategy"].eq("F1_MAX")
    ]["selected_threshold"].iloc[0]
)

for recipe in RECIPES:
    if recipe == "P":
        continue

    new_threshold = float(
        threshold_selection[
            threshold_selection["recipe"].eq(recipe)
            & threshold_selection["strategy"].eq("F1_MAX")
        ]["selected_threshold"].iloc[0]
    )

    for split in [
        "DEV_OOF",
        "TEST_2025",
        "STRESS_2026",
    ]:
        out = paired_bootstrap_vs_p(
            predictions,
            recipe=recipe,
            split=split,
            threshold_base=base_threshold,
            threshold_new=new_threshold,
            n_boot=BOOT_N,
            seed=BOOT_SEED,
        )

        bootstrap_rows.append(out)

bootstrap_results = pd.concat(
    bootstrap_rows,
    ignore_index=True,
)

display(
    bootstrap_results[
        bootstrap_results["metric"].eq(
            "roc_auc"
        )
    ].sort_values(
        ["split", "delta_vs_P"],
        ascending=[True, False],
    )
)

## PART J — 최종 비교표

이 표를 가장 중요하게 보면 됩니다.

- AUC / PR-AUC가 개선되는가?
- Recall / Precision / F1이 개선되는가?
- 2025와 2026에서 같은 방향인가?
- Bootstrap CI가 0을 벗어나는가?

In [ ]:
# 21. Compact final comparison
f1max = classification_results[
    classification_results["strategy"].eq(
        "F1_MAX"
    )
].copy()

wide_metrics = []

for recipe in RECIPES:
    row = {
        "recipe": recipe,
        "feature_n": len(RECIPES[recipe]),
    }

    a = auc_summary[
        auc_summary["recipe"].eq(recipe)
    ].iloc[0]

    row["DEV_WF_AUC"] = a["DEV_WF_AUC"]
    row["DEV_delta_vs_P"] = a["DEV_delta_vs_P"]

    for split, prefix in [
        ("TEST_2025", "TEST25"),
        ("STRESS_2026", "STRESS26"),
    ]:
        q = f1max[
            f1max["recipe"].eq(recipe)
            & f1max["split"].eq(split)
        ].iloc[0]

        row[f"{prefix}_AUC"] = q["roc_auc"]
        row[f"{prefix}_PR_AUC"] = q["pr_auc"]
        row[f"{prefix}_Precision"] = q["precision"]
        row[f"{prefix}_Recall"] = q["recall"]
        row[f"{prefix}_F1"] = q["f1"]
        row[f"{prefix}_BA"] = q["balanced_accuracy"]

    wide_metrics.append(row)

final_comparison = pd.DataFrame(
    wide_metrics
)

display(
    final_comparison.sort_values(
        "DEV_WF_AUC",
        ascending=False,
    )
)

In [ ]:
# 22. Bootstrap significance summary
sig_summary = (
    bootstrap_results
    .pivot_table(
        index=[
            "recipe",
            "metric",
        ],
        columns="split",
        values=[
            "delta_vs_P",
            "ci95_low",
            "ci95_high",
            "bootstrap_p",
        ],
        aggfunc="first",
    )
)

display(sig_summary)

## PART K — AUC 시각화

In [ ]:
# 23. AUC chart
plot_df = final_comparison.copy()

fig, ax = plt.subplots(figsize=(11,6))

x = np.arange(len(plot_df))
w = 0.25

ax.bar(
    x - w,
    plot_df["DEV_WF_AUC"],
    width=w,
    label="DEV WF",
)

ax.bar(
    x,
    plot_df["TEST25_AUC"],
    width=w,
    label="2025",
)

ax.bar(
    x + w,
    plot_df["STRESS26_AUC"],
    width=w,
    label="2026 Stress",
)

ax.axhline(
    0.5,
    linewidth=1,
)

ax.set_xticks(x)
ax.set_xticklabels(
    plot_df["recipe"],
    rotation=45,
    ha="right",
)
ax.set_ylim(0.5, 0.9)
ax.set_ylabel("ROC-AUC")
ax.set_title(
    "RF @ T+5 — Attention / Market / Short Incremental Effect"
)
ax.legend()

plt.tight_layout()
plt.show()

## PART L — Recall / Precision 시각화

In [ ]:
# 24. Future classification chart
for split in ["TEST_2025", "STRESS_2026"]:
    q = f1max[
        f1max["split"].eq(split)
    ].copy()

    q = q.set_index("recipe").loc[
        list(RECIPES.keys())
    ].reset_index()

    fig, ax = plt.subplots(figsize=(11,6))

    x = np.arange(len(q))
    w = 0.25

    ax.bar(
        x - w,
        q["precision"],
        width=w,
        label="Precision",
    )

    ax.bar(
        x,
        q["recall"],
        width=w,
        label="Recall",
    )

    ax.bar(
        x + w,
        q["f1"],
        width=w,
        label="F1",
    )

    ax.set_xticks(x)
    ax.set_xticklabels(
        q["recipe"],
        rotation=45,
        ha="right",
    )
    ax.set_ylim(0,1)
    ax.set_ylabel("Score")
    ax.set_title(
        f"RF @ T+5 Classification — {split}"
    )
    ax.legend()

    plt.tight_layout()
    plt.show()

## PART M — 자동 해석용 채택표

In [ ]:
# 25. Simple decision helper
# 주의: 최종 판단을 자동화하기보다 빠른 요약용입니다.

decision_rows = []

base = final_comparison[
    final_comparison["recipe"].eq("P")
].iloc[0]

for _, r in final_comparison.iterrows():
    if r["recipe"] == "P":
        continue

    dev_delta = (
        r["DEV_WF_AUC"]
        - base["DEV_WF_AUC"]
    )

    t25_delta = (
        r["TEST25_AUC"]
        - base["TEST25_AUC"]
    )

    s26_delta = (
        r["STRESS26_AUC"]
        - base["STRESS26_AUC"]
    )

    signs = [
        dev_delta > 0,
        t25_delta > 0,
        s26_delta > 0,
    ]

    auc_boot = bootstrap_results[
        bootstrap_results["recipe"].eq(
            r["recipe"]
        )
        & bootstrap_results["metric"].eq(
            "roc_auc"
        )
    ]

    sig_positive_n = int(
        (
            (auc_boot["ci95_low"] > 0)
            & (
                auc_boot["delta_vs_P"]
                > 0
            )
        ).sum()
    )

    if all(signs) and sig_positive_n >= 2:
        decision = "강한 채택 후보"
    elif sum(signs) >= 2:
        decision = "보조/추가 검토"
    else:
        decision = "예측모델에서는 기각 후보"

    decision_rows.append({
        "recipe": r["recipe"],
        "DEV_delta_AUC": dev_delta,
        "TEST25_delta_AUC": t25_delta,
        "STRESS26_delta_AUC": s26_delta,
        "positive_splits": sum(signs),
        "significant_positive_splits":
            sig_positive_n,
        "decision_hint": decision,
    })

decision_table = pd.DataFrame(
    decision_rows
)

display(
    decision_table.sort_values(
        [
            "positive_splits",
            "DEV_delta_AUC",
        ],
        ascending=[False, False],
    )
)

## PART N — 저장

In [ ]:
# 26. Save outputs
paths = {
    "recipe_summary":
        OUT_DIR / "recipe_summary.csv",

    "fold_results":
        OUT_DIR / "fold_results.csv",

    "predictions":
        OUT_DIR / "predictions.parquet",

    "auc_summary":
        OUT_DIR / "auc_summary.csv",

    "threshold_selection":
        OUT_DIR / "threshold_selection.csv",

    "classification_results":
        OUT_DIR / "classification_results.csv",

    "paired_bootstrap":
        OUT_DIR / "paired_bootstrap_vs_P.csv",

    "final_comparison":
        OUT_DIR / "final_comparison.csv",

    "decision_table":
        OUT_DIR / "decision_table.csv",
}

recipe_summary.to_csv(
    paths["recipe_summary"],
    index=False,
    encoding="utf-8-sig",
)

fold_results.to_csv(
    paths["fold_results"],
    index=False,
    encoding="utf-8-sig",
)

predictions.to_parquet(
    paths["predictions"],
    index=False,
    compression="zstd",
)

auc_summary.to_csv(
    paths["auc_summary"],
    index=False,
    encoding="utf-8-sig",
)

threshold_selection.to_csv(
    paths["threshold_selection"],
    index=False,
    encoding="utf-8-sig",
)

classification_results.to_csv(
    paths["classification_results"],
    index=False,
    encoding="utf-8-sig",
)

bootstrap_results.to_csv(
    paths["paired_bootstrap"],
    index=False,
    encoding="utf-8-sig",
)

final_comparison.to_csv(
    paths["final_comparison"],
    index=False,
    encoding="utf-8-sig",
)

decision_table.to_csv(
    paths["decision_table"],
    index=False,
    encoding="utf-8-sig",
)

print("saved:", OUT_DIR)

for k, p in paths.items():
    print(k, "->", p)

In [ ]:
# 27. DuckDB meta save
def replace_table(name, df):
    temp = "_tmp_" + name.replace(".", "_")
    con.register(temp, df)

    con.execute(
        f"""
        CREATE OR REPLACE TABLE {name}
        AS
        SELECT * FROM {temp}
        """
    )

    con.unregister(temp)

replace_table(
    "meta.step6c_att_market_short_auc",
    auc_summary,
)

replace_table(
    "meta.step6c_att_market_short_thresholds",
    threshold_selection,
)

replace_table(
    "meta.step6c_att_market_short_classification",
    classification_results,
)

replace_table(
    "meta.step6c_att_market_short_bootstrap",
    bootstrap_results,
)

replace_table(
    "meta.step6c_att_market_short_decision",
    decision_table,
)

con.execute("CHECKPOINT")

print("✅ DuckDB meta 저장")

In [ ]:
# 심사용 실행에서는 팀 GCS에 업로드하지 않습니다 — 이미 로컬(OUT_DIR)에
# 저장되어 있으므로 그 사실만 확인합니다.
for p in paths.values():
    assert p.exists(), p

print("✅ 로컬 저장 확인 완료")

## PART O — Final QA

In [ ]:
# 29. Final QA
checks = []

checks.append({
    "check": "8 recipes exist",
    "passed": len(RECIPES) == 8,
})

checks.append({
    "check": "T+5 only",
    "passed":
        model_frame["prediction_offset"]
        .eq(5).all(),
})

checks.append({
    "check": "unique event",
    "passed":
        model_frame["event_id"].is_unique,
})

checks.append({
    "check": "attention features exist",
    "passed": len(GROUPS["ATT"]) > 0,
})

checks.append({
    "check": "market features exist",
    "passed": len(GROUPS["MARKET"]) > 0,
})

checks.append({
    "check": "short features exist",
    "passed": len(GROUPS["SHORT"]) > 0,
})

checks.append({
    "check": "P baseline reproduced",
    "passed":
        abs(dev_auc - 0.781271) < 0.03
        and abs(test25_auc - 0.786934) < 0.03
        and abs(stress26_auc - 0.778200) < 0.03,
})

checks.append({
    "check": "2025 predictions exist",
    "passed":
        predictions["split"]
        .eq("TEST_2025").any(),
})

checks.append({
    "check": "2026 stress predictions exist",
    "passed":
        predictions["split"]
        .eq("STRESS_2026").any(),
})

validation = pd.DataFrame(checks)

display(validation)

assert validation["passed"].all()

print("✅ STEP 6C validation passed")

# STAGE 18: STEP 6D — T+7 / T+10 모델 최종확정 v2

**태그:** 최종

**원본 노트북:** `6D__T7_T10_모델_최종확정_v2.ipynb`

> STEP 6의 model_predictions.parquet을 재사용해 모델 연구/최종화만 수행합니다.

## FinDA STEP 6D — T+7 / T+10 Model Finalization

### 이 노트북의 범위

이 노트북은 **모델 연구/최종화만** 수행합니다.

#### 수행
1. 기존 STEP 6 `model_predictions.parquet` 재사용
2. `RF + P @ T+7`, `RF + P @ T+10` DEV OOF Classification Threshold 탐색
3. `F1_MAX`, `RECALL_80` 비교
4. Threshold를 고정해 2025 / 2026 평가
5. T+5 / T+7 / T+10 AUC 개선폭 정량화
6. Paired event-level bootstrap
7. T+7 / T+10 Frozen RandomForest model artifact 생성
8. 연구결과 CSV / Manifest / DuckDB meta 저장

#### 하지 않음
- Dashboard table 생성
- Frontend용 JSON 생성
- Serving 폴더 구성
- UI 상태머신 생성

Serving용 산출물은 **별도 STEP 7 Notebook**에서 만듭니다.

---

### Threshold 두 종류

#### Target q35
실제 정답을 만드는 기준입니다.

```text
Direct T+20 BHAR <= Train-period q35 → y=1
```

#### Classification Threshold
학습된 RF가 출력한 risk score를 위험/비위험으로 바꾸는 운영 기준입니다.

```text
prob >= threshold → 위험
```

즉 **Threshold로 RandomForest를 학습하는 것이 아닙니다.**

In [ ]:
# 1. packages / imports
!pip -q install pyarrow scikit-learn imbalanced-learn joblib duckdb

from pathlib import Path
from datetime import datetime
import json, hashlib, subprocess, warnings
import joblib
import duckdb
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    balanced_accuracy_score, accuracy_score,
    confusion_matrix, brier_score_loss
)
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 360)
print("✅ imports 완료")

In [ ]:
# 2. Paths / fixed policy

PROJECT_ID = GCP_PROJECT_ID
BUCKET = GCS_BUCKET_NAME

ROOT = Path(DRIVE_REFACTOR_ROOT)
STEP6_OUT = ROOT / "outputs" / "step6_direct_t20_timing"
STEP6B_OUT = ROOT / "outputs" / "step6b_classification_metrics"
DERIVED_DIR = ROOT / "derived" / "events"

OUT_DIR = ROOT / "outputs" / "step6d_t7_t10_models"
MODEL_DIR = OUT_DIR / "models"
META_DIR = ROOT / "metadata"
DB_PATH = ROOT / "duckdb" / "finda.duckdb"

for p in [OUT_DIR, MODEL_DIR, META_DIR]:
    p.mkdir(parents=True, exist_ok=True)

PRED_PATH = STEP6_OUT / "model_predictions.parquet"
FRAME_PATH = DERIVED_DIR / "direct_t20_model_frame_timing.parquet"
STEP6B_THRESHOLD_PATH = STEP6B_OUT / "threshold_selection.csv"

TIMINGS = [7, 10]
REFERENCE_TIMING = 5

TARGET_RECALL = 0.80
MAIN_THRESHOLD_STRATEGY = "F1_MAX"
BOOT_N = 3000
BOOT_SEED = 20260817
UPLOAD_GCS = False   # 최종 QA 후 True로 변경 가능

RF_PARAMS = dict(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

P_T = [
    "ret_1d_T","ret_5d_T","ret_10d_T","ret_20d_T",
    "volatility_5d_T","volatility_20d_T","volatility_60d_T",
    "volume_ratio_prior20_T","trading_value_ratio_prior20_T",
    "turnover_T","high_low_range_T","up_day_share_10_T",
    "max_daily_return_10_T",
]
P_POST = [
    "early_return","early_peak_return","early_giveback_ratio",
    "early_true_mdd","volume_sustain","volume_change_vs_T",
]
P_FEATURES = P_T + P_POST

print("PRED :", PRED_PATH)
print("FRAME:", FRAME_PATH)
print("OUT  :", OUT_DIR)

In [ ]:
# 3. 로컬에 없으면(예: 이 STAGE만 단독 실행한 경우) 공개 버킷에서 로그인 없이 복구

if not PRED_PATH.exists():
    STEP6_OUT.mkdir(parents=True, exist_ok=True)
    gcs_public_download("derived/modeling_direct_t20/model_predictions.parquet", PRED_PATH)

if not FRAME_PATH.exists():
    DERIVED_DIR.mkdir(parents=True, exist_ok=True)
    gcs_public_download("derived/events/direct_t20_model_frame_timing.parquet", FRAME_PATH)

assert PRED_PATH.exists(), PRED_PATH
assert FRAME_PATH.exists(), FRAME_PATH
print("✅ input artifacts ready")

### PART A — 기존 STEP 6 결과 QA

T+7 / T+10 모델의 **검증용 Prediction은 STEP 6에서 이미 생성한 OOF/OOT Prediction을 재사용**합니다.

따라서 Threshold를 찾기 위해 Event / Snapshot / Target을 다시 만들지 않습니다.

In [ ]:
# 4. Load predictions / model frame

pred = pd.read_parquet(PRED_PATH)
frame = pd.read_parquet(FRAME_PATH)

pred["event_date"] = pd.to_datetime(pred["event_date"], errors="coerce").dt.normalize()
pred["prediction_offset"] = pd.to_numeric(pred["prediction_offset"], errors="coerce").astype(int)

for c in ["event_date","cutoff_date","direct_t20_date","target_start_date","target_end_date"]:
    if c in frame.columns:
        frame[c] = pd.to_datetime(frame[c], errors="coerce").dt.normalize()

frame["prediction_offset"] = pd.to_numeric(frame["prediction_offset"], errors="coerce").astype(int)

required_pred = {
    "prediction_offset","recipe","model","seed","split",
    "event_id","event_date","y","prob","label_q35"
}
missing = sorted(required_pred - set(pred.columns))
assert not missing, missing

required_frame = {
    "event_id","ticker","event_date","prediction_offset",
    "cutoff_date","direct_t20_date","direct_bhar_T20"
}
missing = sorted(required_frame - set(frame.columns))
assert not missing, missing

rfp = pred[
    pred["prediction_offset"].isin([5,7,10])
    & pred["recipe"].eq("P")
    & pred["model"].eq("rf")
].copy()

qa = (
    rfp.groupby(["prediction_offset","split"])
    .agg(rows=("event_id","size"), events=("event_id","nunique"))
    .reset_index()
)
display(qa)

for off in [5,7,10]:
    got = set(rfp.loc[rfp["prediction_offset"].eq(off), "split"].astype(str))
    assert {"DEV_OOF","TEST_2025","STRESS_2026"}.issubset(got), (off, got)

print("✅ RF+P T+5/T+7/T+10 prediction QA")

In [ ]:
# 5. Event-level prediction helper

def get_event_pred(offset, split):
    q = pred[
        pred["prediction_offset"].eq(offset)
        & pred["recipe"].eq("P")
        & pred["model"].eq("rf")
        & pred["split"].eq(split)
    ].copy()

    q = (
        q.groupby(["event_id","event_date","y"], as_index=False)
        .agg(
            prob=("prob","mean"),
            label_q35=("label_q35","mean"),
            seed_n=("seed","nunique"),
        )
    )
    assert not q["event_id"].duplicated().any()
    return q

event_qa = []
for off in [5,7,10]:
    for split in ["DEV_OOF","TEST_2025","STRESS_2026"]:
        q = get_event_pred(off, split)
        event_qa.append({
            "offset":off,
            "split":split,
            "n":len(q),
            "positive_rate":q["y"].mean(),
            "seed_n_max":q["seed_n"].max(),
        })

event_qa = pd.DataFrame(event_qa)
display(event_qa)
assert (event_qa["n"] > 0).all()

### PART B — T+7 / T+10 Classification Threshold

Threshold는 **각 Timing의 DEV OOF에서만** 선택합니다.

- `F1_MAX`: F1 최대
- `RECALL_80`: Recall ≥ 80% 중 Precision 최대

2025 / 2026은 Threshold 선택에 절대 사용하지 않습니다.

In [ ]:
# 6. Metric / threshold helpers

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    return tn / (tn + fp) if (tn + fp) else np.nan

def metric_at_threshold(y, prob, threshold):
    y = np.asarray(y).astype(int)
    prob = np.asarray(prob).astype(float)
    yhat = (prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0,1]).ravel()

    return {
        "n":len(y),
        "positive_n":int(y.sum()),
        "positive_rate":float(y.mean()),
        "threshold":float(threshold),
        "roc_auc":roc_auc_score(y, prob) if len(np.unique(y)) == 2 else np.nan,
        "pr_auc":average_precision_score(y, prob) if len(np.unique(y)) == 2 else np.nan,
        "accuracy":accuracy_score(y, yhat),
        "balanced_accuracy":balanced_accuracy_score(y, yhat),
        "precision":precision_score(y, yhat, zero_division=0),
        "recall":recall_score(y, yhat, zero_division=0),
        "specificity":specificity_score(y, yhat),
        "f1":f1_score(y, yhat, zero_division=0),
        "brier":brier_score_loss(y, prob),
        "tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp),
    }

def threshold_table(y, prob):
    thresholds = np.unique(
        np.r_[np.linspace(0.05,0.95,181), np.asarray(prob)]
    )
    rows = []
    for t in thresholds:
        m = metric_at_threshold(y, prob, t)
        rows.append({
            "threshold":float(t),
            "precision":m["precision"],
            "recall":m["recall"],
            "f1":m["f1"],
            "specificity":m["specificity"],
            "balanced_accuracy":m["balanced_accuracy"],
            "pred_positive_n":m["tp"]+m["fp"],
        })
    return pd.DataFrame(rows).sort_values("threshold").reset_index(drop=True)

def choose_f1_max(tab):
    return tab.sort_values(
        ["f1","recall","precision","threshold"],
        ascending=[False,False,False,False]
    ).iloc[0]

def choose_recall_target(tab, target=0.80):
    eligible = tab[tab["recall"] >= target].copy()
    if len(eligible) == 0:
        return tab.sort_values(
            ["recall","precision","threshold"],
            ascending=[False,False,False]
        ).iloc[0]
    return eligible.sort_values(
        ["precision","f1","threshold"],
        ascending=[False,False,False]
    ).iloc[0]

In [ ]:
# 7. T+7 / T+10 threshold selection

threshold_rows = []
threshold_curves = {}

for off in TIMINGS:
    dev = get_event_pred(off, "DEV_OOF")
    tab = threshold_table(dev["y"], dev["prob"])
    threshold_curves[off] = tab

    f1_row = choose_f1_max(tab)
    r80_row = choose_recall_target(tab, TARGET_RECALL)

    for strategy, row in [("F1_MAX", f1_row), ("RECALL_80", r80_row)]:
        threshold_rows.append({
            "candidate":f"RF_P_T{off}",
            "prediction_offset":off,
            "strategy":strategy,
            "selected_threshold":float(row["threshold"]),
            "DEV_precision":float(row["precision"]),
            "DEV_recall":float(row["recall"]),
            "DEV_f1":float(row["f1"]),
            "DEV_specificity":float(row["specificity"]),
            "DEV_balanced_accuracy":float(row["balanced_accuracy"]),
        })

threshold_selection = pd.DataFrame(threshold_rows)
display(threshold_selection)

# T+5 reference threshold: 기존 STEP6B 파일 우선, 없으면 DEV OOF 재현
t5_threshold = None
if STEP6B_THRESHOLD_PATH.exists():
    old = pd.read_csv(STEP6B_THRESHOLD_PATH)
    q = old[
        old["candidate"].eq("RF_P_T5")
        & old["strategy"].eq("F1_MAX")
    ]
    if len(q):
        t5_threshold = float(q["selected_threshold"].iloc[0])

if t5_threshold is None:
    dev5 = get_event_pred(5, "DEV_OOF")
    t5_threshold = float(
        choose_f1_max(threshold_table(dev5["y"], dev5["prob"]))["threshold"]
    )

print("T+5 reference F1_MAX threshold:", t5_threshold)
assert abs(t5_threshold - 0.496175) < 0.01, (
    "기존 T+5 threshold와 크게 다릅니다. STEP6 prediction 버전을 확인하세요."
)

### PART C — Threshold 고정 미래 평가

각 Timing에서 DEV OOF로 선택한 Threshold를 2025 / 2026에 그대로 적용합니다.

In [ ]:
# 8. Fixed threshold evaluation

eval_rows = []

for off in TIMINGS:
    for strategy in ["F1_MAX","RECALL_80"]:
        t = float(
            threshold_selection[
                threshold_selection["prediction_offset"].eq(off)
                & threshold_selection["strategy"].eq(strategy)
            ]["selected_threshold"].iloc[0]
        )

        for split in ["DEV_OOF","TEST_2025","STRESS_2026"]:
            q = get_event_pred(off, split)
            m = metric_at_threshold(q["y"], q["prob"], t)
            eval_rows.append({
                "candidate":f"RF_P_T{off}",
                "prediction_offset":off,
                "strategy":strategy,
                "split":split,
                **m,
            })

classification_metrics = pd.DataFrame(eval_rows)
display(
    classification_metrics[[
        "candidate","strategy","split","threshold",
        "roc_auc","pr_auc","precision","recall","f1",
        "specificity","balanced_accuracy","tn","fp","fn","tp"
    ]]
)

### PART D — T+5 / T+7 / T+10 개선폭 정량화

T+5를 메인으로 유지하는 근거와 T+10을 추가 checkpoint로 두는 근거를 같은 표에서 확인합니다.

In [ ]:
# 9. Timing AUC comparison
# ------------------------------------------------------------------------------
# [수정] "DEV"는 STEP 6(STAGE 15)의 development_summary.csv에 저장된
# DEV_WF_AUC(walk-forward fold 평균)를 그대로 재사용합니다. 예전 버전은 이 셀에서
# DEV_OOF_AUC(fold 예측을 합친 pooled 값)를 새로 계산해 "DEV"라는 이름으로 같이
# 썼는데, STEP 6 문서(Direct T+20 BHAR Timing/Ablation 설계·결과 보고서)가 실제로
# 보고하는 "DEV"는 전부 DEV_WF 기준이라 서로 다른 값이 섞여 있었습니다.
# (2025/2026 Stress는 원래도 pooled 단일 평가라 이 문제가 없어 그대로 둡니다.)
DEV_WF_PATH = STEP6_OUT / "development_summary.csv"
if not DEV_WF_PATH.exists():
    gcs_public_download("derived/modeling_direct_t20/development_summary.csv", DEV_WF_PATH)

dev_wf_summary = pd.read_csv(DEV_WF_PATH)
dev_wf_p_rf = dev_wf_summary[
    dev_wf_summary["recipe"].eq("P") & dev_wf_summary["model"].eq("rf")
].set_index("prediction_offset")["dev_auc_mean"]

rows = []
for off in [5,7,10]:
    row = {"timing":f"T+{off}", "prediction_offset":off}
    row["DEV_WF_AUC"] = float(dev_wf_p_rf.loc[off]) if off in dev_wf_p_rf.index else np.nan
    for split in ["DEV_OOF","TEST_2025","STRESS_2026"]:
        q = get_event_pred(off, split)
        row[f"{split}_AUC"] = roc_auc_score(q["y"], q["prob"])
        row[f"{split}_PR_AUC"] = average_precision_score(q["y"], q["prob"])
    rows.append(row)

timing_auc = pd.DataFrame(rows)

# STEP 6 문서와 비교 가능한 "보고용 DEV" 델타는 DEV_WF_AUC 기준으로 계산합니다.
# DEV_OOF_AUC 기준 델타는 threshold 선정 등 다른 용도로 별도 보관합니다(컬럼명에 OOF 명시).
for split, dev_col in [("DEV_WF", "DEV_WF_AUC"), ("DEV_OOF", "DEV_OOF_AUC"),
                        ("TEST_2025", "TEST_2025_AUC"), ("STRESS_2026", "STRESS_2026_AUC")]:
    t5 = float(timing_auc.loc[timing_auc["prediction_offset"].eq(5), dev_col].iloc[0])
    t7 = float(timing_auc.loc[timing_auc["prediction_offset"].eq(7), dev_col].iloc[0])
    t10 = float(timing_auc.loc[timing_auc["prediction_offset"].eq(10), dev_col].iloc[0])

    timing_auc[f"{split}_delta_vs_T5"] = timing_auc[dev_col] - t5
    timing_auc.loc[:, f"{split}_T5_to_T7"] = t7 - t5
    timing_auc.loc[:, f"{split}_T7_to_T10"] = t10 - t7
    timing_auc.loc[:, f"{split}_T5_to_T10"] = t10 - t5

display(timing_auc)

print("\n=== 정량 요약 (보고 기준: DEV_WF) ===")
for split in ["DEV_WF","DEV_OOF","TEST_2025","STRESS_2026"]:
    print(
        split,
        "| T5→T7 =", round(float(timing_auc[f"{split}_T5_to_T7"].iloc[0]), 6),
        "| T7→T10 =", round(float(timing_auc[f"{split}_T7_to_T10"].iloc[0]), 6),
        "| T5→T10 =", round(float(timing_auc[f"{split}_T5_to_T10"].iloc[0]), 6),
    )

In [ ]:
# 10. Paired event-level bootstrap AUC

def paired_auc_bootstrap(offset_a, offset_b, split, n_boot=3000, seed=20260817):
    a = get_event_pred(offset_a, split)[["event_id","y","prob"]].rename(
        columns={"prob":"prob_a"}
    )
    b = get_event_pred(offset_b, split)[["event_id","y","prob"]].rename(
        columns={"prob":"prob_b"}
    )
    z = a.merge(b, on=["event_id","y"], how="inner", validate="one_to_one")
    assert len(z) > 0 and z["y"].nunique() == 2

    auc_a = roc_auc_score(z["y"], z["prob_a"])
    auc_b = roc_auc_score(z["y"], z["prob_b"])
    obs = auc_b - auc_a

    rng = np.random.default_rng(seed)
    idx = np.arange(len(z))
    diffs = []

    for _ in range(n_boot):
        samp = rng.choice(idx, size=len(idx), replace=True)
        s = z.iloc[samp]
        if s["y"].nunique() < 2:
            continue
        diffs.append(
            roc_auc_score(s["y"], s["prob_b"])
            - roc_auc_score(s["y"], s["prob_a"])
        )

    diffs = np.asarray(diffs)
    lo, hi = np.quantile(diffs, [0.025,0.975])
    p = 2 * min(np.mean(diffs <= 0), np.mean(diffs >= 0))

    return {
        "split":split,
        "comparison":f"T+{offset_b} - T+{offset_a}",
        "n":len(z),
        "auc_a":auc_a,
        "auc_b":auc_b,
        "delta_auc":obs,
        "ci95_low":lo,
        "ci95_high":hi,
        "bootstrap_p":min(float(p),1.0),
    }

bootstrap_rows = []
for split in ["TEST_2025","STRESS_2026"]:
    for a,b in [(5,7),(7,10),(5,10)]:
        bootstrap_rows.append(
            paired_auc_bootstrap(a,b,split,BOOT_N,BOOT_SEED)
        )

paired_bootstrap_timing = pd.DataFrame(bootstrap_rows)
display(paired_bootstrap_timing)

### PART E — T+7 / T+10 Frozen Model Artifact 생성

검증 성능은 위의 OOF/OOT Prediction으로 판단합니다.

여기서는 **향후 재현/추론을 위한 연구용 Frozen Model Artifact**를 생성합니다.

두 cutoff를 저장합니다.

- `dev2024`: 2025 직전까지 학습 가능한 데이터
- `through2025`: 2026 직전까지 학습 가능한 데이터

이 파일은 아직 Dashboard Serving 파일이 아닙니다.

In [ ]:
# 11. Feature / leakage QA

usable_features = [
    c for c in P_FEATURES
    if c in frame.columns
    and pd.api.types.is_numeric_dtype(frame[c])
    and frame[c].notna().any()
]

bad = [
    c for c in usable_features
    if any(tok in c.lower() for tok in [
        "direct_bhar","target_","label","future_"
    ])
]
assert not bad, bad
assert len(usable_features) == len(P_FEATURES), (
    "P feature 일부가 없습니다.",
    sorted(set(P_FEATURES)-set(usable_features))
)

print("P feature n:", len(usable_features))
print(usable_features)

for off in TIMINGS:
    q = frame[frame["prediction_offset"].eq(off)]
    assert len(q) > 0
    assert (q["cutoff_date"] < q["direct_t20_date"]).all()
print("✅ Feature / timing leakage QA")

In [ ]:
# 12. Frozen model fit helper

def fit_frozen_model(offset, train_cutoff, threshold, artifact_name):
    train_cutoff = pd.Timestamp(train_cutoff)

    x = frame[
        frame["prediction_offset"].eq(offset)
        & (frame["event_date"] < train_cutoff)
        & (frame["direct_t20_date"] < train_cutoff)
    ].copy()

    assert len(x) > 0
    assert x["event_id"].is_unique

    q35 = float(x["direct_bhar_T20"].quantile(0.35))
    x["y"] = (x["direct_bhar_T20"] <= q35).astype(int)

    X = x[usable_features].replace([np.inf,-np.inf], np.nan)
    y = x["y"].to_numpy()

    imputer = SimpleImputer(strategy="median")
    X2 = imputer.fit_transform(X)

    counts = pd.Series(y).value_counts()
    Xfit, yfit = X2, y
    smote_used = False
    smote_k = None

    if len(counts) == 2 and int(counts.min()) >= 2:
        smote_k = min(5, int(counts.min()) - 1)
        Xfit, yfit = SMOTE(
            random_state=42,
            k_neighbors=smote_k
        ).fit_resample(X2, y)
        smote_used = True

    model = RandomForestClassifier(**RF_PARAMS)
    model.fit(Xfit, yfit)

    bundle = {
        "artifact_type":f"FinDA_RF_P_T{offset}_research_frozen",
        "built_at":datetime.now().isoformat(),
        "prediction_offset":int(offset),
        "target":"Direct T+20 BHAR",
        "label_rule":"direct_bhar_T20 <= train_period_q35",
        "label_q35":q35,
        "classification_threshold":float(threshold),
        "threshold_strategy":MAIN_THRESHOLD_STRATEGY,
        "train_cutoff_exclusive":str(train_cutoff.date()),
        "features":usable_features,
        "rf_params":RF_PARAMS,
        "imputer":imputer,
        "model":model,
        "smote_used":smote_used,
        "smote_k":smote_k,
        "train_n":int(len(x)),
        "train_positive_rate":float(x["y"].mean()),
    }

    path = MODEL_DIR / artifact_name
    joblib.dump(bundle, path)

    return {
        "prediction_offset":offset,
        "artifact":artifact_name,
        "train_cutoff_exclusive":str(train_cutoff.date()),
        "train_n":len(x),
        "label_q35":q35,
        "positive_rate":x["y"].mean(),
        "classification_threshold":threshold,
        "smote_used":smote_used,
        "smote_k":smote_k,
    }

main_thresholds = {
    off: float(
        threshold_selection[
            threshold_selection["prediction_offset"].eq(off)
            & threshold_selection["strategy"].eq(MAIN_THRESHOLD_STRATEGY)
        ]["selected_threshold"].iloc[0]
    )
    for off in TIMINGS
}

fit_rows = []
for off in TIMINGS:
    fit_rows.append(
        fit_frozen_model(
            off, "2025-01-01", main_thresholds[off],
            f"rf_p_t{off}_dev2024.joblib"
        )
    )
    fit_rows.append(
        fit_frozen_model(
            off, "2026-01-01", main_thresholds[off],
            f"rf_p_t{off}_through2025.joblib"
        )
    )

model_fit_summary = pd.DataFrame(fit_rows)
display(model_fit_summary)

### PART F — 연구 산출물 저장

Serving Asset은 만들지 않습니다.

이 Notebook의 최종 산출물은:
- Threshold 결과
- 미래검증 성능
- Timing 비교
- Paired Bootstrap
- T+7/T+10 Frozen Model Artifact
- 연구 Manifest

입니다.

In [ ]:
# 13. Save model research outputs

THRESHOLD_PATH = OUT_DIR / "t7_t10_threshold_selection.csv"
CLASS_PATH = OUT_DIR / "t7_t10_classification_metrics.csv"
TIMING_PATH = OUT_DIR / "t5_t7_t10_auc_comparison.csv"
BOOT_PATH = OUT_DIR / "paired_bootstrap_t5_t7_t10.csv"
FIT_PATH = OUT_DIR / "t7_t10_model_fit_summary.csv"
FROZEN_THRESHOLD_PATH = OUT_DIR / "frozen_timing_thresholds.csv"
MANIFEST_PATH = OUT_DIR / "step6d_t7_t10_model_manifest.json"

threshold_selection.to_csv(THRESHOLD_PATH, index=False, encoding="utf-8-sig")
classification_metrics.to_csv(CLASS_PATH, index=False, encoding="utf-8-sig")
timing_auc.to_csv(TIMING_PATH, index=False, encoding="utf-8-sig")
paired_bootstrap_timing.to_csv(BOOT_PATH, index=False, encoding="utf-8-sig")
model_fit_summary.to_csv(FIT_PATH, index=False, encoding="utf-8-sig")

frozen_thresholds = pd.DataFrame([
    {
        "timing":5,
        "candidate":"RF_P_T5",
        "strategy":"F1_MAX",
        "classification_threshold":float(t5_threshold),
        "source":"STEP6B / DEV OOF"
    },
    *[
        {
            "timing":off,
            "candidate":f"RF_P_T{off}",
            "strategy":"F1_MAX",
            "classification_threshold":float(main_thresholds[off]),
            "source":"STEP6D / DEV OOF"
        }
        for off in TIMINGS
    ]
])
frozen_thresholds.to_csv(
    FROZEN_THRESHOLD_PATH, index=False, encoding="utf-8-sig"
)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "built_at":datetime.now().isoformat(),
    "project_id":PROJECT_ID,
    "bucket":BUCKET,
    "scope":"T+7/T+10 model finalization only",
    "target":{
        "continuous":"direct_bhar_T20",
        "binary":"train-period q35",
        "quantile":0.35,
    },
    "recipe":"P",
    "model":"RandomForest",
    "rf_params":RF_PARAMS,
    "threshold_strategy":MAIN_THRESHOLD_STRATEGY,
    "thresholds":{
        f"T+{int(r.timing)}":float(r.classification_threshold)
        for r in frozen_thresholds.itertuples()
    },
    "features":usable_features,
    "files":{
        p.name:sha256_file(p)
        for p in [
            THRESHOLD_PATH, CLASS_PATH, TIMING_PATH,
            BOOT_PATH, FIT_PATH, FROZEN_THRESHOLD_PATH,
            *sorted(MODEL_DIR.glob("*.joblib"))
        ]
    }
}

with open(MANIFEST_PATH,"w",encoding="utf-8") as f:
    json.dump(manifest,f,ensure_ascii=False,indent=2)

print("✅ saved")
for p in [
    THRESHOLD_PATH, CLASS_PATH, TIMING_PATH,
    BOOT_PATH, FIT_PATH, FROZEN_THRESHOLD_PATH, MANIFEST_PATH,
    *sorted(MODEL_DIR.glob("*.joblib"))
]:
    print("-", p)

In [ ]:
# 14. Optional DuckDB meta + GCS upload

if DB_PATH.exists():
    con = duckdb.connect(str(DB_PATH))
    con.execute("CREATE SCHEMA IF NOT EXISTS meta")

    def replace_table(name, df):
        tmp = "_tmp_" + name.replace(".","_")
        con.register(tmp, df)
        con.execute(f"CREATE OR REPLACE TABLE {name} AS SELECT * FROM {tmp}")
        con.unregister(tmp)

    replace_table("meta.step6d_t7_t10_thresholds", frozen_thresholds)
    replace_table("meta.step6d_t7_t10_classification", classification_metrics)
    replace_table("meta.step6d_t7_t10_timing_auc", timing_auc)
    replace_table("meta.step6d_t7_t10_bootstrap", paired_bootstrap_timing)
    replace_table("meta.step6d_t7_t10_model_fit", model_fit_summary)
    con.close()
    print("✅ DuckDB meta saved")

if UPLOAD_GCS:
    from google.colab import auth
    auth.authenticate_user()
    !gcloud -q config set project {PROJECT_ID}

    gcs_root = f"gs://{BUCKET}/derived/model_final_t7_t10"
    for p in [
        THRESHOLD_PATH, CLASS_PATH, TIMING_PATH,
        BOOT_PATH, FIT_PATH, FROZEN_THRESHOLD_PATH, MANIFEST_PATH,
        *sorted(MODEL_DIR.glob("*.joblib"))
    ]:
        subprocess.run(
            ["gcloud","-q","storage","cp",str(p),f"{gcs_root}/{p.name}"],
            check=True
        )
    print("✅ GCS upload complete")
else:
    print("ℹ️ UPLOAD_GCS=False — Drive에만 저장")

# STAGE 19: STEP 7 — Serving / Dashboard Asset Refactor

**태그:** 최종

**원본 노트북:** `7__Serving_Layer_생성.ipynb`

> 모델을 새로 학습하거나 threshold를 새로 탐색하지 않습니다. STEP 6/6D 확정 산출물을 서빙용으로 변환합니다.

## FinDA STEP 7 — Serving / Dashboard Asset Refactor

### 이 노트북의 범위

이 노트북은 **모델을 학습하지 않습니다.**
Threshold도 새로 탐색하지 않습니다.

STEP 6 / STEP 6D에서 확정된 연구 산출물을 읽어 **Serving Layer**만 생성합니다.

#### 입력
- `model_predictions.parquet`
- `direct_t20_model_frame_timing.parquet`
- `frozen_timing_thresholds.csv`
- T+7/T+10 Frozen Model Artifact

#### 출력
- `model_thresholds.json`
- `dashboard_scores_t5_t7_t10.parquet/csv`
- `dashboard_event_timeline.parquet/csv`
- `event_stage_policy.json`
- `model_registry.json`
- T+7/T+10 stable serving model copy
- DuckDB `serving.*`
- GCS `serving/`

#### 금지
- RF 재학습
- Threshold 재탐색
- 2025/2026 결과를 보고 Threshold 변경
- 미래 Timing 결과 선노출 정책을 데이터에 하드코딩

즉 이 Notebook은 **Research Gold → Serving/Consumption Layer** 변환만 담당합니다.

In [ ]:
# 1. imports
!pip -q install pyarrow duckdb joblib

from pathlib import Path
from datetime import datetime
import json, hashlib, shutil, subprocess, warnings
import joblib
import duckdb
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 360)
print("✅ imports 완료")

In [ ]:
# 2. Paths

PROJECT_ID = GCP_PROJECT_ID
BUCKET = GCS_BUCKET_NAME

ROOT = Path(DRIVE_REFACTOR_ROOT)

STEP6_OUT = ROOT / "outputs" / "step6_direct_t20_timing"
STEP6D_OUT = ROOT / "outputs" / "step6d_t7_t10_models"
STEP6D_MODEL_DIR = STEP6D_OUT / "models"
DERIVED_DIR = ROOT / "derived" / "events"
DB_PATH = ROOT / "duckdb" / "finda.duckdb"

SERVING_ROOT = ROOT / "serving"
SERVING_MODEL_DIR = SERVING_ROOT / "models"
SERVING_DASH_DIR = SERVING_ROOT / "dashboard"
SERVING_CONFIG_DIR = SERVING_ROOT / "config"
SERVING_META_DIR = SERVING_ROOT / "metadata"

for p in [
    SERVING_MODEL_DIR, SERVING_DASH_DIR,
    SERVING_CONFIG_DIR, SERVING_META_DIR
]:
    p.mkdir(parents=True, exist_ok=True)

PRED_PATH = STEP6_OUT / "model_predictions.parquet"
FRAME_PATH = DERIVED_DIR / "direct_t20_model_frame_timing.parquet"
THRESHOLD_PATH = STEP6D_OUT / "frozen_timing_thresholds.csv"
MODEL_MANIFEST_PATH = STEP6D_OUT / "step6d_t7_t10_model_manifest.json"

UPLOAD_GCS = False   # 최종 QA 후 True

print(SERVING_ROOT)

In [ ]:
# 3. Input QA / GCS fallback for shared research files

# 로컬에 없으면(예: 이 STAGE만 단독 실행한 경우) 공개 버킷에서 로그인 없이 복구
if not PRED_PATH.exists():
    STEP6_OUT.mkdir(parents=True, exist_ok=True)
    gcs_public_download("derived/modeling_direct_t20/model_predictions.parquet", PRED_PATH)

if not FRAME_PATH.exists():
    DERIVED_DIR.mkdir(parents=True, exist_ok=True)
    gcs_public_download("derived/events/direct_t20_model_frame_timing.parquet", FRAME_PATH)

assert PRED_PATH.exists(), PRED_PATH
assert FRAME_PATH.exists(), FRAME_PATH
assert THRESHOLD_PATH.exists(), (
    f"{THRESHOLD_PATH} 없음. 먼저 STEP6D T7/T10 Model Finalization을 실행하세요."
)
assert MODEL_MANIFEST_PATH.exists(), MODEL_MANIFEST_PATH

required_models = [
    STEP6D_MODEL_DIR / "rf_p_t7_through2025.joblib",
    STEP6D_MODEL_DIR / "rf_p_t10_through2025.joblib",
]
for p in required_models:
    assert p.exists(), p

print("✅ serving inputs ready")

### PART A — Frozen Threshold 읽기

Serving 단계에서는 Threshold를 계산하지 않습니다.

`frozen_timing_thresholds.csv`의 값을 그대로 사용합니다.

In [ ]:
# 4. Load thresholds / validate

thresholds = pd.read_csv(THRESHOLD_PATH)

assert set([5,7,10]).issubset(set(thresholds["timing"].astype(int)))
assert thresholds["strategy"].eq("F1_MAX").all()
assert thresholds["classification_threshold"].between(0,1).all()

threshold_map = {
    int(r.timing): float(r.classification_threshold)
    for r in thresholds.itertuples()
}

print(threshold_map)
assert abs(threshold_map[5] - 0.496175) < 0.01
print("✅ frozen thresholds loaded")

### PART B — 검증 Prediction을 Dashboard Score Table로 변환

Historical Dashboard는 검증 완료된 OOF/OOT Prediction을 사용합니다.

즉:
- DEV → OOF score
- 2025 → Temporal Test score
- 2026 → Stress Test score

를 그대로 사용해 **한 Event 한 행**으로 만듭니다.

In [ ]:
# 5. Load predictions / model frame

pred = pd.read_parquet(PRED_PATH)
frame = pd.read_parquet(FRAME_PATH)

pred["event_date"] = pd.to_datetime(pred["event_date"], errors="coerce").dt.normalize()
pred["prediction_offset"] = pd.to_numeric(pred["prediction_offset"], errors="coerce").astype(int)

for c in ["event_date","cutoff_date","direct_t20_date","target_start_date","target_end_date"]:
    if c in frame.columns:
        frame[c] = pd.to_datetime(frame[c], errors="coerce").dt.normalize()
frame["prediction_offset"] = pd.to_numeric(frame["prediction_offset"], errors="coerce").astype(int)

rfp = pred[
    pred["prediction_offset"].isin([5,7,10])
    & pred["recipe"].eq("P")
    & pred["model"].eq("rf")
].copy()

assert len(rfp) > 0
print("prediction rows:", len(rfp))

In [ ]:
# 6. Event-level validated score

def event_scores(offset):
    q = rfp[rfp["prediction_offset"].eq(offset)].copy()

    q = (
        q.groupby(["event_id","event_date","split","y"], as_index=False)
        .agg(
            score=("prob","mean"),
            seed_n=("seed","nunique"),
        )
    )
    assert not q["event_id"].duplicated().any(), (
        "동일 Event가 여러 split에 존재합니다."
    )

    t = threshold_map[offset]
    q[f"t{offset}_score"] = q["score"]
    q[f"t{offset}_risk"] = (q["score"] >= t).astype("Int64")
    q[f"t{offset}_threshold"] = t
    q[f"t{offset}_split"] = q["split"]

    return q[[
        "event_id","event_date","y",
        f"t{offset}_score",
        f"t{offset}_risk",
        f"t{offset}_threshold",
        f"t{offset}_split",
    ]]

s5 = event_scores(5)
s7 = event_scores(7)
s10 = event_scores(10)

# actual y 동일성 QA
ycheck = (
    s5[["event_id","y"]].rename(columns={"y":"y5"})
    .merge(s7[["event_id","y"]].rename(columns={"y":"y7"}), on="event_id", how="outer")
    .merge(s10[["event_id","y"]].rename(columns={"y":"y10"}), on="event_id", how="outer")
)
same = ycheck.dropna(subset=["y5","y7","y10"])
assert (same["y5"].eq(same["y7"]) & same["y5"].eq(same["y10"])).all()

print("✅ T5/T7/T10 actual label parity")

In [ ]:
# 7. Build two separate assets
#
# A) dashboard_scores_validated:
#    - 모델 점수가 실제 OOF/OOT로 존재하는 Event만 포함
#    - T+5/T+7/T+10 score/risk/threshold에 결측이 없어야 함
#
# B) dashboard_event_catalog_all:
#    - 전체 1,250 Event metadata/timeline 보존
#    - 초기 WF train-only Event는 score가 없는 것이 정상이며,
#      별도 scoring_status로 명시

meta_cols = [
    c for c in ["event_id","ticker","ticker_name","event_date"]
    if c in frame.columns
]
event_meta = frame[meta_cols].drop_duplicates("event_id").copy()

cutoff = (
    frame[
        frame["prediction_offset"].isin([5,7,10])
    ][["event_id","prediction_offset","cutoff_date"]]
    .drop_duplicates(["event_id","prediction_offset"])
)

cutoff_wide = cutoff.pivot(
    index="event_id",
    columns="prediction_offset",
    values="cutoff_date"
).reset_index()

cutoff_wide.columns = [
    "event_id" if c == "event_id" else f"t{int(c)}_cutoff_date"
    for c in cutoff_wide.columns
]

actual_cols = [
    c for c in ["event_id","direct_t20_date","direct_bhar_T20"]
    if c in frame.columns
]
actual = frame[actual_cols].drop_duplicates("event_id").copy()

# -------- A. validated score table --------
# 동일 Event에 대해 세 Timing 모두 검증 score가 존재하는 사건만 사용.
validated = (
    s5.rename(columns={"y":"actual_t20_risk_q35_fold_specific"})
      .merge(
          s7.drop(columns=["event_date","y"]),
          on="event_id",
          how="inner",
          validate="one_to_one"
      )
      .merge(
          s10.drop(columns=["event_date","y"]),
          on="event_id",
          how="inner",
          validate="one_to_one"
      )
)

dashboard_scores_validated = (
    event_meta
    .merge(cutoff_wide, on="event_id", how="inner")
    .merge(validated.drop(columns=["event_date"]), on="event_id", how="inner")
    .merge(actual, on="event_id", how="left")
)

dashboard_scores_validated["scoring_status"] = "validated_oof_oot"

sort_cols = [
    c for c in ["event_date","ticker"]
    if c in dashboard_scores_validated.columns
]
dashboard_scores_validated = (
    dashboard_scores_validated
    .sort_values(sort_cols)
    .reset_index(drop=True)
)

assert dashboard_scores_validated["event_id"].is_unique

required_nonnull = []
for off in [5,7,10]:
    required_nonnull += [
        f"t{off}_score",
        f"t{off}_risk",
        f"t{off}_threshold",
        f"t{off}_split",
    ]

assert dashboard_scores_validated[required_nonnull].notna().all().all(), (
    "validated dashboard score table에 score/risk/threshold 결측이 있습니다."
)

# -------- B. all-event catalog --------
# 전체 Event를 보존하되 score 존재 여부를 명시.
score_presence = (
    validated[["event_id"]]
    .drop_duplicates()
    .assign(scoring_status="validated_oof_oot")
)

dashboard_event_catalog_all = (
    event_meta
    .merge(cutoff_wide, on="event_id", how="left")
    .merge(actual, on="event_id", how="left")
    .merge(score_presence, on="event_id", how="left")
)

dashboard_event_catalog_all["scoring_status"] = (
    dashboard_event_catalog_all["scoring_status"]
    .fillna("train_only_no_oof_score")
)

dashboard_event_catalog_all = (
    dashboard_event_catalog_all
    .sort_values([c for c in ["event_date","ticker"] if c in dashboard_event_catalog_all.columns])
    .reset_index(drop=True)
)

assert dashboard_event_catalog_all["event_id"].is_unique

print("all event rows       :", len(dashboard_event_catalog_all))
print("validated score rows :", len(dashboard_scores_validated))
print("train-only rows      :", (dashboard_event_catalog_all["scoring_status"] == "train_only_no_oof_score").sum())

display(
    dashboard_event_catalog_all["scoring_status"]
    .value_counts(dropna=False)
    .rename_axis("scoring_status")
    .reset_index(name="n")
)

print("\n=== VALIDATED SCORE SAMPLE ===")
display(dashboard_scores_validated.head(10))

### PART C — Dashboard Timeline Asset

UI에서 T+1~4, T+6, T+8~9 등을 계산할 때 단순 calendar day를 더하지 않도록,
실제 cutoff date를 함께 제공합니다.

한 행은 **Event 하나**입니다.

In [ ]:
# 8. Timeline table — 전체 Event 1,250건 보존

timeline_cols = [
    c for c in [
        "event_id","ticker","ticker_name","event_date",
        "t5_cutoff_date","t7_cutoff_date","t10_cutoff_date",
        "direct_t20_date","scoring_status"
    ] if c in dashboard_event_catalog_all.columns
]

dashboard_event_timeline = dashboard_event_catalog_all[timeline_cols].copy()

for a,b in [
    ("event_date","t5_cutoff_date"),
    ("t5_cutoff_date","t7_cutoff_date"),
    ("t7_cutoff_date","t10_cutoff_date"),
    ("t10_cutoff_date","direct_t20_date"),
]:
    if a in dashboard_event_timeline.columns and b in dashboard_event_timeline.columns:
        z = dashboard_event_timeline.dropna(subset=[a,b])
        assert (pd.to_datetime(z[a]) < pd.to_datetime(z[b])).all(), (a,b)

display(dashboard_event_timeline.head())
print("✅ timeline date order QA")

### PART D — Stable Serving Model Copy

STEP 6D에서 만든 `through2025` T+7/T+10 Frozen Model Artifact를
Serving 폴더에 **안정적인 이름으로 복사**합니다.

여기서는 재학습하지 않습니다.

In [ ]:
# 9. Stable model copies + registry

model_copy_map = {
    7: (
        STEP6D_MODEL_DIR / "rf_p_t7_through2025.joblib",
        SERVING_MODEL_DIR / "rf_p_t7.joblib",
    ),
    10: (
        STEP6D_MODEL_DIR / "rf_p_t10_through2025.joblib",
        SERVING_MODEL_DIR / "rf_p_t10.joblib",
    ),
}

registry_models = []

def sha256_file(path):
    h = hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

for timing,(src,dst) in model_copy_map.items():
    shutil.copy2(src,dst)
    bundle = joblib.load(dst)

    assert int(bundle["prediction_offset"]) == timing
    assert abs(float(bundle["classification_threshold"]) - threshold_map[timing]) < 1e-12

    registry_models.append({
        "timing":timing,
        "model_id":f"RF_P_T{timing}",
        "artifact":dst.name,
        "sha256":sha256_file(dst),
        "feature_n":len(bundle["features"]),
        "classification_threshold":float(bundle["classification_threshold"]),
        "threshold_strategy":bundle["threshold_strategy"],
        "train_cutoff_exclusive":bundle["train_cutoff_exclusive"],
        "label_q35":float(bundle["label_q35"]),
    })

print("✅ T7/T10 serving model copies ready")

#### T+5 runtime model에 대한 처리

현재 이 STEP 7은 **T+5 모델을 새로 학습하지 않습니다.**

Historical dashboard의 T+5 score는 STEP 6의 검증 Prediction을 사용합니다.

향후 신규 Event 실시간 추론까지 연결할 때는 T+5 Frozen Runtime Artifact를
별도 model-finalization 단계에서 같은 규격으로 준비하면 됩니다.

즉 현재 웹 시연용 Serving Table은 T+5/T+7/T+10을 모두 지원하지만,
`models/`에는 이번 단계에서 새로 만든 T+7/T+10 stable artifact만 배치합니다.

### PART E — Config / Registry / State Policy

In [ ]:
# 10. Threshold JSON

threshold_payload = {
    "version":"direct_t20_q35_multistage_v1",
    "built_at":datetime.now().isoformat(),
    "target":{
        "name":"Direct T+20 BHAR",
        "label":"train-period q35",
    },
    "score_semantics":"uncalibrated risk score; do not display as literal probability",
    "timings":{
        "t5":{
            "timing":5,
            "role":"main_early_diagnosis",
            "strategy":"F1_MAX",
            "classification_threshold":threshold_map[5],
        },
        "t7":{
            "timing":7,
            "role":"intermediate_update",
            "strategy":"F1_MAX",
            "classification_threshold":threshold_map[7],
        },
        "t10":{
            "timing":10,
            "role":"confirmatory_update",
            "strategy":"F1_MAX",
            "classification_threshold":threshold_map[10],
        },
    }
}

THRESHOLD_JSON = SERVING_CONFIG_DIR / "model_thresholds.json"
with open(THRESHOLD_JSON,"w",encoding="utf-8") as f:
    json.dump(threshold_payload,f,ensure_ascii=False,indent=2)

print(THRESHOLD_JSON)

In [ ]:
# 11. Event stage policy JSON

stage_policy = {
    "version":"multistage_t5_t7_t10_t20_v1",
    "time_basis":"trading_day",
    "checkpoints":[
        {"offset":0, "type":"event", "label":"급등 감지"},
        {"offset":5, "type":"model", "label":"1차 조기진단", "role":"main"},
        {"offset":7, "type":"model", "label":"2차 진단", "role":"intermediate"},
        {"offset":10, "type":"model", "label":"보완진단", "role":"confirmatory"},
        {"offset":20, "type":"actual", "label":"실제 T+20 결과"},
    ],
    "stages":[
        {"from":0,"to":4,"stage":"observing_t5","latest_available":None,"next":5},
        {"from":5,"to":5,"stage":"diagnosed_t5","latest_available":5,"next":7},
        {"from":6,"to":6,"stage":"observing_t7","latest_available":5,"next":7},
        {"from":7,"to":7,"stage":"diagnosed_t7","latest_available":7,"next":10},
        {"from":8,"to":9,"stage":"observing_t10","latest_available":7,"next":10},
        {"from":10,"to":19,"stage":"diagnosed_t10_observing_t20","latest_available":10,"next":20},
        {"from":20,"to":None,"stage":"outcome_available","latest_available":10,"actual":True},
    ],
    "visibility_rules":{
        "t5":"elapsed_trading_days >= 5",
        "t7":"elapsed_trading_days >= 7",
        "t10":"elapsed_trading_days >= 10",
        "actual_t20":"elapsed_trading_days >= 20",
    }
}

STAGE_JSON = SERVING_CONFIG_DIR / "event_stage_policy.json"
with open(STAGE_JSON,"w",encoding="utf-8") as f:
    json.dump(stage_policy,f,ensure_ascii=False,indent=2)

print(STAGE_JSON)

In [ ]:
# 12. Model registry

model_registry = {
    "version":"finda_multistage_model_registry_v1",
    "built_at":datetime.now().isoformat(),
    "historical_dashboard_score_source":"STEP6 validated OOF/OOT predictions",
    "runtime_models":registry_models,
    "t5_runtime_model_status":"not_created_in_this_notebook",
    "t5_historical_score_available":True,
    "notes":[
        "Serving step does not retrain models.",
        "T+5 dashboard score is available from validated prediction table.",
        "T+7/T+10 stable runtime models are copied from STEP6D frozen artifacts."
    ]
}

REGISTRY_JSON = SERVING_META_DIR / "model_registry.json"
with open(REGISTRY_JSON,"w",encoding="utf-8") as f:
    json.dump(model_registry,f,ensure_ascii=False,indent=2)

print(REGISTRY_JSON)

### PART F — Serving 파일 저장 / DuckDB Serving Schema

In [ ]:
# 13. Save dashboard assets

SCORES_PARQUET = SERVING_DASH_DIR / "dashboard_scores_validated_t5_t7_t10.parquet"
SCORES_CSV = SERVING_DASH_DIR / "dashboard_scores_validated_t5_t7_t10.csv"

CATALOG_PARQUET = SERVING_DASH_DIR / "dashboard_event_catalog_all.parquet"
CATALOG_CSV = SERVING_DASH_DIR / "dashboard_event_catalog_all.csv"

TIMELINE_PARQUET = SERVING_DASH_DIR / "dashboard_event_timeline.parquet"
TIMELINE_CSV = SERVING_DASH_DIR / "dashboard_event_timeline.csv"

dashboard_scores_validated.to_parquet(
    SCORES_PARQUET, index=False, compression="zstd"
)
dashboard_scores_validated.to_csv(
    SCORES_CSV, index=False, encoding="utf-8-sig"
)

dashboard_event_catalog_all.to_parquet(
    CATALOG_PARQUET, index=False, compression="zstd"
)
dashboard_event_catalog_all.to_csv(
    CATALOG_CSV, index=False, encoding="utf-8-sig"
)

dashboard_event_timeline.to_parquet(
    TIMELINE_PARQUET, index=False, compression="zstd"
)
dashboard_event_timeline.to_csv(
    TIMELINE_CSV, index=False, encoding="utf-8-sig"
)

print("✅ serving dashboard files saved")

In [ ]:
# 14. DuckDB serving schema

if DB_PATH.exists():
    con = duckdb.connect(str(DB_PATH))
    con.execute("CREATE SCHEMA IF NOT EXISTS serving")

    def replace_table(name, df):
        tmp = "_tmp_" + name.replace(".","_")
        con.register(tmp,df)
        con.execute(f"CREATE OR REPLACE TABLE {name} AS SELECT * FROM {tmp}")
        con.unregister(tmp)

    replace_table(
        "serving.dashboard_scores_validated_t5_t7_t10",
        dashboard_scores_validated
    )
    replace_table(
        "serving.dashboard_event_catalog_all",
        dashboard_event_catalog_all
    )
    replace_table(
        "serving.dashboard_event_timeline",
        dashboard_event_timeline
    )
    con.close()

    print("✅ DuckDB serving.* materialized")
else:
    print("ℹ️ DuckDB 없음 — 파일 산출물만 생성")

In [ ]:
# 15. Serving manifest + optional GCS upload

SERVING_MANIFEST = SERVING_META_DIR / "serving_manifest.json"

asset_paths = [
    SCORES_PARQUET, SCORES_CSV,
    CATALOG_PARQUET, CATALOG_CSV,
    TIMELINE_PARQUET, TIMELINE_CSV,
    THRESHOLD_JSON, STAGE_JSON, REGISTRY_JSON,
    *sorted(SERVING_MODEL_DIR.glob("*.joblib"))
]

manifest = {
    "built_at":datetime.now().isoformat(),
    "layer":"Serving / Consumption",
    "source_layers":[
        "STEP6 validated model predictions",
        "STEP6D frozen thresholds/models",
        "Direct T+20 model frame"
    ],
    "row_grain":{
        "dashboard_scores_validated_t5_t7_t10":"validated event_id 1 row",
        "dashboard_event_catalog_all":"all event_id 1 row",
        "dashboard_event_timeline":"event_id 1 row"
    },
    "assets":{
        p.name:{
            "path":str(p),
            "sha256":sha256_file(p)
        }
        for p in asset_paths
    }
}

with open(SERVING_MANIFEST,"w",encoding="utf-8") as f:
    json.dump(manifest,f,ensure_ascii=False,indent=2)

if UPLOAD_GCS:
    from google.colab import auth
    auth.authenticate_user()
    !gcloud -q config set project {PROJECT_ID}

    for p in [
        SCORES_PARQUET, SCORES_CSV,
        CATALOG_PARQUET, CATALOG_CSV,
        TIMELINE_PARQUET, TIMELINE_CSV
    ]:
        subprocess.run([
            "gcloud","storage","cp",str(p),
            f"gs://{BUCKET}/serving/dashboard/{p.name}"
        ], check=True)

    for p in [THRESHOLD_JSON, STAGE_JSON]:
        subprocess.run([
            "gcloud","storage","cp",str(p),
            f"gs://{BUCKET}/serving/config/{p.name}"
        ], check=True)

    for p in [REGISTRY_JSON, SERVING_MANIFEST]:
        subprocess.run([
            "gcloud","storage","cp",str(p),
            f"gs://{BUCKET}/serving/metadata/{p.name}"
        ], check=True)

    for p in sorted(SERVING_MODEL_DIR.glob("*.joblib")):
        subprocess.run([
            "gcloud","storage","cp",str(p),
            f"gs://{BUCKET}/serving/models/{p.name}"
        ], check=True)

    print("✅ GCS serving upload complete")
else:
    print("ℹ️ UPLOAD_GCS=False — Drive serving/에만 저장")

### PART G — 최종 QA

Serving 단계의 핵심 QA:
- 한 Event 한 행
- Threshold는 Frozen 값 그대로
- T+5/T+7/T+10 score 모두 존재
- Cutoff 날짜가 시간순
- Raw score를 literal probability로 해석하지 않도록 Metadata 명시
- 모델 학습/Threshold 탐색 코드 없음

In [ ]:
# 16. Final QA

assert dashboard_scores_validated["event_id"].is_unique
assert dashboard_event_catalog_all["event_id"].is_unique
assert set([5,7,10]) == set(threshold_map)

for off in [5,7,10]:
    assert f"t{off}_score" in dashboard_scores_validated.columns
    assert f"t{off}_risk" in dashboard_scores_validated.columns
    assert f"t{off}_threshold" in dashboard_scores_validated.columns

score_cols = [f"t{o}_score" for o in [5,7,10]]
risk_cols = [f"t{o}_risk" for o in [5,7,10]]
threshold_cols = [f"t{o}_threshold" for o in [5,7,10]]

assert dashboard_scores_validated[score_cols].notna().all().all()
assert dashboard_scores_validated[risk_cols].notna().all().all()
assert dashboard_scores_validated[threshold_cols].notna().all().all()

print("thresholds:", threshold_map)
print("all event rows       :", len(dashboard_event_catalog_all))
print("validated score rows :", len(dashboard_scores_validated))
print("timeline rows        :", len(dashboard_event_timeline))
print(
    "train-only rows       :",
    (dashboard_event_catalog_all["scoring_status"] == "train_only_no_oof_score").sum()
)
print("models:", [p.name for p in sorted(SERVING_MODEL_DIR.glob("*.joblib"))])

print("\n=== VALIDATED DASHBOARD SAMPLE ===")
display(dashboard_scores_validated.head(10))

print("\n=== ALL EVENT STATUS ===")
display(
    dashboard_event_catalog_all["scoring_status"]
    .value_counts(dropna=False)
    .rename_axis("scoring_status")
    .reset_index(name="n")
)

print("\n✅ STEP7 SERVING REFACTOR v2 COMPLETE")

# STAGE 20: 급등이후 T+20 경로를 가장 잘 구분하는 사건 이전 20D 원천변수 분석

**태그:** 사후 EDA

**원본 노트북:** `급등이후_T_20_경로를_가장_잘_구분하는_사건_이전_20D_원천변수_분석.ipynb`

## 급등이후 T+20 경로를 가장 잘 구분하는 **사건 이전 20D 원천변수** 분석

#### 핵심 질문
> **급등 Event가 확인된 시점 T까지 이미 알 수 있었던 최근 20거래일의
> 외국인·기관·개인 수급, 공매도, 시장상황, 종목 특성 중
> 이후 방향을 가장 잘 구분하는 변수는 무엇인가?**

#### Outcome 2개
1. **Main — 과거 경로유형 재검증**
   - `sustained` vs `overheat_reversal`
   - H20, prediction_offset=0
   - neutral 제외
2. **Secondary — 현재 최종 Direct T20 위험**
   - 비부진(0) vs 부진위험(1)
   - validated OOF/OOT label

#### Predictor
오직:
- `stock_daily.parquet`
- `market_daily.parquet`
- `short_daily.parquet`

에서 Event T 또는 그 이전 정보만 사용합니다.

### 0. 패키지 설치

In [ ]:
!pip -q install "google-cloud-storage>=2.16" "pyarrow>=15" "pandas>=2.1" "numpy>=1.26" "scipy>=1.11" "scikit-learn>=1.4" "statsmodels>=0.14" "matplotlib>=3.8" "tabulate>=0.9"

### 1. 인증 / Config

In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
import json, warnings, zipfile, re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from sklearn.metrics import roc_auc_score
from statsmodels.stats.multitest import multipletests
from google.cloud import storage
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 360)

@dataclass(frozen=True)
class Config:
    project_id: str = GCP_PROJECT_ID
    bucket: str = "finda_project"
    lookback_days: int = 20
    legacy_horizon: int = 20
    legacy_prediction_offset: int = 0
    min_group_n: int = 20
    max_missing_rate: float = 0.60
    alpha: float = 0.05
    top_n: int = 15
    local_root: str = "/content/finda_t20_pre_event_20d_raw_factor"

CFG=Config()
print(json.dumps(asdict(CFG),ensure_ascii=False,indent=2))

ROOT=Path(CFG.local_root)
INPUT=ROOT/"inputs"
TABLE=ROOT/"tables"
FIG=ROOT/"figures"
REPORT=ROOT/"report"
for p in [INPUT,TABLE,FIG,REPORT]:
    p.mkdir(parents=True,exist_ok=True)

client=storage.Client.create_anonymous_client()  # 공개 읽기전용 버킷, 로그인 불필요
bucket=client.bucket(CFG.bucket)
print(f"✅ GCS: gs://{CFG.bucket}")

### 2. GCS canonical / Gold 다운로드

In [ ]:
# 로컬 경로 매핑 — 전부 앞선 STAGE들이 이미 로컬(WORKSPACE_ROOT)에 만들어둔 파일입니다.
# market: STAGE 11·12가 갱신 / events·legacy_targets: STAGE 14 / scores: STAGE 19
PATHS = {
    "market": WORKSPACE_ROOT / "curated" / "market_daily.parquet",
    "events": WORKSPACE_ROOT / "derived" / "events" / "event_master.parquet",
    "legacy_targets": WORKSPACE_ROOT / "derived" / "events" / "event_targets_long.parquet",
    "scores": WORKSPACE_ROOT / "serving" / "dashboard" / "dashboard_scores_validated_t5_t7_t10.parquet",
}

LOCAL = {}
for key, local_path in PATHS.items():
    if not local_path.exists():
        # 이 STAGE만 단독 실행한 경우를 대비해 공개 버킷에서 로그인 없이 복구
        gcs_key = {
            "market": "curated/market_daily.parquet",
            "events": "derived/events/event_master.parquet",
            "legacy_targets": "derived/events/event_targets_long.parquet",
            "scores": "serving/dashboard/dashboard_scores_validated_t5_t7_t10.parquet",
        }[key]
        gcs_public_download(gcs_key, local_path)
    dst = INPUT / local_path.name
    import shutil as _shutil
    _shutil.copyfile(local_path, dst)
    assert dst.exists() and dst.stat().st_size > 0
    LOCAL[key] = dst
    print("✅", key, "->", local_path)

# (재사용) STAGE 3에서 이미 받아온 stock_daily / short_daily를 그대로 사용합니다.
stock=stock_daily.copy()
short=short_daily.copy()
market=pd.read_parquet(LOCAL["market"])
events=pd.read_parquet(LOCAL["events"])
legacy_targets=pd.read_parquet(LOCAL["legacy_targets"])
scores=pd.read_parquet(LOCAL["scores"])

### 3. Schema 확인 — 현재 parquet을 기준으로 동적 컬럼 매핑

In [ ]:
for name,df in [
    ("stock_daily",stock),("market_daily",market),("short_daily",short),
    ("event_master",events),("event_targets_long",legacy_targets),("scores",scores)
]:
    print("\n====",name,df.shape,"====")
    display(pd.DataFrame({"column":df.columns,"dtype":[str(df[c].dtype) for c in df.columns]}))

def detect(df,candidates,required=True):
    lower={str(c).lower():c for c in df.columns}
    for x in candidates:
        if x.lower() in lower:
            return lower[x.lower()]
    if required:
        raise KeyError(f"컬럼 후보 {candidates}를 찾지 못했습니다. 실제={list(df.columns)}")
    return None

def norm_ticker_value(v):
    s=str(v).strip()
    s=re.sub(r"\.0$","",s)
    if s.isdigit() and len(s)<6:
        s=s.zfill(6)
    return s

SC = {
    "ticker": detect(stock, ["ticker", "code", "stock_code"]),
    "date": detect(stock, ["date", "trade_date"]),

    # 가격 / 거래
    "close": detect(stock, ["close", "종가"]),
    "high": detect(stock, ["high", "고가"], False),
    "low": detect(stock, ["low", "저가"], False),
    "volume": detect(stock, ["volume", "거래량"]),
    "trading_value": detect(stock, ["trading_value", "value", "거래대금"]),
    "market_cap": detect(stock, ["market_cap", "marketcap", "시가총액"]),

    # 투자자별 순매수대금
    "foreign": detect(
        stock,
        [
            "foreign_net_buy_value",
            "foreign_net_buy",
            "foreign_net_purchase",
            "외국인순매수",
        ],
    ),
    "institution": detect(
        stock,
        [
            "institution_net_buy_value",
            "institution_net_buy",
            "institutional_net_buy",
            "기관순매수",
        ],
    ),
    "individual": detect(
        stock,
        [
            "individual_net_buy_value",
            "individual_net_buy",
            "retail_net_buy",
            "개인순매수",
        ],
    ),

    # 외국인 보유
    "foreign_ownership": detect(
        stock,
        [
            "foreign_ownership_pct",
            "foreign_ownership_ratio",
            "외국인보유비율",
        ],
        False,
    ),

    # 기업가치
    "per": detect(stock, ["per"], False),
    "pbr": detect(stock, ["pbr"], False),
    "eps": detect(stock, ["eps"], False),
}

print("Detected stock mapping:")
print(json.dumps(SC, ensure_ascii=False, indent=2))

### 4. Key 정규화 / Integrity QA

In [ ]:
def normalize_daily(df,ticker_col,date_col):
    x=df.copy()
    x["ticker_key"]=x[ticker_col].map(norm_ticker_value)
    x["date_key"]=pd.to_datetime(x[date_col],errors="coerce").dt.normalize()
    return x

stock=normalize_daily(stock,SC["ticker"],SC["date"])

market_date=detect(market,["date","trade_date"])
market=market.copy()
market["date_key"]=pd.to_datetime(market[market_date],errors="coerce").dt.normalize()

short_ticker=detect(short,["ticker","code","stock_code"])
short_date=detect(short,["date","trade_date"])
short=normalize_daily(short,short_ticker,short_date)

event_ticker=detect(events,["ticker","code"])
event_date=detect(events,["event_date","date"])
events=events.copy()
events["ticker_key"]=events[event_ticker].map(norm_ticker_value)
events["event_date_key"]=pd.to_datetime(events[event_date],errors="coerce").dt.normalize()
events["event_id"]=events["event_id"].astype(str)

legacy_targets=legacy_targets.copy()
legacy_targets["event_id"]=legacy_targets["event_id"].astype(str)
scores=scores.copy()
scores["event_id"]=scores["event_id"].astype(str)

assert not stock.duplicated(["ticker_key","date_key"]).any()
assert not short.duplicated(["ticker_key","date_key"]).any()
assert not market.duplicated(["date_key"]).any()
assert events["event_id"].is_unique

print("stock:",stock.shape,stock.date_key.min(),"~",stock.date_key.max())
print("market:",market.shape,market.date_key.min(),"~",market.date_key.max())
print("short:",short.shape,short.date_key.min(),"~",short.date_key.max())
print("events:",events.shape,events.event_date_key.min(),"~",events.event_date_key.max())
print("✅ key QA PASS")

### 5. Event별 **T 이전 20D** Feature 생성

예전 분석의 핵심 변수는 현재 canonical에서 다시 계산합니다.

```text
foreign_net_buy_ratio_20d
= sum(foreign net buy, T-19:T)
  / sum(stock trading value, T-19:T)
```

기관/개인도 동일합니다.

`foreign_ownership_change_20d`는 `ownership(T) - ownership(T-20)` percentage-point 차이입니다.

In [ ]:
stock_groups={
    t:g.sort_values("date_key").reset_index(drop=True)
    for t,g in stock.groupby("ticker_key",sort=False)
}

rows=[]
fail_rows=[]

for ev in events[["event_id","ticker_key","event_date_key"]].itertuples(index=False):
    g=stock_groups.get(ev.ticker_key)
    if g is None or g.empty:
        fail_rows.append({"event_id":ev.event_id,"reason":"ticker_missing"})
        continue

    pos=np.flatnonzero(g["date_key"].values==np.datetime64(ev.event_date_key))
    if len(pos)!=1:
        fail_rows.append({"event_id":ev.event_id,"reason":f"event_date_match_{len(pos)}"})
        continue
    i=int(pos[0])

    # 20D sum window T-19:T, plus T-20 anchor for return/ownership change
    if i<CFG.lookback_days:
        fail_rows.append({"event_id":ev.event_id,"reason":"insufficient_20d_history"})
        continue

    w20=g.iloc[i-CFG.lookback_days+1:i+1].copy()
    prior20=g.iloc[i-CFG.lookback_days:i].copy()
    trow=g.iloc[i]
    anchor=g.iloc[i-CFG.lookback_days]

    def num_col(col):
        return pd.to_numeric(w20[col],errors="coerce") if col else pd.Series(dtype=float)

    close_T=pd.to_numeric(pd.Series([trow[SC["close"]]]),errors="coerce").iloc[0]
    close_anchor=pd.to_numeric(pd.Series([anchor[SC["close"]]]),errors="coerce").iloc[0]
    mcap_T=pd.to_numeric(pd.Series([trow[SC["market_cap"]]]),errors="coerce").iloc[0]

    trading20=pd.to_numeric(w20[SC["trading_value"]],errors="coerce")
    trading20_sum=trading20.sum(min_count=max(10,CFG.lookback_days//2))

    volume_prior=pd.to_numeric(prior20[SC["volume"]],errors="coerce")
    tv_prior=pd.to_numeric(prior20[SC["trading_value"]],errors="coerce")
    volume_T=pd.to_numeric(pd.Series([trow[SC["volume"]]]),errors="coerce").iloc[0]
    tv_T=pd.to_numeric(pd.Series([trow[SC["trading_value"]]]),errors="coerce").iloc[0]

    close_path=pd.to_numeric(g.iloc[i-CFG.lookback_days:i+1][SC["close"]],errors="coerce")
    ret_path=close_path.pct_change().dropna()

    rec={
        "event_id":ev.event_id,
        "ticker":ev.ticker_key,
        "event_date":ev.event_date_key,

        "stock_return_20d": close_T/close_anchor-1 if pd.notna(close_T) and pd.notna(close_anchor) and close_anchor!=0 else np.nan,
        "stock_volatility_20d": ret_path.std(ddof=1) if len(ret_path)>=10 else np.nan,
        "volume_ratio_prior20_T": volume_T/volume_prior.mean() if pd.notna(volume_T) and volume_prior.mean()>0 else np.nan,
        "trading_value_ratio_prior20_T": tv_T/tv_prior.mean() if pd.notna(tv_T) and tv_prior.mean()>0 else np.nan,
        "market_cap_T":mcap_T,
        "log_market_cap_T":np.log1p(mcap_T) if pd.notna(mcap_T) and mcap_T>=0 else np.nan,
    }

    if SC["high"] and SC["low"]:
        hi=pd.to_numeric(pd.Series([trow[SC["high"]]]),errors="coerce").iloc[0]
        lo=pd.to_numeric(pd.Series([trow[SC["low"]]]),errors="coerce").iloc[0]
        rec["high_low_range_T"]=(hi-lo)/close_T if pd.notna(hi) and pd.notna(lo) and pd.notna(close_T) and close_T!=0 else np.nan

    for out_key,source_key in [
        ("foreign","foreign"),("institution","institution"),("individual","individual")
    ]:
        s=pd.to_numeric(w20[SC[source_key]],errors="coerce")
        ssum=s.sum(min_count=max(10,CFG.lookback_days//2))
        rec[f"{out_key}_net_buy_20d"]=ssum
        rec[f"{out_key}_net_buy_ratio_20d"]=ssum/trading20_sum if pd.notna(ssum) and pd.notna(trading20_sum) and trading20_sum!=0 else np.nan
        rec[f"{out_key}_net_buy_to_mcap_20d"]=ssum/mcap_T if pd.notna(ssum) and pd.notna(mcap_T) and mcap_T!=0 else np.nan
        rec[f"{out_key}_net_buy_positive_day_share_20d"]=(s>0).mean() if s.notna().sum()>=10 else np.nan

    if SC["foreign_ownership"]:
        own_T=pd.to_numeric(pd.Series([trow[SC["foreign_ownership"]]]),errors="coerce").iloc[0]
        own_anchor=pd.to_numeric(pd.Series([anchor[SC["foreign_ownership"]]]),errors="coerce").iloc[0]
        rec["foreign_ownership_pct_T"]=own_T
        rec["foreign_ownership_change_20d"]=own_T-own_anchor if pd.notna(own_T) and pd.notna(own_anchor) else np.nan

    for k in ["per","pbr","eps"]:
        if SC[k]:
            rec[f"{k}_T"]=pd.to_numeric(pd.Series([trow[SC[k]]]),errors="coerce").iloc[0]

    rows.append(rec)

event_features=pd.DataFrame(rows)
feature_failures=pd.DataFrame(fail_rows)

assert event_features["event_id"].is_unique
print("Feature events:",len(event_features),"/",len(events))
display(event_features.head())
if len(feature_failures):
    display(feature_failures["reason"].value_counts().rename("n").reset_index())

### 6. T시점 market_daily Feature 결합

In [ ]:
# market_daily는 현재 canonical에서 이미 T 이전 정보만으로 계산된 market feature를 T 날짜에 snapshot
MARKET_EXCLUDE={"date_key",market_date}
market_numeric=[
    c for c in market.columns
    if c not in MARKET_EXCLUDE
    and pd.api.types.is_numeric_dtype(market[c])
    and not str(c).lower().endswith("_flag")
]

# date 자체 외 모든 numeric market feature를 사용하되 명시적 미래 토큰은 차단
market_numeric=[
    c for c in market_numeric
    if not any(tok in str(c).lower() for tok in ["future","target","label","score"])
]

market_snap=market[["date_key"]+market_numeric].copy()
rename_market={c:(c if str(c).startswith("market_") else f"market_{c}") for c in market_numeric}
market_snap=market_snap.rename(columns=rename_market)

event_features=event_features.merge(
    market_snap,
    left_on="event_date",
    right_on="date_key",
    how="left",
    validate="many_to_one",
).drop(columns=["date_key"],errors="ignore")

print("market feature N:",len(rename_market))
print(list(rename_market.values()))

### 7. T시점 short_daily Feature 결합

In [ ]:
# short_daily의 현재 canonical feature를 사건일 T에서 사용.
# 미래/target은 제외하고 continuous numeric 중심으로 사용.
short_numeric=[]
for c in short.columns:
    lc=str(c).lower()
    if c in {short_ticker,short_date,"ticker_key","date_key"}:
        continue
    if any(tok in lc for tok in ["future","target","label","score"]):
        continue
    if not pd.api.types.is_numeric_dtype(short[c]):
        continue
    if lc.endswith("_flag") or lc in {"short_ban_period","post_full_reopen"}:
        continue
    short_numeric.append(c)

short_snap=short[["ticker_key","date_key"]+short_numeric].copy()
event_features=event_features.merge(
    short_snap,
    left_on=["ticker","event_date"],
    right_on=["ticker_key","date_key"],
    how="left",
    validate="one_to_one",
).drop(columns=["ticker_key","date_key"],errors="ignore")

print("short feature N:",len(short_numeric))
print(short_numeric)

### 8. Feature Family / leakage 차단

In [ ]:
ID_COLS={"event_id","ticker","event_date"}

CORE_OLD_STYLE=[
    "foreign_net_buy_ratio_20d",
    "foreign_ownership_change_20d",
    "individual_net_buy_ratio_20d",
    "institution_net_buy_ratio_20d",
]

def family(c):
    lc=c.lower()
    if lc.startswith("foreign_"): return "FOREIGN_FLOW"
    if lc.startswith("institution_"): return "INSTITUTION_FLOW"
    if lc.startswith("individual_"): return "INDIVIDUAL_FLOW"
    if lc.startswith("short_"): return "SHORT"
    if lc.startswith("market_"): return "MARKET"
    if lc.startswith(("per_","pbr_","eps_")) or "market_cap" in lc: return "FUNDAMENTAL_SIZE"
    return "STOCK_CONTEXT"

LEAK_TOKENS=("future","target","label","score","direct_bhar","early_","cutoff")

candidate_features=[]
for c in event_features.columns:
    if c in ID_COLS: continue
    if any(t in c.lower() for t in LEAK_TOKENS): continue
    if pd.api.types.is_numeric_dtype(event_features[c]):
        candidate_features.append(c)

catalog=pd.DataFrame({"feature":candidate_features})
catalog["family"]=catalog["feature"].map(family)
display(catalog.groupby("family").size().rename("feature_n").reset_index())
print("candidate N:",len(candidate_features))

### 9. Main Label — H20 `sustained` vs `overheat_reversal`

이 라벨은 새로 만들지 않고 현재 STEP5 Gold `event_targets_long`에서 가져옵니다.

In [ ]:
lt=legacy_targets.copy()
for c in ["prediction_offset","target_horizon"]:
    lt[c]=pd.to_numeric(lt[c],errors="coerce")

required_legacy={"event_id","prediction_offset","target_horizon","label_class"}
missing=required_legacy-set(lt.columns)
if missing:
    raise KeyError(f"legacy target 누락: {missing}")

main_label=lt[
    lt["prediction_offset"].eq(CFG.legacy_prediction_offset)
    & lt["target_horizon"].eq(CFG.legacy_horizon)
][["event_id","label_class"]].copy()

main_label["event_id"]=main_label["event_id"].astype(str)
assert main_label["event_id"].is_unique

print(main_label["label_class"].value_counts(dropna=False))

legacy_data=event_features.merge(
    main_label,on="event_id",how="inner",validate="one_to_one"
)
legacy_data=legacy_data[
    legacy_data["label_class"].isin(["sustained","overheat_reversal"])
].copy()

# favorable=0 sustained, adverse=1 overheat
legacy_data["y"]=legacy_data["label_class"].map({
    "sustained":0,
    "overheat_reversal":1
}).astype(int)

print("Main clean binary N:",len(legacy_data))
display(legacy_data["label_class"].value_counts().rename("n").reset_index())

### 10. Secondary Label — validated Direct T20 비부진 vs 부진위험

In [ ]:
required_score={"event_id","actual_t20_risk_q35_fold_specific"}
missing=required_score-set(scores.columns)
if missing:
    raise KeyError(f"score table 누락: {missing}")

score_cols=["event_id","actual_t20_risk_q35_fold_specific"]
if "scoring_status" in scores.columns:
    score_cols.append("scoring_status")

direct_label=scores[score_cols].copy()
direct_label["event_id"]=direct_label["event_id"].astype(str)
direct_label["actual_t20_risk_q35_fold_specific"]=pd.to_numeric(
    direct_label["actual_t20_risk_q35_fold_specific"],errors="coerce"
)
direct_label=direct_label.dropna(subset=["actual_t20_risk_q35_fold_specific"])
direct_label["y"]=direct_label["actual_t20_risk_q35_fold_specific"].astype(int)

if "scoring_status" in direct_label.columns:
    print(direct_label["scoring_status"].value_counts())

assert direct_label["event_id"].is_unique
assert set(direct_label["y"].unique()).issubset({0,1})

direct_data=event_features.merge(
    direct_label[["event_id","y"]],
    on="event_id",how="inner",validate="one_to_one"
)
print("Secondary validated N:",len(direct_data))
display(direct_data["y"].value_counts().sort_index().rename("n").reset_index())

### 11. 단변량 검정 함수

Rank-biserial 부호는 **favorable group - adverse group** 방향으로 통일합니다.

- Main: `+` = sustained에서 높음 / `-` = overheat_reversal에서 높음
- Secondary: `+` = 비부진에서 높음 / `-` = 부진위험에서 높음

In [ ]:
def holm(p):
    p=np.asarray(p,float)
    out=np.full(len(p),np.nan)
    m=np.isfinite(p)
    if m.sum():
        out[m]=multipletests(p[m],method="holm")[1]
    return out

def run_univariate(df,outcome_name):
    rows=[]
    dropped=[]

    for f in candidate_features:
        x=pd.to_numeric(df[f],errors="coerce").replace([np.inf,-np.inf],np.nan)
        miss=float(x.isna().mean())
        z=pd.DataFrame({"x":x,"y":df["y"]}).dropna()

        favorable=z.loc[z.y.eq(0),"x"].to_numpy(float)
        adverse=z.loc[z.y.eq(1),"x"].to_numpy(float)

        reason=None
        if miss>CFG.max_missing_rate: reason="high_missing"
        elif len(favorable)<CFG.min_group_n or len(adverse)<CFG.min_group_n: reason="small_group"
        elif z.x.nunique()<2: reason="constant"

        if reason:
            dropped.append({
                "outcome":outcome_name,"feature":f,"family":family(f),
                "missing_rate":miss,"fav_n":len(favorable),"adv_n":len(adverse),"reason":reason
            })
            continue

        u=stats.mannwhitneyu(favorable,adverse,alternative="two-sided",method="auto")
        rbc=2*float(u.statistic)/(len(favorable)*len(adverse))-1

        # y=1 adverse. separation AUC ignores sign.
        auc=roc_auc_score(z["y"].astype(int),z["x"].astype(float))
        sep=max(auc,1-auc)

        rows.append({
            "outcome":outcome_name,
            "feature":f,
            "family":family(f),
            "favorable_n":len(favorable),
            "adverse_n":len(adverse),
            "missing_rate":miss,
            "favorable_mean":np.mean(favorable),
            "adverse_mean":np.mean(adverse),
            "favorable_median":np.median(favorable),
            "adverse_median":np.median(adverse),
            "mannwhitney_u":float(u.statistic),
            "p_value":float(u.pvalue),
            "rank_biserial":rbc,
            "abs_rank_biserial":abs(rbc),
            "separation_auc":sep,
        })

    res=pd.DataFrame(rows)
    if res.empty:
        raise RuntimeError(f"{outcome_name}: 분석 가능 feature 없음")

    res["p_value_holm"]=holm(res["p_value"])
    res["significant_holm"]=res["p_value_holm"].le(CFG.alpha)
    res=res.sort_values(
        ["abs_rank_biserial","separation_auc"],
        ascending=False
    ).reset_index(drop=True)
    res["rank"]=np.arange(1,len(res)+1)
    return res,pd.DataFrame(dropped)

legacy_res,legacy_drop=run_univariate(legacy_data,"legacy_h20_path")
direct_res,direct_drop=run_univariate(direct_data,"direct_t20_risk")

print("Legacy analyzed:",len(legacy_res),"Holm sig:",int(legacy_res.significant_holm.sum()))
print("Direct analyzed:",len(direct_res),"Holm sig:",int(direct_res.significant_holm.sum()))
display(legacy_res.head(15))

### 12. 예전 핵심 수급 4개 — 현재 canonical 재검증

In [ ]:
def core_table(res):
    x=res[res["feature"].isin(CORE_OLD_STYLE)].copy()
    order={f:i for i,f in enumerate(CORE_OLD_STYLE)}
    x["_order"]=x["feature"].map(order)
    return x.sort_values("_order").drop(columns="_order")

legacy_core=core_table(legacy_res)
direct_core=core_table(direct_res)

print("=== Main: sustained vs overheat_reversal ===")
display(legacy_core[[
    "feature","family","favorable_median","adverse_median",
    "rank_biserial","p_value_holm","separation_auc"
]])

print("=== Secondary: non-risk vs risk ===")
display(direct_core[[
    "feature","family","favorable_median","adverse_median",
    "rank_biserial","p_value_holm","separation_auc"
]])

### 13. Feature Family 요약

In [ ]:
def family_summary(res):
    out=[]
    for fam,g in res.groupby("family",observed=True):
        g=g.sort_values("abs_rank_biserial",ascending=False)
        b=g.iloc[0]
        out.append({
            "family":fam,
            "feature_n":len(g),
            "holm_significant_n":int(g["significant_holm"].sum()),
            "median_abs_effect":g["abs_rank_biserial"].median(),
            "max_abs_effect":b["abs_rank_biserial"],
            "best_feature":b["feature"],
            "best_effect":b["rank_biserial"],
            "best_holm_p":b["p_value_holm"],
        })
    return pd.DataFrame(out).sort_values(
        ["max_abs_effect","median_abs_effect"],ascending=False
    ).reset_index(drop=True)

legacy_family=family_summary(legacy_res)
direct_family=family_summary(direct_res)

print("=== Legacy H20 Path ===")
display(legacy_family)
print("=== Direct T20 Risk ===")
display(direct_family)

### 14. 시각화 1 — Investor Flow Medians by H20 Path Type

In [ ]:
plot_features=[f for f in CORE_OLD_STYLE if f in legacy_data.columns]
plot_df=legacy_data[["label_class"]+plot_features].copy()

med=plot_df.groupby("label_class")[plot_features].median().T
med=med.reindex(columns=[c for c in ["sustained","overheat_reversal"] if c in med.columns])

fig,ax=plt.subplots(figsize=(10,5))
x=np.arange(len(med))
w=0.36
if "sustained" in med:
    ax.bar(x-w/2,med["sustained"],w,label="Sustained")
if "overheat_reversal" in med:
    ax.bar(x+w/2,med["overheat_reversal"],w,label="Overheat reversal")
ax.axhline(0,linewidth=1)
ax.set_xticks(x,med.index,rotation=28,ha="right")
ax.set_title("Investor Flow Medians by H20 Path Type")
ax.set_ylabel("Median value")
ax.set_xlabel("Pre-event / event-date feature")
ax.legend()
ax.grid(axis="y",alpha=.2)
fig.tight_layout()
FIG1=FIG/"01_investor_flow_medians_legacy_h20.png"
fig.savefig(FIG1,dpi=170,bbox_inches="tight")
plt.show()

### 15. 시각화 2–4 — 전체 변수 / Family

In [ ]:
def effect_plot(res,title,path):
    p=res.head(CFG.top_n).sort_values("rank_biserial")
    fig,ax=plt.subplots(figsize=(10,7))
    ax.barh(p["feature"],p["rank_biserial"])
    ax.axvline(0,linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("Rank-biserial effect (+ favorable / - adverse)")
    ax.set_ylabel("Feature")
    ax.grid(axis="x",alpha=.2)
    fig.tight_layout()
    fig.savefig(path,dpi=170,bbox_inches="tight")
    plt.show()

FIG2=FIG/"02_top_pre_event_separators_legacy_h20.png"
effect_plot(legacy_res,"Top Pre-Event 20D Separators — Legacy H20 Path",FIG2)

lf=legacy_family.sort_values("max_abs_effect")
fig,ax=plt.subplots(figsize=(9,5))
ax.barh(lf["family"],lf["max_abs_effect"])
ax.set_title("Feature Family Separation — Legacy H20 Path")
ax.set_xlabel("Maximum absolute rank-biserial effect")
ax.set_ylabel("Feature family")
ax.grid(axis="x",alpha=.2)
fig.tight_layout()
FIG3=FIG/"03_feature_family_legacy_h20.png"
fig.savefig(FIG3,dpi=170,bbox_inches="tight")
plt.show()

FIG4=FIG/"04_top_pre_event_separators_direct_t20.png"
effect_plot(direct_res,"Top Pre-Event 20D Separators — Direct T20 Risk",FIG4)

### 16. 두 Outcome 효과크기 비교

In [ ]:
effect_compare=legacy_res[[
    "feature","family","rank_biserial","abs_rank_biserial","p_value_holm"
]].rename(columns={
    "rank_biserial":"legacy_effect",
    "abs_rank_biserial":"legacy_abs_effect",
    "p_value_holm":"legacy_holm_p",
}).merge(
    direct_res[[
        "feature","rank_biserial","abs_rank_biserial","p_value_holm"
    ]].rename(columns={
        "rank_biserial":"direct_effect",
        "abs_rank_biserial":"direct_abs_effect",
        "p_value_holm":"direct_holm_p",
    }),
    on="feature",how="inner",validate="one_to_one"
)

effect_compare["min_abs_effect"]=effect_compare[
    ["legacy_abs_effect","direct_abs_effect"]
].min(axis=1)

effect_compare=effect_compare.sort_values(
    "min_abs_effect",ascending=False
).reset_index(drop=True)

display(effect_compare.head(20))

p=effect_compare.head(min(CFG.top_n,len(effect_compare)))
x=np.arange(len(p))
w=.36
fig,ax=plt.subplots(figsize=(13,5))
ax.bar(x-w/2,p["legacy_abs_effect"],w,label="Legacy H20 path")
ax.bar(x+w/2,p["direct_abs_effect"],w,label="Direct T20 risk")
ax.set_xticks(x,p["feature"],rotation=35,ha="right")
ax.set_title("Legacy vs Direct T20 Separation")
ax.set_ylabel("Absolute rank-biserial effect")
ax.set_xlabel("Pre-event feature")
ax.legend()
ax.grid(axis="y",alpha=.2)
fig.tight_layout()
FIG5=FIG/"05_legacy_vs_direct_effect.png"
fig.savefig(FIG5,dpi=170,bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# 연도별 sustained vs overheat_reversal 단변량 구분력 시각화
# - Predictor: Event T 이전 변수만
# - Outcome: legacy H20 path
# - 각 연도별 Top 5 |rank-biserial|
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests

YEAR_MIN_GROUP_N = 15   # 연도별 각 유형 최소 표본
TOP_N_PER_YEAR = 5

year_data = legacy_data.copy()
year_data["year"] = pd.to_datetime(year_data["event_date"]).dt.year

print("=== 연도별 Label 표본 ===")
year_counts = (
    year_data
    .groupby(["year", "label_class"])
    .size()
    .unstack(fill_value=0)
)

display(year_counts)

year_results = []

for year, df_y in year_data.groupby("year"):
    n_sustained = (df_y["y"] == 0).sum()
    n_overheat = (df_y["y"] == 1).sum()

    # 두 집단 중 하나라도 너무 작으면 연도 분석 제외
    if min(n_sustained, n_overheat) < YEAR_MIN_GROUP_N:
        print(
            f"⚠️ {year} 제외: "
            f"sustained={n_sustained}, "
            f"overheat={n_overheat}"
        )
        continue

    temp_rows = []

    for feature in candidate_features:
        x = pd.to_numeric(
            df_y[feature],
            errors="coerce"
        ).replace([np.inf, -np.inf], np.nan)

        tmp = pd.DataFrame({
            "x": x,
            "y": df_y["y"]
        }).dropna()

        sustained = tmp.loc[
            tmp["y"] == 0, "x"
        ].to_numpy(float)

        overheat = tmp.loc[
            tmp["y"] == 1, "x"
        ].to_numpy(float)

        if (
            len(sustained) < YEAR_MIN_GROUP_N
            or len(overheat) < YEAR_MIN_GROUP_N
            or tmp["x"].nunique() < 2
        ):
            continue

        u = stats.mannwhitneyu(
            sustained,
            overheat,
            alternative="two-sided",
            method="auto"
        )

        # + : sustained에서 높음
        # - : overheat_reversal에서 높음
        rank_biserial = (
            2 * float(u.statistic)
            / (len(sustained) * len(overheat))
            - 1
        )

        temp_rows.append({
            "year": year,
            "feature": feature,
            "family": family(feature),

            "sustained_n": len(sustained),
            "overheat_n": len(overheat),

            "sustained_median": np.median(sustained),
            "overheat_median": np.median(overheat),

            "rank_biserial": rank_biserial,
            "abs_rank_biserial": abs(rank_biserial),

            "p_value": float(u.pvalue),
        })

    temp = pd.DataFrame(temp_rows)

    if temp.empty:
        continue

    # 연도별 다중검정 Holm 보정
    temp["p_value_holm"] = multipletests(
        temp["p_value"],
        method="holm"
    )[1]

    temp["significant_holm"] = (
        temp["p_value_holm"] < 0.05
    )

    year_results.append(temp)


year_result = pd.concat(
    year_results,
    ignore_index=True
)

# ------------------------------------------------------------
# 1. 연도별 Top 변수 표
# ------------------------------------------------------------

year_top = (
    year_result
    .sort_values(
        ["year", "abs_rank_biserial"],
        ascending=[True, False]
    )
    .groupby("year")
    .head(TOP_N_PER_YEAR)
    .copy()
)

print("\n=== 연도별 Top 변수 ===")

display(
    year_top[
        [
            "year",
            "feature",
            "family",
            "sustained_median",
            "overheat_median",
            "rank_biserial",
            "p_value_holm",
            "significant_holm",
        ]
    ]
)


# ------------------------------------------------------------
# 2. 연도별 Top 5 시각화
# ------------------------------------------------------------

years = sorted(year_top["year"].unique())

fig, axes = plt.subplots(
    len(years),
    1,
    figsize=(11, 4.2 * len(years))
)

if len(years) == 1:
    axes = [axes]

for ax, year in zip(axes, years):

    p = (
        year_top[
            year_top["year"] == year
        ]
        .sort_values("rank_biserial")
    )

    labels = [
        f"{f} [{fam}]"
        for f, fam in zip(
            p["feature"],
            p["family"]
        )
    ]

    ax.barh(
        labels,
        p["rank_biserial"]
    )

    ax.axvline(
        0,
        linewidth=1
    )

    ax.set_title(
        f"{year} — Top Pre-Event Separators"
    )

    ax.set_xlabel(
        "Rank-biserial effect "
        "(+ Sustained / - Overheat reversal)"
    )

    ax.set_ylabel("Feature")
    ax.grid(
        axis="x",
        alpha=0.2
    )

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 3. 같은 변수의 연도별 효과 변화 Heatmap
# ------------------------------------------------------------

# 전체 기간 기준 강한 변수들을 선택
overall_top_features = (
    legacy_res
    .head(12)["feature"]
    .tolist()
)

heat = (
    year_result[
        year_result["feature"].isin(
            overall_top_features
        )
    ]
    .pivot(
        index="feature",
        columns="year",
        values="rank_biserial"
    )
    .reindex(overall_top_features)
)

fig, ax = plt.subplots(
    figsize=(10, max(5, len(heat) * 0.45))
)

im = ax.imshow(
    heat.values,
    aspect="auto",
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

ax.set_xticks(
    np.arange(len(heat.columns)),
    heat.columns
)

ax.set_yticks(
    np.arange(len(heat.index)),
    heat.index
)

ax.set_title(
    "Yearly Effect Stability of Major Pre-Event Features"
)

ax.set_xlabel("Event year")
ax.set_ylabel("Feature")

cbar = plt.colorbar(
    im,
    ax=ax
)

cbar.set_label(
    "Rank-biserial effect\n"
    "(+ Sustained / - Overheat reversal)"
)

plt.tight_layout()
plt.show()

이 그래프에서 가운데 0을 기준으로:

← 음수                         양수 →
부진위험에서 높음        비부진에서 높음

In [ ]:
# ============================================================
# 발표용 MAIN
# 전체 기간 Direct T+20 Risk — Top Pre-Event Separators
# 전체 validated 1,059 events / 연도 분리 X
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 발표에서 보여줄 상위 변수 수
TOP_N = 8

# 보기 좋은 이름
DISPLAY_NAMES = {
    "high_low_range_T": "Intraday High-Low Range",
    "stock_volatility_20d": "20D Stock Volatility",
    "individual_net_buy_ratio_20d": "20D Individual Net Buy Ratio",
    "trading_value_ratio_prior20_T": "Trading Value Surge",
    "market_below_ma20_ratio": "Market Below MA20 Ratio",
    "foreign_ownership_pct_T": "Foreign Ownership",
    "pbr_T": "PBR",
    "volume_ratio_prior20_T": "Volume Surge",
}

# 전체기간 direct 결과에서 TOP N
plot_df = (
    direct_res
    .head(TOP_N)
    .copy()
)

plot_df["display_name"] = (
    plot_df["feature"]
    .map(DISPLAY_NAMES)
    .fillna(plot_df["feature"])
)

# 수평 bar 정렬
plot_df = plot_df.sort_values(
    "rank_biserial",
    ascending=True
)

fig, ax = plt.subplots(
    figsize=(10, 6)
)

bars = ax.barh(
    plot_df["display_name"],
    plot_df["rank_biserial"]
)

# 0 기준선
ax.axvline(
    0,
    linewidth=1.2
)

# 실제 effect 숫자 표시
for bar, value in zip(
    bars,
    plot_df["rank_biserial"]
):
    y = bar.get_y() + bar.get_height() / 2

    if value < 0:
        x = value - 0.008
        ha = "right"
    else:
        x = value + 0.008
        ha = "left"

    ax.text(
        x,
        y,
        f"{value:+.3f}",
        va="center",
        ha=ha,
        fontsize=10
    )

ax.set_title(
    "Top Pre-Event Separators for T+20 Risk"
)

ax.set_xlabel(
    "Rank-biserial Effect  (+ Non-risk / − Risk)"
)

ax.set_ylabel("")

# 좌우 공간 확보
lim = max(
    0.28,
    plot_df["rank_biserial"].abs().max() * 1.25
)

ax.set_xlim(
    -lim,
    lim
)

ax.grid(
    axis="x",
    alpha=0.2
)

plt.tight_layout()
plt.show()


# 발표용 TOP 변수 표도 같이 확인
display(
    direct_res.head(TOP_N)[
        [
            "feature",
            "family",
            "favorable_median",
            "adverse_median",
            "rank_biserial",
            "p_value_holm",
            "separation_auc",
        ]
    ]
)

## Direct T+20 부진위험 vs 비부진 1059건 기준으로 통일

In [ ]:
# ============================================================
# 발표용 Robustness Check
# Direct T+20 Risk — Year × Feature Family Heatmap
#
# Sample:
#   Validated OOF/OOT 1,059 Events
#
# Outcome:
#   y = 0 : T+20 Non-risk
#   y = 1 : T+20 Risk
#
# Cell value:
#   Mean absolute rank-biserial effect within each year/family
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


# ------------------------------------------------------------
# 0. Config
# ------------------------------------------------------------

YEAR_MIN_GROUP_N = 20

TARGET_FAMILIES = [
    "STOCK_CONTEXT",
    "INDIVIDUAL_FLOW",
    "MARKET",
    "FOREIGN_FLOW",
    "INSTITUTION_FLOW",
    "SHORT",
]

FAMILY_LABELS = {
    "STOCK_CONTEXT": "Stock Context",
    "INDIVIDUAL_FLOW": "Individual Flow",
    "MARKET": "Market",
    "FOREIGN_FLOW": "Foreign Flow",
    "INSTITUTION_FLOW": "Institution Flow",
    "SHORT": "Short Selling",
}


# ------------------------------------------------------------
# 1. Direct T20 데이터에 Event year 추가
# ------------------------------------------------------------

year_df = direct_data.copy()

year_df["year"] = (
    pd.to_datetime(
        year_df["event_date"],
        errors="coerce"
    )
    .dt.year
)

print("Total Direct T20 validated events:", len(year_df))


# ------------------------------------------------------------
# 2. 연도별 Non-risk / Risk 표본 확인
# ------------------------------------------------------------

year_sample = (
    year_df
    .groupby(["year", "y"])
    .size()
    .unstack(fill_value=0)
    .rename(
        columns={
            0: "Non-risk",
            1: "Risk"
        }
    )
)

year_sample["Total"] = (
    year_sample.get("Non-risk", 0)
    + year_sample.get("Risk", 0)
)

print("\n=== Direct T+20 — Yearly Sample Size ===")

display(year_sample)


# ------------------------------------------------------------
# 3. Year × Feature 효과크기 계산
# ------------------------------------------------------------

rows = []

for year, df_y in year_df.groupby("year"):

    nonrisk_n = int((df_y["y"] == 0).sum())
    risk_n = int((df_y["y"] == 1).sum())

    # 한 집단이라도 너무 작으면 해당 연도 제외
    if min(nonrisk_n, risk_n) < YEAR_MIN_GROUP_N:

        print(
            f"⚠️ {year} excluded: "
            f"Non-risk={nonrisk_n}, Risk={risk_n}"
        )

        continue

    for feature in candidate_features:

        fam = family(feature)

        if fam not in TARGET_FAMILIES:
            continue

        x = (
            pd.to_numeric(
                df_y[feature],
                errors="coerce"
            )
            .replace(
                [np.inf, -np.inf],
                np.nan
            )
        )

        tmp = pd.DataFrame({
            "x": x,
            "y": df_y["y"]
        }).dropna()

        nonrisk = (
            tmp.loc[
                tmp["y"] == 0,
                "x"
            ]
            .to_numpy(float)
        )

        risk = (
            tmp.loc[
                tmp["y"] == 1,
                "x"
            ]
            .to_numpy(float)
        )

        # Feature 단위 표본 부족 / 상수 변수 제외
        if (
            len(nonrisk) < YEAR_MIN_GROUP_N
            or len(risk) < YEAR_MIN_GROUP_N
            or tmp["x"].nunique() < 2
        ):
            continue

        u = stats.mannwhitneyu(
            nonrisk,
            risk,
            alternative="two-sided",
            method="auto"
        )

        # ----------------------------------------------------
        # + : Non-risk에서 높은 값
        # - : Risk에서 높은 값
        # ----------------------------------------------------

        rank_biserial = (
            2.0 * float(u.statistic)
            / (len(nonrisk) * len(risk))
            - 1.0
        )

        rows.append({
            "year": int(year),
            "feature": feature,
            "family": fam,

            "nonrisk_n": len(nonrisk),
            "risk_n": len(risk),

            "nonrisk_median": np.median(nonrisk),
            "risk_median": np.median(risk),

            "rank_biserial": rank_biserial,
            "abs_rank_biserial": abs(rank_biserial),

            "p_value": float(u.pvalue),
        })


year_feature_effect_direct = pd.DataFrame(rows)

if year_feature_effect_direct.empty:
    raise RuntimeError(
        "연도별 Direct T20 분석 가능한 변수가 없습니다."
    )


# ------------------------------------------------------------
# 4. Year × Family 요약
# ------------------------------------------------------------

family_year_direct = (
    year_feature_effect_direct
    .groupby(
        ["family", "year"],
        as_index=False
    )
    .agg(
        mean_abs_effect=(
            "abs_rank_biserial",
            "mean"
        ),

        median_abs_effect=(
            "abs_rank_biserial",
            "median"
        ),

        max_abs_effect=(
            "abs_rank_biserial",
            "max"
        ),

        feature_n=(
            "feature",
            "nunique"
        )
    )
)

print(
    "\n=== Direct T+20 — "
    "Year × Feature Family Summary ==="
)

display(
    family_year_direct
    .sort_values(
        ["year", "mean_abs_effect"],
        ascending=[True, False]
    )
)


# ------------------------------------------------------------
# 5. Heatmap용 데이터
# ------------------------------------------------------------

heat_direct = (
    family_year_direct
    .pivot(
        index="family",
        columns="year",
        values="mean_abs_effect"
    )
    .reindex(TARGET_FAMILIES)
)

heat_direct.index = [
    FAMILY_LABELS.get(x, x)
    for x in heat_direct.index
]

print(
    "\n=== Direct T+20 — "
    "Mean Absolute Effect Heatmap Data ==="
)

display(heat_direct)


# ------------------------------------------------------------
# 6. Heatmap
# ------------------------------------------------------------

fig_width = max(
    9,
    1.3 * len(heat_direct.columns) + 3
)

fig_height = max(
    5,
    0.72 * len(heat_direct.index) + 2
)

fig, ax = plt.subplots(
    figsize=(fig_width, fig_height)
)

masked = np.ma.masked_invalid(
    heat_direct.values.astype(float)
)

im = ax.imshow(
    masked,
    aspect="auto",
)

# x-axis
ax.set_xticks(
    np.arange(len(heat_direct.columns))
)

ax.set_xticklabels(
    heat_direct.columns.astype(str)
)

# y-axis
ax.set_yticks(
    np.arange(len(heat_direct.index))
)

ax.set_yticklabels(
    heat_direct.index
)

ax.set_xlabel("Event Year")
ax.set_ylabel("Feature Family")

ax.set_title(
    "Yearly Separation Strength by Feature Family\n"
    "Direct T+20 Non-risk vs Risk"
)


# ------------------------------------------------------------
# 7. Cell 값 표시
# ------------------------------------------------------------

for i in range(len(heat_direct.index)):

    for j in range(len(heat_direct.columns)):

        value = heat_direct.iloc[i, j]

        if pd.isna(value):
            text = "N/A"
        else:
            text = f"{value:.3f}"

        ax.text(
            j,
            i,
            text,
            ha="center",
            va="center",
            fontsize=10,
        )


# ------------------------------------------------------------
# 8. Color bar
# ------------------------------------------------------------

cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.03,
    pad=0.04
)

cbar.set_label(
    "Mean Absolute Rank-biserial Effect"
)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 9. 연도별 가장 강한 Family 확인
# ------------------------------------------------------------

year_best_family = (
    family_year_direct
    .sort_values(
        ["year", "mean_abs_effect"],
        ascending=[True, False]
    )
    .groupby("year")
    .head(1)
    .copy()
)

year_best_family["family_name"] = (
    year_best_family["family"]
    .map(FAMILY_LABELS)
)

print(
    "\n=== Strongest Feature Family by Year ==="
)

display(
    year_best_family[
        [
            "year",
            "family_name",
            "mean_abs_effect",
            "median_abs_effect",
            "max_abs_effect",
            "feature_n",
        ]
    ]
)

In [ ]:
# ============================================================
# 발표용 SECONDARY
# 전체 기간 Direct T+20 Risk — Feature Family 비교
# ============================================================

family_plot = (
    direct_family
    .copy()
    .sort_values(
        "max_abs_effect",
        ascending=True
    )
)

FAMILY_NAMES = {
    "STOCK_CONTEXT": "Stock Context",
    "INDIVIDUAL_FLOW": "Individual Flow",
    "MARKET": "Market",
    "FOREIGN_FLOW": "Foreign Flow",
    "FUNDAMENTAL_SIZE": "Fundamental / Size",
    "INSTITUTION_FLOW": "Institution Flow",
    "SHORT": "Short Selling",
}

family_plot["display_name"] = (
    family_plot["family"]
    .map(FAMILY_NAMES)
    .fillna(family_plot["family"])
)

fig, ax = plt.subplots(
    figsize=(9, 5.5)
)

bars = ax.barh(
    family_plot["display_name"],
    family_plot["max_abs_effect"]
)

for bar, value in zip(
    bars,
    family_plot["max_abs_effect"]
):
    ax.text(
        value + 0.005,
        bar.get_y() + bar.get_height()/2,
        f"{value:.3f}",
        va="center",
        fontsize=10
    )

ax.set_title(
    "Maximum Separation by Feature Family"
)

ax.set_xlabel(
    "Maximum Absolute Rank-biserial Effect"
)

ax.set_ylabel("")

ax.grid(
    axis="x",
    alpha=0.2
)

plt.tight_layout()
plt.show()

### 17. 결과 저장

In [ ]:
paths={
    "event_features":TABLE/"event_pre20d_features.parquet",
    "legacy_result":TABLE/"legacy_h20_univariate_results.csv",
    "direct_result":TABLE/"direct_t20_univariate_results.csv",
    "legacy_core":TABLE/"legacy_old_style_flow_recheck.csv",
    "direct_core":TABLE/"direct_old_style_flow_recheck.csv",
    "legacy_family":TABLE/"legacy_feature_family_summary.csv",
    "direct_family":TABLE/"direct_feature_family_summary.csv",
    "effect_compare":TABLE/"legacy_vs_direct_effect_comparison.csv",
    "catalog":TABLE/"feature_catalog.csv",
    "failures":TABLE/"feature_build_failures.csv",
}

event_features.to_parquet(paths["event_features"],index=False,compression="zstd")
legacy_res.to_csv(paths["legacy_result"],index=False,encoding="utf-8-sig")
direct_res.to_csv(paths["direct_result"],index=False,encoding="utf-8-sig")
legacy_core.to_csv(paths["legacy_core"],index=False,encoding="utf-8-sig")
direct_core.to_csv(paths["direct_core"],index=False,encoding="utf-8-sig")
legacy_family.to_csv(paths["legacy_family"],index=False,encoding="utf-8-sig")
direct_family.to_csv(paths["direct_family"],index=False,encoding="utf-8-sig")
effect_compare.to_csv(paths["effect_compare"],index=False,encoding="utf-8-sig")
catalog.to_csv(paths["catalog"],index=False,encoding="utf-8-sig")
feature_failures.to_csv(paths["failures"],index=False,encoding="utf-8-sig")

for p in list(paths.values())+[FIG1,FIG2,FIG3,FIG4,FIG5]:
    assert p.exists()
    print("✅",p)

### 18. 결과보고서 자동 생성

In [ ]:
def n(v,d=3):
    return "NA" if pd.isna(v) else f"{float(v):.{d}f}"

def row_lines(df,top=10):
    lines=[]
    for r in df.head(top).itertuples(index=False):
        lines.append(
            f"| {int(r.rank)} | `{r.feature}` | {r.family} | "
            f"{n(r.favorable_median,4)} | {n(r.adverse_median,4)} | "
            f"{n(r.rank_biserial,3)} | {n(r.p_value_holm,4)} | {n(r.separation_auc,3)} |"
        )
    return "\n".join(lines)

def fam_lines(df):
    lines=[]
    for r in df.itertuples(index=False):
        lines.append(
            f"| {r.family} | {int(r.feature_n)} | {int(r.holm_significant_n)} | "
            f"{n(r.median_abs_effect,3)} | {n(r.max_abs_effect,3)} | `{r.best_feature}` |"
        )
    return "\n".join(lines)

def core_lines(df):
    lines=[]
    for r in df.itertuples(index=False):
        lines.append(
            f"| `{r.feature}` | {n(r.favorable_median,4)} | {n(r.adverse_median,4)} | "
            f"{n(r.rank_biserial,3)} | {n(r.p_value_holm,4)} |"
        )
    return "\n".join(lines)

legacy_best=legacy_res.iloc[0]
direct_best=direct_res.iloc[0]
common_best=effect_compare.iloc[0] if len(effect_compare) else None

report=f"""# FinDA — T+20 방향을 구분하는 사건 이전 20D 원천변수 분석 결과보고서

## 1. 목적
T 이후 후행경로를 사용하지 않고, `stock_daily`, `market_daily`, `short_daily`의 Event T 또는 이전 20거래일 정보만으로 이후 방향을 가장 잘 구분하는 변수를 분석했다.

## 2. 표본
- Event feature 생성: {len(event_features):,}건 / event_master {len(events):,}건
- Main legacy clean binary: {len(legacy_data):,}건
  - sustained: {(legacy_data.label_class=="sustained").sum():,}
  - overheat_reversal: {(legacy_data.label_class=="overheat_reversal").sum():,}
- Secondary Direct T20 validated: {len(direct_data):,}건
  - non-risk: {(direct_data.y==0).sum():,}
  - risk: {(direct_data.y==1).sum():,}
- 분석 후보 Feature: {len(candidate_features):,}개

## 3. Main — 지속상승형 vs 과열반전형

가장 큰 단변량 구분 변수:
- `{legacy_best.feature}`
- Family: **{legacy_best.family}**
- sustained 중앙값: **{n(legacy_best.favorable_median,4)}**
- overheat_reversal 중앙값: **{n(legacy_best.adverse_median,4)}**
- Rank-biserial: **{n(legacy_best.rank_biserial,3)}**
- Holm p: **{n(legacy_best.p_value_holm,4)}**
- Separation AUC: **{n(legacy_best.separation_auc,3)}**

### 예전 핵심 수급 4개 재검증

| 변수 | 지속상승 중앙값 | 과열반전 중앙값 | Rank-biserial | Holm p |
|---|---:|---:|---:|---:|
{core_lines(legacy_core)}

### 전체 TOP 10

| 순위 | 변수 | Family | 지속상승 중앙값 | 과열반전 중앙값 | Effect | Holm p | Sep. AUC |
|---:|---|---|---:|---:|---:|---:|---:|
{row_lines(legacy_res)}

### Feature Family

| Family | 변수 수 | Holm 유의 수 | Median abs effect | Max abs effect | Best variable |
|---|---:|---:|---:|---:|---|
{fam_lines(legacy_family)}

## 4. Secondary — Direct T20 비부진 vs 부진위험

가장 큰 단변량 구분 변수:
- `{direct_best.feature}`
- Family: **{direct_best.family}**
- 비부진 중앙값: **{n(direct_best.favorable_median,4)}**
- 부진위험 중앙값: **{n(direct_best.adverse_median,4)}**
- Rank-biserial: **{n(direct_best.rank_biserial,3)}**
- Holm p: **{n(direct_best.p_value_holm,4)}**
- Separation AUC: **{n(direct_best.separation_auc,3)}**

### 예전 핵심 수급 4개

| 변수 | 비부진 중앙값 | 부진위험 중앙값 | Rank-biserial | Holm p |
|---|---:|---:|---:|---:|
{core_lines(direct_core)}

### 전체 TOP 10

| 순위 | 변수 | Family | 비부진 중앙값 | 부진위험 중앙값 | Effect | Holm p | Sep. AUC |
|---:|---|---|---:|---:|---:|---:|---:|
{row_lines(direct_res)}

### Feature Family

| Family | 변수 수 | Holm 유의 수 | Median abs effect | Max abs effect | Best variable |
|---|---:|---:|---:|---:|---|
{fam_lines(direct_family)}

## 5. 두 Outcome 공통 강한 변수

공통 순위는 두 Outcome의 `abs(rank-biserial)` 중 작은 값을 기준으로 정렬했다.

{effect_compare.head(15)[["feature","family","legacy_effect","direct_effect","min_abs_effect"]].to_markdown(index=False)}

## 6. 해석 원칙
- Main 결과는 과거 “지속 상승형 vs 단기 과열형” 질문을 현재 canonical 데이터에서 재검증한 것이다.
- Secondary 결과는 동일한 사건 이전 변수가 현재 Direct T20 위험정의에서도 방향성을 유지하는지 보는 강건성 분석이다.
- 모든 Predictor는 Event T 또는 이전만 사용했다.
- `early_return`, `early_true_mdd`, `early_giveback_ratio` 등 T 이후 변수는 사용하지 않았다.
- 효과크기는 단변량 설명력이며 인과관계나 독립 예측성능을 뜻하지 않는다.
"""

REPORT_PATH=REPORT/"FinDA_T20_사전20D_원천변수_구분력_분석_결과보고서.md"
REPORT_PATH.write_text(report,encoding="utf-8")
print(report)
print("\n✅",REPORT_PATH)

In [ ]:
# ============================================================
# Year × Feature Family Heatmap
# sustained vs overheat_reversal
#
# Cell value:
#   Mean absolute rank-biserial effect size
#
# Larger value = stronger univariate separation
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------

YEAR_MIN_GROUP_N = 15

TARGET_FAMILIES = [
    "FOREIGN_FLOW",
    "INSTITUTION_FLOW",
    "INDIVIDUAL_FLOW",
    "SHORT",
    "MARKET",
]

# 발표용 표시명
FAMILY_LABELS = {
    "FOREIGN_FLOW": "Foreign Flow",
    "INSTITUTION_FLOW": "Institution Flow",
    "INDIVIDUAL_FLOW": "Individual Flow",
    "SHORT": "Short Selling",
    "MARKET": "Market",
}


# ------------------------------------------------------------
# 1. Event year 생성
# ------------------------------------------------------------

year_df = legacy_data.copy()

year_df["year"] = (
    pd.to_datetime(year_df["event_date"])
    .dt.year
)


# ------------------------------------------------------------
# 2. 연도별 표본 수 확인
# ------------------------------------------------------------

year_sample = (
    year_df
    .groupby(["year", "label_class"])
    .size()
    .unstack(fill_value=0)
)

print("=== 연도별 sustained / overheat_reversal 표본 ===")
display(year_sample)


# ------------------------------------------------------------
# 3. 연도 × Feature별 Rank-biserial 계산
# ------------------------------------------------------------

rows = []

for year, df_y in year_df.groupby("year"):

    sustained_n = int((df_y["y"] == 0).sum())
    overheat_n = int((df_y["y"] == 1).sum())

    # 연도 전체에서 어느 한쪽 집단이 너무 작으면 제외
    if min(sustained_n, overheat_n) < YEAR_MIN_GROUP_N:

        print(
            f"⚠️ {year} 제외 "
            f"(Sustained={sustained_n}, "
            f"Overheat={overheat_n})"
        )

        continue

    for feature in candidate_features:

        fam = family(feature)

        # 우리가 이번 그림에서 볼 Family만
        if fam not in TARGET_FAMILIES:
            continue

        x = (
            pd.to_numeric(
                df_y[feature],
                errors="coerce"
            )
            .replace([np.inf, -np.inf], np.nan)
        )

        temp = pd.DataFrame({
            "x": x,
            "y": df_y["y"],
        }).dropna()

        sustained = temp.loc[
            temp["y"] == 0,
            "x"
        ].to_numpy(float)

        overheat = temp.loc[
            temp["y"] == 1,
            "x"
        ].to_numpy(float)

        # Feature 자체의 유효 표본도 체크
        if (
            len(sustained) < YEAR_MIN_GROUP_N
            or len(overheat) < YEAR_MIN_GROUP_N
            or temp["x"].nunique() < 2
        ):
            continue

        u = stats.mannwhitneyu(
            sustained,
            overheat,
            alternative="two-sided",
            method="auto",
        )

        # + : sustained에서 높음
        # - : overheat_reversal에서 높음
        rank_biserial = (
            2.0 * float(u.statistic)
            / (len(sustained) * len(overheat))
            - 1.0
        )

        rows.append({
            "year": year,
            "feature": feature,
            "family": fam,
            "rank_biserial": rank_biserial,
            "abs_rank_biserial": abs(rank_biserial),
            "sustained_n": len(sustained),
            "overheat_n": len(overheat),
        })


year_feature_effect = pd.DataFrame(rows)

if year_feature_effect.empty:
    raise RuntimeError(
        "연도별로 분석 가능한 Feature가 없습니다."
    )


# ------------------------------------------------------------
# 4. Family별 평균 효과크기
# ------------------------------------------------------------

family_year_effect = (
    year_feature_effect
    .groupby(
        ["family", "year"],
        as_index=False
    )
    .agg(
        mean_abs_effect=(
            "abs_rank_biserial",
            "mean"
        ),
        median_abs_effect=(
            "abs_rank_biserial",
            "median"
        ),
        max_abs_effect=(
            "abs_rank_biserial",
            "max"
        ),
        feature_n=(
            "feature",
            "nunique"
        ),
    )
)

print("=== Year × Family effect summary ===")

display(
    family_year_effect
    .sort_values(
        ["year", "mean_abs_effect"],
        ascending=[True, False]
    )
)


# ------------------------------------------------------------
# 5. Heatmap용 Pivot
# ------------------------------------------------------------

heat = (
    family_year_effect
    .pivot(
        index="family",
        columns="year",
        values="mean_abs_effect"
    )
    .reindex(TARGET_FAMILIES)
)

heat.index = [
    FAMILY_LABELS.get(x, x)
    for x in heat.index
]

print("=== Mean absolute effect heatmap data ===")
display(heat)


# ------------------------------------------------------------
# 6. Heatmap
# ------------------------------------------------------------

fig_width = max(
    8,
    1.35 * len(heat.columns) + 3
)

fig_height = max(
    4.8,
    0.8 * len(heat.index) + 1.8
)

fig, ax = plt.subplots(
    figsize=(fig_width, fig_height)
)

masked = np.ma.masked_invalid(
    heat.values.astype(float)
)

im = ax.imshow(
    masked,
    aspect="auto",
)

# Axis
ax.set_xticks(
    np.arange(len(heat.columns))
)

ax.set_xticklabels(
    heat.columns.astype(str)
)

ax.set_yticks(
    np.arange(len(heat.index))
)

ax.set_yticklabels(
    heat.index
)

ax.set_xlabel("Event Year")
ax.set_ylabel("Feature Family")

ax.set_title(
    "Yearly Separation Strength by Feature Family\n"
    "Sustained vs Overheat Reversal"
)


# ------------------------------------------------------------
# 7. 각 Cell에 실제 effect size 표시
# ------------------------------------------------------------

finite_values = heat.values[
    np.isfinite(heat.values)
]

threshold = (
    np.nanmedian(finite_values)
    if len(finite_values)
    else 0
)

for i in range(len(heat.index)):

    for j in range(len(heat.columns)):

        value = heat.iloc[i, j]

        if pd.isna(value):
            text = "N/A"
        else:
            text = f"{value:.3f}"

        ax.text(
            j,
            i,
            text,
            ha="center",
            va="center",
            fontsize=10,
        )


# ------------------------------------------------------------
# 8. Color bar
# ------------------------------------------------------------

cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.03,
    pad=0.04
)

cbar.set_label(
    "Mean Absolute Rank-biserial Effect"
)

plt.tight_layout()

plt.show()

### 19. Final QA

In [ ]:
qa=pd.DataFrame([
    ["event_feature_unique",event_features.event_id.is_unique,f"N={len(event_features)}"],
    ["no_post_event_feature",all("early_" not in f.lower() and "future" not in f.lower() for f in candidate_features),"early/future excluded"],
    ["legacy_binary",set(legacy_data.y.unique()).issubset({0,1}),legacy_data.label_class.value_counts().to_dict()],
    ["direct_binary",set(direct_data.y.unique()).issubset({0,1}),direct_data.y.value_counts().to_dict()],
    ["core_flow_built",all(f in event_features.columns for f in ["foreign_net_buy_ratio_20d","individual_net_buy_ratio_20d","institution_net_buy_ratio_20d"]),"old-style ratios"],
    ["legacy_features",len(legacy_res)>0,f"N={len(legacy_res)}"],
    ["direct_features",len(direct_res)>0,f"N={len(direct_res)}"],
    ["effect_bounds",legacy_res.rank_biserial.between(-1,1).all() and direct_res.rank_biserial.between(-1,1).all(),"[-1,1]"],
    ["artifacts",all(p.exists() and p.stat().st_size>0 for p in list(paths.values())+[FIG1,FIG2,FIG3,FIG4,FIG5,REPORT_PATH]),"tables+figures+report"],
],columns=["check","pass","detail"])

display(qa)
if not qa["pass"].all():
    raise RuntimeError("FINAL QA FAIL\n"+qa.loc[~qa["pass"],["check","detail"]].to_string(index=False))
print("✅ FINAL QA — ALL PASS")

### 20. 결과 ZIP 다운로드

In [ ]:
ZIP_PATH=ROOT/"FinDA_T20_PreEvent20D_RawFactor_Results.zip"
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH,"w",zipfile.ZIP_DEFLATED) as z:
    for base in [TABLE,FIG,REPORT]:
        for p in base.rglob("*"):
            if p.is_file():
                z.write(p,arcname=str(p.relative_to(ROOT)))

print("✅",ZIP_PATH)
from google.colab import files
files.download(str(ZIP_PATH))